In [3]:
# === RESUME SETUP (run first) ===
import os

# After you've done "Add Data > Notebook Output Files" with your last saved version,
# set this to the mounted path. Leave as None for a totally fresh run.
PREV_RUN_DIR = "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap" # e.g. "/kaggle/input/rog-ap-6"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # cuts CUDA OOM from fragmentation

if PREV_RUN_DIR and os.path.exists(PREV_RUN_DIR):
    print("Previous run found at:", PREV_RUN_DIR)
else:
    print("No previous run mounted — starting fresh, or PREV_RUN_DIR not set yet.")

Previous run found at: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap


In [4]:
import torch
_torch_pin = f"torch=={torch.__version__.split('+')[0]}"
print("Keeping installed PyTorch:", _torch_pin)

# accelerate==0.33.0 caps numpy<2.0, which drags Kaggle's numpy back from 2.x
# to 1.26.4 -- but Kaggle's pandas wheel is built against numpy 2.x's C ABI,
# so that downgrade breaks pandas at import time ("numpy.dtype size changed").
# accelerate>=0.34 dropped that cap, so bump it and pin numpy explicitly so
# pip can't silently downgrade it again.
!pip install -q \
    --only-binary=transformers,tokenizers,peft,sentencepiece,accelerate,datasets,numpy,torch \
    "transformers==4.44.2" "tokenizers==0.19.1" "peft==0.12.0" \
    "sentencepiece==0.2.0" "accelerate==0.34.2" "datasets==2.20.0" \
    "numpy>=2.0,<2.1" "graph-walker==1.0.6" "{_torch_pin}"

Keeping installed PyTorch: torch==2.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

### Confirmation of the Packages

In [5]:
import transformers, tokenizers, networkx
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("networkx:", networkx.__version__)

transformers: 4.44.2
tokenizers: 0.19.1
networkx: 3.6.1


### Cloning official REPO


In [6]:
# (optional) from huggingface_hub import login; login(token="<your_hf_token>")
import os, sys

REPO_DIR = "/kaggle/working/reasoning-on-graphs"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/RManLuo/reasoning-on-graphs.git {REPO_DIR}

sys.path.append(os.path.join(REPO_DIR, "src"))

# RQ1
# Objective 1: Baseline Establishment

## RoG-WebQSP Baseline

### Config

In [7]:
N_QUESTIONS = 50          # Step 1 checklist: "50-100 questions, not the whole dataset"
DATASET_NAME = "rmanluo/RoG-webqsp"
SPLIT = "test"
MODEL_PATH = "rmanluo/RoG"   # official pre-trained RoG checkpoint (planning + reasoning, same model)
N_BEAM = 3                 # matches the paper's K=3 (Section 5.4) and scripts/planning.sh
OUTPUT_DIR = "/kaggle/working/step1_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Load Dataset Subset

In [8]:
from datasets import load_dataset

full_test = load_dataset(DATASET_NAME, split=SPLIT)
subset = full_test.select(range(min(N_QUESTIONS, len(full_test))))
print(f"Loaded {len(subset)} / {len(full_test)} WebQSP test questions")
print("Fields:", subset.column_names)
subset[0]

README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

data/test-00000-of-00002-9ee8d68f7d951e1(…):   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/2826 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/246 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1628 [00:00<?, ? examples/s]

Loaded 50 / 1628 WebQSP test questions
Fields: ['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']


{'id': 'WebQTest-0',
 'question': 'what does jamaican people speak',
 'answer': ['Jamaican English', 'Jamaican Creole English Language'],
 'q_entity': ['Jamaica'],
 'a_entity': ['Jamaican English', 'Jamaican Creole English Language'],
 'graph': [['Jamaica',
   'meteorology.cyclone_affected_area.cyclones',
   'Tropical Storm Keith'],
  ['Jamaica',
   'location.statistical_region.prevalence_of_undernourisment',
   'g.12tb6gh4f'],
  ['Latoya Greaves', 'olympics.olympic_athlete.country', 'm.0k8nh0b'],
  ['Jamaica',
   'location.statistical_region.electricity_consumption_per_capita',
   'm.0nf4wmg'],
  ['Hurricane Hilda',
   'meteorology.tropical_cyclone.affected_areas',
   'Yucatán Peninsula'],
  ['m.0wj6j0d',
   'sports.competitor_competition_relationship.tournament',
   '2013 World Championships in Athletics'],
  ['Jamaica',
   'location.statistical_region.market_cap_of_listed_companies_as_percent_of_gdp',
   'g.1hhc3gxpy'],
  ['Jamaica',
   'location.statistical_region.energy_use_per_ca

### Load the model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, clean_up_tokenization_spaces=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", torch_dtype=torch.float16)
model.eval()
print("Model loaded.")

tokenizer_config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/78.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

Model loaded.


### Decode Patch

In [10]:
_original_decode = tokenizer.decode  # stash pre-patch decode for later verification (Part 2)

import types

def _sp_decode(self, token_ids, skip_special_tokens=True, **kwargs):
    tokens = self.convert_ids_to_tokens(token_ids, skip_special_tokens=skip_special_tokens)
    return self.sp_model.decode(tokens)

tokenizer.decode = types.MethodType(_sp_decode, tokenizer)
print("Patched tokenizer.decode to use sp_model.decode directly.")

Patched tokenizer.decode to use sp_model.decode directly.


### Diagnostic

In [11]:
# Diagnostic -- run with your CURRENT tokenizer, no reinstall needed
print("Tokenizer class:", type(tokenizer))
print("Has sp_model:", hasattr(tokenizer, "sp_model"))

test = "location.location.languages_spoken"
ids = tokenizer.encode(test, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(ids)
print("tokens:", tokens)
print("tokenizer.decode():   ", repr(tokenizer.decode(ids, clean_up_tokenization_spaces=True)))

if hasattr(tokenizer, "sp_model"):
    print("sp_model.decode() direct:", repr(tokenizer.sp_model.decode(tokens)))

Tokenizer class: <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>
Has sp_model: True
tokens: ['▁location', '.', 'location', '.', 'l', 'anguages', '_', 'sp', 'oken']
tokenizer.decode():    ' location.location.languages_spoken'
sp_model.decode() direct: ' location.location.languages_spoken'


### Importing official planning functions

In [12]:
from utils.utils import InstructFormater
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

prompter = InstructFormater(os.path.join(REPO_DIR, "prompts", "llama2.txt"))
print("Planning instruction:", repr(INSTRUCTION))

Planning instruction: 'Please generate a valid relation path that can be helpful for answering the following question: '


### Run Planning (relation-path generation)

In [13]:
import time, json
from tqdm.auto import tqdm

planning_records = []
for sample in tqdm(subset, desc="Planning"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    t0 = time.time()
    raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
    planning_time = time.time() - t0
    rel_paths = parse_prediction(raw_output["paths"])   # top-N_BEAM predicted relation paths
    planning_records.append({
        "id": sample["id"],
        "question": sample["question"],
        "q_entity": sample["q_entity"],
        "a_entity": sample["a_entity"],
        "graph": sample["graph"],
        "predicted_paths": rel_paths,
        "planning_time_sec": planning_time,
    })

print(f"Generated relation-path plans for {len(planning_records)} questions.")
print("Example plan:", planning_records[0]["predicted_paths"])

Planning:   0%|          | 0/50 [00:00<?, ?it/s]

Generated relation-path plans for 50 questions.
Example plan: [['location.country.languages_spoken'], ['language.human_language.countries_spoken_in'], ['location.country.official_language']]


### Instrumented BFS Wrapper

In [14]:
# Adds READ-ONLY per-hop frontier counters. Does NOT change which nodes get
# expanded or pruned -- traversal order and the relation-match pruning rule
# are copied verbatim from src/utils/graph_utils.py.
from collections import deque
from utils.graph_utils import build_graph, bfs_with_rule   # the OFFICIAL, unmodified function

def bfs_with_rule_instrumented(graph, start_node, target_rule):
    result_paths = []
    hop_frontier = [0] * len(target_rule)   # entities admitted into the frontier at each hop
    queue = deque([(start_node, [])])
    while queue:
        current_node, current_path = queue.popleft()
        if len(current_path) == len(target_rule):
            result_paths.append(current_path)
        if len(current_path) < len(target_rule):
            if current_node not in graph:
                continue
            hop_idx = len(current_path)
            for neighbor in graph.neighbors(current_node):
                rel = graph[current_node][neighbor]["relation"]
                if rel != target_rule[hop_idx] or len(current_path) > len(target_rule):
                    continue
                queue.append((neighbor, current_path + [(current_node, rel, neighbor)]))
                hop_frontier[hop_idx] += 1
    return result_paths, hop_frontier

### Sanity check (Show 0 mismatches)

In [15]:
mismatches = 0
checked = 0
for rec in planning_records:
    graph = build_graph(rec["graph"])
    for entity in rec["q_entity"]:
        for rule in rec["predicted_paths"]:
            checked += 1
            official_result = bfs_with_rule(graph, entity, rule)
            instrumented_result, _ = bfs_with_rule_instrumented(graph, entity, rule)
            if official_result != instrumented_result:
                mismatches += 1

print(f"Checked {checked} (entity, relation-path) calls. Mismatches: {mismatches}")
assert mismatches == 0, "Instrumented BFS diverges from official RoG retrieval -- STOP, do not trust downstream numbers."

Checked 150 (entity, relation-path) calls. Mismatches: 0


### Running Retrieval

In [16]:
def path_endpoints(paths):
    return set(p[-1][-1] for p in paths if len(p) > 0)

retrieval_records = []
for rec in tqdm(planning_records, desc="Retrieval"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        retrieval_records.append({
            "question_id": rec["id"],
            "question": rec["question"],
            "topic_entity": rec["q_entity"],
            "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,   # [hop_1_frontier, hop_2_frontier, ...]
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(retrieval_records)} (question, relation-plan) retrieval records collected.")

Retrieval:   0%|          | 0/50 [00:00<?, ?it/s]

147 (question, relation-plan) retrieval records collected.


### Question Evaluation

In [17]:
import pandas as pd
df = pd.DataFrame(retrieval_records)

question_coverage = (
    df.groupby("question_id")["gold_reachable"]
      .any()
)

print("Questions evaluated:", len(question_coverage))
print("Questions reaching at least one gold answer:", question_coverage.sum())
print("Question-level retrieval coverage:",
      question_coverage.mean() * 100)

Questions evaluated: 50
Questions reaching at least one gold answer: 38
Question-level retrieval coverage: 76.0


### Computing missed_qids

In [18]:
from collections import defaultdict

reachable_by_question = defaultdict(bool)
for rec in retrieval_records:
    reachable_by_question[rec["question_id"]] |= rec["gold_reachable"]

all_qids = {rec["question_id"] for rec in retrieval_records}
missed_qids = [qid for qid in all_qids if not reachable_by_question[qid]]

print(f"{len(missed_qids)} / {len(all_qids)} questions missed")

12 / 50 questions missed


### Revised Failure Analysis

In [19]:
from collections import deque

def find_gold_paths_official_graph(rec, max_hops=2):
    G = build_graph(rec["graph"])
    gold = set(rec["a_entity"])
    found = []

    for start in rec["q_entity"]:
        queue = deque([(start, [], [start])])

        while queue:
            node, relations, entities = queue.popleft()

            if len(relations) > 0 and node in gold:
                found.append({
                    "relations": relations,
                    "entities": entities
                })

            if len(relations) == max_hops:
                continue

            if node not in G:
                continue

            for neighbor in G.neighbors(node):
                rel = G[node][neighbor]["relation"]

                queue.append((
                    neighbor,
                    relations + [rel],
                    entities + [neighbor]
                ))

    return found

for qid in missed_qids:

    rec = next(r for r in planning_records if r["id"] == qid)

    gold_paths = find_gold_paths_official_graph(rec, max_hops=2)

    predicted = {
        tuple(p) for p in rec["predicted_paths"]
    }

    gold_relation_paths = {
        tuple(p["relations"]) for p in gold_paths
    }

    print("\n" + "="*100)
    print("QUESTION:", rec["question"])
    print("GOLD:", rec["a_entity"])

    print("\nPredicted:")
    for p in predicted:
        print(" ", " -> ".join(p))

    print("\nGold-supporting paths in OFFICIAL BUILT GRAPH:")
    for p in gold_paths[:10]:
        print(
            " ",
            " -> ".join(p["entities"]),
            "\n    ",
            " -> ".join(p["relations"])
        )

    if not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"

    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"

    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    print("\nDIAGNOSIS:", diagnosis)


QUESTION: who plays ken barlow in coronation street
GOLD: ['William Roache']

Predicted:
  tv.tv_program.country_of_origin -> people.person.nationality
  tv.regular_tv_appearance.series -> tv.regular_tv_appearance.actor
  tv.regular_tv_appearance.series -> tv.tv_actor.starring_roles

Gold-supporting paths in OFFICIAL BUILT GRAPH:

DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: who is governor of ohio 2011
GOLD: ['John Kasich', 'Ted Strickland', 'Return J. Meigs, Jr.']

Predicted:
  government.governmental_jurisdiction.governing_officials -> government.government_position_held.office_holder
  government.government_position_held.jurisdiction_of_office -> government.government_position_held.office_holder
  government.governmental_jurisdiction.governing_officials -> government.politician.government_positions_held

Gold-supporting paths in OFFICIAL BUILT GRAPH:
  Ohio -> United States of America -> Return J. Meigs, Jr. 
     base.locations.states_and_provences.country -> people.pers

### JSON, CSV save

In [20]:
import pandas as pd

json_path = os.path.join(OUTPUT_DIR, "step1_baseline_webqsp.json")
csv_path = os.path.join(OUTPUT_DIR, "step1_baseline_webqsp.csv")

with open(json_path, "w") as f:
    json.dump(retrieval_records, f, indent=2)

df = pd.DataFrame(retrieval_records)
df.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)
df[["retrieved_paths_count", "gold_reachable", "retrieval_time_sec"]].describe()

Saved: /kaggle/working/step1_baseline/step1_baseline_webqsp.json
Saved: /kaggle/working/step1_baseline/step1_baseline_webqsp.csv


,retrieved_paths_count,retrieval_time_sec
count,147.000000,147.000000
mean,6.564626,0.000197
std,27.793531,0.000231
min,0.000000,0.000007
25%,0.000000,0.000056
50%,1.000000,0.000141
75%,3.000000,0.000217
max,320.000000,0.001233


### Worked Example

In [21]:
example = next(r for r in retrieval_records if len(r["predicted_relation_path"]) > 0)

print("Question:")
print(example["question"])
print()
print("Topic entity:")
print(example["topic_entity"])
print()
print("RoG relation plan:")
print(" -> ".join(example["predicted_relation_path"]))
print()
for i, count in enumerate(example["hop_frontiers"], start=1):
    print(f"Hop {i}:")
    print(f"{count} candidate entities")
    print()
print("Retrieved paths:")
print(example["retrieved_paths_count"])
print()
print("Gold answer reachable:")
print("Yes" if example["gold_reachable"] else "No")
print()
print("Retrieval time:")
print(f"{example['retrieval_time_sec']*1000:.1f} ms")

Question:
what does jamaican people speak

Topic entity:
['Jamaica']

RoG relation plan:
location.country.languages_spoken

Hop 1:
1 candidate entities

Retrieved paths:
1

Gold answer reachable:
Yes

Retrieval time:
1.2 ms


### Final RoG Answer

In [22]:
from qa_prediction.build_qa_input import PromptBuilder

reasoning_prompter = PromptBuilder(
    os.path.join(REPO_DIR, "prompts", "llama2_predict.txt"),
    add_rule=True,
    maximun_token=4096 - 100,
    tokenize=lambda t: len(tokenizer.tokenize(t)),
)

by_question = {}
for rec in planning_records:
    by_question[rec["id"]] = rec

final_answers = {}
for qid, rec in tqdm(by_question.items(), desc="Reasoning"):
    q_dict = {
        "question": rec["question"],
        "graph": rec["graph"],
        "q_entity": rec["q_entity"],
        "predicted_paths": rec["predicted_paths"],
        "choices": [],
    }
    prompt = reasoning_prompter.process_input(q_dict)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
    gen_time = time.time() - t0
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    final_answers[qid] = {"final_RoG_answer": text, "reasoning_time_sec": gen_time}

for rec in retrieval_records:
    rec.update(final_answers.get(rec["question_id"], {}))

with open(json_path, "w") as f:
    json.dump(retrieval_records, f, indent=2)
pd.DataFrame(retrieval_records).to_csv(csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Reasoning:   0%|          | 0/50 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Verify the Decode Patch(full set)

In [23]:
default_decode_records = []
for sample in tqdm(subset, desc="Verify (default decode)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids, num_beams=N_BEAM, num_return_sequences=N_BEAM,
            early_stopping=False, do_sample=True, return_dict_in_generate=True,
            output_scores=True, max_new_tokens=100,
        )
    raw_sequences = output.sequences[:, input_ids.shape[1]:]
    default_text = [_original_decode(seq, skip_special_tokens=True).strip() for seq in raw_sequences]
    default_paths = parse_prediction(default_text)
    default_decode_records.append({"id": sample["id"], "predicted_paths": default_paths})

by_id_default = {r["id"]: r["predicted_paths"] for r in default_decode_records}
diffs = 0
examples_shown = 0
for rec in planning_records:
    patched = rec["predicted_paths"]
    default = by_id_default.get(rec["id"], [])
    if patched != default:
        diffs += 1
        if examples_shown < 5:
            print(f"\nQID {rec['id']}  --  {rec['question']}")
            print("  patched (sp_model.decode): ", patched)
            print("  default (tokenizer.decode):", default)
            examples_shown += 1

print(f"\n{diffs} / {len(planning_records)} questions differ between patched and default decode.")
if diffs == 0:
    print("No difference on the pilot -- the patch is a safe no-op here; keep it for robustness.")
else:
    print("Inspect the examples above: genuine fix, or a new deviation? Decide before scaling.")

Verify (default decode):   0%|          | 0/50 [00:00<?, ?it/s]


0 / 50 questions differ between patched and default decode.
No difference on the pilot -- the patch is a safe no-op here; keep it for robustness.


### Runtime Estimate

In [24]:
df_pilot = pd.DataFrame(retrieval_records)
mean_plan = df_pilot["planning_time_sec"].mean()
mean_retr = df_pilot["retrieval_time_sec"].mean()
n_full = 1628  # official WebQSP test size (RoG paper, Table 6)

est_plan_sec = mean_plan * n_full
est_retr_sec = mean_retr * n_full
est_total_hr = (est_plan_sec + est_retr_sec) / 3600

print(f"Mean planning time/question:  {mean_plan:.2f} s")
print(f"Mean retrieval time/question: {mean_retr:.3f} s")
print(f"Projected for {n_full} questions:")
print(f"  Planning:  ~{est_plan_sec/60:.1f} min")
print(f"  Retrieval: ~{est_retr_sec/60:.1f} min")
print(f"  Total:     ~{est_total_hr:.2f} GPU-hours (planning+retrieval only, not the optional final-answer pass)")
print()
print("Kaggle sessions have historically capped continuous runtime around 9 hours,")
print("with a weekly GPU quota that's fluctuated between ~30 and ~40 hours. Check")
print("your account's current quota (Settings) -- it changes over time. The")
print("checkpointed loops below handle a session cutoff gracefully: just re-run")
print("the cell and it resumes from the last completed question.")

Mean planning time/question:  1.99 s
Mean retrieval time/question: 0.000 s
Projected for 1628 questions:
  Planning:  ~54.0 min
  Retrieval: ~0.0 min
  Total:     ~0.90 GPU-hours (planning+retrieval only, not the optional final-answer pass)

Kaggle sessions have historically capped continuous runtime around 9 hours,
with a weekly GPU quota that's fluctuated between ~30 and ~40 hours. Check
your account's current quota (Settings) -- it changes over time. The
checkpointed loops below handle a session cutoff gracefully: just re-run
the cell and it resumes from the last completed question.


### Full-scale Config

In [25]:
FULL_OUTPUT_DIR = "/kaggle/working/step1_baseline_full"
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
# === RESTORE WebQSP checkpoints from last session ===
import shutil

if PREV_RUN_DIR:
    prev_dir = os.path.join(PREV_RUN_DIR, "step1_baseline_full")
    if os.path.exists(prev_dir):
        for fname in os.listdir(prev_dir):
            dst = os.path.join(FULL_OUTPUT_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(prev_dir, fname), dst)
                print("Restored:", fname)
    else:
        print("No prior WebQSP full-run folder in PREV_RUN_DIR.")
else:
    print("PREV_RUN_DIR not set — nothing to restore.")

PLANNING_CKPT = os.path.join(FULL_OUTPUT_DIR, "planning_webqsp_full.jsonl")
REASON_CKPT = os.path.join(FULL_OUTPUT_DIR, "reasoning_webqsp_full.jsonl")

full_test_all = load_dataset(DATASET_NAME, split=SPLIT)  # all 1,628 WebQSP test questions
print(f"Full test set: {len(full_test_all)} questions")

Full test set: 1628 questions


### Checkpointed planning (full set)

In [26]:
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done[rec["id"]] = rec
    return done

planning_done = load_checkpoint(PLANNING_CKPT)
print(f"Resuming: {len(planning_done)} questions already planned.")

with open(PLANNING_CKPT, "a") as fout:
    for sample in tqdm(full_test_all, desc="Planning (full)"):
        if sample["id"] in planning_done:
            continue
        input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
        t0 = time.time()
        raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=False, max_new_tokens=100)
        planning_time = time.time() - t0
        rel_paths = parse_prediction(raw_output["paths"])
        rec = {
            "id": sample["id"], "question": sample["question"],
            "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
            "graph": sample["graph"], "predicted_paths": rel_paths,
            "planning_time_sec": planning_time,
        }
        fout.write(json.dumps(rec) + "\n")
        fout.flush()
        planning_done[sample["id"]] = rec

planning_records_full = list(planning_done.values())
print(f"Planning complete: {len(planning_records_full)} / {len(full_test_all)} questions.")

Resuming: 1628 questions already planned.


Planning (full):   0%|          | 0/1628 [00:00<?, ?it/s]

Planning complete: 1628 / 1628 questions.


### Retrieval (Full set)

In [27]:
retrieval_records_full = []
for rec in tqdm(planning_records_full, desc="Retrieval (full)"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        retrieval_records_full.append({
            "question_id": rec["id"], "question": rec["question"],
            "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(retrieval_records_full)} retrieval records (full set) collected.")

Retrieval (full):   0%|          | 0/1628 [00:00<?, ?it/s]

4838 retrieval records (full set) collected.


### Question-Level Coverage

In [28]:
df_full = pd.DataFrame(retrieval_records_full)
question_coverage_full = df_full.groupby("question_id")["gold_reachable"].any()
print("Questions evaluated:", len(question_coverage_full))
print("Questions reaching >=1 gold answer:", question_coverage_full.sum())
print("Question-level retrieval coverage: %.2f%%" % (question_coverage_full.mean() * 100))

Questions evaluated: 1628
Questions reaching >=1 gold answer: 1369
Question-level retrieval coverage: 84.09%


### Aggregated Failure Analysis

In [29]:
from collections import Counter

reachable_by_q_full = defaultdict(bool)
for rec in retrieval_records_full:
    reachable_by_q_full[rec["question_id"]] |= rec["gold_reachable"]

all_qids_full = {rec["question_id"] for rec in retrieval_records_full}
missed_qids_full = [q for q in all_qids_full if not reachable_by_q_full[q]]
print(f"{len(missed_qids_full)} / {len(all_qids_full)} questions missed")

planning_by_id_full = {r["id"]: r for r in planning_records_full}
diagnosis_counts = Counter()
diagnosis_by_qid = {}
SAMPLE_PRINT = 5
printed = 0

for qid in tqdm(missed_qids_full, desc="Failure analysis"):
    rec = planning_by_id_full[qid]
    gold_paths = find_gold_paths_official_graph(rec, max_hops=2)  # WebQSP max hop = 2 (Table 6); bump to 4 for CWQ later
    predicted = {tuple(p) for p in rec["predicted_paths"]}
    gold_relation_paths = {tuple(p["relations"]) for p in gold_paths}

    if not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts[diagnosis] += 1
    diagnosis_by_qid[qid] = diagnosis

    if printed < SAMPLE_PRINT:
        print("\n" + "="*100)
        print("QUESTION:", rec["question"])
        print("DIAGNOSIS:", diagnosis)
        printed += 1

print("\n--- Failure breakdown ---")
for k, v in diagnosis_counts.most_common():
    print(f"{k}: {v}  ({100*v/max(len(missed_qids_full),1):.1f}% of missed questions)")

for rec in retrieval_records_full:
    rec["failure_diagnosis"] = diagnosis_by_qid.get(rec["question_id"])

259 / 1628 questions missed


Failure analysis:   0%|          | 0/259 [00:00<?, ?it/s]


QUESTION: when was the printing press invented by gutenberg
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: what sarah dessen books are movies
DIAGNOSIS: PLANNER FAILURE

QUESTION: when was lucy lawless born
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: who was selena gomez in barney and friends
DIAGNOSIS: PLANNER FAILURE

QUESTION: when did the burma cyclone happen
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

--- Failure breakdown ---
PLANNER FAILURE: 188  (72.6% of missed questions)
OFFICIAL GRAPH COVERAGE FAILURE: 71  (27.4% of missed questions)


### Full Scale Output

In [30]:
full_json_path = os.path.join(FULL_OUTPUT_DIR, "step1_baseline_webqsp_full.json")
full_csv_path = os.path.join(FULL_OUTPUT_DIR, "step1_baseline_webqsp_full.csv")

with open(full_json_path, "w") as f:
    json.dump(retrieval_records_full, f, indent=2)
pd.DataFrame(retrieval_records_full).to_csv(full_csv_path, index=False)
print("Saved:", full_json_path)
print("Saved:", full_csv_path)

Saved: /kaggle/working/step1_baseline_full/step1_baseline_webqsp_full.json
Saved: /kaggle/working/step1_baseline_full/step1_baseline_webqsp_full.csv


### Final RoG Answer (Full Planning)

In [31]:
reasoning_done = load_checkpoint(REASON_CKPT)
print(f"Resuming: {len(reasoning_done)} answers already generated.")

with open(REASON_CKPT, "a") as fout:
    for qid, rec in tqdm(planning_by_id_full.items(), desc="Reasoning (full)"):
        if qid in reasoning_done:
            continue
        q_dict = {
            "question": rec["question"], "graph": rec["graph"],
            "q_entity": rec["q_entity"], "predicted_paths": rec["predicted_paths"],
            "choices": [],
        }
        prompt = reasoning_prompter.process_input(q_dict)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
        gen_time = time.time() - t0
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        out_rec = {"id": qid, "final_RoG_answer": text, "reasoning_time_sec": gen_time}
        fout.write(json.dumps(out_rec) + "\n")
        fout.flush()
        reasoning_done[qid] = out_rec

for rec in retrieval_records_full:
    ans = reasoning_done.get(rec["question_id"])
    if ans:
        rec.update({"final_RoG_answer": ans["final_RoG_answer"], "reasoning_time_sec": ans["reasoning_time_sec"]})

with open(full_json_path, "w") as f:
    json.dump(retrieval_records_full, f, indent=2)
pd.DataFrame(retrieval_records_full).to_csv(full_csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Resuming: 1628 answers already generated.


Reasoning (full):   0%|          | 0/1628 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Confirm the full reasoning loop 

In [32]:
missing = [qid for qid in planning_by_id_full if qid not in reasoning_done]
print(f"{len(reasoning_done)} / {len(planning_by_id_full)} questions have a final answer.")
if missing:
    print(f"{len(missing)} still missing -- re-run the Cell 54 reasoning loop (it resumes automatically).")
else:
    print("Reasoning complete. Safe to proceed.")

1628 / 1628 questions have a final answer.
Reasoning complete. Safe to proceed.


### Memory Cache saved for reasoning

In [33]:
# === MEMORY: drop cached subgraphs now that WebQSP reasoning is done ===
import gc
for rec in planning_records_full:
    rec.pop("graph", None)
gc.collect()
print("Dropped cached subgraphs from planning_records_full to free RAM.")

Dropped cached subgraphs from planning_records_full to free RAM.


### Clean per-question table

In [34]:
from collections import defaultdict
import pandas as pd

retrieval_time_by_qid = defaultdict(float)
gold_reachable_by_qid = defaultdict(bool)
for rec in retrieval_records_full:
    retrieval_time_by_qid[rec["question_id"]] += rec["retrieval_time_sec"]
    gold_reachable_by_qid[rec["question_id"]] |= rec["gold_reachable"]

rows = []
for qid, prec in planning_by_id_full.items():
    rrow = reasoning_done.get(qid, {})
    rows.append({
        "id": qid,
        "question": prec["question"],
        "gold_answers": prec["a_entity"],
        "predicted_paths": prec["predicted_paths"],
        "planning_time_sec": prec["planning_time_sec"],
        "retrieval_time_sec": retrieval_time_by_qid.get(qid, 0.0),
        "reasoning_time_sec": rrow.get("reasoning_time_sec"),
        "final_RoG_answer": rrow.get("final_RoG_answer"),
        "gold_reachable": gold_reachable_by_qid.get(qid, False),
    })

per_question = pd.DataFrame(rows)
n = len(per_question)
assert per_question["reasoning_time_sec"].notna().all(), "Some questions missing reasoning -- finish Cell 54 first."
print(f"Per-question baseline table: {n} questions.")

Per-question baseline table: 1628 questions.


### Full per-question table

In [35]:
pd.set_option("display.max_colwidth", 100)  

display_cols = [
    "id", "question", "gold_answers", "predicted_paths",
    "planning_time_sec", "retrieval_time_sec", "reasoning_time_sec",
    "final_RoG_answer", "gold_reachable"
]

per_question_display = per_question[display_cols].rename(columns={"id": "question_id"})
per_question_display

,question_id,question,gold_answers,predicted_paths,planning_time_sec,retrieval_time_sec,reasoning_time_sec,final_RoG_answer,gold_reachable
0,WebQTest-0,what does jamaican people speak,"[Jamaican English, Jamaican Creole English Language]","[[location.country.languages_spoken], [language.human_language.countries_spoken_in], [location.c...",1.324136,0.003920,0.893439,Jamaican English\nJamaican Creole English Language,True
1,WebQTest-1,what did james k polk do before he was president,"[United States Representative, Governor of Tennessee, Speaker of the United States House of Repr...","[[government.government_position_held.office_holder, government.government_position_held.office_...",2.938305,0.000676,0.528926,United States Representative,True
2,WebQTest-3,who plays ken barlow in coronation street,[William Roache],"[[tv.tv_program.country_of_origin, people.person.nationality], [tv.regular_tv_appearance.series,...",1.790577,0.000319,0.472994,David Hanson,False
3,WebQTest-6,where is jamarcus russell from,[Mobile],"[[location.location.people_born_here], [people.person.place_of_birth], [people.person.nationality]]",1.032935,0.000264,0.461035,United States of America\nMobile,True
4,WebQTest-7,where was george washington carver from,[Diamond],"[[people.person.place_of_birth], [location.location.people_born_here], [people.person.nationality]]",1.034217,0.000151,0.583327,United States of America\nDiamond,True
...,...,...,...,...,...,...,...,...,...
1623,WebQTest-2027,what team did david beckham play for before la galaxy,[Manchester United F.C.],"[[sports.pro_athlete.teams, sports.sports_team_roster.team], [soccer.football_player.statistics,...",1.927255,0.000418,1.879107,Paris Saint-Germain F.C.\nA.C. Milan\nManchester United F.C.,True
1624,WebQTest-2028,who is the current leader of france 2010,[Nicolas Sarkozy],"[[people.person.nationality], [base.onephylogeny.type_of_thing.things_of_this_type, people.perso...",2.396295,0.003479,0.968816,Nicolas Sarkozy\nFrançois Hollande,True
1625,WebQTest-2029,where was the palace of knossos located,"[Crete, Greece]","[[location.location.containedby], [architecture.building.building_complex], [architecture.buildi...",0.821460,0.000087,0.154127,Greece,True
1626,WebQTest-2030,where is roswell area 51,"[Lincoln County, Nevada]","[[aviation.airport.serves], [location.location.containedby], [location.location.contains]]",0.892296,0.000057,0.214914,Lincoln County,True


### Saved as CSV and JSON

In [36]:
csv_out = os.path.join(FULL_OUTPUT_DIR, "webqsp_per_question_baseline.csv")
json_out = os.path.join(FULL_OUTPUT_DIR, "webqsp_per_question_baseline.json")

per_question_display.to_csv(csv_out, index=False)
per_question_display.to_json(json_out, orient="records", indent=2)

print("Saved:", csv_out)
print("Saved:", json_out)

Saved: /kaggle/working/step1_baseline_full/webqsp_per_question_baseline.csv
Saved: /kaggle/working/step1_baseline_full/webqsp_per_question_baseline.json


### Timing Statistics

In [37]:
def total_avg(col):
    total = per_question[col].sum()
    return total, total / n

plan_total, plan_avg = total_avg("planning_time_sec")
retr_total, retr_avg = total_avg("retrieval_time_sec")
reas_total, reas_avg = total_avg("reasoning_time_sec")
e2e_total = plan_total + retr_total + reas_total
e2e_avg = e2e_total / n

print(f"Total planning time:   {plan_total:9.1f} s  ({plan_total/60:7.2f} min)   avg/question: {plan_avg:.3f} s")
print(f"Total retrieval time:  {retr_total:9.1f} s  ({retr_total/60:7.2f} min)   avg/question: {retr_avg:.4f} s")
print(f"Total reasoning time:  {reas_total:9.1f} s  ({reas_total/60:7.2f} min)   avg/question: {reas_avg:.3f} s")
print(f"Total end-to-end time: {e2e_total:9.1f} s  ({e2e_total/60:7.2f} min)   avg/question: {e2e_avg:.3f} s")

Total planning time:      2634.3 s  (  43.91 min)   avg/question: 1.618 s
Total retrieval time:        1.2 s  (   0.02 min)   avg/question: 0.0008 s
Total reasoning time:     5148.2 s  (  85.80 min)   avg/question: 3.162 s
Total end-to-end time:    7783.7 s  ( 129.73 min)   avg/question: 4.781 s


### Retrieval Coverage and official Hit/Hits@1/F1

In [38]:
import sys
sys.path.append(os.path.join(REPO_DIR, "src"))
from qa_prediction.evaluate_results import eval_acc, eval_hit, eval_f1

n_covered = int(per_question["gold_reachable"].sum())
coverage_pct = 100 * n_covered / n

hit_list, acc_list, f1_list, prec_list, rec_list = [], [], [], [], []
for _, row in per_question.iterrows():
    prediction = [p for p in row["final_RoG_answer"].split("\n") if p.strip()]
    prediction_str = " ".join(prediction)
    f1, precision, recall = eval_f1(prediction, row["gold_answers"])
    hit_list.append(eval_hit(prediction_str, row["gold_answers"]))
    acc_list.append(eval_acc(prediction_str, row["gold_answers"]))
    f1_list.append(f1); prec_list.append(precision); rec_list.append(recall)

hits1_pct = 100 * sum(hit_list) / n
f1_pct = 100 * sum(f1_list) / n

print(f"Questions reaching >=1 gold: {n_covered}")
print(f"Question-level retrieval coverage: {coverage_pct:.2f}%")
print(f"Hit/Hits@1: {hits1_pct:.2f}%")
print(f"F1: {f1_pct:.2f}%")

Questions reaching >=1 gold: 1369
Question-level retrieval coverage: 84.09%
Hit/Hits@1: 85.93%
F1: 70.27%


### Freezing Baseline

In [39]:
baseline_summary = f"""Dataset: WebQSP
Test questions: {n}

Retrieval:
  Questions reaching >=1 gold: {n_covered}
  Question-level retrieval coverage: {coverage_pct:.2f}%

Timing:
  Total planning time:   {plan_total:.1f} s ({plan_total/60:.2f} min)   Avg/question: {plan_avg:.3f} s
  Total retrieval time:  {retr_total:.1f} s ({retr_total/60:.2f} min)   Avg/question: {retr_avg:.4f} s
  Total reasoning time:  {reas_total:.1f} s ({reas_total/60:.2f} min)   Avg/question: {reas_avg:.3f} s
  Total end-to-end time: {e2e_total:.1f} s ({e2e_total/60:.2f} min)   Avg/question: {e2e_avg:.3f} s

Final QA:
  Hit/Hits@1: {hits1_pct:.2f}%
  F1: {f1_pct:.2f}%
"""
print(baseline_summary)

with open(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_summary.txt"), "w") as f:
    f.write(baseline_summary)

per_question.to_json(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_predictions.json"), orient="records", indent=2)
per_question.to_csv(os.path.join(FULL_OUTPUT_DIR, "webqsp_baseline_predictions.csv"), index=False)
print("Baseline frozen.")

Dataset: WebQSP
Test questions: 1628

Retrieval:
  Questions reaching >=1 gold: 1369
  Question-level retrieval coverage: 84.09%

Timing:
  Total planning time:   2634.3 s (43.91 min)   Avg/question: 1.618 s
  Total retrieval time:  1.2 s (0.02 min)   Avg/question: 0.0008 s
  Total reasoning time:  5148.2 s (85.80 min)   Avg/question: 3.162 s
  Total end-to-end time: 7783.7 s (129.73 min)   Avg/question: 4.781 s

Final QA:
  Hit/Hits@1: 85.93%
  F1: 70.27%

Baseline frozen.


### Memory clear of RoG-WebQSP

In [40]:
# === MEMORY: clear WebQSP intermediates before starting CWQ ===
import torch
for _name in ["retrieval_records_full", "retrieval_records", "planning_records",
              "df_full", "df_pilot", "default_decode_records"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
print("GPU memory reserved:", f"{torch.cuda.memory_reserved()/1e9:.2f} GB")

GPU memory reserved: 6.92 GB


### LeBron James Triple Retrieval

In [41]:
import json
from collections import defaultdict
from utils.graph_utils import build_graph

target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")


target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")

=== Outgoing relations from 'LeBron James' (64 distinct relations) ===
  [ 26x] award.award_winner.awards_won  ->  ['m.0_qrm17', 'm.0yg0zky', 'm.0z66wqh', 'm.0_qrd1p', 'm.0_qrgh9'] ...
  [ 23x] award.award_nominee.award_nominations  ->  ['m.0sgkpd0', 'm.0z1p_l5', 'm.0z9ljvh', 'm.0z5blfv', 'm.0y_yc47'] ...
  [ 20x] award.award_nomination.award_nominee  ->  ['m.0z1n6dn', 'm.0_spmjw', 'm.010w6vqp', 'm.0z43z7_', 'm.0z3v44c'] ...
  [ 16x] award.award_honor.award_winner  ->  ['m.0_qvzz1', 'm.0_qr864', 'm.0x0zlfg', 'm.0_qrlpv', 'm.0x0z10c'] ...
  [ 13x] film.personal_film_appearance.person  ->  ['m.0v4mj0y', 'm.0v4ndsn', 'm.0v4mlsf', 'm.0v462vn', 'm.0v4mk_2'] ...
  [  9x] tv.tv_guest_role.actor  ->  ['m.0y7ls9m', 'm.0y7htk3', 'm.0kb00b3', 'm.0y7j8ry', 'm.0y7klbl'] ...
  [  9x] freebase.valuenotation.is_reviewed  ->  ['Parents', 'Date of birth', 'Children', 'Place of birth', 'Weight'] ...
  [  8x] common.topic.webpage  ->  ['m.0kg5v20', 'm.0bnsx2n', 'm.09wlf17', 'm.09ymk3w', 'm.09x22v7'] ...
 

In [42]:
import json
from collections import defaultdict
from utils.graph_utils import build_graph

target_qid = "WebQTest-268"
rec = None
with open(PLANNING_CKPT) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] == target_qid:
            rec = r
            break

G = build_graph(rec["graph"])
topic = rec["q_entity"][0]  # "LeBron James"

# Group every real 1-hop edge (in either direction) touching the topic entity,
# by relation type -- this is your pool of "Batman / Alfred / Chris Nolan"-style
# context facts for a richer illustrative figure.
outgoing = defaultdict(list)
for nbr in G.neighbors(topic):
    rel = G[topic][nbr]["relation"]
    outgoing[rel].append(nbr)

incoming = defaultdict(list)
for src in G.nodes():
    if src == topic:
        continue
    if G.has_edge(src, topic):
        rel = G[src][topic]["relation"]
        incoming[rel].append(src)

print(f"=== Outgoing relations from '{topic}' ({len(outgoing)} distinct relations) ===")
for rel, targets in sorted(outgoing.items(), key=lambda kv: -len(kv[1])):
    sample = targets[:5]
    print(f"  [{len(targets):>3}x] {rel}  ->  {sample}{' ...' if len(targets) > 5 else ''}")

print(f"\n=== Incoming relations to '{topic}' ({len(incoming)} distinct relations) ===")
for rel, sources in sorted(incoming.items(), key=lambda kv: -len(kv[1])):
    sample = sources[:5]
    print(f"  [{len(sources):>3}x] {rel}  <-  {sample}{' ...' if len(sources) > 5 else ''}")

=== Outgoing relations from 'LeBron James' (64 distinct relations) ===
  [ 26x] award.award_winner.awards_won  ->  ['m.0_qrm17', 'm.0yg0zky', 'm.0z66wqh', 'm.0_qrd1p', 'm.0_qrgh9'] ...
  [ 23x] award.award_nominee.award_nominations  ->  ['m.0sgkpd0', 'm.0z1p_l5', 'm.0z9ljvh', 'm.0z5blfv', 'm.0y_yc47'] ...
  [ 20x] award.award_nomination.award_nominee  ->  ['m.0z1n6dn', 'm.0_spmjw', 'm.010w6vqp', 'm.0z43z7_', 'm.0z3v44c'] ...
  [ 16x] award.award_honor.award_winner  ->  ['m.0_qvzz1', 'm.0_qr864', 'm.0x0zlfg', 'm.0_qrlpv', 'm.0x0z10c'] ...
  [ 13x] film.personal_film_appearance.person  ->  ['m.0v4mj0y', 'm.0v4ndsn', 'm.0v4mlsf', 'm.0v462vn', 'm.0v4mk_2'] ...
  [  9x] tv.tv_guest_role.actor  ->  ['m.0y7ls9m', 'm.0y7htk3', 'm.0kb00b3', 'm.0y7j8ry', 'm.0y7klbl'] ...
  [  9x] freebase.valuenotation.is_reviewed  ->  ['Parents', 'Date of birth', 'Children', 'Place of birth', 'Weight'] ...
  [  8x] common.topic.webpage  ->  ['m.0kg5v20', 'm.0bnsx2n', 'm.09wlf17', 'm.09ymk3w', 'm.09x22v7'] ...
 

## RoG CWQ Baseline

### Config and Load

In [43]:
from datasets import load_dataset
CWQ_DATASET_NAME = "rmanluo/RoG-cwq"
CWQ_SPLIT = "test"
CWQ_N_PILOT = 50
CWQ_MAX_HOPS = 4   # As CWQ goes up to 4-hop 
CWQ_PILOT_DIR = "/kaggle/working/step1_cwq_pilot"
os.makedirs(CWQ_PILOT_DIR, exist_ok=True)

cwq_full_test = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
cwq_pilot = cwq_full_test.select(range(min(CWQ_N_PILOT, len(cwq_full_test))))
print(f"CWQ test set: {len(cwq_full_test)} questions (paper Table 6: 3,531)")
print(f"Pilot subset: {len(cwq_pilot)} questions")
cwq_pilot[0]

README.md:   0%|          | 0.00/913 [00:00<?, ?B/s]

data/train-00000-of-00018-e65d08d5970d44(…):   0%|          | 0.00/130M [00:00<?, ?B/s]

data/train-00001-of-00018-c70342c196c07d(…):   0%|          | 0.00/132M [00:00<?, ?B/s]

data/train-00002-of-00018-d52ad886cb9f05(…):   0%|          | 0.00/128M [00:00<?, ?B/s]

data/train-00003-of-00018-6dac2fb592f087(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

data/test-00000-of-00003-e62a559c5d2b56c(…):   0%|          | 0.00/114M [00:00<?, ?B/s]

data/test-00001-of-00003-2fa9a898639e7d1(…):   0%|          | 0.00/128M [00:00<?, ?B/s]

data/test-00002-of-00003-c659cd388440c4a(…):   0%|          | 0.00/131M [00:00<?, ?B/s]

CWQ test set: 3531 questions (paper Table 6: 3,531)
Pilot subset: 50 questions


{'id': 'WebQTest-832_c334509bb5e02cacae1ba2e80c176499',
 'question': 'Lou Seal is the mascot for the team that last won the World Series when?',
 'answer': ['2014 World Series'],
 'q_entity': ['Lou Seal'],
 'a_entity': ['2014 World Series'],
 'graph': [['Mascot', 'type.type.expected_by', 'sports_team_mascot'],
  ['San Francisco Giants', 'baseball.baseball_team.team_stats', 'm.05n69q3'],
  ['San Francisco Giants', 'baseball.baseball_team.team_stats', 'm.05n6btw'],
  ['San Francisco Giants',
   'freebase.valuenotation.is_reviewed',
   'Contact webpages'],
  ['San Francisco Giants',
   'base.schemastaging.sports_team_extra.training_ground',
   'm.0k079rd'],
  ['Polo Grounds', 'sports.sports_facility.teams', 'San Francisco Giants'],
  ['m.04vy2qp',
   'sports.sports_league_draft_pick.team',
   'San Francisco Giants'],
  ['San Francisco Giants',
   'sports.sports_team.arena_stadium',
   'Seals Stadium'],
  ['m.05n6dtn', 'baseball.baseball_team_stats.team', 'San Francisco Giants'],
  ['San F

### Pilot Planning

In [44]:
cwq_pilot_planning = []
for sample in tqdm(cwq_pilot, desc="CWQ Planning (pilot)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    t0 = time.time()
    raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
    planning_time = time.time() - t0
    rel_paths = parse_prediction(raw_output["paths"])
    cwq_pilot_planning.append({
        "id": sample["id"], "question": sample["question"],
        "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
        "graph": sample["graph"], "predicted_paths": rel_paths,
        "planning_time_sec": planning_time,
    })

path_lengths = sorted({len(p) for rec in cwq_pilot_planning for p in rec["predicted_paths"]})
print(f"Planned {len(cwq_pilot_planning)} CWQ questions. Relation-path lengths seen: {path_lengths}")

CWQ Planning (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

Planned 50 CWQ questions. Relation-path lengths seen: [1, 2, 3, 4]


### BFS Sanity Check (Pilot)

In [45]:
mismatches = checked = 0
for rec in cwq_pilot_planning:
    graph = build_graph(rec["graph"])
    for entity in rec["q_entity"]:
        for rule in rec["predicted_paths"]:
            checked += 1
            if bfs_with_rule(graph, entity, rule) != bfs_with_rule_instrumented(graph, entity, rule)[0]:
                mismatches += 1

print(f"Checked {checked} calls. Mismatches: {mismatches}")
assert mismatches == 0, "Instrumented BFS diverges on CWQ -- STOP, do not trust downstream numbers."

Checked 258 calls. Mismatches: 0


### Retrieval (Pilot)

In [46]:
def run_retrieval(planning_recs, desc):
    out = []
    for rec in tqdm(planning_recs, desc=desc):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
        for rule in plans:
            t0 = time.time()
            all_paths, hop_frontier_total = [], [0] * len(rule)
            if len(rule) > 0:
                for entity in rec["q_entity"]:
                    paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                    all_paths.extend(paths)
                    for h in range(len(rule)):
                        hop_frontier_total[h] += hop_frontier[h]
            retrieval_time = time.time() - t0
            reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
            out.append({
                "question_id": rec["id"], "question": rec["question"],
                "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
                "hop_frontiers": hop_frontier_total,
                "retrieved_paths_count": len(all_paths),
                "gold_answers": sorted(gold_answers),
                "reachable_gold_answers": reachable_gold,
                "gold_reachable": len(reachable_gold) > 0,
                "retrieval_time_sec": retrieval_time,
                "planning_time_sec": rec["planning_time_sec"],
            })
    return out

cwq_pilot_retrieval = run_retrieval(cwq_pilot_planning, "CWQ Retrieval (pilot)")
df_cwq_pilot = pd.DataFrame(cwq_pilot_retrieval)
coverage = df_cwq_pilot.groupby("question_id")["gold_reachable"].any()
print(f"CWQ pilot coverage: {coverage.sum()} / {len(coverage)} = {coverage.mean()*100:.2f}%")

CWQ Retrieval (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

CWQ pilot coverage: 36 / 50 = 72.00%


### Computing missed_qids_cwq

In [47]:
from collections import defaultdict

reachable_by_question_cwq = defaultdict(bool)
for rec in cwq_pilot_retrieval:
    reachable_by_question_cwq[rec["question_id"]] |= rec["gold_reachable"]

all_qids_cwq = {rec["question_id"] for rec in cwq_pilot_retrieval}
missed_qids_cwq = [qid for qid in all_qids_cwq if not reachable_by_question_cwq[qid]]

print(f"{len(missed_qids_cwq)} / {len(all_qids_cwq)} CWQ pilot questions missed")

14 / 50 CWQ pilot questions missed


### Revised Failure Analysis

In [48]:
from collections import deque, Counter

def find_gold_paths_official_graph(rec, max_hops=4, max_expansions=200000):
    G = build_graph(rec["graph"])
    gold = set(rec["a_entity"])
    found = []
    expansions = 0
    capped = False

    for start in rec["q_entity"]:
        queue = deque([(start, [], [start])])
        while queue:
            node, relations, entities = queue.popleft()

            if len(relations) > 0 and node in gold:
                found.append({
                    "relations": relations,
                    "entities": entities
                })

            if len(relations) == max_hops:
                continue
            if node not in G:
                continue

            for neighbor in G.neighbors(node):
                expansions += 1
                if expansions > max_expansions:
                    capped = True
                    return found, capped
                rel = G[node][neighbor]["relation"]
                queue.append((
                    neighbor,
                    relations + [rel],
                    entities + [neighbor]
                ))

    return found, capped


diagnosis_counts = Counter()

for qid in missed_qids_cwq:
    rec = next(r for r in cwq_pilot_planning if r["id"] == qid)   # <-- fixed: pilot planning list

    gold_paths, capped = find_gold_paths_official_graph(rec, max_hops=CWQ_MAX_HOPS)

    predicted = {
        tuple(p) for p in rec["predicted_paths"]
    }
    gold_relation_paths = {
        tuple(p["relations"]) for p in gold_paths
    }

    print("\n" + "="*100)
    print("QUESTION:", rec["question"])
    print("GOLD:", rec["a_entity"])

    print("\nPredicted:")
    for p in predicted:
        print(" ", " -> ".join(p))

    print("\nGold-supporting paths in OFFICIAL BUILT GRAPH:")
    for p in gold_paths[:10]:
        print(
            " ",
            " -> ".join(p["entities"]),
            "\n    ",
            " -> ".join(p["relations"])
        )

    if capped:
        diagnosis = "UNVERIFIED (hit expansion cap)"
    elif not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts[diagnosis] += 1
    print("\nDIAGNOSIS:", diagnosis)

print("\n" + "="*100)
print("FAILURE BREAKDOWN:", diagnosis_counts)


QUESTION: Who holds the position of Prime Minister in the country which contains Dire Dawa?
GOLD: ['Hailemariam Desalegn']

Predicted:
  government.government_position_held.basic_title -> government.government_position_held.office_holder
  location.location.containedby -> people.person.nationality
  government.government_position_held.basic_title -> government.politician.government_positions_held

Gold-supporting paths in OFFICIAL BUILT GRAPH:

DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

QUESTION: Which location in the Anadyr Timezone has the biggest population?
GOLD: ['India']

Predicted:
  time.time_zone.locations_in_this_time_zone
  location.location.time_zones
  location.location.containedby

Gold-supporting paths in OFFICIAL BUILT GRAPH:
  Anadyr Time Zone -> Asia -> India 
     location.location.time_zones -> location.location.containedby
  Anadyr Time Zone -> Day DST ends -> India Time Zone -> India 
     freebase.valuenotation.has_no_value -> freebase.valuenotation.has_no_valu

## CSV/JSON Save

In [49]:
import pandas as pd

json_path = os.path.join(CWQ_PILOT_DIR, "step1_baseline_cwq_pilot.json")
csv_path = os.path.join(CWQ_PILOT_DIR, "step1_baseline_cwq_pilot.csv")

with open(json_path, "w") as f:
    json.dump(cwq_pilot_retrieval, f, indent=2)

df_cwq_pilot = pd.DataFrame(cwq_pilot_retrieval)
df_cwq_pilot.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)
df_cwq_pilot[["retrieved_paths_count", "gold_reachable", "retrieval_time_sec"]].describe()

Saved: /kaggle/working/step1_cwq_pilot/step1_baseline_cwq_pilot.json
Saved: /kaggle/working/step1_cwq_pilot/step1_baseline_cwq_pilot.csv


,retrieved_paths_count,retrieval_time_sec
count,150.000000,150.000000
mean,13.360000,0.000265
std,56.074289,0.000360
min,0.000000,0.000004
25%,0.000000,0.000031
50%,1.000000,0.000103
75%,7.000000,0.000296
max,630.000000,0.002367


### Worked Example

In [50]:
example = next(r for r in cwq_pilot_retrieval if len(r["predicted_relation_path"]) > 0)

print("Question:")
print(example["question"])
print()
print("Topic entity:")
print(example["topic_entity"])
print()
print("RoG relation plan:")
print(" -> ".join(example["predicted_relation_path"]))
print()
for i, count in enumerate(example["hop_frontiers"], start=1):
    print(f"Hop {i}:")
    print(f"{count} candidate entities")
    print()
print("Retrieved paths:")
print(example["retrieved_paths_count"])
print()
print("Gold answer reachable:")
print("Yes" if example["gold_reachable"] else "No")
print()
print("Retrieval time:")
print(f"{example['retrieval_time_sec']*1000:.1f} ms")

Question:
Lou Seal is the mascot for the team that last won the World Series when?

Topic entity:
['Lou Seal']

RoG relation plan:
sports.mascot.team -> sports.sports_championship_event.champion

Hop 1:
1 candidate entities

Hop 2:
1 candidate entities

Retrieved paths:
1

Gold answer reachable:
Yes

Retrieval time:
0.2 ms


### Final RoG Answer

In [51]:
from qa_prediction.build_qa_input import PromptBuilder

reasoning_prompter = PromptBuilder(
    os.path.join(REPO_DIR, "prompts", "llama2_predict.txt"),
    add_rule=True,
    maximun_token=4096 - 100,
    tokenize=lambda t: len(tokenizer.tokenize(t)),
)

by_question_cwq = {}
for rec in cwq_pilot_planning:
    by_question_cwq[rec["id"]] = rec

final_answers_cwq = {}
for qid, rec in tqdm(by_question_cwq.items(), desc="CWQ Reasoning (pilot)"):
    q_dict = {
        "question": rec["question"],
        "graph": rec["graph"],
        "q_entity": rec["q_entity"],
        "predicted_paths": rec["predicted_paths"],
        "choices": [],
    }
    prompt = reasoning_prompter.process_input(q_dict)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
    gen_time = time.time() - t0
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    final_answers_cwq[qid] = {"final_RoG_answer": text, "reasoning_time_sec": gen_time}

for rec in cwq_pilot_retrieval:
    rec.update(final_answers_cwq.get(rec["question_id"], {}))

with open(json_path, "w") as f:
    json.dump(cwq_pilot_retrieval, f, indent=2)
pd.DataFrame(cwq_pilot_retrieval).to_csv(csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

CWQ Reasoning (pilot):   0%|          | 0/50 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Verify the Decode Patch (full set)

In [52]:
cwq_default_decode_records = []
for sample in tqdm(cwq_pilot, desc="CWQ Verify (default decode)"):
    input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids, num_beams=N_BEAM, num_return_sequences=N_BEAM,
            early_stopping=False, do_sample=True, return_dict_in_generate=True,
            output_scores=True, max_new_tokens=100,
        )
    raw_sequences = output.sequences[:, input_ids.shape[1]:]
    default_text = [_original_decode(seq, skip_special_tokens=True).strip() for seq in raw_sequences]
    default_paths = parse_prediction(default_text)
    cwq_default_decode_records.append({"id": sample["id"], "predicted_paths": default_paths})

by_id_default_cwq = {r["id"]: r["predicted_paths"] for r in cwq_default_decode_records}
diffs = 0
examples_shown = 0
for rec in cwq_pilot_planning:
    patched = rec["predicted_paths"]
    default = by_id_default_cwq.get(rec["id"], [])
    if patched != default:
        diffs += 1
        if examples_shown < 5:
            print(f"\nQID {rec['id']}  --  {rec['question']}")
            print("  patched (sp_model.decode): ", patched)
            print("  default (tokenizer.decode):", default)
            examples_shown += 1

print(f"\n{diffs} / {len(cwq_pilot_planning)} questions differ between patched and default decode.")

CWQ Verify (default decode):   0%|          | 0/50 [00:00<?, ?it/s]


0 / 50 questions differ between patched and default decode.


### Runtime Estimate

In [53]:
mean_plan_cwq = df_cwq_pilot["planning_time_sec"].mean()
mean_retr_cwq = df_cwq_pilot["retrieval_time_sec"].mean()
n_cwq_full = 3531  # official CWQ test size (RoG paper, Table 6)

est_plan_sec = mean_plan_cwq * n_cwq_full
est_retr_sec = mean_retr_cwq * n_cwq_full
est_total_hr = (est_plan_sec + est_retr_sec) / 3600

print(f"Mean planning time/question:  {mean_plan_cwq:.2f} s")
print(f"Mean retrieval time/question: {mean_retr_cwq:.3f} s")
print(f"Projected for {n_cwq_full} questions:")
print(f"  Planning:  ~{est_plan_sec/60:.1f} min")
print(f"  Retrieval: ~{est_retr_sec/60:.1f} min")
print(f"  Total:     ~{est_total_hr:.2f} GPU-hours (planning+retrieval only, not the final-answer pass)")

Mean planning time/question:  2.06 s
Mean retrieval time/question: 0.000 s
Projected for 3531 questions:
  Planning:  ~121.0 min
  Retrieval: ~0.0 min
  Total:     ~2.02 GPU-hours (planning+retrieval only, not the final-answer pass)


### Full-Scale Config

In [54]:
CWQ_FULL_DIR = "/kaggle/working/step1_cwq_full"
os.makedirs(CWQ_FULL_DIR, exist_ok=True)
# === RESTORE CWQ checkpoints from last session ===
if PREV_RUN_DIR:
    prev_dir = os.path.join(PREV_RUN_DIR, "step1_cwq_full")
    if os.path.exists(prev_dir):
        for fname in os.listdir(prev_dir):
            dst = os.path.join(CWQ_FULL_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(prev_dir, fname), dst)
                print("Restored:", fname)
    else:
        print("No prior CWQ full-run folder in PREV_RUN_DIR.")
else:
    print("PREV_RUN_DIR not set — nothing to restore.")
CWQ_PLANNING_CKPT = os.path.join(CWQ_FULL_DIR, "planning_cwq_full.jsonl")
CWQ_REASON_CKPT = os.path.join(CWQ_FULL_DIR, "reasoning_cwq_full.jsonl")

SEED = 42
torch.manual_seed(SEED)

cwq_full_test_all = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
print(f"Full CWQ test set: {len(cwq_full_test_all)} questions")

Full CWQ test set: 3531 questions


### Checkpointed Planning (full set)

In [55]:
cwq_planning_done = load_checkpoint(CWQ_PLANNING_CKPT)
print(f"Resuming: {len(cwq_planning_done)} CWQ questions already planned.")

with open(CWQ_PLANNING_CKPT, "a") as fout:
    for sample in tqdm(cwq_full_test_all, desc="Planning (CWQ full)"):
        if sample["id"] in cwq_planning_done:
            continue
        input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
        t0 = time.time()
        raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
        planning_time = time.time() - t0
        rel_paths = parse_prediction(raw_output["paths"])
        rec = {
            "id": sample["id"], "question": sample["question"],
            "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
            "graph": sample["graph"], "predicted_paths": rel_paths,
            "planning_time_sec": planning_time,
        }
        fout.write(json.dumps(rec) + "\n")
        fout.flush()
        cwq_planning_done[sample["id"]] = rec

cwq_planning_records_full = list(cwq_planning_done.values())
print(f"Planning complete: {len(cwq_planning_records_full)} / {len(cwq_full_test_all)} questions.")

Resuming: 3531 CWQ questions already planned.


Planning (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

Planning complete: 3531 / 3531 questions.


### Retrieval (full set)

In [56]:
from tqdm.auto import tqdm
cwq_retrieval_records_full = []
for rec in tqdm(cwq_planning_records_full, desc="Retrieval (CWQ full)"):
    graph = build_graph(rec["graph"])
    gold_answers = set(rec["a_entity"])
    plans = rec["predicted_paths"] if len(rec["predicted_paths"]) > 0 else [[]]
    for rule in plans:
        t0 = time.time()
        all_paths = []
        hop_frontier_total = [0] * len(rule)
        if len(rule) > 0:
            for entity in rec["q_entity"]:
                paths, hop_frontier = bfs_with_rule_instrumented(graph, entity, rule)
                all_paths.extend(paths)
                for h in range(len(rule)):
                    hop_frontier_total[h] += hop_frontier[h]
        retrieval_time = time.time() - t0
        reachable_gold = sorted(path_endpoints(all_paths) & gold_answers)
        cwq_retrieval_records_full.append({
            "question_id": rec["id"], "question": rec["question"],
            "topic_entity": rec["q_entity"], "predicted_relation_path": rule,
            "hop_frontiers": hop_frontier_total,
            "retrieved_paths_count": len(all_paths),
            "gold_answers": sorted(gold_answers),
            "reachable_gold_answers": reachable_gold,
            "gold_reachable": len(reachable_gold) > 0,
            "retrieval_time_sec": retrieval_time,
            "planning_time_sec": rec["planning_time_sec"],
        })

print(f"{len(cwq_retrieval_records_full)} retrieval records (CWQ full) collected.")

Retrieval (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

10566 retrieval records (CWQ full) collected.


### Question-Level Coverage

In [57]:
df_cwq_full = pd.DataFrame(cwq_retrieval_records_full)
question_coverage_cwq_full = df_cwq_full.groupby("question_id")["gold_reachable"].any()
print("Questions evaluated:", len(question_coverage_cwq_full))
print("Questions reaching >=1 gold answer:", question_coverage_cwq_full.sum())
print("Question-level retrieval coverage: %.2f%%" % (question_coverage_cwq_full.mean() * 100))

Questions evaluated: 3531
Questions reaching >=1 gold answer: 2422
Question-level retrieval coverage: 68.59%


### Aggregated Failure Analysis

In [58]:
from collections import Counter

reachable_by_q_cwq_full = defaultdict(bool)
for rec in cwq_retrieval_records_full:
    reachable_by_q_cwq_full[rec["question_id"]] |= rec["gold_reachable"]

all_qids_cwq_full = {rec["question_id"] for rec in cwq_retrieval_records_full}
missed_qids_cwq_full = [q for q in all_qids_cwq_full if not reachable_by_q_cwq_full[q]]
print(f"{len(missed_qids_cwq_full)} / {len(all_qids_cwq_full)} questions missed")

cwq_planning_by_id_full = {r["id"]: r for r in cwq_planning_records_full}
diagnosis_counts_cwq = Counter()
diagnosis_by_qid_cwq = {}
SAMPLE_PRINT = 5
printed = 0

for qid in tqdm(missed_qids_cwq_full, desc="Failure analysis (CWQ full)"):
    rec = cwq_planning_by_id_full[qid]
    gold_paths, capped = find_gold_paths_official_graph(rec, max_hops=CWQ_MAX_HOPS)
    predicted = {tuple(p) for p in rec["predicted_paths"]}
    gold_relation_paths = {tuple(p["relations"]) for p in gold_paths}

    if capped:
        diagnosis = "UNVERIFIED (hit expansion cap)"
    elif not gold_paths:
        diagnosis = "OFFICIAL GRAPH COVERAGE FAILURE"
    elif predicted.isdisjoint(gold_relation_paths):
        diagnosis = "PLANNER FAILURE"
    else:
        diagnosis = "TRUE RETRIEVAL ANOMALY"

    diagnosis_counts_cwq[diagnosis] += 1
    diagnosis_by_qid_cwq[qid] = diagnosis

    if printed < SAMPLE_PRINT:
        print("\n" + "="*100)
        print("QUESTION:", rec["question"])
        print("DIAGNOSIS:", diagnosis)
        printed += 1

print("\n--- Failure breakdown ---")
for k, v in diagnosis_counts_cwq.most_common():
    print(f"{k}: {v}  ({100*v/max(len(missed_qids_cwq_full),1):.2f}%)")

1109 / 3531 questions missed


Failure analysis (CWQ full):   0%|          | 0/1109 [00:00<?, ?it/s]


QUESTION: What Government position holder fought in the battle of Vicksburg?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Which Attorney general fought in the battle of Vicksburg?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Who does Queens hair on a film that James Jordan was a crew member on?
DIAGNOSIS: PLANNER FAILURE

QUESTION: For which teams did the father of the performer in "NFL Super Bowl XLIV Champions: New Orleans Saints" play?
DIAGNOSIS: UNVERIFIED (hit expansion cap)

QUESTION: Who is the prime minister of the country that is the major exports of coffee and tea?
DIAGNOSIS: OFFICIAL GRAPH COVERAGE FAILURE

--- Failure breakdown ---
UNVERIFIED (hit expansion cap): 663  (59.78%)
OFFICIAL GRAPH COVERAGE FAILURE: 352  (31.74%)
PLANNER FAILURE: 94  (8.48%)


### Full-Scale Output

In [59]:
cwq_full_json_path = os.path.join(CWQ_FULL_DIR, "step1_baseline_cwq_full.json")
cwq_full_csv_path = os.path.join(CWQ_FULL_DIR, "step1_baseline_cwq_full.csv")

with open(cwq_full_json_path, "w") as f:
    json.dump(cwq_retrieval_records_full, f, indent=2)
pd.DataFrame(cwq_retrieval_records_full).to_csv(cwq_full_csv_path, index=False)
print("Saved:", cwq_full_json_path)
print("Saved:", cwq_full_csv_path)

Saved: /kaggle/working/step1_cwq_full/step1_baseline_cwq_full.json
Saved: /kaggle/working/step1_cwq_full/step1_baseline_cwq_full.csv


### Final RoG Answer (Full Planning)

In [60]:
final_answers_cwq_full = {}
with open(CWQ_REASON_CKPT, "a") as fout:
    reasoning_done_cwq = load_checkpoint(CWQ_REASON_CKPT)
    print(f"Resuming: {len(reasoning_done_cwq)} CWQ answers already generated.")

    for qid, rec in tqdm(cwq_planning_by_id_full.items(), desc="Reasoning (CWQ full)"):
        if qid in reasoning_done_cwq:
            continue
        q_dict = {
            "question": rec["question"], "graph": rec["graph"],
            "q_entity": rec["q_entity"], "predicted_paths": rec["predicted_paths"],
            "choices": [],
        }
        prompt = reasoning_prompter.process_input(q_dict)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=True)
        gen_time = time.time() - t0
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        out_rec = {"id": qid, "final_RoG_answer": text, "reasoning_time_sec": gen_time}
        fout.write(json.dumps(out_rec) + "\n")
        fout.flush()
        reasoning_done_cwq[qid] = out_rec

for rec in cwq_retrieval_records_full:
    ans = reasoning_done_cwq.get(rec["question_id"])
    if ans:
        rec.update({"final_RoG_answer": ans["final_RoG_answer"], "reasoning_time_sec": ans["reasoning_time_sec"]})

with open(cwq_full_json_path, "w") as f:
    json.dump(cwq_retrieval_records_full, f, indent=2)
pd.DataFrame(cwq_retrieval_records_full).to_csv(cwq_full_csv_path, index=False)
print("Updated with final_RoG_answer and re-saved.")

Resuming: 3531 CWQ answers already generated.


Reasoning (CWQ full):   0%|          | 0/3531 [00:00<?, ?it/s]

Updated with final_RoG_answer and re-saved.


### Confirm the Reasoning Loop

In [61]:
missing_cwq = [qid for qid in cwq_planning_by_id_full if qid not in reasoning_done_cwq]
print(f"{len(reasoning_done_cwq)} / {len(cwq_planning_by_id_full)} CWQ questions have a final answer.")
if missing_cwq:
    print(f"{len(missing_cwq)} still missing -- re-run the reasoning cell above (it resumes automatically).")
else:
    print("CWQ reasoning complete. Safe to proceed.")

3531 / 3531 CWQ questions have a final answer.
CWQ reasoning complete. Safe to proceed.


### Memory Cache for RoG-cwq


In [62]:
# === MEMORY: drop cached subgraphs now that CWQ reasoning is done ===
for rec in cwq_planning_records_full:
    rec.pop("graph", None)
gc.collect()
print("Dropped cached subgraphs from cwq_planning_records_full to free RAM.")

Dropped cached subgraphs from cwq_planning_records_full to free RAM.


### Clean Per-Quesstion Table

In [63]:
retrieval_time_by_qid_cwq = defaultdict(float)
gold_reachable_by_qid_cwq = defaultdict(bool)
for rec in cwq_retrieval_records_full:
    retrieval_time_by_qid_cwq[rec["question_id"]] += rec["retrieval_time_sec"]
    gold_reachable_by_qid_cwq[rec["question_id"]] |= rec["gold_reachable"]

rows_cwq = []
for qid, prec in cwq_planning_by_id_full.items():
    rrow = reasoning_done_cwq.get(qid, {})
    rows_cwq.append({
        "id": qid,
        "question": prec["question"],
        "gold_answers": prec["a_entity"],
        "predicted_paths": prec["predicted_paths"],
        "planning_time_sec": prec["planning_time_sec"],
        "retrieval_time_sec": retrieval_time_by_qid_cwq.get(qid, 0.0),
        "reasoning_time_sec": rrow.get("reasoning_time_sec"),
        "final_RoG_answer": rrow.get("final_RoG_answer"),
        "gold_reachable": gold_reachable_by_qid_cwq.get(qid, False),
    })

cwq_per_question = pd.DataFrame(rows_cwq)
n_cwq = len(cwq_per_question)
assert cwq_per_question["reasoning_time_sec"].notna().all(), "Some CWQ questions missing reasoning -- finish that loop first."
print(f"CWQ per-question baseline table: {n_cwq} questions.")

CWQ per-question baseline table: 3531 questions.


### Full Question-Per Table

In [64]:
pd.set_option("display.max_colwidth", 100)

display_cols = [
    "id", "question", "gold_answers", "predicted_paths",
    "planning_time_sec", "retrieval_time_sec", "reasoning_time_sec",
    "final_RoG_answer", "gold_reachable"
]

cwq_per_question_display = cwq_per_question[display_cols].rename(columns={"id": "question_id"})
cwq_per_question_display

,question_id,question,gold_answers,predicted_paths,planning_time_sec,retrieval_time_sec,reasoning_time_sec,final_RoG_answer,gold_reachable
0,WebQTest-832_c334509bb5e02cacae1ba2e80c176499,Lou Seal is the mascot for the team that last won the World Series when?,[2014 World Series],"[[sports.mascot.team, sports.sports_championship_event.champion], [sports.mascot.team, sports.sp...",2.251861,0.000374,1.090070,2014 World Series\n2012 World Series,True
1,WebQTrn-1259_1997cb4922db71983be26e6a509950f4,"Where did the ""Country Nation World Tour"" concert artist go to college?",[Belmont University],"[[music.concert_tour.artist, people.person.education, education.education.institution], [music.a...",2.370586,0.000483,0.446718,Berklee College of Music,False
2,WebQTest-1384_744a496b907e407b16bc5d7c197dc3f0,What is the predominant religion where the leader is Ovadia Yosef?,[Judaism],"[[people.person.religion], [people.person.nationality, base.argumentmaps.thing_of_disputed_value...",2.165733,0.000606,0.509523,Judaism,True
3,WebQTrn-241_dfb6c97ac9bf2f0ac07f27dd80f9edc2,What country bordering France contains an airport that serves Nijmegen?,[Germany],"[[olympics.olympic_participating_country.olympics_participated_in, olympics.olympic_participatin...",3.910672,0.005167,3.274526,Germany,True
4,WebQTrn-1077_f4a9e5f1e0dcfb82cbadf4771eda7bb5,The national anthem Afghan National Anthem is from the country which practices what religions?,"[Shia Islam, Sunni Islam]","[[music.composition.language, language.human_language.countries_spoken_in], [music.composition.l...",1.790218,0.000139,0.832482,Christianity\nBuddhism\nIslam,False
...,...,...,...,...,...,...,...,...,...
3526,WebQTrn-1938_0e945cac8043fe5af615e4b2f0ddac8f,What is the type of government practiced in the country where the Israeli Lira is used?,[Parliamentary system],"[[finance.currency.countries_formerly_used, government.form_of_government.countries], [finance.c...",2.235173,0.001682,0.282712,Parliamentary system,True
3527,WebQTest-989_20bb2e223d83caf91ca75a04b854377c,"What event with less than 30,000 casualties happened at Dunkirk in WW2?",[Battle of Dunkirk],"[[military.military_conflict.casualties, military.casualties.military_conflict], [military.milit...",2.387669,0.000491,0.680226,Raid on Dunkirk,False
3528,WebQTrn-1722_aca40552e6778874c40c75071d819a54,North America is where Bahamas Creole English Language is spoken belong to.?,[North America],"[[location.location.containedby], [language.human_language.region], [location.country.languages_...",0.910697,0.000039,0.466975,Bahamas\nAmericas,False
3529,WebQTrn-3543_bbb0c8aa3a2941db5bf85e7557241fda,"Find the country with the ISO number of 736 that that imports from Japan, what is the name of th...",[Sudan],"[[base.aareas.schema.administrative_area.administrative_area_type, base.aareas.schema.administra...",3.395276,0.002277,2.505633,Kiribati,True


### Saved as CSV and JSON

In [65]:
cwq_csv_out = os.path.join(CWQ_FULL_DIR, "cwq_per_question_baseline.csv")
cwq_json_out = os.path.join(CWQ_FULL_DIR, "cwq_per_question_baseline.json")

cwq_per_question_display.to_csv(cwq_csv_out, index=False)
cwq_per_question_display.to_json(cwq_json_out, orient="records", indent=2)

print("Saved:", cwq_csv_out)
print("Saved:", cwq_json_out)

Saved: /kaggle/working/step1_cwq_full/cwq_per_question_baseline.csv
Saved: /kaggle/working/step1_cwq_full/cwq_per_question_baseline.json


### Timing Statistics

In [66]:
def total_avg_cwq(col):
    total = cwq_per_question[col].sum()
    return total, total / n_cwq

plan_total_cwq, plan_avg_cwq = total_avg_cwq("planning_time_sec")
retr_total_cwq, retr_avg_cwq = total_avg_cwq("retrieval_time_sec")
reas_total_cwq, reas_avg_cwq = total_avg_cwq("reasoning_time_sec")
e2e_total_cwq = plan_total_cwq + retr_total_cwq + reas_total_cwq
e2e_avg_cwq = e2e_total_cwq / n_cwq

print(f"Total planning time:   {plan_total_cwq:9.1f} s  ({plan_total_cwq/60:7.2f} min)   avg/question: {plan_avg_cwq:.3f} s")
print(f"Total retrieval time:  {retr_total_cwq:9.1f} s  ({retr_total_cwq/60:7.2f} min)   avg/question: {retr_avg_cwq:.4f} s")
print(f"Total reasoning time:  {reas_total_cwq:9.1f} s  ({reas_total_cwq/60:7.2f} min)   avg/question: {reas_avg_cwq:.3f} s")
print(f"Total end-to-end time: {e2e_total_cwq:9.1f} s  ({e2e_total_cwq/60:7.2f} min)   avg/question: {e2e_avg_cwq:.3f} s")

Total planning time:      7044.8 s  ( 117.41 min)   avg/question: 1.995 s
Total retrieval time:        3.1 s  (   0.05 min)   avg/question: 0.0009 s
Total reasoning time:     6027.2 s  ( 100.45 min)   avg/question: 1.707 s
Total end-to-end time:   13075.1 s  ( 217.92 min)   avg/question: 3.703 s


### Retrieval Coverage and Official Hit/Hits@1/F1

In [67]:
import sys
sys.path.append(os.path.join(REPO_DIR, "src"))
from qa_prediction.evaluate_results import eval_acc, eval_hit, eval_f1

n_covered_cwq = int(cwq_per_question["gold_reachable"].sum())
coverage_pct_cwq = 100 * n_covered_cwq / n_cwq

hit_list_cwq, acc_list_cwq, f1_list_cwq, prec_list_cwq, rec_list_cwq = [], [], [], [], []
for _, row in cwq_per_question.iterrows():
    prediction = [p for p in row["final_RoG_answer"].split("\n") if p.strip()]
    prediction_str = " ".join(prediction)
    f1, precision, recall = eval_f1(prediction, row["gold_answers"])
    hit_list_cwq.append(eval_hit(prediction_str, row["gold_answers"]))
    acc_list_cwq.append(eval_acc(prediction_str, row["gold_answers"]))
    f1_list_cwq.append(f1); prec_list_cwq.append(precision); rec_list_cwq.append(recall)

hits1_pct_cwq = 100 * sum(hit_list_cwq) / n_cwq
f1_pct_cwq = 100 * sum(f1_list_cwq) / n_cwq

print(f"Questions reaching >=1 gold: {n_covered_cwq}")
print(f"Question-level retrieval coverage: {coverage_pct_cwq:.2f}%")
print(f"Hit/Hits@1: {hits1_pct_cwq:.2f}%")
print(f"F1: {f1_pct_cwq:.2f}%")

Questions reaching >=1 gold: 2422
Question-level retrieval coverage: 68.59%
Hit/Hits@1: 61.23%
F1: 54.32%


### Freezing Baseline

In [68]:
cwq_baseline_summary = f"""Dataset: CWQ
Test questions: {n_cwq}

Retrieval:
  Questions reaching >=1 gold: {n_covered_cwq}
  Question-level retrieval coverage: {coverage_pct_cwq:.2f}%

Timing:
  Total planning time:   {plan_total_cwq:.1f} s ({plan_total_cwq/60:.2f} min)   Avg/question: {plan_avg_cwq:.3f} s
  Total retrieval time:  {retr_total_cwq:.1f} s ({retr_total_cwq/60:.2f} min)   Avg/question: {retr_avg_cwq:.4f} s
  Total reasoning time:  {reas_total_cwq:.1f} s ({reas_total_cwq/60:.2f} min)   Avg/question: {reas_avg_cwq:.3f} s
  Total end-to-end time: {e2e_total_cwq:.1f} s ({e2e_total_cwq/60:.2f} min)   Avg/question: {e2e_avg_cwq:.3f} s

Final QA:
  Hit/Hits@1: {hits1_pct_cwq:.2f}%
  F1: {f1_pct_cwq:.2f}%
"""
print(cwq_baseline_summary)

with open(os.path.join(CWQ_FULL_DIR, "cwq_baseline_summary.txt"), "w") as f:
    f.write(cwq_baseline_summary)

cwq_per_question.to_json(os.path.join(CWQ_FULL_DIR, "cwq_baseline_predictions.json"), orient="records", indent=2)
cwq_per_question.to_csv(os.path.join(CWQ_FULL_DIR, "cwq_baseline_predictions.csv"), index=False)
print("CWQ Baseline frozen.")

Dataset: CWQ
Test questions: 3531

Retrieval:
  Questions reaching >=1 gold: 2422
  Question-level retrieval coverage: 68.59%

Timing:
  Total planning time:   7044.8 s (117.41 min)   Avg/question: 1.995 s
  Total retrieval time:  3.1 s (0.05 min)   Avg/question: 0.0009 s
  Total reasoning time:  6027.2 s (100.45 min)   Avg/question: 1.707 s
  Total end-to-end time: 13075.1 s (217.92 min)   Avg/question: 3.703 s

Final QA:
  Hit/Hits@1: 61.23%
  F1: 54.32%

CWQ Baseline frozen.


# Objective 2

## Read-Only Search-Space Profiler

## Metric Definitions

**Hop indexing.** Hop `h` executes the planned relation `target_rule[h]`
(0-indexed), corresponding to $r_{h+1}$ in the paper notation.
For a relation plan of length $L$, the final hop is $h=L-1$.

| Counter / Metric | Definition |
|---|---|
| `active_prefixes[h]` | Number of active path-prefix states $|\mathcal{P}_h|$ presented for expansion at hop `h`. Counted before the node-in-graph check; therefore, a start entity absent from its local graph still occupies the initial frontier. |
| `unique_expanded_nodes[h]` | Number of distinct entities whose adjacency lists are actually inspected at hop `h`. Multiple path prefixes ending at the same entity contribute only once to this unique-entity count. |
| `edges_examined[h]` | Number of adjacency entries inspected at hop `h` before relation matching. This is counted per path-prefix expansion; if the same entity is reached through multiple prefixes and expanded repeatedly, each adjacency inspection is counted. |
| `candidate_branches[h]` | Number of relation-valid candidate extensions generated at hop `h`, i.e. $|\mathcal{C}_h|$. These are the branches satisfying the required relation $r_{h+1}$. This has the same counting semantics as the legacy `hop_frontier` counter in Objective 1. |
| `branch_expansion_ratio[h]` | Relation-valid branch expansion ratio $\rho_h = |\mathcal{C}_h| / |\mathcal{P}_h|$, defined only when $|\mathcal{P}_h| > 0$. Values $>1$, $=1$, and $<1$ indicate frontier expansion, unchanged size, and contraction, respectively. |
| `unique_frontier_nodes[h]` | Number of distinct endpoint entities in the relation-valid candidate frontier generated at hop `h`, corresponding to $|\mathcal{U}_{h+1}|$. Distinct path prefixes may terminate at the same entity. |
| `retrieved_paths` | Number of complete reasoning paths produced after execution of the current topic-entity/relation-plan traversal. |
| `peak_frontier_size` | Maximum number of active path prefixes over the hops of a traversal, i.e. $\max_h |\mathcal{P}_h|$. At question level, Peak Frontier is the maximum across all executed topic entities, relation plans, and hops. |
| `peak_queue_size` | Maximum physical `deque` length observed during implementation-level BFS execution. This is retained only as a diagnostic and is **not** treated as the paper's Peak Frontier metric. |
| `afp_eligible` | Indicates whether the relation plan contains at least one intermediate pruning position. Under final-hop protection, a plan is AFP-eligible when $L \geq 2$. |
| `decision_opportunity[h]` | Indicates an intermediate hop at which AFP would have an actual branch-selection decision: $h < L-1$ and $|\mathcal{C}_h| > 1$. |
| `downstream_expansion_edges[h]` | Remaining baseline edge-examination work after hop `h`, defined as $D_h = \sum_{t=h+1}^{L-1} E_t$. This measures the amount of future graph expansion potentially affected by a pruning decision made at hop `h`. |

**Aggregation rule.** Path-prefix, edge, and candidate-branch counts are
aggregated across topic entities and relation plans by **summing** their
counts. Quantities representing distinct entities are aggregated by **set
union**, never by summing per-call unique counts. Peak Frontier is obtained
by a **maximum**, not a sum.

**Interpretation.** The counters are intentionally complementary.
$|\mathcal{P}_h|$ and $|\mathcal{C}_h|$ characterize path-level branching,
unique-entity counters characterize entity-level exploration, and
`edges_examined[h]` measures the actual adjacency work performed by
relation-constrained retrieval. The expansion ratio $\rho_h$ shows how the
relation-valid frontier changes between hops, while
`downstream_expansion_edges[h]` indicates how much future traversal remains
after a possible intermediate pruning decision.

In [69]:
# ## 2.2 Profiled BFS v2 — READ-ONLY per-hop search-space profiler
# Traversal below is copied VERBATIM from bfs_with_rule_instrumented (cell 23), which is
# itself copied verbatim from src/utils/graph_utils.py. Only observation counters were
# added. Returns (result_paths, profile): result_paths must be identical to official
# bfs_with_rule output in contents, order, AND multiplicity.

from collections import deque
import time

def bfs_with_rule_profiled_v2(graph, start_node, target_rule):
    t0 = time.perf_counter()
    L = len(target_rule)
    result_paths = []
    active_prefixes = [0] * L                     # |P_h| dequeued for expansion
    unique_expanded_sets = [set() for _ in range(L)]
    edges_examined = [0] * L                      # adjacency entries inspected pre-match
    candidate_branches = [0] * L                  # == legacy hop_frontier semantics
    unique_frontier_sets = [set() for _ in range(L)]
    peak_queue_size = 1                           # seed state occupies the queue initially
    queue = deque([(start_node, [])])
    while queue:
        if len(queue) > peak_queue_size:
            peak_queue_size = len(queue)
        current_node, current_path = queue.popleft()
        if len(current_path) == L:
            result_paths.append(current_path)
        if len(current_path) < L:
            active_prefixes[len(current_path)] += 1
            if current_node not in graph:
                continue
            hop_idx = len(current_path)
            unique_expanded_sets[hop_idx].add(current_node)
            for neighbor in graph.neighbors(current_node):
                edges_examined[hop_idx] += 1
                rel = graph[current_node][neighbor]["relation"]
                if rel != target_rule[hop_idx] or len(current_path) > len(target_rule):
                    continue
                queue.append((neighbor, current_path + [(current_node, rel, neighbor)]))
                candidate_branches[hop_idx] += 1
                unique_frontier_sets[hop_idx].add(neighbor)
    profile = {
        "plan_length": L,
        "active_prefixes": active_prefixes,
        "unique_expanded_nodes": [len(s) for s in unique_expanded_sets],
        "edges_examined": edges_examined,
        "candidate_branches": candidate_branches,
        "unique_frontier_nodes": [len(s) for s in unique_frontier_sets],
        "peak_active_prefixes": max(active_prefixes) if L > 0 else 0,
        "peak_queue_size": peak_queue_size,
        "retrieved_paths": len(result_paths),
        "time_sec_perf": time.perf_counter() - t0,
        "_unique_expanded_node_sets": unique_expanded_sets,   # raw sets, for correct union-aggregation
        "_unique_frontier_node_sets": unique_frontier_sets,   # raw sets, for correct union-aggregation
    }
    return result_paths, profile

print("bfs_with_rule_profiled_v2 defined.")

bfs_with_rule_profiled_v2 defined.


## Official-vs-profiled equivalence test (hard gate)

In [70]:

# ## 2.3 Official-vs-profiled equivalence test (WebQSP + CWQ, ALL frozen test calls)
# Hard gate per PRD §13: bfs_with_rule_profiled_v2 must reproduce official bfs_with_rule
# EXACTLY -- same paths, order, multiplicity -- before any profile data is trusted.

from datasets import load_dataset
from tqdm.auto import tqdm

# Build graph lookup maps from dataset objects (since in-memory records had 'graph' popped to save RAM)
if "full_test_all" not in globals():
    full_test_all = load_dataset(DATASET_NAME, split=SPLIT)
webqsp_graph_map = {sample["id"]: sample["graph"] for sample in full_test_all}

if "cwq_full_test_all" not in globals():
    cwq_full_test_all = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
cwq_graph_map = {sample["id"]: sample["graph"] for sample in cwq_full_test_all}

def run_equivalence_test(planning_recs, graph_map, dataset_label):
    checked = mismatches = reachability_mismatches = path_count_mismatches = 0
    for rec in tqdm(planning_recs, desc=f"Equivalence ({dataset_label})"):
        # Retrieve graph from rec if present, otherwise lookup from dataset map
        graph_data = rec.get("graph") or graph_map[rec["id"]]
        graph = build_graph(graph_data)
        gold_answers = set(rec["a_entity"])

        for entity in rec["q_entity"]:
            for rule in rec["predicted_paths"]:
                if len(rule) == 0:
                    continue  # Baseline retrieval loops skip empty relation plans

                checked += 1
                official_result = bfs_with_rule(graph, entity, rule)
                profiled_result, _ = bfs_with_rule_profiled_v2(graph, entity, rule)

                # Check 1: Exact path list equality (order + items)
                if official_result != profiled_result:
                    mismatches += 1
                    continue

                # Check 2: Path count equality
                if len(official_result) != len(profiled_result):
                    path_count_mismatches += 1

                # Check 3: Gold answer reachability match
                official_reachable = len(path_endpoints(official_result) & gold_answers) > 0
                profiled_reachable = len(path_endpoints(profiled_result) & gold_answers) > 0
                if official_reachable != profiled_reachable:
                    reachability_mismatches += 1

    print(f"\n[{dataset_label}] Checked {checked} calls | mismatches: {mismatches} | "
              f"reachability mismatches: {reachability_mismatches} | path-count mismatches: {path_count_mismatches}")
    return checked, mismatches, reachability_mismatches, path_count_mismatches

# 1. Run WebQSP Hard Gate
webqsp_checked, webqsp_mm, webqsp_reach_mm, webqsp_count_mm = run_equivalence_test(
    planning_records_full, webqsp_graph_map, "WebQSP test"
)
assert webqsp_mm == 0 and webqsp_reach_mm == 0 and webqsp_count_mm == 0, "Diverges on WebQSP -- STOP."

# 2. Run CWQ Hard Gate
cwq_checked, cwq_mm, cwq_reach_mm, cwq_count_mm = run_equivalence_test(
    cwq_planning_records_full, cwq_graph_map, "CWQ test"
)
assert cwq_mm == 0 and cwq_reach_mm == 0 and cwq_count_mm == 0, "Diverges on CWQ -- STOP."

print("\n=== EQUIVALENCE GATE: PASSED ===")
print(f"WebQSP: {webqsp_checked} calls, 0 mismatches")
print(f"CWQ:    {cwq_checked} calls, 0 mismatches")

Equivalence (WebQSP test):   0%|          | 0/1628 [00:00<?, ?it/s]


[WebQSP test] Checked 4937 calls | mismatches: 0 | reachability mismatches: 0 | path-count mismatches: 0


Equivalence (CWQ test):   0%|          | 0/3531 [00:00<?, ?it/s]


[CWQ test] Checked 16421 calls | mismatches: 0 | reachability mismatches: 0 | path-count mismatches: 0

=== EQUIVALENCE GATE: PASSED ===
WebQSP: 4937 calls, 0 mismatches
CWQ:    16421 calls, 0 mismatches


## Freeze validation relation plans (dev inputs only, test gold untouched)

In [71]:
# ## 2.4 Freeze validation relation plans (WebQSP + CWQ)
# Dev-only planner runs, checkpointed like the frozen test runs. Never touches test gold.
RQ1_DEV_DIR = "/kaggle/working/step2_rq1_dev"
os.makedirs(RQ1_DEV_DIR, exist_ok=True)

WEBQSP_VAL_PLANNING_CKPT = os.path.join(RQ1_DEV_DIR, "planning_webqsp_validation.jsonl")
CWQ_VAL_PLANNING_CKPT = os.path.join(RQ1_DEV_DIR, "planning_cwq_validation.jsonl")

SEED = 42
torch.manual_seed(SEED)

webqsp_val = load_dataset(DATASET_NAME, split="validation")
cwq_val = load_dataset(CWQ_DATASET_NAME, split="validation")
print(f"WebQSP validation: {len(webqsp_val)} questions")
print(f"CWQ validation: {len(cwq_val)} questions  (open item: not yet in verified-facts.md -- record this number there)")

def run_planning_checkpointed(dataset_split, ckpt_path, desc):
    done = load_checkpoint(ckpt_path)
    print(f"Resuming: {len(done)} / {len(dataset_split)} already planned ({desc}).")
    with open(ckpt_path, "a") as fout:
        for sample in tqdm(dataset_split, desc=desc):
            if sample["id"] in done:
                continue
            input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
            t0 = time.time()
            raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
            planning_time = time.time() - t0
            rel_paths = parse_prediction(raw_output["paths"])
            rec = {
                "id": sample["id"], "question": sample["question"],
                "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
                "graph": sample["graph"], "predicted_paths": rel_paths,
                "planning_time_sec": planning_time,
            }
            fout.write(json.dumps(rec) + "\n")
            fout.flush()
            done[sample["id"]] = rec
    return list(done.values())

webqsp_val_planning = run_planning_checkpointed(webqsp_val, WEBQSP_VAL_PLANNING_CKPT, "Planning (WebQSP validation)")
cwq_val_planning = run_planning_checkpointed(cwq_val, CWQ_VAL_PLANNING_CKPT, "Planning (CWQ validation)")

print(f"WebQSP validation plans frozen: {len(webqsp_val_planning)}")
print(f"CWQ validation plans frozen: {len(cwq_val_planning)}")

WebQSP validation: 246 questions
CWQ validation: 3519 questions  (open item: not yet in verified-facts.md -- record this number there)
Resuming: 246 / 246 already planned (Planning (WebQSP validation)).


Planning (WebQSP validation):   0%|          | 0/246 [00:00<?, ?it/s]

Resuming: 3519 / 3519 already planned (Planning (CWQ validation)).


Planning (CWQ validation):   0%|          | 0/3519 [00:00<?, ?it/s]

WebQSP validation plans frozen: 246
CWQ validation plans frozen: 3519


## Development RQ1 profiling over frozen validation plans

In [72]:
# ## 2.5 RQ1 profiling over frozen validation plans -- raw question x plan x hop logs
# Schema per PRD §11 & Paper Definitions:
# rho_h, decision opportunities, downstream edges, peak frontier.
#
# IMPORTANT:
# - Prefix/edge/branch counts are SUMMED across topic entities.
# - Unique-node quantities are aggregated by SET UNION across topic entities.
# - Decision opportunity is detected within an actual topic-entity traversal.
# - Peak Frontier is the maximum active-prefix frontier observed in any traversal.
# - peak_queue_size is retained only as an implementation diagnostic.

import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def profile_validation_set(planning_recs, dataset_label):
    rows = []

    for rec in tqdm(planning_recs, desc=f"RQ1 profiling ({dataset_label})"):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        topic_entity_count = len(rec["q_entity"])

        # First pass: determine plan-level and question-level answer reachability
        plan_reachable_flags = []
        for rule in rec["predicted_paths"]:
            plan_paths = []
            if len(rule) > 0:
                for entity in rec["q_entity"]:
                    paths, _ = bfs_with_rule_profiled_v2(graph, entity, rule)
                    plan_paths.extend(paths)

            plan_reachable = len(path_endpoints(plan_paths) & gold_answers) > 0
            plan_reachable_flags.append(plan_reachable)

        question_gold_reachable = any(plan_reachable_flags)

        # Second pass: collect raw RQ1 profiling statistics
        for plan_rank, rule in enumerate(rec["predicted_paths"]):
            L = len(rule)
            if L == 0:
                continue

            t0 = time.perf_counter()

            agg_active = [0] * L
            agg_edges = [0] * L
            agg_branches = [0] * L
            union_expanded = [set() for _ in range(L)]
            union_frontier = [set() for _ in range(L)]

            total_retrieved_paths = 0
            peak_queue = 0
            plan_peak_frontier = 0
            decision_flags = [False] * L

            for entity in rec["q_entity"]:
                _, profile = bfs_with_rule_profiled_v2(graph, entity, rule)

                entity_peak_frontier = (
                    max(profile["active_prefixes"])
                    if len(profile["active_prefixes"]) > 0 else 0
                )
                plan_peak_frontier = max(plan_peak_frontier, entity_peak_frontier)
                peak_queue = max(peak_queue, profile["peak_queue_size"])

                for h in range(L):
                    agg_active[h] += profile["active_prefixes"][h]
                    agg_edges[h] += profile["edges_examined"][h]
                    agg_branches[h] += profile["candidate_branches"][h]
                    union_expanded[h] |= profile["_unique_expanded_node_sets"][h]
                    union_frontier[h] |= profile["_unique_frontier_node_sets"][h]

                    if h < L - 1 and profile["candidate_branches"][h] > 1:
                        decision_flags[h] = True

                total_retrieved_paths += profile["retrieved_paths"]

            retrieval_time = time.perf_counter() - t0
            plan_gold_reachable = plan_reachable_flags[plan_rank]
            plan_afp_eligible = (L >= 2)

            for h in range(L):
                # rho_h = |C_h| / |P_h|
                branch_expansion_ratio = (
                    agg_branches[h] / agg_active[h]
                    if agg_active[h] > 0 else np.nan
                )

                # AFP decision opportunity at an intermediate hop
                decision_opportunity = decision_flags[h]

                # D_h = sum of future baseline edge examinations after hop h
                downstream_expansion_edges = (
                    sum(agg_edges[h + 1:]) if h < L - 1 else 0
                )

                rows.append({
                    "dataset": dataset_label,
                    "question_id": rec["id"],
                    "plan_id": f"{rec['id']}_p{plan_rank}",
                    "plan_rank": plan_rank,
                    "plan_length": L,
                    "hop": h,
                    "required_relation": rule[h],
                    "topic_entity_count": topic_entity_count,

                    "active_prefixes": agg_active[h],
                    "unique_expanded_nodes": len(union_expanded[h]),
                    "edges_examined": agg_edges[h],
                    "candidate_branches": agg_branches[h],
                    "unique_frontier_nodes": len(union_frontier[h]),
                    "branch_expansion_ratio": branch_expansion_ratio,

                    "retrieved_paths_final_plan": total_retrieved_paths,
                    "peak_frontier_size_plan": plan_peak_frontier,
                    "peak_queue_size_plan": peak_queue,

                    "plan_gold_reachable": plan_gold_reachable,
                    "question_gold_reachable": question_gold_reachable,

                    "afp_eligible_plan": plan_afp_eligible,
                    "decision_opportunity": decision_opportunity,
                    "downstream_expansion_edges": downstream_expansion_edges,

                    "retrieval_time_sec_plan": retrieval_time,
                })

    return pd.DataFrame(rows)


# =========================================================
# Run development RQ1 profiling
# =========================================================
df_rq1_webqsp_dev = profile_validation_set(webqsp_val_planning, "webqsp")
df_rq1_cwq_dev = profile_validation_set(cwq_val_planning, "cwq")


# =========================================================
# Save raw question x plan x hop logs
# =========================================================
RQ1_DEV_DIR_OUT = RQ1_DEV_DIR

df_rq1_webqsp_dev.to_csv(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_webqsp_dev_plan_hop.csv"),
    index=False
)
df_rq1_cwq_dev.to_csv(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_cwq_dev_plan_hop.csv"),
    index=False
)
df_rq1_webqsp_dev.to_json(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_webqsp_dev_plan_hop.jsonl"),
    orient="records",
    lines=True
)
df_rq1_cwq_dev.to_json(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_cwq_dev_plan_hop.jsonl"),
    orient="records",
    lines=True
)

print(f"WebQSP dev plan-hop rows: {len(df_rq1_webqsp_dev)}")
print(f"CWQ dev plan-hop rows: {len(df_rq1_cwq_dev)}")

RQ1 profiling (webqsp):   0%|          | 0/246 [00:00<?, ?it/s]

RQ1 profiling (cwq):   0%|          | 0/3519 [00:00<?, ?it/s]

WebQSP dev plan-hop rows: 1034
CWQ dev plan-hop rows: 18851


## Quick Sanity Check

In [73]:
# ## 2.5.1 Sanity checks for raw RQ1 profiling logs

def sanity_check_rq1(df, label):
    print(f"\n=== {label} RQ1 RAW PROFILE SANITY CHECK ===")
    print(f"Rows: {len(df)}")
    print(f"Questions: {df['question_id'].nunique()}")
    print(f"Plans: {df['plan_id'].nunique()}")
    print(f"Hop range: {df['hop'].min()} -> {df['hop'].max()}")

    # Basic non-negativity
    count_cols = [
        "active_prefixes",
        "unique_expanded_nodes",
        "edges_examined",
        "candidate_branches",
        "unique_frontier_nodes",
        "retrieved_paths_final_plan",
        "peak_frontier_size_plan",
        "peak_queue_size_plan",
        "downstream_expansion_edges",
    ]

    for col in count_cols:
        print(f"{col}: negative values = {(df[col] < 0).sum()}")

    # rho_h should be NaN only when active_prefixes == 0
    invalid_rho_nan = df[
        df["branch_expansion_ratio"].isna()
        & (df["active_prefixes"] > 0)
    ]

    invalid_rho_defined = df[
        df["branch_expansion_ratio"].notna()
        & (df["active_prefixes"] == 0)
    ]

    print(
        "rho NaN despite active_prefixes > 0:",
        len(invalid_rho_nan)
    )
    print(
        "rho defined despite active_prefixes == 0:",
        len(invalid_rho_defined)
    )

    # Candidate branches should be zero if no active prefixes exist
    print(
        "candidate_branches > 0 with active_prefixes == 0:",
        len(df[
            (df["active_prefixes"] == 0)
            & (df["candidate_branches"] > 0)
        ])
    )

    # Decision opportunities must never occur on final hop
    print(
        "decision opportunity on final hop:",
        len(df[
            df["decision_opportunity"]
            & (df["hop"] == df["plan_length"] - 1)
        ])
    )

    # Decision opportunities must belong to AFP-eligible plans
    print(
        "decision opportunity on non-AFP-eligible plan:",
        len(df[
            df["decision_opportunity"]
            & (~df["afp_eligible_plan"])
        ])
    )

    # Final hop must have zero downstream expansion
    print(
        "nonzero downstream edges at final hop:",
        len(df[
            (df["hop"] == df["plan_length"] - 1)
            & (df["downstream_expansion_edges"] != 0)
        ])
    )

    # One row per question x plan x hop
    duplicates = df.duplicated(
        subset=["question_id", "plan_id", "hop"]
    ).sum()

    print(
        "duplicate question-plan-hop rows:",
        duplicates
    )


sanity_check_rq1(df_rq1_webqsp_dev, "WebQSP")
sanity_check_rq1(df_rq1_cwq_dev, "CWQ")


=== WebQSP RQ1 RAW PROFILE SANITY CHECK ===
Rows: 1034
Questions: 246
Plans: 721
Hop range: 0 -> 1
active_prefixes: negative values = 0
unique_expanded_nodes: negative values = 0
edges_examined: negative values = 0
candidate_branches: negative values = 0
unique_frontier_nodes: negative values = 0
retrieved_paths_final_plan: negative values = 0
peak_frontier_size_plan: negative values = 0
peak_queue_size_plan: negative values = 0
downstream_expansion_edges: negative values = 0
rho NaN despite active_prefixes > 0: 0
rho defined despite active_prefixes == 0: 0
candidate_branches > 0 with active_prefixes == 0: 0
decision opportunity on final hop: 0
decision opportunity on non-AFP-eligible plan: 0
nonzero downstream edges at final hop: 0
duplicate question-plan-hop rows: 0

=== CWQ RQ1 RAW PROFILE SANITY CHECK ===
Rows: 18851
Questions: 3519
Plans: 10529
Hop range: 0 -> 4
active_prefixes: negative values = 0
unique_expanded_nodes: negative values = 0
edges_examined: negative values = 0
can

## Per-hop descriptive statistics and branching prevalence

In [74]:
# ## 2.6 RQ1 development descriptive analysis
# Uses ONLY frozen validation profiling outputs from Cell 141.
# Produces hop-level, plan-level, and question-level summaries required for RQ1.
#
# IMPORTANT:
# - "all_active" includes only observed frontiers with active_prefixes > 0.
# - "intermediate_active" excludes final hops and is most relevant to AFP.
# - Branching prevalence uses aggregated question x plan x hop candidate counts.
# - decision_opportunity is stricter: >1 candidate within an actual topic-entity traversal.
# - Downstream D_h is NOT summed as "avoidable work" because different decision hops can overlap.
# - Unique-node counts are never summed into fake question-level unique counts.

import os
import numpy as np
import pandas as pd

PRIMARY_VARS = [
    "active_prefixes",
    "unique_expanded_nodes",
    "edges_examined",
    "candidate_branches",
    "unique_frontier_nodes",
]

PCTS = [0.50, 0.75, 0.90, 0.95]
BRANCH_THRESHOLDS = [1, 5, 10, 50]

def series_summary(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return {
            "n": 0, "mean": np.nan, "median": np.nan,
            "p75": np.nan, "p90": np.nan,
            "p95": np.nan, "max": np.nan
        }
    return {
        "n": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "max": s.max(),
    }

def distribution_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            for metric in PRIMARY_VARS:
                s = series_summary(g[metric])
                rows.append({
                    "dataset": label,
                    "scope": scope_name,
                    "hop": hop_label,
                    "metric": metric,
                    **s
                })

    return pd.DataFrame(rows)

def rho_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            r = g["branch_expansion_ratio"].dropna()
            s = series_summary(r)

            rows.append({
                "dataset": label,
                "scope": scope_name,
                "hop": hop_label,
                **s,
                "rho_gt_1_pct": 100 * (r > 1).mean() if len(r) else np.nan,
                "rho_eq_1_pct": 100 * np.isclose(r, 1.0).mean() if len(r) else np.nan,
                "rho_lt_1_pct": 100 * (r < 1).mean() if len(r) else np.nan,
            })

    return pd.DataFrame(rows)

def branching_prevalence_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            row = {
                "dataset": label,
                "scope": scope_name,
                "hop": hop_label,
                "n_frontiers": len(g),
            }

            for t in BRANCH_THRESHOLDS:
                row[f"candidate_gt_{t}_pct"] = (
                    100 * (g["candidate_branches"] > t).mean()
                    if len(g) else np.nan
                )

            rows.append(row)

    return pd.DataFrame(rows)

def duplicate_pressure_table(df, label):
    work = df[df["active_prefixes"] > 0].copy()

    # Paper ratio |P_h| / |X_h|.
    # Undefined when no entity adjacency was actually expanded.
    work["prefix_to_unique_expanded_ratio"] = np.where(
        work["unique_expanded_nodes"] > 0,
        work["active_prefixes"] / work["unique_expanded_nodes"],
        np.nan
    )

    # Paper ratio |C_h| / |U_{h+1}|.
    # This directly measures repeated candidate endpoints.
    work["candidate_to_unique_frontier_ratio"] = np.where(
        work["unique_frontier_nodes"] > 0,
        work["candidate_branches"] / work["unique_frontier_nodes"],
        np.nan
    )

    rows = []
    metrics = [
        "prefix_to_unique_expanded_ratio",
        "candidate_to_unique_frontier_ratio",
    ]

    groups = [("ALL", work)]
    groups += [(str(h), g) for h, g in work.groupby("hop")]

    for hop_label, g in groups:
        for metric in metrics:
            s = series_summary(g[metric])

            rows.append({
                "dataset": label,
                "hop": hop_label,
                "metric": metric,
                **s,
                "ratio_gt_1_pct": (
                    100 * (g[metric].dropna() > 1).mean()
                    if g[metric].notna().any() else np.nan
                )
            })

    return pd.DataFrame(rows)

def plan_question_tables(df, label):
    plan_df = (
        df.groupby(
            ["dataset", "question_id", "plan_id", "plan_rank"],
            as_index=False
        )
        .agg(
            plan_length=("plan_length", "first"),
            retrieved_paths_final_plan=("retrieved_paths_final_plan", "first"),
            peak_frontier_size_plan=("peak_frontier_size_plan", "first"),
            peak_queue_size_plan=("peak_queue_size_plan", "first"),
            afp_eligible_plan=("afp_eligible_plan", "first"),
            plan_gold_reachable=("plan_gold_reachable", "first"),
            question_gold_reachable=("question_gold_reachable", "first"),
            plan_has_decision=("decision_opportunity", "max"),
            retrieval_time_sec_plan=("retrieval_time_sec_plan", "first"),
        )
    )

    question_work = (
        df.groupby("question_id", as_index=False)
        .agg(
            active_prefixes_total=("active_prefixes", "sum"),
            edges_examined_total=("edges_examined", "sum"),
            candidate_branches_total=("candidate_branches", "sum"),
        )
    )

    question_plan = (
        plan_df.groupby("question_id", as_index=False)
        .agg(
            n_plans=("plan_id", "nunique"),
            retrieved_paths_total=("retrieved_paths_final_plan", "sum"),
            peak_frontier_size_question=("peak_frontier_size_plan", "max"),
            afp_eligible_question=("afp_eligible_plan", "max"),
            decision_opportunity_question=("plan_has_decision", "max"),
            question_gold_reachable=("question_gold_reachable", "max"),
        )
    )

    question_df = question_work.merge(
        question_plan,
        on="question_id",
        how="inner"
    )

    plan_rows = []
    for metric in [
        "retrieved_paths_final_plan",
        "peak_frontier_size_plan",
    ]:
        plan_rows.append({
            "dataset": label,
            "metric": metric,
            **series_summary(plan_df[metric])
        })

    question_rows = []
    for metric in [
        "active_prefixes_total",
        "edges_examined_total",
        "candidate_branches_total",
        "retrieved_paths_total",
        "peak_frontier_size_question",
    ]:
        question_rows.append({
            "dataset": label,
            "metric": metric,
            **series_summary(question_df[metric])
        })

    return (
        plan_df,
        question_df,
        pd.DataFrame(plan_rows),
        pd.DataFrame(question_rows)
    )

def opportunity_table(df, plan_df, question_df, label):
    intermediate_active = df[
        (df["active_prefixes"] > 0) &
        (df["hop"] < df["plan_length"] - 1)
    ]

    decision_hops = intermediate_active[
        intermediate_active["decision_opportunity"]
    ]

    eligible_plans = plan_df[plan_df["afp_eligible_plan"]]
    eligible_questions = question_df[
        question_df["afp_eligible_question"]
    ]

    rows = [{
        "dataset": label,
        "questions": question_df["question_id"].nunique(),
        "plans": len(plan_df),
        "afp_eligible_plans": int(plan_df["afp_eligible_plan"].sum()),
        "afp_eligible_plans_pct": 100 * plan_df["afp_eligible_plan"].mean(),
        "plans_with_decision": int(plan_df["plan_has_decision"].sum()),
        "plans_with_decision_pct": 100 * plan_df["plan_has_decision"].mean(),
        "decision_pct_among_eligible_plans": (
            100 * eligible_plans["plan_has_decision"].mean()
            if len(eligible_plans) else np.nan
        ),
        "afp_eligible_questions": int(question_df["afp_eligible_question"].sum()),
        "afp_eligible_questions_pct": 100 * question_df["afp_eligible_question"].mean(),
        "questions_with_decision": int(question_df["decision_opportunity_question"].sum()),
        "questions_with_decision_pct": 100 * question_df["decision_opportunity_question"].mean(),
        "decision_pct_among_eligible_questions": (
            100 * eligible_questions["decision_opportunity_question"].mean()
            if len(eligible_questions) else np.nan
        ),
        "intermediate_active_hops": len(intermediate_active),
        "decision_hops": len(decision_hops),
        "decision_hop_pct": (
            100 * len(decision_hops) / len(intermediate_active)
            if len(intermediate_active) else np.nan
        ),
        "decision_hops_with_future_work": int(
            (decision_hops["downstream_expansion_edges"] > 0).sum()
        ),
        "decision_hops_with_future_work_pct": (
            100 * (decision_hops["downstream_expansion_edges"] > 0).mean()
            if len(decision_hops) else np.nan
        ),
    }]

    return pd.DataFrame(rows)

def downstream_table(df, label):
    decision_df = df[
        df["decision_opportunity"] &
        (df["hop"] < df["plan_length"] - 1)
    ].copy()

    rows = []
    groups = [("ALL", decision_df)]
    groups += [(str(h), g) for h, g in decision_df.groupby("hop")]

    for hop_label, g in groups:
        s = series_summary(g["downstream_expansion_edges"])

        rows.append({
            "dataset": label,
            "hop": hop_label,
            **s,
            "future_work_gt_0_pct": (
                100 * (g["downstream_expansion_edges"] > 0).mean()
                if len(g) else np.nan
            )
        })

    return pd.DataFrame(rows)

def reachability_table(plan_df, question_df, label):
    return pd.DataFrame([{
        "dataset": label,
        "plans": len(plan_df),
        "reachable_plans": int(plan_df["plan_gold_reachable"].sum()),
        "plan_reachability_pct": 100 * plan_df["plan_gold_reachable"].mean(),
        "questions": len(question_df),
        "reachable_questions": int(question_df["question_gold_reachable"].sum()),
        "question_reachability_pct": 100 * question_df["question_gold_reachable"].mean(),
    }])

def analyze_rq1_dev(df, label):
    dist = distribution_table(df, label)
    rho = rho_table(df, label)
    branching = branching_prevalence_table(df, label)
    duplicate = duplicate_pressure_table(df, label)

    plan_df, question_df, plan_summary, question_summary = (
        plan_question_tables(df, label)
    )

    opportunity = opportunity_table(
        df, plan_df, question_df, label
    )

    downstream = downstream_table(df, label)
    reachability = reachability_table(
        plan_df, question_df, label
    )

    prefix_undefined = int(
        (
            (df["active_prefixes"] > 0) &
            (df["unique_expanded_nodes"] == 0)
        ).sum()
    )

    candidate_undefined = int(
        (
            (df["candidate_branches"] > 0) &
            (df["unique_frontier_nodes"] == 0)
        ).sum()
    )

    print(f"\n{'='*80}")
    print(f"{label.upper()} -- DEVELOPMENT RQ1 CHARACTERIZATION")
    print(f"{'='*80}")

    print("\n[Primary search-space distributions: all active frontiers]")
    display(
        dist[
            (dist["scope"] == "all_active") &
            (dist["hop"] == "ALL")
        ].reset_index(drop=True)
    )

    print("\n[Primary search-space distributions: intermediate active frontiers]")
    display(
        dist[
            (dist["scope"] == "intermediate_active") &
            (dist["hop"] == "ALL")
        ].reset_index(drop=True)
    )

    print("\n[Hop-wise rho_h]")
    display(
        rho[
            rho["scope"] == "all_active"
        ].reset_index(drop=True)
    )

    print("\n[Branching prevalence]")
    display(
        branching[
            branching["hop"] == "ALL"
        ].reset_index(drop=True)
    )

    print("\n[AFP eligibility and actual decision opportunities]")
    display(opportunity)

    print("\n[Downstream expansion at actual decision hops]")
    display(downstream)

    print("\n[Duplicate search pressure]")
    display(
        duplicate[
            duplicate["hop"] == "ALL"
        ].reset_index(drop=True)
    )

    print(
        f"Prefix-duplicate ratio undefined because unique_expanded_nodes=0: "
        f"{prefix_undefined} rows"
    )
    print(
        f"Candidate-duplicate ratio undefined despite candidate_branches>0: "
        f"{candidate_undefined} rows"
    )

    print("\n[Plan-level distributions]")
    display(plan_summary)

    print("\n[Question-level work distributions]")
    display(question_summary)

    print("\n[Reachability]")
    display(reachability)

    return {
        "distribution": dist,
        "rho": rho,
        "branching": branching,
        "duplicate": duplicate,
        "plan_raw": plan_df,
        "question_raw": question_df,
        "plan_summary": plan_summary,
        "question_summary": question_summary,
        "opportunity": opportunity,
        "downstream": downstream,
        "reachability": reachability,
    }


# =========================================================
# Run RQ1 development descriptive analysis
# =========================================================
rq1_webqsp_summary = analyze_rq1_dev(
    df_rq1_webqsp_dev,
    "webqsp"
)

rq1_cwq_summary = analyze_rq1_dev(
    df_rq1_cwq_dev,
    "cwq"
)


# =========================================================
# Save aggregated development summaries separately
# =========================================================
def save_rq1_summaries(summary_dict, dataset_label):
    for name, table in summary_dict.items():
        table.to_csv(
            os.path.join(
                RQ1_DEV_DIR,
                f"rq1_{dataset_label}_dev_{name}.csv"
            ),
            index=False
        )

save_rq1_summaries(rq1_webqsp_summary, "webqsp")
save_rq1_summaries(rq1_cwq_summary, "cwq")

print("\nRQ1 development descriptive summaries saved.")


WEBQSP -- DEVELOPMENT RQ1 CHARACTERIZATION

[Primary search-space distributions: all active frontiers]


,dataset,scope,hop,metric,n,mean,median,p75,p90,p95,max
0,webqsp,all_active,ALL,active_prefixes,971,2.512873,1.0,1.0,4.0,9.5,56
1,webqsp,all_active,ALL,unique_expanded_nodes,971,2.512873,1.0,1.0,4.0,9.5,56
2,webqsp,all_active,ALL,edges_examined,971,351.726056,135.0,402.0,1475.0,1686.0,1969
3,webqsp,all_active,ALL,candidate_branches,971,8.221421,1.0,5.0,18.0,35.0,533
4,webqsp,all_active,ALL,unique_frontier_nodes,971,6.423275,1.0,5.0,17.0,33.0,145



[Primary search-space distributions: intermediate active frontiers]


,dataset,scope,hop,metric,n,mean,median,p75,p90,p95,max
0,webqsp,intermediate_active,ALL,active_prefixes,312,1.048077,1.0,1.00,1.0,1.0,2
1,webqsp,intermediate_active,ALL,unique_expanded_nodes,312,1.048077,1.0,1.00,1.0,1.0,2
2,webqsp,intermediate_active,ALL,edges_examined,312,316.525641,161.0,354.75,569.2,1641.0,1915
3,webqsp,intermediate_active,ALL,candidate_branches,312,5.471154,1.0,5.00,16.0,27.7,56
4,webqsp,intermediate_active,ALL,unique_frontier_nodes,312,5.471154,1.0,5.00,16.0,27.7,56



[Hop-wise rho_h]


...


[Question-level work distributions]


,dataset,metric,n,mean,median,p75,p90,p95,max
0,cwq,active_prefixes_total,3519,16.436772,6.0,14.0,36.2,53.0,1403
1,cwq,edges_examined_total,3519,1493.967604,744.0,1793.0,4858.6,5457.0,52146
2,cwq,candidate_branches_total,3519,70.236147,15.0,44.0,112.2,220.3,21666
3,cwq,retrieved_paths_total,3519,58.421427,9.0,30.0,85.0,175.0,21377
4,cwq,peak_frontier_size_question,3519,6.531117,1.0,5.0,16.0,30.0,603



[Reachability]


,dataset,plans,reachable_plans,plan_reachability_pct,questions,reachable_questions,question_reachability_pct
0,cwq,10529,3971,37.714883,3519,2425,68.911623



RQ1 development descriptive summaries saved.


## Suffix-reachability DP (oracle machinery), brute-force validated

In [75]:
# ## 2.7 Suffix-reachability DP -- validation for oracle diagnostics + future training labels
# Gold answers are used OFFLINE only.
# On validation: used only for oracle diagnostics / decision-gate analysis.
# Later on training data: the same reachability machinery may be used to construct AFP supervision labels.
# Gold answers are NEVER available to AFP at inference time.

import random

def suffix_reachable_dp(graph, rule, gold_answers):
    """
    reachable[h][node] = True iff `node` can reach a gold answer
    by following exactly rule[h:].

    h ranges from 0..L.
    h == L is the base case: the current node itself must be a gold answer.
    """
    L = len(rule)
    reachable = [dict() for _ in range(L + 1)]

    for node in graph.nodes():
        reachable[L][node] = node in gold_answers

    for h in range(L - 1, -1, -1):
        rel = rule[h]

        for node in graph.nodes():
            reachable[h][node] = any(
                graph[node][neighbor]["relation"] == rel
                and reachable[h + 1].get(neighbor, False)
                for neighbor in graph.neighbors(node)
            )

    return reachable


def brute_force_suffix_reachable(graph, node, rule_suffix, gold_answers):
    if len(rule_suffix) == 0:
        return node in gold_answers

    if node not in graph:
        return False

    required_rel = rule_suffix[0]

    for neighbor in graph.neighbors(node):
        if graph[node][neighbor]["relation"] != required_rel:
            continue

        if brute_force_suffix_reachable(
            graph,
            neighbor,
            rule_suffix[1:],
            gold_answers
        ):
            return True

    return False


def validate_suffix_dp(
    planning_recs,
    dataset_label,
    n_questions=30,
    n_nodes=20,
    seed=0
):
    rng = random.Random(seed)

    sample_size = min(n_questions, len(planning_recs))
    validation_sample = rng.sample(planning_recs, sample_size)

    dp_checks = 0
    dp_mismatches = 0

    for rec in validation_sample:
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        graph_nodes = list(graph.nodes())

        if len(graph_nodes) <= n_nodes:
            nodes_to_check = graph_nodes
        else:
            nodes_to_check = rng.sample(graph_nodes, n_nodes)

        for rule in rec["predicted_paths"]:
            if len(rule) == 0:
                continue

            reachable = suffix_reachable_dp(
                graph,
                rule,
                gold_answers
            )

            for h in range(len(rule) + 1):
                suffix = rule[h:]

                for node in nodes_to_check:
                    dp_checks += 1

                    dp_value = reachable[h].get(node, False)

                    brute_value = brute_force_suffix_reachable(
                        graph,
                        node,
                        suffix,
                        gold_answers
                    )

                    if dp_value != brute_value:
                        dp_mismatches += 1

    print(
        f"[{dataset_label}] DP vs brute-force: "
        f"{dp_checks} (node, hop) states checked | "
        f"mismatches: {dp_mismatches}"
    )

    assert dp_mismatches == 0, (
        f"{dataset_label}: suffix-reachability DP diverges "
        f"from brute force -- fix before oracle diagnostics."
    )

    return dp_checks, dp_mismatches


# =========================================================
# Validate on BOTH development datasets
# =========================================================
webqsp_dp_checks, webqsp_dp_mismatches = validate_suffix_dp(
    webqsp_val_planning,
    "WebQSP validation",
    n_questions=30,
    n_nodes=20,
    seed=0
)

cwq_dp_checks, cwq_dp_mismatches = validate_suffix_dp(
    cwq_val_planning,
    "CWQ validation",
    n_questions=30,
    n_nodes=20,
    seed=1
)

assert webqsp_dp_mismatches == 0
assert cwq_dp_mismatches == 0

print("\n=== SUFFIX-REACHABILITY VALIDATION: PASSED ===")
print(f"WebQSP: {webqsp_dp_checks} states, 0 mismatches")
print(f"CWQ:    {cwq_dp_checks} states, 0 mismatches")
print("DP validated against brute force. Safe to proceed to oracle headroom.")

[WebQSP validation] DP vs brute-force: 4420 (node, hop) states checked | mismatches: 0
[CWQ validation] DP vs brute-force: 4999 (node, hop) states checked | mismatches: 0

=== SUFFIX-REACHABILITY VALIDATION: PASSED ===
WebQSP: 4420 states, 0 mismatches
CWQ:    4999 states, 0 mismatches
DP validated against brute force. Safe to proceed to oracle headroom.


## Validation oracle headroom and development decision gate

In [76]:
# ## 2.8 Oracle headroom on validation + decision gate
# ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD.
# Separate read-only re-traversal using validation gold answers offline only.
# Does not modify RoG retrieval, RQ1 profiling, AFP training, or inference-time decisions.
#
# IMPORTANT:
# - Prefix productivity uses reachable_dp[h][current_node].
# - Candidate-branch productivity uses reachable_dp[h+1][neighbor].
# - Only intermediate-hop doomed candidates are oracle-prunable under final-hop protection.
# - Current-hop edge examination is already incurred before AFP acts.
# - Therefore avoidable edge work is downstream work from doomed prefixes at h > 0.

from collections import deque
import numpy as np
import pandas as pd
import os

def oracle_headroom_for_plan(graph, start_node, rule, reachable_dp):
    L = len(rule)

    edges_prod_prefix = [0] * L
    edges_doom_prefix = [0] * L

    branch_prod = [0] * L
    branch_doom = [0] * L

    branch_prunable = [0] * L
    branch_doom_final_protected = [0] * L
    avoidable_downstream_edges = [0] * L

    queue = deque([(start_node, [])])

    while queue:
        current_node, current_path = queue.popleft()
        h = len(current_path)

        if h >= L:
            continue

        if current_node not in graph:
            continue

        # Is this CURRENT PREFIX capable of reaching gold through rule[h:]?
        prefix_is_productive = reachable_dp[h].get(current_node, False)

        for neighbor in graph.neighbors(current_node):
            # Every adjacency entry is examined before relation filtering.
            if prefix_is_productive:
                edges_prod_prefix[h] += 1
            else:
                edges_doom_prefix[h] += 1

                # If h > 0, this doomed prefix was generated at the previous
                # intermediate hop and could theoretically have been removed
                # by a perfect oracle before this expansion occurred.
                if h > 0:
                    avoidable_downstream_edges[h] += 1

            rel = graph[current_node][neighbor]["relation"]

            if rel != rule[h]:
                continue

            # Candidate branch:
            # after taking rule[h], candidate endpoint is evaluated using
            # the remaining suffix rule[h+1:].
            branch_is_productive = reachable_dp[h + 1].get(neighbor, False)

            if branch_is_productive:
                branch_prod[h] += 1
            else:
                branch_doom[h] += 1

                if h < L - 1:
                    # AFP can prune only intermediate candidates.
                    branch_prunable[h] += 1
                else:
                    # Final-hop candidates are protected in the main AFP policy.
                    branch_doom_final_protected[h] += 1

            queue.append(
                (neighbor, current_path + [(current_node, rel, neighbor)])
            )

    return {
        "edges_from_productive_prefix": edges_prod_prefix,
        "edges_from_doomed_prefix": edges_doom_prefix,
        "branches_productive": branch_prod,
        "branches_doomed": branch_doom,
        "oracle_prunable_branches": branch_prunable,
        "doomed_final_branches_protected": branch_doom_final_protected,
        "oracle_avoidable_downstream_edges": avoidable_downstream_edges,
    }


def compute_oracle_headroom(planning_recs, dataset_label):
    rows = []

    for rec in tqdm(
        planning_recs,
        desc=f"Oracle headroom ({dataset_label})"
    ):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])

        for plan_rank, rule in enumerate(rec["predicted_paths"]):
            L = len(rule)

            if L == 0:
                continue

            reachable_dp = suffix_reachable_dp(
                graph,
                rule,
                gold_answers
            )

            plan_gold_reachable = any(
                reachable_dp[0].get(entity, False)
                for entity in rec["q_entity"]
            )

            for entity in rec["q_entity"]:
                hd = oracle_headroom_for_plan(
                    graph,
                    entity,
                    rule,
                    reachable_dp
                )

                for h in range(L):
                    rows.append({
                        "dataset": dataset_label,
                        "question_id": rec["id"],
                        "plan_id": f"{rec['id']}_p{plan_rank}",
                        "plan_rank": plan_rank,
                        "plan_length": L,
                        "topic_entity": entity,
                        "hop": h,
                        "required_relation": rule[h],
                        "is_intermediate_hop": h < L - 1,
                        "plan_gold_reachable": plan_gold_reachable,

                        "edges_from_productive_prefix":
                            hd["edges_from_productive_prefix"][h],

                        "edges_from_doomed_prefix":
                            hd["edges_from_doomed_prefix"][h],

                        "branches_productive":
                            hd["branches_productive"][h],

                        "branches_doomed":
                            hd["branches_doomed"][h],

                        "oracle_prunable_branches":
                            hd["oracle_prunable_branches"][h],

                        "doomed_final_branches_protected":
                            hd["doomed_final_branches_protected"][h],

                        "oracle_avoidable_downstream_edges":
                            hd["oracle_avoidable_downstream_edges"][h],
                    })

    return pd.DataFrame(rows)


# =========================================================
# Run oracle diagnostic on frozen validation plans
# =========================================================
df_oracle_webqsp = compute_oracle_headroom(
    webqsp_val_planning,
    "webqsp"
)

df_oracle_cwq = compute_oracle_headroom(
    cwq_val_planning,
    "cwq"
)


# =========================================================
# Save raw oracle logs
# =========================================================
df_oracle_webqsp.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev.csv"
    ),
    index=False
)

df_oracle_cwq.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev.csv"
    ),
    index=False
)


# =========================================================
# Consistency check:
# oracle re-traversal must reproduce Cell 141 edge/branch totals
# =========================================================
def check_oracle_vs_rq1(df_oracle, df_rq1, label):
    oracle_edges = (
        df_oracle["edges_from_productive_prefix"].sum()
        + df_oracle["edges_from_doomed_prefix"].sum()
    )

    oracle_branches = (
        df_oracle["branches_productive"].sum()
        + df_oracle["branches_doomed"].sum()
    )

    rq1_edges = df_rq1["edges_examined"].sum()
    rq1_branches = df_rq1["candidate_branches"].sum()

    print(f"\n[{label}] ORACLE vs RQ1 CONSISTENCY")
    print(
        f"Edges: oracle={oracle_edges} | "
        f"RQ1={rq1_edges} | "
        f"match={oracle_edges == rq1_edges}"
    )
    print(
        f"Branches: oracle={oracle_branches} | "
        f"RQ1={rq1_branches} | "
        f"match={oracle_branches == rq1_branches}"
    )

    assert oracle_edges == rq1_edges, (
        f"{label}: oracle edge total does not match RQ1 profiler."
    )

    assert oracle_branches == rq1_branches, (
        f"{label}: oracle branch total does not match RQ1 profiler."
    )


check_oracle_vs_rq1(
    df_oracle_webqsp,
    df_rq1_webqsp_dev,
    "WebQSP validation"
)

check_oracle_vs_rq1(
    df_oracle_cwq,
    df_rq1_cwq_dev,
    "CWQ validation"
)


# =========================================================
# Oracle headroom summary
# =========================================================
def safe_pct(num, den):
    return 100 * num / den if den > 0 else np.nan


def summarize_headroom(df, label):
    total_edges = (
        df["edges_from_productive_prefix"].sum()
        + df["edges_from_doomed_prefix"].sum()
    )

    doomed_prefix_edges = df["edges_from_doomed_prefix"].sum()

    # Edge work theoretically removable because the doomed prefix
    # could have been pruned at the preceding intermediate hop.
    avoidable_edges = df[
        "oracle_avoidable_downstream_edges"
    ].sum()

    total_branches = (
        df["branches_productive"].sum()
        + df["branches_doomed"].sum()
    )

    # Only intermediate candidates are eligible for AFP pruning.
    intermediate_df = df[df["is_intermediate_hop"]]

    intermediate_branches = (
        intermediate_df["branches_productive"].sum()
        + intermediate_df["branches_doomed"].sum()
    )

    prunable_branches = (
        intermediate_df["oracle_prunable_branches"].sum()
    )

    final_protected_doomed = (
        df["doomed_final_branches_protected"].sum()
    )

    print(
        f"\n[{label}] ORACLE DIAGNOSTIC -- "
        f"NOT AN INFERENCE METHOD"
    )

    print(
        f"[{label}] Total examined edges: {total_edges}"
    )

    print(
        f"[{label}] Edges from doomed prefixes: "
        f"{doomed_prefix_edges} "
        f"({safe_pct(doomed_prefix_edges, total_edges):.2f}%)"
    )

    print(
        f"[{label}] Oracle-avoidable downstream edges: "
        f"{avoidable_edges} "
        f"({safe_pct(avoidable_edges, total_edges):.2f}% "
        f"of all examined edges)"
    )

    print(
        f"[{label}] Total relation-valid candidate branches: "
        f"{total_branches}"
    )

    print(
        f"[{label}] Intermediate candidate branches: "
        f"{intermediate_branches}"
    )

    print(
        f"[{label}] Oracle-prunable intermediate branches: "
        f"{prunable_branches} "
        f"({safe_pct(prunable_branches, intermediate_branches):.2f}%)"
    )

    print(
        f"[{label}] Doomed final-hop branches protected by policy: "
        f"{final_protected_doomed}"
    )

    # Conservative view restricted to relation plans that actually
    # contain at least one gold-reaching path.
    reachable_plan_df = df[df["plan_gold_reachable"]]
    reachable_intermediate_df = reachable_plan_df[
        reachable_plan_df["is_intermediate_hop"]
    ]

    reachable_intermediate_branches = (
        reachable_intermediate_df["branches_productive"].sum()
        + reachable_intermediate_df["branches_doomed"].sum()
    )

    reachable_prunable = (
        reachable_intermediate_df["oracle_prunable_branches"].sum()
    )

    reachable_avoidable_edges = (
        reachable_plan_df[
            "oracle_avoidable_downstream_edges"
        ].sum()
    )

    reachable_total_edges = (
        reachable_plan_df["edges_from_productive_prefix"].sum()
        + reachable_plan_df["edges_from_doomed_prefix"].sum()
    )

    print(
        f"[{label}] Gold-reachable plans only -- "
        f"oracle-prunable intermediate branches: "
        f"{reachable_prunable}/"
        f"{reachable_intermediate_branches} "
        f"({safe_pct(reachable_prunable, reachable_intermediate_branches):.2f}%)"
    )

    print(
        f"[{label}] Gold-reachable plans only -- "
        f"oracle-avoidable downstream edges: "
        f"{reachable_avoidable_edges}/"
        f"{reachable_total_edges} "
        f"({safe_pct(reachable_avoidable_edges, reachable_total_edges):.2f}%)"
    )

    # Per-hop summary
    per_hop = (
        df.groupby("hop", as_index=False)
        .agg(
            edges_from_productive_prefix=(
                "edges_from_productive_prefix", "sum"
            ),
            edges_from_doomed_prefix=(
                "edges_from_doomed_prefix", "sum"
            ),
            oracle_avoidable_downstream_edges=(
                "oracle_avoidable_downstream_edges", "sum"
            ),
            branches_productive=(
                "branches_productive", "sum"
            ),
            branches_doomed=(
                "branches_doomed", "sum"
            ),
            oracle_prunable_branches=(
                "oracle_prunable_branches", "sum"
            ),
            doomed_final_branches_protected=(
                "doomed_final_branches_protected", "sum"
            ),
        )
    )

    per_hop["edges_total"] = (
        per_hop["edges_from_productive_prefix"]
        + per_hop["edges_from_doomed_prefix"]
    )

    per_hop["avoidable_edge_pct"] = np.where(
        per_hop["edges_total"] > 0,
        100
        * per_hop["oracle_avoidable_downstream_edges"]
        / per_hop["edges_total"],
        np.nan
    )

    per_hop["candidate_branches_total"] = (
        per_hop["branches_productive"]
        + per_hop["branches_doomed"]
    )

    print(f"\n[{label}] Per-hop oracle headroom")
    display(per_hop)

    summary = pd.DataFrame([{
        "dataset": label,
        "total_edges": total_edges,
        "doomed_prefix_edges": doomed_prefix_edges,
        "oracle_avoidable_downstream_edges": avoidable_edges,
        "oracle_avoidable_edge_pct":
            safe_pct(avoidable_edges, total_edges),

        "total_candidate_branches": total_branches,
        "intermediate_candidate_branches": intermediate_branches,
        "oracle_prunable_intermediate_branches": prunable_branches,
        "oracle_prunable_branch_pct":
            safe_pct(prunable_branches, intermediate_branches),

        "doomed_final_branches_protected":
            final_protected_doomed,

        "reachable_plan_intermediate_branches":
            reachable_intermediate_branches,

        "reachable_plan_oracle_prunable_branches":
            reachable_prunable,

        "reachable_plan_prunable_branch_pct":
            safe_pct(
                reachable_prunable,
                reachable_intermediate_branches
            ),

        "reachable_plan_total_edges":
            reachable_total_edges,

        "reachable_plan_oracle_avoidable_edges":
            reachable_avoidable_edges,

        "reachable_plan_avoidable_edge_pct":
            safe_pct(
                reachable_avoidable_edges,
                reachable_total_edges
            ),
    }])

    return summary, per_hop


webqsp_headroom_summary, webqsp_headroom_hop = (
    summarize_headroom(
        df_oracle_webqsp,
        "WebQSP validation"
    )
)

cwq_headroom_summary, cwq_headroom_hop = (
    summarize_headroom(
        df_oracle_cwq,
        "CWQ validation"
    )
)


# =========================================================
# Save oracle summaries
# =========================================================
webqsp_headroom_summary.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev_summary.csv"
    ),
    index=False
)

cwq_headroom_summary.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev_summary.csv"
    ),
    index=False
)

webqsp_headroom_hop.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev_hop.csv"
    ),
    index=False
)

cwq_headroom_hop.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev_hop.csv"
    ),
    index=False
)


# =========================================================
# Development decision gate -- evidence-based answers
# =========================================================

def get_summary_value(df, col):
    return float(df.iloc[0][col])

# RQ1 decision-opportunity evidence
w_opp = rq1_webqsp_summary["opportunity"].iloc[0]
c_opp = rq1_cwq_summary["opportunity"].iloc[0]

w_decision_hop_pct = float(w_opp["decision_hop_pct"])
c_decision_hop_pct = float(c_opp["decision_hop_pct"])

w_question_decision_pct = float(w_opp["questions_with_decision_pct"])
c_question_decision_pct = float(c_opp["questions_with_decision_pct"])

w_eligible_decision_pct = float(w_opp["decision_pct_among_eligible_questions"])
c_eligible_decision_pct = float(c_opp["decision_pct_among_eligible_questions"])

# Oracle evidence
w_avoidable_edge_pct = get_summary_value(
    webqsp_headroom_summary, "oracle_avoidable_edge_pct"
)
c_avoidable_edge_pct = get_summary_value(
    cwq_headroom_summary, "oracle_avoidable_edge_pct"
)

w_prunable_branch_pct = get_summary_value(
    webqsp_headroom_summary, "oracle_prunable_branch_pct"
)
c_prunable_branch_pct = get_summary_value(
    cwq_headroom_summary, "oracle_prunable_branch_pct"
)

w_reachable_prunable_pct = get_summary_value(
    webqsp_headroom_summary, "reachable_plan_prunable_branch_pct"
)
c_reachable_prunable_pct = get_summary_value(
    cwq_headroom_summary, "reachable_plan_prunable_branch_pct"
)

w_reachable_avoidable_pct = get_summary_value(
    webqsp_headroom_summary, "reachable_plan_avoidable_edge_pct"
)
c_reachable_avoidable_pct = get_summary_value(
    cwq_headroom_summary, "reachable_plan_avoidable_edge_pct"
)

w_total_edges = int(
    webqsp_headroom_summary.iloc[0]["total_edges"]
)
c_total_edges = int(
    cwq_headroom_summary.iloc[0]["total_edges"]
)

w_total_branches = int(
    webqsp_headroom_summary.iloc[0]["total_candidate_branches"]
)
c_total_branches = int(
    cwq_headroom_summary.iloc[0]["total_candidate_branches"]
)

w_edges_per_branch = w_total_edges / w_total_branches
c_edges_per_branch = c_total_edges / c_total_branches

# Duplicate-pressure evidence from Cell 143
w_dup = rq1_webqsp_summary["duplicate"]
c_dup = rq1_cwq_summary["duplicate"]

w_prefix_dup_pct = float(
    w_dup[
        (w_dup["hop"] == "ALL") &
        (w_dup["metric"] == "prefix_to_unique_expanded_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

c_prefix_dup_pct = float(
    c_dup[
        (c_dup["hop"] == "ALL") &
        (c_dup["metric"] == "prefix_to_unique_expanded_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

w_frontier_dup_pct = float(
    w_dup[
        (w_dup["hop"] == "ALL") &
        (w_dup["metric"] == "candidate_to_unique_frontier_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

c_frontier_dup_pct = float(
    c_dup[
        (c_dup["hop"] == "ALL") &
        (c_dup["metric"] == "candidate_to_unique_frontier_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

# Hop contributing the largest total oracle-avoidable edge work
w_max_row = webqsp_headroom_hop.loc[
    webqsp_headroom_hop["oracle_avoidable_downstream_edges"].idxmax()
]

c_max_row = cwq_headroom_hop.loc[
    cwq_headroom_hop["oracle_avoidable_downstream_edges"].idxmax()
]

w_max_hop = int(w_max_row["hop"])
c_max_hop = int(c_max_row["hop"])

w_max_hop_edges = int(
    w_max_row["oracle_avoidable_downstream_edges"]
)
c_max_hop_edges = int(
    c_max_row["oracle_avoidable_downstream_edges"]
)


print("\n" + "=" * 80)
print("DEVELOPMENT DECISION GATE -- VALIDATION EVIDENCE")
print("=" * 80)

print("\nQ1. Is residual relation-valid branching frequent enough to justify selective pruning?")
print(
    f"ANSWER: YES. Actual intermediate decision opportunities occur in "
    f"{w_decision_hop_pct:.2f}% of WebQSP and "
    f"{c_decision_hop_pct:.2f}% of CWQ intermediate active hops. "
    f"At question level, {w_question_decision_pct:.2f}% of WebQSP and "
    f"{c_question_decision_pct:.2f}% of CWQ questions contain at least one "
    f"real AFP decision opportunity."
)

print("\nQ2. At which hops and in which dataset is downstream pruning opportunity largest?")
print(
    f"ANSWER: CWQ provides substantially larger downstream structural headroom. "
    f"Oracle-avoidable downstream edges represent {c_avoidable_edge_pct:.2f}% "
    f"of all CWQ examined edges versus {w_avoidable_edge_pct:.2f}% for WebQSP. "
    f"By total oracle-avoidable edge work, WebQSP hop {w_max_hop} contributes "
    f"{w_max_hop_edges:,} edges, while CWQ hop {c_max_hop} contributes "
    f"{c_max_hop_edges:,} edges."
)

print("\nQ3. Is candidate-branch count or edge examination the more appropriate graph-search cost?")
print(
    f"ANSWER: EXAMINED EDGES should remain the primary structural-cost metric. "
    f"WebQSP performs {w_total_edges:,} adjacency examinations for "
    f"{w_total_branches:,} relation-valid branches "
    f"(~{w_edges_per_branch:.2f} examined edges per branch), while CWQ performs "
    f"{c_total_edges:,} examinations for {c_total_branches:,} branches "
    f"(~{c_edges_per_branch:.2f} per branch). Candidate count alone therefore "
    f"does not capture the graph work required before relation matching."
)

print("\nQ4. Are duplicate path/frontier states the main explanation for residual branching?")
print(
    f"ANSWER: NO. Repeated-prefix expansion is rare "
    f"({w_prefix_dup_pct:.2f}% WebQSP; {c_prefix_dup_pct:.2f}% CWQ), and "
    f"candidate-endpoint duplication is also limited "
    f"({w_frontier_dup_pct:.2f}% WebQSP; {c_frontier_dup_pct:.2f}% CWQ). "
    f"Residual search growth is therefore mainly genuine relation-valid "
    f"branch multiplicity rather than simple duplication."
)

print("\nQ5. How do pruning-opportunity frequency and oracle headroom differ between datasets?")
print(
    f"ANSWER: WebQSP has more frequent intermediate decision opportunities "
    f"({w_decision_hop_pct:.2f}% vs {c_decision_hop_pct:.2f}% of active "
    f"intermediate hops), but CWQ has substantially greater downstream edge-saving "
    f"headroom ({c_avoidable_edge_pct:.2f}% vs {w_avoidable_edge_pct:.2f}%). "
    f"Thus CWQ is not necessarily more frequently branching, but its deeper "
    f"retrieval produces greater downstream cost when branching occurs."
)

print("\nQ6. Within gold-reachable plans, does substantial oracle-prunable branching remain?")
print(
    f"ANSWER: YES. Within gold-reachable plans, "
    f"{w_reachable_prunable_pct:.2f}% of WebQSP and "
    f"{c_reachable_prunable_pct:.2f}% of CWQ intermediate candidate branches "
    f"are oracle-doomed. Their corresponding oracle-avoidable examined-edge "
    f"headroom is {w_reachable_avoidable_pct:.2f}% and "
    f"{c_reachable_avoidable_pct:.2f}%, respectively. "
    f"This confirms that pruning opportunity is not explained only by plans "
    f"that already fail to reach a gold answer."
)

print("\nQ7. Is the structural headroom sufficient to justify AFP development?")
print(
    f"ANSWER: YES. Residual branching is non-trivial, real branch-selection "
    f"opportunities exist in both datasets, and oracle analysis identifies "
    f"{w_prunable_branch_pct:.2f}% of WebQSP and "
    f"{c_prunable_branch_pct:.2f}% of CWQ intermediate candidate branches as "
    f"theoretically removable. The evidence therefore justifies proceeding "
    f"to AFP development and controlled RQ2 evaluation."
)

print("\n" + "=" * 80)
print("=== DEVELOPMENT DECISION GATE: PASSED ===")
print("=" * 80)

print(
    "Conclusion: Development RQ1 establishes a meaningful residual search-space "
    "problem and sufficient oracle pruning headroom to proceed to AFP/RQ2."
)

print(
    "Caution: These results establish structural opportunity only. Actual search "
    "reduction, answer preservation, runtime benefit, and pruning overhead remain "
    "unverified until RQ2/RQ3 experiments are completed."
)

print(
    "Protocol: All conclusions above use frozen validation evidence only; "
    "no frozen-test results are used for AFP development decisions."
)

Oracle headroom (webqsp):   0%|          | 0/246 [00:00<?, ?it/s]

Oracle headroom (cwq):   0%|          | 0/3519 [00:00<?, ?it/s]


[WebQSP validation] ORACLE vs RQ1 CONSISTENCY
Edges: oracle=341526 | RQ1=341526 | match=True
Branches: oracle=7983 | RQ1=7983 | match=True

[CWQ validation] ORACLE vs RQ1 CONSISTENCY
Edges: oracle=5257272 | RQ1=5257272 | match=True
Branches: oracle=247161 | RQ1=247161 | match=True

[WebQSP validation] ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD
[WebQSP validation] Total examined edges: 341526
[WebQSP validation] Edges from doomed prefixes: 159039 (46.57%)
[WebQSP validation] Oracle-avoidable downstream edges: 11717 (3.43% of all examined edges)
[WebQSP validation] Total relation-valid candidate branches: 7983
[WebQSP validation] Intermediate candidate branches: 1707
[WebQSP validation] Oracle-prunable intermediate branches: 1340 (78.50%)
[WebQSP validation] Doomed final-hop branches protected by policy: 4882
[WebQSP validation] Gold-reachable plans only -- oracle-prunable intermediate branches: 647/1014 (63.81%)
[WebQSP validation] Gold-reachable plans only -- oracle-avoidable downst

,hop,edges_from_productive_prefix,edges_from_doomed_prefix,oracle_avoidable_downstream_edges,branches_productive,branches_doomed,oracle_prunable_branches,doomed_final_branches_protected,edges_total,avoidable_edge_pct,candidate_branches_total
0,0,173107,147322,0,1144,2554,1340,1214,320429,0.000000,3698
1,1,9380,11717,11717,617,3668,0,3668,21097,55.538702,4285



[CWQ validation] ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD
[CWQ validation] Total examined edges: 5257272
[CWQ validation] Edges from doomed prefixes: 3197488 (60.82%)
[CWQ validation] Oracle-avoidable downstream edges: 1176294 (22.37% of all examined edges)
[CWQ validation] Total relation-valid candidate branches: 247161
[CWQ validation] Intermediate candidate branches: 41576
[CWQ validation] Oracle-prunable intermediate branches: 34431 (82.81%)
[CWQ validation] Doomed final-hop branches protected by policy: 195499
[CWQ validation] Gold-reachable plans only -- oracle-prunable intermediate branches: 14008/21153 (66.22%)
[CWQ validation] Gold-reachable plans only -- oracle-avoidable downstream edges: 196144/2728920 (7.19%)

[CWQ validation] Per-hop oracle headroom


,hop,edges_from_productive_prefix,edges_from_doomed_prefix,oracle_avoidable_downstream_edges,branches_productive,branches_doomed,oracle_prunable_branches,doomed_final_branches_protected,edges_total,avoidable_edge_pct,candidate_branches_total
0,0,1186111,2021194,0,6018,52867,28511,24356,3207305,0.000000,58885
1,1,666705,1036547,1036547,6489,81488,2679,78809,1703252,60.856937,87977
2,2,55475,45373,45373,1901,39543,3230,36313,100848,44.991472,41444
3,3,107617,94287,94287,1351,51999,11,51988,201904,46.698926,53350
4,4,43876,87,87,1472,4033,0,4033,43963,0.197894,5505



DEVELOPMENT DECISION GATE -- VALIDATION EVIDENCE

Q1. Is residual relation-valid branching frequent enough to justify selective pruning?
ANSWER: YES. Actual intermediate decision opportunities occur in 47.12% of WebQSP and 33.77% of CWQ intermediate active hops. At question level, 31.30% of WebQSP and 39.93% of CWQ questions contain at least one real AFP decision opportunity.

Q2. At which hops and in which dataset is downstream pruning opportunity largest?
ANSWER: CWQ provides substantially larger downstream structural headroom. Oracle-avoidable downstream edges represent 22.37% of all CWQ examined edges versus 3.43% for WebQSP. By total oracle-avoidable edge work, WebQSP hop 1 contributes 11,717 edges, while CWQ hop 1 contributes 1,036,547 edges.

Q3. Is candidate-branch count or edge examination the more appropriate graph-search cost?
ANSWER: EXAMINED EDGES should remain the primary structural-cost metric. WebQSP performs 341,526 adjacency examinations for 7,983 relation-valid bran

## Search-space characterization of unpruned RoG. Values summarize the observed hop-level distributions.

In [77]:
# ## RQ1 supervisor table -- development validation results
# Table reports distributions over ALL ACTIVE question x plan x hop frontiers.
# These are development/validation results, NOT final frozen-test RQ1 results.

import os
import pandas as pd
from IPython.display import display

METRICS = {
    "active_prefixes": "Active Path Prefixes",
    "unique_expanded_nodes": "Unique Expanded Nodes",
    "edges_examined": "Edges Examined",
    "candidate_branches": "Candidate Branches",
    "unique_frontier_nodes": "Unique Frontier Nodes",
}

def make_rq1_supervisor_table(df, dataset_name):
    active = df[df["active_prefixes"] > 0].copy()
    rows = []

    for col, label in METRICS.items():
        s = active[col].dropna()

        rows.append({
            "Dataset": dataset_name,
            "Metric": label,
            "Median": s.median(),
            "P90": s.quantile(0.90),
            "P95": s.quantile(0.95),
            "Maximum": s.max(),
        })

    return pd.DataFrame(rows)

table_webqsp = make_rq1_supervisor_table(
    df_rq1_webqsp_dev,
    "WebQSP"
)

table_cwq = make_rq1_supervisor_table(
    df_rq1_cwq_dev,
    "CWQ"
)

rq1_supervisor_table = pd.concat(
    [table_webqsp, table_cwq],
    ignore_index=True
)

# Clean display formatting
for col in ["Median", "P90", "P95", "Maximum"]:
    rq1_supervisor_table[col] = rq1_supervisor_table[col].round(1)

display(rq1_supervisor_table)

# Save for paper drafting
rq1_supervisor_table.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "rq1_development_search_space_summary.csv"
    ),
    index=False
)

print("\nDevelopment RQ1 search-space table saved.")
print("NOTE: Values are from frozen VALIDATION plans, not final test reporting.")

,Dataset,Metric,Median,P90,P95,Maximum
0,WebQSP,Active Path Prefixes,1.0,4.0,9.5,56
1,WebQSP,Unique Expanded Nodes,1.0,4.0,9.5,56
2,WebQSP,Edges Examined,135.0,1475.0,1686.0,1969
3,WebQSP,Candidate Branches,1.0,18.0,35.0,533
4,WebQSP,Unique Frontier Nodes,1.0,17.0,33.0,145
5,CWQ,Active Path Prefixes,1.0,5.0,13.0,603
6,CWQ,Unique Expanded Nodes,1.0,5.0,12.0,210
7,CWQ,Edges Examined,110.0,1111.7,1691.0,23290
8,CWQ,Candidate Branches,1.0,21.0,37.0,21372
9,CWQ,Unique Frontier Nodes,1.0,19.0,33.0,333



Development RQ1 search-space table saved.
NOTE: Values are from frozen VALIDATION plans, not final test reporting.


# RQ2

## RQ2 development setup, reproducibility, leakage guards, and configuration status

In [78]:
# RQ2: Can adaptive frontier pruning reduce unnecessary graph exploration more
# effectively than unpruned RoG and simple fixed-pruning strategies?
#
# IMPORTANT:
# - This cell freezes only methodological/protocol invariants.
# - Model architecture and hyperparameters are DEVELOPMENT choices, not final values.
# - TRAIN is used for branch supervision.
# - VALIDATION is used for model/configuration selection.
# - TEST must not influence AFP development.

import os
import sys
import json
import random
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import networkx as nx
import datasets

from datasets import load_dataset

# =========================================================
# RQ2 stage and output directories
# =========================================================
RQ2_STAGE = "RQ2_DEVELOPMENT"
RQ2_VERSION = "v1"

RQ2_DIR = f"/kaggle/working/step3_rq2_dev_{RQ2_VERSION}"
RQ2_PLAN_DIR = os.path.join(RQ2_DIR, "01_train_plans")
RQ2_LABEL_DIR = os.path.join(RQ2_DIR, "02_branch_labels")
RQ2_FEATURE_DIR = os.path.join(RQ2_DIR, "03_features")
RQ2_MODEL_DIR = os.path.join(RQ2_DIR, "04_models")
RQ2_TUNING_DIR = os.path.join(RQ2_DIR, "05_validation_tuning")
RQ2_COMPARE_DIR = os.path.join(RQ2_DIR, "06_rq2_comparison")
RQ2_ABLATION_DIR = os.path.join(RQ2_DIR, "07_ablation")
RQ2_MANIFEST_DIR = os.path.join(RQ2_DIR, "manifests")

for d in [
    RQ2_DIR,
    RQ2_PLAN_DIR,
    RQ2_LABEL_DIR,
    RQ2_FEATURE_DIR,
    RQ2_MODEL_DIR,
    RQ2_TUNING_DIR,
    RQ2_COMPARE_DIR,
    RQ2_ABLATION_DIR,
    RQ2_MANIFEST_DIR,
]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# Leakage-safe split policy
# =========================================================
RQ2_TRAIN_SPLIT = "train"
RQ2_VALIDATION_SPLIT = "validation"
RQ2_TEST_SPLIT = "test"

RQ2_ALLOWED_DEV_SPLITS = {
    RQ2_TRAIN_SPLIT,
    RQ2_VALIDATION_SPLIT,
}

def assert_rq2_dev_split(split_name):
    assert split_name in RQ2_ALLOWED_DEV_SPLITS, (
        f"RQ2 LEAKAGE GUARD: split='{split_name}' is forbidden during development. "
        f"Only TRAIN and VALIDATION may influence AFP."
    )

assert_rq2_dev_split("train")
assert_rq2_dev_split("validation")

# =========================================================
# Predeclared reproducibility seeds
# These can be frozen now because they should NOT be selected
# according to downstream performance.
# =========================================================
RQ2_SEEDS = [42, 43, 44]
RQ2_CANONICAL_SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(RQ2_CANONICAL_SEED)

# =========================================================
# Verify required frozen RoG components already exist
# =========================================================
required_globals = [
    "DATASET_NAME",
    "CWQ_DATASET_NAME",
    "MODEL_PATH",
    "N_BEAM",
    "INSTRUCTION",
    "build_graph",
    "suffix_reachable_dp",
    "bfs_with_rule_profiled_v2",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

assert len(missing_globals) == 0, (
    "Missing required RoG objects/functions: "
    + ", ".join(missing_globals)
)

RQ2_WEBQSP_DATASET = DATASET_NAME
RQ2_CWQ_DATASET = CWQ_DATASET_NAME
RQ2_PLANNER_MODEL = MODEL_PATH
RQ2_N_BEAM = N_BEAM

RQ2_INSTRUCTION_HASH = hashlib.sha256(
    INSTRUCTION.encode("utf-8")
).hexdigest()

# =========================================================
# Load development splits only
# =========================================================
if "webqsp_train" not in globals():
    webqsp_train = load_dataset(
        RQ2_WEBQSP_DATASET,
        split=RQ2_TRAIN_SPLIT
    )

if "cwq_train" not in globals():
    cwq_train = load_dataset(
        RQ2_CWQ_DATASET,
        split=RQ2_TRAIN_SPLIT
    )

if "webqsp_val" not in globals():
    webqsp_val = load_dataset(
        RQ2_WEBQSP_DATASET,
        split=RQ2_VALIDATION_SPLIT
    )

if "cwq_val" not in globals():
    cwq_val = load_dataset(
        RQ2_CWQ_DATASET,
        split=RQ2_VALIDATION_SPLIT
    )

# =========================================================
# METHOD INVARIANTS -- fixed now
# =========================================================
AFP_FINAL_HOP_PROTECTION = True
AFP_INTERMEDIATE_ONLY = True
AFP_LABEL_RULE = "suffix_gold_reachability"
AFP_EXCLUDE_ALL_NEGATIVE_GROUPS = True

AFP_FEATURE_GROUPS = [
    "semantic",
    "path_context",
    "structural",
    "search_progress",
]

AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES = False
AFP_USE_KGE_CORE = False

RQ2_PRIMARY_COST_METRIC = "edges_examined"
RQ2_PRIMARY_OUTCOME = "SSR"

RQ2_METHODS = [
    "RoG",
    "Fixed-Top-B",
    "Fixed-Threshold",
    "Random-B",
    "Adaptive-Budget-Random",
    "AFP",
]

RQ2_INVARIANTS = {
    "same_questions": True,
    "same_topic_entities": True,
    "same_relation_plans": True,
    "same_question_graph": True,
    "same_graph_construction": True,
    "same_relation_matching": True,
    "same_final_hop_policy": True,
    "same_candidate_generation": True,
    "same_retrieval_semantics_except_selection": True,
    "test_used_for_development": False,
}

# =========================================================
# DEVELOPMENT CONFIGURATION -- NOT FROZEN YET
# =========================================================

# Lightweight semantic encoder from current methodology.
# Keep as primary candidate for now, but NOT frozen.
AFP_SEMANTIC_ENCODER_CANDIDATE = (
    "sentence-transformers/all-MiniLM-L6-v2"
)
AFP_SEMANTIC_DIM = 384

# Scorer architecture candidates.
# 64 remains the manuscript's initial configuration,
# but we allow evidence-based adjustment.
AFP_HIDDEN_DIM_CANDIDATES = [32, 64, 128]

AFP_ACTIVATION_CANDIDATE = "ReLU"
AFP_LOSS_CANDIDATE = "BCEWithLogitsLoss"
AFP_OPTIMIZER_CANDIDATE = "AdamW"

# No final optimizer values yet
AFP_LR_SELECTED = None
AFP_WEIGHT_DECAY_SELECTED = None
AFP_BATCH_SIZE_SELECTED = None
AFP_EPOCHS_SELECTED = None
AFP_HIDDEN_DIM_SELECTED = None
AFP_SEMANTIC_ENCODER_SELECTED = None

# =========================================================
# Fixed-pruning validation grids
# Revised from the manuscript placeholder because RQ1 showed
# intermediate candidate sets well above B=8.
# =========================================================
TOP_B_GRID = [1, 2, 4, 8, 16, 32]

# Wider threshold search than the draft placeholder {0.3,0.5,0.7}.
# Final threshold will be selected on validation only.
THRESHOLD_GRID = [
    0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9
]

# Adaptive-policy grids should NOT be chosen yet.
# Their useful ranges depend on the trained scorer's validation
# logit/probability distribution.
AFP_T_GRID = None
AFP_GAMMA_MIN_GRID = None

AFP_T_SELECTED = None
AFP_GAMMA_MIN_SELECTED = None

# =========================================================
# RQ2 structural metrics
# =========================================================
RQ2_STRUCTURAL_METRICS = [
    "active_prefixes",
    "unique_expanded_nodes",
    "edges_examined",
    "candidate_branches",
    "unique_frontier_nodes",
    "peak_frontier",
    "retrieved_paths",
]

def compute_ssr(method_edges, rog_edges):
    assert rog_edges > 0, "RoG examined-edge denominator must be > 0."
    return 1.0 - (method_edges / rog_edges)

# =========================================================
# Hard methodological assertions
# =========================================================
assert AFP_FINAL_HOP_PROTECTION is True
assert AFP_INTERMEDIATE_ONLY is True
assert AFP_EXCLUDE_ALL_NEGATIVE_GROUPS is True
assert AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES is False
assert AFP_USE_KGE_CORE is False
assert RQ2_INVARIANTS["test_used_for_development"] is False
assert callable(suffix_reachable_dp)

# =========================================================
# Development manifest
# =========================================================
RQ2_CONFIG = {
    "stage": RQ2_STAGE,
    "version": RQ2_VERSION,

    "dataset": {
        "webqsp_name": RQ2_WEBQSP_DATASET,
        "cwq_name": RQ2_CWQ_DATASET,
        "webqsp_train_questions": len(webqsp_train),
        "webqsp_validation_questions": len(webqsp_val),
        "cwq_train_questions": len(cwq_train),
        "cwq_validation_questions": len(cwq_val),
        "test_allowed_for_development": False,
    },

    "frozen_rog": {
        "planner_model": RQ2_PLANNER_MODEL,
        "top_k_relation_plans": RQ2_N_BEAM,
        "instruction_sha256": RQ2_INSTRUCTION_HASH,
    },

    "method_invariants": {
        "final_hop_protection": AFP_FINAL_HOP_PROTECTION,
        "intermediate_only": AFP_INTERMEDIATE_ONLY,
        "label_rule": AFP_LABEL_RULE,
        "exclude_all_negative_groups":
            AFP_EXCLUDE_ALL_NEGATIVE_GROUPS,
        "feature_groups": AFP_FEATURE_GROUPS,
        "future_neighborhood_features":
            AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES,
        "kge_core": AFP_USE_KGE_CORE,
    },

    "development_candidates": {
        "semantic_encoder":
            AFP_SEMANTIC_ENCODER_CANDIDATE,
        "hidden_dims":
            AFP_HIDDEN_DIM_CANDIDATES,
        "activation":
            AFP_ACTIVATION_CANDIDATE,
        "loss":
            AFP_LOSS_CANDIDATE,
        "optimizer":
            AFP_OPTIMIZER_CANDIDATE,
        "top_b_grid":
            TOP_B_GRID,
        "threshold_grid":
            THRESHOLD_GRID,
        "temperature_grid":
            AFP_T_GRID,
        "gamma_min_grid":
            AFP_GAMMA_MIN_GRID,
    },

    "rq2": {
        "methods": RQ2_METHODS,
        "primary_cost_metric":
            RQ2_PRIMARY_COST_METRIC,
        "primary_outcome":
            RQ2_PRIMARY_OUTCOME,
        "structural_metrics":
            RQ2_STRUCTURAL_METRICS,
    },

    "reproducibility": {
        "seeds": RQ2_SEEDS,
        "canonical_seed":
            RQ2_CANONICAL_SEED,
        "python":
            sys.version,
        "platform":
            platform.platform(),
        "torch":
            torch.__version__,
        "numpy":
            np.__version__,
        "pandas":
            pd.__version__,
        "networkx":
            nx.__version__,
        "datasets":
            datasets.__version__,
        "cuda_available":
            torch.cuda.is_available(),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
    },

    "controlled_comparison_invariants":
        RQ2_INVARIANTS,
}

RQ2_CONFIG_PATH = os.path.join(
    RQ2_MANIFEST_DIR,
    "rq2_initial_development_config.json"
)

with open(RQ2_CONFIG_PATH, "w") as f:
    json.dump(
        RQ2_CONFIG,
        f,
        indent=2
    )

# =========================================================
# Setup report
# =========================================================
print("\n" + "=" * 80)
print("RQ2 DEVELOPMENT SETUP")
print("=" * 80)

print(f"Output directory:    {RQ2_DIR}")

print("\nDevelopment datasets:")
print(f"WebQSP train:        {len(webqsp_train)}")
print(f"WebQSP validation:   {len(webqsp_val)}")
print(f"CWQ train:           {len(cwq_train)}")
print(f"CWQ validation:      {len(cwq_val)}")

print("\nFrozen RoG:")
print(f"Planner model:       {RQ2_PLANNER_MODEL}")
print(f"Top-K plans:         {RQ2_N_BEAM}")
print(f"Instruction SHA256:  {RQ2_INSTRUCTION_HASH[:16]}...")

print("\nAFP methodological invariants:")
print(f"Final-hop protection:             {AFP_FINAL_HOP_PROTECTION}")
print(f"Intermediate pruning only:        {AFP_INTERMEDIATE_ONLY}")
print(f"All-negative groups excluded:     {AFP_EXCLUDE_ALL_NEGATIVE_GROUPS}")
print(f"Future-neighborhood features:     {AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES}")
print(f"KGE required in core model:       {AFP_USE_KGE_CORE}")

print("\nDevelopment candidates -- NOT frozen:")
print(f"Semantic encoder:    {AFP_SEMANTIC_ENCODER_CANDIDATE}")
print(f"Hidden dims:         {AFP_HIDDEN_DIM_CANDIDATES}")
print(f"Top-B grid:          {TOP_B_GRID}")
print(f"Threshold grid:      {THRESHOLD_GRID}")
print("T grid:              deferred until scorer validation")
print("gamma_min grid:      deferred until scorer validation")

print("\nControlled RQ2 methods:")
for method in RQ2_METHODS:
    print(f"  - {method}")

print(f"\nPrimary structural cost: {RQ2_PRIMARY_COST_METRIC}")
print(f"Primary RQ2 outcome:     {RQ2_PRIMARY_OUTCOME}")
print(f"Seeds:                   {RQ2_SEEDS}")

print("\n=== RQ2 CELL 1: PASSED ===")
print("TRAIN + VALIDATION only.")
print("No test-set information may influence AFP development.")
print("Next: generate/resume and freeze full TRAIN relation plans.")


RQ2 DEVELOPMENT SETUP
Output directory:    /kaggle/working/step3_rq2_dev_v1

Development datasets:
WebQSP train:        2826
WebQSP validation:   246
CWQ train:           27639
CWQ validation:      3519

Frozen RoG:
Planner model:       rmanluo/RoG
Top-K plans:         3
Instruction SHA256:  e3687b4a5081c22c...

AFP methodological invariants:
Final-hop protection:             True
Intermediate pruning only:        True
All-negative groups excluded:     True
Future-neighborhood features:     False
KGE required in core model:       False

Development candidates -- NOT frozen:
Semantic encoder:    sentence-transformers/all-MiniLM-L6-v2
Hidden dims:         [32, 64, 128]
Top-B grid:          [1, 2, 4, 8, 16, 32]
Threshold grid:      [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
T grid:              deferred until scorer validation
gamma_min grid:      deferred until scorer validation

Controlled RQ2 methods:
  - RoG
  - Fixed-Top-B
  - Fixed-Threshold
  - Random-B
  - Adaptive-Budget-Rand

In [79]:
# Check partial CWQ shard 42 after interruption

import os
import json

path = (
    "/kaggle/working/step3_rq2_dev_v1/"
    "01_train_plans/cwq/shards/"
    "shard_0041_010250_010499.jsonl"
)

print("Exists:", os.path.exists(path))

valid = 0

if os.path.exists(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                json.loads(line)
                valid += 1
            except json.JSONDecodeError:
                break

print("Valid saved records:", valid)
print("Remaining:", 250 - valid)

Exists: True
Valid saved records: 250
Remaining: 0


In [80]:
import os

meta_path = (
    "/kaggle/working/step3_rq2_dev_v1/"
    "01_train_plans/cwq/shards/"
    "shard_0041_010250_010499.meta.json"
)

print("Frozen metadata exists:", os.path.exists(meta_path))

Frozen metadata exists: True


In [81]:
# ## Utility — backup current RQ2 progress
# Run BEFORE resuming the long RQ2 Cell 2.

import os
import shutil
from datetime import datetime

SRC = "/kaggle/working/step3_rq2_dev_v1"

assert os.path.exists(SRC), f"RQ2 directory not found: {SRC}"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP_BASE = (
    f"/kaggle/working/"
    f"rq2_backup_after_shard42_{timestamp}"
)

backup_file = shutil.make_archive(
    BACKUP_BASE,
    "zip",
    root_dir="/kaggle/working",
    base_dir="step3_rq2_dev_v1"
)

print("=" * 70)
print("RQ2 BACKUP CREATED")
print("=" * 70)
print("Source :", SRC)
print("Backup :", backup_file)
print(
    "Size   :",
    round(os.path.getsize(backup_file) / (1024 ** 2), 2),
    "MB"
)
print("\nIMPORTANT: save a Kaggle notebook version/output")
print("to make this backup survive a full session reset.")

RQ2 BACKUP CREATED
Source : /kaggle/working/step3_rq2_dev_v1
Backup : /kaggle/working/rq2_backup_after_shard42_20260831_073227.zip
Size   : 3.86 MB

IMPORTANT: save a Kaggle notebook version/output
to make this backup survive a full session reset.


## Generate/resume and freeze full TRAIN relation plans

In [82]:
# Produces deterministic, resumable RoG relation-plan shards for WebQSP/CWQ TRAIN.
#
# IMPORTANT:
# - Uses TRAIN only.
# - Reuses the SAME RoG planner semantics used for frozen validation planning.
# - Performs a cheap validation-plan fidelity gate BEFORE the long training run.
# - Saves minimal planner outputs only; does NOT duplicate full per-question graphs.
# - source_index is the primary key.
# - Completed shards are immutable.
# - Full training plans are frozen before branch-label construction.

import os
import json
import hashlib
from datetime import datetime, timezone
from tqdm.auto import tqdm

RQ2_CELL2_CODE_TAG = "rq2_train_plan_freeze_v1"
TRAIN_PLAN_SHARD_SIZE = 250
RQ2_PLANNER_DO_SAMPLE = False
RQ2_PLANNER_MAX_NEW_TOKENS = int(
    globals().get("MAX_NEW_TOKENS", 100)
)
PLANNER_PREFLIGHT_N = 3

# =========================================================
# 1. Resolve EXACT planner components already used earlier
# =========================================================
assert "generate_seq" in globals() and callable(generate_seq), (
    "generate_seq() is missing. Re-run the earlier RoG planning-definition cell."
)
assert "parse_prediction" in globals() and callable(parse_prediction), (
    "parse_prediction() is missing. Re-run the earlier RoG planning-definition cell."
)

def first_existing_global(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return globals()[name], name
    return None, None

planner_model, planner_model_var = first_existing_global([
    "planner_model", "rog_model", "model"
])

planner_tokenizer, planner_tokenizer_var = first_existing_global([
    "planner_tokenizer", "rog_tokenizer", "tokenizer"
])

assert planner_model is not None, (
    "Could not locate the already-loaded RoG planner model."
)
assert planner_tokenizer is not None, (
    "Could not locate the already-loaded RoG tokenizer."
)
assert hasattr(planner_model, "generate"), (
    f"Resolved '{planner_model_var}', but it has no .generate() method."
)

planner_model.eval()

print(f"Planner model object:     {planner_model_var}")
print(f"Planner tokenizer object: {planner_tokenizer_var}")

# =========================================================
# 2. Resolve the SAME prompt formatter used for validation
# =========================================================
def infer_prompt_template_from_frozen_validation():
    for records in [
        globals().get("webqsp_val_planning"),
        globals().get("cwq_val_planning"),
    ]:
        if records is None or len(records) == 0:
            continue

        rec = records[0]

        if "input" not in rec or "question" not in rec:
            continue

        question = rec["question"]
        full_input = rec["input"]

        pos = full_input.rfind(question)

        if pos >= 0:
            prefix = full_input[:pos]
            suffix = full_input[pos + len(question):]

            def builder(q):
                return prefix + q + suffix

            assert builder(question) == full_input
            return builder, "inferred_from_frozen_validation_input"

    return None, None


def resolve_prompt_builder():
    # Preferred: an explicit function from the existing notebook
    for fn_name in [
        "format_planning_prompt",
        "build_planning_prompt",
        "format_planner_input",
    ]:
        fn = globals().get(fn_name)

        if callable(fn):
            return fn, fn_name

    # Next: RoG-style InstructFormater object
    for obj_name in [
        "prompter",
        "planner_prompter",
        "planning_prompter",
    ]:
        obj = globals().get(obj_name)

        if obj is None or not hasattr(obj, "format"):
            continue

        def builder(q, formatter=obj):
            return formatter.format(
                instruction=INSTRUCTION,
                message=q
            )

        try:
            _ = builder("RQ2_PROMPT_TEST")
            return builder, obj_name
        except Exception:
            pass

    # Last safe option: infer the exact frozen template
    return infer_prompt_template_from_frozen_validation()


planner_prompt_builder, planner_prompt_source = resolve_prompt_builder()

assert planner_prompt_builder is not None, (
    "Could not safely recover the SAME planning prompt formatter used for "
    "validation. Do NOT invent a new prompt here. Re-run the earlier "
    "validation-planning setup cell so `prompter` or the prompt builder exists."
)

print(f"Planner prompt source:    {planner_prompt_source}")

# Prompt-template fingerprint
prompt_probe = planner_prompt_builder(
    "__RQ2_QUESTION_PLACEHOLDER__"
)

RQ2_PROMPT_TEMPLATE_HASH = hashlib.sha256(
    prompt_probe.encode("utf-8")
).hexdigest()

# =========================================================
# 3. Planner configuration fingerprint
# =========================================================
RQ2_TRAIN_PLANNER_CONFIG = {
    "code_tag": RQ2_CELL2_CODE_TAG,
    "model_path": RQ2_PLANNER_MODEL,
    "n_beam": int(RQ2_N_BEAM),
    "do_sample": bool(RQ2_PLANNER_DO_SAMPLE),
    "max_new_tokens": int(RQ2_PLANNER_MAX_NEW_TOKENS),
    "instruction_sha256": RQ2_INSTRUCTION_HASH,
    "prompt_template_sha256": RQ2_PROMPT_TEMPLATE_HASH,
}

RQ2_TRAIN_PLANNER_SIGNATURE = hashlib.sha256(
    json.dumps(
        RQ2_TRAIN_PLANNER_CONFIG,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print(
    "Planner configuration:   "
    f"{RQ2_TRAIN_PLANNER_SIGNATURE[:16]}..."
)

# =========================================================
# 4. Exact deterministic RoG planning adapter
# =========================================================
def normalize_paths(paths):
    if paths is None:
        return []

    return [
        [str(rel) for rel in path]
        for path in paths
    ]


def generate_frozen_rog_plan(question):
    input_text = planner_prompt_builder(question)

    with torch.inference_mode():
        raw_output = generate_seq(
            planner_model,
            input_text,
            planner_tokenizer,
            num_beam=RQ2_N_BEAM,
            do_sample=RQ2_PLANNER_DO_SAMPLE,
            max_new_tokens=RQ2_PLANNER_MAX_NEW_TOKENS,
        )

    assert isinstance(raw_output, dict)
    assert "paths" in raw_output

    predicted_paths = normalize_paths(
        parse_prediction(raw_output["paths"])
    )

    return {
        "predicted_paths": predicted_paths,
        "raw_paths": [
            str(x) for x in raw_output.get("paths", [])
        ],
        "scores": [
            float(x) for x in raw_output.get("scores", [])
        ],
        "norm_scores": [
            float(x) for x in raw_output.get("norm_scores", [])
        ],
        "planner_input_sha256": hashlib.sha256(
            input_text.encode("utf-8")
        ).hexdigest(),
    }

# =========================================================
# 5. CHEAP FIDELITY GATE against frozen validation plans
# =========================================================
def planner_fidelity_gate(frozen_records, dataset_label, n=3):
    assert frozen_records is not None
    assert len(frozen_records) > 0

    checked = 0
    mismatches = 0

    # Deterministically take first usable records
    for rec in frozen_records:
        if checked >= n:
            break

        if "predicted_paths" not in rec:
            continue

        generated = generate_frozen_rog_plan(
            rec["question"]
        )["predicted_paths"]

        expected = normalize_paths(
            rec["predicted_paths"]
        )

        checked += 1

        if generated != expected:
            mismatches += 1
            print(
                f"\n[{dataset_label}] Planner mismatch "
                f"for question id={rec['id']}"
            )
            print("Expected:", expected)
            print("Generated:", generated)

    assert checked > 0, (
        f"{dataset_label}: no frozen validation records "
        f"were available for planner fidelity testing."
    )

    print(
        f"[{dataset_label}] Planner fidelity: "
        f"{checked} checked | mismatches={mismatches}"
    )

    assert mismatches == 0, (
        f"{dataset_label}: TRAIN planning configuration does not "
        f"reproduce the frozen validation planner. STOP before "
        f"generating training plans."
    )


planner_fidelity_gate(
    webqsp_val_planning,
    "WebQSP validation",
    PLANNER_PREFLIGHT_N
)

planner_fidelity_gate(
    cwq_val_planning,
    "CWQ validation",
    PLANNER_PREFLIGHT_N
)

print("\n=== TRAIN PLANNER FIDELITY GATE: PASSED ===")

# =========================================================
# 6. Dataset fingerprint
# =========================================================
def dataset_planning_fingerprint(dataset):
    ids = dataset["id"]
    questions = dataset["question"]

    h = hashlib.sha256()

    for i, (qid, q) in enumerate(zip(ids, questions)):
        h.update(
            f"{i}\t{qid}\t{q}\n".encode(
                "utf-8",
                errors="replace"
            )
        )

    return h.hexdigest()


webqsp_train_fingerprint = dataset_planning_fingerprint(
    webqsp_train
)

cwq_train_fingerprint = dataset_planning_fingerprint(
    cwq_train
)

print(
    "WebQSP train fingerprint:",
    webqsp_train_fingerprint[:16] + "..."
)

print(
    "CWQ train fingerprint:   ",
    cwq_train_fingerprint[:16] + "..."
)

# =========================================================
# 7. Safe JSON utilities
# =========================================================
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_write_json(obj, path):
    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2
        )
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


def load_jsonl_recover(path):
    if not os.path.exists(path):
        return []

    rows = []
    corrupted_tail = False

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                corrupted_tail = True

                print(
                    f"Recovering {path}: invalid JSON "
                    f"at line {line_no}; dropping tail."
                )
                break

    # Rewrite only valid records after interrupted write
    if corrupted_tail:
        tmp = path + ".recover.tmp"

        with open(tmp, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

            f.flush()
            os.fsync(f.fileno())

        os.replace(tmp, path)

    return rows


def canonicalize_jsonl(rows, path):
    rows = sorted(
        rows,
        key=lambda x: int(x["source_index"])
    )

    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

# =========================================================
# 8. Per-record planning output
# =========================================================
def plan_training_record(
    dataset,
    source_index,
    dataset_label,
    dataset_fingerprint
):
    qid = dataset["id"][source_index]
    question = dataset["question"][source_index]

    result = generate_frozen_rog_plan(question)

    return {
        "dataset": dataset_label,
        "split": "train",
        "source_index": int(source_index),
        "id": qid,
        "question": question,
        "predicted_paths": result["predicted_paths"],
        "raw_paths": result["raw_paths"],
        "scores": result["scores"],
        "norm_scores": result["norm_scores"],
        "planner_input_sha256":
            result["planner_input_sha256"],
        "planner_signature":
            RQ2_TRAIN_PLANNER_SIGNATURE,
        "dataset_fingerprint":
            dataset_fingerprint,
    }

# =========================================================
# 9. Resumable deterministic shard generation
# =========================================================
def generate_train_plan_shards(
    dataset,
    dataset_label,
    dataset_fingerprint,
    shard_size=250
):
    dataset_dir = os.path.join(
        RQ2_PLAN_DIR,
        dataset_label
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    os.makedirs(shard_dir, exist_ok=True)

    n = len(dataset)
    n_shards = (n + shard_size - 1) // shard_size
    shard_metadata = []

    print(
        f"\n{dataset_label}: {n} TRAIN questions | "
        f"{n_shards} shards | shard_size={shard_size}"
    )

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)

        shard_name = (
            f"shard_{shard_id:04d}_"
            f"{start:06d}_{end - 1:06d}"
        )

        shard_path = os.path.join(
            shard_dir,
            shard_name + ".jsonl"
        )

        meta_path = os.path.join(
            shard_dir,
            shard_name + ".meta.json"
        )

        # -------------------------------------------------
        # Completed shard = immutable
        # -------------------------------------------------
        if os.path.exists(meta_path):
            with open(
                meta_path,
                "r",
                encoding="utf-8"
            ) as f:
                meta = json.load(f)

            assert (
                meta["planner_signature"]
                == RQ2_TRAIN_PLANNER_SIGNATURE
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"planner configuration drift."
            )

            assert (
                meta["dataset_fingerprint"]
                == dataset_fingerprint
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"dataset fingerprint drift."
            )

            assert meta["start"] == start
            assert meta["end"] == end
            assert os.path.exists(shard_path)

            actual_hash = sha256_file(shard_path)

            assert (
                actual_hash == meta["sha256"]
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"completed shard hash mismatch."
            )

            shard_metadata.append(meta)

            print(
                f"[{dataset_label}] shard "
                f"{shard_id + 1}/{n_shards}: "
                f"verified frozen ({end-start} records)"
            )

            continue

        # -------------------------------------------------
        # Partial shard: recover/resume by source_index
        # -------------------------------------------------
        existing_rows = load_jsonl_recover(
            shard_path
        )

        existing_by_index = {}

        for row in existing_rows:
            idx = int(row["source_index"])

            assert start <= idx < end, (
                f"{dataset_label} shard {shard_id}: "
                f"record index {idx} outside shard range."
            )

            assert (
                row["planner_signature"]
                == RQ2_TRAIN_PLANNER_SIGNATURE
            )

            assert (
                row["dataset_fingerprint"]
                == dataset_fingerprint
            )

            expected_id = dataset["id"][idx]

            assert str(row["id"]) == str(expected_id), (
                f"{dataset_label} index {idx}: "
                f"dataset ID mismatch."
            )

            assert idx not in existing_by_index, (
                f"{dataset_label} shard {shard_id}: "
                f"duplicate source_index={idx}."
            )

            existing_by_index[idx] = row

        missing_indices = [
            i for i in range(start, end)
            if i not in existing_by_index
        ]

        print(
            f"\n[{dataset_label}] shard "
            f"{shard_id + 1}/{n_shards} "
            f"[{start}:{end}] | "
            f"resume={len(existing_by_index)} | "
            f"remaining={len(missing_indices)}"
        )

        if missing_indices:
            with open(
                shard_path,
                "a",
                encoding="utf-8"
            ) as f:

                for j, idx in enumerate(
                    tqdm(
                        missing_indices,
                        desc=(
                            f"{dataset_label} "
                            f"train shard {shard_id}"
                        ),
                        leave=False
                    ),
                    start=1
                ):
                    row = plan_training_record(
                        dataset,
                        idx,
                        dataset_label,
                        dataset_fingerprint
                    )

                    f.write(
                        json.dumps(
                            row,
                            ensure_ascii=False
                        ) + "\n"
                    )

                    # Make interruption recovery practical
                    f.flush()

                    if j % 25 == 0:
                        os.fsync(f.fileno())

                os.fsync(f.fileno())

        # -------------------------------------------------
        # Verify complete shard before freezing
        # -------------------------------------------------
        complete_rows = load_jsonl_recover(
            shard_path
        )

        assert len(complete_rows) == end - start, (
            f"{dataset_label} shard {shard_id}: "
            f"expected {end-start} records, "
            f"found {len(complete_rows)}."
        )

        complete_indices = sorted(
            int(r["source_index"])
            for r in complete_rows
        )

        assert complete_indices == list(
            range(start, end)
        ), (
            f"{dataset_label} shard {shard_id}: "
            f"missing or duplicate source indices."
        )

        # Canonical deterministic ordering
        canonicalize_jsonl(
            complete_rows,
            shard_path
        )

        shard_hash = sha256_file(
            shard_path
        )

        meta = {
            "dataset": dataset_label,
            "split": "train",
            "shard_id": shard_id,
            "start": start,
            "end": end,
            "n_records": end - start,
            "dataset_fingerprint":
                dataset_fingerprint,
            "planner_signature":
                RQ2_TRAIN_PLANNER_SIGNATURE,
            "sha256":
                shard_hash,
            "code_tag":
                RQ2_CELL2_CODE_TAG,
            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        atomic_write_json(
            meta,
            meta_path
        )

        shard_metadata.append(meta)

        print(
            f"[{dataset_label}] shard "
            f"{shard_id + 1}/{n_shards}: "
            f"FROZEN | sha256={shard_hash[:12]}..."
        )

    return shard_metadata

# =========================================================
# 10. Run resumable TRAIN planning
# =========================================================
webqsp_train_shards = generate_train_plan_shards(
    webqsp_train,
    "webqsp",
    webqsp_train_fingerprint,
    shard_size=TRAIN_PLAN_SHARD_SIZE
)

cwq_train_shards = generate_train_plan_shards(
    cwq_train,
    "cwq",
    cwq_train_fingerprint,
    shard_size=TRAIN_PLAN_SHARD_SIZE
)

# =========================================================
# 11. Combine verified shards into immutable frozen files
# =========================================================
def freeze_combined_train_plans(
    dataset,
    dataset_label,
    dataset_fingerprint,
    shard_metadata
):
    dataset_dir = os.path.join(
        RQ2_PLAN_DIR,
        dataset_label
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    frozen_path = os.path.join(
        dataset_dir,
        f"{dataset_label}_train_plans_frozen.jsonl"
    )

    manifest_path = os.path.join(
        dataset_dir,
        f"{dataset_label}_train_plans_frozen_manifest.json"
    )

    # ---------------------------------------------
    # If already frozen, verify rather than rewrite
    # ---------------------------------------------
    if os.path.exists(manifest_path):
        with open(
            manifest_path,
            "r",
            encoding="utf-8"
        ) as f:
            manifest = json.load(f)

        assert (
            manifest["planner_signature"]
            == RQ2_TRAIN_PLANNER_SIGNATURE
        )

        assert (
            manifest["dataset_fingerprint"]
            == dataset_fingerprint
        )

        assert manifest["n_records"] == len(dataset)
        assert os.path.exists(frozen_path)

        actual_hash = sha256_file(
            frozen_path
        )

        assert (
            actual_hash
            == manifest["combined_sha256"]
        ), (
            f"{dataset_label}: frozen combined "
            f"training-plan hash mismatch."
        )

        print(
            f"\n[{dataset_label}] Existing combined "
            f"TRAIN plan file verified."
        )

        return frozen_path, manifest

    # ---------------------------------------------
    # Create deterministic combined file
    # ---------------------------------------------
    tmp_path = frozen_path + ".tmp"
    total_written = 0

    with open(
        tmp_path,
        "w",
        encoding="utf-8"
    ) as fout:

        for meta in sorted(
            shard_metadata,
            key=lambda x: x["shard_id"]
        ):
            shard_name = (
                f"shard_{meta['shard_id']:04d}_"
                f"{meta['start']:06d}_"
                f"{meta['end'] - 1:06d}.jsonl"
            )

            shard_path = os.path.join(
                shard_dir,
                shard_name
            )

            assert (
                sha256_file(shard_path)
                == meta["sha256"]
            )

            rows = load_jsonl_recover(
                shard_path
            )

            rows = sorted(
                rows,
                key=lambda x: int(
                    x["source_index"]
                )
            )

            for row in rows:
                fout.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

                total_written += 1

        fout.flush()
        os.fsync(fout.fileno())

    assert total_written == len(dataset), (
        f"{dataset_label}: combined file has "
        f"{total_written} records; expected {len(dataset)}."
    )

    os.replace(
        tmp_path,
        frozen_path
    )

    combined_rows = load_jsonl_recover(
        frozen_path
    )

    assert len(combined_rows) == len(dataset)

    indices = [
        int(r["source_index"])
        for r in combined_rows
    ]

    assert indices == list(
        range(len(dataset))
    ), (
        f"{dataset_label}: combined training plans "
        f"are not in exact dataset order."
    )

    # Verify IDs against source dataset
    for i, row in enumerate(combined_rows):
        assert str(row["id"]) == str(
            dataset["id"][i]
        )

    combined_hash = sha256_file(
        frozen_path
    )

    manifest = {
        "dataset": dataset_label,
        "split": "train",
        "n_records": len(dataset),
        "n_shards": len(shard_metadata),
        "shard_size": TRAIN_PLAN_SHARD_SIZE,
        "dataset_fingerprint":
            dataset_fingerprint,
        "planner_signature":
            RQ2_TRAIN_PLANNER_SIGNATURE,
        "planner_config":
            RQ2_TRAIN_PLANNER_CONFIG,
        "combined_sha256":
            combined_hash,
        "code_tag":
            RQ2_CELL2_CODE_TAG,
        "frozen_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        manifest,
        manifest_path
    )

    print(
        f"\n[{dataset_label}] TRAIN plans FROZEN: "
        f"{len(dataset)} records"
    )

    print(
        f"[{dataset_label}] combined SHA256: "
        f"{combined_hash}"
    )

    return frozen_path, manifest


WEBQSP_TRAIN_PLAN_FILE, webqsp_train_plan_manifest = (
    freeze_combined_train_plans(
        webqsp_train,
        "webqsp",
        webqsp_train_fingerprint,
        webqsp_train_shards
    )
)

CWQ_TRAIN_PLAN_FILE, cwq_train_plan_manifest = (
    freeze_combined_train_plans(
        cwq_train,
        "cwq",
        cwq_train_fingerprint,
        cwq_train_shards
    )
)

# =========================================================
# 12. Load minimal frozen planner rows
# =========================================================
webqsp_train_plan_rows = load_jsonl_recover(
    WEBQSP_TRAIN_PLAN_FILE
)

cwq_train_plan_rows = load_jsonl_recover(
    CWQ_TRAIN_PLAN_FILE
)

assert len(webqsp_train_plan_rows) == len(
    webqsp_train
)

assert len(cwq_train_plan_rows) == len(
    cwq_train
)

# =========================================================
# 13. Memory-efficient view for future branch-label cells
# =========================================================
class FrozenPlanningView:
    """
    Lazily attaches frozen predicted_paths to the original
    HuggingFace dataset record without duplicating all graphs
    into a second in-memory list.
    """

    def __init__(self, dataset, plan_rows):
        assert len(dataset) == len(plan_rows)

        self.dataset = dataset
        self.plan_rows = plan_rows

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [
                self[i]
                for i in range(
                    *idx.indices(len(self))
                )
            ]

        rec = dict(self.dataset[int(idx)])
        plan_row = self.plan_rows[int(idx)]

        assert int(
            plan_row["source_index"]
        ) == int(idx)

        assert str(
            rec["id"]
        ) == str(
            plan_row["id"]
        )

        rec["predicted_paths"] = normalize_paths(
            plan_row["predicted_paths"]
        )

        rec["_plan_source_index"] = int(idx)
        rec["_planner_signature"] = (
            plan_row["planner_signature"]
        )

        return rec

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]


webqsp_train_planning = FrozenPlanningView(
    webqsp_train,
    webqsp_train_plan_rows
)

cwq_train_planning = FrozenPlanningView(
    cwq_train,
    cwq_train_plan_rows
)

# =========================================================
# 14. Final completeness and plan-distribution sanity report
# =========================================================
def summarize_frozen_train_plans(
    plan_rows,
    dataset_label
):
    n_questions = len(plan_rows)

    plan_counts = [
        len(r["predicted_paths"])
        for r in plan_rows
    ]

    plan_lengths = [
        len(path)
        for r in plan_rows
        for path in r["predicted_paths"]
    ]

    empty_plan_questions = sum(
        1 for x in plan_counts
        if x == 0
    )

    print(
        f"\n[{dataset_label}] FROZEN TRAIN PLAN SUMMARY"
    )

    print(
        f"Questions:              {n_questions}"
    )

    print(
        f"Questions with 0 plans: {empty_plan_questions}"
    )

    print(
        f"Total predicted plans:  {sum(plan_counts)}"
    )

    print(
        f"Mean plans/question:    "
        f"{np.mean(plan_counts):.3f}"
    )

    if len(plan_lengths) > 0:
        print(
            f"Plan length min/max:    "
            f"{min(plan_lengths)} / "
            f"{max(plan_lengths)}"
        )

        print(
            f"Mean plan length:       "
            f"{np.mean(plan_lengths):.3f}"
        )

    print(
        f"Planner signature:      "
        f"{RQ2_TRAIN_PLANNER_SIGNATURE[:16]}..."
    )


summarize_frozen_train_plans(
    webqsp_train_plan_rows,
    "WebQSP"
)

summarize_frozen_train_plans(
    cwq_train_plan_rows,
    "CWQ"
)

print("\n" + "=" * 80)
print("=== RQ2 CELL 2: TRAIN RELATION PLANS FROZEN ===")
print("=" * 80)
print(f"WebQSP: {len(webqsp_train_plan_rows)} / {len(webqsp_train)}")
print(f"CWQ:    {len(cwq_train_plan_rows)} / {len(cwq_train)}")
print("Planner fidelity gate: PASSED")
print("Completed shards: verified and immutable")
print("Combined TRAIN plan files: frozen with SHA256 manifests")
print("No test data used.")
print("Next: construct intermediate branch supervision from TRAIN only.")

Planner model object:     model
Planner tokenizer object: tokenizer
Planner prompt source:    prompter
Planner configuration:   2c36bd8621e44901...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


[WebQSP validation] Planner fidelity: 3 checked | mismatches=0
[CWQ validation] Planner fidelity: 3 checked | mismatches=0

=== TRAIN PLANNER FIDELITY GATE: PASSED ===
WebQSP train fingerprint: 461cebcb68f95041...
CWQ train fingerprint:    bdc1e12cb379a627...

webqsp: 2826 TRAIN questions | 12 shards | shard_size=250
[webqsp] shard 1/12: verified frozen (250 records)
[webqsp] shard 2/12: verified frozen (250 records)
[webqsp] shard 3/12: verified frozen (250 records)
[webqsp] shard 4/12: verified frozen (250 records)
[webqsp] shard 5/12: verified frozen (250 records)
[webqsp] shard 6/12: verified frozen (250 records)
[webqsp] shard 7/12: verified frozen (250 records)
[webqsp] shard 8/12: verified frozen (250 records)
[webqsp] shard 9/12: verified frozen (250 records)
[webqsp] shard 10/12: verified frozen (250 records)
[webqsp] shard 11/12: verified frozen (250 records)
[webqsp] shard 12/12: verified frozen (76 records)

cwq: 27639 TRAIN questions | 111 shards | shard_size=250
[cwq] sha

cwq train shard 57:   0%|          | 0/33 [00:00<?, ?it/s]

[cwq] shard 58/111: FROZEN | sha256=fc366ca0a26d...

[cwq] shard 59/111 [14500:14750] | resume=0 | remaining=250


...

[cwq] shard 109/111: FROZEN | sha256=284950243d7e...

[cwq] shard 110/111 [27250:27500] | resume=0 | remaining=250


cwq train shard 109:   0%|          | 0/250 [00:00<?, ?it/s]

[cwq] shard 110/111: FROZEN | sha256=b2b0d5239227...

[cwq] shard 111/111 [27500:27639] | resume=0 | remaining=139


cwq train shard 110:   0%|          | 0/139 [00:00<?, ?it/s]

[cwq] shard 111/111: FROZEN | sha256=f7db18193eb3...

[webqsp] TRAIN plans FROZEN: 2826 records
[webqsp] combined SHA256: ac4d388a9a24314b103b3a376812f07a7e4271358a318a02328b2668335cd5cd

[cwq] TRAIN plans FROZEN: 27639 records
[cwq] combined SHA256: 517502c8509aa44773a9a9c2915d15303623dd10410880965a2c49449b579aa1

[WebQSP] FROZEN TRAIN PLAN SUMMARY
Questions:              2826
Questions with 0 plans: 0
Total predicted plans:  8398
Mean plans/question:    2.972
Plan length min/max:    0 / 4
Mean plan length:       1.421
Planner signature:      2c36bd8621e44901...

[CWQ] FROZEN TRAIN PLAN SUMMARY
Questions:              27639
Questions with 0 plans: 0
Total predicted plans:  82738
Mean plans/question:    2.994
Plan length min/max:    0 / 5
Mean plan length:       1.809
Planner signature:      2c36bd8621e44901...

=== RQ2 CELL 2: TRAIN RELATION PLANS FROZEN ===
WebQSP: 2826 / 2826
CWQ:    27639 / 27639
Planner fidelity gate: PASSED
Completed shards: verified and immutable
Combined TRAIN 

In [84]:
#  Frozen TRAIN-plan sanity diagnostic

def inspect_frozen_plans(plan_rows, dataset_name):
    total_plans = 0
    empty_plans = 0
    duplicate_plan_questions = 0
    length_counts = {}

    examples = []

    for row in plan_rows:
        plans = row["predicted_paths"]
        total_plans += len(plans)

        normalized = [tuple(p) for p in plans]

        if len(normalized) != len(set(normalized)):
            duplicate_plan_questions += 1

        for plan_idx, plan in enumerate(plans):
            L = len(plan)
            length_counts[L] = length_counts.get(L, 0) + 1

            if L == 0:
                empty_plans += 1

                if len(examples) < 5:
                    examples.append({
                        "source_index": row["source_index"],
                        "question_id": row["id"],
                        "question": row["question"],
                        "plan_index": plan_idx,
                        "predicted_paths": plans,
                    })

    print("\n" + "=" * 70)
    print(f"{dataset_name} FROZEN TRAIN PLAN DIAGNOSTIC")
    print("=" * 70)
    print("Questions:                ", len(plan_rows))
    print("Total predicted plans:    ", total_plans)
    print("Empty relation plans:     ", empty_plans)
    print("Duplicate-plan questions: ", duplicate_plan_questions)
    print("Plan-length distribution: ", dict(sorted(length_counts.items())))

    if examples:
        print("\nExample questions containing an empty plan:")
        for x in examples:
            print(x)

inspect_frozen_plans(
    webqsp_train_plan_rows,
    "WebQSP"
)

inspect_frozen_plans(
    cwq_train_plan_rows,
    "CWQ"
)


WebQSP FROZEN TRAIN PLAN DIAGNOSTIC
Questions:                 2826
Total predicted plans:     8398
Empty relation plans:      5
Duplicate-plan questions:  15
Plan-length distribution:  {0: 5, 1: 4916, 2: 3415, 3: 60, 4: 2}

Example questions containing an empty plan:
{'source_index': 120, 'question_id': 'WebQTrn-168', 'question': 'what was the name of the original seattle baseball team', 'plan_index': 1, 'predicted_paths': [['sports.defunct_sports_team.later_known_as'], [], ['common.topic.notable_types', 'common.topic.notable_types']]}
{'source_index': 806, 'question_id': 'WebQTrn-1097', 'question': 'what is the new orleans hornets new name', 'plan_index': 0, 'predicted_paths': [[], ['common.topic.notable_types', 'common.topic.notable_types'], ['sports.defunct_sports_team.later_known_as']]}
{'source_index': 2441, 'question_id': 'WebQTrn-3312', 'question': 'which legend of zelda game is the first', 'plan_index': 0, 'predicted_paths': [[], ['common.topic.notable_types'], ['film.film.st

## Construct leakage-free TRAIN branch supervision using validated suffix DP

In [85]:
# ## 3.3 Construct leakage-free TRAIN branch supervision using validated suffix DP
# Positive branch:
#   candidate endpoint can reach >=1 TRAIN gold answer through the exact
#   remaining suffix of the SAME frozen RoG relation plan.
#
# Main label file:
#   contains branches ONLY from feasible intermediate plan-hop groups.
#
# Infeasible groups:
#   candidate set exists but no candidate supports a gold continuation.
#   Logged separately and EXCLUDED from scorer loss.
#
# Empty relation plans:
#   preserved in frozen planner outputs but skipped here because they
#   contain no relation transition and therefore no AFP supervision.
#
# Duplicate predicted plans:
#   preserved exactly as generated by the frozen RoG planner.
#
# Final hop:
#   NEVER labeled because the main AFP configuration does not prune it.

import os
import json
import hashlib
import inspect
from datetime import datetime, timezone
from tqdm import tqdm

RQ2_CELL3_CODE_TAG = "rq2_branch_labels_suffix_dp_v2"
BRANCH_LABEL_SHARD_SIZE = 250

# =========================================================
# 1. Required frozen artifacts / methodological guards
# =========================================================
required = [
    "webqsp_train",
    "cwq_train",
    "webqsp_train_plan_rows",
    "cwq_train_plan_rows",
    "webqsp_train_plan_manifest",
    "cwq_train_plan_manifest",
    "build_graph",
    "suffix_reachable_dp",
    "RQ2_LABEL_DIR",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Run RQ2 Cell 2 to COMPLETE first. Missing: "
    + ", ".join(missing)
)

assert AFP_FINAL_HOP_PROTECTION is True
assert AFP_INTERMEDIATE_ONLY is True
assert AFP_EXCLUDE_ALL_NEGATIVE_GROUPS is True

assert len(webqsp_train_plan_rows) == len(webqsp_train) == 2826
assert len(cwq_train_plan_rows) == len(cwq_train) == 27639

WEBQSP_PLAN_SHA256 = (
    webqsp_train_plan_manifest["combined_sha256"]
)

CWQ_PLAN_SHA256 = (
    cwq_train_plan_manifest["combined_sha256"]
)

# Hash the already-validated suffix-DP implementation
try:
    SUFFIX_DP_SOURCE = inspect.getsource(
        suffix_reachable_dp
    )

    SUFFIX_DP_SHA256 = hashlib.sha256(
        SUFFIX_DP_SOURCE.encode("utf-8")
    ).hexdigest()

except Exception:
    SUFFIX_DP_SHA256 = "source_unavailable"

print(
    "Suffix-DP signature:",
    inspect.signature(suffix_reachable_dp)
)

print(
    "Suffix-DP SHA256:   ",
    SUFFIX_DP_SHA256[:16] + "..."
)

# =========================================================
# 2. Dataset / frozen-plan alignment gate
# =========================================================
def validate_train_schema(
    dataset,
    plan_rows,
    dataset_name
):
    required_fields = {
        "id",
        "question",
        "q_entity",
        "a_entity",
        "graph",
    }

    actual_fields = set(
        dataset.column_names
    )

    missing_fields = (
        required_fields - actual_fields
    )

    assert not missing_fields, (
        f"{dataset_name}: missing fields "
        f"{sorted(missing_fields)}\n"
        f"Available: {sorted(actual_fields)}"
    )

    assert len(dataset) == len(plan_rows)

    check_indices = [
        0,
        len(dataset) // 2,
        len(dataset) - 1,
    ]

    for i in check_indices:
        assert (
            str(dataset[i]["id"])
            == str(plan_rows[i]["id"])
        )

        assert (
            int(plan_rows[i]["source_index"])
            == i
        )

        assert (
            plan_rows[i]["split"]
            == "train"
        )

    print(
        f"[{dataset_name}] "
        f"schema/alignment gate: PASSED | "
        f"{len(dataset)} questions"
    )


validate_train_schema(
    webqsp_train,
    webqsp_train_plan_rows,
    "WebQSP"
)

validate_train_schema(
    cwq_train,
    cwq_train_plan_rows,
    "CWQ"
)

# =========================================================
# 3. Relation-valid neighbor matching
# Uses reproduced RoG graph/relation semantics.
# =========================================================
def relation_valid_neighbors(
    G,
    node,
    required_relation
):
    if node not in G:
        return []

    matched = []

    for nbr in G.neighbors(node):
        edge_data = G.get_edge_data(
            node,
            nbr
        )

        assert edge_data is not None

        assert "relation" in edge_data, (
            "Expected edge attribute "
            "'relation' not found."
        )

        edge_relation = (
            edge_data["relation"]
        )

        if isinstance(
            edge_relation,
            (list, tuple, set)
        ):
            is_match = (
                required_relation
                in edge_relation
            )
        else:
            is_match = (
                edge_relation
                == required_relation
            )

        if is_match:
            matched.append(nbr)

    return matched

# =========================================================
# 4. Adapter to validated suffix-reachability DP
#
# Concept:
# reachable_dp[t][e] =
#   can entity e reach any gold answer by executing plan[t:]?
#
# Candidate generated at hop h already executed plan[h].
# Therefore its remaining suffix begins at h+1.
# =========================================================
def call_suffix_dp(
    G,
    plan,
    gold_answers
):
    sig = inspect.signature(
        suffix_reachable_dp
    )

    params = list(
        sig.parameters.keys()
    )

    if len(params) != 3:
        raise RuntimeError(
            "suffix_reachable_dp does not have "
            "the expected 3-argument interface. "
            f"Current signature: {sig}"
        )

    values = {}

    for p in params:
        low = p.lower()

        if (
            low in {"g", "kg", "graph"}
            or "graph" in low
        ):
            values[p] = G

        elif (
            "plan" in low
            or "rule" in low
            or "relation_path" in low
            or low in {
                "relations",
                "path",
            }
        ):
            values[p] = plan

        elif (
            "gold" in low
            or "answer" in low
            or "target" in low
        ):
            values[p] = gold_answers

        else:
            raise RuntimeError(
                f"Cannot safely map DP "
                f"parameter '{p}'. "
                f"Signature: {sig}"
            )

    reachable_dp = (
        suffix_reachable_dp(**values)
    )

    assert (
        len(reachable_dp)
        == len(plan) + 1
    ), (
        f"Suffix DP returned "
        f"{len(reachable_dp)} layers "
        f"for plan length {len(plan)}; "
        f"expected L+1."
    )

    return reachable_dp


def dp_is_reachable(
    reachable_dp,
    state_idx,
    entity
):
    layer = reachable_dp[state_idx]

    if isinstance(layer, dict):
        return bool(
            layer.get(entity, False)
        )

    if isinstance(
        layer,
        (set, frozenset)
    ):
        return entity in layer

    try:
        return bool(layer[entity])

    except (
        KeyError,
        IndexError,
        TypeError,
    ):
        return False

# =========================================================
# 5. Terminal-state DP sanity gate
# At DP state L, reachability must equal gold membership.
# =========================================================
def suffix_dp_terminal_gate(
    dataset,
    plan_rows,
    dataset_name
):
    checked = 0

    for source_index in range(
        min(len(dataset), 100)
    ):
        rec = dataset[source_index]

        plans = (
            plan_rows[source_index]
            ["predicted_paths"]
        )

        if not plans:
            continue

        G = build_graph(
            rec["graph"]
        )

        gold_answers = set(
            rec["a_entity"]
        )

        for plan in plans:
            plan = list(plan)

            # Empty relation plans contain no
            # AFP-supervisable transition.
            if len(plan) == 0:
                continue

            reachable_dp = call_suffix_dp(
                G,
                plan,
                gold_answers
            )

            L = len(plan)

            nodes_to_check = list(
                gold_answers
            )[:10]

            if len(nodes_to_check) < 20:
                nodes_to_check += list(
                    G.nodes()
                )[:20]

            for node in nodes_to_check:
                observed = dp_is_reachable(
                    reachable_dp,
                    L,
                    node
                )

                expected = (
                    node in gold_answers
                )

                assert observed == expected, (
                    f"{dataset_name}: "
                    f"suffix-DP terminal mismatch\n"
                    f"node={node}\n"
                    f"observed={observed}\n"
                    f"expected={expected}"
                )

                checked += 1

            if checked >= 50:
                print(
                    f"[{dataset_name}] "
                    f"suffix-DP terminal gate: "
                    f"PASSED | checked={checked}"
                )

                return

    assert checked > 0

    print(
        f"[{dataset_name}] "
        f"suffix-DP terminal gate: "
        f"PASSED | checked={checked}"
    )


suffix_dp_terminal_gate(
    webqsp_train,
    webqsp_train_plan_rows,
    "WebQSP"
)

suffix_dp_terminal_gate(
    cwq_train,
    cwq_train_plan_rows,
    "CWQ"
)

print(
    "\n=== SUFFIX-DP "
    "TRAINING-LABEL PREFLIGHT: PASSED ==="
)

# =========================================================
# 6. JSON / hashing helpers
# =========================================================
def json_safe(x):
    if isinstance(x, dict):
        return {
            str(k): json_safe(v)
            for k, v in x.items()
        }

    if isinstance(
        x,
        (list, tuple, set)
    ):
        return [
            json_safe(v)
            for v in x
        ]

    if isinstance(x, np.integer):
        return int(x)

    if isinstance(x, np.floating):
        return float(x)

    if isinstance(x, np.bool_):
        return bool(x)

    return x


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_json(
    obj,
    path
):
    tmp = path + ".tmp"

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            json_safe(obj),
            f,
            ensure_ascii=False,
            indent=2
        )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp,
        path
    )


def atomic_jsonl(
    rows,
    path
):
    tmp = path + ".tmp"

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    json_safe(row),
                    ensure_ascii=False
                )
                + "\n"
            )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp,
        path
    )

# =========================================================
# 7. Label ONE training question
# =========================================================
def label_one_train_question(
    rec,
    plan_row,
    dataset_name,
    source_index
):
    assert (
        str(rec["id"])
        == str(plan_row["id"])
    )

    assert (
        int(plan_row["source_index"])
        == source_index
    )

    G = build_graph(
        rec["graph"]
    )

    topic_entities = list(
        rec["q_entity"]
    )

    gold_answers = set(
        rec["a_entity"]
    )

    plans = [
        list(p)
        for p in plan_row[
            "predicted_paths"
        ]
    ]

    branch_rows = []
    group_rows = []

    question_stats = {
        "plans": len(plans),

        # NEW: explicit audit of empty plans
        "empty_plans_skipped": 0,

        "topic_entities":
            len(topic_entities),

        "intermediate_groups": 0,
        "feasible_groups": 0,
        "infeasible_groups": 0,

        "empty_terminated": 0,
        "final_hop_groups": 0,
        "decision_groups": 0,

        "labeled_branches": 0,
        "positive_branches": 0,
        "negative_branches": 0,
    }

    # -----------------------------------------------------
    # Preserve EVERY frozen predicted plan, including
    # duplicate plan instances. We do not deduplicate.
    # -----------------------------------------------------
    for plan_idx, plan in enumerate(
        plans
    ):
        L = len(plan)

        # ---------------------------------------------
        # Frozen planner occasionally produced [].
        # Keep it in planner artifact, but it has
        # no relation transition and no AFP label.
        # ---------------------------------------------
        if L == 0:
            question_stats[
                "empty_plans_skipped"
            ] += 1
            continue

        # Plan-specific DP computed once,
        # reused across topic entities.
        reachable_dp = call_suffix_dp(
            G,
            plan,
            gold_answers
        )

        for (
            topic_idx,
            topic_entity
        ) in enumerate(
            topic_entities
        ):
            # Path multiplicity is preserved.
            active_prefixes = [
                (topic_entity,)
            ]

            for h in range(L):
                required_relation = (
                    plan[h]
                )

                candidates = []

                # -----------------------------------------
                # Construct exact relation-valid C_h
                # -----------------------------------------
                for (
                    parent_idx,
                    prefix
                ) in enumerate(
                    active_prefixes
                ):
                    current_entity = (
                        prefix[-1]
                    )

                    neighbors = (
                        relation_valid_neighbors(
                            G,
                            current_entity,
                            required_relation
                        )
                    )

                    for nbr in neighbors:
                        candidates.append({
                            "parent_prefix_index":
                                parent_idx,

                            "prefix_entities":
                                prefix,

                            "candidate_entity":
                                nbr,

                            "branch_entities":
                                prefix + (nbr,),
                        })

                group_id = (
                    f"{dataset_name}|"
                    f"{source_index}|"
                    f"p{plan_idx}|"
                    f"t{topic_idx}|"
                    f"h{h}"
                )

                # -----------------------------------------
                # C_h = empty:
                # terminate THIS topic-plan traversal.
                # -----------------------------------------
                if len(candidates) == 0:
                    group_rows.append({
                        "dataset":
                            dataset_name,

                        "split":
                            "train",

                        "source_index":
                            source_index,

                        "question_id":
                            rec["id"],

                        "group_id":
                            group_id,

                        "plan_index":
                            plan_idx,

                        "topic_index":
                            topic_idx,

                        "topic_entity":
                            topic_entity,

                        "hop":
                            h,

                        "plan_length":
                            L,

                        "required_relation":
                            required_relation,

                        "candidate_count":
                            0,

                        "positive_count":
                            0,

                        "negative_count":
                            0,

                        "decision_opportunity":
                            False,

                        "status":
                            "empty_terminated",
                    })

                    question_stats[
                        "empty_terminated"
                    ] += 1

                    break

                # -----------------------------------------
                # FINAL HOP:
                # deliberately excluded from supervision.
                # -----------------------------------------
                if h == L - 1:
                    group_rows.append({
                        "dataset":
                            dataset_name,

                        "split":
                            "train",

                        "source_index":
                            source_index,

                        "question_id":
                            rec["id"],

                        "group_id":
                            group_id,

                        "plan_index":
                            plan_idx,

                        "topic_index":
                            topic_idx,

                        "topic_entity":
                            topic_entity,

                        "hop":
                            h,

                        "plan_length":
                            L,

                        "required_relation":
                            required_relation,

                        "candidate_count":
                            len(candidates),

                        "positive_count":
                            None,

                        "negative_count":
                            None,

                        "decision_opportunity":
                            False,

                        "status":
                            "final_hop_excluded",
                    })

                    question_stats[
                        "final_hop_groups"
                    ] += 1

                    break

                # -----------------------------------------
                # INTERMEDIATE SUPERVISION
                #
                # Candidate already executed plan[h].
                # Remaining suffix = plan[h+1:].
                #
                # y_i = reachable_dp[h+1][candidate]
                # -----------------------------------------
                labels = [
                    int(
                        dp_is_reachable(
                            reachable_dp,
                            h + 1,
                            c[
                                "candidate_entity"
                            ]
                        )
                    )
                    for c in candidates
                ]

                n_pos = int(
                    sum(labels)
                )

                n_neg = (
                    len(labels) - n_pos
                )

                feasible = (
                    n_pos > 0
                )

                decision = (
                    len(candidates) > 1
                )

                question_stats[
                    "intermediate_groups"
                ] += 1

                question_stats[
                    "decision_groups"
                ] += int(decision)

                group_status = (
                    "feasible"
                    if feasible
                    else
                    "infeasible_all_negative"
                )

                group_rows.append({
                    "dataset":
                        dataset_name,

                    "split":
                        "train",

                    "source_index":
                        source_index,

                    "question_id":
                        rec["id"],

                    "group_id":
                        group_id,

                    "plan_index":
                        plan_idx,

                    "topic_index":
                        topic_idx,

                    "topic_entity":
                        topic_entity,

                    "hop":
                        h,

                    "plan_length":
                        L,

                    "required_relation":
                        required_relation,

                    "candidate_count":
                        len(candidates),

                    "positive_count":
                        n_pos,

                    "negative_count":
                        n_neg,

                    "decision_opportunity":
                        decision,

                    "status":
                        group_status,
                })

                if feasible:
                    question_stats[
                        "feasible_groups"
                    ] += 1

                    # -------------------------------------
                    # ONLY feasible groups enter
                    # scorer-supervision dataset.
                    # -------------------------------------
                    for (
                        candidate_idx,
                        (cand, y)
                    ) in enumerate(
                        zip(
                            candidates,
                            labels
                        )
                    ):
                        branch_rows.append({
                            "dataset":
                                dataset_name,

                            "split":
                                "train",

                            "source_index":
                                source_index,

                            "question_id":
                                rec["id"],

                            "group_id":
                                group_id,

                            "plan_index":
                                plan_idx,

                            "plan":
                                plan,

                            "plan_length":
                                L,

                            "topic_index":
                                topic_idx,

                            "topic_entity":
                                topic_entity,

                            "hop":
                                h,

                            "required_relation":
                                required_relation,

                            "remaining_suffix":
                                plan[h + 1:],

                            "candidate_index":
                                candidate_idx,

                            "parent_prefix_index":
                                cand[
                                    "parent_prefix_index"
                                ],

                            "prefix_entities":
                                list(
                                    cand[
                                        "prefix_entities"
                                    ]
                                ),

                            "candidate_entity":
                                cand[
                                    "candidate_entity"
                                ],

                            "branch_entities":
                                list(
                                    cand[
                                        "branch_entities"
                                    ]
                                ),

                            "candidate_count":
                                len(candidates),

                            "decision_opportunity":
                                decision,

                            "label":
                                int(y),
                        })

                    question_stats[
                        "labeled_branches"
                    ] += len(candidates)

                    question_stats[
                        "positive_branches"
                    ] += n_pos

                    question_stats[
                        "negative_branches"
                    ] += n_neg

                else:
                    # -------------------------------------
                    # Planner/graph coverage failure:
                    # logged but excluded from BCE labels.
                    # -------------------------------------
                    question_stats[
                        "infeasible_groups"
                    ] += 1

                # -----------------------------------------
                # CRITICAL:
                # Label construction follows UNPRUNED RoG.
                #
                # Gold labels NEVER influence traversal.
                # All relation-valid candidates propagate.
                # -----------------------------------------
                active_prefixes = [
                    tuple(
                        c["branch_entities"]
                    )
                    for c in candidates
                ]

    return (
        branch_rows,
        group_rows,
        question_stats
    )

# =========================================================
# 8. Resumable label-shard generation
# =========================================================
def generate_branch_label_shards(
    dataset,
    plan_rows,
    dataset_name,
    plan_sha256,
    shard_size=250
):
    dataset_dir = os.path.join(
        RQ2_LABEL_DIR,
        dataset_name
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    os.makedirs(
        shard_dir,
        exist_ok=True
    )

    n = len(dataset)

    n_shards = (
        n + shard_size - 1
    ) // shard_size

    metadata = []

    print(
        f"\n{dataset_name.upper()}: "
        f"{n} TRAIN questions | "
        f"{n_shards} label shards"
    )

    for shard_id in range(
        n_shards
    ):
        start = (
            shard_id * shard_size
        )

        end = min(
            start + shard_size,
            n
        )

        stem = (
            f"shard_{shard_id:04d}_"
            f"{start:06d}_"
            f"{end - 1:06d}"
        )

        labels_path = os.path.join(
            shard_dir,
            stem + ".labels.jsonl"
        )

        groups_path = os.path.join(
            shard_dir,
            stem + ".groups.jsonl"
        )

        meta_path = os.path.join(
            shard_dir,
            stem + ".meta.json"
        )

        # ---------------------------------------------
        # Frozen shard: verify, never regenerate.
        # ---------------------------------------------
        if os.path.exists(meta_path):
            with open(
                meta_path,
                "r",
                encoding="utf-8"
            ) as f:
                meta = json.load(f)

            assert (
                meta["plan_sha256"]
                == plan_sha256
            )

            assert (
                meta["suffix_dp_sha256"]
                == SUFFIX_DP_SHA256
            )

            assert (
                meta["start"]
                == start
            )

            assert (
                meta["end"]
                == end
            )

            assert (
                sha256_file(
                    labels_path
                )
                == meta[
                    "labels_sha256"
                ]
            )

            assert (
                sha256_file(
                    groups_path
                )
                == meta[
                    "groups_sha256"
                ]
            )

            metadata.append(meta)

            print(
                f"[{dataset_name}] "
                f"label shard "
                f"{shard_id + 1}/"
                f"{n_shards}: "
                f"verified frozen"
            )

            continue

        # Remove stale temp files
        for p in [
            labels_path + ".tmp",
            groups_path + ".tmp",
        ]:
            if os.path.exists(p):
                os.remove(p)

        all_labels = []
        all_groups = []

        stats = {
            "questions": 0,

            # NEW
            "empty_plans_skipped": 0,

            "intermediate_groups": 0,
            "feasible_groups": 0,
            "infeasible_groups": 0,

            "empty_terminated": 0,
            "final_hop_groups": 0,
            "decision_groups": 0,

            "labeled_branches": 0,
            "positive_branches": 0,
            "negative_branches": 0,
        }

        for idx in tqdm(
            range(start, end),
            desc=(
                f"{dataset_name} "
                f"label shard {shard_id}"
            ),
            leave=False
        ):
            rec = dataset[idx]

            plan_row = (
                plan_rows[idx]
            )

            (
                labels,
                groups,
                qstats
            ) = (
                label_one_train_question(
                    rec,
                    plan_row,
                    dataset_name,
                    idx
                )
            )

            all_labels.extend(
                labels
            )

            all_groups.extend(
                groups
            )

            stats[
                "questions"
            ] += 1

            for key in stats:
                if key == "questions":
                    continue

                stats[key] += (
                    qstats[key]
                )

        # ---------------------------------------------
        # Atomic shard freeze
        # ---------------------------------------------
        atomic_jsonl(
            all_labels,
            labels_path
        )

        atomic_jsonl(
            all_groups,
            groups_path
        )

        labels_hash = (
            sha256_file(
                labels_path
            )
        )

        groups_hash = (
            sha256_file(
                groups_path
            )
        )

        meta = {
            "dataset":
                dataset_name,

            "split":
                "train",

            "shard_id":
                shard_id,

            "start":
                start,

            "end":
                end,

            "n_questions":
                end - start,

            "plan_sha256":
                plan_sha256,

            "suffix_dp_sha256":
                SUFFIX_DP_SHA256,

            "code_tag":
                RQ2_CELL3_CODE_TAG,

            **stats,

            "labels_sha256":
                labels_hash,

            "groups_sha256":
                groups_hash,

            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        atomic_json(
            meta,
            meta_path
        )

        metadata.append(meta)

        print(
            f"[{dataset_name}] "
            f"label shard "
            f"{shard_id + 1}/{n_shards}: "
            f"FROZEN | "
            f"branches="
            f"{stats['labeled_branches']} | "
            f"pos="
            f"{stats['positive_branches']} | "
            f"neg="
            f"{stats['negative_branches']} | "
            f"infeasible="
            f"{stats['infeasible_groups']} | "
            f"empty_plans="
            f"{stats['empty_plans_skipped']}"
        )

    return metadata

# =========================================================
# 9. Generate TRAIN branch supervision
# =========================================================
webqsp_branch_label_shards = (
    generate_branch_label_shards(
        webqsp_train,
        webqsp_train_plan_rows,
        "webqsp",
        WEBQSP_PLAN_SHA256,
        shard_size=
            BRANCH_LABEL_SHARD_SIZE
    )
)

cwq_branch_label_shards = (
    generate_branch_label_shards(
        cwq_train,
        cwq_train_plan_rows,
        "cwq",
        CWQ_PLAN_SHA256,
        shard_size=
            BRANCH_LABEL_SHARD_SIZE
    )
)

# =========================================================
# 10. Combine and freeze label artifacts
# =========================================================
def combine_label_artifacts(
    dataset_name,
    shard_metadata,
    plan_sha256
):
    dataset_dir = os.path.join(
        RQ2_LABEL_DIR,
        dataset_name
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    labels_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_labels_frozen.jsonl"
    )

    groups_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_groups_frozen.jsonl"
    )

    infeasible_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_infeasible_groups_frozen.jsonl"
    )

    label_count = 0
    group_count = 0
    infeasible_count = 0

    with \
        open(
            labels_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as lf, \
        open(
            groups_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as gf, \
        open(
            infeasible_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as inf:

        for meta in sorted(
            shard_metadata,
            key=lambda x:
                x["shard_id"]
        ):
            stem = (
                f"shard_"
                f"{meta['shard_id']:04d}_"
                f"{meta['start']:06d}_"
                f"{meta['end'] - 1:06d}"
            )

            labels_path = os.path.join(
                shard_dir,
                stem + ".labels.jsonl"
            )

            groups_path = os.path.join(
                shard_dir,
                stem + ".groups.jsonl"
            )

            assert (
                sha256_file(
                    labels_path
                )
                == meta[
                    "labels_sha256"
                ]
            )

            assert (
                sha256_file(
                    groups_path
                )
                == meta[
                    "groups_sha256"
                ]
            )

            with open(
                labels_path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:
                    if line.strip():
                        lf.write(line)
                        label_count += 1

            with open(
                groups_path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:
                    if not line.strip():
                        continue

                    gf.write(line)
                    group_count += 1

                    row = json.loads(
                        line
                    )

                    if (
                        row["status"]
                        ==
                        "infeasible_all_negative"
                    ):
                        inf.write(line)
                        infeasible_count += 1

        for f in [
            lf,
            gf,
            inf
        ]:
            f.flush()
            os.fsync(f.fileno())

    os.replace(
        labels_out + ".tmp",
        labels_out
    )

    os.replace(
        groups_out + ".tmp",
        groups_out
    )

    os.replace(
        infeasible_out + ".tmp",
        infeasible_out
    )

    totals = {
        key: sum(
            int(
                m.get(
                    key,
                    0
                )
            )
            for m in shard_metadata
        )
        for key in [
            "questions",

            # NEW
            "empty_plans_skipped",

            "intermediate_groups",
            "feasible_groups",
            "infeasible_groups",
            "empty_terminated",
            "final_hop_groups",
            "decision_groups",
            "labeled_branches",
            "positive_branches",
            "negative_branches",
        ]
    }

    assert (
        label_count
        == totals[
            "labeled_branches"
        ]
    )

    assert (
        infeasible_count
        == totals[
            "infeasible_groups"
        ]
    )

    assert (
        totals[
            "labeled_branches"
        ]
        ==
        totals[
            "positive_branches"
        ]
        +
        totals[
            "negative_branches"
        ]
    )

    manifest = {
        "dataset":
            dataset_name,

        "split":
            "train",

        "code_tag":
            RQ2_CELL3_CODE_TAG,

        "plan_sha256":
            plan_sha256,

        "suffix_dp_sha256":
            SUFFIX_DP_SHA256,

        **totals,

        "labels_file":
            labels_out,

        "groups_file":
            groups_out,

        "infeasible_file":
            infeasible_out,

        "labels_sha256":
            sha256_file(
                labels_out
            ),

        "groups_sha256":
            sha256_file(
                groups_out
            ),

        "infeasible_sha256":
            sha256_file(
                infeasible_out
            ),

        "frozen_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    manifest_path = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_labels_manifest.json"
    )

    atomic_json(
        manifest,
        manifest_path
    )

    return manifest

# =========================================================
# 11. Freeze combined artifacts
# =========================================================
webqsp_branch_manifest = (
    combine_label_artifacts(
        "webqsp",
        webqsp_branch_label_shards,
        WEBQSP_PLAN_SHA256
    )
)

cwq_branch_manifest = (
    combine_label_artifacts(
        "cwq",
        cwq_branch_label_shards,
        CWQ_PLAN_SHA256
    )
)

# =========================================================
# 12. Scientific sanity checks / report
# =========================================================
def report_branch_manifest(m):
    total = (
        m["labeled_branches"]
    )

    pos = (
        m["positive_branches"]
    )

    neg = (
        m["negative_branches"]
    )

    assert total == pos + neg
    assert m["feasible_groups"] > 0
    assert pos > 0
    assert neg >= 0

    pos_rate = (
        100.0 * pos / total
        if total > 0
        else 0.0
    )

    decision_rate = (
        100.0
        * m["decision_groups"]
        / m["intermediate_groups"]
        if m[
            "intermediate_groups"
        ] > 0
        else 0.0
    )

    print("\n" + "=" * 78)

    print(
        f"{m['dataset'].upper()} "
        f"TRAIN BRANCH SUPERVISION"
    )

    print("=" * 78)

    print(
        f"Questions:                   "
        f"{m['questions']}"
    )

    print(
        f"Empty relation plans skipped:"
        f" {m['empty_plans_skipped']}"
    )

    print(
        f"Intermediate groups:         "
        f"{m['intermediate_groups']}"
    )

    print(
        f"Feasible groups:             "
        f"{m['feasible_groups']}"
    )

    print(
        f"Infeasible groups:           "
        f"{m['infeasible_groups']}"
    )

    print(
        f"Decision groups |C_h|>1:     "
        f"{m['decision_groups']}"
    )

    print(
        f"Decision-group rate:         "
        f"{decision_rate:.2f}%"
    )

    print(
        f"Final-hop groups excluded:   "
        f"{m['final_hop_groups']}"
    )

    print(
        f"Empty/terminated groups:     "
        f"{m['empty_terminated']}"
    )

    print(
        f"Labeled feasible branches:   "
        f"{total}"
    )

    print(
        f"Positive branches:           "
        f"{pos}"
    )

    print(
        f"Negative branches:           "
        f"{neg}"
    )

    print(
        f"Positive rate:               "
        f"{pos_rate:.2f}%"
    )

    print(
        f"Labels SHA256:               "
        f"{m['labels_sha256'][:16]}..."
    )

    print(
        f"Groups SHA256:               "
        f"{m['groups_sha256'][:16]}..."
    )


report_branch_manifest(
    webqsp_branch_manifest
)

report_branch_manifest(
    cwq_branch_manifest
)

# =========================================================
# 13. Verify observed empty-plan counts against frozen-plan diagnostic
# =========================================================
assert (
    webqsp_branch_manifest[
        "empty_plans_skipped"
    ]
    == 5
), (
    "Unexpected WebQSP empty-plan count."
)

assert (
    cwq_branch_manifest[
        "empty_plans_skipped"
    ]
    == 132
), (
    "Unexpected CWQ empty-plan count."
)

print("\n" + "=" * 78)
print(
    "=== RQ2 CELL 3: "
    "TRAIN BRANCH SUPERVISION FROZEN ==="
)
print("=" * 78)

print(
    "TRAIN gold answers only."
)

print(
    "Empty relation plans preserved in planner artifacts "
    "but skipped from supervision."
)

print(
    "Duplicate frozen relation plans preserved."
)

print(
    "Final-hop candidates excluded from pruning supervision."
)

print(
    "All-negative infeasible groups logged separately."
)

print(
    "Only feasible intermediate groups enter scorer labels."
)

print(
    "Gold labels never alter unpruned RoG traversal."
)

print(
    "No validation/test gold answers used."
)

print(
    "Next: label sanity analysis and "
    "class-distribution diagnostics."
)

Suffix-DP signature: (graph, rule, gold_answers)
Suffix-DP SHA256:    5e60f720cc392a9a...
[WebQSP] schema/alignment gate: PASSED | 2826 questions
[CWQ] schema/alignment gate: PASSED | 27639 questions
[WebQSP] suffix-DP terminal gate: PASSED | checked=63
[CWQ] suffix-DP terminal gate: PASSED | checked=63

=== SUFFIX-DP TRAINING-LABEL PREFLIGHT: PASSED ===

WEBQSP: 2826 TRAIN questions | 12 label shards


[webqsp] label shard 1/12: FROZEN | branches=1319 | pos=549 | neg=770 | infeasible=82 | empty_plans=1


[webqsp] label shard 2/12: FROZEN | branches=1683 | pos=739 | neg=944 | infeasible=85 | empty_plans=0


[webqsp] label shard 3/12: FROZEN | branches=1585 | pos=663 | neg=922 | infeasible=80 | empty_plans=0


[webqsp] label shard 4/12: FROZEN | branches=1466 | pos=548 | neg=918 | infeasible=86 | empty_plans=1


...

[cwq] label shard 107/111: FROZEN | branches=2503 | pos=1221 | neg=1282 | infeasible=203 | empty_plans=0


[cwq] label shard 108/111: FROZEN | branches=2175 | pos=744 | neg=1431 | infeasible=210 | empty_plans=0


[cwq] label shard 109/111: FROZEN | branches=1657 | pos=535 | neg=1122 | infeasible=195 | empty_plans=0


[cwq] label shard 110/111: FROZEN | branches=2164 | pos=617 | neg=1547 | infeasible=204 | empty_plans=0


[cwq] label shard 111/111: FROZEN | branches=1815 | pos=345 | neg=1470 | infeasible=122 | empty_plans=0

WEBQSP TRAIN BRANCH SUPERVISION
Questions:                   2826
Empty relation plans skipped: 5
Intermediate groups:         3099
Feasible groups:             2177
Infeasible groups:           922
Decision groups |C_h|>1:     2000
Decision-group rate:         64.54%
Final-hop groups excluded:   6223
Empty/terminated groups:     2369
Labeled feasible branches:   19157
Positive branches:           8021
Negative branches:           11136
Positive rate:               41.87%
Labels SHA256:               428992995f787122...
Groups SHA256:               2cf2515efa647e28...

CWQ TRAIN BRANCH SUPERVISION
Questions:                   27639
Empty relation plans skipped: 132
Intermediate groups:         57866
Feasible groups:             33837
Infeasible groups:           24029
Decision groups |C_h|>1:     24222
Decision-group rate:         41.86%
Final-hop groups excluded:   64839
Empty/term

## Training-label sanity checks and class-balance statistics

In [86]:
# Purpose:
#   1. verify frozen label/group artifacts,
#   2. quantify feasible decision groups vs singleton groups,
#   3. inspect class balance by dataset and hop,
#   4. inspect candidate-set sizes and positive multiplicity,
#   5. determine whether scorer training should use:
#         all feasible groups
#      or feasible decision groups only.
#
# NO model training occurs in this cell.
# NO validation/test gold data are used.

import os
import json
import math
from collections import Counter, defaultdict

CELL4_TAG = "rq2_label_diagnostics_v1"

# =========================================================
# 1. Required artifacts
# =========================================================
required = [
    "webqsp_branch_manifest",
    "cwq_branch_manifest",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Run Cell 3 first. Missing: "
    + ", ".join(missing)
)

# =========================================================
# 2. Streaming JSONL readers
# =========================================================
def iter_jsonl(path):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                yield json.loads(line)


def load_group_stats(manifest):
    groups = list(
        iter_jsonl(
            manifest["groups_file"]
        )
    )

    labels = list(
        iter_jsonl(
            manifest["labels_file"]
        )
    )

    return groups, labels

# =========================================================
# 3. Dataset diagnostic
# =========================================================
def diagnose_branch_supervision(
    manifest,
    dataset_name
):
    groups, labels = load_group_stats(
        manifest
    )

    # -----------------------------------------------------
    # Group categories
    # -----------------------------------------------------
    feasible = [
        g for g in groups
        if g["status"] == "feasible"
    ]

    infeasible = [
        g for g in groups
        if g["status"]
        == "infeasible_all_negative"
    ]

    final_groups = [
        g for g in groups
        if g["status"]
        == "final_hop_excluded"
    ]

    empty_groups = [
        g for g in groups
        if g["status"]
        == "empty_terminated"
    ]

    feasible_decision = [
        g for g in feasible
        if g["candidate_count"] > 1
    ]

    feasible_singleton = [
        g for g in feasible
        if g["candidate_count"] == 1
    ]

    infeasible_decision = [
        g for g in infeasible
        if g["candidate_count"] > 1
    ]

    infeasible_singleton = [
        g for g in infeasible
        if g["candidate_count"] == 1
    ]

    # -----------------------------------------------------
    # Label rows split by actual decision opportunity
    # -----------------------------------------------------
    decision_labels = [
        r for r in labels
        if r["decision_opportunity"]
    ]

    singleton_labels = [
        r for r in labels
        if not r["decision_opportunity"]
    ]

    def class_stats(rows):
        n = len(rows)
        pos = sum(
            int(r["label"])
            for r in rows
        )

        neg = n - pos

        return {
            "n": n,
            "pos": pos,
            "neg": neg,
            "pos_rate":
                pos / n if n else 0.0,
            "neg_pos_ratio":
                neg / pos
                if pos > 0 else float("inf"),
        }

    all_stats = class_stats(labels)
    decision_stats = class_stats(
        decision_labels
    )
    singleton_stats = class_stats(
        singleton_labels
    )

    # -----------------------------------------------------
    # Candidate-count distribution among feasible groups
    # -----------------------------------------------------
    candidate_counts = [
        int(g["candidate_count"])
        for g in feasible
    ]

    candidate_counter = Counter(
        candidate_counts
    )

    # -----------------------------------------------------
    # Positive-count multiplicity per feasible group
    # -----------------------------------------------------
    positive_counts = [
        int(g["positive_count"])
        for g in feasible
    ]

    positive_counter = Counter(
        positive_counts
    )

    # -----------------------------------------------------
    # Hop-wise statistics over LABEL ROWS
    # -----------------------------------------------------
    hop_rows = defaultdict(list)

    for r in labels:
        hop_rows[
            int(r["hop"])
        ].append(r)

    # -----------------------------------------------------
    # Hop-wise statistics over feasible DECISION rows
    # -----------------------------------------------------
    hop_decision_rows = defaultdict(list)

    for r in decision_labels:
        hop_decision_rows[
            int(r["hop"])
        ].append(r)

    # -----------------------------------------------------
    # Sanity invariants
    # -----------------------------------------------------
    assert (
        len(feasible)
        ==
        manifest["feasible_groups"]
    )

    assert (
        len(infeasible)
        ==
        manifest["infeasible_groups"]
    )

    assert (
        len(final_groups)
        ==
        manifest["final_hop_groups"]
    )

    assert (
        len(empty_groups)
        ==
        manifest["empty_terminated"]
    )

    assert (
        len(labels)
        ==
        manifest["labeled_branches"]
    )

    assert (
        all_stats["pos"]
        ==
        manifest["positive_branches"]
    )

    assert (
        all_stats["neg"]
        ==
        manifest["negative_branches"]
    )

    # Every branch row must come from a feasible group.
    feasible_ids = {
        g["group_id"]
        for g in feasible
    }

    assert all(
        r["group_id"] in feasible_ids
        for r in labels
    )

    # Every singleton feasible group must have exactly
    # one branch row.
    singleton_group_ids = {
        g["group_id"]
        for g in feasible_singleton
    }

    singleton_label_counts = Counter(
        r["group_id"]
        for r in singleton_labels
    )

    assert all(
        singleton_label_counts[g] == 1
        for g in singleton_group_ids
    )

    # All feasible groups must have >=1 positive.
    assert all(
        int(g["positive_count"]) >= 1
        for g in feasible
    )

    # All infeasible groups must have exactly 0 positives.
    assert all(
        int(g["positive_count"]) == 0
        for g in infeasible
    )

    # -----------------------------------------------------
    # Report
    # -----------------------------------------------------
    print("\n" + "=" * 84)
    print(
        f"{dataset_name.upper()} "
        f"TRAIN LABEL DIAGNOSTICS"
    )
    print("=" * 84)

    print("\n[GROUP SPACE]")
    print(
        f"Feasible groups:                 "
        f"{len(feasible)}"
    )
    print(
        f"  feasible decision |C_h|>1:     "
        f"{len(feasible_decision)}"
    )
    print(
        f"  feasible singleton |C_h|=1:    "
        f"{len(feasible_singleton)}"
    )

    if len(feasible) > 0:
        print(
            f"  decision share of feasible:    "
            f"{100 * len(feasible_decision) / len(feasible):.2f}%"
        )

    print(
        f"Infeasible groups:               "
        f"{len(infeasible)}"
    )
    print(
        f"  infeasible decision |C_h|>1:   "
        f"{len(infeasible_decision)}"
    )
    print(
        f"  infeasible singleton |C_h|=1:  "
        f"{len(infeasible_singleton)}"
    )

    print("\n[BRANCH LABEL SPACE]")

    def print_class_block(
        name,
        stats
    ):
        print(
            f"{name:<28}"
            f"n={stats['n']:<8} "
            f"pos={stats['pos']:<8} "
            f"neg={stats['neg']:<8} "
            f"pos%={100 * stats['pos_rate']:.2f} "
            f"neg/pos={stats['neg_pos_ratio']:.3f}"
        )

    print_class_block(
        "All feasible branches",
        all_stats
    )

    print_class_block(
        "Decision-group branches",
        decision_stats
    )

    print_class_block(
        "Singleton-group branches",
        singleton_stats
    )

    print("\n[CANDIDATE COUNT DISTRIBUTION: FEASIBLE GROUPS]")

    for k in sorted(candidate_counter):
        print(
            f"|C_h|={k:<4} "
            f"groups={candidate_counter[k]}"
        )

    print("\n[POSITIVE MULTIPLICITY: FEASIBLE GROUPS]")

    for k in sorted(positive_counter):
        print(
            f"positive_count={k:<4} "
            f"groups={positive_counter[k]}"
        )

    print("\n[CLASS BALANCE BY HOP: ALL FEASIBLE LABELS]")

    for h in sorted(hop_rows):
        s = class_stats(
            hop_rows[h]
        )

        print(
            f"hop={h:<2} "
            f"n={s['n']:<8} "
            f"pos={s['pos']:<8} "
            f"neg={s['neg']:<8} "
            f"pos%={100 * s['pos_rate']:.2f}"
        )

    print("\n[CLASS BALANCE BY HOP: DECISION GROUPS ONLY]")

    for h in sorted(
        hop_decision_rows
    ):
        s = class_stats(
            hop_decision_rows[h]
        )

        print(
            f"hop={h:<2} "
            f"n={s['n']:<8} "
            f"pos={s['pos']:<8} "
            f"neg={s['neg']:<8} "
            f"pos%={100 * s['pos_rate']:.2f}"
        )

    print("\n[SANITY]")
    print(
        "Manifest counts:              PASSED"
    )
    print(
        "Feasible-group linkage:       PASSED"
    )
    print(
        "Feasible groups >=1 positive: PASSED"
    )
    print(
        "Infeasible groups =0 positive:PASSED"
    )
    print(
        "Singleton row cardinality:    PASSED"
    )

    return {
        "dataset":
            dataset_name,

        "feasible_groups":
            len(feasible),

        "feasible_decision_groups":
            len(feasible_decision),

        "feasible_singleton_groups":
            len(feasible_singleton),

        "infeasible_groups":
            len(infeasible),

        "all_stats":
            all_stats,

        "decision_stats":
            decision_stats,

        "singleton_stats":
            singleton_stats,

        "candidate_counter":
            dict(candidate_counter),

        "positive_counter":
            dict(positive_counter),
    }

# =========================================================
# 4. Run diagnostics
# =========================================================
webqsp_label_diag = (
    diagnose_branch_supervision(
        webqsp_branch_manifest,
        "webqsp"
    )
)

cwq_label_diag = (
    diagnose_branch_supervision(
        cwq_branch_manifest,
        "cwq"
    )
)

print("\n" + "=" * 84)
print(
    "=== RQ2 CELL 4: "
    "TRAIN LABEL DIAGNOSTICS COMPLETE ==="
)
print("=" * 84)

print(
    "No scorer-training choice has been made yet."
)

print(
    "Next decision: whether BCE training should use "
    "all feasible groups or feasible decision groups only."
)


WEBQSP TRAIN LABEL DIAGNOSTICS

[GROUP SPACE]
Feasible groups:                 2177
  feasible decision |C_h|>1:     1457
  feasible singleton |C_h|=1:    720
  decision share of feasible:    66.93%
Infeasible groups:               922
  infeasible decision |C_h|>1:   543
  infeasible singleton |C_h|=1:  379

[BRANCH LABEL SPACE]
All feasible branches       n=19157    pos=8021     neg=11136    pos%=41.87 neg/pos=1.388
Decision-group branches     n=18437    pos=7301     neg=11136    pos%=39.60 neg/pos=1.525
Singleton-group branches    n=720      pos=720      neg=0        pos%=100.00 neg/pos=0.000

[CANDIDATE COUNT DISTRIBUTION: FEASIBLE GROUPS]
|C_h|=1    groups=720
|C_h|=2    groups=261
|C_h|=3    groups=167
|C_h|=4    groups=145
|C_h|=5    groups=73
|C_h|=6    groups=80
|C_h|=7    groups=54
|C_h|=8    groups=62
|C_h|=9    groups=34
|C_h|=10   groups=45
|C_h|=11   groups=34
|C_h|=12   groups=33
|C_h|=13   groups=28
|C_h|=14   groups=24
|C_h|=15   groups=27
|C_h|=16   groups=24
|C_h|=1

In [94]:
# ======================================================================
# RECOVER / BIND EXISTING FROZEN VALIDATION PLAN ROWS
# ======================================================================
#
# Purpose:
#   Find the already-existing frozen validation relation plans from RQ1.
#
# We DO NOT regenerate any plans.
# We DO NOT call the planner.
# We only identify and verify existing objects.
# ======================================================================

import json
from collections.abc import Sequence


def looks_like_plan_row(row):
    """
    A frozen plan row should contain:
      - predicted_paths
      - id
    source_index is strongly preferred.
    """
    if not isinstance(row, dict):
        return False

    return (
        "predicted_paths" in row
        and
        "id" in row
    )


def inspect_plan_candidate(
    obj,
    expected_len,
    dataset,
    name
):
    """
    Verify whether obj behaves like the frozen validation-plan rows.
    """

    # Must support len()
    try:
        n = len(obj)
    except Exception:
        return False, None

    if n != expected_len:
        return False, None

    # Must support indexing
    try:
        first = obj[0]
        middle = obj[n // 2]
        last = obj[n - 1]
    except Exception:
        return False, None

    for row in [first, middle, last]:
        if not looks_like_plan_row(row):
            return False, None

    # Verify ID alignment against actual validation dataset
    check_indices = [
        0,
        n // 2,
        n - 1,
    ]

    try:
        for i in check_indices:

            row = obj[i]
            rec = dataset[i]

            if str(row["id"]) != str(rec["id"]):
                return False, None

            if (
                "source_index" in row
                and
                int(row["source_index"]) != i
            ):
                return False, None

    except Exception:
        return False, None

    return True, {
        "name": name,
        "length": n,
        "sample_id": str(first["id"]),
        "sample_plans": first["predicted_paths"],
    }


def find_validation_plan_object(
    dataset,
    expected_len,
    dataset_name
):
    matches = []

    # --------------------------------------------------------------
    # Search all notebook globals
    # --------------------------------------------------------------
    for name, obj in list(globals().items()):

        # Skip obvious irrelevant namespaces / modules
        if name.startswith("_"):
            continue

        ok, info = inspect_plan_candidate(
            obj,
            expected_len,
            dataset,
            name
        )

        if ok:
            matches.append(
                (name, obj, info)
            )

    print("\n" + "=" * 80)
    print(
        f"{dataset_name.upper()} "
        f"VALIDATION PLAN OBJECT SEARCH"
    )
    print("=" * 80)

    if len(matches) == 0:

        print(
            "No matching frozen validation-plan object "
            "was found in current globals."
        )

        return None, None

    for name, _, info in matches:
        print(
            f"Candidate: {name}"
        )
        print(
            f"  rows:       {info['length']}"
        )
        print(
            f"  sample id:  {info['sample_id']}"
        )
        print(
            f"  sample plan:{info['sample_plans']}"
        )

    # Prefer names containing dataset + val terminology
    def preference(item):
        name = item[0].lower()

        score = 0

        if dataset_name.lower() in name:
            score += 10

        if "val" in name:
            score += 5

        if "plan" in name:
            score += 5

        if "frozen" in name:
            score += 2

        if "row" in name:
            score += 1

        return score

    matches.sort(
        key=preference,
        reverse=True
    )

    selected_name, selected_obj, _ = (
        matches[0]
    )

    print(
        f"\nAUTO-SELECTED: {selected_name}"
    )

    return (
        selected_obj,
        selected_name
    )


# ======================================================================
# 1. Find WebQSP frozen validation plans
# ======================================================================

(
    webqsp_val_plan_rows,
    WEBQSP_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=webqsp_val,
    expected_len=246,
    dataset_name="webqsp"
)


# ======================================================================
# 2. Find CWQ frozen validation plans
# ======================================================================

(
    cwq_val_plan_rows,
    CWQ_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=cwq_val,
    expected_len=3519,
    dataset_name="cwq"
)


# ======================================================================
# 3. Hard gate
# ======================================================================

assert webqsp_val_plan_rows is not None, (
    "WebQSP frozen validation plans were not found "
    "in current notebook memory."
)

assert cwq_val_plan_rows is not None, (
    "CWQ frozen validation plans were not found "
    "in current notebook memory."
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


# ======================================================================
# 4. Full alignment verification
# ======================================================================

def verify_full_plan_alignment(
    dataset,
    plan_rows,
    dataset_name
):

    zero_plan_questions = 0
    total_plans = 0

    for i in range(
        len(dataset)
    ):

        rec = dataset[i]
        row = plan_rows[i]

        assert (
            str(rec["id"])
            ==
            str(row["id"])
        ), (
            f"{dataset_name}: ID mismatch "
            f"at index {i}"
        )

        if "source_index" in row:
            assert (
                int(
                    row[
                        "source_index"
                    ]
                )
                == i
            )

        plans = row[
            "predicted_paths"
        ]

        assert isinstance(
            plans,
            (list, tuple)
        )

        if len(plans) == 0:
            zero_plan_questions += 1

        total_plans += len(plans)

    print(
        f"\n[{dataset_name}] "
        "FULL VALIDATION PLAN ALIGNMENT: PASSED"
    )

    print(
        "Questions:",
        len(dataset)
    )

    print(
        "Total predicted plans:",
        total_plans
    )

    print(
        "Questions with 0 plans:",
        zero_plan_questions
    )


verify_full_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_full_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)


print("\n" + "=" * 80)
print(
    "=== FROZEN VALIDATION PLAN OBJECTS RECOVERED ==="
)
print("=" * 80)

print(
    "WebQSP:",
    WEBQSP_VAL_PLAN_VAR
)

print(
    "CWQ:   ",
    CWQ_VAL_PLAN_VAR
)

print(
    "\nNo relation plans regenerated."
)

print(
    "You can now rerun Cell 6."
)


WEBQSP VALIDATION PLAN OBJECT SEARCH
Candidate: webqsp_val_planning
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]

AUTO-SELECTED: webqsp_val_planning

CWQ VALIDATION PLAN OBJECT SEARCH
Candidate: cwq_val_planning
  rows:       3519
  sample id:  WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  sample plan:[['government.government_position_held.office_position_or_title', 'government.politician.government_positions_held'], ['location.location.containedby', 'people.person.nationality'], ['government.government_position_held.office_position_or_title', 'government.government_position_held.office_holder']]

AUTO-SELECTED: cwq_val_planning

[WebQSP] FULL VALIDATION PLAN ALIGNMENT: PASSED
Questions: 246
Total predicted plans: 721
Questions with 0 plans: 0

[CWQ] FULL VALIDATION PLAN ALIGNMENT: PASSED
Questions: 3519
Total predicte

## Defining AFP Feature Extraction

In [92]:
# ======================================================================
# 3.5 AFP FEATURE EXTRACTION — FINAL SPECIFICATION v2
# ======================================================================
#
# Revision reason:
#   A large fraction of candidate nodes are unresolved Freebase MIDs:
#       WebQSP: 58.44%
#       CWQ:    46.14%
#
# Policy:
#   - NEVER embed raw Freebase MIDs as natural-language text.
#   - Unresolved entity -> zero semantic embedding.
#   - Explicit availability indicators distinguish missing semantics
#     from a genuine cosine similarity near zero.
#   - Prefix semantic mean uses readable entities only.
#   - No external MID->name lookup is introduced.
#
# No model training occurs here.
# No gold-answer / suffix-DP information is available to this extractor.
# No future candidate neighborhood is inspected.
# ======================================================================

import re
import json
import math
import hashlib
from collections import Counter
from typing import Optional, Dict

import numpy as np


# ======================================================================
# 1. Freeze FINAL feature policy
# ======================================================================

AFP_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"

AFP_TRAIN_DECISION_ONLY = True
AFP_EXCLUDE_SINGLETON_FEASIBLE = True

AFP_USE_FUTURE_NEIGHBORHOOD = False
AFP_USE_GOLD_AS_FEATURE = False
AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE = False
AFP_USE_KGE_CORE = False

AFP_USE_EXTERNAL_ENTITY_RESOLVER = False
AFP_MASK_UNRESOLVED_ENTITY_IDS = True

AFP_SEMANTIC_ENCODER_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

AFP_SEMANTIC_EXPECTED_DIM = 384


# ======================================================================
# 2. Raw Freebase-ID detection
# ======================================================================

FREEBASE_ID_RE = re.compile(
    r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
)


def is_raw_freebase_id(x):
    if x is None:
        return False

    return bool(
        FREEBASE_ID_RE.match(
            str(x).strip()
        )
    )


def has_readable_entity_surface(
    entity,
    entity_name_map: Optional[Dict] = None
):
    # Verified resolver can be supplied later only if one exists.
    if (
        entity_name_map is not None
        and entity in entity_name_map
    ):
        resolved = entity_name_map[entity]

        return (
            resolved is not None
            and str(resolved).strip() != ""
            and not is_raw_freebase_id(resolved)
        )

    return (
        entity is not None
        and str(entity).strip() != ""
        and not is_raw_freebase_id(entity)
    )


# ======================================================================
# 3. Text normalization
# ======================================================================

def normalize_surface_text(x):
    if x is None:
        return ""

    text = str(x).strip()

    text = text.replace("_", " ")
    text = text.replace(".", " ")
    text = text.replace("/", " ")

    text = re.sub(
        r"(?<=[a-z])(?=[A-Z])",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().lower()


def relation_surface_text(relation):
    return normalize_surface_text(
        relation
    )


def entity_surface_text(
    entity,
    entity_name_map=None
):
    if (
        entity_name_map is not None
        and entity in entity_name_map
    ):
        resolved = entity_name_map[
            entity
        ]

        if (
            resolved is not None
            and not is_raw_freebase_id(
                resolved
            )
        ):
            return normalize_surface_text(
                resolved
            )

    # CRITICAL:
    # never convert raw MID to MiniLM text.
    if is_raw_freebase_id(entity):
        return ""

    return normalize_surface_text(
        entity
    )


def relation_sequence_text(relations):
    if not relations:
        return ""

    return " ; ".join(
        relation_surface_text(r)
        for r in relations
    )


# ======================================================================
# 4. Vector helpers
# ======================================================================

def safe_l2_normalize(
    x,
    eps=1e-12
):
    x = np.asarray(
        x,
        dtype=np.float32
    )

    norm = float(
        np.linalg.norm(x)
    )

    if norm <= eps:
        return np.zeros_like(
            x,
            dtype=np.float32
        )

    return (
        x / norm
    ).astype(
        np.float32
    )


def safe_cosine(a, b):
    a = safe_l2_normalize(a)
    b = safe_l2_normalize(b)

    if (
        not np.any(a)
        or not np.any(b)
    ):
        return 0.0

    return float(
        np.clip(
            np.dot(a, b),
            -1.0,
            1.0
        )
    )


def mean_embedding(
    embeddings,
    dim
):
    if len(embeddings) == 0:
        return np.zeros(
            dim,
            dtype=np.float32
        )

    x = np.mean(
        np.asarray(
            embeddings,
            dtype=np.float32
        ),
        axis=0
    )

    return safe_l2_normalize(x)


# ======================================================================
# 5. Safe entity embedding
# ======================================================================
#
# Raw MID:
#     embedding = 0
#     available = 0
#
# Readable entity:
#     MiniLM embedding
#     available = 1
# ======================================================================

def get_safe_entity_embedding(
    semantic_encoder,
    entity,
    entity_name_map=None
):
    available = (
        has_readable_entity_surface(
            entity,
            entity_name_map
        )
    )

    if not available:
        return (
            np.zeros(
                semantic_encoder.dim,
                dtype=np.float32
            ),
            0.0
        )

    text = entity_surface_text(
        entity,
        entity_name_map
    )

    emb = semantic_encoder.get(
        "entity",
        entity,
        text
    )

    return (
        np.asarray(
            emb,
            dtype=np.float32
        ),
        1.0
    )


# ======================================================================
# 6. FINAL feature schema
# ======================================================================

AFP_SEMANTIC_FEATURE_NAMES = [

    # Candidate semantics
    "sem_q_candidate",
    "sem_candidate_surface_available",

    # Current entity semantics
    "sem_q_current_entity",
    "sem_current_surface_available",

    # Relation/plan semantics
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",

    # Candidate ↔ current relation
    "sem_candidate_current_relation",
]


AFP_PATH_FEATURE_NAMES = [

    # Prefix semantic context
    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",

    # NEW: fraction of readable entities in prefix
    "path_prefix_surface_fraction",

    # Symbolic/path-history features
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",
]


AFP_STRUCTURAL_FEATURE_NAMES = [

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",
]


AFP_PROGRESS_FEATURE_NAMES = [

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]


AFP_FEATURE_NAMES = (
    AFP_SEMANTIC_FEATURE_NAMES
    + AFP_PATH_FEATURE_NAMES
    + AFP_STRUCTURAL_FEATURE_NAMES
    + AFP_PROGRESS_FEATURE_NAMES
)


AFP_FEATURE_DIM = len(
    AFP_FEATURE_NAMES
)

assert AFP_FEATURE_DIM == 27


# ======================================================================
# 7. Feature-group slices
# ======================================================================

_n_sem = len(
    AFP_SEMANTIC_FEATURE_NAMES
)

_n_path = len(
    AFP_PATH_FEATURE_NAMES
)

_n_struct = len(
    AFP_STRUCTURAL_FEATURE_NAMES
)


AFP_FEATURE_GROUP_SLICES = {

    "semantic": (
        0,
        _n_sem
    ),

    "path": (
        _n_sem,
        _n_sem + _n_path
    ),

    "structural": (
        _n_sem + _n_path,
        _n_sem + _n_path + _n_struct
    ),

    "progress": (
        _n_sem + _n_path + _n_struct,
        AFP_FEATURE_DIM
    ),
}


# ======================================================================
# 8. Feature specification fingerprint
# ======================================================================

AFP_FEATURE_SPEC = {

    "version":
        AFP_FEATURE_VERSION,

    "semantic_encoder":
        AFP_SEMANTIC_ENCODER_NAME,

    "semantic_encoder_dim":
        AFP_SEMANTIC_EXPECTED_DIM,

    "feature_dim":
        AFP_FEATURE_DIM,

    "decision_only_training":
        AFP_TRAIN_DECISION_ONLY,

    "exclude_singleton_feasible":
        AFP_EXCLUDE_SINGLETON_FEASIBLE,

    "mask_raw_freebase_ids":
        AFP_MASK_UNRESOLVED_ENTITY_IDS,

    "external_entity_resolver":
        AFP_USE_EXTERNAL_ENTITY_RESOLVER,

    "future_neighborhood":
        AFP_USE_FUTURE_NEIGHBORHOOD,

    "gold_features":
        AFP_USE_GOLD_AS_FEATURE,

    "suffix_dp_feature":
        AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE,

    "kge_core":
        AFP_USE_KGE_CORE,

    "feature_names":
        AFP_FEATURE_NAMES,
}


AFP_FEATURE_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_FEATURE_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()


# ======================================================================
# 9. FINAL candidate feature extractor
# ======================================================================

def extract_afp_candidate_features(
    question_id,
    question,
    plan,
    hop,
    candidates,
    candidate_index,
    semantic_encoder,
    entity_name_map=None
):

    plan = list(plan)
    L = len(plan)

    assert L >= 2

    assert (
        0 <= hop < L - 1
    )

    n_candidates = len(
        candidates
    )

    assert n_candidates > 1

    cand = candidates[
        candidate_index
    ]

    prefix_entities = list(
        cand["prefix_entities"]
    )

    assert len(
        prefix_entities
    ) >= 1

    current_entity = (
        prefix_entities[-1]
    )

    candidate_entity = (
        cand["candidate_entity"]
    )

    parent_prefix_index = int(
        cand["parent_prefix_index"]
    )

    current_relation = (
        plan[hop]
    )

    remaining_suffix = (
        plan[hop + 1:]
    )


    # ==================================================================
    # Reusable text embeddings
    # ==================================================================

    q_emb = semantic_encoder.get(
        "question",
        question_id,
        str(question)
    )


    (
        current_entity_emb,
        current_surface_available
    ) = get_safe_entity_embedding(
        semantic_encoder,
        current_entity,
        entity_name_map
    )


    (
        candidate_entity_emb,
        candidate_surface_available
    ) = get_safe_entity_embedding(
        semantic_encoder,
        candidate_entity,
        entity_name_map
    )


    current_relation_text = (
        relation_surface_text(
            current_relation
        )
    )

    current_relation_emb = (
        semantic_encoder.get(
            "relation",
            current_relation,
            current_relation_text
        )
    )


    full_plan_text = (
        relation_sequence_text(
            plan
        )
    )

    full_plan_key = "||".join(
        str(x)
        for x in plan
    )

    full_plan_emb = semantic_encoder.get(
        "plan",
        full_plan_key,
        full_plan_text
    )


    suffix_text = (
        relation_sequence_text(
            remaining_suffix
        )
    )

    suffix_key = "||".join(
        str(x)
        for x in remaining_suffix
    )

    suffix_emb = semantic_encoder.get(
        "suffix",
        suffix_key,
        suffix_text
    )


    # ==================================================================
    # A. SEMANTIC FEATURES (8)
    # ==================================================================

    f_sem = np.asarray(
        [
            safe_cosine(
                q_emb,
                candidate_entity_emb
            ),

            candidate_surface_available,

            safe_cosine(
                q_emb,
                current_entity_emb
            ),

            current_surface_available,

            safe_cosine(
                q_emb,
                current_relation_emb
            ),

            safe_cosine(
                q_emb,
                full_plan_emb
            ),

            safe_cosine(
                q_emb,
                suffix_emb
            ),

            safe_cosine(
                candidate_entity_emb,
                current_relation_emb
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # B. PATH-CONTEXT FEATURES (7)
    # ==================================================================

    readable_prefix_embeddings = []

    readable_prefix_count = 0

    for entity in prefix_entities:

        (
            entity_emb,
            entity_available
        ) = get_safe_entity_embedding(
            semantic_encoder,
            entity,
            entity_name_map
        )

        if entity_available > 0:
            readable_prefix_embeddings.append(
                entity_emb
            )

            readable_prefix_count += 1


    prefix_mean_emb = mean_embedding(
        readable_prefix_embeddings,
        semantic_encoder.dim
    )


    prefix_len = len(
        prefix_entities
    )


    prefix_surface_fraction = (
        readable_prefix_count
        / max(
            1,
            prefix_len
        )
    )


    candidate_occurrences = sum(
        1
        for e in prefix_entities
        if e == candidate_entity
    )


    unique_prefix_entities = len(
        set(
            prefix_entities
        )
    )


    previous_relations = (
        plan[:hop]
    )


    relation_repeat_before = sum(
        1
        for r in previous_relations
        if r == current_relation
    )


    f_path = np.asarray(
        [
            safe_cosine(
                q_emb,
                prefix_mean_emb
            ),

            safe_cosine(
                candidate_entity_emb,
                prefix_mean_emb
            ),

            float(
                prefix_surface_fraction
            ),

            float(
                candidate_occurrences > 0
            ),

            float(
                candidate_occurrences
                / max(
                    1,
                    prefix_len
                )
            ),

            float(
                unique_prefix_entities
                / max(
                    1,
                    prefix_len
                )
            ),

            float(
                relation_repeat_before
                / max(
                    1,
                    hop
                )
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # C. CURRENT-FRONTIER STRUCTURAL FEATURES (8)
    # ==================================================================
    #
    # Only C_h is used.
    # No Adj(candidate), degree, or next-hop expansion.
    # ==================================================================

    endpoint_counter = Counter(
        c["candidate_entity"]
        for c in candidates
    )


    parent_counter = Counter(
        int(
            c[
                "parent_prefix_index"
            ]
        )
        for c in candidates
    )


    unique_candidate_entities = len(
        endpoint_counter
    )


    contributing_parents = len(
        parent_counter
    )


    parent_fanout = int(
        parent_counter[
            parent_prefix_index
        ]
    )


    endpoint_multiplicity = int(
        endpoint_counter[
            candidate_entity
        ]
    )


    duplicate_endpoint_ratio = (
        1.0
        -
        (
            unique_candidate_entities
            / n_candidates
        )
    )


    f_struct = np.asarray(
        [
            math.log1p(
                n_candidates
            ),

            math.log1p(
                unique_candidate_entities
            ),

            math.log1p(
                contributing_parents
            ),

            math.log1p(
                parent_fanout
            ),

            float(
                parent_fanout
                / n_candidates
            ),

            math.log1p(
                endpoint_multiplicity
            ),

            float(
                endpoint_multiplicity
                / n_candidates
            ),

            float(
                duplicate_endpoint_ratio
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # D. PROGRESS FEATURES (4)
    # ==================================================================

    remaining_hops = (
        L - hop - 1
    )


    hop_fraction = (
        hop
        / max(
            1,
            L - 1
        )
    )


    remaining_fraction = (
        remaining_hops
        / L
    )


    penultimate_indicator = float(
        remaining_hops == 1
    )


    f_prog = np.asarray(
        [
            float(
                hop_fraction
            ),

            float(
                remaining_fraction
            ),

            float(
                math.log1p(L)
            ),

            penultimate_indicator,
        ],
        dtype=np.float32
    )


    # ==================================================================
    # FINAL VECTOR
    # ==================================================================

    x = np.concatenate(
        [
            f_sem,
            f_path,
            f_struct,
            f_prog
        ]
    ).astype(
        np.float32
    )


    assert x.shape == (
        AFP_FEATURE_DIM,
    )

    assert np.all(
        np.isfinite(x)
    )

    return x


# ======================================================================
# 10. Group extractor
# ======================================================================

def extract_afp_group_features(
    question_id,
    question,
    plan,
    hop,
    candidate_rows,
    semantic_encoder,
    entity_name_map=None
):

    assert len(
        candidate_rows
    ) > 1


    candidates = []

    for row in candidate_rows:

        candidates.append(
            {
                "prefix_entities":
                    list(
                        row[
                            "prefix_entities"
                        ]
                    ),

                "candidate_entity":
                    row[
                        "candidate_entity"
                    ],

                "parent_prefix_index":
                    int(
                        row[
                            "parent_prefix_index"
                        ]
                    ),
            }
        )


    X = np.vstack(
        [
            extract_afp_candidate_features(
                question_id=
                    question_id,

                question=
                    question,

                plan=
                    plan,

                hop=
                    hop,

                candidates=
                    candidates,

                candidate_index=
                    i,

                semantic_encoder=
                    semantic_encoder,

                entity_name_map=
                    entity_name_map
            )

            for i in range(
                len(candidates)
            )
        ]
    ).astype(
        np.float32
    )


    assert X.shape == (
        len(candidates),
        AFP_FEATURE_DIM
    )

    return X


# ======================================================================
# 11. Feature-group helper
# ======================================================================

def split_afp_feature_groups(X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    result = {}

    for (
        group_name,
        (start, end)
    ) in (
        AFP_FEATURE_GROUP_SLICES
        .items()
    ):

        result[
            group_name
        ] = X[
            ...,
            start:end
        ]

    return result


# ======================================================================
# 12. SANITY TEST WITH RAW FREEBASE IDs
# ======================================================================

# Reuse MockSemanticEncoder from previous Cell 5.
# If unavailable, define a tiny one.

if "MockSemanticEncoder" not in globals():

    class MockSemanticEncoder:

        def __init__(
            self,
            dim=32
        ):
            self.dim = dim

        def get(
            self,
            namespace,
            identifier,
            raw_text
        ):
            # Raw MID must NEVER arrive here as entity text.
            if (
                namespace == "entity"
                and is_raw_freebase_id(
                    raw_text
                )
            ):
                raise AssertionError(
                    "Raw Freebase MID reached semantic encoder."
                )

            token = (
                f"{namespace}|"
                f"{identifier}|"
                f"{raw_text}"
            )

            digest = hashlib.sha256(
                token.encode(
                    "utf-8"
                )
            ).digest()

            seed = int.from_bytes(
                digest[:8],
                "little"
            )

            rng = (
                np.random.default_rng(
                    seed
                )
            )

            vec = rng.normal(
                size=self.dim
            ).astype(
                np.float32
            )

            return safe_l2_normalize(
                vec
            )


_mock_encoder = (
    MockSemanticEncoder(
        dim=32
    )
)


_mock_rows = [

    {
        "prefix_entities": [
            "Natalie Portman"
        ],
        "candidate_entity":
            "m.0k3qzz",
        "parent_prefix_index":
            0,
    },

    {
        "prefix_entities": [
            "Natalie Portman"
        ],
        "candidate_entity":
            "Canada",
        "parent_prefix_index":
            0,
    },
]


_mock_X = (
    extract_afp_group_features(

        question_id=
            "mock-mid-test",

        question=
            "where was the person born",

        plan=[
            "people.person.place_of_birth",
            "location.location.containedby",
        ],

        hop=0,

        candidate_rows=
            _mock_rows,

        semantic_encoder=
            _mock_encoder
    )
)


assert _mock_X.shape == (
    2,
    27
)


# ----------------------------------------------------------
# Raw-MID candidate:
#
# feature 0 = sem_q_candidate -> must be 0
# feature 1 = candidate_surface_available -> must be 0
# feature 7 = sem_candidate_current_relation -> must be 0
# feature 9 = path_candidate_prefix_entity_mean -> must be 0
# ----------------------------------------------------------

assert np.isclose(
    _mock_X[0, 0],
    0.0
)

assert np.isclose(
    _mock_X[0, 1],
    0.0
)

assert np.isclose(
    _mock_X[0, 7],
    0.0
)

assert np.isclose(
    _mock_X[0, 9],
    0.0
)


# Readable candidate must report availability=1
assert np.isclose(
    _mock_X[1, 1],
    1.0
)


assert np.all(
    np.isfinite(
        _mock_X
    )
)


# ======================================================================
# 13. FINAL REPORT
# ======================================================================

print(
    "\n" + "=" * 84
)

print(
    "=== RQ2 CELL 5: "
    "AFP FEATURE EXTRACTION v2 FROZEN ==="
)

print("=" * 84)

print(
    f"Semantic features:     "
    f"{len(AFP_SEMANTIC_FEATURE_NAMES)}"
)

print(
    f"Path features:         "
    f"{len(AFP_PATH_FEATURE_NAMES)}"
)

print(
    f"Structural features:   "
    f"{len(AFP_STRUCTURAL_FEATURE_NAMES)}"
)

print(
    f"Progress features:     "
    f"{len(AFP_PROGRESS_FEATURE_NAMES)}"
)

print(
    f"TOTAL FEATURE DIM:     "
    f"{AFP_FEATURE_DIM}"
)

print(
    f"Feature-spec SHA256:   "
    f"{AFP_FEATURE_SPEC_SHA256}"
)


print(
    "\nEntity semantic policy:"
)

print(
    "  Readable entity name       -> frozen MiniLM embedding"
)

print(
    "  Raw Freebase MID           -> ZERO embedding"
)

print(
    "  Candidate availability     -> explicit feature"
)

print(
    "  Current availability       -> explicit feature"
)

print(
    "  Prefix semantic mean       -> readable entities only"
)

print(
    "  Prefix readable fraction   -> explicit feature"
)


print(
    "\nMethodological safeguards:"
)

print(
    "  Raw MID sent to MiniLM:          NO"
)

print(
    "  External MID resolver added:     NO"
)

print(
    "  Gold-answer feature:             NO"
)

print(
    "  Suffix-reachability feature:     NO"
)

print(
    "  Future-neighborhood inspection:  NO"
)

print(
    "  Candidate degree lookup:         NO"
)

print(
    "  Singleton scorer invocation:     NO"
)

print(
    "  Training groups: feasible decisions only"
)


print(
    "\nRaw-MID masking synthetic gate: PASSED"
)

print(
    "\nThis v2 specification supersedes "
    "afp_features_v1."
)

print(
    "Next: build/cache TRAIN + VALIDATION "
    "feature datasets."
)


=== RQ2 CELL 5: AFP FEATURE EXTRACTION v2 FROZEN ===
Semantic features:     8
Path features:         7
Structural features:   8
Progress features:     4
TOTAL FEATURE DIM:     27
Feature-spec SHA256:   738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86

Entity semantic policy:
  Readable entity name       -> frozen MiniLM embedding
  Raw Freebase MID           -> ZERO embedding
  Candidate availability     -> explicit feature
  Current availability       -> explicit feature
  Prefix semantic mean       -> readable entities only
  Prefix readable fraction   -> explicit feature

Methodological safeguards:
  Raw MID sent to MiniLM:          NO
  External MID resolver added:     NO
  Gold-answer feature:             NO
  Suffix-reachability feature:     NO
  Future-neighborhood inspection:  NO
  Candidate degree lookup:         NO
  Singleton scorer invocation:     NO
  Training groups: feasible decisions only

Raw-MID masking synthetic gate: PASSED

This v2 specification su

In [89]:
# =========================================================
# Semantic surface-form sanity check
# =========================================================

def inspect_entity_surface_forms(
    branch_manifest,
    dataset_name,
    n=20
):
    rows = []

    with open(
        branch_manifest["labels_file"],
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

            if len(rows) >= n:
                break

    print("\n" + "=" * 72)
    print(f"{dataset_name.upper()} ENTITY-SURFACE SANITY")
    print("=" * 72)

    for i, row in enumerate(rows[:n]):
        candidate = row["candidate_entity"]

        prefix_last = (
            row["prefix_entities"][-1]
            if row["prefix_entities"]
            else None
        )

        print(
            f"{i:02d} | "
            f"current={prefix_last} | "
            f"candidate={candidate}"
        )

    print()


inspect_entity_surface_forms(
    webqsp_branch_manifest,
    "webqsp",
    n=20
)

inspect_entity_surface_forms(
    cwq_branch_manifest,
    "cwq",
    n=20
)


WEBQSP ENTITY-SURFACE SANITY
00 | current=Justin Bieber | candidate=Canada
01 | current=Natalie Portman | candidate=m.0k3qzz
02 | current=Natalie Portman | candidate=m.03jt5jq
03 | current=Natalie Portman | candidate=m.0nfnhrj
04 | current=Natalie Portman | candidate=m.040myw2
05 | current=Natalie Portman | candidate=m.04dcjy9
06 | current=Natalie Portman | candidate=m.0k3r0b
07 | current=Natalie Portman | candidate=m.0cs2bt7
08 | current=Natalie Portman | candidate=m.0j_xb7
09 | current=Natalie Portman | candidate=m.0nh4bnk
10 | current=Natalie Portman | candidate=m.02vb_fz
11 | current=Natalie Portman | candidate=m.0k7msn
12 | current=Natalie Portman | candidate=m.0109rjwm
13 | current=Natalie Portman | candidate=m.0jtns5
14 | current=Natalie Portman | candidate=m.0cccx3m
15 | current=Natalie Portman | candidate=m.07zm_x0
16 | current=Natalie Portman | candidate=m.0jwjgq
17 | current=Natalie Portman | candidate=m.0k3qy8
18 | current=Natalie Portman | candidate=g.11b7qqspgv
19 | curr

In [90]:
# ======================================================================
# AUDIT ENTITY SURFACE-FORM COVERAGE
# ======================================================================
#
# Purpose:
#   1. quantify how often candidate/current entities are raw Freebase IDs,
#   2. inspect whether NetworkX graph nodes already contain useful
#      name/label attributes,
#   3. inspect available graph-record structure before deciding how
#      entity surface resolution should be implemented.
#
# No feature cache is constructed here.
# No gold/test information is used.
# ======================================================================

import re
import json
from collections import Counter

FREEBASE_ID_RE = re.compile(
    r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
)


def is_raw_freebase_id(x):
    if x is None:
        return False

    return bool(
        FREEBASE_ID_RE.match(
            str(x).strip()
        )
    )


# ----------------------------------------------------------------------
# 1. Stream decision-group branch rows only
# ----------------------------------------------------------------------
def audit_entity_id_rate(
    manifest,
    dataset_name
):
    total = 0

    candidate_raw = 0
    current_raw = 0

    both_named = 0
    any_raw = 0

    raw_examples = []

    with open(
        manifest["labels_file"],
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            # Scorer is trained only on actual decision groups.
            if not row.get(
                "decision_opportunity",
                False
            ):
                continue

            total += 1

            candidate = row[
                "candidate_entity"
            ]

            current = row[
                "prefix_entities"
            ][-1]

            cand_is_raw = (
                is_raw_freebase_id(
                    candidate
                )
            )

            curr_is_raw = (
                is_raw_freebase_id(
                    current
                )
            )

            candidate_raw += int(
                cand_is_raw
            )

            current_raw += int(
                curr_is_raw
            )

            if (
                not cand_is_raw
                and not curr_is_raw
            ):
                both_named += 1

            if (
                cand_is_raw
                or curr_is_raw
            ):
                any_raw += 1

                if len(
                    raw_examples
                ) < 10:

                    raw_examples.append({
                        "source_index":
                            row[
                                "source_index"
                            ],

                        "question_id":
                            row[
                                "question_id"
                            ],

                        "current":
                            current,

                        "candidate":
                            candidate,
                    })

    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"DECISION-BRANCH ENTITY AUDIT"
    )

    print(
        "=" * 78
    )

    print(
        f"Decision branches:           "
        f"{total}"
    )

    print(
        f"Raw candidate IDs:           "
        f"{candidate_raw} "
        f"({100*candidate_raw/max(1,total):.2f}%)"
    )

    print(
        f"Raw current-entity IDs:      "
        f"{current_raw} "
        f"({100*current_raw/max(1,total):.2f}%)"
    )

    print(
        f"Any raw endpoint:            "
        f"{any_raw} "
        f"({100*any_raw/max(1,total):.2f}%)"
    )

    print(
        f"Both endpoints readable:     "
        f"{both_named} "
        f"({100*both_named/max(1,total):.2f}%)"
    )

    print(
        "\nExamples:"
    )

    for x in raw_examples:
        print(x)

    return raw_examples


webqsp_raw_examples = (
    audit_entity_id_rate(
        webqsp_branch_manifest,
        "webqsp"
    )
)

cwq_raw_examples = (
    audit_entity_id_rate(
        cwq_branch_manifest,
        "cwq"
    )
)


# ======================================================================
# 2. Inspect graph representation around raw-ID examples
# ======================================================================

def inspect_graph_node_metadata(
    dataset,
    examples,
    dataset_name,
    max_examples=5
):
    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"RAW-ID GRAPH METADATA INSPECTION"
    )

    print(
        "=" * 78
    )

    checked = 0

    for ex in examples:

        if checked >= max_examples:
            break

        idx = int(
            ex["source_index"]
        )

        rec = dataset[idx]

        G = build_graph(
            rec["graph"]
        )

        for role in [
            "current",
            "candidate"
        ]:

            entity = ex[role]

            if not is_raw_freebase_id(
                entity
            ):
                continue

            print(
                f"\nsource_index={idx}"
            )

            print(
                f"role={role}"
            )

            print(
                f"entity={entity}"
            )

            if entity in G:

                attrs = dict(
                    G.nodes[
                        entity
                    ]
                )

                print(
                    "NetworkX node attrs:",
                    attrs
                )

            else:

                print(
                    "Entity not found "
                    "as NetworkX node."
                )

            checked += 1

            if checked >= max_examples:
                break


inspect_graph_node_metadata(
    webqsp_train,
    webqsp_raw_examples,
    "webqsp"
)

inspect_graph_node_metadata(
    cwq_train,
    cwq_raw_examples,
    "cwq"
)


# ======================================================================
# 3. Inspect raw source graph schema for a few affected questions
# ======================================================================

def inspect_source_graph_schema(
    dataset,
    examples,
    dataset_name,
    max_questions=2
):
    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"SOURCE GRAPH SCHEMA"
    )

    print(
        "=" * 78
    )

    seen = set()

    shown = 0

    for ex in examples:

        idx = int(
            ex["source_index"]
        )

        if idx in seen:
            continue

        seen.add(idx)

        rec = dataset[idx]

        raw_graph = rec[
            "graph"
        ]

        print(
            f"\nsource_index={idx}"
        )

        print(
            "graph python type:",
            type(raw_graph)
        )

        try:
            print(
                "graph sample:",
                raw_graph[:3]
            )
        except Exception:
            print(
                "graph sample:",
                str(raw_graph)[:1500]
            )

        shown += 1

        if shown >= max_questions:
            break


inspect_source_graph_schema(
    webqsp_train,
    webqsp_raw_examples,
    "webqsp"
)

inspect_source_graph_schema(
    cwq_train,
    cwq_raw_examples,
    "cwq"
)


print(
    "\n" + "=" * 78
)

print(
    "=== ENTITY SURFACE-FORM AUDIT COMPLETE ==="
)

print(
    "Do NOT build MiniLM feature cache yet."
)


WEBQSP DECISION-BRANCH ENTITY AUDIT
Decision branches:           18437
Raw candidate IDs:           10774 (58.44%)
Raw current-entity IDs:      0 (0.00%)
Any raw endpoint:            10774 (58.44%)
Both endpoints readable:     7663 (41.56%)

Examples:
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0k3qzz'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.03jt5jq'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0nfnhrj'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.040myw2'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.04dcjy9'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0k3r0b'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0cs2bt7'}
{'source_index': 1, '

In [91]:
# ======================================================================
# CHECK FOR EXISTING FREEBASE MID -> ENTITY NAME RESOLVER
# ======================================================================
#
# Goal:
#   Find whether the current RoG environment already contains a verified
#   MID-to-readable-name mapping.
#
# IMPORTANT:
#   We do NOT construct semantic features yet.
# ======================================================================

import os
import re
import json

MID_RE = re.compile(r"^(?:m|g)\.[A-Za-z0-9_\-]+$")


def is_mid(x):
    return bool(
        MID_RE.match(
            str(x).strip()
        )
    )


# ======================================================================
# 1. Dataset columns
# ======================================================================

print("=" * 80)
print("DATASET COLUMN AUDIT")
print("=" * 80)

print("\nWebQSP columns:")
print(webqsp_train.column_names)

print("\nCWQ columns:")
print(cwq_train.column_names)


# ======================================================================
# 2. Search notebook globals for possible entity-name dictionaries
# ======================================================================

keywords = (
    "name",
    "label",
    "entity",
    "mid",
    "id2",
    "2id",
    "alias",
)


candidate_globals = []

for var_name, obj in list(globals().items()):

    low = var_name.lower()

    if not any(
        k in low
        for k in keywords
    ):
        continue

    if isinstance(obj, dict):

        # Avoid dumping giant dictionaries
        size = len(obj)

        sample_items = list(
            obj.items()
        )[:5]

        candidate_globals.append(
            (
                var_name,
                size,
                sample_items,
            )
        )


print("\n" + "=" * 80)
print("POSSIBLE MAPPING DICTIONARIES IN NOTEBOOK")
print("=" * 80)

if not candidate_globals:
    print("No obvious dictionary candidates found.")

else:
    for name, size, sample in candidate_globals:

        print(
            f"\n{name} | size={size}"
        )

        for k, v in sample:
            print(
                "   ",
                repr(k),
                "->",
                repr(v)
            )


# ======================================================================
# 3. Test candidate dictionaries against known unresolved MIDs
# ======================================================================

KNOWN_MIDS = [
    "m.0k3qzz",
    "m.03jt5jq",
    "m.0nfnhrj",
    "m.0hqf007",
    "m.0hqf002",
]


print("\n" + "=" * 80)
print("KNOWN MID LOOKUP TEST")
print("=" * 80)


successful_resolvers = []


for name, size, _ in candidate_globals:

    obj = globals()[name]

    hits = {}

    for mid in KNOWN_MIDS:

        if mid in obj:

            value = obj[mid]

            # We only count readable mappings
            if (
                value is not None
                and str(value).strip()
                and not is_mid(value)
            ):
                hits[mid] = value

    if hits:

        successful_resolvers.append(
            name
        )

        print(
            f"\n{name}: "
            f"{len(hits)}/{len(KNOWN_MIDS)} readable hits"
        )

        for mid, value in hits.items():
            print(
                f"  {mid} -> {value}"
            )


if not successful_resolvers:
    print(
        "\nNo existing notebook dictionary resolved "
        "the sampled Freebase IDs."
    )


# ======================================================================
# 4. Search dataset columns for likely entity-name mappings
# ======================================================================

def inspect_mapping_like_columns(
    dataset,
    dataset_name,
    n_rows=3
):

    interesting = []

    for col in dataset.column_names:

        low = col.lower()

        if any(
            k in low
            for k in [
                "name",
                "label",
                "entity",
                "alias",
                "mid",
            ]
        ):
            interesting.append(col)

    print(
        "\n" + "=" * 80
    )

    print(
        f"{dataset_name.upper()} "
        f"MAPPING-LIKE DATASET COLUMNS"
    )

    print("=" * 80)

    print(
        "Candidate columns:",
        interesting
    )

    for col in interesting:

        print(
            f"\nCOLUMN: {col}"
        )

        for i in range(
            min(n_rows, len(dataset))
        ):

            value = dataset[i][col]

            text = repr(value)

            if len(text) > 1000:
                text = (
                    text[:1000]
                    + " ..."
                )

            print(
                f"row {i}: {text}"
            )


inspect_mapping_like_columns(
    webqsp_train,
    "WebQSP"
)

inspect_mapping_like_columns(
    cwq_train,
    "CWQ"
)


# ======================================================================
# 5. Final status
# ======================================================================

print(
    "\n" + "=" * 80
)

print(
    "=== EXISTING ENTITY RESOLVER AUDIT COMPLETE ==="
)

print("=" * 80)

if successful_resolvers:

    print(
        "Potential existing resolver(s):"
    )

    for x in successful_resolvers:
        print("  ", x)

    print(
        "\nDo NOT use automatically yet; "
        "we will verify coverage first."
    )

else:

    print(
        "No verified existing MID->name resolver found yet."
    )

    print(
        "If none exists, we will revise AFP semantic "
        "features rather than embedding raw MIDs."
    )

DATASET COLUMN AUDIT

WebQSP columns:
['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']

CWQ columns:
['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']

POSSIBLE MAPPING DICTIONARIES IN NOTEBOOK

webqsp_label_diag | size=10
    'dataset' -> 'webqsp'
    'feasible_groups' -> 2177
    'feasible_decision_groups' -> 1457
    'feasible_singleton_groups' -> 720
    'infeasible_groups' -> 922

cwq_label_diag | size=10
    'dataset' -> 'cwq'
    'feasible_groups' -> 33837
    'feasible_decision_groups' -> 15937
    'feasible_singleton_groups' -> 17900
    'infeasible_groups' -> 24029

KNOWN MID LOOKUP TEST

No existing notebook dictionary resolved the sampled Freebase IDs.

WEBQSP MAPPING-LIKE DATASET COLUMNS
Candidate columns: ['q_entity', 'a_entity']

COLUMN: q_entity
row 0: ['Justin Bieber']
row 1: ['Natalie Portman']
row 2: ['Grand Bahama']

COLUMN: a_entity
row 0: ['Jaxon Bieber']
row 1: ['Padmé Amidala']
row 2: ['Bahamas']

CWQ MAPPING-LIKE D

In [95]:
# ======================================================================
# CELL 6A — RECOVER / BIND EXISTING FROZEN VALIDATION PLAN ROWS
# ======================================================================
#
# Purpose:
#   Find the already-existing frozen validation relation plans from RQ1.
#
# We DO NOT regenerate any plans.
# We DO NOT call the planner.
# We only identify and verify existing objects.
# ======================================================================

import json
from collections.abc import Sequence


def looks_like_plan_row(row):
    """
    A frozen plan row should contain:
      - predicted_paths
      - id
    source_index is strongly preferred.
    """
    if not isinstance(row, dict):
        return False

    return (
        "predicted_paths" in row
        and
        "id" in row
    )


def inspect_plan_candidate(
    obj,
    expected_len,
    dataset,
    name
):
    """
    Verify whether obj behaves like the frozen validation-plan rows.
    """

    # Must support len()
    try:
        n = len(obj)
    except Exception:
        return False, None

    if n != expected_len:
        return False, None

    # Must support indexing
    try:
        first = obj[0]
        middle = obj[n // 2]
        last = obj[n - 1]
    except Exception:
        return False, None

    for row in [first, middle, last]:
        if not looks_like_plan_row(row):
            return False, None

    # Verify ID alignment against actual validation dataset
    check_indices = [
        0,
        n // 2,
        n - 1,
    ]

    try:
        for i in check_indices:

            row = obj[i]
            rec = dataset[i]

            if str(row["id"]) != str(rec["id"]):
                return False, None

            if (
                "source_index" in row
                and
                int(row["source_index"]) != i
            ):
                return False, None

    except Exception:
        return False, None

    return True, {
        "name": name,
        "length": n,
        "sample_id": str(first["id"]),
        "sample_plans": first["predicted_paths"],
    }


def find_validation_plan_object(
    dataset,
    expected_len,
    dataset_name
):
    matches = []

    # --------------------------------------------------------------
    # Search all notebook globals
    # --------------------------------------------------------------
    for name, obj in list(globals().items()):

        # Skip obvious irrelevant namespaces / modules
        if name.startswith("_"):
            continue

        ok, info = inspect_plan_candidate(
            obj,
            expected_len,
            dataset,
            name
        )

        if ok:
            matches.append(
                (name, obj, info)
            )

    print("\n" + "=" * 80)
    print(
        f"{dataset_name.upper()} "
        f"VALIDATION PLAN OBJECT SEARCH"
    )
    print("=" * 80)

    if len(matches) == 0:

        print(
            "No matching frozen validation-plan object "
            "was found in current globals."
        )

        return None, None

    for name, _, info in matches:
        print(
            f"Candidate: {name}"
        )
        print(
            f"  rows:       {info['length']}"
        )
        print(
            f"  sample id:  {info['sample_id']}"
        )
        print(
            f"  sample plan:{info['sample_plans']}"
        )

    # Prefer names containing dataset + val terminology
    def preference(item):
        name = item[0].lower()

        score = 0

        if dataset_name.lower() in name:
            score += 10

        if "val" in name:
            score += 5

        if "plan" in name:
            score += 5

        if "frozen" in name:
            score += 2

        if "row" in name:
            score += 1

        return score

    matches.sort(
        key=preference,
        reverse=True
    )

    selected_name, selected_obj, _ = (
        matches[0]
    )

    print(
        f"\nAUTO-SELECTED: {selected_name}"
    )

    return (
        selected_obj,
        selected_name
    )


# ======================================================================
# 1. Find WebQSP frozen validation plans
# ======================================================================

(
    webqsp_val_plan_rows,
    WEBQSP_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=webqsp_val,
    expected_len=246,
    dataset_name="webqsp"
)


# ======================================================================
# 2. Find CWQ frozen validation plans
# ======================================================================

(
    cwq_val_plan_rows,
    CWQ_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=cwq_val,
    expected_len=3519,
    dataset_name="cwq"
)


# ======================================================================
# 3. Hard gate
# ======================================================================

assert webqsp_val_plan_rows is not None, (
    "WebQSP frozen validation plans were not found "
    "in current notebook memory."
)

assert cwq_val_plan_rows is not None, (
    "CWQ frozen validation plans were not found "
    "in current notebook memory."
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


# ======================================================================
# 4. Full alignment verification
# ======================================================================

def verify_full_plan_alignment(
    dataset,
    plan_rows,
    dataset_name
):

    zero_plan_questions = 0
    total_plans = 0

    for i in range(
        len(dataset)
    ):

        rec = dataset[i]
        row = plan_rows[i]

        assert (
            str(rec["id"])
            ==
            str(row["id"])
        ), (
            f"{dataset_name}: ID mismatch "
            f"at index {i}"
        )

        if "source_index" in row:
            assert (
                int(
                    row[
                        "source_index"
                    ]
                )
                == i
            )

        plans = row[
            "predicted_paths"
        ]

        assert isinstance(
            plans,
            (list, tuple)
        )

        if len(plans) == 0:
            zero_plan_questions += 1

        total_plans += len(plans)

    print(
        f"\n[{dataset_name}] "
        "FULL VALIDATION PLAN ALIGNMENT: PASSED"
    )

    print(
        "Questions:",
        len(dataset)
    )

    print(
        "Total predicted plans:",
        total_plans
    )

    print(
        "Questions with 0 plans:",
        zero_plan_questions
    )


verify_full_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_full_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)


print("\n" + "=" * 80)
print(
    "=== FROZEN VALIDATION PLAN OBJECTS RECOVERED ==="
)
print("=" * 80)

print(
    "WebQSP:",
    WEBQSP_VAL_PLAN_VAR
)

print(
    "CWQ:   ",
    CWQ_VAL_PLAN_VAR
)

print(
    "\nNo relation plans regenerated."
)

print(
    "You can now rerun Cell 6."
)


WEBQSP VALIDATION PLAN OBJECT SEARCH
Candidate: webqsp_val_planning
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]
Candidate: webqsp_val_plan_rows
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]

AUTO-SELECTED: webqsp_val_plan_rows

CWQ VALIDATION PLAN OBJECT SEARCH
Candidate: cwq_val_planning
  rows:       3519
  sample id:  WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  sample plan:[['government.government_position_held.office_position_or_title', 'government.politician.government_positions_held'], ['location.location.containedby', 'people.person.nationality'], ['government.government_position_held.office_position_or_title', 'government.government_position_hel

## Build/cache train + validation feature datasets

In [97]:
# ======================================================================
# 3.6 BUILD + CACHE TRAIN AND VALIDATION AFP FEATURE DATASETS
# ======================================================================
#
# TRAIN:
#   Uses already-frozen Cell-3 supervision.
#
# VALIDATION:
#   Constructs suffix-DP labels using VALIDATION gold answers only.
#
# Scorer dataset:
#   feasible INTERMEDIATE DECISION groups only:
#       |C_h| > 1
#       and at least one positive candidate
#
# IMPORTANT:
#   - NO TEST data.
#   - NO test gold.
#   - Gold is NEVER an AFP feature.
#   - Gold labels NEVER alter unpruned RoG traversal.
#   - Final hop excluded.
#   - Singleton groups excluded from scorer dataset.
#   - Raw Freebase MID -> masked semantic representation.
#
# Validation-plan compatibility:
#   Older frozen RQ1 validation rows may not contain source_index.
#   Their already-verified positional index is used instead.
#
# Output:
#   X              [N_branches, 27]
#   y              [N_branches]
#   group_ptr      [N_groups + 1]
#   group metadata
# ======================================================================

import os
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
from tqdm import tqdm

# ======================================================================
# 1. HARD FEATURE-SPECIFICATION GATES
# ======================================================================

EXPECTED_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"
EXPECTED_FEATURE_SHA256 = (
    "738985d1232a8ac5935c397ed95eca233"
    "77bc59b4a99494547fa7779062ade86"
)

assert AFP_FEATURE_VERSION == EXPECTED_FEATURE_VERSION
assert AFP_FEATURE_SPEC_SHA256 == EXPECTED_FEATURE_SHA256
assert AFP_FEATURE_DIM == 27

assert AFP_TRAIN_DECISION_ONLY is True
assert AFP_EXCLUDE_SINGLETON_FEASIBLE is True
assert AFP_USE_FUTURE_NEIGHBORHOOD is False
assert AFP_USE_GOLD_AS_FEATURE is False
assert AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE is False
assert AFP_USE_KGE_CORE is False

required_objects = [
    "webqsp_train",
    "cwq_train",
    "webqsp_branch_manifest",
    "cwq_branch_manifest",
    "build_graph",
    "relation_valid_neighbors",
    "call_suffix_dp",
    "dp_is_reachable",
    "extract_afp_group_features",
    "FrozenMiniLMEncoder",
]

missing = [x for x in required_objects if x not in globals()]

assert not missing, (
    "Missing required previous-cell objects: "
    + ", ".join(missing)
)

print("Feature version:", AFP_FEATURE_VERSION)
print("Feature dimension:", AFP_FEATURE_DIM)
print("Feature SHA256:", AFP_FEATURE_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. RESOLVE VALIDATION DATASETS
# ======================================================================

def resolve_global(names):
    for name in names:
        if name in globals():
            return globals()[name], name

    raise RuntimeError(
        "Could not resolve any of these globals:\n"
        + "\n".join(names)
    )

webqsp_val, WEBQSP_VAL_VAR = resolve_global([
    "webqsp_val",
    "webqsp_validation",
    "webqsp_valid",
])

cwq_val, CWQ_VAL_VAR = resolve_global([
    "cwq_val",
    "cwq_validation",
    "cwq_valid",
])

assert len(webqsp_val) == 246
assert len(cwq_val) == 3519

print("WebQSP validation dataset:", WEBQSP_VAL_VAR)
print("CWQ validation dataset:   ", CWQ_VAL_VAR)

# ======================================================================
# 3. RESOLVE FROZEN VALIDATION PLAN ROWS
# ======================================================================

webqsp_val_plan_rows, WEBQSP_VAL_PLAN_VAR = resolve_global([
    "webqsp_val_plan_rows",
    "webqsp_val_planning",
    "webqsp_validation_plan_rows",
    "webqsp_valid_plan_rows",
    "webqsp_val_plans",
    "webqsp_frozen_val_plan_rows",
])

cwq_val_plan_rows, CWQ_VAL_PLAN_VAR = resolve_global([
    "cwq_val_plan_rows",
    "cwq_val_planning",
    "cwq_validation_plan_rows",
    "cwq_valid_plan_rows",
    "cwq_val_plans",
    "cwq_frozen_val_plan_rows",
])

assert len(webqsp_val_plan_rows) == len(webqsp_val)
assert len(cwq_val_plan_rows) == len(cwq_val)

print("WebQSP validation plans:", WEBQSP_VAL_PLAN_VAR)
print("CWQ validation plans:   ", CWQ_VAL_PLAN_VAR)

# ======================================================================
# 4. FULL VALIDATION PLAN ALIGNMENT GATE
# ======================================================================

def verify_validation_plan_alignment(dataset, plan_rows, dataset_name):
    total_plans = 0
    empty_plans = 0

    for i, (rec, row) in enumerate(zip(dataset, plan_rows)):
        assert "id" in row
        assert "predicted_paths" in row

        assert str(rec["id"]) == str(row["id"]), (
            f"{dataset_name}: ID mismatch at index {i}\n"
            f"dataset={rec['id']}\n"
            f"plan_row={row['id']}"
        )

        # Older RQ1 validation artifacts may not contain this field.
        if "source_index" in row:
            assert int(row["source_index"]) == i, (
                f"{dataset_name}: source_index mismatch at {i}"
            )

        plans = row["predicted_paths"]
        assert isinstance(plans, (list, tuple))

        total_plans += len(plans)
        empty_plans += sum(len(p) == 0 for p in plans)

    print(f"\n[{dataset_name}] validation alignment: PASSED")
    print("Questions:          ", len(dataset))
    print("Predicted plans:    ", total_plans)
    print("Empty plans:        ", empty_plans)

verify_validation_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_validation_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)

# ======================================================================
# 5. OUTPUT DIRECTORY
# ======================================================================

if "RQ2_FEATURE_DIR" not in globals():
    RQ2_FEATURE_DIR = os.path.join(
        os.path.dirname(os.path.abspath(RQ2_LABEL_DIR)),
        "03_features"
    )

os.makedirs(RQ2_FEATURE_DIR, exist_ok=True)

print("\nFeature directory:", RQ2_FEATURE_DIR)

# ======================================================================
# 6. HASH / ATOMIC-WRITE HELPERS
# ======================================================================

def sha256_file_local(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def atomic_json_local(obj, path):
    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


def stable_plan_rows_sha256(plan_rows):
    """
    Deterministic hash for frozen validation planner outputs.

    Older RQ1 validation rows may not contain source_index.
    Their already-verified positional index is used instead.
    """
    h = hashlib.sha256()

    for i, row in enumerate(plan_rows):
        source_index = int(row.get("source_index", i))

        assert source_index == i, (
            f"Validation plan ordering mismatch: "
            f"row={i}, source_index={source_index}"
        )

        payload = {
            "source_index": source_index,
            "id": str(row["id"]),
            "predicted_paths": row["predicted_paths"],
        }

        line = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True
        )

        h.update(line.encode("utf-8"))
        h.update(b"\n")

    return h.hexdigest()


WEBQSP_VAL_PLAN_SHA256 = stable_plan_rows_sha256(
    webqsp_val_plan_rows
)

CWQ_VAL_PLAN_SHA256 = stable_plan_rows_sha256(
    cwq_val_plan_rows
)

print(
    "\nWebQSP VAL-plan SHA256:",
    WEBQSP_VAL_PLAN_SHA256[:16] + "..."
)

print(
    "CWQ VAL-plan SHA256:   ",
    CWQ_VAL_PLAN_SHA256[:16] + "..."
)

# ======================================================================
# 7. VALIDATION SUPERVISION DIRECTORY
# ======================================================================

VAL_LABEL_DIR = os.path.join(
    RQ2_FEATURE_DIR,
    "validation_supervision"
)

os.makedirs(VAL_LABEL_DIR, exist_ok=True)

# ======================================================================
# 8. CONSTRUCT VALIDATION DECISION SUPERVISION
# ======================================================================
#
# Mirrors TRAIN branch-label construction, but:
#   - validation gold only
#   - feasible decision groups written
#   - final hop excluded
#   - singleton groups excluded
#   - traversal always remains unpruned
# ======================================================================

def generate_validation_decision_rows(
    dataset,
    plan_rows,
    dataset_name,
    plan_sha256
):
    out_path = os.path.join(
        VAL_LABEL_DIR,
        f"{dataset_name}_validation_decision_labels.jsonl"
    )

    manifest_path = os.path.join(
        VAL_LABEL_DIR,
        f"{dataset_name}_validation_decision_labels_manifest.json"
    )

    # Reuse already-completed artifact if valid.
    if os.path.exists(out_path) and os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        valid = (
            manifest.get("feature_spec_sha256") == AFP_FEATURE_SPEC_SHA256
            and manifest.get("val_plan_sha256") == plan_sha256
            and sha256_file_local(out_path) == manifest.get("labels_sha256")
        )

        if valid:
            print(
                f"[{dataset_name}] validation supervision: "
                "verified frozen"
            )
            return out_path, manifest

    tmp_path = out_path + ".tmp"

    stats = {
        "questions": len(dataset),
        "empty_plans": 0,
        "intermediate_groups": 0,
        "feasible_groups": 0,
        "feasible_decision_groups": 0,
        "feasible_singletons": 0,
        "infeasible_groups": 0,
        "final_hop_groups": 0,
        "empty_terminated": 0,
        "decision_branches": 0,
        "positive_branches": 0,
        "negative_branches": 0,
    }

    with open(tmp_path, "w", encoding="utf-8") as out:
        for source_index in tqdm(
            range(len(dataset)),
            desc=f"{dataset_name} validation labels"
        ):
            rec = dataset[source_index]
            plan_row = plan_rows[source_index]

            assert str(rec["id"]) == str(plan_row["id"])

            # Positional source index is canonical for old RQ1 artifacts.
            if "source_index" in plan_row:
                assert int(plan_row["source_index"]) == source_index

            G = build_graph(rec["graph"])
            gold_answers = set(rec["a_entity"])
            topic_entities = list(rec["q_entity"])

            plans = [
                list(p)
                for p in plan_row["predicted_paths"]
            ]

            for plan_idx, plan in enumerate(plans):
                L = len(plan)

                if L == 0:
                    stats["empty_plans"] += 1
                    continue

                reachable_dp = call_suffix_dp(
                    G,
                    plan,
                    gold_answers
                )

                for topic_idx, topic_entity in enumerate(topic_entities):
                    active_prefixes = [(topic_entity,)]

                    for h in range(L):
                        required_relation = plan[h]
                        candidates = []

                        # Exact relation-valid RoG expansion
                        for parent_idx, prefix in enumerate(active_prefixes):
                            current_entity = prefix[-1]

                            neighbors = relation_valid_neighbors(
                                G,
                                current_entity,
                                required_relation
                            )

                            for nbr in neighbors:
                                candidates.append({
                                    "parent_prefix_index": parent_idx,
                                    "prefix_entities": list(prefix),
                                    "candidate_entity": nbr,
                                    "branch_entities": list(prefix + (nbr,)),
                                })

                        # No valid continuation
                        if not candidates:
                            stats["empty_terminated"] += 1
                            break

                        # Final-hop protection
                        if h == L - 1:
                            stats["final_hop_groups"] += 1
                            break

                        stats["intermediate_groups"] += 1

                        labels = [
                            int(
                                dp_is_reachable(
                                    reachable_dp,
                                    h + 1,
                                    c["candidate_entity"]
                                )
                            )
                            for c in candidates
                        ]

                        n_pos = int(sum(labels))
                        n_neg = len(labels) - n_pos

                        feasible = n_pos > 0
                        decision = len(candidates) > 1

                        if feasible:
                            stats["feasible_groups"] += 1

                            if decision:
                                stats["feasible_decision_groups"] += 1

                                group_id = (
                                    f"{dataset_name}|validation|"
                                    f"{source_index}|"
                                    f"p{plan_idx}|"
                                    f"t{topic_idx}|"
                                    f"h{h}"
                                )

                                for candidate_idx, (cand, y) in enumerate(
                                    zip(candidates, labels)
                                ):
                                    row = {
                                        "dataset": dataset_name,
                                        "split": "validation",
                                        "source_index": source_index,
                                        "question_id": rec["id"],
                                        "group_id": group_id,
                                        "plan_index": plan_idx,
                                        "plan": plan,
                                        "plan_length": L,
                                        "topic_index": topic_idx,
                                        "topic_entity": topic_entity,
                                        "hop": h,
                                        "required_relation": required_relation,
                                        "candidate_index": candidate_idx,
                                        "candidate_count": len(candidates),
                                        "parent_prefix_index":
                                            cand["parent_prefix_index"],
                                        "prefix_entities":
                                            cand["prefix_entities"],
                                        "candidate_entity":
                                            cand["candidate_entity"],
                                        "label": int(y),
                                    }

                                    out.write(
                                        json.dumps(
                                            row,
                                            ensure_ascii=False
                                        ) + "\n"
                                    )

                                stats["decision_branches"] += len(candidates)
                                stats["positive_branches"] += n_pos
                                stats["negative_branches"] += n_neg

                            else:
                                stats["feasible_singletons"] += 1

                        else:
                            stats["infeasible_groups"] += 1

                        # CRITICAL:
                        # labels do NOT modify traversal.
                        active_prefixes = [
                            tuple(c["branch_entities"])
                            for c in candidates
                        ]

        out.flush()
        os.fsync(out.fileno())

    os.replace(tmp_path, out_path)

    manifest = {
        "dataset": dataset_name,
        "split": "validation",
        "val_plan_sha256": plan_sha256,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        **stats,
        "labels_sha256": sha256_file_local(out_path),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json_local(
        manifest,
        manifest_path
    )

    print(
        f"\n[{dataset_name}] VALIDATION SUPERVISION FROZEN"
    )
    print(
        "Intermediate groups:      ",
        stats["intermediate_groups"]
    )
    print(
        "Feasible decision groups: ",
        stats["feasible_decision_groups"]
    )
    print(
        "Decision branches:        ",
        stats["decision_branches"]
    )
    print(
        "Positive / Negative:      ",
        stats["positive_branches"],
        "/",
        stats["negative_branches"]
    )
    print(
        "Empty plans skipped:      ",
        stats["empty_plans"]
    )

    return out_path, manifest

# ======================================================================
# 9. BUILD/FREEZE VALIDATION SUPERVISION
# ======================================================================

(
    webqsp_val_labels_file,
    webqsp_val_label_manifest
) = generate_validation_decision_rows(
    webqsp_val,
    webqsp_val_plan_rows,
    "webqsp",
    WEBQSP_VAL_PLAN_SHA256
)

(
    cwq_val_labels_file,
    cwq_val_label_manifest
) = generate_validation_decision_rows(
    cwq_val,
    cwq_val_plan_rows,
    "cwq",
    CWQ_VAL_PLAN_SHA256
)

# ======================================================================
# 10. TRAIN LABEL FILES FROM CELL 3
# ======================================================================

webqsp_train_labels_file = webqsp_branch_manifest["labels_file"]
cwq_train_labels_file = cwq_branch_manifest["labels_file"]

# ======================================================================
# 11. ITERATE FEASIBLE DECISION GROUPS
# ======================================================================

def iter_decision_groups(labels_file):
    """
    Rows are expected to be group-contiguous.

    TRAIN file:
        singleton feasible groups exist but are skipped.

    VALIDATION file:
        only feasible decision groups were written.
    """
    current_id = None
    current_rows = []

    with open(labels_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)

            # TRAIN labels contain singleton feasible rows.
            if row.get("decision_opportunity", True) is False:
                continue

            group_id = row["group_id"]

            if current_id is not None and group_id != current_id:
                assert len(current_rows) > 1
                yield current_rows
                current_rows = []

            current_id = group_id
            current_rows.append(row)

    if current_rows:
        assert len(current_rows) > 1
        yield current_rows

# ======================================================================
# 12. COLLECT REUSABLE SEMANTIC TEXTS
# ======================================================================

def collect_semantic_texts(labels_file, dataset):
    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    n_groups = 0
    n_branches = 0

    for rows in iter_decision_groups(labels_file):
        n_groups += 1
        n_branches += len(rows)

        first = rows[0]
        source_index = int(first["source_index"])
        rec = dataset[source_index]

        question_id = str(first["question_id"])
        questions[question_id] = rec["question"]

        plan = list(first["plan"])
        hop = int(first["hop"])

        for relation in plan:
            relations[str(relation)] = relation_surface_text(relation)

        plan_key = "||".join(str(x) for x in plan)
        plans[plan_key] = relation_sequence_text(plan)

        suffix = plan[hop + 1:]
        suffix_key = "||".join(str(x) for x in suffix)
        suffixes[suffix_key] = relation_sequence_text(suffix)

        # Readable entity names only.
        # Raw Freebase MIDs are excluded from MiniLM inventory.
        for row in rows:
            candidate = row["candidate_entity"]

            if has_readable_entity_surface(candidate):
                entities[str(candidate)] = entity_surface_text(candidate)

            for entity in row["prefix_entities"]:
                if has_readable_entity_surface(entity):
                    entities[str(entity)] = entity_surface_text(entity)

    return {
        "questions": questions,
        "entities": entities,
        "relations": relations,
        "plans": plans,
        "suffixes": suffixes,
        "n_groups": n_groups,
        "n_branches": n_branches,
    }


def merge_mapping_dicts(*dicts):
    result = {}

    for d in dicts:
        result.update(d)

    return result

print("\nCollecting semantic-text inventory...")

wq_train_text = collect_semantic_texts(
    webqsp_train_labels_file,
    webqsp_train
)

wq_val_text = collect_semantic_texts(
    webqsp_val_labels_file,
    webqsp_val
)

cwq_train_text = collect_semantic_texts(
    cwq_train_labels_file,
    cwq_train
)

cwq_val_text = collect_semantic_texts(
    cwq_val_labels_file,
    cwq_val
)

ALL_QUESTIONS = merge_mapping_dicts(
    wq_train_text["questions"],
    wq_val_text["questions"],
    cwq_train_text["questions"],
    cwq_val_text["questions"],
)

ALL_ENTITIES = merge_mapping_dicts(
    wq_train_text["entities"],
    wq_val_text["entities"],
    cwq_train_text["entities"],
    cwq_val_text["entities"],
)

ALL_RELATIONS = merge_mapping_dicts(
    wq_train_text["relations"],
    wq_val_text["relations"],
    cwq_train_text["relations"],
    cwq_val_text["relations"],
)

ALL_PLANS = merge_mapping_dicts(
    wq_train_text["plans"],
    wq_val_text["plans"],
    cwq_train_text["plans"],
    cwq_val_text["plans"],
)

ALL_SUFFIXES = merge_mapping_dicts(
    wq_train_text["suffixes"],
    wq_val_text["suffixes"],
    cwq_train_text["suffixes"],
    cwq_val_text["suffixes"],
)

print("Unique questions:  ", len(ALL_QUESTIONS))
print("Readable entities: ", len(ALL_ENTITIES))
print("Relations:         ", len(ALL_RELATIONS))
print("Full plans:        ", len(ALL_PLANS))
print("Suffixes:          ", len(ALL_SUFFIXES))

# ======================================================================
# 13. LOAD FROZEN MINILM
# ======================================================================

try:
    import torch
    FEATURE_DEVICE = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
except Exception:
    FEATURE_DEVICE = "cpu"

print("\nMiniLM device:", FEATURE_DEVICE)

afp_semantic_encoder = FrozenMiniLMEncoder(
    model_name=AFP_SEMANTIC_ENCODER_NAME,
    device=FEATURE_DEVICE,
    batch_size=256
)

# ======================================================================
# 14. BATCH CACHE SEMANTIC EMBEDDINGS
# ======================================================================

print("\nCaching question embeddings...")
afp_semantic_encoder.prefill(
    "question",
    ALL_QUESTIONS
)

print("Caching readable entity embeddings...")
afp_semantic_encoder.prefill(
    "entity",
    ALL_ENTITIES
)

print("Caching relation embeddings...")
afp_semantic_encoder.prefill(
    "relation",
    ALL_RELATIONS
)

print("Caching full-plan embeddings...")
afp_semantic_encoder.prefill(
    "plan",
    ALL_PLANS
)

print("Caching suffix embeddings...")
afp_semantic_encoder.prefill(
    "suffix",
    ALL_SUFFIXES
)

print(
    "Semantic cache entries:",
    len(afp_semantic_encoder.cache)
)

# ======================================================================
# 15. BUILD ONE FEATURE DATASET
# ======================================================================

def build_feature_dataset(
    dataset_name,
    split,
    dataset,
    labels_file
):
    X_blocks = []
    y_blocks = []

    group_ptr = [0]
    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []

    n_groups = 0

    for rows in tqdm(
        iter_decision_groups(labels_file),
        desc=f"{dataset_name} {split} features"
    ):
        first = rows[0]

        source_index = int(first["source_index"])
        rec = dataset[source_index]

        assert str(rec["id"]) == str(first["question_id"])

        plan = list(first["plan"])
        hop = int(first["hop"])

        X_group = extract_afp_group_features(
            question_id=first["question_id"],
            question=rec["question"],
            plan=plan,
            hop=hop,
            candidate_rows=rows,
            semantic_encoder=afp_semantic_encoder,
            entity_name_map=None
        )

        y_group = np.asarray(
            [int(r["label"]) for r in rows],
            dtype=np.uint8
        )

        assert X_group.shape == (
            len(rows),
            AFP_FEATURE_DIM
        )

        assert len(y_group) == len(rows)
        assert int(y_group.sum()) >= 1
        assert len(rows) > 1
        assert np.all(np.isfinite(X_group))

        X_blocks.append(X_group)
        y_blocks.append(y_group)

        group_ptr.append(
            group_ptr[-1] + len(rows)
        )

        group_source_index.append(source_index)
        group_hop.append(hop)
        group_plan_length.append(len(plan))
        group_candidate_count.append(len(rows))

        n_groups += 1

    assert n_groups > 0

    X = np.concatenate(
        X_blocks,
        axis=0
    ).astype(np.float32)

    y = np.concatenate(
        y_blocks,
        axis=0
    ).astype(np.uint8)

    group_ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    group_source_index = np.asarray(
        group_source_index,
        dtype=np.int32
    )

    group_hop = np.asarray(
        group_hop,
        dtype=np.int16
    )

    group_plan_length = np.asarray(
        group_plan_length,
        dtype=np.int16
    )

    group_candidate_count = np.asarray(
        group_candidate_count,
        dtype=np.int32
    )

    # Final invariants
    assert X.shape[0] == len(y)
    assert X.shape[1] == AFP_FEATURE_DIM
    assert group_ptr[0] == 0
    assert group_ptr[-1] == len(y)
    assert len(group_ptr) == n_groups + 1
    assert np.all(np.diff(group_ptr) > 1)
    assert np.all(np.isfinite(X))
    assert set(np.unique(y)).issubset({0, 1})

    return {
        "X": X,
        "y": y,
        "group_ptr": group_ptr,
        "group_source_index": group_source_index,
        "group_hop": group_hop,
        "group_plan_length": group_plan_length,
        "group_candidate_count": group_candidate_count,
    }

# ======================================================================
# 16. FREEZE ONE FEATURE DATASET
# ======================================================================

def freeze_feature_dataset(
    dataset_name,
    split,
    data
):
    out_dir = os.path.join(
        RQ2_FEATURE_DIR,
        dataset_name
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    npz_path = os.path.join(
        out_dir,
        f"{dataset_name}_{split}_afp_features_v2.npz"
    )

    manifest_path = os.path.join(
        out_dir,
        f"{dataset_name}_{split}_afp_features_v2_manifest.json"
    )

    np.savez(
        npz_path,
        X=data["X"],
        y=data["y"],
        group_ptr=data["group_ptr"],
        group_source_index=data["group_source_index"],
        group_hop=data["group_hop"],
        group_plan_length=data["group_plan_length"],
        group_candidate_count=data["group_candidate_count"],
    )

    y = data["y"]
    group_ptr = data["group_ptr"]

    n_groups = len(group_ptr) - 1
    n_branches = len(y)

    n_pos = int(y.sum())
    n_neg = n_branches - n_pos

    manifest = {
        "dataset": dataset_name,
        "split": split,
        "feature_version": AFP_FEATURE_VERSION,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "feature_dim": AFP_FEATURE_DIM,
        "semantic_encoder": AFP_SEMANTIC_ENCODER_NAME,
        "raw_mid_policy":
            "zero_embedding_plus_availability",
        "decision_only": True,
        "n_groups": int(n_groups),
        "n_branches": int(n_branches),
        "positive_branches": int(n_pos),
        "negative_branches": int(n_neg),
        "positive_rate": float(
            n_pos / n_branches
        ),
        "npz_sha256":
            sha256_file_local(npz_path),
        "created_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    atomic_json_local(
        manifest,
        manifest_path
    )

    return npz_path, manifest

# ======================================================================
# 17. BUILD WEBQSP TRAIN
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING WEBQSP TRAIN FEATURES")
print("=" * 80)

webqsp_train_features = build_feature_dataset(
    "webqsp",
    "train",
    webqsp_train,
    webqsp_train_labels_file
)

(
    webqsp_train_feature_file,
    webqsp_train_feature_manifest
) = freeze_feature_dataset(
    "webqsp",
    "train",
    webqsp_train_features
)

# ======================================================================
# 18. BUILD WEBQSP VALIDATION
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING WEBQSP VALIDATION FEATURES")
print("=" * 80)

webqsp_val_features = build_feature_dataset(
    "webqsp",
    "validation",
    webqsp_val,
    webqsp_val_labels_file
)

(
    webqsp_val_feature_file,
    webqsp_val_feature_manifest
) = freeze_feature_dataset(
    "webqsp",
    "validation",
    webqsp_val_features
)

# ======================================================================
# 19. BUILD CWQ TRAIN
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING CWQ TRAIN FEATURES")
print("=" * 80)

cwq_train_features = build_feature_dataset(
    "cwq",
    "train",
    cwq_train,
    cwq_train_labels_file
)

(
    cwq_train_feature_file,
    cwq_train_feature_manifest
) = freeze_feature_dataset(
    "cwq",
    "train",
    cwq_train_features
)

# ======================================================================
# 20. BUILD CWQ VALIDATION
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING CWQ VALIDATION FEATURES")
print("=" * 80)

cwq_val_features = build_feature_dataset(
    "cwq",
    "validation",
    cwq_val,
    cwq_val_labels_file
)

(
    cwq_val_feature_file,
    cwq_val_feature_manifest
) = freeze_feature_dataset(
    "cwq",
    "validation",
    cwq_val_features
)

# ======================================================================
# 21. FINAL FEATURE REPORT
# ======================================================================

def report_feature_manifest(m):
    print("\n" + "-" * 72)
    print(
        f"{m['dataset'].upper()} "
        f"{m['split'].upper()}"
    )
    print("-" * 72)

    print("Decision groups:  ", m["n_groups"])
    print("Branches:         ", m["n_branches"])
    print("Positive:         ", m["positive_branches"])
    print("Negative:         ", m["negative_branches"])
    print(
        "Positive rate:    ",
        f"{100*m['positive_rate']:.2f}%"
    )
    print("Feature dimension:", m["feature_dim"])
    print(
        "NPZ SHA256:       ",
        m["npz_sha256"][:16] + "..."
    )

report_feature_manifest(
    webqsp_train_feature_manifest
)

report_feature_manifest(
    webqsp_val_feature_manifest
)

report_feature_manifest(
    cwq_train_feature_manifest
)

report_feature_manifest(
    cwq_val_feature_manifest
)

# ======================================================================
# 22. CROSS-CHECK TRAIN COUNTS AGAINST CELL 4
# ======================================================================

assert (
    webqsp_train_feature_manifest["n_groups"]
    ==
    webqsp_label_diag["feasible_decision_groups"]
)

assert (
    cwq_train_feature_manifest["n_groups"]
    ==
    cwq_label_diag["feasible_decision_groups"]
)

assert (
    webqsp_train_feature_manifest["n_branches"]
    ==
    webqsp_label_diag["decision_stats"]["n"]
)

assert (
    cwq_train_feature_manifest["n_branches"]
    ==
    cwq_label_diag["decision_stats"]["n"]
)

assert (
    webqsp_train_feature_manifest["positive_branches"]
    ==
    webqsp_label_diag["decision_stats"]["pos"]
)

assert (
    cwq_train_feature_manifest["positive_branches"]
    ==
    cwq_label_diag["decision_stats"]["pos"]
)

# ======================================================================
# 23. FINAL FREEZE GATE
# ======================================================================

print("\n" + "=" * 84)
print(
    "=== RQ2 CELL 6: "
    "TRAIN + VALIDATION FEATURE DATASETS FROZEN ==="
)
print("=" * 84)

print("Feature spec:", AFP_FEATURE_SPEC_SHA256)
print("Feature dimension:", AFP_FEATURE_DIM)
print("Raw Freebase MIDs were never passed to MiniLM.")
print("TRAIN supervision: training gold only.")
print("VALIDATION supervision: validation gold only.")
print("TEST data/gold: NOT USED.")
print("All scorer datasets contain feasible decision groups only.")
print("Group boundaries preserved for group-aware loss/ranking.")
print("Frozen validation plans reused; none regenerated.")
print("\nNext: define AFP scorer and validation metrics.")

Feature version: afp_features_v2_masked_entity_semantics
Feature dimension: 27
Feature SHA256: 738985d1232a8ac5...
WebQSP validation dataset: webqsp_val
CWQ validation dataset:    cwq_val
WebQSP validation plans: webqsp_val_plan_rows
CWQ validation plans:    cwq_val_plan_rows

[WebQSP] validation alignment: PASSED
Questions:           246
Predicted plans:     721
Empty plans:         0

[CWQ] validation alignment: PASSED
Questions:           3519
Predicted plans:     10536
Empty plans:         7

Feature directory: /kaggle/working/step3_rq2_dev_v1/03_features

WebQSP VAL-plan SHA256: df44197f9cf9e244...
CWQ VAL-plan SHA256:    1a0590c1cd183860...


webqsp validation labels: 100%|██████████| 246/246 [00:09<00:00, 26.62it/s]



[webqsp] VALIDATION SUPERVISION FROZEN
Intermediate groups:       253
Feasible decision groups:  87
Decision branches:         966
Positive / Negative:       319 / 647
Empty plans skipped:       0


cwq validation labels: 100%|██████████| 3519/3519 [02:40<00:00, 21.96it/s]



[cwq] VALIDATION SUPERVISION FROZEN
Intermediate groups:       6462
Feasible decision groups:  1352
Decision branches:         18688
Positive / Negative:       5604 / 13084
Empty plans skipped:       7

Unique questions:   10133
Readable entities:  21218
Relations:          795
Full plans:         1846
Suffixes:           801

MiniLM device: cuda

Caching question embeddings...


...


BUILDING CWQ TRAIN FEATURES


cwq train features: 15937it [07:08, 37.21it/s]



BUILDING CWQ VALIDATION FEATURES


cwq validation features: 1352it [00:35, 37.68it/s]



------------------------------------------------------------------------
WEBQSP TRAIN
------------------------------------------------------------------------
Decision groups:   1457
Branches:          18437
Positive:          7301
Negative:          11136
Positive rate:     39.60%
Feature dimension: 27
NPZ SHA256:        55db1698520e6c6b...

------------------------------------------------------------------------
WEBQSP VALIDATION
------------------------------------------------------------------------
Decision groups:   87
Branches:          966
Positive:          319
Negative:          647
Positive rate:     33.02%
Feature dimension: 27
NPZ SHA256:        de6b1ed170f93181...

------------------------------------------------------------------------
CWQ TRAIN
------------------------------------------------------------------------
Decision groups:   15937
Branches:          218544
Positive:          52746
Negative:          165798
Positive rate:     24.14%
Feature dimension: 27
NPZ S

## Defining lightweight AFP scorer and group aware validation metrics


In [98]:
# ======================================================================
# DEFINE LIGHTWEIGHT AFP SCORER + GROUP-AWARE VALIDATION METRICS
# ======================================================================
#
# Scorer:
#   h_i = ReLU(W1 x_i + b1)
#   l_i = W2 h_i + b2
#   s_i = sigmoid(l_i)
#
# Input:
#   27-d frozen AFP feature vector.
#
# This cell DEFINES:
#   - train-only feature standardization
#   - lightweight AFP MLP
#   - branch BCE
#   - group-aware ranking metrics
#   - evaluation utilities
#
# This cell DOES NOT train the scorer.
# ======================================================================

import math
import copy
import json
import hashlib
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================================================================
# 1. Frozen scorer search space
# ======================================================================

AFP_SCORER_VERSION = "afp_mlp_v1"

AFP_HIDDEN_CANDIDATES = [32, 64, 128]
AFP_DROPOUT = 0.0

AFP_INPUT_DIM = AFP_FEATURE_DIM
assert AFP_INPUT_DIM == 27

# We keep architecture intentionally lightweight:
# 27 -> H -> 1
AFP_SCORER_SPEC = {
    "version": AFP_SCORER_VERSION,
    "input_dim": AFP_INPUT_DIM,
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "activation": "ReLU",
    "output": "single_logit",
    "dropout": AFP_DROPOUT,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
}

AFP_SCORER_SPEC_SHA256 = hashlib.sha256(
    json.dumps(AFP_SCORER_SPEC, sort_keys=True).encode("utf-8")
).hexdigest()

print("AFP scorer version:", AFP_SCORER_VERSION)
print("Input dimension:   ", AFP_INPUT_DIM)
print("Hidden candidates: ", AFP_HIDDEN_CANDIDATES)
print("Scorer SHA256:     ", AFP_SCORER_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. TRAIN-ONLY feature standardizer
# ======================================================================
#
# IMPORTANT:
# Mean/std must be fitted on TRAIN only.
# The same fitted statistics are then applied to validation/test.
#
# Constant features are protected with std = 1.
# ======================================================================

class AFPFeatureStandardizer:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None
        self.fitted = False

    def fit(self, X):
        X = np.asarray(X, dtype=np.float32)

        assert X.ndim == 2
        assert X.shape[1] == AFP_INPUT_DIM
        assert np.all(np.isfinite(X))

        self.mean_ = X.mean(axis=0).astype(np.float32)
        self.std_ = X.std(axis=0).astype(np.float32)

        self.std_[self.std_ < self.eps] = 1.0
        self.fitted = True
        return self

    def transform(self, X):
        assert self.fitted

        X = np.asarray(X, dtype=np.float32)
        Z = (X - self.mean_) / self.std_

        assert np.all(np.isfinite(Z))
        return Z.astype(np.float32)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def state_dict(self):
        assert self.fitted
        return {
            "mean": self.mean_.tolist(),
            "std": self.std_.tolist(),
            "eps": float(self.eps),
        }

    def load_state_dict(self, state):
        self.mean_ = np.asarray(state["mean"], dtype=np.float32)
        self.std_ = np.asarray(state["std"], dtype=np.float32)
        self.eps = float(state["eps"])
        self.fitted = True
        return self

# ======================================================================
# 3. Lightweight AFP scorer
# ======================================================================

class AFPScorer(nn.Module):
    def __init__(self, input_dim=AFP_INPUT_DIM, hidden_dim=64, dropout=0.0):
        super().__init__()

        assert hidden_dim in AFP_HIDDEN_CANDIDATES

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

        self.dropout = (
            nn.Dropout(dropout)
            if dropout > 0
            else nn.Identity()
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        """
        Returns raw logits.

        x:
            [N, 27]

        output:
            [N]
        """
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        logits = self.fc2(h).squeeze(-1)

        return logits

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))

# ======================================================================
# 4. Branch-level BCE
# ======================================================================

def branch_bce_from_logits(logits, targets):
    """
    Ordinary branch-averaged BCE.

    This is an evaluation metric and can also be used as a training loss.
    """
    logits = logits.float()
    targets = targets.float()

    assert logits.shape == targets.shape

    return F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="mean"
    )

# ======================================================================
# 5. Group-balanced BCE
# ======================================================================
#
# Important because candidate-frontier sizes are heavy-tailed.
#
# Each decision GROUP gets equal weight:
#
# L = mean_g [ mean_{i in g} BCE_i ]
#
# This prevents a 600-candidate frontier from contributing 300x
# more loss than a 2-candidate frontier merely because of size.
# ======================================================================

def group_balanced_bce_from_logits(logits, targets, group_ptr):
    logits = logits.float()
    targets = targets.float()

    assert logits.shape == targets.shape

    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    losses = []

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        assert end > start

        group_loss = F.binary_cross_entropy_with_logits(
            logits[start:end],
            targets[start:end],
            reduction="mean"
        )

        losses.append(group_loss)

    assert losses

    return torch.stack(losses).mean()

# ======================================================================
# 6. Ranking helpers
# ======================================================================

def _group_average_precision(labels_sorted):
    """
    Average Precision inside one decision group.

    labels_sorted:
        binary labels ordered from highest score to lowest score.
    """
    labels_sorted = np.asarray(labels_sorted, dtype=np.int32)

    total_pos = int(labels_sorted.sum())

    if total_pos == 0:
        return np.nan

    hit_count = 0
    precision_sum = 0.0

    for rank, y in enumerate(labels_sorted, start=1):
        if y == 1:
            hit_count += 1
            precision_sum += hit_count / rank

    return precision_sum / total_pos

# ======================================================================
# 7. Group-aware ranking metrics
# ======================================================================

def compute_group_ranking_metrics(scores, labels, group_ptr, ks=(1, 2, 4)):
    """
    scores:
        higher = better candidate

    labels:
        1 iff answer-supporting under exact suffix supervision

    group_ptr:
        boundaries of feasible decision groups.

    Returns:
        top1_positive_hit
        mrr_first_positive
        group_average_precision
        hit@k
        positive_recall@k
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    assert len(scores) == len(labels)
    assert group_ptr[0] == 0
    assert group_ptr[-1] == len(labels)

    top1_hits = []
    reciprocal_ranks = []
    average_precisions = []

    hit_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}

    group_sizes = []
    positive_counts = []

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        g_scores = scores[start:end]
        g_labels = labels[start:end]

        assert len(g_labels) > 1
        assert g_labels.sum() >= 1

        # Stable descending sort.
        order = np.argsort(
            -g_scores,
            kind="mergesort"
        )

        ranked_labels = g_labels[order]

        group_sizes.append(len(g_labels))
        positive_counts.append(int(g_labels.sum()))

        # ----------------------------------------------------------
        # Top-1 positive hit
        # ----------------------------------------------------------
        top1_hits.append(
            float(ranked_labels[0] == 1)
        )

        # ----------------------------------------------------------
        # Reciprocal rank of FIRST positive
        # ----------------------------------------------------------
        first_positive_rank = (
            np.flatnonzero(ranked_labels == 1)[0] + 1
        )

        reciprocal_ranks.append(
            1.0 / first_positive_rank
        )

        # ----------------------------------------------------------
        # Group Average Precision
        # ----------------------------------------------------------
        average_precisions.append(
            _group_average_precision(ranked_labels)
        )

        # ----------------------------------------------------------
        # Hit@k and Positive Recall@k
        # ----------------------------------------------------------
        total_positive = int(ranked_labels.sum())

        for k in ks:
            effective_k = min(k, len(ranked_labels))
            top_k = ranked_labels[:effective_k]

            positive_in_top_k = int(top_k.sum())

            hit_at_k[k].append(
                float(positive_in_top_k > 0)
            )

            recall_at_k[k].append(
                positive_in_top_k / total_positive
            )

    metrics = {
        "n_groups": int(len(group_ptr) - 1),
        "mean_group_size": float(np.mean(group_sizes)),
        "median_group_size": float(np.median(group_sizes)),
        "mean_positive_count": float(np.mean(positive_counts)),

        "top1_positive_hit": float(np.mean(top1_hits)),
        "mrr_first_positive": float(np.mean(reciprocal_ranks)),
        "group_average_precision": float(np.mean(average_precisions)),
    }

    for k in ks:
        metrics[f"hit@{k}"] = float(
            np.mean(hit_at_k[k])
        )

        metrics[f"positive_recall@{k}"] = float(
            np.mean(recall_at_k[k])
        )

    return metrics

# ======================================================================
# 8. Complete validation evaluator
# ======================================================================

@torch.no_grad()
def evaluate_afp_scorer(
    model,
    X,
    y,
    group_ptr,
    device="cpu"
):
    model.eval()

    X_tensor = torch.as_tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    y_tensor = torch.as_tensor(
        y,
        dtype=torch.float32,
        device=device
    )

    logits = model(X_tensor)

    branch_bce = branch_bce_from_logits(
        logits,
        y_tensor
    ).item()

    group_bce = group_balanced_bce_from_logits(
        logits,
        y_tensor,
        group_ptr
    ).item()

    probs = torch.sigmoid(
        logits
    ).detach().cpu().numpy()

    ranking = compute_group_ranking_metrics(
        scores=probs,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

    result = {
        "branch_bce": float(branch_bce),
        "group_balanced_bce": float(group_bce),
        **ranking,
    }

    return result

# ======================================================================
# 9. Random-ranking sanity baseline
# ======================================================================
#
# This is NOT one of the final retrieval baselines.
# It is only a diagnostic baseline for scorer ranking metrics.
# ======================================================================

def random_ranking_metrics(y, group_ptr, seed=42):
    rng = np.random.default_rng(seed)

    random_scores = rng.random(
        len(y)
    )

    return compute_group_ranking_metrics(
        scores=random_scores,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

# ======================================================================
# 10. Oracle-ranking sanity ceiling
# ======================================================================
#
# Again, diagnostic only.
# Gold labels are NEVER used by the model.
# ======================================================================

def oracle_ranking_metrics(y, group_ptr):
    # Positive candidates receive a higher synthetic score.
    oracle_scores = np.asarray(
        y,
        dtype=np.float64
    )

    return compute_group_ranking_metrics(
        scores=oracle_scores,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

# ======================================================================
# 11. Definition sanity gate
# ======================================================================

mock_group_ptr = np.asarray(
    [0, 3, 6],
    dtype=np.int64
)

mock_y = np.asarray(
    [
        0, 1, 0,   # group 1
        1, 0, 1,   # group 2
    ],
    dtype=np.uint8
)

mock_scores = np.asarray(
    [
        0.2, 0.9, 0.1,
        0.8, 0.2, 0.7,
    ],
    dtype=np.float32
)

mock_metrics = compute_group_ranking_metrics(
    scores=mock_scores,
    labels=mock_y,
    group_ptr=mock_group_ptr,
    ks=(1, 2, 4)
)

assert np.isclose(
    mock_metrics["top1_positive_hit"],
    1.0
)

assert np.isclose(
    mock_metrics["mrr_first_positive"],
    1.0
)

mock_model = AFPScorer(
    input_dim=27,
    hidden_dim=64,
    dropout=AFP_DROPOUT
)

mock_X = torch.randn(
    6,
    27
)

mock_logits = mock_model(
    mock_X
)

assert mock_logits.shape == (6,)
assert torch.all(
    torch.isfinite(mock_logits)
)

# ======================================================================
# 12. Inspect random/oracle validation ranking ranges
# ======================================================================

print("\n" + "=" * 80)
print("WEBQSP VALIDATION RANKING SANITY")
print("=" * 80)

webqsp_random_metrics = random_ranking_metrics(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"],
    seed=42
)

webqsp_oracle_metrics = oracle_ranking_metrics(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"]
)

print("Random ranking:")
for k, v in webqsp_random_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\nOracle ranking:")
for k, v in webqsp_oracle_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\n" + "=" * 80)
print("CWQ VALIDATION RANKING SANITY")
print("=" * 80)

cwq_random_metrics = random_ranking_metrics(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"],
    seed=42
)

cwq_oracle_metrics = oracle_ranking_metrics(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"]
)

print("Random ranking:")
for k, v in cwq_random_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\nOracle ranking:")
for k, v in cwq_oracle_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

# ======================================================================
# 13. Final report
# ======================================================================

print("\n" + "=" * 84)
print("=== RQ2 CELL 7: LIGHTWEIGHT AFP SCORER DEFINED ===")
print("=" * 84)

print("Architecture:")
print("  Input -> Linear(H) -> ReLU -> Linear(1)")
print("  Input dimension:", AFP_INPUT_DIM)
print("  Hidden candidates:", AFP_HIDDEN_CANDIDATES)
print("  Dropout:", AFP_DROPOUT)

print("\nDefined loss/statistics:")
print("  Branch BCE")
print("  Group-balanced BCE")

print("\nDefined ranking metrics:")
print("  Top-1 Positive Hit")
print("  MRR of first positive")
print("  Group Average Precision")
print("  Hit@1 / Hit@2 / Hit@4")
print("  Positive Recall@1 / @2 / @4")

print("\nNormalization:")
print("  Feature mean/std fitted on TRAIN only")
print("  Same statistics applied to validation/test")

print("\nImportant:")
print("  No scorer training performed.")
print("  No test data used.")
print("  Gold labels used only for evaluation diagnostics.")
print("\nNext: train AFP scorer on TRAIN and evaluate on VALIDATION.")

AFP scorer version: afp_mlp_v1
Input dimension:    27
Hidden candidates:  [32, 64, 128]
Scorer SHA256:      0d096c8aa07d893a...

WEBQSP VALIDATION RANKING SANITY
Random ranking:
  mean_group_size            11.1034
  median_group_size          5.0000
  mean_positive_count        3.6667
  top1_positive_hit          0.4943
  mrr_first_positive         0.6701
  group_average_precision    0.6319
  hit@1                      0.4943
  positive_recall@1          0.2418
  hit@2                      0.7241
  positive_recall@2          0.4367
  hit@4                      0.8621
  positive_recall@4          0.6903

Oracle ranking:
  mean_group_size            11.1034
  median_group_size          5.0000
  mean_positive_count        3.6667
  top1_positive_hit          1.0000
  mrr_first_positive         1.0000
  group_average_precision    1.0000
  hit@1                      1.0000
  positive_recall@1          0.5708
  hit@2                      1.0000
  positive_recall@2          0.7737
  hit@4    

## Training scorer

In [99]:
# ======================================================================
# 3.8 TRAIN LIGHTWEIGHT AFP SCORER
# ======================================================================
#
# Development grid:
#   Hidden size: {32, 64, 128}
#   Loss:        {branch BCE, group-balanced BCE}
#   Seeds:       {42, 43, 44}
#
# Fixed for all runs:
#   AdamW
#   LR = 1e-3
#   Weight decay = 1e-4
#   Epochs = 80
#   Full-batch optimization
#   No class weighting
#   No dropout
#
# IMPORTANT:
#   - Feature normalization fitted on TRAIN only.
#   - Validation is NEVER used to fit normalization.
#   - No test data/gold used.
#   - This cell does NOT select the winning configuration.
#   - Cell 9 will perform validation-based model selection.
# ======================================================================

import os
import json
import time
import random
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_HIDDEN_CANDIDATES == [32, 64, 128]
assert AFP_DROPOUT == 0.0

required = [
    "AFPScorer",
    "AFPFeatureStandardizer",
    "evaluate_afp_scorer",
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing Cell 6/7 objects: " + ", ".join(missing)

# ======================================================================
# 2. Fixed training protocol
# ======================================================================

AFP_TRAINING_VERSION = "afp_train_v1"

AFP_TRAIN_SEEDS = [42, 43, 44]
AFP_LOSS_CANDIDATES = [
    "branch_bce",
    "group_balanced_bce",
]

AFP_LEARNING_RATE = 1e-3
AFP_WEIGHT_DECAY = 1e-4
AFP_EPOCHS = 80
AFP_GRAD_CLIP = 5.0

AFP_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AFP_TRAINING_SPEC = {
    "version": AFP_TRAINING_VERSION,
    "scorer_version": AFP_SCORER_VERSION,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "loss_candidates": AFP_LOSS_CANDIDATES,
    "seeds": AFP_TRAIN_SEEDS,
    "optimizer": "AdamW",
    "learning_rate": AFP_LEARNING_RATE,
    "weight_decay": AFP_WEIGHT_DECAY,
    "epochs": AFP_EPOCHS,
    "gradient_clip": AFP_GRAD_CLIP,
    "dropout": AFP_DROPOUT,
    "class_weighting": False,
    "training_mode": "full_batch",
}

AFP_TRAINING_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_TRAINING_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Training version:", AFP_TRAINING_VERSION)
print("Device:          ", AFP_DEVICE)
print("Hidden sizes:    ", AFP_HIDDEN_CANDIDATES)
print("Losses:          ", AFP_LOSS_CANDIDATES)
print("Seeds:           ", AFP_TRAIN_SEEDS)
print("Epochs:          ", AFP_EPOCHS)
print("Learning rate:   ", AFP_LEARNING_RATE)
print("Weight decay:    ", AFP_WEIGHT_DECAY)
print("Training SHA256: ", AFP_TRAINING_SPEC_SHA256[:16] + "...")

# ======================================================================
# 3. Output directory
# ======================================================================

RQ2_SCORER_DIR = os.path.join(
    os.path.dirname(RQ2_FEATURE_DIR),
    "04_scorer"
)

os.makedirs(RQ2_SCORER_DIR, exist_ok=True)

print("Scorer directory:", RQ2_SCORER_DIR)

# ======================================================================
# 4. Reproducibility helper
# ======================================================================

def set_afp_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ======================================================================
# 5. Exact group-balanced branch weights
# ======================================================================
#
# For group g with n_g branches:
#
#       w_i = 1 / n_g
#
# Therefore:
#
#   sum_i w_i BCE_i / G
#
# equals:
#
#   (1/G) sum_g (1/n_g) sum_i BCE_i
#
# which is exact group-balanced BCE.
# ======================================================================

def make_group_balanced_weights(group_ptr, n_branches):
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    weights = np.zeros(
        n_branches,
        dtype=np.float32
    )

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        size = end - start
        assert size > 1

        weights[start:end] = 1.0 / size

    assert np.all(weights > 0)

    n_groups = len(group_ptr) - 1

    # Sum of weights must equal number of groups.
    assert np.isclose(
        weights.sum(),
        n_groups,
        rtol=1e-5
    )

    return weights

# ======================================================================
# 6. Vectorized training loss
# ======================================================================

def compute_training_loss(logits, targets, loss_name, group_weights=None):
    per_branch = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    if loss_name == "branch_bce":
        return per_branch.mean()

    if loss_name == "group_balanced_bce":
        assert group_weights is not None

        # Exact group-balanced objective.
        return (
            per_branch * group_weights
        ).sum() / group_weights.sum()

    raise ValueError(
        f"Unknown training loss: {loss_name}"
    )

# ======================================================================
# 7. Save standardizer
# ======================================================================

def save_standardizer(dataset_name, standardizer):
    path = os.path.join(
        RQ2_SCORER_DIR,
        f"{dataset_name}_train_standardizer.json"
    )

    payload = {
        "dataset": dataset_name,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "fit_split": "train",
        "state": standardizer.state_dict(),
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False
        )

    return path

# ======================================================================
# 8. Prepare one dataset
# ======================================================================

def prepare_scorer_dataset(
    dataset_name,
    train_features,
    val_features
):
    X_train = train_features["X"]
    y_train = train_features["y"].astype(np.float32)
    train_group_ptr = train_features["group_ptr"]

    X_val = val_features["X"]
    y_val = val_features["y"].astype(np.float32)
    val_group_ptr = val_features["group_ptr"]

    assert X_train.shape[1] == AFP_FEATURE_DIM
    assert X_val.shape[1] == AFP_FEATURE_DIM

    # --------------------------------------------------------------
    # TRAIN-ONLY normalization
    # --------------------------------------------------------------
    standardizer = AFPFeatureStandardizer()

    X_train_z = standardizer.fit_transform(
        X_train
    )

    X_val_z = standardizer.transform(
        X_val
    )

    standardizer_file = save_standardizer(
        dataset_name,
        standardizer
    )

    group_weights = make_group_balanced_weights(
        train_group_ptr,
        len(y_train)
    )

    print(f"\n[{dataset_name.upper()}] preparation")
    print("Train branches:     ", len(y_train))
    print("Train groups:       ", len(train_group_ptr) - 1)
    print("Validation branches:", len(y_val))
    print("Validation groups:  ", len(val_group_ptr) - 1)
    print("Positive train rate:",
          f"{100*y_train.mean():.2f}%")
    print("Positive val rate:  ",
          f"{100*y_val.mean():.2f}%")
    print("Standardizer:       ", standardizer_file)

    return {
        "X_train": X_train_z,
        "y_train": y_train,
        "train_group_ptr": train_group_ptr,
        "group_weights": group_weights,
        "X_val": X_val_z,
        "y_val": y_val,
        "val_group_ptr": val_group_ptr,
        "standardizer": standardizer,
        "standardizer_file": standardizer_file,
    }

# ======================================================================
# 9. Train ONE run
# ======================================================================

def train_one_afp_run(
    dataset_name,
    prepared,
    hidden_dim,
    loss_name,
    seed
):
    set_afp_seed(seed)

    X_train_t = torch.as_tensor(
        prepared["X_train"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    y_train_t = torch.as_tensor(
        prepared["y_train"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    group_weights_t = torch.as_tensor(
        prepared["group_weights"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=hidden_dim,
        dropout=AFP_DROPOUT
    ).to(AFP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=AFP_LEARNING_RATE,
        weight_decay=AFP_WEIGHT_DECAY
    )

    start_time = time.time()
    final_train_loss = None

    # --------------------------------------------------------------
    # Fixed-epoch training.
    #
    # No validation-based early stopping here.
    # This avoids introducing another tuning dimension.
    # --------------------------------------------------------------
    for epoch in range(1, AFP_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(X_train_t)

        loss = compute_training_loss(
            logits=logits,
            targets=y_train_t,
            loss_name=loss_name,
            group_weights=group_weights_t
        )

        assert torch.isfinite(loss), (
            f"Non-finite training loss: "
            f"{dataset_name}, H={hidden_dim}, "
            f"loss={loss_name}, seed={seed}"
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            AFP_GRAD_CLIP
        )

        optimizer.step()

        final_train_loss = float(
            loss.detach().cpu()
        )

    runtime_sec = time.time() - start_time

    # --------------------------------------------------------------
    # Training-set loss diagnostics
    # --------------------------------------------------------------
    model.eval()

    with torch.no_grad():
        train_logits = model(X_train_t)

        final_train_branch_bce = float(
            F.binary_cross_entropy_with_logits(
                train_logits,
                y_train_t
            ).cpu()
        )

        train_group_loss = compute_training_loss(
            logits=train_logits,
            targets=y_train_t,
            loss_name="group_balanced_bce",
            group_weights=group_weights_t
        )

        final_train_group_bce = float(
            train_group_loss.cpu()
        )

    # --------------------------------------------------------------
    # Validation metrics from frozen Cell-7 evaluator
    # --------------------------------------------------------------
    val_metrics = evaluate_afp_scorer(
        model=model,
        X=prepared["X_val"],
        y=prepared["y_val"],
        group_ptr=prepared["val_group_ptr"],
        device=AFP_DEVICE
    )

    # --------------------------------------------------------------
    # Save final checkpoint.
    #
    # This is NOT yet the selected/frozen deployment model.
    # Cell 9 will select among these runs using validation only.
    # --------------------------------------------------------------
    run_name = (
        f"{dataset_name}"
        f"_h{hidden_dim}"
        f"_{loss_name}"
        f"_seed{seed}"
    )

    checkpoint_path = os.path.join(
        RQ2_SCORER_DIR,
        run_name + ".pt"
    )

    cpu_state = {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
    }

    checkpoint = {
        "dataset": dataset_name,
        "run_name": run_name,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "scorer_spec_sha256": AFP_SCORER_SPEC_SHA256,
        "training_spec_sha256": AFP_TRAINING_SPEC_SHA256,
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
        "seed": seed,
        "epochs": AFP_EPOCHS,
        "learning_rate": AFP_LEARNING_RATE,
        "weight_decay": AFP_WEIGHT_DECAY,
        "model_state_dict": cpu_state,
        "standardizer_state":
            prepared["standardizer"].state_dict(),
        "validation_metrics": val_metrics,
    }

    torch.save(
        checkpoint,
        checkpoint_path
    )

    result = {
        "dataset": dataset_name,
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
        "seed": seed,
        "epochs": AFP_EPOCHS,

        "final_train_objective":
            final_train_loss,

        "train_branch_bce":
            final_train_branch_bce,

        "train_group_balanced_bce":
            final_train_group_bce,

        "val_branch_bce":
            val_metrics["branch_bce"],

        "val_group_balanced_bce":
            val_metrics["group_balanced_bce"],

        "val_top1_positive_hit":
            val_metrics["top1_positive_hit"],

        "val_mrr_first_positive":
            val_metrics["mrr_first_positive"],

        "val_group_average_precision":
            val_metrics["group_average_precision"],

        "val_hit@1":
            val_metrics["hit@1"],

        "val_hit@2":
            val_metrics["hit@2"],

        "val_hit@4":
            val_metrics["hit@4"],

        "val_positive_recall@1":
            val_metrics["positive_recall@1"],

        "val_positive_recall@2":
            val_metrics["positive_recall@2"],

        "val_positive_recall@4":
            val_metrics["positive_recall@4"],

        "runtime_sec":
            runtime_sec,

        "checkpoint":
            checkpoint_path,
    }

    del model
    del optimizer
    del X_train_t
    del y_train_t
    del group_weights_t

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# ======================================================================
# 10. Train full predefined grid for one dataset
# ======================================================================

def train_afp_grid(
    dataset_name,
    train_features,
    val_features
):
    prepared = prepare_scorer_dataset(
        dataset_name,
        train_features,
        val_features
    )

    results = []

    total_runs = (
        len(AFP_HIDDEN_CANDIDATES)
        * len(AFP_LOSS_CANDIDATES)
        * len(AFP_TRAIN_SEEDS)
    )

    run_no = 0

    print("\n" + "=" * 84)
    print(f"TRAINING {dataset_name.upper()} AFP SCORER GRID")
    print(f"Total runs: {total_runs}")
    print("=" * 84)

    for hidden_dim in AFP_HIDDEN_CANDIDATES:
        for loss_name in AFP_LOSS_CANDIDATES:
            for seed in AFP_TRAIN_SEEDS:
                run_no += 1

                print(
                    f"\n[{run_no:02d}/{total_runs}] "
                    f"H={hidden_dim} | "
                    f"loss={loss_name} | "
                    f"seed={seed}"
                )

                result = train_one_afp_run(
                    dataset_name=dataset_name,
                    prepared=prepared,
                    hidden_dim=hidden_dim,
                    loss_name=loss_name,
                    seed=seed
                )

                results.append(result)

                print(
                    f"  train objective = "
                    f"{result['final_train_objective']:.5f}"
                )
                print(
                    f"  val Group AP    = "
                    f"{result['val_group_average_precision']:.4f}"
                )
                print(
                    f"  val MRR         = "
                    f"{result['val_mrr_first_positive']:.4f}"
                )
                print(
                    f"  val Top-1 Hit   = "
                    f"{result['val_top1_positive_hit']:.4f}"
                )
                print(
                    f"  val Group BCE   = "
                    f"{result['val_group_balanced_bce']:.4f}"
                )
                print(
                    f"  runtime         = "
                    f"{result['runtime_sec']:.2f}s"
                )

    return prepared, results

# ======================================================================
# 11. WEBQSP training grid
# ======================================================================

webqsp_scorer_prepared, webqsp_scorer_results = train_afp_grid(
    dataset_name="webqsp",
    train_features=webqsp_train_features,
    val_features=webqsp_val_features
)

# ======================================================================
# 12. CWQ training grid
# ======================================================================

cwq_scorer_prepared, cwq_scorer_results = train_afp_grid(
    dataset_name="cwq",
    train_features=cwq_train_features,
    val_features=cwq_val_features
)

# ======================================================================
# 13. Combine and save run-level results
# ======================================================================

afp_scorer_results = (
    webqsp_scorer_results
    + cwq_scorer_results
)

afp_scorer_results_df = pd.DataFrame(
    afp_scorer_results
)

results_csv = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_runs.csv"
)

afp_scorer_results_df.to_csv(
    results_csv,
    index=False
)

results_json = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_runs.json"
)

with open(
    results_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        afp_scorer_results,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 14. Aggregate across seeds
# ======================================================================
#
# IMPORTANT:
# This is descriptive only.
# We are NOT selecting the winner in this cell.
# ======================================================================

metric_columns = [
    "val_group_average_precision",
    "val_mrr_first_positive",
    "val_top1_positive_hit",
    "val_group_balanced_bce",
    "val_branch_bce",
    "val_positive_recall@1",
    "val_positive_recall@2",
    "val_positive_recall@4",
]

aggregate = (
    afp_scorer_results_df
    .groupby(
        ["dataset", "hidden_dim", "loss_name"]
    )[metric_columns]
    .agg(["mean", "std"])
    .reset_index()
)

aggregate_csv = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_aggregate.csv"
)

aggregate.to_csv(
    aggregate_csv,
    index=False
)

# ======================================================================
# 15. Compact validation summary
# ======================================================================

summary = (
    afp_scorer_results_df
    .groupby(
        ["dataset", "hidden_dim", "loss_name"]
    )
    .agg(
        group_ap_mean=(
            "val_group_average_precision",
            "mean"
        ),
        group_ap_std=(
            "val_group_average_precision",
            "std"
        ),
        mrr_mean=(
            "val_mrr_first_positive",
            "mean"
        ),
        top1_mean=(
            "val_top1_positive_hit",
            "mean"
        ),
        group_bce_mean=(
            "val_group_balanced_bce",
            "mean"
        ),
        branch_bce_mean=(
            "val_branch_bce",
            "mean"
        ),
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("VALIDATION SUMMARY ACROSS 3 SEEDS — DESCRIPTIVE ONLY")
print("=" * 100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    subset = summary[
        summary["dataset"] == dataset_name
    ]

    print(
        subset[
            [
                "hidden_dim",
                "loss_name",
                "group_ap_mean",
                "group_ap_std",
                "mrr_mean",
                "top1_mean",
                "group_bce_mean",
                "branch_bce_mean",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 16. Final training manifest
# ======================================================================

training_manifest = {
    "training_version": AFP_TRAINING_VERSION,
    "training_spec_sha256": AFP_TRAINING_SPEC_SHA256,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
    "scorer_spec_sha256": AFP_SCORER_SPEC_SHA256,
    "datasets": ["webqsp", "cwq"],
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "loss_candidates": AFP_LOSS_CANDIDATES,
    "seeds": AFP_TRAIN_SEEDS,
    "epochs": AFP_EPOCHS,
    "learning_rate": AFP_LEARNING_RATE,
    "weight_decay": AFP_WEIGHT_DECAY,
    "class_weighting": False,
    "validation_early_stopping": False,
    "test_used": False,
    "n_runs_total": len(afp_scorer_results),
    "results_csv": results_csv,
    "aggregate_csv": aggregate_csv,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_manifest,
        f,
        indent=2
    )

print("\n" + "=" * 84)
print("=== RQ2 CELL 8: AFP SCORER TRAINING COMPLETE ===")
print("=" * 84)
print("Total runs:", len(afp_scorer_results))
print("Runs per dataset: 18")
print("Hidden sizes:", AFP_HIDDEN_CANDIDATES)
print("Losses:", AFP_LOSS_CANDIDATES)
print("Seeds:", AFP_TRAIN_SEEDS)
print("Validation early stopping: NO")
print("Class weighting: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Results:", results_csv)
print("\nNext: validation comparison + scorer selection/freeze.")

Training version: afp_train_v1
Device:           cuda
Hidden sizes:     [32, 64, 128]
Losses:           ['branch_bce', 'group_balanced_bce']
Seeds:            [42, 43, 44]
Epochs:           80
Learning rate:    0.001
Weight decay:     0.0001
Training SHA256:  c91adb1c90f9e75d...
Scorer directory: /kaggle/working/step3_rq2_dev_v1/04_scorer

[WEBQSP] preparation
Train branches:      18437
Train groups:        1457
Validation branches: 966
Validation groups:   87
Positive train rate: 39.60%
Positive val rate:   33.02%
Standardizer:        /kaggle/working/step3_rq2_dev_v1/04_scorer/webqsp_train_standardizer.json

TRAINING WEBQSP AFP SCORER GRID
Total runs: 18

[01/18] H=32 | loss=branch_bce | seed=42
  train objective = 0.57420
  val Group AP    = 0.6104
  val MRR         = 0.6512
  val Top-1 Hit   = 0.5057
  val Group BCE   = 0.6264
  runtime         = 0.39s

[02/18] H=32 | loss=branch_bce | seed=43
  train objective = 0.57767
  val Group AP    = 0.6059
  val MRR         = 0.6436
  val To

## AFP Scorer Diagnostic Gate

In [100]:
# ======================================================================
# 3.9A AFP SCORER DIAGNOSTIC GATE
# ======================================================================
#
# Purpose:
#   Diagnose scorer v1 BEFORE validation selection/final freeze.
#
# Tests:
#   1. Proper random-ranking null distribution
#   2. TRAIN vs VALIDATION ranking/generalization
#   3. Within-group feature variation
#   4. Candidate-level feature signal after removing group-level effects
#   5. Diagnostic subgroup performance
#
# IMPORTANT:
#   - TRAIN + VALIDATION only
#   - NO TEST data/gold
#   - NO new training
#   - NO winner frozen
#   - NO architecture changed
# ======================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"
assert AFP_FEATURE_DIM == 27
assert len(afp_scorer_results_df) == 36

required = [
    "compute_group_ranking_metrics",
    "AFPScorer",
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
    "webqsp_scorer_prepared",
    "cwq_scorer_prepared",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing required objects: " + ", ".join(missing)

DIAG_DIR = os.path.join(
    os.path.dirname(RQ2_SCORER_DIR),
    "05_scorer_diagnostics"
)
os.makedirs(DIAG_DIR, exist_ok=True)

print("Diagnostic directory:", DIAG_DIR)

# ======================================================================
# 2. Frozen Feature-v2 names
# ======================================================================

AFP_FEATURE_NAMES_V2 = [
    "sem_q_candidate",                    # 0
    "sem_candidate_surface_available",    # 1
    "sem_q_current_entity",               # 2
    "sem_current_surface_available",      # 3
    "sem_q_current_relation",             # 4
    "sem_q_full_plan",                    # 5
    "sem_q_remaining_suffix",             # 6
    "sem_candidate_current_relation",     # 7

    "path_q_prefix_entity_mean",           # 8
    "path_candidate_prefix_entity_mean",   # 9
    "path_prefix_surface_fraction",        # 10
    "path_candidate_repeats_entity",       # 11
    "path_candidate_occurrence_fraction",  # 12
    "path_unique_entity_ratio",            # 13
    "path_relation_repeat_fraction_before",# 14

    "struct_log_candidate_count",          # 15
    "struct_log_unique_candidate_entities",# 16
    "struct_log_contributing_parents",     # 17
    "struct_log_parent_fanout",            # 18
    "struct_parent_frontier_share",        # 19
    "struct_log_endpoint_multiplicity",    # 20
    "struct_endpoint_frontier_share",      # 21
    "struct_duplicate_endpoint_ratio",     # 22

    "prog_hop_fraction",                   # 23
    "prog_remaining_fraction",             # 24
    "prog_log_plan_length",                # 25
    "prog_penultimate_indicator",          # 26
]

assert len(AFP_FEATURE_NAMES_V2) == 27

# ======================================================================
# 3. Proper random-ranking null distribution
# ======================================================================
#
# Cell 7 used only one random seed.
# Here we build an empirical null distribution over 1000 random rankings.
# ======================================================================

RANDOM_NULL_TRIALS = 1000
RANDOM_NULL_BASE_SEED = 20260831

def build_random_null(y, group_ptr, n_trials=1000, base_seed=20260831):
    y = np.asarray(y, dtype=np.uint8)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    rows = []

    for trial in tqdm(
        range(n_trials),
        desc="Random null",
        leave=False
    ):
        rng = np.random.default_rng(base_seed + trial)
        scores = rng.random(len(y))

        m = compute_group_ranking_metrics(
            scores=scores,
            labels=y,
            group_ptr=group_ptr,
            ks=(1, 2, 4)
        )

        rows.append({
            "trial": trial,
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
            "hit2": m["hit@2"],
            "recall2": m["positive_recall@2"],
        })

    return pd.DataFrame(rows)

def summarize_null(df):
    rows = []

    for metric in ["group_ap", "mrr", "top1", "hit2", "recall2"]:
        values = df[metric].values

        rows.append({
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
            "p2.5": np.percentile(values, 2.5),
            "p50": np.percentile(values, 50),
            "p97.5": np.percentile(values, 97.5),
        })

    return pd.DataFrame(rows)

print("\nBuilding WebQSP random null...")
webqsp_random_null = build_random_null(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"],
    RANDOM_NULL_TRIALS,
    RANDOM_NULL_BASE_SEED
)

print("Building CWQ random null...")
cwq_random_null = build_random_null(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"],
    RANDOM_NULL_TRIALS,
    RANDOM_NULL_BASE_SEED + 10000
)

webqsp_null_summary = summarize_null(webqsp_random_null)
cwq_null_summary = summarize_null(cwq_random_null)

print("\n" + "="*90)
print("RANDOM-RANKING NULL — WEBQSP")
print("="*90)
print(webqsp_null_summary.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\n" + "="*90)
print("RANDOM-RANKING NULL — CWQ")
print("="*90)
print(cwq_null_summary.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

# ======================================================================
# 4. Checkpoint loader
# ======================================================================

def load_afp_checkpoint(path, device="cpu"):
    try:
        ckpt = torch.load(
            path,
            map_location=device,
            weights_only=False
        )
    except TypeError:
        ckpt = torch.load(
            path,
            map_location=device
        )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=int(ckpt["hidden_dim"]),
        dropout=AFP_DROPOUT
    ).to(device)

    model.load_state_dict(
        ckpt["model_state_dict"]
    )
    model.eval()

    return model, ckpt

@torch.no_grad()
def predict_afp_scores(model, X, device):
    X_t = torch.as_tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    logits = model(X_t)
    probs = torch.sigmoid(logits)

    return probs.detach().cpu().numpy()

# ======================================================================
# 5. TRAIN vs VALIDATION ranking for every saved run
# ======================================================================

def get_prepared(dataset_name):
    if dataset_name == "webqsp":
        return webqsp_scorer_prepared
    if dataset_name == "cwq":
        return cwq_scorer_prepared
    raise ValueError(dataset_name)

generalization_rows = []

print("\nEvaluating TRAIN vs VALIDATION ranking...")

for _, row in tqdm(
    afp_scorer_results_df.iterrows(),
    total=len(afp_scorer_results_df)
):
    dataset_name = row["dataset"]
    prepared = get_prepared(dataset_name)

    model, ckpt = load_afp_checkpoint(
        row["checkpoint"],
        AFP_DEVICE
    )

    train_scores = predict_afp_scores(
        model,
        prepared["X_train"],
        AFP_DEVICE
    )

    val_scores = predict_afp_scores(
        model,
        prepared["X_val"],
        AFP_DEVICE
    )

    train_rank = compute_group_ranking_metrics(
        train_scores,
        prepared["y_train"],
        prepared["train_group_ptr"],
        ks=(1, 2, 4)
    )

    val_rank = compute_group_ranking_metrics(
        val_scores,
        prepared["y_val"],
        prepared["val_group_ptr"],
        ks=(1, 2, 4)
    )

    generalization_rows.append({
        "dataset": dataset_name,
        "hidden_dim": int(row["hidden_dim"]),
        "loss_name": row["loss_name"],
        "seed": int(row["seed"]),

        "train_group_ap":
            train_rank["group_average_precision"],
        "val_group_ap":
            val_rank["group_average_precision"],
        "gap_group_ap":
            train_rank["group_average_precision"]
            - val_rank["group_average_precision"],

        "train_mrr":
            train_rank["mrr_first_positive"],
        "val_mrr":
            val_rank["mrr_first_positive"],
        "gap_mrr":
            train_rank["mrr_first_positive"]
            - val_rank["mrr_first_positive"],

        "train_top1":
            train_rank["top1_positive_hit"],
        "val_top1":
            val_rank["top1_positive_hit"],
        "gap_top1":
            train_rank["top1_positive_hit"]
            - val_rank["top1_positive_hit"],
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

generalization_df = pd.DataFrame(generalization_rows)

generalization_agg = (
    generalization_df
    .groupby(["dataset", "hidden_dim", "loss_name"])
    .agg(
        train_group_ap=("train_group_ap", "mean"),
        val_group_ap=("val_group_ap", "mean"),
        gap_group_ap=("gap_group_ap", "mean"),
        train_mrr=("train_mrr", "mean"),
        val_mrr=("val_mrr", "mean"),
        gap_mrr=("gap_mrr", "mean"),
        train_top1=("train_top1", "mean"),
        val_top1=("val_top1", "mean"),
        gap_top1=("gap_top1", "mean"),
    )
    .reset_index()
)

print("\n" + "="*105)
print("TRAIN → VALIDATION GENERALIZATION")
print("="*105)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = generalization_agg[
        generalization_agg["dataset"] == dataset_name
    ]

    print(sub.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 6. Compare validation configuration means against random null
# ======================================================================

config_mean = (
    generalization_df
    .groupby(["dataset", "hidden_dim", "loss_name"])
    .agg(
        group_ap=("val_group_ap", "mean"),
        mrr=("val_mrr", "mean"),
        top1=("val_top1", "mean"),
    )
    .reset_index()
)

def empirical_upper_p(null_values, observed):
    null_values = np.asarray(null_values)
    return float(
        (1 + np.sum(null_values >= observed))
        / (len(null_values) + 1)
    )

null_comparison_rows = []

for _, row in config_mean.iterrows():
    dataset_name = row["dataset"]

    null_df = (
        webqsp_random_null
        if dataset_name == "webqsp"
        else cwq_random_null
    )

    null_comparison_rows.append({
        "dataset": dataset_name,
        "hidden_dim": int(row["hidden_dim"]),
        "loss_name": row["loss_name"],

        "group_ap": row["group_ap"],
        "random_group_ap_mean":
            null_df["group_ap"].mean(),
        "group_ap_delta":
            row["group_ap"] - null_df["group_ap"].mean(),
        "group_ap_null_p":
            empirical_upper_p(
                null_df["group_ap"],
                row["group_ap"]
            ),

        "mrr": row["mrr"],
        "random_mrr_mean":
            null_df["mrr"].mean(),
        "mrr_delta":
            row["mrr"] - null_df["mrr"].mean(),
        "mrr_null_p":
            empirical_upper_p(
                null_df["mrr"],
                row["mrr"]
            ),

        "top1": row["top1"],
        "random_top1_mean":
            null_df["top1"].mean(),
        "top1_delta":
            row["top1"] - null_df["top1"].mean(),
        "top1_null_p":
            empirical_upper_p(
                null_df["top1"],
                row["top1"]
            ),
    })

null_comparison_df = pd.DataFrame(
    null_comparison_rows
)

print("\n" + "="*110)
print("VALIDATION SCORER vs RANDOM-RANKING NULL")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = null_comparison_df[
        null_comparison_df["dataset"] == dataset_name
    ]

    cols = [
        "hidden_dim",
        "loss_name",
        "group_ap",
        "random_group_ap_mean",
        "group_ap_delta",
        "group_ap_null_p",
        "mrr_delta",
        "mrr_null_p",
        "top1_delta",
        "top1_null_p",
    ]

    print(sub[cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 7. Within-group feature variation
# ======================================================================
#
# A feature that is constant across every candidate in a frontier
# cannot rank those candidates.
# ======================================================================

def feature_variation_profile(X, group_ptr, feature_names, tol=1e-8):
    X = np.asarray(X, dtype=np.float32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    n_features = X.shape[1]
    varying_count = np.zeros(n_features, dtype=np.int64)
    std_sum = np.zeros(n_features, dtype=np.float64)

    n_groups = len(group_ptr) - 1

    for g in range(n_groups):
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        Xg = X[s:e]
        ranges = Xg.max(axis=0) - Xg.min(axis=0)
        stds = Xg.std(axis=0)

        varying_count += (ranges > tol)
        std_sum += stds

    return pd.DataFrame({
        "feature_index": np.arange(n_features),
        "feature": feature_names,
        "varying_groups": varying_count,
        "n_groups": n_groups,
        "varying_group_rate":
            varying_count / n_groups,
        "mean_within_group_std":
            std_sum / n_groups,
    })

variation_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    v = feature_variation_profile(
        data["X"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    v.insert(0, "split", split)
    v.insert(0, "dataset", dataset_name)
    variation_frames.append(v)

feature_variation_df = pd.concat(
    variation_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("WITHIN-GROUP FEATURE VARIATION — VALIDATION")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = feature_variation_df[
        (feature_variation_df["dataset"] == dataset_name)
        &
        (feature_variation_df["split"] == "validation")
    ].sort_values(
        "varying_group_rate",
        ascending=False
    )

    print(sub[
        [
            "feature_index",
            "feature",
            "varying_group_rate",
            "mean_within_group_std",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 8. Group-center features
# ======================================================================
#
# Removing each frontier's feature mean eliminates pure group-level
# offsets. What remains is candidate-relative variation.
# ======================================================================

def group_center_features(X, group_ptr):
    X = np.asarray(X, dtype=np.float32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    Z = np.empty_like(X)

    for g in range(len(group_ptr) - 1):
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        Xg = X[s:e]
        Z[s:e] = Xg - Xg.mean(axis=0, keepdims=True)

    return Z

# ======================================================================
# 9. Candidate-relative feature signal
# ======================================================================
#
# AUC = 0.5  -> no directionally useful signal
# AUC > 0.5  -> higher feature tends to indicate positive branch
# AUC < 0.5  -> lower feature tends to indicate positive branch
#
# best_auc = max(AUC, 1-AUC)
#
# This is diagnostic only; it is NOT feature selection.
# ======================================================================

def feature_signal_profile(X, y, group_ptr, feature_names):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.uint8)

    centered = group_center_features(
        X,
        group_ptr
    )

    rows = []

    for j, name in enumerate(feature_names):
        x = centered[:, j]

        pos = x[y == 1]
        neg = x[y == 0]

        if np.allclose(x, x[0]):
            auc = 0.5
        else:
            auc = roc_auc_score(y, x)

        rows.append({
            "feature_index": j,
            "feature": name,
            "positive_mean_centered":
                float(pos.mean()),
            "negative_mean_centered":
                float(neg.mean()),
            "centered_delta":
                float(pos.mean() - neg.mean()),
            "auc_higher_is_positive":
                float(auc),
            "best_orientation_auc":
                float(max(auc, 1.0 - auc)),
            "preferred_direction":
                "higher"
                if auc >= 0.5
                else "lower",
        })

    return pd.DataFrame(rows)

signal_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    sig = feature_signal_profile(
        data["X"],
        data["y"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    sig.insert(0, "split", split)
    sig.insert(0, "dataset", dataset_name)
    signal_frames.append(sig)

feature_signal_df = pd.concat(
    signal_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("CANDIDATE-RELATIVE FEATURE SIGNAL — VALIDATION")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = feature_signal_df[
        (feature_signal_df["dataset"] == dataset_name)
        &
        (feature_signal_df["split"] == "validation")
    ].sort_values(
        "best_orientation_auc",
        ascending=False
    )

    print(sub[
        [
            "feature_index",
            "feature",
            "centered_delta",
            "auc_higher_is_positive",
            "best_orientation_auc",
            "preferred_direction",
        ]
    ].head(15).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 10. Diagnostic reference configuration
# ======================================================================
#
# For subgroup inspection ONLY:
#   choose the configuration with best MEAN validation Group AP.
#
# This is NOT final model selection/freeze.
# The three seeds are ensembled to reduce initialization noise.
# ======================================================================

def get_diagnostic_reference(dataset_name):
    sub = (
        config_mean[
            config_mean["dataset"] == dataset_name
        ]
        .sort_values(
            ["group_ap", "mrr", "top1"],
            ascending=[False, False, False]
        )
        .iloc[0]
    )

    return (
        int(sub["hidden_dim"]),
        sub["loss_name"]
    )

def ensemble_scores_for_config(
    dataset_name,
    hidden_dim,
    loss_name,
    prepared
):
    rows = afp_scorer_results_df[
        (afp_scorer_results_df["dataset"] == dataset_name)
        &
        (afp_scorer_results_df["hidden_dim"] == hidden_dim)
        &
        (afp_scorer_results_df["loss_name"] == loss_name)
    ]

    assert len(rows) == 3

    scores = []

    for _, row in rows.iterrows():
        model, _ = load_afp_checkpoint(
            row["checkpoint"],
            AFP_DEVICE
        )

        scores.append(
            predict_afp_scores(
                model,
                prepared["X_val"],
                AFP_DEVICE
            )
        )

        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.mean(
        np.stack(scores, axis=0),
        axis=0
    )

# ======================================================================
# 11. Subgroup-ranking helper
# ======================================================================

def subset_groups_metrics(
    scores,
    labels,
    group_ptr,
    selected_group_indices
):
    new_scores = []
    new_labels = []
    new_ptr = [0]

    for g in selected_group_indices:
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        new_scores.extend(scores[s:e])
        new_labels.extend(labels[s:e])
        new_ptr.append(
            new_ptr[-1] + (e - s)
        )

    if len(selected_group_indices) == 0:
        return None

    return compute_group_ranking_metrics(
        np.asarray(new_scores),
        np.asarray(new_labels),
        np.asarray(new_ptr),
        ks=(1, 2, 4)
    )

def diagnostic_subgroups(dataset_name, features, scores):
    y = features["y"]
    ptr = features["group_ptr"]
    hops = features["group_hop"]
    sizes = features["group_candidate_count"]
    X = features["X"]

    rows = []
    n_groups = len(ptr) - 1

    # --------------------------------------------------------------
    # Hop
    # --------------------------------------------------------------
    for hop in sorted(np.unique(hops)):
        idx = np.flatnonzero(hops == hop)

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "hop",
            "subgroup": f"hop_{int(hop)}",
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    # --------------------------------------------------------------
    # Candidate-set size
    # --------------------------------------------------------------
    size_bins = [
        ("2-4", 2, 4),
        ("5-8", 5, 8),
        ("9-16", 9, 16),
        ("17+", 17, np.inf),
    ]

    for label, lo, hi in size_bins:
        mask = (sizes >= lo) & (sizes <= hi)
        idx = np.flatnonzero(mask)

        if len(idx) == 0:
            continue

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "candidate_size",
            "subgroup": label,
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    # --------------------------------------------------------------
    # Candidate entity-surface availability
    # --------------------------------------------------------------
    availability = []

    for g in range(n_groups):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        a = X[
            s:e,
            1  # candidate-surface availability
        ]

        if np.all(a == 0):
            availability.append("all_raw_mid")
        elif np.all(a == 1):
            availability.append("all_readable")
        else:
            availability.append("mixed")

    availability = np.asarray(availability)

    for category in [
        "all_raw_mid",
        "mixed",
        "all_readable"
    ]:
        idx = np.flatnonzero(
            availability == category
        )

        if len(idx) == 0:
            continue

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "candidate_surface",
            "subgroup": category,
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    return pd.DataFrame(rows)

diag_reference = {}
subgroup_frames = []

for dataset_name, features, prepared in [
    (
        "webqsp",
        webqsp_val_features,
        webqsp_scorer_prepared
    ),
    (
        "cwq",
        cwq_val_features,
        cwq_scorer_prepared
    ),
]:
    hidden_dim, loss_name = get_diagnostic_reference(
        dataset_name
    )

    diag_reference[dataset_name] = {
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
    }

    scores = ensemble_scores_for_config(
        dataset_name,
        hidden_dim,
        loss_name,
        prepared
    )

    sg = diagnostic_subgroups(
        dataset_name,
        features,
        scores
    )

    subgroup_frames.append(sg)

    print(
        f"\nDiagnostic reference {dataset_name.upper()}: "
        f"H={hidden_dim}, loss={loss_name}"
    )

subgroup_df = pd.concat(
    subgroup_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("DIAGNOSTIC REFERENCE — VALIDATION SUBGROUP PERFORMANCE")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    print(
        subgroup_df[
            subgroup_df["dataset"] == dataset_name
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 12. Save diagnostics
# ======================================================================

webqsp_random_null.to_csv(
    os.path.join(
        DIAG_DIR,
        "webqsp_random_null.csv"
    ),
    index=False
)

cwq_random_null.to_csv(
    os.path.join(
        DIAG_DIR,
        "cwq_random_null.csv"
    ),
    index=False
)

null_comparison_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "scorer_vs_random_null.csv"
    ),
    index=False
)

generalization_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "train_validation_generalization_runs.csv"
    ),
    index=False
)

generalization_agg.to_csv(
    os.path.join(
        DIAG_DIR,
        "train_validation_generalization_aggregate.csv"
    ),
    index=False
)

feature_variation_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "within_group_feature_variation.csv"
    ),
    index=False
)

feature_signal_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "candidate_relative_feature_signal.csv"
    ),
    index=False
)

subgroup_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "validation_subgroup_diagnostics.csv"
    ),
    index=False
)

diagnostic_manifest = {
    "diagnostic_version":
        "afp_scorer_diagnostic_v1",

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_spec_sha256":
        AFP_SCORER_SPEC_SHA256,

    "training_spec_sha256":
        AFP_TRAINING_SPEC_SHA256,

    "random_null_trials":
        RANDOM_NULL_TRIALS,

    "diagnostic_reference":
        diag_reference,

    "train_used":
        True,

    "validation_used":
        True,

    "test_used":
        False,

    "new_training_performed":
        False,

    "winner_selected":
        False,

    "final_scorer_frozen":
        False,
}

with open(
    os.path.join(
        DIAG_DIR,
        "scorer_diagnostic_manifest.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        diagnostic_manifest,
        f,
        indent=2
    )

# ======================================================================
# 13. Final diagnostic gate report
# ======================================================================

print("\n" + "="*90)
print("=== RQ2 CELL 9A: AFP SCORER DIAGNOSTIC GATE COMPLETE ===")
print("="*90)

print("Random null trials:", RANDOM_NULL_TRIALS)
print("TRAIN ranking evaluated: YES")
print("VALIDATION ranking evaluated: YES")
print("Within-group feature variation evaluated: YES")
print("Candidate-relative feature signal evaluated: YES")
print("Subgroup diagnostics evaluated: YES")
print("New scorer training: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Final scorer frozen: NO")
print("\nSTOP HERE.")
print("Send the diagnostic output before any scorer selection/freeze.")

Diagnostic directory: /kaggle/working/step3_rq2_dev_v1/05_scorer_diagnostics

Building WebQSP random null...


Building CWQ random null...



RANDOM-RANKING NULL — WEBQSP
  metric   mean    std   p2.5    p50  p97.5
group_ap 0.6098 0.0202 0.5725 0.6095 0.6523
     mrr 0.6458 0.0274 0.5942 0.6459 0.7015
    top1 0.4754 0.0431 0.3908 0.4713 0.5632
    hit2 0.6917 0.0361 0.6207 0.6897 0.7586
 recall2 0.4328 0.0269 0.3823 0.4320 0.4850

RANDOM-RANKING NULL — CWQ
  metric   mean    std   p2.5    p50  p97.5
group_ap 0.5257 0.0058 0.5144 0.5257 0.5370
     mrr 0.5496 0.0069 0.5357 0.5496 0.5629
    top1 0.3617 0.0103 0.3402 0.3617 0.3809
    hit2 0.5669 0.0098 0.5481 0.5666 0.5858
 recall2 0.3987 0.0083 0.3830 0.3987 0.4148

Evaluating TRAIN vs VALIDATION ranking...


100%|██████████| 36/36 [00:11<00:00,  3.06it/s]



TRAIN → VALIDATION GENERALIZATION

WEBQSP
dataset  hidden_dim          loss_name  train_group_ap  val_group_ap  gap_group_ap  train_mrr  val_mrr  gap_mrr  train_top1  val_top1  gap_top1
 webqsp          32         branch_bce          0.6353        0.6065        0.0288     0.6728   0.6448   0.0280      0.5109    0.4904    0.0204
 webqsp          32 group_balanced_bce          0.6372        0.6052        0.0320     0.6729   0.6462   0.0266      0.5102    0.4943    0.0159
 webqsp          64         branch_bce          0.6369        0.5991        0.0378     0.6748   0.6335   0.0413      0.5125    0.4751    0.0374
 webqsp          64 group_balanced_bce          0.6416        0.6064        0.0351     0.6775   0.6475   0.0300      0.5164    0.4981    0.0183
 webqsp         128         branch_bce          0.6383        0.5930        0.0453     0.6722   0.6275   0.0447      0.5081    0.4598    0.0484
 webqsp         128 group_balanced_bce          0.6412        0.6041        0.0371     0.6771

## Candidate Distinguishability & Feature Revision Gate

In [101]:
# ======================================================================
# CANDIDATE DISTINGUISHABILITY and FEATURE REVISION GATE
# ======================================================================
#
# Purpose:
#   Determine whether AFP scorer v1 is limited primarily by:
#     (A) candidate representation / feature indistinguishability, or
#     (B) training objective / scorer capacity.
#
# Diagnostics:
#   1. Exact feature-vector collisions within each decision group
#   2. All-identical frontier rate
#   3. Unique representation ratio
#   4. Positive/negative conflicts inside identical feature classes
#   5. Unavoidable positive-negative tie fraction
#   6. Representation-aware optimistic Top-1 ceiling
#   7. Raw-MID vs readable candidate groups
#   8. Candidate-wise pairwise feature discrimination
#
# IMPORTANT:
#   - TRAIN + VALIDATION only
#   - NO TEST data/gold
#   - NO scorer training
#   - NO feature modification
#   - NO final scorer selection/freeze
# ======================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"

required = [
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
    "webqsp_scorer_prepared",
    "cwq_scorer_prepared",
    "feature_variation_df",
    "null_comparison_df",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing Cell 6/8/9A objects: " + ", ".join(missing)

FEATURE_REVISION_DIR = os.path.join(
    os.path.dirname(RQ2_SCORER_DIR),
    "06_feature_revision_gate"
)
os.makedirs(FEATURE_REVISION_DIR, exist_ok=True)

print("Feature revision gate directory:", FEATURE_REVISION_DIR)

# ======================================================================
# 2. Feature names
# ======================================================================

AFP_FEATURE_NAMES_V2 = [
    "sem_q_candidate",
    "sem_candidate_surface_available",
    "sem_q_current_entity",
    "sem_current_surface_available",
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",
    "sem_candidate_current_relation",

    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",
    "path_prefix_surface_fraction",
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]

assert len(AFP_FEATURE_NAMES_V2) == 27

# Candidate-surface availability feature
CANDIDATE_SURFACE_INDEX = 1

# ======================================================================
# 3. Stable feature signature
# ======================================================================
#
# Features are deterministic float32 values.
# We round to 7 decimals only to avoid meaningless floating-point noise.
#
# This is NOT approximate clustering.
# It is used only to identify effectively identical Feature-v2 vectors.
# ======================================================================

FEATURE_SIGNATURE_DECIMALS = 7

def feature_signature(x):
    x = np.asarray(x, dtype=np.float64)
    return tuple(
        np.round(
            x,
            FEATURE_SIGNATURE_DECIMALS
        ).tolist()
    )

# ======================================================================
# 4. Analyze one decision group
# ======================================================================

def analyze_group_representation(Xg, yg):
    Xg = np.asarray(Xg, dtype=np.float32)
    yg = np.asarray(yg, dtype=np.uint8)

    n = len(yg)
    n_pos = int(yg.sum())
    n_neg = n - n_pos

    assert n > 1
    assert n_pos >= 1

    classes = {}

    for i in range(n):
        sig = feature_signature(Xg[i])

        if sig not in classes:
            classes[sig] = []

        classes[sig].append(i)

    n_unique = len(classes)
    all_identical = (n_unique == 1)

    mixed_classes = 0
    mixed_candidates = 0

    unavoidable_tie_pairs = 0

    class_positive_rates = []

    for indices in classes.values():
        labels = yg[indices]

        p = int(labels.sum())
        q = len(labels) - p

        class_positive_rates.append(
            p / len(labels)
        )

        # Identical feature vector contains BOTH labels.
        if p > 0 and q > 0:
            mixed_classes += 1
            mixed_candidates += len(labels)

        # Every positive-negative pair inside the same exact feature
        # class is impossible for a deterministic scorer to order.
        unavoidable_tie_pairs += p * q

    total_pos_neg_pairs = n_pos * n_neg

    if total_pos_neg_pairs > 0:
        unavoidable_pair_tie_fraction = (
            unavoidable_tie_pairs
            / total_pos_neg_pairs
        )
    else:
        unavoidable_pair_tie_fraction = 0.0

    # --------------------------------------------------------------
    # Optimistic representation-aware Top-1 ceiling
    # --------------------------------------------------------------
    #
    # Suppose an oracle knew which REPRESENTATION CLASS was best,
    # but still could not distinguish candidates sharing the exact
    # same vector.
    #
    # Within the selected class, expected Top-1 success is its
    # positive fraction.
    #
    # This therefore quantifies ambiguity induced by the representation.
    # --------------------------------------------------------------
    optimistic_top1_ceiling = max(
        class_positive_rates
    )

    return {
        "n_candidates": n,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "n_unique_vectors": n_unique,
        "unique_vector_ratio": n_unique / n,
        "all_identical": int(all_identical),
        "mixed_label_classes": mixed_classes,
        "has_label_conflict": int(mixed_classes > 0),
        "mixed_class_candidates": mixed_candidates,
        "mixed_candidate_fraction": mixed_candidates / n,
        "unavoidable_tie_pairs": unavoidable_tie_pairs,
        "total_pos_neg_pairs": total_pos_neg_pairs,
        "unavoidable_pair_tie_fraction":
            unavoidable_pair_tie_fraction,
        "optimistic_top1_representation_ceiling":
            optimistic_top1_ceiling,
    }

# ======================================================================
# 5. Candidate-surface subgroup
# ======================================================================

def candidate_surface_group(Xg):
    availability = Xg[:, CANDIDATE_SURFACE_INDEX]

    if np.all(availability < 0.5):
        return "all_raw_mid"

    if np.all(availability >= 0.5):
        return "all_readable"

    return "mixed"

# ======================================================================
# 6. Analyze complete feature dataset
# ======================================================================

def analyze_dataset_representation(
    dataset_name,
    split,
    features
):
    X = np.asarray(
        features["X"],
        dtype=np.float32
    )

    y = np.asarray(
        features["y"],
        dtype=np.uint8
    )

    ptr = np.asarray(
        features["group_ptr"],
        dtype=np.int64
    )

    hops = np.asarray(
        features["group_hop"],
        dtype=np.int64
    )

    sizes = np.asarray(
        features["group_candidate_count"],
        dtype=np.int64
    )

    rows = []

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        Xg = X[s:e]
        yg = y[s:e]

        r = analyze_group_representation(
            Xg,
            yg
        )

        r.update({
            "dataset": dataset_name,
            "split": split,
            "group_index": g,
            "hop": int(hops[g]),
            "candidate_size": int(sizes[g]),
            "surface_group":
                candidate_surface_group(Xg),
        })

        rows.append(r)

    return pd.DataFrame(rows)

# ======================================================================
# 7. Run representation diagnostics
# ======================================================================

representation_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    print(
        f"Analyzing {dataset_name.upper()} {split}..."
    )

    df = analyze_dataset_representation(
        dataset_name,
        split,
        data
    )

    representation_frames.append(df)

representation_df = pd.concat(
    representation_frames,
    ignore_index=True
)

# ======================================================================
# 8. Aggregate representation diagnostics
# ======================================================================

def aggregate_representation(df):
    total_pos_neg_pairs = df[
        "total_pos_neg_pairs"
    ].sum()

    total_unavoidable = df[
        "unavoidable_tie_pairs"
    ].sum()

    weighted_pair_tie_fraction = (
        total_unavoidable / total_pos_neg_pairs
        if total_pos_neg_pairs > 0
        else 0.0
    )

    return {
        "n_groups": len(df),

        "all_identical_groups":
            int(df["all_identical"].sum()),

        "all_identical_rate":
            float(df["all_identical"].mean()),

        "groups_with_label_conflict":
            int(df["has_label_conflict"].sum()),

        "label_conflict_rate":
            float(df["has_label_conflict"].mean()),

        "mean_unique_vector_ratio":
            float(df["unique_vector_ratio"].mean()),

        "median_unique_vector_ratio":
            float(df["unique_vector_ratio"].median()),

        "mean_mixed_candidate_fraction":
            float(df["mixed_candidate_fraction"].mean()),

        "weighted_unavoidable_pair_tie_fraction":
            float(weighted_pair_tie_fraction),

        "mean_optimistic_top1_ceiling":
            float(
                df[
                    "optimistic_top1_representation_ceiling"
                ].mean()
            ),
    }

representation_summary_rows = []

for (dataset_name, split), sub in representation_df.groupby(
    ["dataset", "split"]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "split": split,
    })

    representation_summary_rows.append(stats)

representation_summary_df = pd.DataFrame(
    representation_summary_rows
)

print("\n" + "="*110)
print("FEATURE-v2 REPRESENTATION DISTINGUISHABILITY")
print("="*110)

print(
    representation_summary_df[
        [
            "dataset",
            "split",
            "n_groups",
            "all_identical_rate",
            "label_conflict_rate",
            "mean_unique_vector_ratio",
            "weighted_unavoidable_pair_tie_fraction",
            "mean_optimistic_top1_ceiling",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ======================================================================
# 9. Representation diagnostics by candidate-surface availability
# ======================================================================

surface_summary_rows = []

for (
    dataset_name,
    split,
    surface_group
), sub in representation_df.groupby(
    [
        "dataset",
        "split",
        "surface_group"
    ]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "split": split,
        "surface_group": surface_group,
    })

    surface_summary_rows.append(stats)

surface_summary_df = pd.DataFrame(
    surface_summary_rows
)

print("\n" + "="*110)
print("REPRESENTATION DISTINGUISHABILITY BY CANDIDATE SURFACE")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()} VALIDATION")

    sub = surface_summary_df[
        (surface_summary_df["dataset"] == dataset_name)
        &
        (surface_summary_df["split"] == "validation")
    ]

    cols = [
        "surface_group",
        "n_groups",
        "all_identical_rate",
        "label_conflict_rate",
        "mean_unique_vector_ratio",
        "weighted_unavoidable_pair_tie_fraction",
        "mean_optimistic_top1_ceiling",
    ]

    print(
        sub[cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 10. Candidate-wise pairwise feature discrimination
# ======================================================================
#
# This replaces the misleading centered-AUC interpretation for
# group-constant features.
#
# For each feature and each feasible decision group:
#
#   compare every positive candidate with every negative candidate.
#
# We count:
#   positive feature > negative feature
#   positive feature < negative feature
#   exact tie
#
# Then report:
#
#   best_orientation_accuracy
#
# A feature that never varies within groups will have:
#   tie_rate = 1.0
#
# and cannot be mistaken for useful ranking signal.
# ======================================================================

def pairwise_feature_signal(
    X,
    y,
    group_ptr,
    feature_names,
    tol=1e-8
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.uint8)
    ptr = np.asarray(group_ptr, dtype=np.int64)

    rows = []

    for j, feature_name in enumerate(feature_names):
        higher = 0
        lower = 0
        ties = 0
        varying_groups = 0
        total_groups = len(ptr) - 1

        for g in range(total_groups):
            s = int(ptr[g])
            e = int(ptr[g + 1])

            xg = X[s:e, j]
            yg = y[s:e]

            if (
                float(xg.max() - xg.min())
                > tol
            ):
                varying_groups += 1

            pos = xg[yg == 1]
            neg = xg[yg == 0]

            if len(pos) == 0 or len(neg) == 0:
                continue

            diff = (
                pos[:, None]
                -
                neg[None, :]
            )

            higher += int(
                np.sum(diff > tol)
            )

            lower += int(
                np.sum(diff < -tol)
            )

            ties += int(
                np.sum(np.abs(diff) <= tol)
            )

        total_pairs = higher + lower + ties

        if total_pairs == 0:
            best_accuracy = np.nan
            tie_rate = np.nan
            direction = "none"
        else:
            # Ties count as 0.5 because the feature cannot order them.
            higher_acc = (
                higher + 0.5 * ties
            ) / total_pairs

            lower_acc = (
                lower + 0.5 * ties
            ) / total_pairs

            best_accuracy = max(
                higher_acc,
                lower_acc
            )

            tie_rate = (
                ties / total_pairs
            )

            if higher_acc > lower_acc:
                direction = "higher"
            elif lower_acc > higher_acc:
                direction = "lower"
            else:
                direction = "tie"

        rows.append({
            "feature_index": j,
            "feature": feature_name,
            "varying_groups": varying_groups,
            "n_groups": total_groups,
            "varying_group_rate":
                varying_groups / total_groups,
            "positive_gt_negative_pairs": higher,
            "positive_lt_negative_pairs": lower,
            "tie_pairs": ties,
            "total_pairs": total_pairs,
            "pairwise_tie_rate": tie_rate,
            "best_orientation_accuracy":
                best_accuracy,
            "preferred_direction": direction,
        })

    return pd.DataFrame(rows)

pairwise_signal_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    sig = pairwise_feature_signal(
        data["X"],
        data["y"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    sig.insert(
        0,
        "split",
        split
    )

    sig.insert(
        0,
        "dataset",
        dataset_name
    )

    pairwise_signal_frames.append(sig)

pairwise_signal_df = pd.concat(
    pairwise_signal_frames,
    ignore_index=True
)

print("\n" + "="*110)
print("CANDIDATE-WISE PAIRWISE FEATURE SIGNAL — VALIDATION")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = pairwise_signal_df[
        (pairwise_signal_df["dataset"] == dataset_name)
        &
        (pairwise_signal_df["split"] == "validation")
    ].sort_values(
        [
            "best_orientation_accuracy",
            "varying_group_rate"
        ],
        ascending=False
    )

    print(
        sub[
            [
                "feature_index",
                "feature",
                "varying_group_rate",
                "pairwise_tie_rate",
                "best_orientation_accuracy",
                "preferred_direction",
            ]
        ].head(15).to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 11. Candidate-size representation breakdown
# ======================================================================

def size_bin(x):
    if x <= 4:
        return "2-4"
    if x <= 8:
        return "5-8"
    if x <= 16:
        return "9-16"
    return "17+"

validation_repr = representation_df[
    representation_df["split"] == "validation"
].copy()

validation_repr["size_bin"] = (
    validation_repr[
        "candidate_size"
    ].map(size_bin)
)

size_summary_rows = []

for (
    dataset_name,
    bin_name
), sub in validation_repr.groupby(
    ["dataset", "size_bin"]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "size_bin": bin_name,
    })

    size_summary_rows.append(stats)

size_summary_df = pd.DataFrame(
    size_summary_rows
)

print("\n" + "="*110)
print("REPRESENTATION DISTINGUISHABILITY BY FRONTIER SIZE")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = size_summary_df[
        size_summary_df["dataset"]
        == dataset_name
    ]

    print(
        sub[
            [
                "size_bin",
                "n_groups",
                "all_identical_rate",
                "label_conflict_rate",
                "mean_unique_vector_ratio",
                "weighted_unavoidable_pair_tie_fraction",
                "mean_optimistic_top1_ceiling",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 12. Representation-limited frontier counts
# ======================================================================

print("\n" + "="*100)
print("VALIDATION REPRESENTATION-LIMITED FRONTIERS")
print("="*100)

revision_gate_summary = {}

for dataset_name in ["webqsp", "cwq"]:
    sub = validation_repr[
        validation_repr["dataset"] == dataset_name
    ]

    n = len(sub)

    all_identical = int(
        sub["all_identical"].sum()
    )

    conflicting = int(
        sub["has_label_conflict"].sum()
    )

    raw = sub[
        sub["surface_group"] == "all_raw_mid"
    ]

    raw_identical = int(
        raw["all_identical"].sum()
    )

    revision_gate_summary[dataset_name] = {
        "n_validation_groups": n,

        "all_identical_groups":
            all_identical,

        "all_identical_rate":
            all_identical / n,

        "label_conflict_groups":
            conflicting,

        "label_conflict_rate":
            conflicting / n,

        "all_raw_mid_groups":
            len(raw),

        "all_raw_mid_rate":
            len(raw) / n,

        "all_raw_mid_identical_groups":
            raw_identical,

        "all_raw_mid_identical_rate":
            (
                raw_identical / len(raw)
                if len(raw) > 0
                else 0.0
            ),

        "weighted_unavoidable_pair_tie_fraction":
            aggregate_representation(
                sub
            )[
                "weighted_unavoidable_pair_tie_fraction"
            ],
    }

    print(f"\n{dataset_name.upper()}")
    print(
        "Validation decision groups:       ",
        n
    )
    print(
        "All-identical Feature-v2 groups:  ",
        f"{all_identical}/{n} "
        f"({100*all_identical/n:.2f}%)"
    )
    print(
        "Groups with +/- feature conflict: ",
        f"{conflicting}/{n} "
        f"({100*conflicting/n:.2f}%)"
    )
    print(
        "All-raw-MID groups:               ",
        f"{len(raw)}/{n} "
        f"({100*len(raw)/n:.2f}%)"
    )

    if len(raw) > 0:
        print(
            "Raw-MID groups all-identical:    ",
            f"{raw_identical}/{len(raw)} "
            f"({100*raw_identical/len(raw):.2f}%)"
        )

# ======================================================================
# 13. Scientific interpretation gate
# ======================================================================
#
# This is deliberately descriptive.
# We DO NOT automatically redesign AFP from arbitrary thresholds.
#
# Instead we identify which of three situations the evidence supports:
#
#   REPRESENTATION-LIMITED
#   MIXED
#   OBJECTIVE/MODEL-LIMITED
#
# The printed label is a diagnostic heuristic, not a statistical test.
# ======================================================================

def diagnostic_gate_label(stats):
    identical = stats["all_identical_rate"]
    raw = stats["all_raw_mid_rate"]
    ties = stats[
        "weighted_unavoidable_pair_tie_fraction"
    ]

    if identical >= 0.50 and raw >= 0.50:
        return "REPRESENTATION-LIMITED"

    if identical <= 0.20 and ties <= 0.20:
        return "OBJECTIVE/MODEL-LIMITED OR MIXED"

    return "MIXED / PARTLY REPRESENTATION-LIMITED"

gate_labels = {}

print("\n" + "="*100)
print("FEATURE REVISION GATE")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    label = diagnostic_gate_label(
        revision_gate_summary[
            dataset_name
        ]
    )

    gate_labels[dataset_name] = label

    print(
        f"{dataset_name.upper()}: {label}"
    )

print(
    "\nNOTE: These labels are diagnostic heuristics, "
    "not hypothesis-test results."
)

# ======================================================================
# 14. Save diagnostics
# ======================================================================

representation_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "group_representation_diagnostics.csv"
    ),
    index=False
)

representation_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_summary.csv"
    ),
    index=False
)

surface_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_by_candidate_surface.csv"
    ),
    index=False
)

pairwise_signal_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "pairwise_feature_signal.csv"
    ),
    index=False
)

size_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_by_frontier_size.csv"
    ),
    index=False
)

# ======================================================================
# 15. Hash helper
# ======================================================================

def sha256_file_gate(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

summary_path = os.path.join(
    FEATURE_REVISION_DIR,
    "representation_summary.csv"
)

signal_path = os.path.join(
    FEATURE_REVISION_DIR,
    "pairwise_feature_signal.csv"
)

# ======================================================================
# 16. Manifest
# ======================================================================

gate_manifest = {
    "gate_version":
        "afp_feature_revision_gate_v1",

    "feature_version":
        AFP_FEATURE_VERSION,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_version":
        AFP_SCORER_VERSION,

    "training_version":
        AFP_TRAINING_VERSION,

    "signature_decimals":
        FEATURE_SIGNATURE_DECIMALS,

    "datasets":
        ["webqsp", "cwq"],

    "splits_used":
        ["train", "validation"],

    "test_used":
        False,

    "new_training_performed":
        False,

    "feature_spec_modified":
        False,

    "winner_selected":
        False,

    "final_scorer_frozen":
        False,

    "gate_labels":
        gate_labels,

    "validation_summary":
        revision_gate_summary,

    "representation_summary_sha256":
        sha256_file_gate(summary_path),

    "pairwise_signal_sha256":
        sha256_file_gate(signal_path),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

manifest_path = os.path.join(
    FEATURE_REVISION_DIR,
    "feature_revision_gate_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        gate_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 17. Final gate report
# ======================================================================

print("\n" + "="*92)
print("=== RQ2 CELL 9B: CANDIDATE DISTINGUISHABILITY GATE COMPLETE ===")
print("="*92)

print("Feature version examined:", AFP_FEATURE_VERSION)
print("Exact/effective feature collisions examined: YES")
print("Positive-negative representation conflicts examined: YES")
print("Unavoidable pairwise ties examined: YES")
print("Raw MID vs readable groups examined: YES")
print("Pairwise candidate-specific feature signal examined: YES")
print("New features added: NO")
print("New scorer training: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Final scorer frozen: NO")

print("\nDiagnostic gate:")
print("  WebQSP:", gate_labels["webqsp"])
print("  CWQ:   ", gate_labels["cwq"])

print("\nSTOP HERE.")
print(
    "Send the full Cell 9B output before changing Feature-v2, "
    "retraining, or freezing a scorer."
)

Feature revision gate directory: /kaggle/working/step3_rq2_dev_v1/06_feature_revision_gate
Analyzing WEBQSP train...
Analyzing WEBQSP validation...
Analyzing CWQ train...
Analyzing CWQ validation...

FEATURE-v2 REPRESENTATION DISTINGUISHABILITY
dataset      split  n_groups  all_identical_rate  label_conflict_rate  mean_unique_vector_ratio  weighted_unavoidable_pair_tie_fraction  mean_optimistic_top1_ceiling
    cwq      train     15937              0.5022               0.4873                    0.5897                                  0.3320                        0.6387
    cwq validation      1352              0.4771               0.4453                    0.6254                                  0.2233                        0.6798
 webqsp      train      1457              0.6205               0.4935                    0.5132                                  0.6511                        0.6870
 webqsp validation        87              0.7701               0.6437                    0.

In [1]:
# ======================================================================
# RQ2 ARTIFACT RELOAD AFTER KAGGLE SESSION / ACCELERATOR RESTART
# ======================================================================
#
# Restores:
#   - Feature-v2 TRAIN + VALIDATION NPZ files
#   - Feature manifests
#   - Train-only standardizers
#   - All scorer-v1 checkpoints/results
#   - Cell 9A scorer diagnostics
#   - Cell 9B representation diagnostics
#   - Core AFP constants/classes
#
# Does NOT:
#   - regenerate plans
#   - rebuild features
#   - retrain scorers
#   - use test data
# ======================================================================

from pathlib import Path
import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_NAME = "step3_rq2_dev_v1"
WORK_ROOT = Path("/kaggle/working") / PROJECT_NAME

# ======================================================================
# 1. Find saved artifact root
# ======================================================================

def find_saved_project():
    # Best case: /kaggle/working survived restart.
    if WORK_ROOT.exists():
        return WORK_ROOT

    # Otherwise look inside mounted Kaggle inputs.
    input_root = Path("/kaggle/input")

    if input_root.exists():
        matches = [
            p for p in input_root.rglob(PROJECT_NAME)
            if p.is_dir()
        ]

        if matches:
            print("Found saved project under Kaggle input:")
            for p in matches:
                print(" ", p)
            return matches[0]

    raise FileNotFoundError(
        "\nCould not find the saved RQ2 artifacts.\n\n"
        "If you changed accelerator and /kaggle/working was cleared:\n"
        "1. Add your latest saved Kaggle notebook VERSION/OUTPUT as an Input.\n"
        "2. Then rerun this cell.\n\n"
        f"Expected folder: {PROJECT_NAME}"
    )

SOURCE_ROOT = find_saved_project()

print("Artifact source:", SOURCE_ROOT)

# ======================================================================
# 2. If source is read-only /kaggle/input, copy back to /kaggle/working
# ======================================================================

if SOURCE_ROOT.resolve() != WORK_ROOT.resolve():
    print("\nRestoring saved artifacts to /kaggle/working...")

    WORK_ROOT.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        SOURCE_ROOT,
        WORK_ROOT,
        dirs_exist_ok=True
    )

    RQ2_ROOT = WORK_ROOT
else:
    RQ2_ROOT = SOURCE_ROOT

print("Active project root:", RQ2_ROOT)

# ======================================================================
# 3. Restore directory variables
# ======================================================================

RQ2_FEATURE_DIR = str(RQ2_ROOT / "03_features")
RQ2_SCORER_DIR = str(RQ2_ROOT / "04_scorer")
DIAG_DIR = str(RQ2_ROOT / "05_scorer_diagnostics")
FEATURE_REVISION_DIR = str(RQ2_ROOT / "06_feature_revision_gate")

assert Path(RQ2_FEATURE_DIR).exists()
assert Path(RQ2_SCORER_DIR).exists()
assert Path(DIAG_DIR).exists()
assert Path(FEATURE_REVISION_DIR).exists()

# ======================================================================
# 4. Helpers
# ======================================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

def load_feature_npz(path):
    z = np.load(path, allow_pickle=False)

    required = [
        "X",
        "y",
        "group_ptr",
        "group_source_index",
        "group_hop",
        "group_plan_length",
        "group_candidate_count",
    ]

    missing = [k for k in required if k not in z.files]
    assert not missing, f"Missing NPZ arrays: {missing}"

    return {
        k: z[k]
        for k in required
    }

# ======================================================================
# 5. Locate feature files
# ======================================================================

FEATURE_FILES = {
    "webqsp_train":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_train_afp_features_v2.npz",

    "webqsp_validation":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_validation_afp_features_v2.npz",

    "cwq_train":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_train_afp_features_v2.npz",

    "cwq_validation":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_validation_afp_features_v2.npz",
}

FEATURE_MANIFEST_FILES = {
    "webqsp_train":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_train_afp_features_v2_manifest.json",

    "webqsp_validation":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_validation_afp_features_v2_manifest.json",

    "cwq_train":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_train_afp_features_v2_manifest.json",

    "cwq_validation":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_validation_afp_features_v2_manifest.json",
}

for path in list(FEATURE_FILES.values()) + list(FEATURE_MANIFEST_FILES.values()):
    assert path.exists(), f"Missing artifact: {path}"

# ======================================================================
# 6. Reload feature manifests
# ======================================================================

webqsp_train_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["webqsp_train"]
)

webqsp_val_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["webqsp_validation"]
)

cwq_train_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["cwq_train"]
)

cwq_val_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["cwq_validation"]
)

# ======================================================================
# 7. Restore frozen feature constants
# ======================================================================

AFP_FEATURE_VERSION = webqsp_train_feature_manifest["feature_version"]
AFP_FEATURE_SPEC_SHA256 = webqsp_train_feature_manifest["feature_spec_sha256"]
AFP_FEATURE_DIM = int(webqsp_train_feature_manifest["feature_dim"])
AFP_SEMANTIC_ENCODER_NAME = webqsp_train_feature_manifest["semantic_encoder"]

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27

EXPECTED_FEATURE_SHA = (
    "738985d1232a8ac5935c397ed95eca233"
    "77bc59b4a99494547fa7779062ade86"
)

assert AFP_FEATURE_SPEC_SHA256 == EXPECTED_FEATURE_SHA

# ======================================================================
# 8. Verify feature-file hashes BEFORE loading
# ======================================================================

for key in FEATURE_FILES:
    expected = {
        "webqsp_train": webqsp_train_feature_manifest,
        "webqsp_validation": webqsp_val_feature_manifest,
        "cwq_train": cwq_train_feature_manifest,
        "cwq_validation": cwq_val_feature_manifest,
    }[key]["npz_sha256"]

    actual = sha256_file(FEATURE_FILES[key])

    assert actual == expected, (
        f"{key}: NPZ SHA256 mismatch!\n"
        f"expected={expected}\n"
        f"actual={actual}"
    )

print("\nFeature artifact SHA256 gates: PASSED")

# ======================================================================
# 9. Reload feature datasets
# ======================================================================

webqsp_train_features = load_feature_npz(
    FEATURE_FILES["webqsp_train"]
)

webqsp_val_features = load_feature_npz(
    FEATURE_FILES["webqsp_validation"]
)

cwq_train_features = load_feature_npz(
    FEATURE_FILES["cwq_train"]
)

cwq_val_features = load_feature_npz(
    FEATURE_FILES["cwq_validation"]
)

# ======================================================================
# 10. Restore lightweight scorer definition
# ======================================================================

AFP_SCORER_VERSION = "afp_mlp_v1"
AFP_INPUT_DIM = AFP_FEATURE_DIM
AFP_HIDDEN_CANDIDATES = [32, 64, 128]
AFP_DROPOUT = 0.0

class AFPScorer(nn.Module):
    def __init__(
        self,
        input_dim=27,
        hidden_dim=64,
        dropout=0.0
    ):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            1
        )

        self.dropout = (
            nn.Dropout(dropout)
            if dropout > 0
            else nn.Identity()
        )

    def forward(self, x):
        h = F.relu(
            self.fc1(x)
        )

        h = self.dropout(h)

        return self.fc2(
            h
        ).squeeze(-1)

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.sigmoid(
            self.forward(x)
        )

# ======================================================================
# 11. Restore standardizer definition
# ======================================================================

class AFPFeatureStandardizer:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None
        self.fitted = False

    def transform(self, X):
        assert self.fitted

        X = np.asarray(
            X,
            dtype=np.float32
        )

        Z = (
            X - self.mean_
        ) / self.std_

        assert np.all(np.isfinite(Z))

        return Z.astype(np.float32)

    def load_state_dict(self, state):
        self.mean_ = np.asarray(
            state["mean"],
            dtype=np.float32
        )

        self.std_ = np.asarray(
            state["std"],
            dtype=np.float32
        )

        self.eps = float(
            state["eps"]
        )

        self.fitted = True
        return self

# ======================================================================
# 12. Reload train-only standardizers
# ======================================================================

def load_standardizer(path):
    payload = load_json(path)

    assert payload["fit_split"] == "train"
    assert payload["feature_spec_sha256"] == AFP_FEATURE_SPEC_SHA256

    obj = AFPFeatureStandardizer()
    obj.load_state_dict(
        payload["state"]
    )

    return obj, payload

webqsp_standardizer, webqsp_standardizer_manifest = load_standardizer(
    Path(RQ2_SCORER_DIR) /
    "webqsp_train_standardizer.json"
)

cwq_standardizer, cwq_standardizer_manifest = load_standardizer(
    Path(RQ2_SCORER_DIR) /
    "cwq_train_standardizer.json"
)

# ======================================================================
# 13. Reconstruct standardized scorer datasets
# ======================================================================

webqsp_scorer_prepared = {
    "X_train":
        webqsp_standardizer.transform(
            webqsp_train_features["X"]
        ),

    "y_train":
        webqsp_train_features["y"].astype(np.float32),

    "train_group_ptr":
        webqsp_train_features["group_ptr"],

    "X_val":
        webqsp_standardizer.transform(
            webqsp_val_features["X"]
        ),

    "y_val":
        webqsp_val_features["y"].astype(np.float32),

    "val_group_ptr":
        webqsp_val_features["group_ptr"],

    "standardizer":
        webqsp_standardizer,
}

cwq_scorer_prepared = {
    "X_train":
        cwq_standardizer.transform(
            cwq_train_features["X"]
        ),

    "y_train":
        cwq_train_features["y"].astype(np.float32),

    "train_group_ptr":
        cwq_train_features["group_ptr"],

    "X_val":
        cwq_standardizer.transform(
            cwq_val_features["X"]
        ),

    "y_val":
        cwq_val_features["y"].astype(np.float32),

    "val_group_ptr":
        cwq_val_features["group_ptr"],

    "standardizer":
        cwq_standardizer,
}

# ======================================================================
# 14. Reload scorer training results / manifest
# ======================================================================

training_manifest_path = (
    Path(RQ2_SCORER_DIR) /
    "afp_scorer_training_manifest.json"
)

training_results_path = (
    Path(RQ2_SCORER_DIR) /
    "afp_scorer_training_runs.csv"
)

assert training_manifest_path.exists()
assert training_results_path.exists()

afp_training_manifest = load_json(
    training_manifest_path
)

afp_scorer_results_df = pd.read_csv(
    training_results_path
)

AFP_TRAINING_VERSION = (
    afp_training_manifest[
        "training_version"
    ]
)

AFP_TRAINING_SPEC_SHA256 = (
    afp_training_manifest[
        "training_spec_sha256"
    ]
)

AFP_SCORER_SPEC_SHA256 = (
    afp_training_manifest[
        "scorer_spec_sha256"
    ]
)

assert len(
    afp_scorer_results_df
) == 36

# ======================================================================
# 15. Repair checkpoint paths after restore
# ======================================================================
#
# CSV contains old /kaggle/working paths.
# Rebuild paths by checkpoint filename so this also works when restored
# from a saved Kaggle version.
# ======================================================================

def repair_checkpoint_path(old_path):
    filename = Path(
        str(old_path)
    ).name

    new_path = (
        Path(RQ2_SCORER_DIR) /
        filename
    )

    assert new_path.exists(), (
        f"Missing checkpoint: {new_path}"
    )

    return str(new_path)

afp_scorer_results_df[
    "checkpoint"
] = afp_scorer_results_df[
    "checkpoint"
].apply(
    repair_checkpoint_path
)

checkpoint_files = list(
    Path(RQ2_SCORER_DIR).glob(
        "*.pt"
    )
)

print(
    "Scorer checkpoints found:",
    len(checkpoint_files)
)

assert len(checkpoint_files) >= 36

# ======================================================================
# 16. Safe checkpoint loader
# ======================================================================

AFP_DEVICE = "cpu"

def load_afp_checkpoint(
    path,
    device="cpu"
):
    try:
        ckpt = torch.load(
            path,
            map_location=device,
            weights_only=False
        )
    except TypeError:
        ckpt = torch.load(
            path,
            map_location=device
        )

    assert (
        ckpt["feature_spec_sha256"]
        == AFP_FEATURE_SPEC_SHA256
    )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=int(
            ckpt["hidden_dim"]
        ),
        dropout=AFP_DROPOUT
    ).to(device)

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    model.eval()

    return model, ckpt

# ======================================================================
# 17. Reload Cell 9A diagnostics
# ======================================================================

diagnostic_files = {
    "null_comparison":
        Path(DIAG_DIR) /
        "scorer_vs_random_null.csv",

    "generalization_runs":
        Path(DIAG_DIR) /
        "train_validation_generalization_runs.csv",

    "generalization_aggregate":
        Path(DIAG_DIR) /
        "train_validation_generalization_aggregate.csv",

    "feature_variation":
        Path(DIAG_DIR) /
        "within_group_feature_variation.csv",

    "feature_signal":
        Path(DIAG_DIR) /
        "candidate_relative_feature_signal.csv",

    "subgroups":
        Path(DIAG_DIR) /
        "validation_subgroup_diagnostics.csv",
}

for path in diagnostic_files.values():
    assert path.exists(), f"Missing Cell 9A artifact: {path}"

null_comparison_df = pd.read_csv(
    diagnostic_files["null_comparison"]
)

generalization_df = pd.read_csv(
    diagnostic_files["generalization_runs"]
)

generalization_agg = pd.read_csv(
    diagnostic_files["generalization_aggregate"]
)

feature_variation_df = pd.read_csv(
    diagnostic_files["feature_variation"]
)

feature_signal_df = pd.read_csv(
    diagnostic_files["feature_signal"]
)

subgroup_df = pd.read_csv(
    diagnostic_files["subgroups"]
)

# ======================================================================
# 18. Reload Cell 9B diagnostics
# ======================================================================

revision_files = {
    "representation":
        Path(FEATURE_REVISION_DIR) /
        "group_representation_diagnostics.csv",

    "summary":
        Path(FEATURE_REVISION_DIR) /
        "representation_summary.csv",

    "surface":
        Path(FEATURE_REVISION_DIR) /
        "representation_by_candidate_surface.csv",

    "pairwise":
        Path(FEATURE_REVISION_DIR) /
        "pairwise_feature_signal.csv",

    "size":
        Path(FEATURE_REVISION_DIR) /
        "representation_by_frontier_size.csv",

    "manifest":
        Path(FEATURE_REVISION_DIR) /
        "feature_revision_gate_manifest.json",
}

for path in revision_files.values():
    assert path.exists(), f"Missing Cell 9B artifact: {path}"

representation_df = pd.read_csv(
    revision_files["representation"]
)

representation_summary_df = pd.read_csv(
    revision_files["summary"]
)

surface_summary_df = pd.read_csv(
    revision_files["surface"]
)

pairwise_signal_df = pd.read_csv(
    revision_files["pairwise"]
)

size_summary_df = pd.read_csv(
    revision_files["size"]
)

feature_revision_gate_manifest = load_json(
    revision_files["manifest"]
)

# ======================================================================
# 19. Structural sanity gates
# ======================================================================

assert webqsp_train_features["X"].shape == (18437, 27)
assert webqsp_val_features["X"].shape == (966, 27)

assert cwq_train_features["X"].shape == (218544, 27)
assert cwq_val_features["X"].shape == (18688, 27)

assert len(webqsp_train_features["group_ptr"]) - 1 == 1457
assert len(webqsp_val_features["group_ptr"]) - 1 == 87

assert len(cwq_train_features["group_ptr"]) - 1 == 15937
assert len(cwq_val_features["group_ptr"]) - 1 == 1352

assert feature_revision_gate_manifest["test_used"] is False
assert feature_revision_gate_manifest["winner_selected"] is False

# ======================================================================
# 20. Final report
# ======================================================================

print("\n" + "=" * 86)
print("=== RQ2 DEVELOPMENT ARTIFACTS SUCCESSFULLY RELOADED ===")
print("=" * 86)

print("Project root:", RQ2_ROOT)
print("Device:", AFP_DEVICE)

print("\nFeature-v2")
print("  dimension:      ", AFP_FEATURE_DIM)
print("  SHA256:         ", AFP_FEATURE_SPEC_SHA256[:16] + "...")

print("\nWEBQSP")
print("  train branches: ", len(webqsp_train_features["y"]))
print("  train groups:   ", len(webqsp_train_features["group_ptr"]) - 1)
print("  val branches:   ", len(webqsp_val_features["y"]))
print("  val groups:     ", len(webqsp_val_features["group_ptr"]) - 1)

print("\nCWQ")
print("  train branches: ", len(cwq_train_features["y"]))
print("  train groups:   ", len(cwq_train_features["group_ptr"]) - 1)
print("  val branches:   ", len(cwq_val_features["y"]))
print("  val groups:     ", len(cwq_val_features["group_ptr"]) - 1)

print("\nScorer")
print("  training runs:  ", len(afp_scorer_results_df))
print("  checkpoints:    ", len(checkpoint_files))

print("\nDiagnostics")
print("  Cell 9A: LOADED")
print("  Cell 9B: LOADED")
print(
    "  WebQSP gate:",
    feature_revision_gate_manifest["gate_labels"]["webqsp"]
)
print(
    "  CWQ gate:   ",
    feature_revision_gate_manifest["gate_labels"]["cwq"]
)

print("\nTEST DATA/GOLD LOADED: NO")
print("FINAL SCORER FROZEN: NO")
print("\nReady for the adaptive-selector stage.")

Artifact source: /kaggle/working/step3_rq2_dev_v1
Active project root: /kaggle/working/step3_rq2_dev_v1

Feature artifact SHA256 gates: PASSED
Scorer checkpoints found: 36

=== RQ2 DEVELOPMENT ARTIFACTS SUCCESSFULLY RELOADED ===
Project root: /kaggle/working/step3_rq2_dev_v1
Device: cpu

Feature-v2
  dimension:       27
  SHA256:          738985d1232a8ac5...

WEBQSP
  train branches:  18437
  train groups:    1457
  val branches:    966
  val groups:      87

CWQ
  train branches:  218544
  train groups:    15937
  val branches:    18688
  val groups:      1352

Scorer
  training runs:   36
  checkpoints:     36

Diagnostics
  Cell 9A: LOADED
  Cell 9B: LOADED
  WebQSP gate: REPRESENTATION-LIMITED
  CWQ gate:    MIXED / PARTLY REPRESENTATION-LIMITED

TEST DATA/GOLD LOADED: NO
FINAL SCORER FROZEN: NO

Ready for the adaptive-selector stage.


## Tie-Aware Scorer Validation and checkpoint selection/freeze

In [3]:

# WHY TIE-AWARE:
#   Cell 9B established extensive representation collisions.
#   Ordinary stable-sort AP/MRR/Top1 can depend on arbitrary candidate
#   ordering when logits are tied.
#
# Therefore config selection now uses EXPECTED ranking performance under
# random ordering inside tied-score blocks.
#
# Selection:
#   1. Evaluate all 36 SAVED checkpoints on VALIDATION only.
#   2. Aggregate each (hidden_dim, loss) across seeds 42/43/44.
#   3. Select by:
#        a) highest mean tie-aware Group AP
#        b) highest mean tie-aware MRR
#        c) highest mean tie-aware Top-1 Hit
#        d) lowest mean Group-Balanced BCE
#   4. Deployment seed remains FIXED at 42.
#
# IMPORTANT:
#   - NO new training.
#   - NO test data/gold.
#   - NO seed cherry-picking.
#   - Feature-v2 remains frozen.
# ======================================================================

import os
import json
import math
import shutil
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"
assert len(afp_scorer_results_df) == 36

assert feature_revision_gate_manifest["test_used"] is False
assert feature_revision_gate_manifest["winner_selected"] is False

DEPLOYMENT_SEED = 42

# Treat only numerically indistinguishable logits as tied.
TIE_ATOL = 1e-8

print("Deployment seed:", DEPLOYMENT_SEED)
print("Tie tolerance:  ", TIE_ATOL)

# ======================================================================
# 2. Checkpoint-specific standardization
# ======================================================================

def transform_with_checkpoint_standardizer(X, ckpt):
    state = ckpt["standardizer_state"]

    mean = np.asarray(
        state["mean"],
        dtype=np.float32
    )

    std = np.asarray(
        state["std"],
        dtype=np.float32
    )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    Z = (X - mean) / std

    assert Z.shape[1] == AFP_FEATURE_DIM
    assert np.all(np.isfinite(Z))

    return Z.astype(np.float32)

# ======================================================================
# 3. Build score-tie blocks
# ======================================================================

def make_tie_blocks(logits, atol=TIE_ATOL):
    """
    Sort candidates by descending logit and group numerically tied scores.

    Returns list of arrays containing original candidate indices.
    """
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    blocks = []
    current = [int(order[0])]
    reference = float(logits[order[0]])

    for idx in order[1:]:
        value = float(logits[idx])

        if abs(value - reference) <= atol:
            current.append(int(idx))
        else:
            blocks.append(
                np.asarray(current, dtype=np.int64)
            )

            current = [int(idx)]
            reference = value

    blocks.append(
        np.asarray(current, dtype=np.int64)
    )

    return blocks

# ======================================================================
# 4. Expected Top-1 under random tie-breaking
# ======================================================================

def expected_top1_from_blocks(blocks, labels):
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    top_block = blocks[0]

    return float(
        labels[top_block].mean()
    )

# ======================================================================
# 5. Expected reciprocal rank of first positive
# ======================================================================

def expected_mrr_from_blocks(blocks, labels):
    """
    Exact expectation under uniform random ordering within each tie block.
    """
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    offset = 0

    for block in blocks:
        block_y = labels[block]

        n = len(block_y)
        p = int(block_y.sum())

        if p == 0:
            offset += n
            continue

        # First block containing at least one positive.
        denominator = math.comb(n, p)

        expected_rr = 0.0

        # If p positives are randomly placed among n positions,
        # probability first positive occurs at local rank k:
        #
        # C(n-k, p-1) / C(n, p)
        #
        max_first_rank = n - p + 1

        for k in range(1, max_first_rank + 1):
            probability = (
                math.comb(n - k, p - 1)
                / denominator
            )

            expected_rr += (
                probability
                / (offset + k)
            )

        return float(expected_rr)

    raise AssertionError(
        "Feasible decision group has no positive candidate."
    )

# ======================================================================
# 6. Expected AP under random tie-breaking
# ======================================================================

def expected_ap_from_blocks(blocks, labels):
    """
    Exact expected Average Precision under uniform random ordering
    inside tied-score blocks.

    No Monte-Carlo approximation is used.
    """
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    total_positive = int(
        labels.sum()
    )

    assert total_positive > 0

    offset = 0
    positives_before = 0
    expected_precision_sum = 0.0

    for block in blocks:
        block_y = labels[block]

        n = len(block_y)
        p = int(block_y.sum())

        if p == 0:
            offset += n
            continue

        for local_rank in range(1, n + 1):
            # P(position local_rank is positive)
            py = p / n

            # E[Y_r * K_r]
            #
            # K_r = number of positives in this block up to position r.
            if n == 1:
                ey_times_k = 1.0
            else:
                ey_times_k = (
                    py
                    +
                    (local_rank - 1)
                    * p
                    * (p - 1)
                    / (n * (n - 1))
                )

            expected_numerator = (
                positives_before * py
                + ey_times_k
            )

            global_rank = (
                offset + local_rank
            )

            expected_precision_sum += (
                expected_numerator
                / global_rank
            )

        offset += n
        positives_before += p

    return float(
        expected_precision_sum
        / total_positive
    )

# ======================================================================
# 7. Complete tie-aware group ranking evaluator
# ======================================================================

def tie_aware_group_ranking_metrics(
    logits,
    labels,
    group_ptr,
    atol=TIE_ATOL
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    assert len(logits) == len(labels)
    assert ptr[0] == 0
    assert ptr[-1] == len(labels)

    aps = []
    mrrs = []
    top1s = []

    groups_with_ties = 0
    fully_tied_groups = 0

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        group_logits = logits[s:e]
        group_labels = labels[s:e]

        assert len(group_labels) > 1
        assert group_labels.sum() >= 1

        blocks = make_tie_blocks(
            group_logits,
            atol=atol
        )

        if any(len(b) > 1 for b in blocks):
            groups_with_ties += 1

        if len(blocks) == 1:
            fully_tied_groups += 1

        aps.append(
            expected_ap_from_blocks(
                blocks,
                group_labels
            )
        )

        mrrs.append(
            expected_mrr_from_blocks(
                blocks,
                group_labels
            )
        )

        top1s.append(
            expected_top1_from_blocks(
                blocks,
                group_labels
            )
        )

    n_groups = len(ptr) - 1

    return {
        "tie_aware_group_ap":
            float(np.mean(aps)),

        "tie_aware_mrr":
            float(np.mean(mrrs)),

        "tie_aware_top1":
            float(np.mean(top1s)),

        "groups_with_score_ties":
            int(groups_with_ties),

        "score_tie_group_rate":
            float(groups_with_ties / n_groups),

        "fully_tied_score_groups":
            int(fully_tied_groups),

        "fully_tied_score_group_rate":
            float(fully_tied_groups / n_groups),
    }

# ======================================================================
# 8. BCE metrics
# ======================================================================

def bce_metrics(logits, y, group_ptr):
    logits_t = torch.as_tensor(
        logits,
        dtype=torch.float32
    )

    y_t = torch.as_tensor(
        y,
        dtype=torch.float32
    )

    branch_bce = float(
        F.binary_cross_entropy_with_logits(
            logits_t,
            y_t,
            reduction="mean"
        )
    )

    ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    group_losses = []

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        group_losses.append(
            F.binary_cross_entropy_with_logits(
                logits_t[s:e],
                y_t[s:e],
                reduction="mean"
            )
        )

    group_bce = float(
        torch.stack(
            group_losses
        ).mean()
    )

    return branch_bce, group_bce

# ======================================================================
# 9. Evaluate one saved checkpoint
# ======================================================================

@torch.no_grad()
def evaluate_saved_run(row):
    dataset_name = row["dataset"]

    if dataset_name == "webqsp":
        raw_val = webqsp_val_features
    elif dataset_name == "cwq":
        raw_val = cwq_val_features
    else:
        raise ValueError(dataset_name)

    model, ckpt = load_afp_checkpoint(
        row["checkpoint"],
        device="cpu"
    )

    assert ckpt["dataset"] == dataset_name
    assert int(ckpt["seed"]) == int(row["seed"])
    assert (
        ckpt["feature_spec_sha256"]
        == AFP_FEATURE_SPEC_SHA256
    )

    X_val = transform_with_checkpoint_standardizer(
        raw_val["X"],
        ckpt
    )

    X_t = torch.as_tensor(
        X_val,
        dtype=torch.float32
    )

    logits = model(
        X_t
    ).cpu().numpy()

    y = raw_val[
        "y"
    ].astype(np.uint8)

    ptr = raw_val[
        "group_ptr"
    ]

    branch_bce, group_bce = bce_metrics(
        logits,
        y,
        ptr
    )

    ranking = tie_aware_group_ranking_metrics(
        logits,
        y,
        ptr,
        atol=TIE_ATOL
    )

    return {
        "dataset":
            dataset_name,

        "hidden_dim":
            int(row["hidden_dim"]),

        "loss_name":
            row["loss_name"],

        "seed":
            int(row["seed"]),

        "branch_bce":
            branch_bce,

        "group_bce":
            group_bce,

        **ranking,

        "checkpoint":
            row["checkpoint"],
    }

# ======================================================================
# 10. Re-evaluate all 36 checkpoints
# ======================================================================

print("\nRe-evaluating all 36 checkpoints with tie-aware metrics...")

tie_aware_rows = []

for i, row in afp_scorer_results_df.iterrows():
    result = evaluate_saved_run(row)

    tie_aware_rows.append(
        result
    )

    print(
        f"[{i+1:02d}/36] "
        f"{result['dataset']} "
        f"H={result['hidden_dim']} "
        f"{result['loss_name']} "
        f"seed={result['seed']} | "
        f"AP={result['tie_aware_group_ap']:.4f} "
        f"MRR={result['tie_aware_mrr']:.4f} "
        f"Top1={result['tie_aware_top1']:.4f}"
    )

tie_aware_results_df = pd.DataFrame(
    tie_aware_rows
)

# ======================================================================
# 11. Aggregate configurations over fixed seeds
# ======================================================================

tie_aware_config_summary = (
    tie_aware_results_df
    .groupby(
        [
            "dataset",
            "hidden_dim",
            "loss_name",
        ]
    )
    .agg(
        n_seeds=(
            "seed",
            "count"
        ),

        group_ap_mean=(
            "tie_aware_group_ap",
            "mean"
        ),

        group_ap_std=(
            "tie_aware_group_ap",
            "std"
        ),

        mrr_mean=(
            "tie_aware_mrr",
            "mean"
        ),

        mrr_std=(
            "tie_aware_mrr",
            "std"
        ),

        top1_mean=(
            "tie_aware_top1",
            "mean"
        ),

        top1_std=(
            "tie_aware_top1",
            "std"
        ),

        group_bce_mean=(
            "group_bce",
            "mean"
        ),

        branch_bce_mean=(
            "branch_bce",
            "mean"
        ),

        score_tie_rate_mean=(
            "score_tie_group_rate",
            "mean"
        ),
    )
    .reset_index()
)

assert np.all(
    tie_aware_config_summary[
        "n_seeds"
    ] == 3
)

# ======================================================================
# 12. Deterministic configuration selection
# ======================================================================

def rank_configs(dataset_name):
    sub = tie_aware_config_summary[
        tie_aware_config_summary[
            "dataset"
        ] == dataset_name
    ].copy()

    assert len(sub) == 6

    sub = sub.sort_values(
        by=[
            "group_ap_mean",
            "mrr_mean",
            "top1_mean",
            "group_bce_mean",
            "hidden_dim",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort"
    ).reset_index(
        drop=True
    )

    return sub

webqsp_ranked_configs = rank_configs(
    "webqsp"
)

cwq_ranked_configs = rank_configs(
    "cwq"
)

# ======================================================================
# 13. Print tie-aware validation rankings
# ======================================================================

def print_ranked(dataset_name, ranked):
    print("\n" + "=" * 110)
    print(
        f"{dataset_name.upper()} "
        "TIE-AWARE VALIDATION CONFIGURATION RANKING"
    )
    print("=" * 110)

    cols = [
        "hidden_dim",
        "loss_name",
        "group_ap_mean",
        "group_ap_std",
        "mrr_mean",
        "top1_mean",
        "group_bce_mean",
        "score_tie_rate_mean",
    ]

    print(
        ranked[cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

print_ranked(
    "webqsp",
    webqsp_ranked_configs
)

print_ranked(
    "cwq",
    cwq_ranked_configs
)

# ======================================================================
# 14. Select configuration, NOT seed
# ======================================================================

webqsp_selected_config = (
    webqsp_ranked_configs.iloc[0]
)

cwq_selected_config = (
    cwq_ranked_configs.iloc[0]
)

def fixed_seed_run(
    dataset_name,
    selected_config
):
    sub = tie_aware_results_df[
        (tie_aware_results_df["dataset"] == dataset_name)
        &
        (
            tie_aware_results_df["hidden_dim"]
            ==
            int(
                selected_config["hidden_dim"]
            )
        )
        &
        (
            tie_aware_results_df["loss_name"]
            ==
            selected_config["loss_name"]
        )
        &
        (
            tie_aware_results_df["seed"]
            ==
            DEPLOYMENT_SEED
        )
    ]

    assert len(sub) == 1

    return sub.iloc[0]

webqsp_deployment_run = fixed_seed_run(
    "webqsp",
    webqsp_selected_config
)

cwq_deployment_run = fixed_seed_run(
    "cwq",
    cwq_selected_config
)

# ======================================================================
# 15. Save selected development checkpoints
# ======================================================================

FINAL_SCORER_DIR = (
    Path(RQ2_ROOT)
    / "07_final_scorer"
)

FINAL_SCORER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

def sha256_file_final(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

def copy_selected_checkpoint(
    dataset_name,
    deployment_run
):
    source = Path(
        deployment_run[
            "checkpoint"
        ]
    )

    assert source.exists()

    destination = (
        FINAL_SCORER_DIR
        / f"{dataset_name}_afp_scorer_selected.pt"
    )

    shutil.copy2(
        source,
        destination
    )

    return (
        str(destination),
        sha256_file_final(destination)
    )

(
    webqsp_best_checkpoint,
    webqsp_best_checkpoint_sha256
) = copy_selected_checkpoint(
    "webqsp",
    webqsp_deployment_run
)

(
    cwq_best_checkpoint,
    cwq_best_checkpoint_sha256
) = copy_selected_checkpoint(
    "cwq",
    cwq_deployment_run
)

# ======================================================================
# 16. Save tie-aware evaluation tables
# ======================================================================

tie_aware_runs_file = (
    FINAL_SCORER_DIR
    / "tie_aware_validation_runs.csv"
)

tie_aware_configs_file = (
    FINAL_SCORER_DIR
    / "tie_aware_validation_configurations.csv"
)

tie_aware_results_df.to_csv(
    tie_aware_runs_file,
    index=False
)

tie_aware_config_summary.to_csv(
    tie_aware_configs_file,
    index=False
)

# ======================================================================
# 17. Build selection record
# ======================================================================

def build_selection_record(
    dataset_name,
    selected_config,
    deployment_run,
    checkpoint_path,
    checkpoint_sha
):
    return {
        "dataset":
            dataset_name,

        "selected_hidden_dim":
            int(
                selected_config[
                    "hidden_dim"
                ]
            ),

        "selected_loss":
            selected_config[
                "loss_name"
            ],

        "selection_seeds":
            [42, 43, 44],

        "deployment_seed":
            DEPLOYMENT_SEED,

        "mean_tie_aware_group_ap":
            float(
                selected_config[
                    "group_ap_mean"
                ]
            ),

        "std_tie_aware_group_ap":
            float(
                selected_config[
                    "group_ap_std"
                ]
            ),

        "mean_tie_aware_mrr":
            float(
                selected_config[
                    "mrr_mean"
                ]
            ),

        "mean_tie_aware_top1":
            float(
                selected_config[
                    "top1_mean"
                ]
            ),

        "mean_group_bce":
            float(
                selected_config[
                    "group_bce_mean"
                ]
            ),

        "deployment_tie_aware_group_ap":
            float(
                deployment_run[
                    "tie_aware_group_ap"
                ]
            ),

        "deployment_tie_aware_mrr":
            float(
                deployment_run[
                    "tie_aware_mrr"
                ]
            ),

        "deployment_tie_aware_top1":
            float(
                deployment_run[
                    "tie_aware_top1"
                ]
            ),

        "deployment_group_bce":
            float(
                deployment_run[
                    "group_bce"
                ]
            ),

        "checkpoint":
            checkpoint_path,

        "checkpoint_sha256":
            checkpoint_sha,
    }

webqsp_scorer_selection = build_selection_record(
    "webqsp",
    webqsp_selected_config,
    webqsp_deployment_run,
    webqsp_best_checkpoint,
    webqsp_best_checkpoint_sha256
)

cwq_scorer_selection = build_selection_record(
    "cwq",
    cwq_selected_config,
    cwq_deployment_run,
    cwq_best_checkpoint,
    cwq_best_checkpoint_sha256
)

# ======================================================================
# 18. Selection manifest
# ======================================================================

selection_manifest = {
    "selection_version":
        "afp_scorer_selection_v2_tie_aware",

    "feature_version":
        AFP_FEATURE_VERSION,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_version":
        AFP_SCORER_VERSION,

    "training_version":
        AFP_TRAINING_VERSION,

    "selection_metric_revision": {
        "reason":
            "Extensive representation/score ties discovered by "
            "Cells 9A-9B make ordinary stable-sort ranking metrics "
            "candidate-order dependent.",

        "tie_handling":
            "Exact expected metric under uniform random ordering "
            "within numerically tied score blocks.",

        "tie_atol":
            TIE_ATOL,

        "new_training":
            False,
    },

    "selection_policy": [
        "highest mean tie-aware Group AP",
        "highest mean tie-aware MRR",
        "highest mean tie-aware Top-1",
        "lowest mean Group-Balanced BCE",
    ],

    "configuration_selection_seeds":
        [42, 43, 44],

    "deployment_seed":
        DEPLOYMENT_SEED,

    "webqsp":
        webqsp_scorer_selection,

    "cwq":
        cwq_scorer_selection,

    "feature_revision":
        False,

    "test_used_for_selection":
        False,

    "test_metrics_observed":
        False,

    "complete_afp_frozen":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

selection_manifest_path = (
    FINAL_SCORER_DIR
    / "afp_scorer_selection_manifest.json"
)

with open(
    selection_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selection_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 19. Final report
# ======================================================================

def print_selection(dataset_name, selection):
    print("\n" + "-" * 78)
    print(dataset_name.upper())
    print("-" * 78)

    print(
        "Selected H:              ",
        selection[
            "selected_hidden_dim"
        ]
    )

    print(
        "Selected loss:           ",
        selection[
            "selected_loss"
        ]
    )

    print(
        "Mean tie-aware Group AP: ",
        f"{selection['mean_tie_aware_group_ap']:.4f}"
    )

    print(
        "Mean tie-aware MRR:      ",
        f"{selection['mean_tie_aware_mrr']:.4f}"
    )

    print(
        "Mean tie-aware Top-1:    ",
        f"{selection['mean_tie_aware_top1']:.4f}"
    )

    print(
        "Deployment seed:         ",
        selection[
            "deployment_seed"
        ]
    )

    print(
        "Seed-42 tie-aware AP:    ",
        f"{selection['deployment_tie_aware_group_ap']:.4f}"
    )

    print(
        "Checkpoint SHA256:       ",
        selection[
            "checkpoint_sha256"
        ][:16] + "..."
    )

print_selection(
    "WebQSP",
    webqsp_scorer_selection
)

print_selection(
    "CWQ",
    cwq_scorer_selection
)

print("\n" + "=" * 92)
print("=== RQ2 CELL 9C: TIE-AWARE SCORER VALIDATION + SELECTION COMPLETE ===")
print("=" * 92)

print("All 36 saved checkpoints revalidated: YES")
print("Tie-aware ranking used for selection: YES")
print("Configuration aggregated over 3 seeds: YES")
print("Validation-selected seed:             NO")
print("Fixed deployment seed:                42")
print("New scorer training:                  NO")
print("Feature-v2 modified:                  NO")
print("TEST data/gold used:                  NO")
print("TEST metrics observed:                NO")
print("Selected scorer checkpoints saved:    YES")
print("Complete AFP configuration frozen:    NO")
print()
print("Next: adaptive selector definition + validation tuning.")

Deployment seed: 42
Tie tolerance:   1e-08

Re-evaluating all 36 checkpoints with tie-aware metrics...
[01/36] webqsp H=32 branch_bce seed=42 | AP=0.6122 MRR=0.6480 Top1=0.4841
[02/36] webqsp H=32 branch_bce seed=43 | AP=0.6071 MRR=0.6386 Top1=0.4713
[03/36] webqsp H=32 branch_bce seed=44 | AP=0.6048 MRR=0.6364 Top1=0.4495
[04/36] webqsp H=32 group_balanced_bce seed=42 | AP=0.6024 MRR=0.6359 Top1=0.4604
[05/36] webqsp H=32 group_balanced_bce seed=43 | AP=0.6178 MRR=0.6557 Top1=0.4956
[06/36] webqsp H=32 group_balanced_bce seed=44 | AP=0.6004 MRR=0.6369 Top1=0.4611
[07/36] webqsp H=64 branch_bce seed=42 | AP=0.6069 MRR=0.6415 Top1=0.4725
[08/36] webqsp H=64 branch_bce seed=43 | AP=0.5941 MRR=0.6162 Top1=0.4259
[09/36] webqsp H=64 branch_bce seed=44 | AP=0.6003 MRR=0.6307 Top1=0.4598
[10/36] webqsp H=64 group_balanced_bce seed=42 | AP=0.6135 MRR=0.6501 Top1=0.4833
[11/36] webqsp H=64 group_balanced_bce seed=43 | AP=0.5952 MRR=0.6314 Top1=0.4603
[12/36] webqsp H=64 group_balanced_bce seed

## Implement AFP confidence-aware adaptive selector

In [4]:
# ======================================================================
# 3.10 DEFINE ADAPTIVE AFP SELECTOR
# ======================================================================
#
# pi_i    = softmax(logit_i / T)
# u_h     = normalized entropy(pi)
# gamma_h = gamma_min + u_h * (1 - gamma_min)
#
# B_h = smallest cumulative top-B probability mass reaching gamma_h.
#
# Safeguards:
#   - singleton -> retain all
#   - final hop -> retain all
#   - fully tied logits -> retain all
#   - score tie at pruning boundary -> preserve whole tied class
#
# NO tuning here.
# NO graph traversal here.
# NO test data.
# ======================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

# ======================================================================
# 1. Selector specification
# ======================================================================

AFP_SELECTOR_VERSION = "afp_adaptive_selector_v1"

AFP_SELECTOR_SPEC = {
    "version": AFP_SELECTOR_VERSION,
    "probability": "softmax(logits/T)",
    "uncertainty": "normalized_entropy",
    "gamma": "gamma_min + u*(1-gamma_min)",
    "budget": "minimum cumulative probability mass",
    "singleton_bypass": True,
    "final_hop_protection": True,
    "all_score_ties_retain_all": True,
    "cutoff_tie_expansion": True,
    "temperature_validation_tuned": True,
    "gamma_min_validation_tuned": True,
    "test_used_for_tuning": False,
}

AFP_SELECTOR_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_SELECTOR_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Selector version:", AFP_SELECTOR_VERSION)
print("Selector SHA256:", AFP_SELECTOR_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. Scorer invocation policy
# ======================================================================

def afp_should_score(candidate_count, hop, plan_length):
    assert candidate_count >= 0
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    if candidate_count <= 1:
        return False

    if hop == plan_length - 1:
        return False

    return True

# ======================================================================
# 3. Stable temperature-scaled softmax
# ======================================================================

def afp_softmax(logits, temperature):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert temperature > 0

    z = logits / float(temperature)
    z -= np.max(z)

    exp_z = np.exp(z)
    probs = exp_z / exp_z.sum()

    assert np.all(np.isfinite(probs))
    assert np.all(probs >= 0)
    assert np.isclose(probs.sum(), 1.0)

    return probs

# ======================================================================
# 4. Normalized entropy
# ======================================================================

def afp_normalized_entropy(probs):
    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    n = len(probs)

    if n <= 1:
        return 0.0

    assert np.all(probs >= 0)
    assert np.isclose(probs.sum(), 1.0)

    nz = probs > 0

    entropy = -np.sum(
        probs[nz] * np.log(probs[nz])
    )

    u = entropy / np.log(n)

    return float(
        np.clip(u, 0.0, 1.0)
    )

# ======================================================================
# 5. Adaptive gamma
# ======================================================================

def afp_gamma(uncertainty, gamma_min):
    assert 0.0 <= uncertainty <= 1.0
    assert 0.0 < gamma_min <= 1.0

    gamma = (
        gamma_min
        + uncertainty
        * (1.0 - gamma_min)
    )

    return float(
        np.clip(
            gamma,
            gamma_min,
            1.0
        )
    )

# ======================================================================
# 6. Adaptive selector for an intermediate decision frontier
# ======================================================================

def afp_select_from_logits(
    logits,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))

    n = len(logits)

    # Singleton safety
    if n == 1:
        return {
            "selected_indices":
                np.asarray([0], dtype=np.int64),
            "probabilities":
                np.asarray([1.0], dtype=np.float64),
            "uncertainty": 0.0,
            "gamma": 1.0,
            "requested_B": 1,
            "retained_B": 1,
            "retained_mass": 1.0,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_retain_all",
        }

    probs = afp_softmax(
        logits,
        temperature
    )

    uncertainty = afp_normalized_entropy(
        probs
    )

    gamma = afp_gamma(
        uncertainty,
        gamma_min
    )

    # --------------------------------------------------------------
    # Fully tied logits:
    # scorer cannot distinguish candidates -> abstain from pruning.
    # --------------------------------------------------------------
    if (
        float(logits.max() - logits.min())
        <= tie_tolerance
    ):
        return {
            "selected_indices":
                np.arange(n, dtype=np.int64),
            "probabilities": probs,
            "uncertainty": uncertainty,
            "gamma": gamma,
            "requested_B": n,
            "retained_B": n,
            "retained_mass": 1.0,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "all_scores_tied_retain_all",
        }

    # Stable descending score order
    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    ranked_probs = probs[order]
    cumulative = np.cumsum(
        ranked_probs
    )

    # Numerical safety
    cumulative[-1] = 1.0

    requested_B = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        ) + 1
    )

    requested_B = min(
        requested_B,
        n
    )

    # --------------------------------------------------------------
    # Preserve score ties at the cutoff.
    # --------------------------------------------------------------
    cutoff_score = float(
        logits[
            order[
                requested_B - 1
            ]
        ]
    )

    retained_B = requested_B

    while retained_B < n:
        next_score = float(
            logits[
                order[retained_B]
            ]
        )

        if (
            abs(next_score - cutoff_score)
            <= tie_tolerance
        ):
            retained_B += 1
        else:
            break

    selected = order[
        :retained_B
    ].astype(np.int64)

    retained_mass = float(
        probs[selected].sum()
    )

    assert 1 <= requested_B <= retained_B <= n
    assert retained_mass + 1e-12 >= gamma

    return {
        "selected_indices": selected,
        "probabilities": probs,
        "uncertainty": uncertainty,
        "gamma": gamma,
        "requested_B": requested_B,
        "retained_B": retained_B,
        "retained_mass": retained_mass,
        "pruned_count": n - retained_B,
        "pruning_fraction":
            (n - retained_B) / n,
        "reason": (
            "adaptive_with_tie_expansion"
            if retained_B > requested_B
            else "adaptive"
        ),
    }

# ======================================================================
# 7. Traversal-level wrapper
# ======================================================================

def afp_select_frontier(
    logits,
    candidate_count,
    hop,
    plan_length,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    assert candidate_count >= 1
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    # Final-hop protection
    if hop == plan_length - 1:
        return {
            "selected_indices":
                np.arange(
                    candidate_count,
                    dtype=np.int64
                ),
            "retained_B":
                candidate_count,
            "requested_B":
                candidate_count,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "final_hop_protection",
            "scorer_invoked": False,
        }

    # Singleton bypass
    if candidate_count == 1:
        return {
            "selected_indices":
                np.asarray(
                    [0],
                    dtype=np.int64
                ),
            "retained_B": 1,
            "requested_B": 1,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_bypass",
            "scorer_invoked": False,
        }

    assert logits is not None
    assert len(logits) == candidate_count

    result = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    result["scorer_invoked"] = True

    return result

# ======================================================================
# 8. Sanity A — invocation policy
# ======================================================================

assert not afp_should_score(
    candidate_count=1,
    hop=0,
    plan_length=3
)

assert not afp_should_score(
    candidate_count=5,
    hop=2,
    plan_length=3
)

assert afp_should_score(
    candidate_count=5,
    hop=1,
    plan_length=3
)

print("\nScorer invocation gate: PASSED")

# ======================================================================
# 9. Sanity B — uniform logits -> maximal uncertainty -> retain all
# ======================================================================

r = afp_select_from_logits(
    logits=[0.0, 0.0, 0.0, 0.0],
    temperature=1.0,
    gamma_min=0.70
)

assert np.isclose(
    r["uncertainty"],
    1.0
)

assert np.isclose(
    r["gamma"],
    1.0
)

assert r["retained_B"] == 4
assert r["pruned_count"] == 0

print("Uniform-score abstention gate: PASSED")

# ======================================================================
# 10. Sanity C — confident frontier can prune
# ======================================================================

r = afp_select_from_logits(
    logits=[10.0, 0.0, -1.0, -2.0],
    temperature=1.0,
    gamma_min=0.70
)

assert r["retained_B"] < 4
assert r["pruned_count"] > 0
assert 0 in r["selected_indices"]

print("Confident-pruning gate: PASSED")

# ======================================================================
# 11. Sanity D — final hop is fully protected
# ======================================================================

r = afp_select_frontier(
    logits=None,
    candidate_count=7,
    hop=2,
    plan_length=3,
    temperature=1.0,
    gamma_min=0.70
)

assert r["retained_B"] == 7
assert r["scorer_invoked"] is False
assert r["reason"] == "final_hop_protection"

print("Final-hop protection gate: PASSED")

# ======================================================================
# 12. Sanity E — cutoff score ties are not split
# ======================================================================

r = afp_select_from_logits(
    logits=[3.0, 1.0, 1.0, 1.0],
    temperature=1.0,
    gamma_min=0.50
)

if r["requested_B"] > 1:
    assert r["retained_B"] == 4

print("Cutoff-tie preservation gate: PASSED")

# ======================================================================
# 13. Sanity F — permutation equivariance
# ======================================================================

base_logits = np.asarray(
    [4.0, 2.5, 1.0, -1.0]
)

base = afp_select_from_logits(
    base_logits,
    temperature=1.0,
    gamma_min=0.70
)

perm = np.asarray(
    [2, 0, 3, 1]
)

permuted = afp_select_from_logits(
    base_logits[perm],
    temperature=1.0,
    gamma_min=0.70
)

selected_original = set(
    base["selected_indices"].tolist()
)

selected_after_permutation = set(
    perm[
        permuted["selected_indices"]
    ].tolist()
)

assert (
    selected_original
    ==
    selected_after_permutation
)

print("Permutation-equivariance gate: PASSED")

# ======================================================================
# 14. Illustrative behavior only
# ======================================================================
#
# T=1 and gamma_min=0.70 are NOT selected values here.
# ======================================================================

examples = {
    "uniform":
        [0.0, 0.0, 0.0, 0.0],

    "weak":
        [1.0, 0.9, 0.8, 0.7],

    "moderate":
        [2.0, 1.0, 0.5, 0.0],

    "strong":
        [8.0, 1.0, 0.0, -1.0],
}

print("\n" + "=" * 86)
print("ILLUSTRATIVE SELECTOR BEHAVIOR — NOT TUNED")
print("=" * 86)

for name, logits in examples.items():
    r = afp_select_from_logits(
        logits=logits,
        temperature=1.0,
        gamma_min=0.70
    )

    print(
        f"{name:<10} "
        f"u={r['uncertainty']:.4f}  "
        f"gamma={r['gamma']:.4f}  "
        f"B={r['retained_B']}/{len(logits)}  "
        f"{r['reason']}"
    )

# ======================================================================
# 15. Save definition manifest
# ======================================================================

SELECTOR_DIR = (
    Path(RQ2_ROOT)
    / "08_adaptive_selector"
)

SELECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

selector_manifest = {
    "selector_version":
        AFP_SELECTOR_VERSION,

    "selector_spec_sha256":
        AFP_SELECTOR_SPEC_SHA256,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "selected_scorers": {
        "webqsp":
            webqsp_best_checkpoint,

        "cwq":
            cwq_best_checkpoint,
    },

    "formula": {
        "probability":
            "softmax(logits/T)",

        "uncertainty":
            "-sum(pi*log(pi))/log(n)",

        "gamma":
            "gamma_min + u*(1-gamma_min)",

        "budget":
            "smallest cumulative top-B probability mass reaching gamma",
    },

    "safeguards": {
        "singleton_bypass": True,
        "final_hop_protection": True,
        "fully_tied_scores_retain_all": True,
        "cutoff_score_ties_preserved": True,
    },

    "temperature_selected":
        False,

    "gamma_min_selected":
        False,

    "validation_traversal_tuning_pending":
        True,

    "test_used":
        False,

    "complete_afp_frozen":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

selector_manifest_path = (
    SELECTOR_DIR
    / "adaptive_selector_definition_manifest.json"
)

with open(
    selector_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selector_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 16. Final report
# ======================================================================

print("\n" + "=" * 90)
print("=== RQ2 CELL 10: ADAPTIVE AFP SELECTOR DEFINED ===")
print("=" * 90)

print("Temperature-scaled softmax:    YES")
print("Normalized entropy:            YES")
print("Adaptive gamma:                YES")
print("Cumulative-mass budget:        YES")
print("Singleton bypass:              YES")
print("Final-hop protection:          YES")
print("Fully tied scores retain all:  YES")
print("Cutoff score ties preserved:   YES")
print()
print("Temperature tuned:             NO")
print("gamma_min tuned:               NO")
print("New training:                  NO")
print("TEST data/gold used:           NO")
print("Complete AFP frozen:           NO")
print()
print("Next: validation traversal integration + selector tuning.")

Selector version: afp_adaptive_selector_v1
Selector SHA256: 62fad1e1f5869a54...

Scorer invocation gate: PASSED
Uniform-score abstention gate: PASSED
Confident-pruning gate: PASSED
Final-hop protection gate: PASSED
Cutoff-tie preservation gate: PASSED
Permutation-equivariance gate: PASSED

ILLUSTRATIVE SELECTOR BEHAVIOR — NOT TUNED
uniform    u=1.0000  gamma=1.0000  B=4/4  all_scores_tied_retain_all
weak       u=0.9955  gamma=0.9987  B=4/4  adaptive
moderate   u=0.8005  gamma=0.9402  B=4/4  adaptive
strong     u=0.0083  gamma=0.7025  B=1/4  adaptive

=== RQ2 CELL 10: ADAPTIVE AFP SELECTOR DEFINED ===
Temperature-scaled softmax:    YES
Normalized entropy:            YES
Adaptive gamma:                YES
Cumulative-mass budget:        YES
Singleton bypass:              YES
Final-hop protection:          YES
Fully tied scores retain all:  YES
Cutoff score ties preserved:   YES

Temperature tuned:             NO
gamma_min tuned:               NO
New training:                  NO
TEST data

## Implement controlled baselines — RoG, Top-\(B\), threshold, Random-\(B\), adaptive-budget random

In [5]:
# ======================================================================
# 3.11 IMPLEMENT CONTROLLED PRUNING BASELINES
# ======================================================================
#
# Methods:
#   1. RoG
#   2. Fixed Top-B
#   3. Fixed Threshold
#   4. Random-B
#   5. Adaptive-Budget Random
#   6. AFP
#
# Controlled comparison principle:
#   Everything except frontier-selection policy remains fixed.
#
# Shared safeguards:
#   - singleton frontier -> retain all
#   - final hop -> retain all
#
# IMPORTANT:
#   - No hyperparameter tuning here.
#   - No graph traversal here.
#   - No test data.
#   - Random methods use deterministic per-group seeds.
# ======================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_SELECTOR_VERSION == "afp_adaptive_selector_v1"
assert "afp_select_from_logits" in globals()
assert "afp_select_frontier" in globals()

CONTROLLED_METHODS = [
    "rog",
    "fixed_top_b",
    "fixed_threshold",
    "random_b",
    "adaptive_budget_random",
    "afp",
]

RANDOM_BASELINE_SEEDS = [42, 43, 44]

print("Controlled methods:", CONTROLLED_METHODS)
print("Random seeds:", RANDOM_BASELINE_SEEDS)

# ======================================================================
# 2. Baseline specification
# ======================================================================

CONTROLLED_BASELINE_VERSION = "afp_controlled_baselines_v1"

CONTROLLED_BASELINE_SPEC = {
    "version": CONTROLLED_BASELINE_VERSION,
    "methods": CONTROLLED_METHODS,
    "shared_final_hop_protection": True,
    "shared_singleton_bypass": True,

    "rog": "retain_all",

    "fixed_top_b": {
        "ranking": "AFP scorer logits",
        "budget": "fixed_B",
        "cutoff_score_ties": "expand_full_tied_class",
    },

    "fixed_threshold": {
        "score": "sigmoid(AFP scorer logit)",
        "rule": "retain score >= tau",
        "empty_frontier_allowed": True,
    },

    "random_b": {
        "budget": "fixed_B",
        "selection": "uniform_without_replacement",
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "adaptive_budget_random": {
        "budget": "AFP adaptive retained_B",
        "selection": "uniform_without_replacement",
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "afp": {
        "budget": "confidence-aware adaptive",
        "selection": "AFP scorer ranking",
    },

    "test_used": False,
}

CONTROLLED_BASELINE_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        CONTROLLED_BASELINE_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Baseline version:", CONTROLLED_BASELINE_VERSION)
print("Baseline SHA256:", CONTROLLED_BASELINE_SPEC_SHA256[:16] + "...")

# ======================================================================
# 3. Helpers
# ======================================================================

def sigmoid_np(logits):
    logits = np.asarray(logits, dtype=np.float64)

    out = np.empty_like(logits)

    positive = logits >= 0
    negative = ~positive

    out[positive] = (
        1.0 /
        (1.0 + np.exp(-logits[positive]))
    )

    exp_x = np.exp(logits[negative])
    out[negative] = exp_x / (1.0 + exp_x)

    return out


def preserve_original_candidate_order(indices):
    """
    Membership may be determined by ranking/random selection,
    but surviving candidates propagate in their ORIGINAL RoG order.

    This prevents traversal/path-order changes becoming a confound.
    """
    return np.asarray(
        sorted(
            int(i) for i in indices
        ),
        dtype=np.int64
    )


def deterministic_group_rng(base_seed, group_key):
    """
    Stable per-group RNG.

    Python's built-in hash() is intentionally NOT used because its
    value may differ across interpreter sessions.
    """
    payload = (
        f"{int(base_seed)}|{str(group_key)}"
    ).encode("utf-8")

    digest = hashlib.sha256(payload).digest()

    group_seed = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )

    return np.random.default_rng(group_seed)

# ======================================================================
# 4. RoG — no pruning
# ======================================================================

def select_rog(candidate_count):
    assert candidate_count >= 1

    selected = np.arange(
        candidate_count,
        dtype=np.int64
    )

    return {
        "selected_indices": selected,
        "requested_B": candidate_count,
        "retained_B": candidate_count,
        "pruned_count": 0,
        "pruning_fraction": 0.0,
        "reason": "rog_retain_all",
        "scorer_invoked": False,
        "budget_source": "none",
        "selection_source": "none",
    }

# ======================================================================
# 5. Fixed Top-B
# ======================================================================
#
# Top-B is determined by AFP scorer logits.
#
# IMPORTANT:
# If the B-th score is tied with later candidates, the whole tied
# score class is retained. We do not use arbitrary list order to
# decide between representation-identical candidates.
# ======================================================================

def select_fixed_top_b(
    logits,
    B,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert int(B) >= 1

    n = len(logits)
    requested_B = min(int(B), n)

    if requested_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )

        return {
            "selected_indices": selected,
            "requested_B": requested_B,
            "retained_B": n,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "fixed_top_b_retain_all",
            "scorer_invoked": True,
            "budget_source": "fixed",
            "selection_source": "scorer",
        }

    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    cutoff_score = float(
        logits[
            order[requested_B - 1]
        ]
    )

    retained_B = requested_B

    while retained_B < n:
        next_score = float(
            logits[
                order[retained_B]
            ]
        )

        if (
            abs(next_score - cutoff_score)
            <= tie_tolerance
        ):
            retained_B += 1
        else:
            break

    ranked_selected = order[:retained_B]

    # Propagate in original candidate order.
    selected = preserve_original_candidate_order(
        ranked_selected
    )

    return {
        "selected_indices": selected,
        "requested_B": requested_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": (
            "fixed_top_b_with_tie_expansion"
            if retained_B > requested_B
            else "fixed_top_b"
        ),
        "scorer_invoked": True,
        "budget_source": "fixed",
        "selection_source": "scorer",
    }

# ======================================================================
# 6. Fixed probability threshold
# ======================================================================
#
# s_i = sigmoid(logit_i)
#
# retain iff:
#       s_i >= tau
#
# No forced Top-1 fallback is introduced.
# If no branch passes tau, the intermediate traversal terminates.
# That behavior is part of the threshold baseline itself.
# ======================================================================

def select_fixed_threshold(
    logits,
    threshold
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert 0.0 <= threshold <= 1.0

    probs = sigmoid_np(
        logits
    )

    selected = np.flatnonzero(
        probs >= float(threshold)
    ).astype(np.int64)

    n = len(logits)
    retained_B = len(selected)

    return {
        "selected_indices": selected,
        "probabilities": probs,
        "requested_B": retained_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": (
            "fixed_threshold"
            if retained_B > 0
            else "fixed_threshold_empty"
        ),
        "scorer_invoked": True,
        "budget_source": "threshold",
        "selection_source": "scorer",
    }

# ======================================================================
# 7. Fixed Random-B
# ======================================================================

def select_random_b(
    candidate_count,
    B,
    seed,
    group_key
):
    assert candidate_count >= 1
    assert int(B) >= 1
    assert int(seed) in RANDOM_BASELINE_SEEDS

    n = candidate_count
    retained_B = min(
        int(B),
        n
    )

    if retained_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )
    else:
        rng = deterministic_group_rng(
            seed,
            group_key
        )

        selected = rng.choice(
            n,
            size=retained_B,
            replace=False
        )

        selected = (
            preserve_original_candidate_order(
                selected
            )
        )

    return {
        "selected_indices": selected,
        "requested_B": retained_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": "random_b",
        "scorer_invoked": False,
        "budget_source": "fixed",
        "selection_source": "random",
        "random_seed": int(seed),
    }

# ======================================================================
# 8. Adaptive-Budget Random
# ======================================================================
#
# This baseline uses AFP ONLY to determine HOW MANY candidates should
# survive.
#
# It deliberately ignores AFP's ranking when deciding WHICH candidates
# survive.
#
# Therefore:
#
#   AFP vs Adaptive-Budget Random
#
# isolates the value of learned ranking while holding adaptive budget
# behavior approximately fixed.
# ======================================================================

def select_adaptive_budget_random(
    logits,
    temperature,
    gamma_min,
    seed,
    group_key,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert int(seed) in RANDOM_BASELINE_SEEDS

    n = len(logits)

    # Compute EXACT AFP budget.
    afp_reference = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    retained_B = int(
        afp_reference[
            "retained_B"
        ]
    )

    assert 1 <= retained_B <= n

    if retained_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )
    else:
        rng = deterministic_group_rng(
            seed,
            group_key
        )

        selected = rng.choice(
            n,
            size=retained_B,
            replace=False
        )

        selected = (
            preserve_original_candidate_order(
                selected
            )
        )

    return {
        "selected_indices": selected,

        "requested_B":
            int(
                afp_reference[
                    "requested_B"
                ]
            ),

        "retained_B":
            retained_B,

        "pruned_count":
            n - retained_B,

        "pruning_fraction":
            (n - retained_B) / n,

        "uncertainty":
            afp_reference[
                "uncertainty"
            ],

        "gamma":
            afp_reference[
                "gamma"
            ],

        "retained_mass":
            afp_reference[
                "retained_mass"
            ],

        "afp_budget_reason":
            afp_reference[
                "reason"
            ],

        "reason":
            "adaptive_budget_random",

        "scorer_invoked":
            True,

        "budget_source":
            "afp_adaptive",

        "selection_source":
            "random",

        "random_seed":
            int(seed),
    }

# ======================================================================
# 9. AFP policy wrapper
# ======================================================================

def select_afp_policy(
    logits,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    result = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    # Preserve original RoG candidate ordering after membership selection.
    result = dict(result)

    result["selected_indices"] = (
        preserve_original_candidate_order(
            result["selected_indices"]
        )
    )

    result["scorer_invoked"] = True
    result["budget_source"] = "afp_adaptive"
    result["selection_source"] = "scorer"

    return result

# ======================================================================
# 10. Unified controlled-selection interface
# ======================================================================

def controlled_select_frontier(
    method,
    candidate_count,
    hop,
    plan_length,
    logits=None,

    # Fixed Top-B / Random-B
    B=None,

    # Fixed threshold
    threshold=None,

    # AFP / Adaptive-Budget Random
    temperature=None,
    gamma_min=None,

    # Random baselines
    seed=None,
    group_key=None,

    tie_tolerance=1e-8
):
    """
    Common selection interface for controlled validation/test traversal.

    All methods receive the SAME candidate frontier.

    The only difference is the selection policy.
    """
    assert method in CONTROLLED_METHODS
    assert candidate_count >= 1
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    # --------------------------------------------------------------
    # RoG never prunes.
    # --------------------------------------------------------------
    if method == "rog":
        return select_rog(
            candidate_count
        )

    # --------------------------------------------------------------
    # Shared FINAL-HOP protection for ALL pruning baselines.
    # --------------------------------------------------------------
    if hop == plan_length - 1:
        selected = np.arange(
            candidate_count,
            dtype=np.int64
        )

        return {
            "selected_indices": selected,
            "requested_B": candidate_count,
            "retained_B": candidate_count,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "final_hop_protection",
            "scorer_invoked": False,
            "budget_source": "protected",
            "selection_source": "protected",
        }

    # --------------------------------------------------------------
    # Shared singleton bypass.
    # --------------------------------------------------------------
    if candidate_count == 1:
        return {
            "selected_indices":
                np.asarray(
                    [0],
                    dtype=np.int64
                ),
            "requested_B": 1,
            "retained_B": 1,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_bypass",
            "scorer_invoked": False,
            "budget_source": "bypass",
            "selection_source": "bypass",
        }

    # --------------------------------------------------------------
    # Fixed Top-B
    # --------------------------------------------------------------
    if method == "fixed_top_b":
        assert logits is not None
        assert len(logits) == candidate_count
        assert B is not None

        return select_fixed_top_b(
            logits=logits,
            B=B,
            tie_tolerance=tie_tolerance
        )

    # --------------------------------------------------------------
    # Fixed Threshold
    # --------------------------------------------------------------
    if method == "fixed_threshold":
        assert logits is not None
        assert len(logits) == candidate_count
        assert threshold is not None

        return select_fixed_threshold(
            logits=logits,
            threshold=threshold
        )

    # --------------------------------------------------------------
    # Fixed Random-B
    # --------------------------------------------------------------
    if method == "random_b":
        assert B is not None
        assert seed is not None
        assert group_key is not None

        return select_random_b(
            candidate_count=candidate_count,
            B=B,
            seed=seed,
            group_key=group_key
        )

    # --------------------------------------------------------------
    # Adaptive-Budget Random
    # --------------------------------------------------------------
    if method == "adaptive_budget_random":
        assert logits is not None
        assert len(logits) == candidate_count
        assert temperature is not None
        assert gamma_min is not None
        assert seed is not None
        assert group_key is not None

        return select_adaptive_budget_random(
            logits=logits,
            temperature=temperature,
            gamma_min=gamma_min,
            seed=seed,
            group_key=group_key,
            tie_tolerance=tie_tolerance
        )

    # --------------------------------------------------------------
    # AFP
    # --------------------------------------------------------------
    if method == "afp":
        assert logits is not None
        assert len(logits) == candidate_count
        assert temperature is not None
        assert gamma_min is not None

        return select_afp_policy(
            logits=logits,
            temperature=temperature,
            gamma_min=gamma_min,
            tie_tolerance=tie_tolerance
        )

    raise RuntimeError(
        f"Unhandled method: {method}"
    )

# ======================================================================
# 11. Sanity Gate A — RoG retains all
# ======================================================================

r = controlled_select_frontier(
    method="rog",
    candidate_count=5,
    hop=0,
    plan_length=3
)

assert r["retained_B"] == 5
assert r["pruned_count"] == 0
assert np.array_equal(
    r["selected_indices"],
    np.arange(5)
)

print("\nRoG baseline gate: PASSED")

# ======================================================================
# 12. Sanity Gate B — Fixed Top-B
# ======================================================================

r = controlled_select_frontier(
    method="fixed_top_b",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=[5.0, 4.0, 3.0, 2.0, 1.0],
    B=2
)

assert r["requested_B"] == 2
assert r["retained_B"] == 2
assert set(
    r["selected_indices"].tolist()
) == {0, 1}

print("Fixed Top-B gate: PASSED")

# ======================================================================
# 13. Sanity Gate C — Top-B does not split score ties
# ======================================================================

r = controlled_select_frontier(
    method="fixed_top_b",
    candidate_count=4,
    hop=0,
    plan_length=3,
    logits=[4.0, 2.0, 2.0, 2.0],
    B=2
)

assert r["requested_B"] == 2
assert r["retained_B"] == 4

print("Fixed Top-B tie-preservation gate: PASSED")

# ======================================================================
# 14. Sanity Gate D — Threshold
# ======================================================================

r = controlled_select_frontier(
    method="fixed_threshold",
    candidate_count=4,
    hop=0,
    plan_length=3,
    logits=[2.0, 0.5, -1.0, -3.0],
    threshold=0.50
)

# sigmoid(2), sigmoid(.5) >= .5
# sigmoid(-1), sigmoid(-3) < .5
assert set(
    r["selected_indices"].tolist()
) == {0, 1}

print("Fixed-threshold gate: PASSED")

# ======================================================================
# 15. Sanity Gate E — Threshold may terminate frontier
# ======================================================================

r = controlled_select_frontier(
    method="fixed_threshold",
    candidate_count=3,
    hop=0,
    plan_length=3,
    logits=[-5.0, -4.0, -3.0],
    threshold=0.95
)

assert r["retained_B"] == 0
assert r["reason"] == "fixed_threshold_empty"

print("Threshold-empty gate: PASSED")

# ======================================================================
# 16. Sanity Gate F — Random-B reproducibility
# ======================================================================

r1 = controlled_select_frontier(
    method="random_b",
    candidate_count=20,
    hop=0,
    plan_length=3,
    B=4,
    seed=42,
    group_key="example-question|plan0|topic0|hop0"
)

r2 = controlled_select_frontier(
    method="random_b",
    candidate_count=20,
    hop=0,
    plan_length=3,
    B=4,
    seed=42,
    group_key="example-question|plan0|topic0|hop0"
)

assert np.array_equal(
    r1["selected_indices"],
    r2["selected_indices"]
)

assert r1["retained_B"] == 4

print("Random-B reproducibility gate: PASSED")

# ======================================================================
# 17. Sanity Gate G — random seeds actually differ
# ======================================================================

seed_sets = []

for seed in RANDOM_BASELINE_SEEDS:
    r = controlled_select_frontier(
        method="random_b",
        candidate_count=50,
        hop=0,
        plan_length=3,
        B=5,
        seed=seed,
        group_key="seed-difference-check"
    )

    seed_sets.append(
        tuple(
            r["selected_indices"].tolist()
        )
    )

assert len(
    set(seed_sets)
) > 1

print("Random-seed differentiation gate: PASSED")

# ======================================================================
# 18. Sanity Gate H — Adaptive-Budget Random uses exact AFP budget
# ======================================================================

test_logits = np.asarray(
    [8.0, 2.0, 1.0, 0.0, -1.0]
)

afp_reference = controlled_select_frontier(
    method="afp",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=test_logits,
    temperature=1.0,
    gamma_min=0.70
)

adaptive_random = controlled_select_frontier(
    method="adaptive_budget_random",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=test_logits,
    temperature=1.0,
    gamma_min=0.70,
    seed=42,
    group_key="adaptive-budget-check"
)

assert (
    adaptive_random["retained_B"]
    ==
    afp_reference["retained_B"]
)

assert (
    adaptive_random["requested_B"]
    ==
    afp_reference["requested_B"]
)

print("Adaptive-budget Random budget-equivalence gate: PASSED")

# ======================================================================
# 19. Sanity Gate I — final-hop protection shared by all methods
# ======================================================================

for method in CONTROLLED_METHODS:
    kwargs = {}

    if method in [
        "fixed_top_b",
        "fixed_threshold",
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["logits"] = [5.0, 1.0, -1.0]

    if method in [
        "fixed_top_b",
        "random_b",
    ]:
        kwargs["B"] = 1

    if method == "fixed_threshold":
        kwargs["threshold"] = 0.99

    if method in [
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["temperature"] = 1.0
        kwargs["gamma_min"] = 0.70

    if method in [
        "random_b",
        "adaptive_budget_random",
    ]:
        kwargs["seed"] = 42
        kwargs["group_key"] = "final-hop-test"

    r = controlled_select_frontier(
        method=method,
        candidate_count=3,
        hop=2,
        plan_length=3,
        **kwargs
    )

    assert r["retained_B"] == 3
    assert r["pruned_count"] == 0

print("Shared final-hop protection gate: PASSED")

# ======================================================================
# 20. Sanity Gate J — singleton bypass shared
# ======================================================================

for method in CONTROLLED_METHODS:
    kwargs = {}

    if method in [
        "fixed_top_b",
        "fixed_threshold",
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["logits"] = [0.0]

    if method in [
        "fixed_top_b",
        "random_b",
    ]:
        kwargs["B"] = 1

    if method == "fixed_threshold":
        kwargs["threshold"] = 0.99

    if method in [
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["temperature"] = 1.0
        kwargs["gamma_min"] = 0.70

    if method in [
        "random_b",
        "adaptive_budget_random",
    ]:
        kwargs["seed"] = 42
        kwargs["group_key"] = "singleton-test"

    r = controlled_select_frontier(
        method=method,
        candidate_count=1,
        hop=0,
        plan_length=3,
        **kwargs
    )

    assert r["retained_B"] == 1
    assert r["pruned_count"] == 0

print("Shared singleton-bypass gate: PASSED")

# ======================================================================
# 21. Show policy characteristics
# ======================================================================

policy_table = [
    {
        "method": "RoG",
        "uses_scorer": False,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Fixed Top-B",
        "uses_scorer": True,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Fixed Threshold",
        "uses_scorer": True,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Random-B",
        "uses_scorer": False,
        "adaptive_budget": False,
        "random_selection": True,
    },
    {
        "method": "Adaptive-Budget Random",
        "uses_scorer": True,
        "adaptive_budget": True,
        "random_selection": True,
    },
    {
        "method": "AFP",
        "uses_scorer": True,
        "adaptive_budget": True,
        "random_selection": False,
    },
]

print("\n" + "=" * 92)
print("CONTROLLED METHOD CHARACTERISTICS")
print("=" * 92)

for p in policy_table:
    print(
        f"{p['method']:<24} "
        f"scorer={str(p['uses_scorer']):<5} "
        f"adaptive_budget={str(p['adaptive_budget']):<5} "
        f"random={str(p['random_selection']):<5}"
    )

# ======================================================================
# 22. Save baseline-definition manifest
# ======================================================================

BASELINE_DIR = (
    Path(RQ2_ROOT)
    / "09_controlled_baselines"
)

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_manifest = {
    "baseline_version":
        CONTROLLED_BASELINE_VERSION,

    "baseline_spec_sha256":
        CONTROLLED_BASELINE_SPEC_SHA256,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "selector_spec_sha256":
        AFP_SELECTOR_SPEC_SHA256,

    "scorer_selection_manifest":
        str(
            FINAL_SCORER_DIR
            / "afp_scorer_selection_manifest.json"
        ),

    "methods":
        CONTROLLED_METHODS,

    "random_seeds":
        RANDOM_BASELINE_SEEDS,

    "shared_controls": {
        "same_candidate_frontier": True,
        "same_relation_matching": True,
        "same_final_hop_protection": True,
        "same_singleton_bypass": True,
        "preserve_original_candidate_order_after_selection": True,
    },

    "fixed_top_b": {
        "hyperparameter_tuned": False,
        "cutoff_tie_expansion": True,
    },

    "fixed_threshold": {
        "hyperparameter_tuned": False,
        "empty_frontier_allowed": True,
        "forced_top1_fallback": False,
    },

    "random_b": {
        "hyperparameter_tuned": False,
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "adaptive_budget_random": {
        "temperature_tuned": False,
        "gamma_min_tuned": False,
        "budget_exactly_matches_afp": True,
    },

    "afp": {
        "temperature_tuned": False,
        "gamma_min_tuned": False,
    },

    "validation_traversal_run":
        False,

    "test_used":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

baseline_manifest_path = (
    BASELINE_DIR
    / "controlled_baseline_definition_manifest.json"
)

with open(
    baseline_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 23. Final report
# ======================================================================

print("\n" + "=" * 94)
print("=== RQ2 CELL 11: CONTROLLED BASELINE POLICIES IMPLEMENTED ===")
print("=" * 94)

print("RoG:                       YES")
print("Fixed Top-B:               YES")
print("Fixed Threshold:           YES")
print("Random-B:                  YES")
print("Adaptive-Budget Random:    YES")
print("AFP:                       YES")
print()
print("Shared final-hop protection: YES")
print("Shared singleton bypass:     YES")
print("Original candidate order:    PRESERVED after selection")
print("Random seeds:               ", RANDOM_BASELINE_SEEDS)
print()
print("Top-B tuned:                 NO")
print("Threshold tuned:             NO")
print("AFP temperature tuned:       NO")
print("AFP gamma_min tuned:         NO")
print("Validation traversal run:    NO")
print("TEST data/gold used:         NO")
print("Complete AFP frozen:         NO")
print()
print("Next: integrate all policies into the shared validation traversal.")

Controlled methods: ['rog', 'fixed_top_b', 'fixed_threshold', 'random_b', 'adaptive_budget_random', 'afp']
Random seeds: [42, 43, 44]
Baseline version: afp_controlled_baselines_v1
Baseline SHA256: 7170fa41d1bb1cd2...

RoG baseline gate: PASSED
Fixed Top-B gate: PASSED
Fixed Top-B tie-preservation gate: PASSED
Fixed-threshold gate: PASSED
Threshold-empty gate: PASSED
Random-B reproducibility gate: PASSED
Random-seed differentiation gate: PASSED
Adaptive-budget Random budget-equivalence gate: PASSED
Shared final-hop protection gate: PASSED
Shared singleton-bypass gate: PASSED

CONTROLLED METHOD CHARACTERISTICS
RoG                      scorer=False adaptive_budget=False random=False
Fixed Top-B              scorer=True  adaptive_budget=False random=False
Fixed Threshold          scorer=True  adaptive_budget=False random=False
Random-B                 scorer=False adaptive_budget=False random=True 
Adaptive-Budget Random   scorer=True  adaptive_budget=True  random=True 
AFP                

## Restore and Audit Validation Traversal Prerequisites

In [7]:
# ======================================================================
# RECOVER FROZEN VALIDATION PLANS FROM RQ1 ARTIFACTS
# ======================================================================
#
# PURPOSE
# -------
# Recover the EXACT persisted validation planning artifacts generated
# during RQ1 development.
#
# We DO NOT rerun:
#   - RoG planner
#   - MiniLM
#   - traversal
#   - tuning
#
# Instead:
#   1. Locate planning_<dataset>_validation.jsonl
#   2. Inspect its actual schema
#   3. Automatically identify the predicted-plan list field
#   4. Require exact frozen question/plan/empty-plan counts
#   5. Restore the rows + plan-access helper for Cell 12B
#
# NO TEST DATA.
# ======================================================================

import os
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Project roots
# ======================================================================

RQ2_ROOT = Path("/kaggle/working/step3_rq2_dev_v1")

assert RQ2_ROOT.exists(), (
    f"Missing RQ2 root: {RQ2_ROOT}"
)

print("RQ2 root:", RQ2_ROOT)
print("Device:  ", "cuda" if torch.cuda.is_available() else "cpu")


# ======================================================================
# 2. Frozen validation expectations
# ======================================================================

FROZEN_VALIDATION_EXPECTATIONS = {
    "webqsp": {
        "questions": 246,
        "total_plans": 721,
        "empty_plans": 0,
        "nonempty_plans": 721,
    },

    "cwq": {
        "questions": 3519,
        "total_plans": 10536,
        "empty_plans": 7,
        "nonempty_plans": 10529,
    },
}


# ======================================================================
# 3. Locate exact RQ1 planning artifacts
# ======================================================================

def locate_rq1_validation_planning(dataset):
    filename = f"planning_{dataset}_validation.jsonl"

    preferred = [
        Path("/kaggle/working/step2_rq1_dev") / filename,
    ]

    # Known saved-notebook style location.
    input_root = Path("/kaggle/input")

    for p in preferred:
        if p.exists():
            return p

    if input_root.exists():
        matches = list(
            input_root.rglob(filename)
        )

        if matches:
            # Prefer paths containing step2_rq1_dev.
            matches = sorted(
                matches,
                key=lambda p: (
                    "step2_rq1_dev" not in str(p),
                    len(str(p))
                )
            )

            return matches[0]

    raise FileNotFoundError(
        f"Could not locate {filename}. "
        "Do NOT rerun the planner."
    )


WEBQSP_VAL_PLAN_PATH = locate_rq1_validation_planning(
    "webqsp"
)

CWQ_VAL_PLAN_PATH = locate_rq1_validation_planning(
    "cwq"
)

print("\nFrozen RQ1 planning artifacts:")
print("  WebQSP:", WEBQSP_VAL_PLAN_PATH)
print("  CWQ:   ", CWQ_VAL_PLAN_PATH)


# ======================================================================
# 4. Leakage gate
# ======================================================================

def assert_validation_only_path(path):
    text = str(path).lower()

    assert "validation" in text, (
        f"Not a validation artifact: {path}"
    )

    parts = (
        text
        .replace("\\", "/")
        .replace("-", "_")
        .replace(".", "_")
        .split("/")
    )

    for part in parts:
        tokens = part.split("_")

        assert "test" not in tokens, (
            f"TEST-like artifact rejected: {path}"
        )


assert_validation_only_path(
    WEBQSP_VAL_PLAN_PATH
)

assert_validation_only_path(
    CWQ_VAL_PLAN_PATH
)

print("\nValidation-only leakage gate: PASSED")


# ======================================================================
# 5. Load JSONL exactly as persisted
# ======================================================================

def load_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line_no, line in enumerate(
            f,
            start=1
        ):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(
                    json.loads(line)
                )
            except Exception as e:
                raise RuntimeError(
                    f"JSON error in {path}, "
                    f"line {line_no}: {e}"
                )

    return rows


webqsp_val_plan_rows = load_jsonl(
    WEBQSP_VAL_PLAN_PATH
)

cwq_val_plan_rows = load_jsonl(
    CWQ_VAL_PLAN_PATH
)

print("\nRaw JSONL rows:")
print("  WebQSP:", len(webqsp_val_plan_rows))
print("  CWQ:   ", len(cwq_val_plan_rows))


# ======================================================================
# 6. Question-count fidelity gate
# ======================================================================

assert len(webqsp_val_plan_rows) == (
    FROZEN_VALIDATION_EXPECTATIONS[
        "webqsp"
    ][
        "questions"
    ]
)

assert len(cwq_val_plan_rows) == (
    FROZEN_VALIDATION_EXPECTATIONS[
        "cwq"
    ][
        "questions"
    ]
)

print("Frozen question-count gate: PASSED")


# ======================================================================
# 7. Show actual planning-row schema
# ======================================================================

def show_row_schema(dataset, rows):
    row = rows[0]

    print("\n" + "=" * 88)
    print(f"{dataset.upper()} FIRST PLANNING ROW SCHEMA")
    print("=" * 88)

    print("Top-level keys:")

    for key in row.keys():
        value = row[key]

        if isinstance(value, list):
            desc = f"list[{len(value)}]"

            if value:
                desc += (
                    f" -> {type(value[0]).__name__}"
                )

        elif isinstance(value, dict):
            desc = (
                "dict keys="
                + str(
                    list(value.keys())[:15]
                )
            )

        else:
            text = str(value)

            if len(text) > 100:
                text = text[:97] + "..."

            desc = (
                f"{type(value).__name__}: "
                f"{text}"
            )

        print(
            f"  {key:<30} {desc}"
        )


show_row_schema(
    "webqsp",
    webqsp_val_plan_rows
)

show_row_schema(
    "cwq",
    cwq_val_plan_rows
)


# ======================================================================
# 8. Enumerate list-valued dictionary field paths
# ======================================================================
#
# We inspect dictionary structure but do NOT descend into lists.
# This finds candidates such as:
#
#   ("predicted_paths",)
#   ("planning", "paths")
#   ("prediction", "relation_paths")
#
# ======================================================================

def enumerate_list_paths(
    obj,
    prefix=()
):
    paths = []

    if not isinstance(
        obj,
        dict
    ):
        return paths

    for key, value in obj.items():
        current = (
            *prefix,
            key
        )

        if isinstance(
            value,
            list
        ):
            paths.append(
                current
            )

        elif isinstance(
            value,
            dict
        ):
            paths.extend(
                enumerate_list_paths(
                    value,
                    current
                )
            )

    return paths


def get_nested_value(
    row,
    path
):
    value = row

    for key in path:
        if not isinstance(
            value,
            dict
        ):
            return None

        if key not in value:
            return None

        value = value[key]

    return value


def path_to_string(path):
    return ".".join(
        str(x)
        for x in path
    )


# ======================================================================
# 9. Empty-plan detector
# ======================================================================

def plan_is_empty(plan):
    if plan is None:
        return True

    if isinstance(
        plan,
        str
    ):
        text = plan.strip().lower()

        return text in {
            "",
            "[]",
            "()",
            "{}",
            "none",
            "null",
        }

    if isinstance(
        plan,
        (
            list,
            tuple,
            dict,
            set,
        )
    ):
        return len(plan) == 0

    return False


# ======================================================================
# 10. Evaluate every possible list field
# ======================================================================

SEMANTIC_PLAN_TOKENS = [
    "plan",
    "path",
    "relation",
    "prediction",
    "predict",
    "beam",
]


def evaluate_list_paths(
    dataset,
    rows
):
    expected = (
        FROZEN_VALIDATION_EXPECTATIONS[
            dataset
        ]
    )

    discovered_paths = set()

    # Inspect enough rows to catch optional nested structures.
    for row in rows[:min(
        100,
        len(rows)
    )]:
        for path in enumerate_list_paths(
            row
        ):
            discovered_paths.add(
                path
            )

    evaluations = []

    for path in sorted(
        discovered_paths
    ):
        values = []

        valid = True

        for row in rows:
            value = get_nested_value(
                row,
                path
            )

            if not isinstance(
                value,
                list
            ):
                valid = False
                break

            values.append(
                value
            )

        if not valid:
            continue

        total_items = sum(
            len(v)
            for v in values
        )

        empty_items = sum(
            int(
                plan_is_empty(item)
            )
            for value in values
            for item in value
        )

        path_text = (
            path_to_string(
                path
            )
            .lower()
        )

        semantic_score = sum(
            token in path_text
            for token
            in SEMANTIC_PLAN_TOKENS
        )

        exact_counts = (
            total_items
            == expected[
                "total_plans"
            ]
            and
            empty_items
            == expected[
                "empty_plans"
            ]
        )

        evaluations.append(
            {
                "path": path,
                "path_text":
                    path_to_string(
                        path
                    ),
                "total_items":
                    total_items,
                "empty_items":
                    empty_items,
                "semantic_score":
                    semantic_score,
                "exact_counts":
                    exact_counts,
            }
        )

    evaluations.sort(
        key=lambda x: (
            not x[
                "exact_counts"
            ],
            -x[
                "semantic_score"
            ],
            abs(
                x["total_items"]
                - expected["total_plans"]
            ),
            x["path_text"],
        )
    )

    return evaluations


webqsp_path_evaluations = (
    evaluate_list_paths(
        "webqsp",
        webqsp_val_plan_rows
    )
)

cwq_path_evaluations = (
    evaluate_list_paths(
        "cwq",
        cwq_val_plan_rows
    )
)


# ======================================================================
# 11. Print discovered field candidates
# ======================================================================

def print_path_candidates(
    dataset,
    evaluations,
    limit=15
):
    print("\n" + "=" * 96)
    print(
        f"{dataset.upper()} LIST-FIELD CANDIDATES"
    )
    print("=" * 96)

    print(
        f"{'field':<45}"
        f"{'items':>10}"
        f"{'empty':>10}"
        f"{'semantic':>11}"
        f"{'exact':>8}"
    )

    for item in evaluations[
        :limit
    ]:
        print(
            f"{item['path_text']:<45}"
            f"{item['total_items']:>10}"
            f"{item['empty_items']:>10}"
            f"{item['semantic_score']:>11}"
            f"{str(item['exact_counts']):>8}"
        )


print_path_candidates(
    "webqsp",
    webqsp_path_evaluations
)

print_path_candidates(
    "cwq",
    cwq_path_evaluations
)


# ======================================================================
# 12. Select exact frozen plan field
# ======================================================================

def select_exact_plan_path(
    dataset,
    evaluations
):
    exact = [
        x
        for x in evaluations
        if x["exact_counts"]
    ]

    assert len(exact) > 0, (
        f"{dataset.upper()}: no list field matches "
        "the frozen validation-plan counts."
    )

    # Require semantic relation to plans/predictions where possible.
    semantic_exact = [
        x
        for x in exact
        if x[
            "semantic_score"
        ] > 0
    ]

    if semantic_exact:
        exact = semantic_exact

    # Highest semantic score first.
    exact = sorted(
        exact,
        key=lambda x: (
            -x[
                "semantic_score"
            ],
            x[
                "path_text"
            ],
        )
    )

    # If multiple exact candidates remain, report them.
    if len(exact) > 1:
        print(
            f"\n{dataset.upper()}: "
            "multiple exact-count fields found:"
        )

        for x in exact:
            print(
                " ",
                x[
                    "path_text"
                ]
            )

        print(
            "Using highest-ranked semantic candidate:",
            exact[0][
                "path_text"
            ]
        )

    return exact[0]


webqsp_selected_plan_field = (
    select_exact_plan_path(
        "webqsp",
        webqsp_path_evaluations
    )
)

cwq_selected_plan_field = (
    select_exact_plan_path(
        "cwq",
        cwq_path_evaluations
    )
)

WEBQSP_VAL_PLAN_FIELD_PATH = (
    webqsp_selected_plan_field[
        "path"
    ]
)

CWQ_VAL_PLAN_FIELD_PATH = (
    cwq_selected_plan_field[
        "path"
    ]
)

print("\nSelected frozen plan fields:")
print(
    "  WebQSP:",
    path_to_string(
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

print(
    "  CWQ:   ",
    path_to_string(
        CWQ_VAL_PLAN_FIELD_PATH
    )
)


# ======================================================================
# 13. Canonical frozen-plan accessor
# ======================================================================

def get_frozen_relation_plans(
    row,
    field_path
):
    plans = get_nested_value(
        row,
        field_path
    )

    assert isinstance(
        plans,
        list
    )

    return plans


# ======================================================================
# 14. Exact plan-count fidelity audit
# ======================================================================

def audit_restored_plans(
    dataset,
    rows,
    field_path
):
    expected = (
        FROZEN_VALIDATION_EXPECTATIONS[
            dataset
        ]
    )

    plan_lists = [
        get_frozen_relation_plans(
            row,
            field_path
        )
        for row in rows
    ]

    total_plans = sum(
        len(plans)
        for plans in plan_lists
    )

    empty_plans = sum(
        int(
            plan_is_empty(plan)
        )
        for plans in plan_lists
        for plan in plans
    )

    nonempty_plans = (
        total_plans
        - empty_plans
    )

    result = {
        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "nonempty_plans":
            nonempty_plans,
    }

    assert result == {
        "questions":
            expected[
                "questions"
            ],

        "total_plans":
            expected[
                "total_plans"
            ],

        "empty_plans":
            expected[
                "empty_plans"
            ],

        "nonempty_plans":
            expected[
                "nonempty_plans"
            ],
    }, (
        f"{dataset.upper()} frozen-plan "
        f"fidelity failure:\n"
        f"observed={result}\n"
        f"expected={expected}"
    )

    return result


webqsp_val_plan_audit = (
    audit_restored_plans(
        "webqsp",
        webqsp_val_plan_rows,
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

cwq_val_plan_audit = (
    audit_restored_plans(
        "cwq",
        cwq_val_plan_rows,
        CWQ_VAL_PLAN_FIELD_PATH
    )
)

print(
    "\nExact frozen validation-plan "
    "count gate: PASSED"
)


# ======================================================================
# 15. Fingerprint ORIGINAL persisted JSONL files
# ======================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


WEBQSP_VAL_PLAN_FILE_SHA256 = (
    sha256_file(
        WEBQSP_VAL_PLAN_PATH
    )
)

CWQ_VAL_PLAN_FILE_SHA256 = (
    sha256_file(
        CWQ_VAL_PLAN_PATH
    )
)

print("\nPersisted artifact SHA256:")
print(
    "  WebQSP:",
    WEBQSP_VAL_PLAN_FILE_SHA256
)

print(
    "  CWQ:   ",
    CWQ_VAL_PLAN_FILE_SHA256
)


# ======================================================================
# 16. Inspect a representative plan from each dataset
# ======================================================================

def first_nonempty_plan(
    rows,
    field_path
):
    for row_index, row in enumerate(
        rows
    ):
        plans = get_frozen_relation_plans(
            row,
            field_path
        )

        for plan_index, plan in enumerate(
            plans
        ):
            if not plan_is_empty(
                plan
            ):
                return (
                    row_index,
                    plan_index,
                    plan,
                )

    return None


webqsp_sample_plan = (
    first_nonempty_plan(
        webqsp_val_plan_rows,
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

cwq_sample_plan = (
    first_nonempty_plan(
        cwq_val_plan_rows,
        CWQ_VAL_PLAN_FIELD_PATH
    )
)

print("\nRepresentative frozen plan objects:")

print(
    "  WebQSP:",
    webqsp_sample_plan
)

print(
    "  CWQ:   ",
    cwq_sample_plan
)


# ======================================================================
# 17. Verify selected scorer manifests still exist
# ======================================================================

FINAL_SCORER_DIR = (
    RQ2_ROOT
    / "07_final_scorer"
)

SCORER_SELECTION_MANIFEST = (
    FINAL_SCORER_DIR
    / "afp_scorer_selection_manifest.json"
)

SELECTOR_MANIFEST = (
    RQ2_ROOT
    / "08_adaptive_selector"
    / "adaptive_selector_definition_manifest.json"
)

BASELINE_MANIFEST = (
    RQ2_ROOT
    / "09_controlled_baselines"
    / "controlled_baseline_definition_manifest.json"
)

assert SCORER_SELECTION_MANIFEST.exists()
assert SELECTOR_MANIFEST.exists()
assert BASELINE_MANIFEST.exists()

with open(
    SCORER_SELECTION_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    scorer_selection_manifest_12a = (
        json.load(f)
    )

with open(
    SELECTOR_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    selector_manifest_12a = (
        json.load(f)
    )

with open(
    BASELINE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    baseline_manifest_12a = (
        json.load(f)
    )

assert (
    scorer_selection_manifest_12a[
        "test_used_for_selection"
    ]
    is False
)

assert (
    scorer_selection_manifest_12a[
        "test_metrics_observed"
    ]
    is False
)

assert (
    selector_manifest_12a[
        "test_used"
    ]
    is False
)

assert (
    baseline_manifest_12a[
        "test_used"
    ]
    is False
)

print(
    "\nScorer/selector/baseline "
    "artifact gates: PASSED"
)


# ======================================================================
# 18. Verify Cell 10/11 policy functions
# ======================================================================

required_policy_functions = [
    "afp_select_from_logits",
    "afp_select_frontier",
    "controlled_select_frontier",
    "select_rog",
    "select_fixed_top_b",
    "select_fixed_threshold",
    "select_random_b",
    "select_adaptive_budget_random",
    "select_afp_policy",
]

missing = [
    name
    for name in required_policy_functions
    if name not in globals()
    or not callable(
        globals()[name]
    )
]

assert not missing, (
    "Missing Cell 10/11 functions: "
    + str(missing)
)

print(
    "Cell 10/11 policy functions: PASSED"
)


# ======================================================================
# 19. Discover likely validation/KG artifacts for Cell 12B
# ======================================================================

def discover_dataset_artifacts(
    dataset,
    limit=30
):
    aliases = {
        "webqsp": [
            "webqsp",
            "web_qsp",
        ],

        "cwq": [
            "cwq",
            "complexwebquestions",
            "complex_web_questions",
        ],
    }[
        dataset
    ]

    roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    rows = []

    for root in roots:
        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d != "__pycache__"
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                text = str(
                    path
                ).lower()

                if not any(
                    alias in text
                    for alias in aliases
                ):
                    continue

                # No test artifacts.
                parts = (
                    text
                    .replace("\\", "/")
                    .replace("-", "_")
                    .replace(".", "_")
                    .split("/")
                )

                if any(
                    "test" in part.split("_")
                    for part in parts
                ):
                    continue

                tags = []

                for token in [
                    "validation",
                    "val",
                    "graph",
                    "subgraph",
                    "kg",
                    "adj",
                    "profile",
                    "question",
                    "entity",
                    "relation",
                    "plan",
                ]:
                    if token in text:
                        tags.append(
                            token
                        )

                if not tags:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )
                except Exception:
                    size_mb = np.nan

                score = (
                    10 * int(
                        "graph" in tags
                        or "subgraph" in tags
                        or "kg" in tags
                    )
                    +
                    6 * int(
                        "validation" in tags
                        or "val" in tags
                    )
                    +
                    4 * int(
                        "profile" in tags
                    )
                )

                rows.append(
                    {
                        "path":
                            str(path),

                        "size_mb":
                            size_mb,

                        "tags":
                            ",".join(tags),

                        "score":
                            score,
                    }
                )

    df = pd.DataFrame(
        rows
    )

    if len(df) == 0:
        return df

    df = (
        df
        .drop_duplicates(
            subset=[
                "path"
            ]
        )
        .sort_values(
            [
                "score",
                "path",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .head(limit)
        .reset_index(
            drop=True
        )
    )

    return df


webqsp_artifact_inventory = (
    discover_dataset_artifacts(
        "webqsp"
    )
)

cwq_artifact_inventory = (
    discover_dataset_artifacts(
        "cwq"
    )
)


def print_inventory(
    dataset,
    df
):
    print("\n" + "=" * 100)
    print(
        f"{dataset.upper()} "
        "LIKELY VALIDATION / KG ARTIFACTS"
    )
    print("=" * 100)

    if len(df) == 0:
        print("NONE FOUND")
        return

    for _, row in df.iterrows():
        size = row[
            "size_mb"
        ]

        size_text = (
            f"{size:.2f} MB"
            if np.isfinite(size)
            else "?"
        )

        print(
            f"{size_text:>10}  "
            f"[{row['tags']}]  "
            f"{row['path']}"
        )


print_inventory(
    "webqsp",
    webqsp_artifact_inventory
)

print_inventory(
    "cwq",
    cwq_artifact_inventory
)


# ======================================================================
# 20. Inspect surviving traversal/data globals
# ======================================================================

TRAVERSAL_KEYWORDS = [
    "travers",
    "expand",
    "match",
    "candidate",
    "graph",
    "neighbor",
    "adj",
    "profile",
    "feature",
]

existing_traversal_callables = sorted(
    name
    for name, obj
    in globals().items()
    if callable(obj)
    and any(
        token in name.lower()
        for token
        in TRAVERSAL_KEYWORDS
    )
)

DATA_GLOBAL_KEYWORDS = [
    "webqsp",
    "cwq",
    "graph",
    "dataset",
    "validation",
    "val_data",
]

existing_data_globals = sorted(
    name
    for name, obj
    in globals().items()
    if not callable(obj)
    and any(
        token in name.lower()
        for token
        in DATA_GLOBAL_KEYWORDS
    )
    and not name.startswith("_")
)

print(
    "\nExisting traversal-related callables:"
)

print(
    existing_traversal_callables[:50]
    if existing_traversal_callables
    else "NONE"
)

print(
    "\nExisting data-related globals:"
)

print(
    existing_data_globals[:80]
    if existing_data_globals
    else "NONE"
)


# ======================================================================
# 21. Save recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12A_RECOVERY_MANIFEST = {
    "cell":
        "RQ2_12A_R",

    "webqsp": {
        "artifact":
            str(
                WEBQSP_VAL_PLAN_PATH
            ),

        "artifact_sha256":
            WEBQSP_VAL_PLAN_FILE_SHA256,

        "plan_field":
            path_to_string(
                WEBQSP_VAL_PLAN_FIELD_PATH
            ),

        **webqsp_val_plan_audit,
    },

    "cwq": {
        "artifact":
            str(
                CWQ_VAL_PLAN_PATH
            ),

        "artifact_sha256":
            CWQ_VAL_PLAN_FILE_SHA256,

        "plan_field":
            path_to_string(
                CWQ_VAL_PLAN_FIELD_PATH
            ),

        **cwq_val_plan_audit,
    },

    "planner_rerun":
        False,

    "semantic_encoder_rerun":
        False,

    "validation_traversal_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "rog_fidelity_checked":
        False,
}

CELL12A_RECOVERY_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12a_frozen_plan_recovery.json"
)

with open(
    CELL12A_RECOVERY_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        CELL12A_RECOVERY_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 22. Final report
# ======================================================================

print("\n" + "=" * 98)
print(
    "=== RQ2 CELL 12A-R: FROZEN VALIDATION PLAN RECOVERY COMPLETE ==="
)
print("=" * 98)

print("\nWEBQSP")
print(
    "  artifact:       ",
    WEBQSP_VAL_PLAN_PATH
)
print(
    "  plan field:     ",
    path_to_string(
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)
print(
    "  questions:      ",
    webqsp_val_plan_audit[
        "questions"
    ]
)
print(
    "  total plans:    ",
    webqsp_val_plan_audit[
        "total_plans"
    ]
)
print(
    "  empty plans:    ",
    webqsp_val_plan_audit[
        "empty_plans"
    ]
)
print(
    "  executable:     ",
    webqsp_val_plan_audit[
        "nonempty_plans"
    ]
)

print("\nCWQ")
print(
    "  artifact:       ",
    CWQ_VAL_PLAN_PATH
)
print(
    "  plan field:     ",
    path_to_string(
        CWQ_VAL_PLAN_FIELD_PATH
    )
)
print(
    "  questions:      ",
    cwq_val_plan_audit[
        "questions"
    ]
)
print(
    "  total plans:    ",
    cwq_val_plan_audit[
        "total_plans"
    ]
)
print(
    "  empty plans:    ",
    cwq_val_plan_audit[
        "empty_plans"
    ]
)
print(
    "  executable:     ",
    cwq_val_plan_audit[
        "nonempty_plans"
    ]
)

print("\nIntegrity")
print(
    "  Exact frozen plan counts:       PASSED"
)
print(
    "  Persisted artifact fingerprint: SAVED"
)
print(
    "  Scorer artifacts:               PASSED"
)
print(
    "  Selector artifacts:             PASSED"
)
print(
    "  Baseline artifacts:             PASSED"
)
print(
    "  Cell 10/11 functions:           PASSED"
)

print("\nExecution status")
print(
    "  Planner rerun:                  NO"
)
print(
    "  MiniLM rerun:                   NO"
)
print(
    "  Validation traversal run:       NO"
)
print(
    "  Hyperparameter tuning:          NO"
)
print(
    "  TEST data/gold loaded:          NO"
)
print(
    "  RoG fidelity checked:           NO"
)

print(
    "\nRecovery manifest:",
    CELL12A_RECOVERY_MANIFEST_PATH
)

print(
    "\nNext: Cell 12B — shared validation traversal "
    "+ RoG fidelity gate."
)

RQ2 root: /kaggle/working/step3_rq2_dev_v1
Device:   cpu

Frozen RQ1 planning artifacts:
  WebQSP: /kaggle/working/step2_rq1_dev/planning_webqsp_validation.jsonl
  CWQ:    /kaggle/working/step2_rq1_dev/planning_cwq_validation.jsonl

Validation-only leakage gate: PASSED

Raw JSONL rows:
  WebQSP: 246
  CWQ:    3519
Frozen question-count gate: PASSED

WEBQSP FIRST PLANNING ROW SCHEMA
Top-level keys:
  id                             str: WebQTrn-9
  question                       str: how old is sacha baron cohen
  q_entity                       list[1] -> str
  a_entity                       list[1] -> str
  graph                          list[6293] -> list
  predicted_paths                list[3] -> list
  planning_time_sec              float: 1.4861440658569336

CWQ FIRST PLANNING ROW SCHEMA
Top-level keys:
  id                             str: WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  question                       str: Who was the president in 1980 of the country that has Azad 

In [9]:
# ======================================================================
# EXACT RoG GRAPH SEMANTICS + SHARED TRAVERSAL FIDELITY
# ======================================================================
#
# FIX:
# Official RoG uses:
#
#   G = nx.Graph()
#   G.add_edge(h, t, relation=r.strip())
#
# Therefore:
#   - graph is UNDIRECTED
#   - only UNIQUE entity-pair neighbors exist
#   - repeated (h,t) edges overwrite relation attribute
#   - neighbor order follows first insertion
#
# This cell reproduces those semantics WITHOUT rerunning RoG planner.
#
# NO tuning.
# NO MiniLM.
# NO TEST.
# ======================================================================

import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

assert len(webqsp_val_plan_rows) == 246
assert len(cwq_val_plan_rows) == 3519

assert "controlled_select_frontier" in globals()
assert callable(controlled_select_frontier)

assert "get_frozen_relation_plans" in globals()
assert callable(get_frozen_relation_plans)

print("Cell 12A-R prerequisites: PASSED")


# ======================================================================
# 2. Frozen RQ1 references
# ======================================================================

FROZEN_RQ1_CORE = {
    "webqsp": {
        "active_hop_rows": 971,
        "edges_examined": 341526,
        "candidate_branches": 7983,
    },

    "cwq": {
        "active_hop_rows": 16564,
        "edges_examined": 5257272,
        "candidate_branches": 247161,
    },
}

FROZEN_RQ1_REACHABILITY = {
    "webqsp": {
        "reachable_plans": 345,
        "reachable_questions": 205,
    },

    "cwq": {
        "reachable_plans": 3971,
        "reachable_questions": 2425,
    },
}

print("Frozen RQ1 references: LOADED")


# ======================================================================
# 3. Exact RoG graph construction
# ======================================================================
#
# Equivalent to:
#
#   G = nx.Graph()
#   for h, r, t in graph:
#       G.add_edge(h, t, relation=r.strip())
#
# Dict behavior is intentional:
#
#   - first appearance fixes neighbor insertion order
#   - later same-pair edges replace relation
#   - no parallel edges
#
# ======================================================================

def build_rog_adjacency(graph):
    adjacency = defaultdict(dict)

    for triple in graph:
        if (
            not isinstance(triple, (list, tuple))
            or len(triple) != 3
        ):
            continue

        h, r, t = triple

        h = str(h)
        t = str(t)
        r = str(r).strip()

        # ----------------------------------------------------------
        # NetworkX Graph.add_edge semantics:
        # same undirected pair => relation attr is overwritten.
        # Existing dictionary key keeps its insertion position.
        # ----------------------------------------------------------
        adjacency[h][t] = r
        adjacency[t][h] = r

    return adjacency


# ======================================================================
# 4. Initial prefix + extension helpers
# ======================================================================

def make_rog_initial_prefixes(topic_entities):
    return [
        {
            "entities": (str(entity),),
            "relations": tuple(),
        }
        for entity in topic_entities
    ]


def extend_rog_prefix(
    prefix,
    relation,
    next_entity
):
    return {
        "entities":
            prefix["entities"]
            + (str(next_entity),),

        "relations":
            prefix["relations"]
            + (str(relation),),
    }


# ======================================================================
# 5. Exact RoG candidate generation
# ======================================================================
#
# Equivalent to:
#
#   for neighbor in graph.neighbors(current_node):
#       rel = graph[current_node][neighbor]["relation"]
#       if rel == target_relation:
#           append...
#
# edges_examined = number of UNIQUE neighbors inspected.
# ======================================================================

def generate_rog_candidates(
    prefixes,
    target_relation,
    adjacency
):
    target_relation = str(
        target_relation
    ).strip()

    candidates = []
    edges_examined = 0

    for prefix in prefixes:
        current_entity = (
            prefix["entities"][-1]
        )

        neighbors = adjacency.get(
            current_entity,
            {}
        )

        # RoG checks every unique graph neighbor.
        edges_examined += len(
            neighbors
        )

        for neighbor, edge_relation in neighbors.items():

            if edge_relation != target_relation:
                continue

            candidates.append(
                extend_rog_prefix(
                    prefix=prefix,
                    relation=target_relation,
                    next_entity=neighbor
                )
            )

    return (
        candidates,
        edges_examined
    )


# ======================================================================
# 6. Exact unpruned RoG plan traversal
# ======================================================================

def traverse_rog_plan(
    row,
    plan,
    question_index,
    plan_index
):
    assert isinstance(plan, list)
    assert len(plan) > 0

    adjacency = build_rog_adjacency(
        row.get("graph", [])
    )

    prefixes = make_rog_initial_prefixes(
        row.get("q_entity", [])
    )

    hop_records = []

    for hop, relation in enumerate(plan):

        if len(prefixes) == 0:
            break

        active_prefixes = len(
            prefixes
        )

        candidates, edges_examined = (
            generate_rog_candidates(
                prefixes=prefixes,
                target_relation=relation,
                adjacency=adjacency
            )
        )

        hop_records.append(
            {
                "question_index":
                    question_index,

                "question_id":
                    str(
                        row.get(
                            "id",
                            question_index
                        )
                    ),

                "plan_index":
                    plan_index,

                "hop":
                    hop,

                "plan_length":
                    len(plan),

                "relation":
                    str(relation),

                "active_prefixes":
                    active_prefixes,

                "edges_examined":
                    edges_examined,

                "candidate_branches":
                    len(candidates),

                "unique_candidate_entities":
                    len({
                        p["entities"][-1]
                        for p in candidates
                    }),
            }
        )

        prefixes = candidates

    return (
        prefixes,
        hop_records
    )


# ======================================================================
# 7. Dataset-level exact RoG traversal
# ======================================================================

def run_exact_rog_validation(
    dataset,
    rows,
    plan_field_path
):
    all_hops = []

    total_plans = 0
    empty_plans = 0
    executable_plans = 0

    reachable_plans = 0
    reachable_questions = set()

    for q_idx, row in enumerate(rows):

        qid = str(
            row.get(
                "id",
                q_idx
            )
        )

        answers = {
            str(x)
            for x in row.get(
                "a_entity",
                []
            )
        }

        plans = get_frozen_relation_plans(
            row,
            plan_field_path
        )

        for plan_idx, plan in enumerate(
            plans
        ):
            total_plans += 1

            if (
                not isinstance(plan, list)
                or len(plan) == 0
            ):
                empty_plans += 1
                continue

            executable_plans += 1

            final_prefixes, hop_records = (
                traverse_rog_plan(
                    row=row,
                    plan=plan,
                    question_index=q_idx,
                    plan_index=plan_idx
                )
            )

            all_hops.extend(
                hop_records
            )

            final_entities = {
                p["entities"][-1]
                for p in final_prefixes
            }

            if (
                len(
                    final_entities
                    & answers
                )
                > 0
            ):
                reachable_plans += 1
                reachable_questions.add(
                    qid
                )

    hop_df = pd.DataFrame(
        all_hops
    )

    return {
        "dataset":
            dataset,

        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "executable_plans":
            executable_plans,

        "active_hop_rows":
            int(len(hop_df)),

        "edges_examined":
            int(
                hop_df[
                    "edges_examined"
                ].sum()
            ),

        "candidate_branches":
            int(
                hop_df[
                    "candidate_branches"
                ].sum()
            ),

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            len(
                reachable_questions
            ),

        "reachable_question_ids":
            sorted(
                reachable_questions
            ),

        "hop_df":
            hop_df,
    }


# ======================================================================
# 8. WebQSP exact RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "WEBQSP — OFFICIAL RoG GRAPH SEMANTICS FIDELITY"
)

print(
    "=" * 94
)

webqsp_exact_rog = (
    run_exact_rog_validation(
        dataset="webqsp",
        rows=webqsp_val_plan_rows,
        plan_field_path=
            WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

print(
    "Observed core:",
    {
        k: webqsp_exact_rog[k]
        for k in [
            "active_hop_rows",
            "edges_examined",
            "candidate_branches",
        ]
    }
)

print(
    "Expected core:",
    FROZEN_RQ1_CORE[
        "webqsp"
    ]
)

for key, expected in (
    FROZEN_RQ1_CORE[
        "webqsp"
    ].items()
):
    assert (
        webqsp_exact_rog[key]
        == expected
    ), (
        f"WEBQSP mismatch: {key} | "
        f"observed={webqsp_exact_rog[key]} "
        f"expected={expected}"
    )

print(
    "WebQSP search-cost fidelity: PASSED"
)

print(
    "Observed reachability:",
    {
        "reachable_plans":
            webqsp_exact_rog[
                "reachable_plans"
            ],

        "reachable_questions":
            webqsp_exact_rog[
                "reachable_questions"
            ],
    }
)

print(
    "Expected reachability:",
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ]
)

assert (
    webqsp_exact_rog[
        "reachable_plans"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ][
        "reachable_plans"
    ]
)

assert (
    webqsp_exact_rog[
        "reachable_questions"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ][
        "reachable_questions"
    ]
)

print(
    "WebQSP reachability fidelity: PASSED"
)


# ======================================================================
# 9. CWQ exact RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "CWQ — OFFICIAL RoG GRAPH SEMANTICS FIDELITY"
)

print(
    "=" * 94
)

cwq_exact_rog = (
    run_exact_rog_validation(
        dataset="cwq",
        rows=cwq_val_plan_rows,
        plan_field_path=
            CWQ_VAL_PLAN_FIELD_PATH
    )
)

print(
    "Observed core:",
    {
        k: cwq_exact_rog[k]
        for k in [
            "active_hop_rows",
            "edges_examined",
            "candidate_branches",
        ]
    }
)

print(
    "Expected core:",
    FROZEN_RQ1_CORE[
        "cwq"
    ]
)

for key, expected in (
    FROZEN_RQ1_CORE[
        "cwq"
    ].items()
):
    assert (
        cwq_exact_rog[key]
        == expected
    ), (
        f"CWQ mismatch: {key} | "
        f"observed={cwq_exact_rog[key]} "
        f"expected={expected}"
    )

print(
    "CWQ search-cost fidelity: PASSED"
)

print(
    "Observed reachability:",
    {
        "reachable_plans":
            cwq_exact_rog[
                "reachable_plans"
            ],

        "reachable_questions":
            cwq_exact_rog[
                "reachable_questions"
            ],
    }
)

print(
    "Expected reachability:",
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ]
)

assert (
    cwq_exact_rog[
        "reachable_plans"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ][
        "reachable_plans"
    ]
)

assert (
    cwq_exact_rog[
        "reachable_questions"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ][
        "reachable_questions"
    ]
)

print(
    "CWQ reachability fidelity: PASSED"
)


# ======================================================================
# 10. Plan-count gates
# ======================================================================

assert (
    webqsp_exact_rog[
        "total_plans"
    ] == 721
)

assert (
    webqsp_exact_rog[
        "empty_plans"
    ] == 0
)

assert (
    webqsp_exact_rog[
        "executable_plans"
    ] == 721
)

assert (
    cwq_exact_rog[
        "total_plans"
    ] == 10536
)

assert (
    cwq_exact_rog[
        "empty_plans"
    ] == 7
)

assert (
    cwq_exact_rog[
        "executable_plans"
    ] == 10529
)

print(
    "\nPlan-count fidelity: PASSED"
)


# ======================================================================
# 11. Shared controlled traversal
# ======================================================================
#
# THIS is the single engine used later by every pruning method.
#
# Only:
#
#     controlled_select_frontier(...)
#
# changes between methods.
#
# Scorer callback will be connected in Cell 12C.
# ======================================================================

SCORE_REQUIRED_METHODS = {
    "fixed_top_b",
    "fixed_threshold",
    "adaptive_budget_random",
    "afp",
}


def run_shared_validation_traversal(
    dataset,
    rows,
    plan_field_path,
    method,

    scorer_callback=None,

    B=None,
    threshold=None,

    temperature=None,
    gamma_min=None,

    seed=None,

    tie_tolerance=1e-8,

    collect_hop_records=True
):
    assert method in CONTROLLED_METHODS

    total_edges_examined = 0
    total_candidate_branches = 0
    total_retained_branches = 0
    total_pruned_branches = 0

    active_hop_rows = 0
    decision_hops = 0

    scorer_invocations = 0

    final_hop_protections = 0
    singleton_bypasses = 0
    tied_abstentions = 0
    cutoff_tie_expansions = 0
    empty_after_selection = 0

    total_plans = 0
    empty_plans = 0
    executable_plans = 0

    reachable_plans = 0
    reachable_question_ids = set()

    question_edge_counts = defaultdict(
        int
    )

    hop_records = []

    for q_idx, row in enumerate(
        rows
    ):
        qid = str(
            row.get(
                "id",
                q_idx
            )
        )

        answers = {
            str(x)
            for x in row.get(
                "a_entity",
                []
            )
        }

        adjacency = build_rog_adjacency(
            row.get(
                "graph",
                []
            )
        )

        plans = get_frozen_relation_plans(
            row,
            plan_field_path
        )

        for plan_idx, plan in enumerate(
            plans
        ):
            total_plans += 1

            if (
                not isinstance(plan, list)
                or len(plan) == 0
            ):
                empty_plans += 1
                continue

            executable_plans += 1

            prefixes = make_rog_initial_prefixes(
                row.get(
                    "q_entity",
                    []
                )
            )

            for hop, relation in enumerate(
                plan
            ):
                if len(prefixes) == 0:
                    break

                active_hop_rows += 1

                active_prefix_count = len(
                    prefixes
                )

                candidates, current_edges = (
                    generate_rog_candidates(
                        prefixes=prefixes,
                        target_relation=relation,
                        adjacency=adjacency
                    )
                )

                candidate_count = len(
                    candidates
                )

                total_edges_examined += (
                    current_edges
                )

                total_candidate_branches += (
                    candidate_count
                )

                question_edge_counts[
                    qid
                ] += current_edges

                # --------------------------------------------------
                # No relation-valid candidate.
                # --------------------------------------------------
                if candidate_count == 0:

                    if collect_hop_records:
                        hop_records.append(
                            {
                                "dataset":
                                    dataset,

                                "method":
                                    method,

                                "question_id":
                                    qid,

                                "question_index":
                                    q_idx,

                                "plan_index":
                                    plan_idx,

                                "hop":
                                    hop,

                                "plan_length":
                                    len(plan),

                                "relation":
                                    str(relation),

                                "active_prefixes":
                                    active_prefix_count,

                                "edges_examined":
                                    current_edges,

                                "candidate_branches":
                                    0,

                                "retained_branches":
                                    0,

                                "pruned_branches":
                                    0,

                                "selection_reason":
                                    "no_relation_match",
                            }
                        )

                    prefixes = []
                    break

                is_final_hop = (
                    hop
                    == len(plan) - 1
                )

                if (
                    not is_final_hop
                    and candidate_count > 1
                ):
                    decision_hops += 1

                # --------------------------------------------------
                # Compute logits only when this method/hop needs them.
                # --------------------------------------------------
                logits = None

                needs_scores = (
                    method
                    in SCORE_REQUIRED_METHODS
                    and
                    not is_final_hop
                    and
                    candidate_count > 1
                )

                if needs_scores:
                    assert (
                        scorer_callback
                        is not None
                    ), (
                        f"{method} requires scorer_callback."
                    )

                    logits = scorer_callback(
                        dataset=dataset,
                        row=row,
                        plan=plan,
                        plan_index=plan_idx,
                        hop=hop,
                        prefixes=prefixes,
                        candidates=candidates,
                        relation=str(
                            relation
                        ).strip(),
                        adjacency=adjacency,
                    )

                    logits = np.asarray(
                        logits,
                        dtype=np.float64
                    )

                    assert logits.shape == (
                        candidate_count,
                    )

                    assert np.all(
                        np.isfinite(
                            logits
                        )
                    )

                group_key = (
                    f"{dataset}|"
                    f"{qid}|"
                    f"plan={plan_idx}|"
                    f"hop={hop}"
                )

                selection = (
                    controlled_select_frontier(
                        method=method,

                        candidate_count=
                            candidate_count,

                        hop=hop,

                        plan_length=
                            len(plan),

                        logits=logits,

                        B=B,

                        threshold=
                            threshold,

                        temperature=
                            temperature,

                        gamma_min=
                            gamma_min,

                        seed=seed,

                        group_key=
                            group_key,

                        tie_tolerance=
                            tie_tolerance
                    )
                )

                selected_indices = np.asarray(
                    selection[
                        "selected_indices"
                    ],
                    dtype=np.int64
                )

                retained_count = len(
                    selected_indices
                )

                pruned_count = (
                    candidate_count
                    - retained_count
                )

                assert (
                    retained_count
                    ==
                    selection[
                        "retained_B"
                    ]
                )

                assert (
                    pruned_count
                    ==
                    selection[
                        "pruned_count"
                    ]
                )

                total_retained_branches += (
                    retained_count
                )

                total_pruned_branches += (
                    pruned_count
                )

                if selection.get(
                    "scorer_invoked",
                    False
                ):
                    scorer_invocations += 1

                reason = selection.get(
                    "reason",
                    ""
                )

                if (
                    reason
                    == "final_hop_protection"
                ):
                    final_hop_protections += 1

                if (
                    reason
                    == "singleton_bypass"
                ):
                    singleton_bypasses += 1

                if (
                    reason
                    == "all_scores_tied_retain_all"
                ):
                    tied_abstentions += 1

                if "tie_expansion" in reason:
                    cutoff_tie_expansions += 1

                if retained_count == 0:
                    empty_after_selection += 1
                    prefixes = []

                else:
                    prefixes = [
                        candidates[
                            int(i)
                        ]
                        for i
                        in selected_indices
                    ]

                if collect_hop_records:
                    hop_records.append(
                        {
                            "dataset":
                                dataset,

                            "method":
                                method,

                            "question_id":
                                qid,

                            "question_index":
                                q_idx,

                            "plan_index":
                                plan_idx,

                            "hop":
                                hop,

                            "plan_length":
                                len(plan),

                            "relation":
                                str(
                                    relation
                                ).strip(),

                            "active_prefixes":
                                active_prefix_count,

                            "edges_examined":
                                current_edges,

                            "candidate_branches":
                                candidate_count,

                            "retained_branches":
                                retained_count,

                            "pruned_branches":
                                pruned_count,

                            "selection_reason":
                                reason,
                        }
                    )

            # ------------------------------------------------------
            # Plan-level answer reachability
            # ------------------------------------------------------
            if len(prefixes) > 0:

                final_entities = {
                    p["entities"][-1]
                    for p in prefixes
                }

                if (
                    len(
                        final_entities
                        & answers
                    )
                    > 0
                ):
                    reachable_plans += 1

                    reachable_question_ids.add(
                        qid
                    )

    hop_df = (
        pd.DataFrame(
            hop_records
        )
        if collect_hop_records
        else None
    )

    return {
        "dataset":
            dataset,

        "method":
            method,

        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "executable_plans":
            executable_plans,

        "active_hop_rows":
            active_hop_rows,

        "decision_hops":
            decision_hops,

        "edges_examined":
            total_edges_examined,

        "candidate_branches":
            total_candidate_branches,

        "retained_branches":
            total_retained_branches,

        "pruned_branches":
            total_pruned_branches,

        "scorer_invocations":
            scorer_invocations,

        "final_hop_protections":
            final_hop_protections,

        "singleton_bypasses":
            singleton_bypasses,

        "tied_abstentions":
            tied_abstentions,

        "cutoff_tie_expansions":
            cutoff_tie_expansions,

        "empty_after_selection":
            empty_after_selection,

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            len(
                reachable_question_ids
            ),

        "reachable_question_ids":
            sorted(
                reachable_question_ids
            ),

        "question_edge_counts":
            dict(
                question_edge_counts
            ),

        "hop_df":
            hop_df,
    }


# ======================================================================
# 12. Shared-engine RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "SHARED ENGINE — RoG FIDELITY"
)

print(
    "=" * 94
)

webqsp_shared_rog = (
    run_shared_validation_traversal(
        dataset="webqsp",
        rows=webqsp_val_plan_rows,
        plan_field_path=
            WEBQSP_VAL_PLAN_FIELD_PATH,
        method="rog",
        scorer_callback=None
    )
)

cwq_shared_rog = (
    run_shared_validation_traversal(
        dataset="cwq",
        rows=cwq_val_plan_rows,
        plan_field_path=
            CWQ_VAL_PLAN_FIELD_PATH,
        method="rog",
        scorer_callback=None
    )
)


def assert_rog_fidelity(
    dataset,
    result
):
    core = FROZEN_RQ1_CORE[
        dataset
    ]

    reach = FROZEN_RQ1_REACHABILITY[
        dataset
    ]

    for key, expected in core.items():

        assert (
            result[key]
            == expected
        ), (
            f"{dataset.upper()} "
            f"shared-engine mismatch {key}: "
            f"{result[key]} != {expected}"
        )

    assert (
        result[
            "reachable_plans"
        ]
        ==
        reach[
            "reachable_plans"
        ]
    )

    assert (
        result[
            "reachable_questions"
        ]
        ==
        reach[
            "reachable_questions"
        ]
    )

    assert (
        result[
            "pruned_branches"
        ] == 0
    )

    assert (
        result[
            "scorer_invocations"
        ] == 0
    )

    print(
        f"{dataset.upper()} shared "
        "RoG fidelity: PASSED"
    )


assert_rog_fidelity(
    "webqsp",
    webqsp_shared_rog
)

assert_rog_fidelity(
    "cwq",
    cwq_shared_rog
)


# ======================================================================
# 13. Save fidelity artifacts
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

webqsp_shared_rog[
    "hop_df"
].to_csv(
    TRAVERSAL_DIR
    / "webqsp_shared_rog_validation_hops.csv",
    index=False
)

cwq_shared_rog[
    "hop_df"
].to_csv(
    TRAVERSAL_DIR
    / "cwq_shared_rog_validation_hops.csv",
    index=False
)


def compact_result(result):
    return {
        k: v
        for k, v
        in result.items()
        if k not in {
            "hop_df",
            "reachable_question_ids",
            "question_edge_counts",
        }
    }


CELL12B_RECOVERY_MANIFEST = {
    "cell":
        "RQ2_12B_R",

    "graph_semantics":
        "official_rog_networkx_graph_equivalent",

    "graph_type":
        "undirected_simple_graph",

    "duplicate_pair_behavior":
        "last_relation_attribute_overwrites",

    "neighbor_behavior":
        "unique_neighbors",

    "relation_matching":
        "exact_stored_relation_match",

    "webqsp":
        compact_result(
            webqsp_shared_rog
        ),

    "cwq":
        compact_result(
            cwq_shared_rog
        ),

    "search_fidelity_passed":
        True,

    "reachability_fidelity_passed":
        True,

    "shared_engine_defined":
        True,

    "planner_rerun":
        False,

    "semantic_encoder_rerun":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "complete_afp_frozen":
        False,
}

CELL12B_RECOVERY_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12b_exact_rog_shared_traversal.json"
)

with open(
    CELL12B_RECOVERY_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12B_RECOVERY_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 14. Final report
# ======================================================================

print(
    "\n"
    + "=" * 98
)

print(
    "=== RQ2 CELL 12B-R: EXACT RoG SHARED TRAVERSAL FIDELITY PASSED ==="
)

print(
    "=" * 98
)

print("\nGraph semantics")
print(
    "  Undirected simple graph:          YES"
)
print(
    "  Unique entity-pair edges:         YES"
)
print(
    "  Duplicate-pair relation overwrite:YES"
)
print(
    "  Exact relation matching:          YES"
)

print("\nWEBQSP")
print(
    "  Active hop rows:       ",
    webqsp_shared_rog[
        "active_hop_rows"
    ]
)
print(
    "  Examined edges:        ",
    webqsp_shared_rog[
        "edges_examined"
    ]
)
print(
    "  Candidate branches:    ",
    webqsp_shared_rog[
        "candidate_branches"
    ]
)
print(
    "  Reachable plans:       ",
    webqsp_shared_rog[
        "reachable_plans"
    ]
)
print(
    "  Reachable questions:   ",
    webqsp_shared_rog[
        "reachable_questions"
    ]
)

print("\nCWQ")
print(
    "  Active hop rows:       ",
    cwq_shared_rog[
        "active_hop_rows"
    ]
)
print(
    "  Examined edges:        ",
    cwq_shared_rog[
        "edges_examined"
    ]
)
print(
    "  Candidate branches:    ",
    cwq_shared_rog[
        "candidate_branches"
    ]
)
print(
    "  Reachable plans:       ",
    cwq_shared_rog[
        "reachable_plans"
    ]
)
print(
    "  Reachable questions:   ",
    cwq_shared_rog[
        "reachable_questions"
    ]
)

print("\nIntegrity")
print(
    "  Search-cost fidelity:  PASSED"
)
print(
    "  Reachability fidelity: PASSED"
)
print(
    "  Shared engine:         DEFINED"
)

print("\nExecution")
print(
    "  Planner rerun:         NO"
)
print(
    "  MiniLM rerun:          NO"
)
print(
    "  Pruning run:           NO"
)
print(
    "  Tuning run:            NO"
)
print(
    "  TEST loaded:           NO"
)
print(
    "  AFP fully frozen:      NO"
)

print(
    "\nManifest:",
    CELL12B_RECOVERY_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C — online Feature-v2 scorer "
    "integration fidelity gate."
)

Cell 12A-R prerequisites: PASSED
Frozen RQ1 references: LOADED

WEBQSP — OFFICIAL RoG GRAPH SEMANTICS FIDELITY
Observed core: {'active_hop_rows': 971, 'edges_examined': 341526, 'candidate_branches': 7983}
Expected core: {'active_hop_rows': 971, 'edges_examined': 341526, 'candidate_branches': 7983}
WebQSP search-cost fidelity: PASSED
Observed reachability: {'reachable_plans': 345, 'reachable_questions': 205}
Expected reachability: {'reachable_plans': 345, 'reachable_questions': 205}
WebQSP reachability fidelity: PASSED

CWQ — OFFICIAL RoG GRAPH SEMANTICS FIDELITY
Observed core: {'active_hop_rows': 16564, 'edges_examined': 5257272, 'candidate_branches': 247161}
Expected core: {'active_hop_rows': 16564, 'edges_examined': 5257272, 'candidate_branches': 247161}
CWQ search-cost fidelity: PASSED
Observed reachability: {'reachable_plans': 3971, 'reachable_questions': 2425}
Expected reachability: {'reachable_plans': 3971, 'reachable_questions': 2425}
CWQ reachability fidelity: PASSED

Plan-coun

In [10]:
# ======================================================================
# RECOVER EXACT FROZEN FEATURE-v2 RUNTIME SPECIFICATION
# ======================================================================
#
# PURPOSE
# -------
# Before implementing ONLINE AFP scoring inside the dynamic traversal,
# recover as much of the EXACT frozen Feature-v2 implementation as
# possible from:
#
#   - feature manifests
#   - validation supervision manifests/records
#   - saved NPZ structure
#   - saved notebook/source files, if present
#   - semantic embedding/cache artifacts, if present
#   - surviving Python globals
#
# WHY:
#   We must NOT re-invent Feature-v2 from feature names alone.
#   Cell 12C-B must reproduce the cached validation features before
#   being allowed to score dynamically changed frontiers.
#
# THIS CELL:
#   - does NOT compute new features
#   - does NOT run MiniLM
#   - does NOT run pruning
#   - does NOT tune hyperparameters
#   - does NOT touch TEST
# ======================================================================

import os
import re
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ======================================================================
# 1. Roots
# ======================================================================

RQ2_ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_DIR = (
    RQ2_ROOT
    / "03_features"
)

assert FEATURE_DIR.exists()

print("RQ2 root:     ", RQ2_ROOT)
print("Feature root: ", FEATURE_DIR)


# ======================================================================
# 2. Expected frozen Feature-v2 identity
# ======================================================================

EXPECTED_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

EXPECTED_FEATURE_DIM = 27

EXPECTED_FEATURE_SHA256 = (
    "738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86"
)

EXPECTED_FEATURE_NAMES = [
    "sem_q_candidate",
    "sem_candidate_surface_available",
    "sem_q_current_entity",
    "sem_current_surface_available",
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",
    "sem_candidate_current_relation",

    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",
    "path_prefix_surface_fraction",
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]

assert len(EXPECTED_FEATURE_NAMES) == 27

print("\nExpected Feature-v2 identity:")
print("  version:", EXPECTED_FEATURE_VERSION)
print("  dim:    ", EXPECTED_FEATURE_DIM)
print("  SHA256: ", EXPECTED_FEATURE_SHA256)


# ======================================================================
# 3. Locate frozen manifests
# ======================================================================

WEBQSP_VAL_FEATURE_MANIFEST = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2_manifest.json"
)

CWQ_VAL_FEATURE_MANIFEST = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2_manifest.json"
)

WEBQSP_VAL_FEATURE_NPZ = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2.npz"
)

CWQ_VAL_FEATURE_NPZ = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2.npz"
)

WEBQSP_LABEL_MANIFEST = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels_manifest.json"
)

CWQ_LABEL_MANIFEST = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels_manifest.json"
)

WEBQSP_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels.jsonl"
)

CWQ_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels.jsonl"
)

for path in [
    WEBQSP_VAL_FEATURE_MANIFEST,
    CWQ_VAL_FEATURE_MANIFEST,
    WEBQSP_VAL_FEATURE_NPZ,
    CWQ_VAL_FEATURE_NPZ,
    WEBQSP_LABEL_MANIFEST,
    CWQ_LABEL_MANIFEST,
    WEBQSP_LABEL_FILE,
    CWQ_LABEL_FILE,
]:
    assert path.exists(), (
        f"Missing artifact: {path}"
    )

print("\nFrozen feature/supervision artifacts: FOUND")


# ======================================================================
# 4. JSON helpers
# ======================================================================

def load_json(path):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def load_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


webqsp_feature_manifest_12c = load_json(
    WEBQSP_VAL_FEATURE_MANIFEST
)

cwq_feature_manifest_12c = load_json(
    CWQ_VAL_FEATURE_MANIFEST
)

webqsp_label_manifest_12c = load_json(
    WEBQSP_LABEL_MANIFEST
)

cwq_label_manifest_12c = load_json(
    CWQ_LABEL_MANIFEST
)


# ======================================================================
# 5. Pretty-print manifest structure
# ======================================================================

def print_json_structure(
    obj,
    prefix="",
    depth=0,
    max_depth=4
):
    if depth > max_depth:
        return

    if isinstance(obj, dict):
        for key, value in obj.items():

            full = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if isinstance(value, dict):
                print(
                    f"  {full}: dict[{len(value)}]"
                )

                print_json_structure(
                    value,
                    full,
                    depth + 1,
                    max_depth
                )

            elif isinstance(value, list):
                print(
                    f"  {full}: list[{len(value)}]"
                )

                if (
                    len(value) <= 40
                    and all(
                        not isinstance(
                            x,
                            (dict, list)
                        )
                        for x in value
                    )
                ):
                    print(
                        "    ",
                        value
                    )

            else:
                text = str(value)

                if len(text) > 180:
                    text = (
                        text[:177]
                        + "..."
                    )

                print(
                    f"  {full}: {text}"
                )


print(
    "\n"
    + "=" * 100
)

print(
    "WEBQSP FEATURE MANIFEST STRUCTURE"
)

print(
    "=" * 100
)

print_json_structure(
    webqsp_feature_manifest_12c
)


print(
    "\n"
    + "=" * 100
)

print(
    "CWQ FEATURE MANIFEST STRUCTURE"
)

print(
    "=" * 100
)

print_json_structure(
    cwq_feature_manifest_12c
)


# ======================================================================
# 6. Search manifests recursively for important terms
# ======================================================================

IMPORTANT_TERMS = [
    "feature",
    "semantic",
    "surface",
    "encoder",
    "minilm",
    "embedding",
    "prefix",
    "candidate",
    "struct",
    "progress",
    "log1p",
    "cosine",
    "relation",
    "suffix",
    "mask",
    "mid",
]


def flatten_json(
    obj,
    prefix=""
):
    output = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            output.extend(
                flatten_json(
                    value,
                    new_prefix
                )
            )

    elif isinstance(obj, list):

        if all(
            not isinstance(
                x,
                (dict, list)
            )
            for x in obj
        ):
            output.append(
                (
                    prefix,
                    obj
                )
            )

        else:
            for i, value in enumerate(
                obj
            ):
                output.extend(
                    flatten_json(
                        value,
                        f"{prefix}[{i}]"
                    )
                )

    else:
        output.append(
            (
                prefix,
                obj
            )
        )

    return output


def print_relevant_manifest_entries(
    name,
    manifest
):
    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{name} — RELEVANT MANIFEST ENTRIES"
    )

    print(
        "=" * 100
    )

    flattened = flatten_json(
        manifest
    )

    shown = set()

    for path, value in flattened:
        combined = (
            path
            + " "
            + str(value)
        ).lower()

        if not any(
            term in combined
            for term
            in IMPORTANT_TERMS
        ):
            continue

        key = (
            path,
            str(value)
        )

        if key in shown:
            continue

        shown.add(
            key
        )

        text = str(value)

        if len(text) > 500:
            text = (
                text[:497]
                + "..."
            )

        print(
            f"{path}: {text}"
        )


print_relevant_manifest_entries(
    "WEBQSP",
    webqsp_feature_manifest_12c
)

print_relevant_manifest_entries(
    "CWQ",
    cwq_feature_manifest_12c
)


# ======================================================================
# 7. NPZ schema inspection
# ======================================================================

def inspect_npz(
    dataset,
    path
):
    data = np.load(
        path,
        allow_pickle=True
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset.upper()} FEATURE NPZ SCHEMA"
    )

    print(
        "=" * 100
    )

    for key in data.files:
        arr = data[key]

        print(
            f"{key:<30} "
            f"shape={str(arr.shape):<18} "
            f"dtype={arr.dtype}"
        )

        if (
            arr.ndim == 1
            and len(arr) <= 30
        ):
            print(
                "   ",
                arr.tolist()
            )

    return {
        key: data[key]
        for key in data.files
    }


webqsp_npz_12c = inspect_npz(
    "webqsp",
    WEBQSP_VAL_FEATURE_NPZ
)

cwq_npz_12c = inspect_npz(
    "cwq",
    CWQ_VAL_FEATURE_NPZ
)


# ======================================================================
# 8. Basic frozen shape gates
# ======================================================================

def resolve_npz_array(
    d,
    aliases
):
    for key in aliases:
        if key in d:
            return d[key]

    return None


webqsp_X_12c = resolve_npz_array(
    webqsp_npz_12c,
    [
        "X",
        "features",
        "x",
    ]
)

cwq_X_12c = resolve_npz_array(
    cwq_npz_12c,
    [
        "X",
        "features",
        "x",
    ]
)

assert webqsp_X_12c is not None
assert cwq_X_12c is not None

assert webqsp_X_12c.shape == (
    966,
    27
)

assert cwq_X_12c.shape == (
    18688,
    27
)

print(
    "\nFrozen validation feature shape gate: PASSED"
)


# ======================================================================
# 9. Feature-column numeric summary
# ======================================================================

def feature_numeric_summary(
    dataset,
    X
):
    assert X.shape[1] == 27

    rows = []

    for j, name in enumerate(
        EXPECTED_FEATURE_NAMES
    ):
        col = np.asarray(
            X[:, j],
            dtype=np.float64
        )

        rows.append(
            {
                "index":
                    j,

                "feature":
                    name,

                "min":
                    float(
                        np.min(col)
                    ),

                "max":
                    float(
                        np.max(col)
                    ),

                "mean":
                    float(
                        np.mean(col)
                    ),

                "std":
                    float(
                        np.std(col)
                    ),

                "zero_rate":
                    float(
                        np.mean(
                            np.isclose(
                                col,
                                0.0
                            )
                        )
                    ),

                "unique_rounded_6":
                    int(
                        len(
                            np.unique(
                                np.round(
                                    col,
                                    6
                                )
                            )
                        )
                    ),
            }
        )

    df = pd.DataFrame(
        rows
    )

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"{dataset.upper()} FEATURE NUMERIC SUMMARY"
    )

    print(
        "=" * 110
    )

    print(
        df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    return df


webqsp_feature_numeric_12c = (
    feature_numeric_summary(
        "webqsp",
        webqsp_X_12c
    )
)

cwq_feature_numeric_12c = (
    feature_numeric_summary(
        "cwq",
        cwq_X_12c
    )
)


# ======================================================================
# 10. Validation supervision schema
# ======================================================================

webqsp_labels_12c = load_jsonl(
    WEBQSP_LABEL_FILE
)

cwq_labels_12c = load_jsonl(
    CWQ_LABEL_FILE
)

print(
    "\nValidation supervision rows:"
)

print(
    "  WebQSP:",
    len(webqsp_labels_12c)
)

print(
    "  CWQ:   ",
    len(cwq_labels_12c)
)


def describe_json_record(
    dataset,
    row
):
    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset.upper()} VALIDATION SUPERVISION SAMPLE"
    )

    print(
        "=" * 100
    )

    for key, value in row.items():

        if isinstance(value, list):
            preview = value[:5]

            print(
                f"{key:<35} "
                f"list[{len(value)}] "
                f"{preview}"
            )

        elif isinstance(value, dict):
            print(
                f"{key:<35} "
                f"dict keys={list(value.keys())[:20]}"
            )

        else:
            text = str(value)

            if len(text) > 250:
                text = (
                    text[:247]
                    + "..."
                )

            print(
                f"{key:<35} "
                f"{type(value).__name__}: {text}"
            )


if webqsp_labels_12c:
    describe_json_record(
        "webqsp",
        webqsp_labels_12c[0]
    )

if cwq_labels_12c:
    describe_json_record(
        "cwq",
        cwq_labels_12c[0]
    )


# ======================================================================
# 11. Supervision manifest structures
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "WEBQSP VALIDATION SUPERVISION MANIFEST"
)

print(
    "=" * 100
)

print_json_structure(
    webqsp_label_manifest_12c
)


print(
    "\n"
    + "=" * 100
)

print(
    "CWQ VALIDATION SUPERVISION MANIFEST"
)

print(
    "=" * 100
)

print_json_structure(
    cwq_label_manifest_12c
)


# ======================================================================
# 12. Search filesystem for original Feature-v2 source
# ======================================================================
#
# Look for literal frozen feature names/version in:
#
#   .py
#   .ipynb
#   .md
#   .txt
#   .json
#
# Large graph/data files are skipped.
# ======================================================================

SOURCE_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

SOURCE_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".md",
    ".txt",
    ".json",
}

SOURCE_NEEDLES = [
    "afp_features_v2_masked_entity_semantics",
    "sem_q_candidate",
    "path_candidate_prefix_entity_mean",
    "struct_log_endpoint_multiplicity",
    "prog_penultimate_indicator",
]

MAX_SOURCE_FILE_MB = 25


def search_source_files():
    hits = []

    for root in SOURCE_ROOTS:

        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d not in {
                    "__pycache__",
                    "node_modules",
                }
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                if (
                    path.suffix.lower()
                    not in SOURCE_EXTENSIONS
                ):
                    continue

                text_path = str(
                    path
                ).lower()

                # Explicit TEST leakage safety.
                normalized_parts = re.split(
                    r"[/\\_\-.]+",
                    text_path
                )

                if "test" in normalized_parts:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )

                    if (
                        size_mb
                        > MAX_SOURCE_FILE_MB
                    ):
                        continue

                    content = path.read_text(
                        encoding="utf-8",
                        errors="ignore"
                    )

                except Exception:
                    continue

                matched = [
                    needle
                    for needle
                    in SOURCE_NEEDLES
                    if needle in content
                ]

                if matched:
                    hits.append(
                        {
                            "path":
                                str(path),

                            "size_mb":
                                size_mb,

                            "matched":
                                matched,
                        }
                    )

    hits.sort(
        key=lambda x: (
            -len(
                x["matched"]
            ),
            x["size_mb"],
            x["path"],
        )
    )

    return hits


feature_source_hits_12c = (
    search_source_files()
)

print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 SOURCE SEARCH"
)

print(
    "=" * 100
)

if not feature_source_hits_12c:
    print(
        "No saved source file containing Feature-v2 literals found."
    )

else:
    for hit in feature_source_hits_12c[
        :30
    ]:
        print(
            f"{hit['size_mb']:.3f} MB  "
            f"{hit['path']}"
        )

        print(
            "   matched:",
            hit[
                "matched"
            ]
        )


# ======================================================================
# 13. Extract relevant notebook/code snippets when possible
# ======================================================================

def extract_source_snippets(
    path,
    needles,
    context_chars=2500
):
    path = Path(
        path
    )

    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    snippets = []

    for needle in needles:

        pos = text.find(
            needle
        )

        if pos < 0:
            continue

        start = max(
            0,
            pos - context_chars
        )

        end = min(
            len(text),
            pos + context_chars
        )

        snippets.append(
            {
                "needle":
                    needle,

                "snippet":
                    text[
                        start:end
                    ],
            }
        )

    return snippets


print(
    "\n"
    + "=" * 100
)

print(
    "TOP SOURCE SNIPPETS"
)

print(
    "=" * 100
)

if feature_source_hits_12c:

    for hit in feature_source_hits_12c[
        :3
    ]:

        print(
            "\nSOURCE:",
            hit["path"]
        )

        snippets = (
            extract_source_snippets(
                hit["path"],
                hit["matched"]
            )
        )

        for snippet in snippets[
            :3
        ]:

            print(
                "\n--- around:",
                snippet[
                    "needle"
                ],
                "---"
            )

            print(
                snippet[
                    "snippet"
                ][:6000]
            )


# ======================================================================
# 14. Search for semantic embedding/cache artifacts
# ======================================================================

SEMANTIC_FILE_TOKENS = [
    "embedding",
    "embeddings",
    "semantic",
    "minilm",
    "sentence",
    "encoder",
    "cache",
]


def discover_semantic_artifacts():
    hits = []

    for root in SOURCE_ROOTS:

        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d != "__pycache__"
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                text = str(
                    path
                ).lower()

                if not any(
                    token in text
                    for token
                    in SEMANTIC_FILE_TOKENS
                ):
                    continue

                # Avoid TEST files.
                parts = re.split(
                    r"[/\\_\-.]+",
                    text
                )

                if "test" in parts:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )
                except Exception:
                    size_mb = np.nan

                hits.append(
                    {
                        "path":
                            str(path),

                        "size_mb":
                            size_mb,
                    }
                )

    # Deduplicate
    unique = {
        x["path"]: x
        for x in hits
    }

    hits = list(
        unique.values()
    )

    hits.sort(
        key=lambda x: (
            x["path"]
        )
    )

    return hits


semantic_artifacts_12c = (
    discover_semantic_artifacts()
)

print(
    "\n"
    + "=" * 100
)

print(
    "SEMANTIC / EMBEDDING / CACHE ARTIFACTS"
)

print(
    "=" * 100
)

if not semantic_artifacts_12c:
    print(
        "No obvious semantic-cache artifact found."
    )

else:
    for hit in semantic_artifacts_12c[
        :50
    ]:

        size = hit[
            "size_mb"
        ]

        size_text = (
            f"{size:.3f} MB"
            if np.isfinite(size)
            else "?"
        )

        print(
            f"{size_text:>12}  "
            f"{hit['path']}"
        )


# ======================================================================
# 15. Search current globals for encoder/cache/extractor objects
# ======================================================================

GLOBAL_TOKENS = [
    "feature",
    "semantic",
    "embed",
    "encoder",
    "minilm",
    "sentence",
    "surface",
    "cache",
]


runtime_globals_12c = []

for name, obj in globals().items():

    if name.startswith(
        "_"
    ):
        continue

    low = name.lower()

    if any(
        token in low
        for token
        in GLOBAL_TOKENS
    ):

        runtime_globals_12c.append(
            {
                "name":
                    name,

                "type":
                    type(
                        obj
                    ).__name__,

                "callable":
                    callable(
                        obj
                    ),
            }
        )


runtime_globals_12c = sorted(
    runtime_globals_12c,
    key=lambda x:
        x["name"]
)

print(
    "\n"
    + "=" * 100
)

print(
    "CURRENT FEATURE/SEMANTIC-RELATED GLOBALS"
)

print(
    "=" * 100
)

for row in runtime_globals_12c:
    print(
        f"{row['name']:<50} "
        f"type={row['type']:<25} "
        f"callable={row['callable']}"
    )


# ======================================================================
# 16. Determine whether exact runtime implementation is recoverable
# ======================================================================

SOURCE_FOUND = (
    len(
        feature_source_hits_12c
    ) > 0
)

SEMANTIC_CACHE_FOUND = (
    len(
        semantic_artifacts_12c
    ) > 0
)

EXTRACTOR_GLOBALS = [
    row["name"]
    for row in runtime_globals_12c
    if row[
        "callable"
    ]
    and any(
        token
        in row["name"].lower()
        for token
        in [
            "feature",
            "embed",
            "semantic",
            "surface",
        ]
    )
]


# ======================================================================
# 17. Save audit manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_A_MANIFEST = {
    "cell":
        "RQ2_12C_A",

    "purpose":
        "recover_exact_frozen_feature_v2_runtime_specification",

    "feature_version":
        EXPECTED_FEATURE_VERSION,

    "feature_dim":
        EXPECTED_FEATURE_DIM,

    "feature_spec_sha256":
        EXPECTED_FEATURE_SHA256,

    "webqsp_validation_shape":
        list(
            webqsp_X_12c.shape
        ),

    "cwq_validation_shape":
        list(
            cwq_X_12c.shape
        ),

    "source_hits":
        feature_source_hits_12c,

    "semantic_artifact_count":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "source_found":
        SOURCE_FOUND,

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "new_feature_computation":
        False,

    "semantic_encoder_run":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,
}

CELL12C_A_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_a_feature_runtime_recovery_audit.json"
)

with open(
    CELL12C_A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 18. Final report
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== RQ2 CELL 12C-A: FROZEN FEATURE-v2 RUNTIME RECOVERY AUDIT COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "\nFrozen Feature-v2"
)

print(
    "  Version:                     ",
    EXPECTED_FEATURE_VERSION
)

print(
    "  Dimension:                   ",
    EXPECTED_FEATURE_DIM
)

print(
    "  SHA256:                      ",
    EXPECTED_FEATURE_SHA256
)

print(
    "\nValidation caches"
)

print(
    "  WebQSP:                      ",
    webqsp_X_12c.shape
)

print(
    "  CWQ:                         ",
    cwq_X_12c.shape
)

print(
    "\nRecovery"
)

print(
    "  Feature source found:        ",
    SOURCE_FOUND
)

print(
    "  Semantic/cache candidates:   ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "  Extractor globals found:     ",
    EXTRACTOR_GLOBALS
)

print(
    "\nExecution"
)

print(
    "  New features computed:       NO"
)

print(
    "  MiniLM/encoder run:          NO"
)

print(
    "  Pruning run:                 NO"
)

print(
    "  Hyperparameter tuning:       NO"
)

print(
    "  TEST loaded:                 NO"
)

print(
    "\nManifest:",
    CELL12C_A_MANIFEST_PATH
)

print(
    "\nNext: use this audit to implement Cell 12C-B "
    "and require exact reproduction of cached Feature-v2 values "
    "before enabling online AFP scoring."
)

RQ2 root:      /kaggle/working/step3_rq2_dev_v1
Feature root:  /kaggle/working/step3_rq2_dev_v1/03_features

Expected Feature-v2 identity:
  version: afp_features_v2_masked_entity_semantics
  dim:     27
  SHA256:  738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86

Frozen feature/supervision artifacts: FOUND

WEBQSP FEATURE MANIFEST STRUCTURE
  dataset: webqsp
  split: validation
  feature_version: afp_features_v2_masked_entity_semantics
  feature_spec_sha256: 738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86
  feature_dim: 27
  semantic_encoder: sentence-transformers/all-MiniLM-L6-v2
  raw_mid_policy: zero_embedding_plus_availability
  decision_only: True
  n_groups: 87
  n_branches: 966
  positive_branches: 319
  negative_branches: 647
  positive_rate: 0.3302277432712215
  npz_sha256: de6b1ed170f931812561aaee141439c22fb23bfdb63cd025f637caa6e1f7edeb
  created_utc: 2026-08-31T16:40:38.107811+00:00

CWQ FEATURE MANIFEST STRUCTURE
  dataset: cwq
  split: val

RuntimeError: dictionary changed size during iteration

In [11]:
# ======================================================================
# A RECOVERY — CONTINUE FROM SECTION 15
# ======================================================================
#
# Fixes:
#   RuntimeError: dictionary changed size during iteration
#
# Everything before Section 15 from the previous run remains valid.
# ======================================================================

import json
from pathlib import Path


# ======================================================================
# 15. Search current globals for encoder/cache/extractor objects
# ======================================================================

GLOBAL_TOKENS = [
    "feature",
    "semantic",
    "embed",
    "encoder",
    "minilm",
    "sentence",
    "surface",
    "cache",
]

runtime_globals_12c = []

# IMPORTANT:
# Snapshot globals BEFORE iterating.
# This prevents Jupyter/global-scope assignments from modifying the
# dictionary being iterated.
global_snapshot_12c = list(
    globals().items()
)

for name, obj in global_snapshot_12c:

    if name.startswith("_"):
        continue

    low = name.lower()

    if any(
        token in low
        for token in GLOBAL_TOKENS
    ):
        runtime_globals_12c.append(
            {
                "name":
                    name,

                "type":
                    type(obj).__name__,

                "callable":
                    callable(obj),
            }
        )


runtime_globals_12c = sorted(
    runtime_globals_12c,
    key=lambda x: x["name"]
)


print(
    "\n"
    + "=" * 100
)

print(
    "CURRENT FEATURE/SEMANTIC-RELATED GLOBALS"
)

print(
    "=" * 100
)

for row in runtime_globals_12c:

    print(
        f"{row['name']:<50} "
        f"type={row['type']:<25} "
        f"callable={row['callable']}"
    )


# ======================================================================
# 16. Determine exact runtime-recovery status
# ======================================================================

SOURCE_FOUND = (
    len(
        feature_source_hits_12c
    ) > 0
)

SEMANTIC_CACHE_FOUND = (
    len(
        semantic_artifacts_12c
    ) > 0
)

EXTRACTOR_GLOBALS = [
    row["name"]
    for row in runtime_globals_12c
    if row["callable"]
    and any(
        token in row["name"].lower()
        for token in [
            "feature",
            "embed",
            "semantic",
            "surface",
        ]
    )
]


# ======================================================================
# 17. Additional recovery interpretation
# ======================================================================
#
# We found the exact saved notebook source containing Feature-v2.
# This is sufficient to reconstruct the original feature code itself.
#
# However:
#   semantic cache candidate was not found by filename search.
#
# Therefore Cell 12C-B must separately determine whether:
#
#   A) MiniLM model/cache is locally available, or
#   B) original semantic embeddings can be recovered from notebook
#      artifacts, or
#   C) semantic encoding must be rerun.
#
# We will NOT silently change the feature representation.
# ======================================================================

FEATURE_RUNTIME_RECOVERY_STATUS = {
    "exact_feature_source_found":
        SOURCE_FOUND,

    "feature_source_primary":
        (
            feature_source_hits_12c[0]["path"]
            if SOURCE_FOUND
            else None
        ),

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "semantic_cache_candidates":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "safe_to_reconstruct_feature_code":
        SOURCE_FOUND,

    "safe_to_claim_online_semantic_runtime_ready":
        False,
}


print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 RUNTIME RECOVERY STATUS"
)

print(
    "=" * 100
)

print(
    "Exact Feature-v2 source found:       ",
    SOURCE_FOUND
)

if SOURCE_FOUND:
    print(
        "Primary source:                    ",
        feature_source_hits_12c[0]["path"]
    )

print(
    "Semantic/cache candidates found:     ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "Feature/semantic callable globals:   ",
    EXTRACTOR_GLOBALS
)

print(
    "Feature code reconstructable:        ",
    SOURCE_FOUND
)

print(
    "Online semantic runtime ready:       ",
    "NOT YET VERIFIED"
)


# ======================================================================
# 18. Save corrected audit manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_A_MANIFEST = {
    "cell":
        "RQ2_12C_A",

    "purpose":
        "recover_exact_frozen_feature_v2_runtime_specification",

    "feature_version":
        EXPECTED_FEATURE_VERSION,

    "feature_dim":
        EXPECTED_FEATURE_DIM,

    "feature_spec_sha256":
        EXPECTED_FEATURE_SHA256,

    "webqsp_validation_shape":
        list(
            webqsp_X_12c.shape
        ),

    "cwq_validation_shape":
        list(
            cwq_X_12c.shape
        ),

    "source_hits":
        feature_source_hits_12c,

    "primary_feature_source":
        (
            feature_source_hits_12c[0]["path"]
            if SOURCE_FOUND
            else None
        ),

    "semantic_artifact_count":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "source_found":
        SOURCE_FOUND,

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "runtime_recovery_status":
        FEATURE_RUNTIME_RECOVERY_STATUS,

    "new_feature_computation":
        False,

    "semantic_encoder_run":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,
}

CELL12C_A_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_a_feature_runtime_recovery_audit.json"
)

with open(
    CELL12C_A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 19. Final report
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== RQ2 CELL 12C-A: FROZEN FEATURE-v2 RUNTIME RECOVERY AUDIT COMPLETE ==="
)

print(
    "=" * 104
)

print("\nFrozen Feature-v2")

print(
    "  Version:                     ",
    EXPECTED_FEATURE_VERSION
)

print(
    "  Dimension:                   ",
    EXPECTED_FEATURE_DIM
)

print(
    "  SHA256:                      ",
    EXPECTED_FEATURE_SHA256
)


print("\nValidation caches")

print(
    "  WebQSP:                      ",
    webqsp_X_12c.shape
)

print(
    "  CWQ:                         ",
    cwq_X_12c.shape
)


print("\nRecovery")

print(
    "  Exact feature source found:  ",
    SOURCE_FOUND
)

print(
    "  Primary source:              ",
    (
        feature_source_hits_12c[0]["path"]
        if SOURCE_FOUND
        else "NONE"
    )
)

print(
    "  Semantic/cache candidates:   ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "  Extractor globals found:     ",
    EXTRACTOR_GLOBALS
)


print("\nExecution")

print(
    "  New features computed:       NO"
)

print(
    "  MiniLM/encoder run:          NO"
)

print(
    "  Pruning run:                 NO"
)

print(
    "  Hyperparameter tuning:       NO"
)

print(
    "  TEST loaded:                 NO"
)


print(
    "\nManifest:",
    CELL12C_A_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C-B — reconstruct exact Feature-v2 runtime "
    "from the saved notebook source and verify online features "
    "against the frozen validation NPZ before pruning."
)


CURRENT FEATURE/SEMANTIC-RELATED GLOBALS
AFPFeatureStandardizer                             type=type                      callable=True
AFP_FEATURE_DIM                                    type=int                       callable=False
AFP_FEATURE_SPEC_SHA256                            type=str                       callable=False
AFP_FEATURE_VERSION                                type=str                       callable=False
AFP_SEMANTIC_ENCODER_NAME                          type=str                       callable=False
CWQ_VAL_FEATURE_MANIFEST                           type=PosixPath                 callable=False
CWQ_VAL_FEATURE_NPZ                                type=PosixPath                 callable=False
EXPECTED_FEATURE_DIM                               type=int                       callable=False
EXPECTED_FEATURE_NAMES                             type=list                      callable=False
EXPECTED_FEATURE_SHA                               type=str                       call

In [13]:
# ======================================================================
# STATIC RECOVERY OF EXACT FEATURE-v2 SOURCE
# ======================================================================
#
# FIX:
#   Previous B1 executed top-level assignments from Cell 167.
#   One of those instantiated MockSemanticEncoder, whose definition
#   lives elsewhere in the notebook.
#
# This recovery cell performs STATIC SOURCE ANALYSIS ONLY.
#
# It does NOT execute Cell 167.
# Therefore:
#   - no MockSemanticEncoder dependency
#   - no MiniLM
#   - no feature computation
#   - no pruning
#   - no tuning
#   - no TEST
# ======================================================================

import ast
import json
import re
from pathlib import Path


# ======================================================================
# 1. Load saved notebook
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_notebook_12cb = json.load(f)

print("Notebook:", NOTEBOOK_PATH)
print("Cells:   ", len(saved_notebook_12cb["cells"]))


# ======================================================================
# 2. Collect code cells
# ======================================================================

code_cells_12cb = []

for cell_index, cell in enumerate(
    saved_notebook_12cb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    code_cells_12cb.append(
        {
            "cell_index": cell_index,
            "source": source,
        }
    )

print("Code cells:", len(code_cells_12cb))


# ======================================================================
# 3. Locate exact Feature-v2 definition cell
# ======================================================================

feature_definition_candidates = []

for item in code_cells_12cb:
    source = item["source"]

    if (
        'AFP_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"'
        in source
        and "sem_q_candidate" in source
        and "struct_log_endpoint_multiplicity" in source
        and "prog_penultimate_indicator" in source
    ):
        feature_definition_candidates.append(
            item
        )

assert len(feature_definition_candidates) == 1, (
    "Expected exactly one Feature-v2 definition cell; "
    f"found {len(feature_definition_candidates)}"
)

FEATURE_DEFINITION_CELL = (
    feature_definition_candidates[0]
)

FEATURE_SOURCE_12CB = (
    FEATURE_DEFINITION_CELL["source"]
)

FEATURE_CELL_INDEX_12CB = (
    FEATURE_DEFINITION_CELL["cell_index"]
)

print(
    "\nFeature-v2 definition cell:",
    FEATURE_CELL_INDEX_12CB
)


# ======================================================================
# 4. Parse source WITHOUT executing it
# ======================================================================

feature_tree_12cb = ast.parse(
    FEATURE_SOURCE_12CB,
    filename="saved_notebook_feature_v2"
)

print("Static AST parse: PASSED")


# ======================================================================
# 5. Static constant extractor
# ======================================================================

def get_assignment_node(
    tree,
    variable_name
):
    for node in tree.body:

        if isinstance(node, ast.Assign):
            for target in node.targets:

                if (
                    isinstance(target, ast.Name)
                    and target.id == variable_name
                ):
                    return node.value

        elif isinstance(node, ast.AnnAssign):

            if (
                isinstance(node.target, ast.Name)
                and node.target.id == variable_name
            ):
                return node.value

    return None


def literal_assignment(
    tree,
    variable_name
):
    node = get_assignment_node(
        tree,
        variable_name
    )

    if node is None:
        return None

    try:
        return ast.literal_eval(
            node
        )
    except Exception:
        return None


# ======================================================================
# 6. Recover literal Feature-v2 identity
# ======================================================================

STATIC_FEATURE_VERSION = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_FEATURE_VERSION"
    )
)

STATIC_SEMANTIC_ENCODER = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_SEMANTIC_ENCODER_NAME"
    )
)

STATIC_SEMANTIC_DIM = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_SEMANTIC_EXPECTED_DIM"
    )
)

print("\nStatic identity recovery:")
print(
    "  feature version:  ",
    STATIC_FEATURE_VERSION
)
print(
    "  encoder:          ",
    STATIC_SEMANTIC_ENCODER
)
print(
    "  semantic dim:     ",
    STATIC_SEMANTIC_DIM
)

assert (
    STATIC_FEATURE_VERSION
    ==
    EXPECTED_FEATURE_VERSION
)

assert (
    STATIC_SEMANTIC_ENCODER
    ==
    "sentence-transformers/all-MiniLM-L6-v2"
)

assert (
    STATIC_SEMANTIC_DIM
    == 384
)

print("Static identity gates: PASSED")


# ======================================================================
# 7. Recover exact feature-name lists statically
# ======================================================================

FEATURE_NAME_VARIABLES = [
    "AFP_SEMANTIC_FEATURE_NAMES",
    "AFP_PATH_FEATURE_NAMES",
    "AFP_STRUCTURAL_FEATURE_NAMES",
    "AFP_PROGRESS_FEATURE_NAMES",
]

static_feature_groups_12cb = {}

for variable in FEATURE_NAME_VARIABLES:

    value = literal_assignment(
        feature_tree_12cb,
        variable
    )

    assert isinstance(
        value,
        list
    ), (
        f"Could not statically recover {variable}"
    )

    static_feature_groups_12cb[
        variable
    ] = value


STATIC_FEATURE_NAMES = (
    static_feature_groups_12cb[
        "AFP_SEMANTIC_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_PATH_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_STRUCTURAL_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_PROGRESS_FEATURE_NAMES"
    ]
)

assert len(
    STATIC_FEATURE_NAMES
) == 27

assert (
    STATIC_FEATURE_NAMES
    ==
    EXPECTED_FEATURE_NAMES
)

print(
    "Feature-name/order static gate: PASSED"
)


# ======================================================================
# 8. Verify frozen SHA from already persisted manifest
# ======================================================================
#
# We do NOT need to execute the hashing expression inside Cell 167.
# The persisted feature artifact already carries the frozen SHA.
# ======================================================================

assert (
    webqsp_feature_manifest_12c[
        "feature_spec_sha256"
    ]
    ==
    EXPECTED_FEATURE_SHA256
)

assert (
    cwq_feature_manifest_12c[
        "feature_spec_sha256"
    ]
    ==
    EXPECTED_FEATURE_SHA256
)

print(
    "Persisted Feature-v2 SHA gate: PASSED"
)


# ======================================================================
# 9. Enumerate exact function definitions in Feature-v2 cell
# ======================================================================

feature_functions_12cb = []
feature_classes_12cb = []

for node in feature_tree_12cb.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        source_segment = (
            ast.get_source_segment(
                FEATURE_SOURCE_12CB,
                node
            )
            or ""
        )

        # Reconstruct readable signature directly from source line.
        first_line = (
            source_segment
            .splitlines()[0]
            if source_segment
            else node.name
        )

        feature_functions_12cb.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        getattr(
                            node,
                            "lineno",
                            -1
                        )
                    ),

                "end_line":
                    int(
                        getattr(
                            node,
                            "end_lineno",
                            -1
                        )
                    ),

                "first_line":
                    first_line,

                "source":
                    source_segment,
            }
        )

    elif isinstance(
        node,
        ast.ClassDef
    ):

        feature_classes_12cb.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        getattr(
                            node,
                            "lineno",
                            -1
                        )
                    ),

                "source":
                    (
                        ast.get_source_segment(
                            FEATURE_SOURCE_12CB,
                            node
                        )
                        or ""
                    ),
            }
        )


print(
    "\n"
    + "=" * 105
)

print(
    "FUNCTIONS RECOVERED FROM EXACT FEATURE-v2 CELL"
)

print(
    "=" * 105
)

for item in feature_functions_12cb:
    print(
        f"line={item['line']:<5} "
        f"{item['first_line']}"
    )


print(
    "\nClasses defined directly in Feature-v2 cell:"
)

if feature_classes_12cb:
    for item in feature_classes_12cb:
        print(
            f"line={item['line']:<5} "
            f"class {item['name']}"
        )
else:
    print("NONE")


# ======================================================================
# 10. Rank likely feature-builder functions by SOURCE CONTENT
# ======================================================================

BUILDER_MARKERS = [
    "candidate_entity",
    "prefix_entities",
    "semantic_encoder",
    "candidate_count",
    "AFP_FEATURE_DIM",
    "AFP_FEATURE_NAMES",
    "struct_log_candidate_count",
    "prog_hop_fraction",
]

feature_builder_candidates_12cb = []

for item in feature_functions_12cb:

    source = item[
        "source"
    ]

    score = sum(
        marker in source
        for marker
        in BUILDER_MARKERS
    )

    low_name = item[
        "name"
    ].lower()

    if any(
        token in low_name
        for token in [
            "feature",
            "candidate",
            "branch",
            "extract",
            "build",
        ]
    ):
        score += 2

    if score > 0:
        feature_builder_candidates_12cb.append(
            {
                "name":
                    item["name"],

                "line":
                    item["line"],

                "score":
                    score,

                "first_line":
                    item[
                        "first_line"
                    ],

                "source":
                    source,
            }
        )


feature_builder_candidates_12cb = sorted(
    feature_builder_candidates_12cb,
    key=lambda x: (
        -x["score"],
        x["line"]
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "LIKELY FEATURE BUILDER FUNCTIONS"
)

print(
    "=" * 105
)

for item in feature_builder_candidates_12cb:
    print(
        f"score={item['score']:<3} "
        f"line={item['line']:<5} "
        f"{item['first_line']}"
    )


# ======================================================================
# 11. Print top builder SOURCE exactly
# ======================================================================

TOP_FEATURE_BUILDER_12CB = (
    feature_builder_candidates_12cb[0]
    if feature_builder_candidates_12cb
    else None
)

print(
    "\n"
    + "=" * 105
)

print(
    "TOP FEATURE BUILDER SOURCE"
)

print(
    "=" * 105
)

if TOP_FEATURE_BUILDER_12CB is None:

    print(
        "No likely builder found."
    )

else:

    print(
        "Name:",
        TOP_FEATURE_BUILDER_12CB[
            "name"
        ]
    )

    print(
        "Line:",
        TOP_FEATURE_BUILDER_12CB[
            "line"
        ]
    )

    print()

    print(
        TOP_FEATURE_BUILDER_12CB[
            "source"
        ]
    )


# ======================================================================
# 12. Search entire notebook for MockSemanticEncoder
# ======================================================================

mock_semantic_locations_12cb = []

for item in code_cells_12cb:

    if "MockSemanticEncoder" in item[
        "source"
    ]:

        mock_semantic_locations_12cb.append(
            item[
                "cell_index"
            ]
        )


print(
    "\nMockSemanticEncoder appears in cells:",
    mock_semantic_locations_12cb
)


# ======================================================================
# 13. Search whole notebook for semantic encoder classes/functions
# ======================================================================

SEMANTIC_MARKERS = [
    "SentenceTransformer",
    "semantic_encoder",
    "SemanticEncoder",
    "MockSemanticEncoder",
    "all-MiniLM-L6-v2",
]


semantic_source_cells_12cb = []

for item in code_cells_12cb:

    source = item[
        "source"
    ]

    score = sum(
        marker.lower()
        in source.lower()
        for marker
        in SEMANTIC_MARKERS
    )

    if score > 0:

        semantic_source_cells_12cb.append(
            {
                "cell_index":
                    item[
                        "cell_index"
                    ],

                "score":
                    score,

                "source":
                    source,
            }
        )


semantic_source_cells_12cb = sorted(
    semantic_source_cells_12cb,
    key=lambda x: (
        -x["score"],
        x["cell_index"]
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "SEMANTIC ENCODER SOURCE-CELL CANDIDATES"
)

print(
    "=" * 105
)

for item in semantic_source_cells_12cb[
    :20
]:

    print(
        f"cell={item['cell_index']:<5} "
        f"score={item['score']}"
    )


# ======================================================================
# 14. Static inventory of semantic-related definitions
# ======================================================================

semantic_definition_inventory_12cb = []

for item in semantic_source_cells_12cb:

    try:
        tree = ast.parse(
            item[
                "source"
            ]
        )
    except SyntaxError:
        continue

    for node in tree.body:

        if isinstance(
            node,
            ast.ClassDef
        ):

            low = node.name.lower()

            if any(
                token in low
                for token in [
                    "semantic",
                    "embed",
                    "encoder",
                    "mock",
                ]
            ):

                semantic_definition_inventory_12cb.append(
                    {
                        "cell":
                            item[
                                "cell_index"
                            ],

                        "type":
                            "class",

                        "name":
                            node.name,

                        "source":
                            (
                                ast.get_source_segment(
                                    item[
                                        "source"
                                    ],
                                    node
                                )
                                or ""
                            ),
                    }
                )

        elif isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            low = node.name.lower()

            if any(
                token in low
                for token in [
                    "semantic",
                    "embed",
                    "encoder",
                    "surface",
                ]
            ):

                semantic_definition_inventory_12cb.append(
                    {
                        "cell":
                            item[
                                "cell_index"
                            ],

                        "type":
                            "function",

                        "name":
                            node.name,

                        "source":
                            (
                                ast.get_source_segment(
                                    item[
                                        "source"
                                    ],
                                    node
                                )
                                or ""
                            ),
                    }
                )


print(
    "\n"
    + "=" * 105
)

print(
    "SEMANTIC/ENCODER DEFINITIONS FOUND IN SAVED NOTEBOOK"
)

print(
    "=" * 105
)

for item in semantic_definition_inventory_12cb:

    print(
        f"cell={item['cell']:<5} "
        f"{item['type']:<10} "
        f"{item['name']}"
    )


# ======================================================================
# 15. Print sources of relevant semantic classes/functions
# ======================================================================

print(
    "\n"
    + "=" * 105
)

print(
    "RELEVANT SEMANTIC DEFINITION SOURCES"
)

print(
    "=" * 105
)

for item in semantic_definition_inventory_12cb:

    print(
        "\n"
        + "-" * 100
    )

    print(
        f"Cell {item['cell']} | "
        f"{item['type']} {item['name']}"
    )

    print(
        "-" * 100
    )

    print(
        item[
            "source"
        ]
    )


# ======================================================================
# 16. Check whether SentenceTransformer package/model is available
#     WITHOUT loading a model
# ======================================================================

sentence_transformers_importable_12cb = False
sentence_transformers_version_12cb = None

try:

    import sentence_transformers

    sentence_transformers_importable_12cb = True

    sentence_transformers_version_12cb = getattr(
        sentence_transformers,
        "__version__",
        "unknown"
    )

except Exception:

    sentence_transformers_importable_12cb = False


print(
    "\nSentence-transformers importable:",
    sentence_transformers_importable_12cb
)

print(
    "Version:",
    sentence_transformers_version_12cb
)


# ======================================================================
# 17. Save static recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_B1_STATIC_MANIFEST = {
    "cell":
        "RQ2_12C_B1_R",

    "recovery_mode":
        "static_source_analysis_only",

    "feature_source":
        str(
            NOTEBOOK_PATH
        ),

    "feature_definition_cell":
        FEATURE_CELL_INDEX_12CB,

    "feature_version":
        STATIC_FEATURE_VERSION,

    "semantic_encoder":
        STATIC_SEMANTIC_ENCODER,

    "semantic_dim":
        STATIC_SEMANTIC_DIM,

    "feature_dim":
        len(
            STATIC_FEATURE_NAMES
        ),

    "feature_names_match":
        True,

    "feature_sha_manifest_match":
        True,

    "functions":
        [
            {
                "name":
                    x["name"],

                "line":
                    x["line"],

                "first_line":
                    x["first_line"],
            }
            for x in
            feature_functions_12cb
        ],

    "builder_candidates":
        [
            {
                "name":
                    x["name"],

                "line":
                    x["line"],

                "score":
                    x["score"],

                "first_line":
                    x["first_line"],
            }
            for x in
            feature_builder_candidates_12cb
        ],

    "semantic_definitions":
        [
            {
                "cell":
                    x["cell"],

                "type":
                    x["type"],

                "name":
                    x["name"],
            }
            for x in
            semantic_definition_inventory_12cb
        ],

    "sentence_transformers_importable":
        sentence_transformers_importable_12cb,

    "sentence_transformers_version":
        sentence_transformers_version_12cb,

    "notebook_code_executed":
        False,

    "minilm_loaded":
        False,

    "new_features_computed":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning":
        False,

    "test_loaded":
        False,
}


CELL12C_B1_STATIC_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b1_static_feature_source_recovery.json"
)


with open(
    CELL12C_B1_STATIC_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_B1_STATIC_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 18. Final report
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "=== RQ2 CELL 12C-B1-R: EXACT FEATURE-v2 SOURCE STATICALLY RECOVERED ==="
)

print(
    "=" * 108
)

print(
    "Feature definition cell:    ",
    FEATURE_CELL_INDEX_12CB
)

print(
    "Feature version:            ",
    STATIC_FEATURE_VERSION
)

print(
    "Semantic encoder:           ",
    STATIC_SEMANTIC_ENCODER
)

print(
    "Semantic dimension:         ",
    STATIC_SEMANTIC_DIM
)

print(
    "Feature dimension:          ",
    len(
        STATIC_FEATURE_NAMES
    )
)

print(
    "Feature name/order gate:    PASS"
)

print(
    "Feature SHA manifest gate:  PASS"
)

print(
    "Notebook Feature cell run:  NO"
)

print(
    "MiniLM loaded:              NO"
)

print(
    "New features computed:      NO"
)

print(
    "Pruning run:                NO"
)

print(
    "Hyperparameter tuning:      NO"
)

print(
    "TEST loaded:                NO"
)

print(
    "\nManifest:",
    CELL12C_B1_STATIC_MANIFEST_PATH
)

print(
    "\nNext: use the recovered builder + encoder definitions "
    "to construct the runtime scorer and reproduce frozen NPZ "
    "features before any tuning."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Cells:    190
Code cells: 94

Feature-v2 definition cell: 167
Static AST parse: PASSED

Static identity recovery:
  feature version:   afp_features_v2_masked_entity_semantics
  encoder:           sentence-transformers/all-MiniLM-L6-v2
  semantic dim:      384
Static identity gates: PASSED
Feature-name/order static gate: PASSED
Persisted Feature-v2 SHA gate: PASSED

FUNCTIONS RECOVERED FROM EXACT FEATURE-v2 CELL
line=66    def is_raw_freebase_id(x):
line=77    def has_readable_entity_surface(
line=105   def normalize_surface_text(x):
line=130   def relation_surface_text(relation):
line=136   def entity_surface_text(
line=168   def relation_sequence_text(relations):
line=182   def safe_l2_normalize(
line=208   def safe_cosine(a, b):
line=227   def mean_embedding(
line=261   def get_safe_entity_embedding(
line=480   def extract_afp_candidate_features(
line=964   def extract_afp_group_features(
line=1056  def spli

In [14]:
# ======================================================================
# STATIC RECOVERY OF EXACT SEMANTIC RUNTIME + GROUP BUILDER
# ======================================================================
#
# PURPOSE
# -------
# Recover, without executing:
#
#   1. Exact semantic-encoder implementation/configuration.
#   2. Exact extract_afp_group_features(...) source.
#   3. Exact notebook call sites used to build Feature-v2.
#   4. Any encoder constructor / device / batch / normalization settings.
#
# NO MiniLM loading.
# NO new features.
# NO pruning.
# NO tuning.
# NO TEST.
# ======================================================================

import ast
import json
import re
from pathlib import Path


# ======================================================================
# 1. Reload saved notebook independently
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb_12cb2 = json.load(f)


def notebook_cell_source(cell_index):
    cell = nb_12cb2["cells"][cell_index]

    assert cell["cell_type"] == "code"

    return "".join(
        cell.get("source", [])
    )


# Known from B1-R
CELL_FEATURE = 167
CELL_SEMANTIC_CANDIDATE = 154
CELL_BUILD_CANDIDATE = 173

feature_source = notebook_cell_source(
    CELL_FEATURE
)

semantic_source_154 = notebook_cell_source(
    CELL_SEMANTIC_CANDIDATE
)

build_source_173 = notebook_cell_source(
    CELL_BUILD_CANDIDATE
)

print("Notebook:", NOTEBOOK_PATH)
print("Feature cell:", CELL_FEATURE)
print("Semantic candidate cell:", CELL_SEMANTIC_CANDIDATE)
print("Build candidate cell:", CELL_BUILD_CANDIDATE)


# ======================================================================
# 2. Generic exact AST source extractor
# ======================================================================

def parse_source(source, label):
    try:
        return ast.parse(
            source,
            filename=label
        )
    except SyntaxError as e:
        raise RuntimeError(
            f"AST parse failed for {label}: {e}"
        )


feature_tree = parse_source(
    feature_source,
    "feature_cell_167"
)

semantic_tree_154 = parse_source(
    semantic_source_154,
    "semantic_cell_154"
)

build_tree_173 = parse_source(
    build_source_173,
    "build_cell_173"
)


def exact_source(source, node):
    return (
        ast.get_source_segment(
            source,
            node
        )
        or ""
    )


# ======================================================================
# 3. Recover exact extract_afp_group_features source
# ======================================================================

group_builder_node = None

for node in feature_tree.body:
    if (
        isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
        and node.name
        == "extract_afp_group_features"
    ):
        group_builder_node = node
        break


assert group_builder_node is not None

GROUP_BUILDER_SOURCE_12CB2 = (
    exact_source(
        feature_source,
        group_builder_node
    )
)

print(
    "\n"
    + "=" * 110
)

print(
    "EXACT extract_afp_group_features SOURCE"
)

print(
    "=" * 110
)

print(
    GROUP_BUILDER_SOURCE_12CB2
)


# ======================================================================
# 4. Inventory ALL top-level definitions in Cell 154
# ======================================================================

semantic_defs_154 = []

for node in semantic_tree_154.body:

    if isinstance(
        node,
        ast.ClassDef
    ):
        semantic_defs_154.append(
            {
                "type":
                    "class",

                "name":
                    node.name,

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        semantic_source_154,
                        node
                    ),
            }
        )

    elif isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):
        semantic_defs_154.append(
            {
                "type":
                    "function",

                "name":
                    node.name,

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        semantic_source_154,
                        node
                    ),
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "ALL TOP-LEVEL DEFINITIONS IN SEMANTIC CELL 154"
)

print(
    "=" * 110
)

if not semantic_defs_154:
    print("NONE")

for item in semantic_defs_154:
    first_line = (
        item["source"]
        .splitlines()[0]
        if item["source"]
        else item["name"]
    )

    print(
        f"line={item['line']:<5} "
        f"{item['type']:<10} "
        f"{first_line}"
    )


# ======================================================================
# 5. Print exact Cell-154 definitions
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "EXACT SEMANTIC CELL 154 DEFINITION SOURCES"
)

print(
    "=" * 110
)

for item in semantic_defs_154:

    print(
        "\n"
        + "-" * 105
    )

    print(
        f"{item['type'].upper()} "
        f"{item['name']} "
        f"(line {item['line']})"
    )

    print(
        "-" * 105
    )

    print(
        item["source"]
    )


# ======================================================================
# 6. Find relevant imports in Cell 154
# ======================================================================

semantic_imports_154 = []

for node in semantic_tree_154.body:

    if isinstance(
        node,
        (
            ast.Import,
            ast.ImportFrom,
        )
    ):
        semantic_imports_154.append(
            exact_source(
                semantic_source_154,
                node
            )
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 154 IMPORTS"
)

print(
    "=" * 110
)

for src in semantic_imports_154:
    print(src)


# ======================================================================
# 7. Extract relevant top-level assignments / expressions
# ======================================================================

RUNTIME_TERMS = [
    "semantic",
    "encoder",
    "sentence",
    "transformer",
    "minilm",
    "model",
    "device",
    "batch",
    "cache",
    "normalize",
    "embedding",
]


def relevant_top_level_statements(
    source,
    tree
):
    rows = []

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
                ast.Import,
                ast.ImportFrom,
            )
        ):
            continue

        segment = exact_source(
            source,
            node
        )

        low = segment.lower()

        if any(
            term in low
            for term in RUNTIME_TERMS
        ):
            rows.append(
                {
                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        segment,
                }
            )

    return rows


semantic_runtime_statements_154 = (
    relevant_top_level_statements(
        semantic_source_154,
        semantic_tree_154
    )
)

build_runtime_statements_173 = (
    relevant_top_level_statements(
        build_source_173,
        build_tree_173
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 154 RELEVANT RUNTIME STATEMENTS"
)

print(
    "=" * 110
)

if not semantic_runtime_statements_154:
    print("NONE")

for item in semantic_runtime_statements_154:
    print(
        f"\n[line {item['line']}]\n"
        f"{item['source']}"
    )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 173 RELEVANT RUNTIME STATEMENTS"
)

print(
    "=" * 110
)

if not build_runtime_statements_173:
    print("NONE")

for item in build_runtime_statements_173:
    print(
        f"\n[line {item['line']}]\n"
        f"{item['source']}"
    )


# ======================================================================
# 8. Search ALL notebook cells for encoder construction
# ======================================================================

SEARCH_PATTERNS = [
    "SentenceTransformer(",
    "semantic_encoder =",
    "semantic_encoder=",
    "SemanticEncoder(",
    "Cached",
    "MiniLM",
    "all-MiniLM-L6-v2",
]

encoder_runtime_hits = []

for cell_index, cell in enumerate(
    nb_12cb2["cells"]
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    matched = [
        pattern
        for pattern in SEARCH_PATTERNS
        if pattern.lower()
        in source.lower()
    ]

    if matched:
        encoder_runtime_hits.append(
            {
                "cell":
                    cell_index,

                "matched":
                    matched,

                "source":
                    source,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "NOTEBOOK CELLS CONTAINING SEMANTIC ENCODER CONSTRUCTION"
)

print(
    "=" * 110
)

for hit in encoder_runtime_hits:

    print(
        f"\nCELL {hit['cell']} "
        f"| matched={hit['matched']}"
    )

    print(
        "-" * 100
    )

    # Print only relevant lines + nearby context.
    lines = hit[
        "source"
    ].splitlines()

    relevant_indices = []

    for i, line in enumerate(
        lines
    ):
        low = line.lower()

        if any(
            pattern.lower()
            in low
            for pattern
            in SEARCH_PATTERNS
        ):
            relevant_indices.extend(
                range(
                    max(0, i - 5),
                    min(
                        len(lines),
                        i + 12
                    )
                )
            )

    relevant_indices = sorted(
        set(
            relevant_indices
        )
    )

    for i in relevant_indices:
        print(
            f"{i+1:04d}: "
            f"{lines[i]}"
        )


# ======================================================================
# 9. Find exact Feature-v2 call sites across notebook
# ======================================================================

CALL_TARGETS = {
    "extract_afp_candidate_features",
    "extract_afp_group_features",
    "collect_semantic_texts",
}


feature_call_sites = []

for cell_index, cell in enumerate(
    nb_12cb2["cells"]
):

    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    try:
        tree = ast.parse(
            source
        )
    except Exception:
        continue

    for node in ast.walk(
        tree
    ):
        if not isinstance(
            node,
            ast.Call
        ):
            continue

        function_name = None

        if isinstance(
            node.func,
            ast.Name
        ):
            function_name = (
                node.func.id
            )

        elif isinstance(
            node.func,
            ast.Attribute
        ):
            function_name = (
                node.func.attr
            )

        if (
            function_name
            not in CALL_TARGETS
        ):
            continue

        segment = exact_source(
            source,
            node
        )

        feature_call_sites.append(
            {
                "cell":
                    cell_index,

                "line":
                    getattr(
                        node,
                        "lineno",
                        -1
                    ),

                "function":
                    function_name,

                "source":
                    segment,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "EXACT FEATURE-v2 CALL SITES"
)

print(
    "=" * 110
)

for item in feature_call_sites:

    print(
        f"\ncell={item['cell']} "
        f"line={item['line']} "
        f"function={item['function']}"
    )

    print(
        item["source"]
    )


# ======================================================================
# 10. Recover other builder/helper definitions from Cell 173
# ======================================================================

defs_173 = []

for node in build_tree_173.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        defs_173.append(
            {
                "name":
                    node.name,

                "type":
                    (
                        "class"
                        if isinstance(
                            node,
                            ast.ClassDef
                        )
                        else "function"
                    ),

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        build_source_173,
                        node
                    ),
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 173 DEFINITIONS"
)

print(
    "=" * 110
)

for item in defs_173:

    first_line = (
        item["source"]
        .splitlines()[0]
        if item["source"]
        else item["name"]
    )

    print(
        f"line={item['line']:<5} "
        f"{item['type']:<10} "
        f"{first_line}"
    )


# ======================================================================
# 11. Print build/helper definitions likely needed for reproduction
# ======================================================================

BUILD_HELPER_TERMS = [
    "semantic",
    "feature",
    "decision",
    "group",
    "cache",
    "encoder",
    "build",
]


print(
    "\n"
    + "=" * 110
)

print(
    "RELEVANT CELL 173 HELPER SOURCES"
)

print(
    "=" * 110
)

for item in defs_173:

    low = item[
        "name"
    ].lower()

    if not any(
        term in low
        for term
        in BUILD_HELPER_TERMS
    ):
        continue

    print(
        "\n"
        + "-" * 100
    )

    print(
        f"{item['type'].upper()} "
        f"{item['name']} "
        f"(line {item['line']})"
    )

    print(
        "-" * 100
    )

    print(
        item[
            "source"
        ]
    )


# ======================================================================
# 12. Static check for encoder normalization / batching behavior
# ======================================================================

combined_relevant_source = "\n".join(
    [
        semantic_source_154,
        build_source_173,
    ]
)

runtime_flags_12cb2 = {
    "uses_sentence_transformer":
        "SentenceTransformer"
        in combined_relevant_source,

    "uses_encode":
        ".encode("
        in combined_relevant_source,

    "normalize_embeddings_true":
        bool(
            re.search(
                r"normalize_embeddings\s*=\s*True",
                combined_relevant_source
            )
        ),

    "normalize_embeddings_false":
        bool(
            re.search(
                r"normalize_embeddings\s*=\s*False",
                combined_relevant_source
            )
        ),

    "batch_size_explicit":
        "batch_size"
        in combined_relevant_source,

    "device_explicit":
        "device"
        in combined_relevant_source,

    "cache_explicit":
        "cache"
        in combined_relevant_source.lower(),
}


print(
    "\n"
    + "=" * 110
)

print(
    "SEMANTIC RUNTIME FLAGS"
)

print(
    "=" * 110
)

for key, value in (
    runtime_flags_12cb2.items()
):
    print(
        f"{key:<35} {value}"
    )


# ======================================================================
# 13. Save static recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_B2_MANIFEST = {
    "cell":
        "RQ2_12C_B2",

    "mode":
        "static_source_recovery",

    "feature_definition_cell":
        CELL_FEATURE,

    "semantic_candidate_cell":
        CELL_SEMANTIC_CANDIDATE,

    "build_candidate_cell":
        CELL_BUILD_CANDIDATE,

    "group_builder_found":
        group_builder_node
        is not None,

    "semantic_definitions":
        [
            {
                "type":
                    x["type"],

                "name":
                    x["name"],

                "line":
                    x["line"],
            }
            for x in semantic_defs_154
        ],

    "feature_call_sites":
        [
            {
                "cell":
                    x["cell"],

                "line":
                    x["line"],

                "function":
                    x["function"],
            }
            for x in feature_call_sites
        ],

    "runtime_flags":
        runtime_flags_12cb2,

    "notebook_code_executed":
        False,

    "model_loaded":
        False,

    "new_features_computed":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning":
        False,

    "test_loaded":
        False,
}


CELL12C_B2_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b2_semantic_runtime_static_recovery.json"
)


with open(
    CELL12C_B2_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_B2_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 14. Final report
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "=== RQ2 CELL 12C-B2: EXACT SEMANTIC RUNTIME STATIC RECOVERY COMPLETE ==="
)

print(
    "=" * 110
)

print(
    "Group Feature-v2 builder found: ",
    group_builder_node is not None
)

print(
    "Cell-154 definitions found:      ",
    len(
        semantic_defs_154
    )
)

print(
    "Feature-build call sites found:  ",
    len(
        feature_call_sites
    )
)

print(
    "Notebook code executed:          NO"
)

print(
    "MiniLM loaded:                   NO"
)

print(
    "New features computed:           NO"
)

print(
    "Pruning run:                     NO"
)

print(
    "Hyperparameter tuning:           NO"
)

print(
    "TEST loaded:                     NO"
)

print(
    "\nManifest:",
    CELL12C_B2_MANIFEST_PATH
)

print(
    "\nNext: instantiate the EXACT recovered semantic encoder on CPU, "
    "rebuild the cached validation decision-group features, and require "
    "numerical equality with the frozen NPZ before enabling online AFP."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Feature cell: 167
Semantic candidate cell: 154
Build candidate cell: 173

EXACT extract_afp_group_features SOURCE
def extract_afp_group_features(
    question_id,
    question,
    plan,
    hop,
    candidate_rows,
    semantic_encoder,
    entity_name_map=None
):

    assert len(
        candidate_rows
    ) > 1


    candidates = []

    for row in candidate_rows:

        candidates.append(
            {
                "prefix_entities":
                    list(
                        row[
                            "prefix_entities"
                        ]
                    ),

                "candidate_entity":
                    row[
                        "candidate_entity"
                    ],

                "parent_prefix_index":
                    int(
                        row[
                            "parent_prefix_index"
                        ]
                    ),
     

In [16]:
# ======================================================================
# LOCATE HOW FrozenMiniLMEncoder WAS DEFINED
# ======================================================================
#
# STATIC ONLY.
# No model loading.
# No feature computation.
# No pruning.
# No tuning.
# No TEST.
# ======================================================================

import ast
import json
import re
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

TARGET = "FrozenMiniLMEncoder"

print("Notebook:", NOTEBOOK_PATH)
print("Cells:", len(nb["cells"]))


# ======================================================================
# 1. Find EVERY textual occurrence
# ======================================================================

text_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    if TARGET not in src:
        continue

    lines = src.splitlines()

    for line_idx, line in enumerate(
        lines
    ):
        if TARGET in line:

            start = max(
                0,
                line_idx - 12
            )

            end = min(
                len(lines),
                line_idx + 20
            )

            context = "\n".join(
                f"{i+1:04d}: {lines[i]}"
                for i in range(
                    start,
                    end
                )
            )

            text_hits.append(
                {
                    "cell":
                        cell_idx,

                    "line":
                        line_idx + 1,

                    "context":
                        context,
                }
            )


print(
    "\n"
    + "=" * 100
)

print(
    "ALL TEXTUAL OCCURRENCES OF FrozenMiniLMEncoder"
)

print(
    "=" * 100
)

print(
    "Total occurrences:",
    len(text_hits)
)

for hit in text_hits:

    print(
        "\n"
        + "-" * 95
    )

    print(
        f"CELL {hit['cell']} "
        f"| LINE {hit['line']}"
    )

    print(
        "-" * 95
    )

    print(
        hit["context"]
    )


# ======================================================================
# 2. AST search for all ways the symbol could be introduced
# ======================================================================

ast_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    try:
        tree = ast.parse(src)
    except Exception:
        continue

    for node in ast.walk(
        tree
    ):

        # ----------------------------------------------------------
        # class FrozenMiniLMEncoder:
        # ----------------------------------------------------------
        if (
            isinstance(node, ast.ClassDef)
            and node.name == TARGET
        ):
            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "ClassDef",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # def FrozenMiniLMEncoder(...):
        # ----------------------------------------------------------
        elif (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                )
            )
            and node.name == TARGET
        ):
            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "FunctionDef",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # FrozenMiniLMEncoder = ...
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.Assign
        ):

            target_names = []

            for target in node.targets:

                if isinstance(
                    target,
                    ast.Name
                ):
                    target_names.append(
                        target.id
                    )

            if TARGET in target_names:

                ast_hits.append(
                    {
                        "cell":
                            cell_idx,

                        "kind":
                            "Assign",

                        "line":
                            getattr(
                                node,
                                "lineno",
                                -1
                            ),

                        "source":
                            ast.get_source_segment(
                                src,
                                node
                            ) or "",
                    }
                )

        # ----------------------------------------------------------
        # FrozenMiniLMEncoder: X = ...
        # ----------------------------------------------------------
        elif (
            isinstance(
                node,
                ast.AnnAssign
            )
            and isinstance(
                node.target,
                ast.Name
            )
            and node.target.id == TARGET
        ):

            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "AnnAssign",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # from x import FrozenMiniLMEncoder
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.ImportFrom
        ):

            for alias in node.names:

                if (
                    alias.name == TARGET
                    or alias.asname == TARGET
                ):

                    ast_hits.append(
                        {
                            "cell":
                                cell_idx,

                            "kind":
                                "ImportFrom",

                            "line":
                                getattr(
                                    node,
                                    "lineno",
                                    -1
                                ),

                            "source":
                                ast.get_source_segment(
                                    src,
                                    node
                                ) or "",
                        }
                    )

        # ----------------------------------------------------------
        # import something as FrozenMiniLMEncoder
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.Import
        ):

            for alias in node.names:

                if (
                    alias.name == TARGET
                    or alias.asname == TARGET
                ):

                    ast_hits.append(
                        {
                            "cell":
                                cell_idx,

                            "kind":
                                "Import",

                            "line":
                                getattr(
                                    node,
                                    "lineno",
                                    -1
                                ),

                            "source":
                                ast.get_source_segment(
                                    src,
                                    node
                                ) or "",
                        }
                    )


print(
    "\n"
    + "=" * 100
)

print(
    "AST DEFINITIONS / IMPORTS / ASSIGNMENTS"
)

print(
    "=" * 100
)

if not ast_hits:
    print("NONE FOUND")

else:

    for hit in ast_hits:

        print(
            f"\ncell={hit['cell']} "
            f"line={hit['line']} "
            f"kind={hit['kind']}"
        )

        print(
            hit["source"]
        )


# ======================================================================
# 3. Search for likely encoder classes even if name differs
# ======================================================================

encoder_like_defs = []

TOKENS = [
    "encoder",
    "embedding",
    "minilm",
    "sentence",
    "transformer",
    "cache",
]


for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    try:
        tree = ast.parse(
            src
        )
    except Exception:
        continue

    for node in tree.body:

        if isinstance(
            node,
            ast.ClassDef
        ):

            name_low = (
                node.name.lower()
            )

            body_src = (
                ast.get_source_segment(
                    src,
                    node
                )
                or ""
            )

            body_low = (
                body_src.lower()
            )

            score = sum(
                token in name_low
                or token in body_low
                for token in TOKENS
            )

            if score > 0:

                encoder_like_defs.append(
                    {
                        "cell":
                            cell_idx,

                        "name":
                            node.name,

                        "line":
                            node.lineno,

                        "score":
                            score,

                        "source":
                            body_src,
                    }
                )


encoder_like_defs = sorted(
    encoder_like_defs,
    key=lambda x: (
        -x["score"],
        x["cell"],
        x["line"],
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "ENCODER-LIKE CLASS DEFINITIONS"
)

print(
    "=" * 100
)

if not encoder_like_defs:
    print("NONE FOUND")

else:

    for item in encoder_like_defs[
        :20
    ]:

        first = (
            item["source"]
            .splitlines()[0]
            if item["source"]
            else item["name"]
        )

        print(
            f"score={item['score']:<3} "
            f"cell={item['cell']:<4} "
            f"line={item['line']:<5} "
            f"{first}"
        )


# ======================================================================
# 4. Search for SentenceTransformer creation / .encode behavior
# ======================================================================

runtime_hits = []

PATTERNS = [
    "SentenceTransformer(",
    ".encode(",
    "normalize_embeddings",
    "convert_to_numpy",
    "show_progress_bar",
    "batch_size",
    "self.cache",
    "self.model",
    "self.dim",
]


for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    matched = [
        p
        for p in PATTERNS
        if p.lower()
        in src.lower()
    ]

    if not matched:
        continue

    runtime_hits.append(
        {
            "cell":
                cell_idx,

            "matched":
                matched,

            "source":
                src,
        }
    )


print(
    "\n"
    + "=" * 100
)

print(
    "SENTENCE-TRANSFORMER / ENCODER RUNTIME CELLS"
)

print(
    "=" * 100
)

for hit in runtime_hits:

    print(
        f"\nCELL {hit['cell']} "
        f"| matched={hit['matched']}"
    )

    lines = hit[
        "source"
    ].splitlines()

    keep = set()

    for i, line in enumerate(
        lines
    ):

        if any(
            pattern.lower()
            in line.lower()
            for pattern
            in PATTERNS
        ):

            for j in range(
                max(0, i - 8),
                min(
                    len(lines),
                    i + 18
                )
            ):
                keep.add(j)

    for j in sorted(
        keep
    ):
        print(
            f"{j+1:04d}: "
            f"{lines[j]}"
        )


# ======================================================================
# 5. Search for dynamic creation via exec/eval
# ======================================================================

dynamic_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    if (
        "exec(" in src
        or "eval(" in src
    ):

        dynamic_hits.append(
            {
                "cell":
                    cell_idx,

                "source":
                    src,
            }
        )


print(
    "\n"
    + "=" * 100
)

print(
    "EXEC / EVAL CELLS"
)

print(
    "=" * 100
)

if not dynamic_hits:
    print("NONE")

else:

    for hit in dynamic_hits:
        print(
            f"\nCELL {hit['cell']}"
        )

        print(
            hit["source"]
        )


# ======================================================================
# 6. Final diagnosis
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== FrozenMiniLMEncoder DEFINITION DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "Text occurrences:            ",
    len(text_hits)
)

print(
    "AST definitions/imports:     ",
    len(ast_hits)
)

print(
    "Encoder-like classes:        ",
    len(encoder_like_defs)
)

print(
    "Runtime encoder cells:       ",
    len(runtime_hits)
)

print(
    "Dynamic exec/eval cells:     ",
    len(dynamic_hits)
)

print(
    "\nMiniLM loaded:               NO"
)

print(
    "Features recomputed:         NO"
)

print(
    "Pruning run:                NO"
)

print(
    "Hyperparameter tuning:      NO"
)

print(
    "TEST loaded:                NO"
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Cells: 190

ALL TEXTUAL OCCURRENCES OF FrozenMiniLMEncoder
Total occurrences: 2

-----------------------------------------------------------------------------------------------
CELL 173 | LINE 75
-----------------------------------------------------------------------------------------------
0063: assert AFP_USE_KGE_CORE is False
0064: 
0065: required_objects = [
0066:     "webqsp_train",
0067:     "cwq_train",
0068:     "webqsp_branch_manifest",
0069:     "cwq_branch_manifest",
0070:     "build_graph",
0071:     "relation_valid_neighbors",
0072:     "call_suffix_dp",
0073:     "dp_is_reachable",
0074:     "extract_afp_group_features",
0075:     "FrozenMiniLMEncoder",
0076: ]
0077: 
0078: missing = [x for x in required_objects if x not in globals()]
0079: 
0080: assert not missing, (
0081:     "Missing required previous-cell objects: "
0082:     + ", ".join(missing)
0083: )
0084: 
0085: print("Feature version:"

In [17]:
# ======================================================================
# BEHAVIORAL RECOVERY OF MINILM RUNTIME
# ======================================================================
#
# FrozenMiniLMEncoder source is absent from the saved notebook.
#
# Therefore we recover its observable behavior using:
#   - exact frozen MiniLM model
#   - exact Feature-v2 extractor from Cell 167
#   - exact frozen validation supervision
#   - frozen validation Feature-v2 NPZ
#
# We compare:
#   A. raw SentenceTransformer embeddings
#   B. L2-normalized SentenceTransformer embeddings
#
# The frozen NPZ decides SOFTWARE FIDELITY only.
#
# NO pruning.
# NO selector tuning.
# NO TEST.
# ======================================================================

import ast
import json
import math
import re
from pathlib import Path
from collections import Counter
from typing import Optional, Dict

import numpy as np
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer


# ======================================================================
# 1. Paths + hard gates
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

RQ2_ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_DIR = (
    RQ2_ROOT
    / "03_features"
)

assert NOTEBOOK_PATH.exists()
assert FEATURE_DIR.exists()

assert len(webqsp_val_plan_rows) == 246
assert len(cwq_val_plan_rows) == 3519

print("Notebook:", NOTEBOOK_PATH)
print("Runtime device: CPU")


# ======================================================================
# 2. Recover exact Feature-v2 functions from Cell 167
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

FEATURE_CELL = 167

feature_source = "".join(
    nb["cells"][FEATURE_CELL]["source"]
)

tree = ast.parse(
    feature_source
)

REQUIRED_FUNCTIONS = [
    "is_raw_freebase_id",
    "has_readable_entity_surface",
    "normalize_surface_text",
    "relation_surface_text",
    "entity_surface_text",
    "relation_sequence_text",
    "safe_l2_normalize",
    "safe_cosine",
    "mean_embedding",
    "get_safe_entity_embedding",
    "extract_afp_candidate_features",
    "extract_afp_group_features",
]

nodes = {}

for node in tree.body:
    if (
        isinstance(node, ast.FunctionDef)
        and node.name in REQUIRED_FUNCTIONS
    ):
        nodes[node.name] = node

missing = [
    name
    for name in REQUIRED_FUNCTIONS
    if name not in nodes
]

assert not missing, (
    "Missing Feature-v2 functions: "
    + str(missing)
)


feature_ns = {
    "np": np,
    "math": math,
    "re": re,
    "Counter": Counter,
    "Optional": Optional,
    "Dict": Dict,

    "AFP_FEATURE_DIM": 27,

    "FREEBASE_ID_RE":
        re.compile(
            r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
        ),

    "__name__":
        "recovered_feature_v2",
}

module = ast.Module(
    body=[
        nodes[name]
        for name in REQUIRED_FUNCTIONS
    ],
    type_ignores=[]
)

ast.fix_missing_locations(
    module
)

exec(
    compile(
        module,
        filename="recovered_feature_v2",
        mode="exec"
    ),
    feature_ns
)


extract_group = (
    feature_ns[
        "extract_afp_group_features"
    ]
)

relation_surface_text = (
    feature_ns[
        "relation_surface_text"
    ]
)

relation_sequence_text = (
    feature_ns[
        "relation_sequence_text"
    ]
)

has_readable_entity_surface = (
    feature_ns[
        "has_readable_entity_surface"
    ]
)

entity_surface_text = (
    feature_ns[
        "entity_surface_text"
    ]
)

safe_l2_normalize = (
    feature_ns[
        "safe_l2_normalize"
    ]
)

print(
    "Exact Feature-v2 extractor recovery: PASSED"
)


# ======================================================================
# 3. Frozen artifacts
# ======================================================================

WEBQSP_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels.jsonl"
)

CWQ_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels.jsonl"
)

WEBQSP_NPZ = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2.npz"
)

CWQ_NPZ = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2.npz"
)

for path in [
    WEBQSP_LABEL_FILE,
    CWQ_LABEL_FILE,
    WEBQSP_NPZ,
    CWQ_NPZ,
]:
    assert path.exists()


# ======================================================================
# 4. Exact decision-group iterator
# ======================================================================

def iter_groups(labels_file):

    current_id = None
    current_rows = []

    with open(
        labels_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(
                line
            )

            if (
                row.get(
                    "decision_opportunity",
                    True
                )
                is False
            ):
                continue

            gid = row[
                "group_id"
            ]

            if (
                current_id is not None
                and gid != current_id
            ):
                assert len(
                    current_rows
                ) > 1

                yield current_rows

                current_rows = []

            current_id = gid

            current_rows.append(
                row
            )

    if current_rows:

        assert len(
            current_rows
        ) > 1

        yield current_rows


# ======================================================================
# 5. Collect exact validation semantic inventory
# ======================================================================

def collect_semantic_inventory(
    labels_file,
    dataset_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    groups = 0
    branches = 0

    for rows in iter_groups(
        labels_file
    ):

        groups += 1
        branches += len(rows)

        first = rows[0]

        source_index = int(
            first["source_index"]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first["question_id"]
        )

        qid = str(
            first["question_id"]
        )

        questions[qid] = (
            rec["question"]
        )

        plan = list(
            first["plan"]
        )

        hop = int(
            first["hop"]
        )

        for relation in plan:

            relations[
                str(relation)
            ] = relation_surface_text(
                relation
            )

        plan_key = "||".join(
            str(x)
            for x in plan
        )

        plans[
            plan_key
        ] = relation_sequence_text(
            plan
        )

        suffix = plan[
            hop + 1:
        ]

        suffix_key = "||".join(
            str(x)
            for x in suffix
        )

        suffixes[
            suffix_key
        ] = relation_sequence_text(
            suffix
        )

        for row in rows:

            candidate = row[
                "candidate_entity"
            ]

            if has_readable_entity_surface(
                candidate
            ):

                entities[
                    str(candidate)
                ] = entity_surface_text(
                    candidate
                )

            for entity in row[
                "prefix_entities"
            ]:

                if has_readable_entity_surface(
                    entity
                ):

                    entities[
                        str(entity)
                    ] = entity_surface_text(
                        entity
                    )

    return {
        "questions": questions,
        "entities": entities,
        "relations": relations,
        "plans": plans,
        "suffixes": suffixes,
        "groups": groups,
        "branches": branches,
    }


print(
    "\nCollecting validation semantic inventory..."
)

wq_inventory = (
    collect_semantic_inventory(
        WEBQSP_LABEL_FILE,
        webqsp_val_plan_rows
    )
)

cwq_inventory = (
    collect_semantic_inventory(
        CWQ_LABEL_FILE,
        cwq_val_plan_rows
    )
)


def merge_maps(*maps):

    out = {}

    for mapping in maps:
        out.update(
            mapping
        )

    return out


ALL_QUESTIONS = merge_maps(
    wq_inventory["questions"],
    cwq_inventory["questions"]
)

ALL_ENTITIES = merge_maps(
    wq_inventory["entities"],
    cwq_inventory["entities"]
)

ALL_RELATIONS = merge_maps(
    wq_inventory["relations"],
    cwq_inventory["relations"]
)

ALL_PLANS = merge_maps(
    wq_inventory["plans"],
    cwq_inventory["plans"]
)

ALL_SUFFIXES = merge_maps(
    wq_inventory["suffixes"],
    cwq_inventory["suffixes"]
)


print("\nInventory")
print(
    "  Questions:         ",
    len(ALL_QUESTIONS)
)
print(
    "  Readable entities: ",
    len(ALL_ENTITIES)
)
print(
    "  Relations:         ",
    len(ALL_RELATIONS)
)
print(
    "  Plans:             ",
    len(ALL_PLANS)
)
print(
    "  Suffixes:          ",
    len(ALL_SUFFIXES)
)


# ======================================================================
# 6. Load exact frozen MiniLM model
# ======================================================================

MODEL_NAME = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

print(
    "\nLoading:",
    MODEL_NAME
)

minilm_model = (
    SentenceTransformer(
        MODEL_NAME,
        device="cpu"
    )
)

MINILM_DIM = int(
    minilm_model
    .get_sentence_embedding_dimension()
)

assert MINILM_DIM == 384

print(
    "Embedding dimension:",
    MINILM_DIM
)


# ======================================================================
# 7. Encode raw embeddings exactly once
# ======================================================================
#
# Original Cell 173 used:
#   batch_size = 256
#
# We preserve category separation:
#   question
#   entity
#   relation
#   plan
#   suffix
#
# normalize_embeddings=False gives the raw SentenceTransformer output.
# The second candidate behavior is obtained by L2-normalizing these
# same vectors, so MiniLM inference is performed only once.
# ======================================================================

RAW_CACHE = {}


def prefill_raw(
    namespace,
    mapping
):

    items = list(
        mapping.items()
    )

    if not items:
        return

    identifiers = [
        str(k)
        for k, _
        in items
    ]

    texts = [
        str(v)
        for _, v
        in items
    ]

    print(
        f"Encoding {namespace:<10}: "
        f"{len(texts)}"
    )

    embeddings = (
        minilm_model.encode(
            texts,
            batch_size=256,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=False
        )
    )

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )

    assert embeddings.shape == (
        len(texts),
        384
    )

    for identifier, vector in zip(
        identifiers,
        embeddings
    ):

        RAW_CACHE[
            (
                namespace,
                identifier
            )
        ] = np.asarray(
            vector,
            dtype=np.float32
        )


print(
    "\nBuilding raw validation semantic cache..."
)

prefill_raw(
    "question",
    ALL_QUESTIONS
)

prefill_raw(
    "entity",
    ALL_ENTITIES
)

prefill_raw(
    "relation",
    ALL_RELATIONS
)

prefill_raw(
    "plan",
    ALL_PLANS
)

prefill_raw(
    "suffix",
    ALL_SUFFIXES
)

print(
    "\nRaw cache entries:",
    len(RAW_CACHE)
)


# ======================================================================
# 8. Behavioral encoder wrapper
# ======================================================================

class RecoveredMiniLMEncoder:

    def __init__(
        self,
        model,
        raw_cache,
        vector_mode
    ):
        assert vector_mode in {
            "raw",
            "unit_normalized",
        }

        self.model = model
        self.raw_cache = raw_cache
        self.vector_mode = (
            vector_mode
        )

        self.dim = 384

        # Runtime cache follows expected API.
        self.cache = {}


    def _transform(
        self,
        vector
    ):
        vector = np.asarray(
            vector,
            dtype=np.float32
        )

        if (
            self.vector_mode
            == "raw"
        ):
            return vector.copy()

        return safe_l2_normalize(
            vector
        )


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        key = (
            str(namespace),
            str(identifier)
        )

        if key in self.cache:
            return self.cache[
                key
            ]

        # Prefer precomputed raw vector.
        if key in self.raw_cache:

            raw = self.raw_cache[
                key
            ]

        else:
            # This path will later support dynamic frontiers.
            raw = self.model.encode(
                [str(raw_text)],
                batch_size=1,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            )[0]

            raw = np.asarray(
                raw,
                dtype=np.float32
            )

            self.raw_cache[
                key
            ] = raw

        output = self._transform(
            raw
        )

        self.cache[
            key
        ] = output

        return output


encoder_raw = (
    RecoveredMiniLMEncoder(
        model=minilm_model,
        raw_cache=RAW_CACHE,
        vector_mode="raw"
    )
)

encoder_unit = (
    RecoveredMiniLMEncoder(
        model=minilm_model,
        raw_cache=RAW_CACHE,
        vector_mode=
            "unit_normalized"
    )
)


# ======================================================================
# 9. Rebuild validation feature matrix
# ======================================================================

def rebuild_features(
    dataset_name,
    dataset_rows,
    labels_file,
    encoder
):

    blocks = []

    group_ptr = [0]

    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []

    for rows in tqdm(
        iter_groups(
            labels_file
        ),
        desc=(
            f"{dataset_name} "
            f"{encoder.vector_mode}"
        )
    ):

        first = rows[0]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first[
                "question_id"
            ]
        )

        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )

        X_group = extract_group(
            question_id=
                first[
                    "question_id"
                ],

            question=
                rec[
                    "question"
                ],

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                rows,

            semantic_encoder=
                encoder,

            entity_name_map=
                None
        )

        assert X_group.shape == (
            len(rows),
            27
        )

        blocks.append(
            X_group
        )

        group_ptr.append(
            group_ptr[-1]
            + len(rows)
        )

        group_source_index.append(
            source_index
        )

        group_hop.append(
            hop
        )

        group_plan_length.append(
            len(plan)
        )

        group_candidate_count.append(
            len(rows)
        )

    return {
        "X":
            np.concatenate(
                blocks,
                axis=0
            ).astype(
                np.float32
            ),

        "group_ptr":
            np.asarray(
                group_ptr,
                dtype=np.int64
            ),

        "group_source_index":
            np.asarray(
                group_source_index,
                dtype=np.int32
            ),

        "group_hop":
            np.asarray(
                group_hop,
                dtype=np.int16
            ),

        "group_plan_length":
            np.asarray(
                group_plan_length,
                dtype=np.int16
            ),

        "group_candidate_count":
            np.asarray(
                group_candidate_count,
                dtype=np.int32
            ),
    }


# ======================================================================
# 10. Frozen arrays
# ======================================================================

def load_npz(path):

    z = np.load(
        path,
        allow_pickle=False
    )

    return {
        k: z[k]
        for k in z.files
    }


wq_frozen = load_npz(
    WEBQSP_NPZ
)

cwq_frozen = load_npz(
    CWQ_NPZ
)


# ======================================================================
# 11. Rebuild BOTH plausible runtime behaviors
# ======================================================================

print(
    "\nRebuilding RAW behavior..."
)

wq_raw = rebuild_features(
    "webqsp",
    webqsp_val_plan_rows,
    WEBQSP_LABEL_FILE,
    encoder_raw
)

cwq_raw = rebuild_features(
    "cwq",
    cwq_val_plan_rows,
    CWQ_LABEL_FILE,
    encoder_raw
)


print(
    "\nRebuilding UNIT-NORMALIZED behavior..."
)

wq_unit = rebuild_features(
    "webqsp",
    webqsp_val_plan_rows,
    WEBQSP_LABEL_FILE,
    encoder_unit
)

cwq_unit = rebuild_features(
    "cwq",
    cwq_val_plan_rows,
    CWQ_LABEL_FILE,
    encoder_unit
)


# ======================================================================
# 12. Metadata fidelity
# ======================================================================

META_KEYS = [
    "group_ptr",
    "group_source_index",
    "group_hop",
    "group_plan_length",
    "group_candidate_count",
]


def assert_metadata(
    dataset,
    rebuilt,
    frozen
):

    for key in META_KEYS:

        assert np.array_equal(
            rebuilt[key],
            frozen[key]
        ), (
            f"{dataset}: "
            f"metadata mismatch {key}"
        )


assert_metadata(
    "WebQSP/raw",
    wq_raw,
    wq_frozen
)

assert_metadata(
    "WebQSP/unit",
    wq_unit,
    wq_frozen
)

assert_metadata(
    "CWQ/raw",
    cwq_raw,
    cwq_frozen
)

assert_metadata(
    "CWQ/unit",
    cwq_unit,
    cwq_frozen
)

print(
    "\nMetadata fidelity: PASSED"
)


# ======================================================================
# 13. Numerical comparison
# ======================================================================

SEMANTIC_COLUMNS = [
    0, 2, 4, 5, 6, 7, 8, 9
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(27)
    if i not in SEMANTIC_COLUMNS
]


def compare_features(
    rebuilt,
    frozen
):

    A = np.asarray(
        rebuilt["X"],
        dtype=np.float64
    )

    B = np.asarray(
        frozen["X"],
        dtype=np.float64
    )

    assert A.shape == B.shape

    diff = np.abs(
        A - B
    )

    sem = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym = diff[
        :,
        SYMBOLIC_COLUMNS
    ]

    return {
        "max_all":
            float(
                diff.max()
            ),

        "mean_all":
            float(
                diff.mean()
            ),

        "max_semantic":
            float(
                sem.max()
            ),

        "mean_semantic":
            float(
                sem.mean()
            ),

        "max_symbolic":
            float(
                sym.max()
            ),

        "mean_symbolic":
            float(
                sym.mean()
            ),
    }


results = {
    "webqsp_raw":
        compare_features(
            wq_raw,
            wq_frozen
        ),

    "webqsp_unit":
        compare_features(
            wq_unit,
            wq_frozen
        ),

    "cwq_raw":
        compare_features(
            cwq_raw,
            cwq_frozen
        ),

    "cwq_unit":
        compare_features(
            cwq_unit,
            cwq_frozen
        ),
}


print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 BEHAVIORAL FIDELITY"
)

print(
    "=" * 100
)

for name, values in (
    results.items()
):

    print(
        f"\n{name}"
    )

    for key, value in (
        values.items()
    ):

        print(
            f"  {key:<18} "
            f"{value:.10g}"
        )


# ======================================================================
# 14. Determine which embedding behavior matches frozen features
# ======================================================================

raw_score = (
    results[
        "webqsp_raw"
    ][
        "mean_semantic"
    ]
    +
    results[
        "cwq_raw"
    ][
        "mean_semantic"
    ]
)

unit_score = (
    results[
        "webqsp_unit"
    ][
        "mean_semantic"
    ]
    +
    results[
        "cwq_unit"
    ][
        "mean_semantic"
    ]
)


if raw_score <= unit_score:

    SELECTED_VECTOR_MODE = (
        "raw"
    )

    AFP_RUNTIME_SEMANTIC_ENCODER = (
        encoder_raw
    )

    selected_results = [
        results["webqsp_raw"],
        results["cwq_raw"],
    ]

else:

    SELECTED_VECTOR_MODE = (
        "unit_normalized"
    )

    AFP_RUNTIME_SEMANTIC_ENCODER = (
        encoder_unit
    )

    selected_results = [
        results["webqsp_unit"],
        results["cwq_unit"],
    ]


print(
    "\nSelected runtime behavior:",
    SELECTED_VECTOR_MODE
)


# ======================================================================
# 15. Strict software-fidelity gate
# ======================================================================

SYMBOLIC_ATOL = 1e-7

# CPU vs original CUDA MiniLM may differ slightly.
SEMANTIC_ATOL = 5e-5


for result in selected_results:

    assert (
        result[
            "max_symbolic"
        ]
        <= SYMBOLIC_ATOL
    ), (
        "Symbolic Feature-v2 reproduction failed."
    )

    assert (
        result[
            "max_semantic"
        ]
        <= SEMANTIC_ATOL
    ), (
        "Semantic Feature-v2 reproduction failed. "
        "Do NOT increase tolerance automatically."
    )


print(
    "Feature-v2 behavioral fidelity: PASSED"
)


# ======================================================================
# 16. Difference between the two candidate behaviors
# ======================================================================

RAW_UNIT_WEBQSP_MAX = float(
    np.max(
        np.abs(
            wq_raw["X"]
            -
            wq_unit["X"]
        )
    )
)

RAW_UNIT_CWQ_MAX = float(
    np.max(
        np.abs(
            cwq_raw["X"]
            -
            cwq_unit["X"]
        )
    )
)

print(
    "\nRaw vs unit-normalized feature difference"
)

print(
    "  WebQSP max:",
    RAW_UNIT_WEBQSP_MAX
)

print(
    "  CWQ max:   ",
    RAW_UNIT_CWQ_MAX
)


# ======================================================================
# 17. Freeze recovered runtime objects for B4
# ======================================================================

AFP_RUNTIME_FEATURE_EXTRACTOR = (
    extract_group
)

AFP_RUNTIME_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

AFP_RUNTIME_FEATURE_DIM = 27

AFP_RUNTIME_MINILM_MODEL = (
    minilm_model
)

AFP_RUNTIME_VECTOR_MODE = (
    SELECTED_VECTOR_MODE
)


# ======================================================================
# 18. Save recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest = {
    "cell":
        "RQ2_12C_B3_R",

    "recovery_reason":
        (
            "FrozenMiniLMEncoder definition absent "
            "from persisted notebook source"
        ),

    "recovery_type":
        "behavioral_feature_reproduction",

    "semantic_model":
        MODEL_NAME,

    "runtime_device":
        "cpu",

    "batch_size":
        256,

    "candidate_vector_modes": [
        "raw",
        "unit_normalized",
    ],

    "selected_vector_mode":
        SELECTED_VECTOR_MODE,

    "webqsp_raw":
        results[
            "webqsp_raw"
        ],

    "webqsp_unit":
        results[
            "webqsp_unit"
        ],

    "cwq_raw":
        results[
            "cwq_raw"
        ],

    "cwq_unit":
        results[
            "cwq_unit"
        ],

    "raw_unit_webqsp_max":
        RAW_UNIT_WEBQSP_MAX,

    "raw_unit_cwq_max":
        RAW_UNIT_CWQ_MAX,

    "symbolic_atol":
        SYMBOLIC_ATOL,

    "semantic_atol":
        SEMANTIC_ATOL,

    "feature_fidelity_passed":
        True,

    "software_fidelity_only":
        True,

    "validation_labels_used_for_model_selection":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "complete_afp_frozen":
        False,
}


manifest_path = (
    TRAVERSAL_DIR
    / "cell12c_b3_behavioral_minilm_recovery.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 19. Final report
# ======================================================================

print(
    "\n"
    + "=" * 106
)

print(
    "=== RQ2 CELL 12C-B3-R: MINILM RUNTIME BEHAVIOR RECOVERED ==="
)

print(
    "=" * 106
)

print(
    "FrozenMiniLMEncoder source:     MISSING"
)

print(
    "Recovery basis:                 FROZEN Feature-v2 NPZ"
)

print(
    "Semantic model:                 ",
    MODEL_NAME
)

print(
    "Selected embedding behavior:    ",
    SELECTED_VECTOR_MODE
)

print(
    "Feature-v2 fidelity:            PASS"
)

print(
    "Runtime extractor ready:        YES"
)

print(
    "Runtime semantic encoder ready: YES"
)

print(
    "\nPruning run:                    NO"
)

print(
    "Hyperparameter tuning:          NO"
)

print(
    "TEST loaded:                    NO"
)

print(
    "Complete AFP frozen:            NO"
)

print(
    "\nManifest:",
    manifest_path
)

print(
    "\nNext: Cell 12C-B4 — attach train-only standardizer + "
    "selected scorer checkpoint, verify online logits, then "
    "Cell 13 hyperparameter tuning."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Runtime device: CPU
Exact Feature-v2 extractor recovery: PASSED


Inventory
  Questions:          882
  Readable entities:  2637
  Relations:          274
  Plans:              292
  Suffixes:           190

Loading: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Rebuilding UNIT-NORMALIZED behavior...


webqsp unit_normalized: 0it [00:00, ?it/s]

cwq unit_normalized: 0it [00:00, ?it/s]


Metadata fidelity: PASSED

FEATURE-v2 BEHAVIORAL FIDELITY

webqsp_raw
  max_all            3.576278687e-07
  mean_all           1.56384136e-08
  max_semantic       3.576278687e-07
  mean_semantic      5.277964589e-08
  max_symbolic       0
  mean_symbolic      0

webqsp_unit
  max_all            3.576278687e-07
  mean_all           1.56059912e-08
  max_semantic       3.576278687e-07
  mean_semantic      5.267022031e-08
  max_symbolic       0
  mean_symbolic      0

cwq_raw
  max_all            0.4881388545
  mean_all           0.0001131997791
  max_semantic       0.4881388545
  mean_semantic      0.0003820492545
  max_symbolic       0
  mean_symbolic      0

cwq_unit
  max_all            0.4881388545
  mean_all           0.0001131997428
  max_semantic       0.4881388545
  mean_semantic      0.0003820491321
  max_symbolic       0
  mean_symbolic      0

Selected runtime behavior: unit_normalized


AssertionError: Semantic Feature-v2 reproduction failed. Do NOT increase tolerance automatically.

In [18]:
# ======================================================================
# 3.12C-B3-DIAG LOCALIZE CWQ SEMANTIC FIDELITY FAILURE
# ======================================================================
#
# Uses existing:
#   cwq_unit
#   cwq_frozen
#   cwq_val_plan_rows
#
# No MiniLM rerun.
# No pruning.
# No tuning.
# No TEST.
# ======================================================================

import numpy as np
import pandas as pd


FEATURE_NAMES = [
    "sem_q_candidate",                     # 0
    "sem_candidate_surface_available",     # 1
    "sem_q_current_entity",                # 2
    "sem_current_surface_available",       # 3
    "sem_q_current_relation",              # 4
    "sem_q_full_plan",                     # 5
    "sem_q_remaining_suffix",              # 6
    "sem_candidate_current_relation",      # 7
    "path_q_prefix_entity_mean",            # 8
    "path_candidate_prefix_entity_mean",    # 9
    "path_prefix_surface_fraction",         # 10
    "path_candidate_repeats_entity",        # 11
    "path_candidate_occurrence_fraction",   # 12
    "path_unique_entity_ratio",             # 13
    "path_relation_repeat_fraction_before", # 14
    "struct_log_candidate_count",           # 15
    "struct_log_unique_candidate_entities", # 16
    "struct_log_contributing_parents",      # 17
    "struct_log_parent_fanout",             # 18
    "struct_parent_frontier_share",         # 19
    "struct_log_endpoint_multiplicity",     # 20
    "struct_endpoint_frontier_share",       # 21
    "struct_duplicate_endpoint_ratio",      # 22
    "prog_hop_fraction",                    # 23
    "prog_remaining_fraction",              # 24
    "prog_log_plan_length",                 # 25
    "prog_penultimate_indicator",           # 26
]

assert len(FEATURE_NAMES) == 27

A = np.asarray(
    cwq_unit["X"],
    dtype=np.float64
)

B = np.asarray(
    cwq_frozen["X"],
    dtype=np.float64
)

assert A.shape == B.shape

diff = np.abs(A - B)

FIDELITY_ATOL = 5e-5


# ======================================================================
# 1. Global discrepancy summary
# ======================================================================

bad_mask = diff > FIDELITY_ATOL

bad_rows = np.flatnonzero(
    np.any(
        bad_mask,
        axis=1
    )
)

bad_cells = np.argwhere(
    bad_mask
)

print("=" * 100)
print("CWQ SEMANTIC FIDELITY FAILURE LOCALIZATION")
print("=" * 100)

print("Feature matrix shape:       ", A.shape)
print("Bad feature cells (>5e-5):  ", len(bad_cells))
print("Bad branch rows:             ", len(bad_rows))
print(
    "Bad branch-row rate:        ",
    f"{100 * len(bad_rows) / len(A):.4f}%"
)
print("Maximum difference:          ", diff.max())


# ======================================================================
# 2. Which feature columns are failing?
# ======================================================================

column_rows = []

for j, name in enumerate(FEATURE_NAMES):

    col_diff = diff[:, j]

    count = int(
        np.sum(
            col_diff > FIDELITY_ATOL
        )
    )

    column_rows.append(
        {
            "feature_index": j,
            "feature": name,
            "bad_values": count,
            "max_abs_diff":
                float(
                    col_diff.max()
                ),
            "mean_abs_diff":
                float(
                    col_diff.mean()
                ),
        }
    )


column_df = pd.DataFrame(
    column_rows
)

column_df = column_df[
    column_df["bad_values"] > 0
].sort_values(
    [
        "bad_values",
        "max_abs_diff",
    ],
    ascending=False
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILING FEATURE COLUMNS"
)

print(
    "=" * 100
)

if len(column_df) == 0:
    print("NONE")
else:
    print(
        column_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.9f}"
        )
    )


# ======================================================================
# 3. Map branch rows -> decision groups
# ======================================================================

ptr = np.asarray(
    cwq_frozen[
        "group_ptr"
    ],
    dtype=np.int64
)

source_indices = np.asarray(
    cwq_frozen[
        "group_source_index"
    ],
    dtype=np.int64
)

hops = np.asarray(
    cwq_frozen[
        "group_hop"
    ],
    dtype=np.int64
)

plan_lengths = np.asarray(
    cwq_frozen[
        "group_plan_length"
    ],
    dtype=np.int64
)

candidate_counts = np.asarray(
    cwq_frozen[
        "group_candidate_count"
    ],
    dtype=np.int64
)


def branch_to_group(row_idx):
    return int(
        np.searchsorted(
            ptr[1:],
            row_idx,
            side="right"
        )
    )


bad_groups = sorted(
    set(
        branch_to_group(i)
        for i in bad_rows
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "AFFECTED GROUPS"
)

print(
    "=" * 100
)

print(
    "Bad groups:",
    len(bad_groups),
    "/",
    len(ptr) - 1,
    "=",
    f"{100 * len(bad_groups)/(len(ptr)-1):.4f}%"
)


group_summary = []

for g in bad_groups:

    s = int(ptr[g])
    e = int(ptr[g + 1])

    group_diff = diff[
        s:e
    ]

    source_index = int(
        source_indices[g]
    )

    rec_plan = (
        cwq_val_plan_rows[
            source_index
        ]
    )

    group_summary.append(
        {
            "group_index":
                g,

            "source_index":
                source_index,

            "question_id":
                str(
                    rec_plan.get(
                        "id",
                        ""
                    )
                ),

            "hop":
                int(
                    hops[g]
                ),

            "plan_length":
                int(
                    plan_lengths[g]
                ),

            "candidate_count":
                int(
                    candidate_counts[g]
                ),

            "max_abs_diff":
                float(
                    group_diff.max()
                ),

            "bad_cells":
                int(
                    np.sum(
                        group_diff
                        > FIDELITY_ATOL
                    )
                ),
        }
    )


group_df = pd.DataFrame(
    group_summary
).sort_values(
    "max_abs_diff",
    ascending=False
)


print(
    group_df.head(
        50
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ======================================================================
# 4. Inspect the largest individual differences
# ======================================================================

flat_order = np.argsort(
    diff.ravel()
)[::-1]

print(
    "\n"
    + "=" * 110
)

print(
    "TOP 30 INDIVIDUAL FEATURE DIFFERENCES"
)

print(
    "=" * 110
)

shown = 0

for flat_idx in flat_order:

    row_idx, feature_idx = (
        np.unravel_index(
            flat_idx,
            diff.shape
        )
    )

    d = float(
        diff[
            row_idx,
            feature_idx
        ]
    )

    if d <= FIDELITY_ATOL:
        break

    g = branch_to_group(
        row_idx
    )

    src_idx = int(
        source_indices[g]
    )

    qid = str(
        cwq_val_plan_rows[
            src_idx
        ].get(
            "id",
            ""
        )
    )

    print(
        f"row={row_idx:<6} "
        f"group={g:<5} "
        f"source={src_idx:<5} "
        f"feature={feature_idx:02d} "
        f"{FEATURE_NAMES[feature_idx]:<38} "
        f"rebuilt={A[row_idx, feature_idx]: .7f} "
        f"frozen={B[row_idx, feature_idx]: .7f} "
        f"diff={d:.7f} "
        f"id={qid}"
    )

    shown += 1

    if shown >= 30:
        break


# ======================================================================
# 5. Diagnostic pattern
# ======================================================================

QUESTION_DEPENDENT = {
    0,  # q-candidate
    2,  # q-current entity
    4,  # q-current relation
    5,  # q-full plan
    6,  # q-suffix
    8,  # q-prefix mean
}

NONQUESTION_SEMANTIC = {
    7,  # candidate-current relation
    9,  # candidate-prefix mean
}


bad_feature_indices = set(
    int(x[1])
    for x in bad_cells
)

question_only_failure = (
    len(bad_feature_indices) > 0
    and bad_feature_indices.issubset(
        QUESTION_DEPENDENT
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILURE PATTERN"
)

print(
    "=" * 100
)

print(
    "Failing feature indices:",
    sorted(
        bad_feature_indices
    )
)

print(
    "Only question-dependent semantic features fail:",
    question_only_failure
)

print(
    "Candidate/entity-only semantic failures:",
    sorted(
        bad_feature_indices
        & NONQUESTION_SEMANTIC
    )
)


# ======================================================================
# 6. Compare actual validation dataset question text if available
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CWQ DATASET vs FROZEN PLAN QUESTION TEXT CHECK"
)

print(
    "=" * 100
)


if "cwq_val" not in globals():

    print(
        "cwq_val object is not currently loaded."
    )

    print(
        "Cannot yet compare the original validation dataset text "
        "against frozen planning-row text."
    )

else:

    assert len(
        cwq_val
    ) == len(
        cwq_val_plan_rows
    )

    question_mismatches = []

    for i in range(
        len(cwq_val)
    ):

        dataset_id = str(
            cwq_val[i]["id"]
        )

        plan_id = str(
            cwq_val_plan_rows[i]["id"]
        )

        assert dataset_id == plan_id

        q_dataset = str(
            cwq_val[i][
                "question"
            ]
        )

        q_plan = str(
            cwq_val_plan_rows[i][
                "question"
            ]
        )

        if q_dataset != q_plan:

            question_mismatches.append(
                {
                    "source_index":
                        i,

                    "id":
                        dataset_id,

                    "dataset_question":
                        q_dataset,

                    "plan_question":
                        q_plan,

                    "dataset_repr":
                        repr(
                            q_dataset
                        ),

                    "plan_repr":
                        repr(
                            q_plan
                        ),
                }
            )


    print(
        "Exact question-text mismatches:",
        len(
            question_mismatches
        )
    )


    bad_source_set = set(
        int(
            source_indices[g]
        )
        for g in bad_groups
    )

    mismatch_source_set = set(
        x[
            "source_index"
        ]
        for x in question_mismatches
    )


    print(
        "Bad feature source indices:",
        len(
            bad_source_set
        )
    )

    print(
        "Question-mismatch source indices:",
        len(
            mismatch_source_set
        )
    )

    print(
        "Intersection:",
        len(
            bad_source_set
            & mismatch_source_set
        )
    )


    if question_mismatches:

        print(
            "\nFirst question-text mismatches:"
        )

        for item in question_mismatches[
            :20
        ]:

            marker = (
                " <-- FEATURE FAILURE"
                if item[
                    "source_index"
                ]
                in bad_source_set
                else ""
            )

            print(
                "\nsource_index=",
                item[
                    "source_index"
                ],
                " id=",
                item[
                    "id"
                ],
                marker,
                sep=""
            )

            print(
                "dataset:",
                item[
                    "dataset_repr"
                ]
            )

            print(
                "plan:   ",
                item[
                    "plan_repr"
                ]
            )


# ======================================================================
# 7. Final diagnosis summary
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== CWQ FEATURE FIDELITY DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "Bad branch rows:              ",
    len(
        bad_rows
    )
)

print(
    "Bad decision groups:          ",
    len(
        bad_groups
    )
)

print(
    "Maximum semantic difference:  ",
    float(
        diff.max()
    )
)

print(
    "Question-only failure pattern:",
    question_only_failure
)

print(
    "\nNo tolerance changed."
)

print(
    "No MiniLM rerun."
)

print(
    "No pruning."
)

print(
    "No hyperparameter tuning."
)

print(
    "No TEST."
)

CWQ SEMANTIC FIDELITY FAILURE LOCALIZATION
Feature matrix shape:        (18688, 27)
Bad feature cells (>5e-5):   4754
Bad branch rows:              839
Bad branch-row rate:         4.4895%
Maximum difference:           0.4881388545036316

FAILING FEATURE COLUMNS
 feature_index                   feature  bad_values  max_abs_diff  mean_abs_diff
             2      sem_q_current_entity         839   0.274980783    0.000811751
             8 path_q_prefix_entity_mean         839   0.274980783    0.000793556
             6    sem_q_remaining_suffix         839   0.040323436    0.000360753
             4    sem_q_current_relation         839   0.037393421    0.000351435
             5           sem_q_full_plan         804   0.034087270    0.000334386
             0           sem_q_candidate         594   0.488138855    0.000404427

AFFECTED GROUPS
Bad groups: 70 / 1352 = 5.1775%
 group_index  source_index                                    question_id  hop  plan_length  candidate_count  max_

In [20]:
# ======================================================================
# FAST CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT
# ======================================================================
#
# Uses only id + question columns.
# Does NOT materialize the huge graph field row-by-row.
# CPU only.
# ======================================================================

from collections import defaultdict

question_versions = defaultdict(list)

print("\nReading CWQ TRAIN id/question columns...")

train_ids = cwq_train_exact["id"]
train_questions = cwq_train_exact["question"]

assert len(train_ids) == len(train_questions)
assert len(train_ids) == 27639

for i, (qid, q) in enumerate(
    zip(
        train_ids,
        train_questions
    )
):
    question_versions[
        str(qid)
    ].append(
        (
            "train",
            i,
            str(q)
        )
    )


print("Reading CWQ VALIDATION id/question columns...")

val_ids = cwq_val_exact["id"]
val_questions = cwq_val_exact["question"]

assert len(val_ids) == len(val_questions)
assert len(val_ids) == 3519

for i, (qid, q) in enumerate(
    zip(
        val_ids,
        val_questions
    )
):
    question_versions[
        str(qid)
    ].append(
        (
            "validation",
            i,
            str(q)
        )
    )


# --------------------------------------------------------------
# Find IDs associated with multiple DISTINCT question strings
# --------------------------------------------------------------

collision_rows = []

for qid, versions in question_versions.items():

    unique_texts = {
        x[2]
        for x in versions
    }

    if len(unique_texts) <= 1:
        continue

    val_sources = [
        x[1]
        for x in versions
        if x[0] == "validation"
    ]

    collision_rows.append(
        {
            "question_id":
                qid,

            "n_occurrences":
                len(versions),

            "n_unique_texts":
                len(unique_texts),

            "validation_source_indices":
                val_sources,

            "versions":
                versions,
        }
    )


collision_val_sources = set()

for item in collision_rows:

    collision_val_sources.update(
        int(x)
        for x in item[
            "validation_source_indices"
        ]
    )


print(
    "\n"
    + "=" * 105
)

print(
    "CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT"
)

print(
    "=" * 105
)

print(
    "IDs with >1 distinct question text:",
    len(collision_rows)
)

print(
    "Validation source indices involved:",
    len(collision_val_sources)
)

print(
    "Intersection with failed feature sources:",
    len(
        collision_val_sources
        & bad_source_set
    )
)


print(
    "\nFailed sources:",
    len(bad_source_set)
)

print(
    "Failed sources explained by collisions:",
    len(
        bad_source_set
        & collision_val_sources
    )
)


# --------------------------------------------------------------
# Show relevant collisions
# --------------------------------------------------------------

shown = 0

for item in collision_rows:

    relevant_sources = (
        set(
            item[
                "validation_source_indices"
            ]
        )
        & bad_source_set
    )

    if not relevant_sources:
        continue

    print(
        "\n"
        + "-" * 100
    )

    print(
        "question_id:",
        item[
            "question_id"
        ]
    )

    print(
        "affected validation indices:",
        sorted(
            relevant_sources
        )
    )

    for split, idx, text in item[
        "versions"
    ]:

        print(
            f"  {split:<10} "
            f"index={idx:<6} "
            f"{repr(text)}"
        )

    shown += 1

    if shown >= 30:
        break


# --------------------------------------------------------------
# Root-cause classification
# --------------------------------------------------------------

if (
    bad_source_set
    and bad_source_set.issubset(
        mismatch_source_set
    )
):

    ROOT_CAUSE = (
        "planning_row_question_text_differs_from_"
        "original_validation_dataset"
    )

elif bad_source_set.issubset(
    mismatch_source_set
    | collision_val_sources
):

    ROOT_CAUSE = (
        "mixed_question_text_source_and_or_"
        "question_id_cache_collision"
    )

else:

    ROOT_CAUSE = (
        "not_fully_explained_yet"
    )


print(
    "\n"
    + "=" * 108
)

print(
    "=== CWQ QUESTION-EMBEDDING SOURCE DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 108
)

print(
    "Failed feature source questions:",
    len(bad_source_set)
)

print(
    "Plan-vs-dataset text mismatches:",
    len(mismatch_source_set)
)

print(
    "Question-ID collision sources:",
    len(collision_val_sources)
)

print(
    "Failed sources explained by collisions:",
    len(
        bad_source_set
        & collision_val_sources
    )
)

print(
    "\nROOT CAUSE CLASSIFICATION:"
)

print(
    ROOT_CAUSE
)

print("\nMiniLM rerun:          NO")
print("Feature rebuild:       NO")
print("Pruning:               NO")
print("Hyperparameter tuning: NO")
print("TEST used for dev:     NO")


Reading CWQ TRAIN id/question columns...
Reading CWQ VALIDATION id/question columns...

CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT
IDs with >1 distinct question text: 0
Validation source indices involved: 0
Intersection with failed feature sources: 0

Failed sources: 45
Failed sources explained by collisions: 0

=== CWQ QUESTION-EMBEDDING SOURCE DIAGNOSTIC COMPLETE ===
Failed feature source questions: 45
Plan-vs-dataset text mismatches: 0
Question-ID collision sources: 0
Failed sources explained by collisions: 0

ROOT CAUSE CLASSIFICATION:
not_fully_explained_yet

MiniLM rerun:          NO
Feature rebuild:       NO
Pruning:               NO
Hyperparameter tuning: NO
TEST used for dev:     NO


In [21]:
# ======================================================================
# 3.12C-B3-DIAG3 — MINILM QUESTION TRUNCATION / MAX-LENGTH AUDIT
# ======================================================================
#
# PURPOSE
# -------
# Determine whether the missing FrozenMiniLMEncoder used a different
# max_seq_length / truncation behavior.
#
# Uses ONLY:
#   - CWQ VALIDATION
#   - the 45 already-identified affected questions
#   - frozen validation Feature-v2 values as SOFTWARE-FIDELITY reference
#
# No scorer.
# No pruning.
# No selector tuning.
# No TEST examples accessed.
# ======================================================================

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "cwq_val_exact",
    "cwq_frozen",
    "cwq_val_plan_rows",
    "bad_groups",
    "bad_source_set",
    "source_indices",
    "safe_l2_normalize",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Missing previous diagnostic objects: "
    + ", ".join(missing)
)

assert len(cwq_val_exact) == 3519
assert len(bad_source_set) == 45
assert len(bad_groups) == 70

tokenizer = minilm_model.tokenizer

print("Affected CWQ source questions:", len(bad_source_set))
print("Affected decision groups:     ", len(bad_groups))
print("Current MiniLM max_seq_length:", minilm_model.max_seq_length)


# ======================================================================
# 2. Exact token lengths WITHOUT truncation
# ======================================================================

all_token_lengths = []

for i in range(len(cwq_val_exact)):

    q = str(
        cwq_val_exact[i]["question"]
    )

    encoded = tokenizer(
        q,
        add_special_tokens=True,
        truncation=False
    )

    n_tokens = len(
        encoded["input_ids"]
    )

    all_token_lengths.append(
        n_tokens
    )


all_token_lengths = np.asarray(
    all_token_lengths,
    dtype=np.int32
)

bad_indices = np.asarray(
    sorted(bad_source_set),
    dtype=np.int32
)

bad_token_lengths = (
    all_token_lengths[
        bad_indices
    ]
)


# ======================================================================
# 3. Token-length profile
# ======================================================================

print("\n" + "=" * 100)
print("CWQ QUESTION TOKEN-LENGTH PROFILE")
print("=" * 100)

print("\nALL VALIDATION")
print("  n:      ", len(all_token_lengths))
print("  mean:   ", float(all_token_lengths.mean()))
print("  median: ", float(np.median(all_token_lengths)))
print("  p90:    ", float(np.quantile(all_token_lengths, 0.90)))
print("  p95:    ", float(np.quantile(all_token_lengths, 0.95)))
print("  p99:    ", float(np.quantile(all_token_lengths, 0.99)))
print("  max:    ", int(all_token_lengths.max()))

print("\nAFFECTED 45 QUESTIONS")
print("  n:      ", len(bad_token_lengths))
print("  mean:   ", float(bad_token_lengths.mean()))
print("  median: ", float(np.median(bad_token_lengths)))
print("  min:    ", int(bad_token_lengths.min()))
print("  max:    ", int(bad_token_lengths.max()))


for threshold in [
    64,
    96,
    128,
    160,
    192,
    256,
]:

    all_over = int(
        np.sum(
            all_token_lengths > threshold
        )
    )

    bad_over = int(
        np.sum(
            bad_token_lengths > threshold
        )
    )

    print(
        f"\n> {threshold:<3} tokens:"
        f"  all={all_over:<4}"
        f"  affected={bad_over:<3}"
        f" / {len(bad_token_lengths)}"
    )


# ======================================================================
# 4. Print affected question lengths
# ======================================================================

affected_length_rows = []

for idx in sorted(
    bad_source_set
):

    rec = cwq_val_exact[
        idx
    ]

    affected_length_rows.append(
        {
            "source_index":
                int(idx),

            "question_id":
                str(
                    rec["id"]
                ),

            "n_tokens":
                int(
                    all_token_lengths[
                        idx
                    ]
                ),

            "question":
                str(
                    rec["question"]
                ),
        }
    )


affected_length_df = (
    pd.DataFrame(
        affected_length_rows
    )
    .sort_values(
        "n_tokens",
        ascending=False
    )
)


print("\n" + "=" * 110)
print("AFFECTED QUESTION LENGTHS")
print("=" * 110)

print(
    affected_length_df[
        [
            "source_index",
            "n_tokens",
            "question_id",
            "question",
        ]
    ].to_string(
        index=False
    )
)


# ======================================================================
# 5. Reconstruct affected group metadata
# ======================================================================

ptr = np.asarray(
    cwq_frozen["group_ptr"],
    dtype=np.int64
)

assert len(ptr) == 1353


# We need the exact decision-group rows.
assert "iter_groups" in globals(), (
    "iter_groups() from B3-R is missing."
)

assert "CWQ_LABEL_FILE" in globals(), (
    "CWQ_LABEL_FILE from B3-R is missing."
)


all_cwq_group_rows = list(
    iter_groups(
        CWQ_LABEL_FILE
    )
)

assert len(
    all_cwq_group_rows
) == 1352


affected_groups = {}

for g in bad_groups:

    rows = all_cwq_group_rows[
        g
    ]

    first = rows[0]

    src_idx = int(
        first["source_index"]
    )

    assert src_idx in bad_source_set

    affected_groups[
        g
    ] = rows


# ======================================================================
# 6. Need the currently-correct NON-question semantic embeddings
# ======================================================================
#
# encoder_unit already reproduced all candidate/entity-only semantics.
# We use it only for relation / plan / suffix reference embeddings.
# ======================================================================

assert "encoder_unit" in globals()

base_encoder = encoder_unit


def cosine_np(a, b):

    a = np.asarray(
        a,
        dtype=np.float32
    )

    b = np.asarray(
        b,
        dtype=np.float32
    )

    na = float(
        np.linalg.norm(a)
    )

    nb = float(
        np.linalg.norm(b)
    )

    if na <= 1e-12 or nb <= 1e-12:
        return 0.0

    return float(
        np.dot(a, b)
        / (na * nb)
    )


# ======================================================================
# 7. Software-fidelity reference
# ======================================================================
#
# For each affected group we compare:
#
# feature 4: cos(q, current_relation)
# feature 5: cos(q, full_plan)
# feature 6: cos(q, remaining_suffix)
#
# These three depend on q but NOT candidate identity.
# Therefore one row per group is sufficient.
# ======================================================================

REFERENCE_FEATURES = [
    4,
    5,
    6,
]


def evaluate_question_embeddings(
    q_embedding_by_source
):

    diffs = []

    per_feature = {
        4: [],
        5: [],
        6: [],
    }

    for g in bad_groups:

        rows = affected_groups[
            g
        ]

        first = rows[0]

        source_index = int(
            first["source_index"]
        )

        q_emb = q_embedding_by_source[
            source_index
        ]

        plan = list(
            first["plan"]
        )

        hop = int(
            first["hop"]
        )

        current_relation = (
            plan[hop]
        )

        remaining_suffix = (
            plan[
                hop + 1:
            ]
        )

        relation_emb = (
            base_encoder.get(
                "relation",
                current_relation,
                feature_ns[
                    "relation_surface_text"
                ](
                    current_relation
                )
            )
        )

        plan_key = "||".join(
            str(x)
            for x in plan
        )

        full_plan_text = (
            feature_ns[
                "relation_sequence_text"
            ](
                plan
            )
        )

        plan_emb = (
            base_encoder.get(
                "plan",
                plan_key,
                full_plan_text
            )
        )

        suffix_key = "||".join(
            str(x)
            for x in remaining_suffix
        )

        suffix_text = (
            feature_ns[
                "relation_sequence_text"
            ](
                remaining_suffix
            )
        )

        suffix_emb = (
            base_encoder.get(
                "suffix",
                suffix_key,
                suffix_text
            )
        )

        predicted = {
            4:
                cosine_np(
                    q_emb,
                    relation_emb
                ),

            5:
                cosine_np(
                    q_emb,
                    plan_emb
                ),

            6:
                cosine_np(
                    q_emb,
                    suffix_emb
                ),
        }

        frozen_row = int(
            ptr[g]
        )

        for feature_idx in REFERENCE_FEATURES:

            observed = float(
                cwq_frozen["X"][
                    frozen_row,
                    feature_idx
                ]
            )

            delta = abs(
                predicted[
                    feature_idx
                ]
                - observed
            )

            diffs.append(
                delta
            )

            per_feature[
                feature_idx
            ].append(
                delta
            )


    diffs = np.asarray(
        diffs,
        dtype=np.float64
    )

    return {
        "n_values":
            int(
                len(diffs)
            ),

        "mean_abs":
            float(
                diffs.mean()
            ),

        "max_abs":
            float(
                diffs.max()
            ),

        "p95_abs":
            float(
                np.quantile(
                    diffs,
                    0.95
                )
            ),

        "f4_mean":
            float(
                np.mean(
                    per_feature[4]
                )
            ),

        "f5_mean":
            float(
                np.mean(
                    per_feature[5]
                )
            ),

        "f6_mean":
            float(
                np.mean(
                    per_feature[6]
                )
            ),
    }


# ======================================================================
# 8. Test PREDECLARED plausible max_seq_length behaviors
# ======================================================================
#
# These are software-fidelity candidates, NOT AFP hyperparameters.
#
# Common sentence-transformer truncation limits:
#   64, 128, 256
#
# Also retain the currently loaded model value.
# ======================================================================

ORIGINAL_RUNTIME_MAX_SEQ = int(
    minilm_model.max_seq_length
)

candidate_max_lengths = sorted(
    set(
        [
            64,
            128,
            256,
            ORIGINAL_RUNTIME_MAX_SEQ,
        ]
    )
)


affected_questions = [
    str(
        cwq_val_exact[idx][
            "question"
        ]
    )
    for idx in sorted(
        bad_source_set
    )
]

affected_indices_sorted = sorted(
    bad_source_set
)


max_length_results = {}


print("\n" + "=" * 100)
print("MAX-SEQUENCE-LENGTH SOFTWARE-FIDELITY TEST")
print("=" * 100)


for max_len in candidate_max_lengths:

    print(
        f"\nTesting max_seq_length={max_len} ..."
    )

    minilm_model.max_seq_length = int(
        max_len
    )

    with torch.inference_mode():

        E = minilm_model.encode(
            affected_questions,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )

    E = np.asarray(
        E,
        dtype=np.float32
    )

    assert E.shape == (
        len(
            affected_indices_sorted
        ),
        384
    )

    q_map = {
        int(idx):
            E[j]
        for j, idx in enumerate(
            affected_indices_sorted
        )
    }

    result = evaluate_question_embeddings(
        q_map
    )

    max_length_results[
        int(max_len)
    ] = result

    print(
        f"  mean abs diff: "
        f"{result['mean_abs']:.10f}"
    )

    print(
        f"  max abs diff:  "
        f"{result['max_abs']:.10f}"
    )

    print(
        f"  p95 abs diff:  "
        f"{result['p95_abs']:.10f}"
    )

    print(
        "  feature means: "
        f"f4={result['f4_mean']:.10f} | "
        f"f5={result['f5_mean']:.10f} | "
        f"f6={result['f6_mean']:.10f}"
    )


# Always restore current model setting.
minilm_model.max_seq_length = (
    ORIGINAL_RUNTIME_MAX_SEQ
)


# ======================================================================
# 9. Rank candidate behaviors
# ======================================================================

ranking_rows = []

for max_len, result in (
    max_length_results.items()
):

    ranking_rows.append(
        {
            "max_seq_length":
                int(max_len),

            "mean_abs_diff":
                result[
                    "mean_abs"
                ],

            "max_abs_diff":
                result[
                    "max_abs"
                ],

            "p95_abs_diff":
                result[
                    "p95_abs"
                ],
        }
    )


ranking_df = (
    pd.DataFrame(
        ranking_rows
    )
    .sort_values(
        [
            "mean_abs_diff",
            "max_abs_diff",
        ]
    )
)


print("\n" + "=" * 100)
print("MAX-LENGTH FIDELITY RANKING")
print("=" * 100)

print(
    ranking_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.10f}"
    )
)


best_row = ranking_df.iloc[0]

BEST_MAX_SEQ_LENGTH = int(
    best_row[
        "max_seq_length"
    ]
)

BEST_MEAN_DIFF = float(
    best_row[
        "mean_abs_diff"
    ]
)

BEST_MAX_DIFF = float(
    best_row[
        "max_abs_diff"
    ]
)


print(
    "\nBest candidate max_seq_length:",
    BEST_MAX_SEQ_LENGTH
)

print(
    "Best mean difference:",
    BEST_MEAN_DIFF
)

print(
    "Best max difference:",
    BEST_MAX_DIFF
)


# ======================================================================
# 10. Classification
# ======================================================================

FIDELITY_ATOL = 5e-5

if BEST_MAX_DIFF <= FIDELITY_ATOL:

    MAX_LENGTH_EXPLAINS_FAILURE = True

    print(
        "\nRESULT: max_seq_length behavior "
        "FULLY explains the frozen CWQ features."
    )

else:

    MAX_LENGTH_EXPLAINS_FAILURE = False

    print(
        "\nRESULT: max_seq_length alone does NOT "
        "fully explain the frozen CWQ features."
    )


print("\n" + "=" * 108)
print("=== MINILM QUESTION-RUNTIME DIAGNOSTIC COMPLETE ===")
print("=" * 108)

print(
    "Current runtime max_seq_length: ",
    ORIGINAL_RUNTIME_MAX_SEQ
)

print(
    "Best candidate max_seq_length:  ",
    BEST_MAX_SEQ_LENGTH
)

print(
    "Failure explained:              ",
    MAX_LENGTH_EXPLAINS_FAILURE
)

print(
    "\nMiniLM encoded affected validation questions only."
)

print(
    "No scorer."
)

print(
    "No pruning."
)

print(
    "No AFP hyperparameter tuning."
)

print(
    "No TEST examples accessed."
)

Affected CWQ source questions: 45
Affected decision groups:      70
Current MiniLM max_seq_length: 256

CWQ QUESTION TOKEN-LENGTH PROFILE

ALL VALIDATION
  n:       3519
  mean:    18.429667519181585
  median:  18.0
  p90:     24.0
  p95:     25.0
  p99:     29.0
  max:     39

AFFECTED 45 QUESTIONS
  n:       45
  mean:    20.733333333333334
  median:  20.0
  min:     15
  max:     30

> 64  tokens:  all=0     affected=0   / 45

> 96  tokens:  all=0     affected=0   / 45

> 128 tokens:  all=0     affected=0   / 45

> 160 tokens:  all=0     affected=0   / 45

> 192 tokens:  all=0     affected=0   / 45

> 256 tokens:  all=0     affected=0   / 45

AFFECTED QUESTION LENGTHS
 source_index  n_tokens                                    question_id                                                                                                              question
         1313        30  WebQTrn-3012_da5f6afb91d1bbe3f283829b281fe01b Of the countries that share a border with China, which count

In [22]:
# ======================================================================
# 3.12C-B3-DIAG4 — QUESTION NORMALIZATION FIDELITY AUDIT
# ======================================================================
#
# Hypothesis:
#   Missing FrozenMiniLMEncoder normalized raw_text using the existing
#   Feature-v2 normalize_surface_text() before SentenceTransformer.encode.
#
# Why plausible:
#   questions  -> stored RAW in semantic inventory
#   entities   -> already normalized
#   relations  -> already normalized
#   plans      -> already normalized
#   suffixes   -> already normalized
#
# Therefore an encoder-side normalizer would selectively alter QUESTION
# embeddings, matching the observed failure pattern.
#
# VALIDATION SOFTWARE-FIDELITY ONLY.
# No scorer.
# No pruning.
# No AFP hyperparameter tuning.
# No TEST examples accessed.
# ======================================================================

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "cwq_val_exact",
    "cwq_frozen",
    "bad_source_set",
    "bad_groups",
    "encoder_unit",
    "extract_group",
    "iter_groups",
    "CWQ_LABEL_FILE",
    "feature_ns",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing previous B3 diagnostic objects: "
    + ", ".join(missing)
)


normalize_surface_text_fn = (
    feature_ns[
        "normalize_surface_text"
    ]
)

assert callable(
    normalize_surface_text_fn
)

tokenizer = (
    minilm_model.tokenizer
)

print(
    "Affected CWQ source questions:",
    len(bad_source_set)
)

print(
    "Affected decision groups:",
    len(bad_groups)
)


# ======================================================================
# 2. Compare RAW vs Feature-v2-normalized tokenization
# ======================================================================

token_change_rows = []

for source_index in range(
    len(cwq_val_exact)
):

    raw_question = str(
        cwq_val_exact[
            source_index
        ][
            "question"
        ]
    )

    normalized_question = (
        normalize_surface_text_fn(
            raw_question
        )
    )

    raw_ids = tokenizer(
        raw_question,
        add_special_tokens=True,
        truncation=False
    )[
        "input_ids"
    ]

    normalized_ids = tokenizer(
        normalized_question,
        add_special_tokens=True,
        truncation=False
    )[
        "input_ids"
    ]

    tokens_differ = (
        raw_ids
        != normalized_ids
    )

    if tokens_differ:

        token_change_rows.append(
            {
                "source_index":
                    int(
                        source_index
                    ),

                "question_id":
                    str(
                        cwq_val_exact[
                            source_index
                        ][
                            "id"
                        ]
                    ),

                "raw_question":
                    raw_question,

                "normalized_question":
                    normalized_question,

                "raw_token_count":
                    len(
                        raw_ids
                    ),

                "normalized_token_count":
                    len(
                        normalized_ids
                    ),
            }
        )


token_change_df = (
    pd.DataFrame(
        token_change_rows
    )
)

token_change_sources = {
    int(x)
    for x in (
        token_change_df[
            "source_index"
        ].tolist()
        if len(token_change_df)
        else []
    )
}


# ======================================================================
# 3. Compare token-change set with the 45 failure sources
# ======================================================================

intersection = (
    token_change_sources
    & bad_source_set
)

bad_not_changed = (
    bad_source_set
    - token_change_sources
)

changed_not_bad = (
    token_change_sources
    - bad_source_set
)


print(
    "\n"
    + "=" * 106
)

print(
    "RAW QUESTION vs normalize_surface_text() TOKENIZATION"
)

print(
    "=" * 106
)

print(
    "Validation questions:",
    len(cwq_val_exact)
)

print(
    "Questions whose token IDs change:",
    len(
        token_change_sources
    )
)

print(
    "Known failed source questions:",
    len(
        bad_source_set
    )
)

print(
    "Intersection:",
    len(
        intersection
    )
)

print(
    "Failed but tokenization unchanged:",
    len(
        bad_not_changed
    )
)

print(
    "Tokenization changed but feature did not fail:",
    len(
        changed_not_bad
    )
)


if bad_source_set:

    recall = (
        len(intersection)
        /
        len(bad_source_set)
    )

else:
    recall = 0.0


if token_change_sources:

    precision = (
        len(intersection)
        /
        len(token_change_sources)
    )

else:
    precision = 0.0


print(
    f"\nFailure-source recall:    "
    f"{100*recall:.2f}%"
)

print(
    f"Failure-source precision: "
    f"{100*precision:.2f}%"
)

print(
    "Exact set equality:",
    token_change_sources
    == bad_source_set
)


# ======================================================================
# 4. Show affected normalization examples
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "NORMALIZATION EXAMPLES AMONG FAILED QUESTIONS"
)

print(
    "=" * 110
)


shown = 0

for _, row in token_change_df.iterrows():

    src = int(
        row[
            "source_index"
        ]
    )

    if src not in bad_source_set:
        continue

    print(
        "\n"
        + "-" * 104
    )

    print(
        "source_index:",
        src
    )

    print(
        "id:",
        row[
            "question_id"
        ]
    )

    print(
        "RAW:       ",
        repr(
            row[
                "raw_question"
            ]
        )
    )

    print(
        "NORMALIZED:",
        repr(
            row[
                "normalized_question"
            ]
        )
    )

    print(
        "token counts:",
        row[
            "raw_token_count"
        ],
        "->",
        row[
            "normalized_token_count"
        ]
    )

    shown += 1

    if shown >= 25:
        break


# ======================================================================
# 5. Encode normalized versions of ONLY the 45 affected questions
# ======================================================================

affected_indices = sorted(
    bad_source_set
)

affected_normalized_questions = [
    normalize_surface_text_fn(
        str(
            cwq_val_exact[
                idx
            ][
                "question"
            ]
        )
    )
    for idx in affected_indices
]


print(
    "\nEncoding normalized affected questions only..."
)


with torch.inference_mode():

    normalized_embeddings = (
        minilm_model.encode(
            affected_normalized_questions,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )
    )


normalized_embeddings = np.asarray(
    normalized_embeddings,
    dtype=np.float32
)

assert normalized_embeddings.shape == (
    len(
        affected_indices
    ),
    384
)


normalized_qid_embedding = {}

normalized_source_embedding = {}

for j, source_index in enumerate(
    affected_indices
):

    qid = str(
        cwq_val_exact[
            source_index
        ][
            "id"
        ]
    )

    normalized_qid_embedding[
        qid
    ] = (
        normalized_embeddings[
            j
        ]
    )

    normalized_source_embedding[
        int(
            source_index
        )
    ] = (
        normalized_embeddings[
            j
        ]
    )


# ======================================================================
# 6. Encoder override
# ======================================================================
#
# QUESTION:
#   use the normalized-question embedding above.
#
# Everything else:
#   delegate to encoder_unit, which already reproduced candidate/entity/
#   relation/plan/suffix semantics.
# ======================================================================

class NormalizedQuestionEncoder:

    def __init__(
        self,
        question_embedding_by_id,
        fallback_encoder
    ):

        self.question_embedding_by_id = (
            question_embedding_by_id
        )

        self.fallback_encoder = (
            fallback_encoder
        )

        self.dim = 384


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        namespace = str(
            namespace
        )

        identifier = str(
            identifier
        )

        if (
            namespace
            == "question"
            and identifier
            in self.question_embedding_by_id
        ):

            return np.asarray(
                self.question_embedding_by_id[
                    identifier
                ],
                dtype=np.float32
            )

        return (
            self.fallback_encoder.get(
                namespace,
                identifier,
                raw_text
            )
        )


normalized_question_encoder = (
    NormalizedQuestionEncoder(
        question_embedding_by_id=
            normalized_qid_embedding,

        fallback_encoder=
            encoder_unit
    )
)


# ======================================================================
# 7. Reload exact validation decision groups
# ======================================================================

cwq_group_rows_diag4 = list(
    iter_groups(
        CWQ_LABEL_FILE
    )
)

assert len(
    cwq_group_rows_diag4
) == 1352


group_ptr = np.asarray(
    cwq_frozen[
        "group_ptr"
    ],
    dtype=np.int64
)


# ======================================================================
# 8. Rebuild ONLY the 70 previously failing groups
# ======================================================================

all_differences = []

semantic_differences = []

symbolic_differences = []

SEMANTIC_COLUMNS = [
    0, 2, 4, 5, 6, 7, 8, 9
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(27)
    if i not in SEMANTIC_COLUMNS
]


group_results = []


for g in sorted(
    bad_groups
):

    rows = (
        cwq_group_rows_diag4[
            g
        ]
    )

    first = rows[0]

    source_index = int(
        first[
            "source_index"
        ]
    )

    assert (
        source_index
        in bad_source_set
    )

    rec = (
        cwq_val_exact[
            source_index
        ]
    )

    assert str(
        rec["id"]
    ) == str(
        first[
            "question_id"
        ]
    )

    plan = list(
        first[
            "plan"
        ]
    )

    hop = int(
        first[
            "hop"
        ]
    )


    X_rebuilt = extract_group(
        question_id=
            first[
                "question_id"
            ],

        question=
            rec[
                "question"
            ],

        plan=
            plan,

        hop=
            hop,

        candidate_rows=
            rows,

        semantic_encoder=
            normalized_question_encoder,

        entity_name_map=
            None
    )


    start = int(
        group_ptr[
            g
        ]
    )

    end = int(
        group_ptr[
            g + 1
        ]
    )

    X_frozen = np.asarray(
        cwq_frozen[
            "X"
        ][
            start:end
        ],
        dtype=np.float32
    )


    assert X_rebuilt.shape == (
        X_frozen.shape
    )


    diff = np.abs(
        X_rebuilt.astype(
            np.float64
        )
        -
        X_frozen.astype(
            np.float64
        )
    )


    sem_diff = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym_diff = diff[
        :,
        SYMBOLIC_COLUMNS
    ]


    all_differences.append(
        diff.ravel()
    )

    semantic_differences.append(
        sem_diff.ravel()
    )

    symbolic_differences.append(
        sym_diff.ravel()
    )


    group_results.append(
        {
            "group_index":
                int(g),

            "source_index":
                source_index,

            "question_id":
                str(
                    rec[
                        "id"
                    ]
                ),

            "max_all":
                float(
                    diff.max()
                ),

            "max_semantic":
                float(
                    sem_diff.max()
                ),

            "max_symbolic":
                float(
                    sym_diff.max()
                ),
        }
    )


all_differences = np.concatenate(
    all_differences
)

semantic_differences = np.concatenate(
    semantic_differences
)

symbolic_differences = np.concatenate(
    symbolic_differences
)


# ======================================================================
# 9. Fidelity report
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "NORMALIZED-QUESTION FEATURE-v2 FIDELITY"
)

print(
    "=" * 108
)

print(
    "Groups rebuilt:",
    len(
        group_results
    )
)

print(
    "Max all-feature difference:",
    float(
        all_differences.max()
    )
)

print(
    "Mean all-feature difference:",
    float(
        all_differences.mean()
    )
)

print(
    "Max semantic difference:",
    float(
        semantic_differences.max()
    )
)

print(
    "Mean semantic difference:",
    float(
        semantic_differences.mean()
    )
)

print(
    "Max symbolic difference:",
    float(
        symbolic_differences.max()
    )
)


group_result_df = (
    pd.DataFrame(
        group_results
    )
    .sort_values(
        "max_semantic",
        ascending=False
    )
)


print(
    "\nWorst groups after normalization:"
)

print(
    group_result_df.head(
        20
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.10g}"
    )
)


# ======================================================================
# 10. Strong software-fidelity gate
# ======================================================================

SEMANTIC_ATOL = 5e-5
SYMBOLIC_ATOL = 1e-7

NORMALIZATION_EXPLAINS_FAILURE = (
    float(
        semantic_differences.max()
    )
    <= SEMANTIC_ATOL

    and

    float(
        symbolic_differences.max()
    )
    <= SYMBOLIC_ATOL
)


print(
    "\n"
    + "=" * 108
)

print(
    "QUESTION NORMALIZATION ROOT-CAUSE TEST"
)

print(
    "=" * 108
)

print(
    "Token-change exact set equality:",
    token_change_sources
    == bad_source_set
)

print(
    "Feature-fidelity gate:",
    NORMALIZATION_EXPLAINS_FAILURE
)


if NORMALIZATION_EXPLAINS_FAILURE:

    QUESTION_RUNTIME_ROOT_CAUSE = (
        "FrozenMiniLMEncoder normalized raw question text "
        "before MiniLM encoding"
    )

    print(
        "\nRESULT: QUESTION NORMALIZATION "
        "FULLY EXPLAINS THE CWQ FAILURE."
    )

else:

    QUESTION_RUNTIME_ROOT_CAUSE = (
        "question normalization alone does not fully explain failure"
    )

    print(
        "\nRESULT: QUESTION NORMALIZATION DOES NOT "
        "FULLY EXPLAIN THE CWQ FAILURE."
    )


print(
    "\nRoot-cause classification:"
)

print(
    QUESTION_RUNTIME_ROOT_CAUSE
)


print(
    "\nMiniLM encoded only 45 affected validation questions."
)

print(
    "No scorer."
)

print(
    "No pruning."
)

print(
    "No AFP hyperparameter tuning."
)

print(
    "No TEST examples accessed."
)

Affected CWQ source questions: 45
Affected decision groups: 70

RAW QUESTION vs normalize_surface_text() TOKENIZATION
Validation questions: 3519
Questions whose token IDs change: 265
Known failed source questions: 45
Intersection: 45
Failed but tokenization unchanged: 0
Tokenization changed but feature did not fail: 220

Failure-source recall:    100.00%
Failure-source precision: 16.98%
Exact set equality: False

NORMALIZATION EXAMPLES AMONG FAILED QUESTIONS

--------------------------------------------------------------------------------------------------------
source_index: 76
id: WebQTrn-836_4dbb23937c062d0f3a3496d2fbf26a12
RAW:        "What major religion in the UK has a place of worship named St. Mary's Cathedral, Batticaloa?"
NORMALIZED: "what major religion in the uk has a place of worship named st mary's cathedral, batticaloa?"
token counts: 25 -> 24

--------------------------------------------------------------------------------------------------------
source_index: 347
id: W

In [23]:
# ======================================================================
# 3.12C-B3-FINAL
# BEHAVIORALLY RECOVERED MINILM RUNTIME + FULL FEATURE-v2 FIDELITY GATE
# ======================================================================
#
# RECOVERED BEHAVIOR
# ------------------
# The original FrozenMiniLMEncoder source was not persisted in the
# notebook. Validation diagnostics established that its observable
# behavior includes:
#
#       normalize_surface_text(raw_text)
#               ↓
#             MiniLM
#
# Evidence:
#   - WebQSP already reproduced to ~3e-7.
#   - CWQ failure occurred ONLY in q-dependent semantic features.
#   - Question normalization reduced CWQ failing-group max error from
#       0.4881388545
#     to
#       2.980232e-07.
#
# THIS CELL:
#   1. Defines recovered runtime encoder.
#   2. Prefills validation semantic inventory.
#   3. Rebuilds ALL frozen validation Feature-v2 groups.
#   4. Requires full WebQSP + CWQ numerical fidelity.
#   5. Exposes runtime encoder/extractor for Cell 12C-B4.
#
# NO scorer selection.
# NO pruning.
# NO AFP hyperparameter tuning.
# NO TEST examples accessed.
# ======================================================================

import json
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "feature_ns",
    "extract_group",
    "iter_groups",
    "WEBQSP_LABEL_FILE",
    "CWQ_LABEL_FILE",
    "WEBQSP_NPZ",
    "CWQ_NPZ",
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing B3/diagnostic objects: "
    + ", ".join(missing)
)


normalize_surface_text_runtime = (
    feature_ns[
        "normalize_surface_text"
    ]
)

relation_surface_text_runtime = (
    feature_ns[
        "relation_surface_text"
    ]
)

relation_sequence_text_runtime = (
    feature_ns[
        "relation_sequence_text"
    ]
)

has_readable_entity_surface_runtime = (
    feature_ns[
        "has_readable_entity_surface"
    ]
)

entity_surface_text_runtime = (
    feature_ns[
        "entity_surface_text"
    ]
)

assert callable(
    normalize_surface_text_runtime
)

assert callable(
    extract_group
)


print(
    "Recovered normalization function: READY"
)

print(
    "MiniLM embedding dimension:",
    minilm_model.get_embedding_dimension()
)


# ======================================================================
# 2. Resolve validation question records
# ======================================================================
#
# Original Cell 173 used the validation dataset's question field.
#
# CWQ exact dataset was loaded during diagnostics.
#
# For WebQSP:
#   use webqsp_val if still present;
#   otherwise frozen planning rows contain the aligned question text
#   already shown to reproduce the frozen matrix.
# ======================================================================

if (
    "webqsp_val" in globals()
    and globals()["webqsp_val"] is not None
):

    webqsp_val_runtime = globals()[
        "webqsp_val"
    ]

    WEBQSP_RUNTIME_SOURCE = (
        "existing_global:webqsp_val"
    )

else:

    webqsp_val_runtime = (
        webqsp_val_plan_rows
    )

    WEBQSP_RUNTIME_SOURCE = (
        "frozen_validation_planning_rows"
    )


if (
    "cwq_val_exact" in globals()
    and globals()["cwq_val_exact"] is not None
):

    cwq_val_runtime = (
        cwq_val_exact
    )

    CWQ_RUNTIME_SOURCE = (
        "exact_validation_dataset"
    )

elif (
    "cwq_val" in globals()
    and globals()["cwq_val"] is not None
):

    cwq_val_runtime = (
        globals()["cwq_val"]
    )

    CWQ_RUNTIME_SOURCE = (
        "existing_global:cwq_val"
    )

else:

    cwq_val_runtime = (
        cwq_val_plan_rows
    )

    CWQ_RUNTIME_SOURCE = (
        "frozen_validation_planning_rows"
    )


assert len(
    webqsp_val_runtime
) == 246

assert len(
    cwq_val_runtime
) == 3519


print(
    "\nValidation sources:"
)

print(
    "  WebQSP:",
    WEBQSP_RUNTIME_SOURCE
)

print(
    "  CWQ:   ",
    CWQ_RUNTIME_SOURCE
)


# ======================================================================
# 3. Exact validation group iterator alias
# ======================================================================

iter_runtime_groups = (
    iter_groups
)


# ======================================================================
# 4. Collect semantic inventory exactly as Feature-v2 expects
# ======================================================================

def collect_runtime_semantic_inventory(
    labels_file,
    dataset_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    n_groups = 0
    n_branches = 0

    for rows in iter_runtime_groups(
        labels_file
    ):

        n_groups += 1
        n_branches += len(
            rows
        )

        first = rows[0]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first[
                "question_id"
            ]
        )


        # --------------------------------------------------------------
        # CRITICAL:
        # Store the RAW question here.
        #
        # The recovered semantic encoder performs normalization.
        # This mirrors the original Cell-173 inventory behavior.
        # --------------------------------------------------------------

        question_id = str(
            first[
                "question_id"
            ]
        )

        questions[
            question_id
        ] = str(
            rec[
                "question"
            ]
        )


        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )


        for relation in plan:

            relations[
                str(
                    relation
                )
            ] = (
                relation_surface_text_runtime(
                    relation
                )
            )


        plan_key = "||".join(
            str(x)
            for x in plan
        )

        plans[
            plan_key
        ] = (
            relation_sequence_text_runtime(
                plan
            )
        )


        suffix = (
            plan[
                hop + 1:
            ]
        )

        suffix_key = "||".join(
            str(x)
            for x in suffix
        )

        suffixes[
            suffix_key
        ] = (
            relation_sequence_text_runtime(
                suffix
            )
        )


        for row in rows:

            candidate = row[
                "candidate_entity"
            ]

            if (
                has_readable_entity_surface_runtime(
                    candidate
                )
            ):

                entities[
                    str(
                        candidate
                    )
                ] = (
                    entity_surface_text_runtime(
                        candidate
                    )
                )


            for entity in row[
                "prefix_entities"
            ]:

                if (
                    has_readable_entity_surface_runtime(
                        entity
                    )
                ):

                    entities[
                        str(
                            entity
                        )
                    ] = (
                        entity_surface_text_runtime(
                            entity
                        )
                    )


    return {
        "questions":
            questions,

        "entities":
            entities,

        "relations":
            relations,

        "plans":
            plans,

        "suffixes":
            suffixes,

        "n_groups":
            n_groups,

        "n_branches":
            n_branches,
    }


print(
    "\nCollecting full validation semantic inventory..."
)


webqsp_inventory_final = (
    collect_runtime_semantic_inventory(
        WEBQSP_LABEL_FILE,
        webqsp_val_runtime
    )
)

cwq_inventory_final = (
    collect_runtime_semantic_inventory(
        CWQ_LABEL_FILE,
        cwq_val_runtime
    )
)


assert (
    webqsp_inventory_final[
        "n_groups"
    ]
    == 87
)

assert (
    webqsp_inventory_final[
        "n_branches"
    ]
    == 966
)

assert (
    cwq_inventory_final[
        "n_groups"
    ]
    == 1352
)

assert (
    cwq_inventory_final[
        "n_branches"
    ]
    == 18688
)


print(
    "WebQSP:",
    webqsp_inventory_final[
        "n_groups"
    ],
    "groups |",
    webqsp_inventory_final[
        "n_branches"
    ],
    "branches"
)

print(
    "CWQ:   ",
    cwq_inventory_final[
        "n_groups"
    ],
    "groups |",
    cwq_inventory_final[
        "n_branches"
    ],
    "branches"
)


# ======================================================================
# 5. Merge semantic inventory
# ======================================================================

def merge_runtime_dicts(
    *dicts
):

    result = {}

    for d in dicts:

        result.update(
            d
        )

    return result


RUNTIME_QUESTIONS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "questions"
        ],
        cwq_inventory_final[
            "questions"
        ],
    )
)

RUNTIME_ENTITIES = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "entities"
        ],
        cwq_inventory_final[
            "entities"
        ],
    )
)

RUNTIME_RELATIONS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "relations"
        ],
        cwq_inventory_final[
            "relations"
        ],
    )
)

RUNTIME_PLANS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "plans"
        ],
        cwq_inventory_final[
            "plans"
        ],
    )
)

RUNTIME_SUFFIXES = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "suffixes"
        ],
        cwq_inventory_final[
            "suffixes"
        ],
    )
)


print(
    "\nSemantic inventory"
)

print(
    "  Questions:         ",
    len(
        RUNTIME_QUESTIONS
    )
)

print(
    "  Readable entities: ",
    len(
        RUNTIME_ENTITIES
    )
)

print(
    "  Relations:         ",
    len(
        RUNTIME_RELATIONS
    )
)

print(
    "  Plans:             ",
    len(
        RUNTIME_PLANS
    )
)

print(
    "  Suffixes:          ",
    len(
        RUNTIME_SUFFIXES
    )
)


# ======================================================================
# 6. Behaviorally recovered FrozenMiniLM runtime
# ======================================================================
#
# Important:
#
# The lost wrapper source cannot be claimed to have been recovered
# textually.
#
# What IS recovered and verified is its observable feature behavior:
#
#       raw_text
#          ↓
#       normalize_surface_text
#          ↓
#       all-MiniLM-L6-v2
#
# We use unit-normalized MiniLM vectors.
#
# Feature-v2 cosine operations and validation fidelity determine the
# relevant observable behavior.
# ======================================================================

class BehaviorallyRecoveredMiniLMEncoder:

    def __init__(
        self,
        model,
        batch_size=256
    ):

        self.model = model

        self.batch_size = int(
            batch_size
        )

        self.dim = int(
            model.get_embedding_dimension()
        )

        assert self.dim == 384

        self.cache = {}


    def _normalize_text(
        self,
        raw_text
    ):

        return (
            normalize_surface_text_runtime(
                raw_text
            )
        )


    def prefill(
        self,
        namespace,
        mapping
    ):

        namespace = str(
            namespace
        )

        items = list(
            mapping.items()
        )

        missing_items = []

        for identifier, raw_text in items:

            key = (
                namespace,
                str(
                    identifier
                )
            )

            if key not in self.cache:

                missing_items.append(
                    (
                        str(
                            identifier
                        ),
                        raw_text
                    )
                )


        if not missing_items:
            return


        texts = [
            self._normalize_text(
                raw_text
            )
            for _, raw_text
            in missing_items
        ]


        embeddings = (
            self.model.encode(
                texts,
                batch_size=
                    self.batch_size,

                show_progress_bar=
                    False,

                convert_to_numpy=
                    True,

                normalize_embeddings=
                    True
            )
        )


        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )


        assert embeddings.shape == (
            len(
                missing_items
            ),
            self.dim
        )


        for (
            (
                identifier,
                _
            ),
            vector
        ) in zip(
            missing_items,
            embeddings
        ):

            self.cache[
                (
                    namespace,
                    identifier
                )
            ] = np.asarray(
                vector,
                dtype=np.float32
            )


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        namespace = str(
            namespace
        )

        identifier = str(
            identifier
        )

        key = (
            namespace,
            identifier
        )


        if key not in self.cache:

            text = self._normalize_text(
                raw_text
            )

            vector = (
                self.model.encode(
                    [text],
                    batch_size=1,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=True
                )[0]
            )

            self.cache[
                key
            ] = np.asarray(
                vector,
                dtype=np.float32
            )


        return self.cache[
            key
        ]


# ======================================================================
# 7. Instantiate recovered runtime
# ======================================================================

AFP_RECOVERED_SEMANTIC_ENCODER = (
    BehaviorallyRecoveredMiniLMEncoder(
        model=
            minilm_model,

        batch_size=
            256
    )
)


print(
    "\nPrefilling recovered semantic runtime..."
)


for namespace, mapping in [
    (
        "question",
        RUNTIME_QUESTIONS
    ),
    (
        "entity",
        RUNTIME_ENTITIES
    ),
    (
        "relation",
        RUNTIME_RELATIONS
    ),
    (
        "plan",
        RUNTIME_PLANS
    ),
    (
        "suffix",
        RUNTIME_SUFFIXES
    ),
]:

    print(
        f"  {namespace:<10}"
        f"{len(mapping):>6}"
    )

    AFP_RECOVERED_SEMANTIC_ENCODER.prefill(
        namespace,
        mapping
    )


print(
    "\nRecovered semantic cache entries:",
    len(
        AFP_RECOVERED_SEMANTIC_ENCODER.cache
    )
)


# ======================================================================
# 8. Full validation Feature-v2 reconstruction
# ======================================================================

def rebuild_full_validation_features(
    dataset_name,
    dataset_rows,
    labels_file,
    encoder
):

    X_blocks = []
    y_blocks = []

    group_ptr = [
        0
    ]

    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []


    n_groups = 0


    for rows in tqdm(
        iter_runtime_groups(
            labels_file
        ),
        desc=(
            f"{dataset_name} full "
            "Feature-v2 fidelity"
        )
    ):

        first = rows[
            0
        ]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]


        assert str(
            rec[
                "id"
            ]
        ) == str(
            first[
                "question_id"
            ]
        )


        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )


        X_group = extract_group(
            question_id=
                first[
                    "question_id"
                ],

            question=
                rec[
                    "question"
                ],

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                rows,

            semantic_encoder=
                encoder,

            entity_name_map=
                None
        )


        y_group = np.asarray(
            [
                int(
                    row[
                        "label"
                    ]
                )
                for row in rows
            ],
            dtype=np.uint8
        )


        assert X_group.shape == (
            len(
                rows
            ),
            27
        )


        X_blocks.append(
            X_group
        )

        y_blocks.append(
            y_group
        )


        group_ptr.append(
            group_ptr[-1]
            + len(
                rows
            )
        )


        group_source_index.append(
            source_index
        )

        group_hop.append(
            hop
        )

        group_plan_length.append(
            len(
                plan
            )
        )

        group_candidate_count.append(
            len(
                rows
            )
        )


        n_groups += 1


    return {
        "X":
            np.concatenate(
                X_blocks,
                axis=0
            ).astype(
                np.float32
            ),

        "y":
            np.concatenate(
                y_blocks,
                axis=0
            ).astype(
                np.uint8
            ),

        "group_ptr":
            np.asarray(
                group_ptr,
                dtype=np.int64
            ),

        "group_source_index":
            np.asarray(
                group_source_index,
                dtype=np.int32
            ),

        "group_hop":
            np.asarray(
                group_hop,
                dtype=np.int16
            ),

        "group_plan_length":
            np.asarray(
                group_plan_length,
                dtype=np.int16
            ),

        "group_candidate_count":
            np.asarray(
                group_candidate_count,
                dtype=np.int32
            ),

        "n_groups":
            int(
                n_groups
            ),
    }


print(
    "\nRebuilding COMPLETE WebQSP validation matrix..."
)

webqsp_runtime_features = (
    rebuild_full_validation_features(
        dataset_name=
            "webqsp",

        dataset_rows=
            webqsp_val_runtime,

        labels_file=
            WEBQSP_LABEL_FILE,

        encoder=
            AFP_RECOVERED_SEMANTIC_ENCODER
    )
)


print(
    "\nRebuilding COMPLETE CWQ validation matrix..."
)

cwq_runtime_features = (
    rebuild_full_validation_features(
        dataset_name=
            "cwq",

        dataset_rows=
            cwq_val_runtime,

        labels_file=
            CWQ_LABEL_FILE,

        encoder=
            AFP_RECOVERED_SEMANTIC_ENCODER
    )
)


# ======================================================================
# 9. Load frozen Feature-v2 references
# ======================================================================

def load_frozen_npz_final(
    path
):

    z = np.load(
        path,
        allow_pickle=False
    )

    return {
        key:
            z[
                key
            ]
        for key in z.files
    }


webqsp_frozen_final = (
    load_frozen_npz_final(
        WEBQSP_NPZ
    )
)

cwq_frozen_final = (
    load_frozen_npz_final(
        CWQ_NPZ
    )
)


# ======================================================================
# 10. Exact metadata gates
# ======================================================================

META_KEYS = [
    "y",
    "group_ptr",
    "group_source_index",
    "group_hop",
    "group_plan_length",
    "group_candidate_count",
]


def exact_metadata_gate(
    dataset_name,
    rebuilt,
    frozen
):

    print(
        f"\n{dataset_name.upper()} metadata"
    )

    for key in META_KEYS:

        assert np.array_equal(
            rebuilt[
                key
            ],
            frozen[
                key
            ]
        ), (
            f"{dataset_name}: "
            f"{key} mismatch"
        )

        print(
            f"  {key:<25} PASS"
        )


exact_metadata_gate(
    "webqsp",
    webqsp_runtime_features,
    webqsp_frozen_final
)

exact_metadata_gate(
    "cwq",
    cwq_runtime_features,
    cwq_frozen_final
)


# ======================================================================
# 11. Numerical fidelity
# ======================================================================

SEMANTIC_COLUMNS = [
    0,
    2,
    4,
    5,
    6,
    7,
    8,
    9,
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(
        27
    )
    if i not in (
        SEMANTIC_COLUMNS
    )
]


def full_feature_fidelity_report(
    dataset_name,
    rebuilt,
    frozen
):

    A = np.asarray(
        rebuilt[
            "X"
        ],
        dtype=np.float64
    )

    B = np.asarray(
        frozen[
            "X"
        ],
        dtype=np.float64
    )

    assert A.shape == B.shape


    diff = np.abs(
        A - B
    )

    sem = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym = diff[
        :,
        SYMBOLIC_COLUMNS
    ]


    report = {
        "shape":
            list(
                A.shape
            ),

        "max_all":
            float(
                diff.max()
            ),

        "mean_all":
            float(
                diff.mean()
            ),

        "p99_all":
            float(
                np.quantile(
                    diff,
                    0.99
                )
            ),

        "max_semantic":
            float(
                sem.max()
            ),

        "mean_semantic":
            float(
                sem.mean()
            ),

        "max_symbolic":
            float(
                sym.max()
            ),

        "mean_symbolic":
            float(
                sym.mean()
            ),

        "n_values_gt_5e_5":
            int(
                np.sum(
                    diff
                    > 5e-5
                )
            ),
    }


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} FULL FEATURE-v2 FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in report.items():

        print(
            f"{key:<24}",
            value
        )


    return report


webqsp_full_fidelity = (
    full_feature_fidelity_report(
        "webqsp",
        webqsp_runtime_features,
        webqsp_frozen_final
    )
)

cwq_full_fidelity = (
    full_feature_fidelity_report(
        "cwq",
        cwq_runtime_features,
        cwq_frozen_final
    )
)


# ======================================================================
# 12. HARD software-fidelity gate
# ======================================================================

SEMANTIC_ATOL = 5e-5
SYMBOLIC_ATOL = 1e-7


for dataset_name, result in [
    (
        "WebQSP",
        webqsp_full_fidelity
    ),
    (
        "CWQ",
        cwq_full_fidelity
    ),
]:

    assert (
        result[
            "max_semantic"
        ]
        <= SEMANTIC_ATOL
    ), (
        f"{dataset_name}: semantic fidelity failed."
    )


    assert (
        result[
            "max_symbolic"
        ]
        <= SYMBOLIC_ATOL
    ), (
        f"{dataset_name}: symbolic fidelity failed."
    )


    assert (
        result[
            "n_values_gt_5e_5"
        ]
        == 0
    )


print(
    "\nFULL Feature-v2 software fidelity: PASSED"
)


# ======================================================================
# 13. Expose FINAL online runtime objects
# ======================================================================

AFP_RUNTIME_SEMANTIC_ENCODER = (
    AFP_RECOVERED_SEMANTIC_ENCODER
)

AFP_RUNTIME_FEATURE_EXTRACTOR = (
    extract_group
)

AFP_RUNTIME_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

AFP_RUNTIME_FEATURE_DIM = (
    27
)

AFP_RUNTIME_SEMANTIC_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

AFP_RUNTIME_TEXT_POLICY = (
    "normalize_surface_text_before_minilm"
)

AFP_RUNTIME_RECOVERY_STATUS = (
    "behaviorally_recovered_and_feature_fidelity_verified"
)


# ======================================================================
# 14. Save final fidelity manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(
        "/kaggle/working/"
        "step3_rq2_dev_v1/"
        "10_validation_traversal"
    )
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


B3_FINAL_MANIFEST = {
    "cell":
        "RQ2_12C_B3_FINAL",

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "feature_dim":
        27,

    "semantic_encoder":
        AFP_RUNTIME_SEMANTIC_MODEL,

    "original_wrapper_source_available":
        False,

    "recovery_type":
        "behavioral_software_fidelity",

    "recovered_text_policy":
        AFP_RUNTIME_TEXT_POLICY,

    "recovered_vector_policy":
        "unit_normalized_minilm_embedding",

    "recovery_evidence": {
        "cwq_initial_max_semantic_error":
            0.4881388545036316,

        "cwq_normalized_question_diagnostic_max_error":
            2.980232238769531e-07,
    },

    "webqsp":
        webqsp_full_fidelity,

    "cwq":
        cwq_full_fidelity,

    "semantic_atol":
        SEMANTIC_ATOL,

    "symbolic_atol":
        SYMBOLIC_ATOL,

    "metadata_fidelity":
        True,

    "full_feature_fidelity":
        True,

    "online_feature_runtime_ready":
        True,

    "scorer_integration_ready":
        False,

    "pruning_run":
        False,

    "afp_hyperparameter_tuning_run":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,
}


B3_FINAL_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b3_final_feature_runtime_fidelity.json"
)


with open(
    B3_FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        B3_FINAL_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 15. Final report
# ======================================================================

print(
    "\n"
    + "=" * 112
)

print(
    "=== RQ2 CELL 12C-B3-FINAL: FEATURE-v2 ONLINE RUNTIME VERIFIED ==="
)

print(
    "=" * 112
)


print(
    "\nRecovered semantic behavior:"
)

print(
    "  raw text"
)

print(
    "    -> normalize_surface_text"
)

print(
    "    -> all-MiniLM-L6-v2"
)

print(
    "    -> unit-normalized embedding"
)


print(
    "\nWebQSP full validation:"
)

print(
    "  branches:",
    webqsp_runtime_features[
        "X"
    ].shape[
        0
    ]
)

print(
    "  max semantic error:",
    webqsp_full_fidelity[
        "max_semantic"
    ]
)

print(
    "  max symbolic error:",
    webqsp_full_fidelity[
        "max_symbolic"
    ]
)


print(
    "\nCWQ full validation:"
)

print(
    "  branches:",
    cwq_runtime_features[
        "X"
    ].shape[
        0
    ]
)

print(
    "  max semantic error:",
    cwq_full_fidelity[
        "max_semantic"
    ]
)

print(
    "  max symbolic error:",
    cwq_full_fidelity[
        "max_symbolic"
    ]
)


print(
    "\nRuntime status:"
)

print(
    "  Exact Feature-v2 code:      VERIFIED"
)

print(
    "  Semantic behavior:          RECOVERED"
)

print(
    "  Full frozen-NPZ fidelity:   PASS"
)

print(
    "  Online feature runtime:     READY"
)

print(
    "  Scorer integration:         NEXT"
)

print(
    "  Pruning run:                NO"
)

print(
    "  Hyperparameter tuning:      NO"
)

print(
    "  TEST examples accessed:     NO"
)

print(
    "  Complete AFP frozen:        NO"
)


print(
    "\nManifest:",
    B3_FINAL_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C-B4 — frozen standardizer + selected scorer "
    "checkpoint integration and online-logit fidelity gate."
)

Recovered normalization function: READY
MiniLM embedding dimension: 384

Validation sources:
  WebQSP: frozen_validation_planning_rows
  CWQ:    exact_validation_dataset

WebQSP: 87 groups | 966 branches
CWQ:    1352 groups | 18688 branches

Semantic inventory
  Questions:          882
  Readable entities:  2637
  Relations:          274
  Plans:              292
  Suffixes:           190

Prefilling recovered semantic runtime...
  question     882
  entity      2637
  relation     274
  plan         292
  suffix       190

Recovered semantic cache entries: 4275

Rebuilding COMPLETE WebQSP validation matrix...


webqsp full Feature-v2 fidelity: 0it [00:00, ?it/s]


Rebuilding COMPLETE CWQ validation matrix...


cwq full Feature-v2 fidelity: 0it [00:00, ?it/s]


WEBQSP metadata
  y                         PASS
  group_ptr                 PASS
  group_source_index        PASS
  group_hop                 PASS
  group_plan_length         PASS
  group_candidate_count     PASS

CWQ metadata
  y                         PASS
  group_ptr                 PASS
  group_source_index        PASS
  group_hop                 PASS
  group_plan_length         PASS
  group_candidate_count     PASS

WEBQSP FULL FEATURE-v2 FIDELITY
shape                    [966, 27]
max_all                  3.5762786865234375e-07
mean_all                 1.6312285219084568e-08
p99_all                  2.0559877157210816e-07
max_semantic             3.5762786865234375e-07
mean_semantic            5.5053962614410415e-08
max_symbolic             0.0
mean_symbolic            0.0
n_values_gt_5e_5         0

CWQ FULL FEATURE-v2 FIDELITY
shape                    [18688, 27]
max_all                  5.364418029785156e-07
mean_all                 1.8326335775376288e-08
p99_all           

In [25]:
# ======================================================================
# RQ2 CELL 12C-B4 — REVISED
# FROZEN STANDARDIZER + SELECTED SCORER INTEGRATION
# + ONLINE LOGIT FIDELITY GATE
# ======================================================================
#
# PURPOSE
# -------
# 1. Recover the exact persisted AFPScorer definition.
# 2. Recover its required AFP_* namespace constants safely.
# 3. Load the already-selected development checkpoints:
#
#       WebQSP: H=32, branch_bce, seed=42
#       CWQ:    H=64, branch_bce, seed=42
#
# 4. Restore frozen TRAIN-only standardizers.
# 5. Verify selected checkpoint identities.
# 6. Compare scorer logits from:
#
#       frozen Feature-v2 NPZ
#                 vs
#       recovered online Feature-v2
#
# 7. Build the gold-free runtime scoring callback for Cell 13.
# 8. Spot-check actual group-level online scoring.
#
# IMPORTANT
# ---------
# SOFTWARE/RUNTIME FIDELITY ONLY.
#
# NO scorer re-selection.
# NO pruning.
# NO AFP hyperparameter tuning.
# NO TEST examples accessed.
# ======================================================================

import ast
import hashlib
import inspect
import json
from pathlib import Path
from typing import *

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ======================================================================
# 1. HARD PREREQUISITES FROM B3-FINAL
# ======================================================================

required_b3_objects = [
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_FEATURE_EXTRACTOR",
    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_runtime_features",
    "cwq_runtime_features",

    "webqsp_frozen_final",
    "cwq_frozen_final",

    "WEBQSP_LABEL_FILE",
    "CWQ_LABEL_FILE",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "iter_runtime_groups",
]

missing_b3_objects = [
    name
    for name in required_b3_objects
    if name not in globals()
]

assert not missing_b3_objects, (
    "Missing B3-FINAL objects:\n  "
    + "\n  ".join(missing_b3_objects)
    + "\nDo NOT continue. B3-FINAL runtime objects are required."
)

assert int(
    AFP_RUNTIME_FEATURE_DIM
) == 27

DEVICE = torch.device(
    "cpu"
)

print(
    "Runtime device:",
    DEVICE
)

print(
    "Feature runtime:",
    AFP_RUNTIME_FEATURE_VERSION
)

print(
    "Feature dimension:",
    AFP_RUNTIME_FEATURE_DIM
)


# ======================================================================
# 2. ARTIFACT PATHS
# ======================================================================

ROOT = Path(
    "/kaggle/working/"
    "step3_rq2_dev_v1"
)

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

FINAL_SCORER_DIR = (
    ROOT
    / "07_final_scorer"
)

WEBQSP_CKPT = (
    FINAL_SCORER_DIR
    / "webqsp_afp_scorer_selected.pt"
)

CWQ_CKPT = (
    FINAL_SCORER_DIR
    / "cwq_afp_scorer_selected.pt"
)


assert NOTEBOOK_PATH.exists(), (
    f"Notebook not found: {NOTEBOOK_PATH}"
)

assert WEBQSP_CKPT.exists(), (
    f"WebQSP checkpoint not found: {WEBQSP_CKPT}"
)

assert CWQ_CKPT.exists(), (
    f"CWQ checkpoint not found: {CWQ_CKPT}"
)


print(
    "\nSelected checkpoints:"
)

print(
    "  WebQSP:",
    WEBQSP_CKPT
)

print(
    "  CWQ:   ",
    CWQ_CKPT
)


# ======================================================================
# 3. SHA256 IDENTITY GATE
# ======================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


webqsp_ckpt_sha = sha256_file(
    WEBQSP_CKPT
)

cwq_ckpt_sha = sha256_file(
    CWQ_CKPT
)


print(
    "\nCheckpoint SHA256"
)

print(
    "  WebQSP:",
    webqsp_ckpt_sha
)

print(
    "  CWQ:   ",
    cwq_ckpt_sha
)


# Selected scorer artifacts frozen at Cell 9C.
assert webqsp_ckpt_sha.startswith(
    "bee65146403d5656"
), (
    "WebQSP selected checkpoint identity mismatch."
)

assert cwq_ckpt_sha.startswith(
    "91a531c057bb02d0"
), (
    "CWQ selected checkpoint identity mismatch."
)


print(
    "Selected checkpoint identity: PASSED"
)


# ======================================================================
# 4. LOAD PERSISTED NOTEBOOK + RECOVER CLASS SOURCE
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    nb_b4 = json.load(
        f
    )


def recover_class_source(
    notebook,
    class_name
):

    matches = []

    for cell_idx, cell in enumerate(
        notebook["cells"]
    ):

        if (
            cell.get(
                "cell_type"
            )
            != "code"
        ):
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        try:

            tree = ast.parse(
                source
            )

        except Exception:

            continue

        for node in tree.body:

            if not (
                isinstance(
                    node,
                    ast.ClassDef
                )
                and
                node.name
                == class_name
            ):
                continue

            class_source = (
                ast.get_source_segment(
                    source,
                    node
                )
            )

            if class_source:

                matches.append(
                    {
                        "cell_idx":
                            int(
                                cell_idx
                            ),

                        "source":
                            class_source,
                    }
                )

    return matches


scorer_matches = (
    recover_class_source(
        nb_b4,
        "AFPScorer"
    )
)

standardizer_matches = (
    recover_class_source(
        nb_b4,
        "AFPFeatureStandardizer"
    )
)


assert len(
    scorer_matches
) >= 1, (
    "Persisted AFPScorer class definition not found."
)

assert len(
    standardizer_matches
) >= 1, (
    "Persisted AFPFeatureStandardizer class definition not found."
)


# Latest persisted definition.
scorer_match = (
    scorer_matches[
        -1
    ]
)

standardizer_match = (
    standardizer_matches[
        -1
    ]
)

scorer_cell_idx = (
    scorer_match[
        "cell_idx"
    ]
)

scorer_source = (
    scorer_match[
        "source"
    ]
)

std_cell_idx = (
    standardizer_match[
        "cell_idx"
    ]
)

std_source = (
    standardizer_match[
        "source"
    ]
)


print(
    "\nRecovered classes:"
)

print(
    "  AFPScorer:",
    f"cell {scorer_cell_idx}"
)

print(
    "  AFPFeatureStandardizer:",
    f"cell {std_cell_idx}"
)


# ======================================================================
# 5. RECOVER CLASS NAMESPACE DEPENDENCIES
# ======================================================================
#
# Previous B4 failure:
#
#     NameError: AFP_INPUT_DIM is not defined
#
# The persisted class uses AFP_* constants in constructor defaults.
#
# Here we:
#   - start from the verified current runtime namespace,
#   - recover literal AFP_* constants from the class source cell,
#   - explicitly bind AFP_INPUT_DIM = 27 from verified Feature-v2.
#
# No notebook training code is executed.
# ======================================================================

class_ns = dict(
    globals()
)

class_ns.update(
    {
        "np":
            np,

        "torch":
            torch,

        "nn":
            nn,

        "F":
            F,

        "Path":
            Path,

        "Optional":
            Optional,

        "Dict":
            Dict,

        "List":
            List,

        "Tuple":
            Tuple,

        "Any":
            Any,
    }
)


def recover_literal_afp_constants_from_cell(
    notebook,
    cell_idx
):

    source = "".join(
        notebook[
            "cells"
        ][
            cell_idx
        ].get(
            "source",
            []
        )
    )

    tree = ast.parse(
        source
    )

    recovered = {}


    for node in tree.body:

        # --------------------------------------------------------------
        # AFP_SOMETHING = literal
        # --------------------------------------------------------------

        if isinstance(
            node,
            ast.Assign
        ):

            try:

                value = ast.literal_eval(
                    node.value
                )

            except Exception:

                continue


            for target in node.targets:

                if (
                    isinstance(
                        target,
                        ast.Name
                    )
                    and
                    target.id.startswith(
                        "AFP_"
                    )
                ):

                    recovered[
                        target.id
                    ] = value


        # --------------------------------------------------------------
        # AFP_SOMETHING: type = literal
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.AnnAssign
        ):

            if not (
                isinstance(
                    node.target,
                    ast.Name
                )
                and
                node.target.id.startswith(
                    "AFP_"
                )
            ):

                continue

            if node.value is None:
                continue


            try:

                value = ast.literal_eval(
                    node.value
                )

            except Exception:

                continue


            recovered[
                node.target.id
            ] = value


    return recovered


dependency_cells = sorted(
    set(
        [
            scorer_cell_idx,
            std_cell_idx,
        ]
    )
)


recovered_afp_constants = {}


for cell_idx in dependency_cells:

    recovered_afp_constants.update(
        recover_literal_afp_constants_from_cell(
            nb_b4,
            cell_idx
        )
    )


for name, value in (
    recovered_afp_constants.items()
):

    class_ns[
        name
    ] = value


# ----------------------------------------------------------------------
# Verified dependency, NOT guessed.
# ----------------------------------------------------------------------

class_ns[
    "AFP_INPUT_DIM"
] = int(
    AFP_RUNTIME_FEATURE_DIM
)


print(
    "\nRecovered scorer namespace dependencies:"
)

print(
    "  AFP_INPUT_DIM =",
    class_ns[
        "AFP_INPUT_DIM"
    ]
)


if recovered_afp_constants:

    print(
        "  Persisted literal AFP_* constants:"
    )

    for name in sorted(
        recovered_afp_constants
    ):

        print(
            f"    {name} = "
            f"{recovered_afp_constants[name]!r}"
        )

else:

    print(
        "  No additional literal AFP_* constants found."
    )


# ======================================================================
# 6. EXECUTE ONLY THE EXACT PERSISTED CLASS DEFINITIONS
# ======================================================================

exec(
    scorer_source,
    class_ns
)

exec(
    std_source,
    class_ns
)


AFPScorerExact = (
    class_ns[
        "AFPScorer"
    ]
)

AFPFeatureStandardizerExact = (
    class_ns[
        "AFPFeatureStandardizer"
    ]
)


print(
    "\nExact persisted classes: LOADED"
)


print(
    "\nAFPScorer signature:"
)

print(
    inspect.signature(
        AFPScorerExact
    )
)


print(
    "AFPFeatureStandardizer signature:"
)

print(
    inspect.signature(
        AFPFeatureStandardizerExact
    )
)


scorer_signature = (
    inspect.signature(
        AFPScorerExact
    )
)


assert (
    "input_dim"
    in scorer_signature.parameters
), (
    "Persisted AFPScorer does not expose input_dim."
)


input_default = (
    scorer_signature
    .parameters[
        "input_dim"
    ]
    .default
)


if (
    input_default
    is not
    inspect._empty
):

    assert int(
        input_default
    ) == 27, (
        "Persisted scorer input dimension "
        "does not equal Feature-v2 dimension 27."
    )


print(
    "AFPScorer input-dimension gate: PASSED"
)


# ======================================================================
# 7. LOAD SELECTED CHECKPOINTS ON CPU
# ======================================================================

def torch_load_cpu(
    path
):

    try:

        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        return torch.load(
            path,
            map_location="cpu"
        )


webqsp_ckpt_obj = (
    torch_load_cpu(
        WEBQSP_CKPT
    )
)

cwq_ckpt_obj = (
    torch_load_cpu(
        CWQ_CKPT
    )
)


print(
    "\nCheckpoint top-level types:"
)

print(
    "  WebQSP:",
    type(
        webqsp_ckpt_obj
    )
)

print(
    "  CWQ:   ",
    type(
        cwq_ckpt_obj
    )
)


if isinstance(
    webqsp_ckpt_obj,
    dict
):

    print(
        "  WebQSP keys:",
        sorted(
            webqsp_ckpt_obj.keys()
        )
    )


if isinstance(
    cwq_ckpt_obj,
    dict
):

    print(
        "  CWQ keys:",
        sorted(
            cwq_ckpt_obj.keys()
        )
    )


# ======================================================================
# 8. EXTRACT MODEL STATE_DICT
# ======================================================================

MODEL_STATE_KEYS = [
    "model_state_dict",
    "state_dict",
    "model_state",
    "scorer_state_dict",
    "scorer_state",
]


def looks_like_state_dict(
    obj
):

    if not isinstance(
        obj,
        dict
    ):
        return False

    if len(
        obj
    ) == 0:
        return False

    tensor_count = sum(
        int(
            torch.is_tensor(
                value
            )
        )
        for value in obj.values()
    )

    if tensor_count == 0:
        return False

    return any(
        str(
            key
        ).endswith(
            "weight"
        )
        for key in obj.keys()
    )


def extract_model_state(
    checkpoint
):

    if looks_like_state_dict(
        checkpoint
    ):

        return checkpoint


    assert isinstance(
        checkpoint,
        dict
    )


    # Direct known locations.
    for key in MODEL_STATE_KEYS:

        if key not in checkpoint:
            continue

        value = checkpoint[
            key
        ]

        if looks_like_state_dict(
            value
        ):

            return value


    # One nested level.
    for parent_key, parent_value in (
        checkpoint.items()
    ):

        if not isinstance(
            parent_value,
            dict
        ):
            continue

        for key in MODEL_STATE_KEYS:

            if key not in parent_value:
                continue

            value = parent_value[
                key
            ]

            if looks_like_state_dict(
                value
            ):

                return value


    raise AssertionError(
        "Could not locate scorer state_dict "
        "inside selected checkpoint."
    )


webqsp_model_state = (
    extract_model_state(
        webqsp_ckpt_obj
    )
)

cwq_model_state = (
    extract_model_state(
        cwq_ckpt_obj
    )
)


print(
    "\nModel state keys"
)

print(
    "  WebQSP:",
    list(
        webqsp_model_state.keys()
    )
)

print(
    "  CWQ:   ",
    list(
        cwq_model_state.keys()
    )
)


# ======================================================================
# 9. INFER FROZEN MLP ARCHITECTURE DIRECTLY FROM WEIGHTS
# ======================================================================

def infer_mlp_dimensions(
    state_dict
):

    matrix_weights = []

    for key, value in (
        state_dict.items()
    ):

        if (
            torch.is_tensor(
                value
            )
            and
            value.ndim == 2
        ):

            matrix_weights.append(
                (
                    str(
                        key
                    ),
                    tuple(
                        value.shape
                    )
                )
            )


    assert len(
        matrix_weights
    ) == 2, (
        "Expected exactly two 2-D Linear weight tensors; "
        f"found {matrix_weights}"
    )


    first_candidates = [
        item
        for item in matrix_weights
        if item[
            1
        ][
            1
        ] == 27
    ]


    assert len(
        first_candidates
    ) == 1, (
        "Could not uniquely identify first 27-D Linear layer."
    )


    first_key, first_shape = (
        first_candidates[
            0
        ]
    )


    hidden_dim = int(
        first_shape[
            0
        ]
    )


    second_candidates = [
        item
        for item in matrix_weights
        if item[
            1
        ] == (
            1,
            hidden_dim
        )
    ]


    assert len(
        second_candidates
    ) == 1, (
        "Could not uniquely identify output Linear layer."
    )


    second_key = (
        second_candidates[
            0
        ][
            0
        ]
    )


    return {
        "input_dim":
            27,

        "hidden_dim":
            hidden_dim,

        "output_dim":
            1,

        "first_weight_key":
            first_key,

        "second_weight_key":
            second_key,
    }


webqsp_arch = (
    infer_mlp_dimensions(
        webqsp_model_state
    )
)

cwq_arch = (
    infer_mlp_dimensions(
        cwq_model_state
    )
)


print(
    "\nRecovered scorer architectures"
)

print(
    "  WebQSP:",
    webqsp_arch
)

print(
    "  CWQ:   ",
    cwq_arch
)


assert (
    webqsp_arch[
        "hidden_dim"
    ]
    == 32
), (
    "WebQSP selected scorer should be H=32."
)

assert (
    cwq_arch[
        "hidden_dim"
    ]
    == 64
), (
    "CWQ selected scorer should be H=64."
)


print(
    "Selected H=32/H=64 architecture gate: PASSED"
)


# ======================================================================
# 10. INSTANTIATE EXACT PERSISTED SCORER
# ======================================================================

def instantiate_exact_scorer(
    hidden_dim
):

    signature = (
        inspect.signature(
            AFPScorerExact
        )
    )

    kwargs = {}


    for name, param in (
        signature.parameters.items()
    ):

        lname = str(
            name
        ).lower()


        if lname in {
            "input_dim",
            "in_dim",
            "feature_dim",
            "n_features",
        }:

            kwargs[
                name
            ] = 27


        elif lname in {
            "hidden_dim",
            "hidden_size",
            "hidden",
        }:

            kwargs[
                name
            ] = int(
                hidden_dim
            )


        elif lname in {
            "dropout",
            "dropout_p",
            "dropout_rate",
        }:

            kwargs[
                name
            ] = 0.0


        elif (
            param.default
            is not
            inspect._empty
        ):

            # Preserve persisted default.
            continue


        else:

            raise AssertionError(
                "Unknown required AFPScorer constructor "
                f"parameter: {name}"
            )


    model = AFPScorerExact(
        **kwargs
    )


    model = model.to(
        DEVICE
    )

    model.eval()

    return model


webqsp_scorer = (
    instantiate_exact_scorer(
        hidden_dim=32
    )
)

cwq_scorer = (
    instantiate_exact_scorer(
        hidden_dim=64
    )
)


webqsp_scorer.load_state_dict(
    webqsp_model_state,
    strict=True
)

cwq_scorer.load_state_dict(
    cwq_model_state,
    strict=True
)


webqsp_scorer.eval()
cwq_scorer.eval()


print(
    "\nExact scorer state restoration: PASSED"
)


# ======================================================================
# 11. RECOVER FROZEN TRAIN-ONLY STANDARDIZER
# ======================================================================

def to_numpy_float32(
    value
):

    if torch.is_tensor(
        value
    ):

        value = (
            value
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        value,
        dtype=np.float32
    )


def find_mean_std_dict(
    obj,
    path="root"
):

    matches = []


    if not isinstance(
        obj,
        dict
    ):

        return matches


    lowercase_keys = {
        str(
            key
        ).lower():
            key
        for key in obj.keys()
    }


    possible_mean_keys = [
        "mean",
        "mean_",
        "feature_mean",
        "feature_means",
    ]

    possible_std_keys = [
        "std",
        "std_",
        "feature_std",
        "feature_stds",
        "scale",
        "scale_",
    ]


    mean_key = next(
        (
            lowercase_keys[
                name
            ]
            for name in possible_mean_keys
            if name in lowercase_keys
        ),
        None
    )

    std_key = next(
        (
            lowercase_keys[
                name
            ]
            for name in possible_std_keys
            if name in lowercase_keys
        ),
        None
    )


    if (
        mean_key is not None
        and
        std_key is not None
    ):

        try:

            mean = to_numpy_float32(
                obj[
                    mean_key
                ]
            )

            std = to_numpy_float32(
                obj[
                    std_key
                ]
            )

        except Exception:

            mean = None
            std = None


        if (
            mean is not None
            and
            std is not None
            and
            mean.shape == (
                27,
            )
            and
            std.shape == (
                27,
            )
        ):

            matches.append(
                {
                    "path":
                        path,

                    "mean":
                        mean,

                    "std":
                        std,
                }
            )


    for key, value in (
        obj.items()
    ):

        if isinstance(
            value,
            dict
        ):

            matches.extend(
                find_mean_std_dict(
                    value,
                    path=(
                        f"{path}.{key}"
                    )
                )
            )


    return matches


def extract_standardizer_state(
    checkpoint,
    dataset_name
):

    matches = find_mean_std_dict(
        checkpoint
    )


    assert len(
        matches
    ) >= 1, (
        f"{dataset_name}: no frozen 27-D "
        "standardizer mean/std found."
    )


    reference = (
        matches[
            0
        ]
    )


    # Multiple copies are acceptable only if identical.
    for other in matches[
        1:
    ]:

        assert np.array_equal(
            reference[
                "mean"
            ],
            other[
                "mean"
            ]
        ), (
            f"{dataset_name}: inconsistent standardizer means."
        )

        assert np.array_equal(
            reference[
                "std"
            ],
            other[
                "std"
            ]
        ), (
            f"{dataset_name}: inconsistent standardizer stds."
        )


    return reference


webqsp_std_state = (
    extract_standardizer_state(
        webqsp_ckpt_obj,
        "WebQSP"
    )
)

cwq_std_state = (
    extract_standardizer_state(
        cwq_ckpt_obj,
        "CWQ"
    )
)


print(
    "\nFrozen standardizer state"
)

print(
    "  WebQSP source:",
    webqsp_std_state[
        "path"
    ]
)

print(
    "  CWQ source:   ",
    cwq_std_state[
        "path"
    ]
)


for dataset_name, state in [
    (
        "WebQSP",
        webqsp_std_state
    ),
    (
        "CWQ",
        cwq_std_state
    ),
]:

    assert state[
        "mean"
    ].shape == (
        27,
    )

    assert state[
        "std"
    ].shape == (
        27,
    )

    assert np.all(
        np.isfinite(
            state[
                "mean"
            ]
        )
    )

    assert np.all(
        np.isfinite(
            state[
                "std"
            ]
        )
    )

    assert np.all(
        state[
            "std"
        ] > 0
    ), (
        f"{dataset_name}: standardizer contains non-positive std."
    )


print(
    "Frozen standardizer validity: PASSED"
)


# ======================================================================
# 12. STANDARDIZATION FUNCTION
# ======================================================================

def apply_frozen_standardizer(
    X,
    state
):

    X = np.asarray(
        X,
        dtype=np.float32
    )


    mean = np.asarray(
        state[
            "mean"
        ],
        dtype=np.float32
    )

    std = np.asarray(
        state[
            "std"
        ],
        dtype=np.float32
    )


    assert X.ndim == 2
    assert X.shape[
        1
    ] == 27


    return (
        (
            X
            - mean
        )
        /
        std
    ).astype(
        np.float32
    )


# ======================================================================
# 13. SCORER LOGIT INFERENCE
# ======================================================================

def scorer_logits(
    model,
    X_standardized
):

    X_standardized = np.asarray(
        X_standardized,
        dtype=np.float32
    )


    assert X_standardized.ndim == 2
    assert X_standardized.shape[
        1
    ] == 27


    with torch.inference_mode():

        tensor = (
            torch.from_numpy(
                X_standardized
            )
            .to(
                DEVICE
            )
        )


        output = model(
            tensor
        )


        logits = (
            output
            .reshape(
                -1
            )
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )


    assert logits.shape == (
        X_standardized.shape[
            0
        ],
    )


    return logits


# ======================================================================
# 14. FROZEN NPZ vs RECOVERED ONLINE LOGIT FIDELITY
# ======================================================================

def sigmoid_numpy(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -logits
            )
        )
    )


def evaluate_logit_fidelity(
    dataset_name,
    frozen_features,
    runtime_features,
    standardizer_state,
    scorer
):

    X_frozen = np.asarray(
        frozen_features[
            "X"
        ],
        dtype=np.float32
    )

    X_runtime = np.asarray(
        runtime_features[
            "X"
        ],
        dtype=np.float32
    )


    assert X_frozen.shape == (
        X_runtime.shape
    )


    Z_frozen = (
        apply_frozen_standardizer(
            X_frozen,
            standardizer_state
        )
    )

    Z_runtime = (
        apply_frozen_standardizer(
            X_runtime,
            standardizer_state
        )
    )


    logits_frozen = (
        scorer_logits(
            scorer,
            Z_frozen
        )
    )

    logits_runtime = (
        scorer_logits(
            scorer,
            Z_runtime
        )
    )


    probs_frozen = sigmoid_numpy(
        logits_frozen
    )

    probs_runtime = sigmoid_numpy(
        logits_runtime
    )


    feature_diff = np.abs(
        X_frozen.astype(
            np.float64
        )
        -
        X_runtime.astype(
            np.float64
        )
    )

    standardized_diff = np.abs(
        Z_frozen.astype(
            np.float64
        )
        -
        Z_runtime.astype(
            np.float64
        )
    )

    logit_diff = np.abs(
        logits_frozen.astype(
            np.float64
        )
        -
        logits_runtime.astype(
            np.float64
        )
    )

    probability_diff = np.abs(
        probs_frozen
        -
        probs_runtime
    )


    report = {
        "n_branches":
            int(
                X_frozen.shape[
                    0
                ]
            ),

        "max_feature_diff":
            float(
                feature_diff.max()
            ),

        "mean_feature_diff":
            float(
                feature_diff.mean()
            ),

        "max_standardized_diff":
            float(
                standardized_diff.max()
            ),

        "mean_standardized_diff":
            float(
                standardized_diff.mean()
            ),

        "max_logit_diff":
            float(
                logit_diff.max()
            ),

        "mean_logit_diff":
            float(
                logit_diff.mean()
            ),

        "max_probability_diff":
            float(
                probability_diff.max()
            ),

        "mean_probability_diff":
            float(
                probability_diff.mean()
            ),
    }


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} "
        "SCORER LOGIT FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in (
        report.items()
    ):

        print(
            f"{key:<30}",
            value
        )


    return {
        "report":
            report,

        "logits_frozen":
            logits_frozen,

        "logits_runtime":
            logits_runtime,

        "standardized_frozen":
            Z_frozen,

        "standardized_runtime":
            Z_runtime,
    }


webqsp_logit_result = (
    evaluate_logit_fidelity(
        dataset_name=
            "webqsp",

        frozen_features=
            webqsp_frozen_final,

        runtime_features=
            webqsp_runtime_features,

        standardizer_state=
            webqsp_std_state,

        scorer=
            webqsp_scorer
    )
)


cwq_logit_result = (
    evaluate_logit_fidelity(
        dataset_name=
            "cwq",

        frozen_features=
            cwq_frozen_final,

        runtime_features=
            cwq_runtime_features,

        standardizer_state=
            cwq_std_state,

        scorer=
            cwq_scorer
    )
)


webqsp_logit_fidelity = (
    webqsp_logit_result[
        "report"
    ]
)

cwq_logit_fidelity = (
    cwq_logit_result[
        "report"
    ]
)

webqsp_frozen_logits = (
    webqsp_logit_result[
        "logits_frozen"
    ]
)

webqsp_runtime_logits = (
    webqsp_logit_result[
        "logits_runtime"
    ]
)

cwq_frozen_logits = (
    cwq_logit_result[
        "logits_frozen"
    ]
)

cwq_runtime_logits = (
    cwq_logit_result[
        "logits_runtime"
    ]
)


# ======================================================================
# 15. HARD LOGIT FIDELITY GATE
# ======================================================================
#
# Tiny Feature-v2 float32 differences can be amplified by standardized
# low-variance columns.
#
# Observable scorer logits are therefore the main software-fidelity gate.
#
# Do NOT loosen this automatically if it fails.
# ======================================================================

LOGIT_ATOL = 1e-4


for dataset_name, result in [
    (
        "WebQSP",
        webqsp_logit_fidelity
    ),
    (
        "CWQ",
        cwq_logit_fidelity
    ),
]:

    assert (
        result[
            "max_logit_diff"
        ]
        <= LOGIT_ATOL
    ), (
        f"{dataset_name}: scorer logit fidelity failed. "
        "Do NOT increase LOGIT_ATOL automatically."
    )


print(
    "\nFrozen-NPZ -> online-logit fidelity: PASSED"
)


# ======================================================================
# 16. EXPOSE FINAL RUNTIME SCORERS/STANDARDIZERS
# ======================================================================

AFP_RUNTIME_SCORERS = {
    "webqsp":
        webqsp_scorer,

    "cwq":
        cwq_scorer,
}


AFP_RUNTIME_STANDARDIZERS = {
    "webqsp": {
        "mean":
            webqsp_std_state[
                "mean"
            ].copy(),

        "std":
            webqsp_std_state[
                "std"
            ].copy(),
    },

    "cwq": {
        "mean":
            cwq_std_state[
                "mean"
            ].copy(),

        "std":
            cwq_std_state[
                "std"
            ].copy(),
    },
}


# ======================================================================
# 17. GOLD-FREE ONLINE GROUP SCORING CALLBACK
# ======================================================================
#
# This is the scorer interface for the later online validation traversal.
#
# Inputs available at inference time:
#   - question ID
#   - question text
#   - relation plan
#   - hop
#   - current candidate group
#
# NO:
#   - answer entities
#   - gold labels
#   - suffix reachability labels
# ======================================================================

def afp_runtime_score_group(
    dataset_name,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    dataset_key = str(
        dataset_name
    ).strip().lower()


    assert dataset_key in {
        "webqsp",
        "cwq",
    }


    assert len(
        candidate_rows
    ) >= 1


    X = (
        AFP_RUNTIME_FEATURE_EXTRACTOR(
            question_id=
                question_id,

            question=
                question,

            plan=
                list(
                    plan
                ),

            hop=
                int(
                    hop
                ),

            candidate_rows=
                candidate_rows,

            semantic_encoder=
                AFP_RUNTIME_SEMANTIC_ENCODER,

            entity_name_map=
                None
        )
    )


    X = np.asarray(
        X,
        dtype=np.float32
    )


    assert X.shape == (
        len(
            candidate_rows
        ),
        27
    )


    Z = (
        apply_frozen_standardizer(
            X,
            AFP_RUNTIME_STANDARDIZERS[
                dataset_key
            ]
        )
    )


    logits = (
        scorer_logits(
            AFP_RUNTIME_SCORERS[
                dataset_key
            ],
            Z
        )
    )


    assert logits.shape == (
        len(
            candidate_rows
        ),
    )


    return {
        "features":
            X,

        "standardized_features":
            Z,

        "logits":
            logits,
    }


AFP_RUNTIME_SCORE_GROUP = (
    afp_runtime_score_group
)


# ======================================================================
# 18. GROUP-LEVEL CALLBACK SPOT CHECK
# ======================================================================

def callback_spotcheck(
    dataset_name,
    dataset_rows,
    labels_file,
    frozen_features,
    frozen_logits,
    requested_group_indices
):

    groups = list(
        iter_runtime_groups(
            labels_file
        )
    )


    group_ptr = np.asarray(
        frozen_features[
            "group_ptr"
        ],
        dtype=np.int64
    )


    assert len(
        group_ptr
    ) == (
        len(
            groups
        )
        + 1
    )


    results = []


    for group_index in (
        requested_group_indices
    ):

        rows = groups[
            group_index
        ]

        first = rows[
            0
        ]


        source_index = int(
            first[
                "source_index"
            ]
        )


        rec = dataset_rows[
            source_index
        ]


        assert str(
            rec[
                "id"
            ]
        ) == str(
            first[
                "question_id"
            ]
        )


        runtime_output = (
            AFP_RUNTIME_SCORE_GROUP(
                dataset_name=
                    dataset_name,

                question_id=
                    first[
                        "question_id"
                    ],

                question=
                    rec[
                        "question"
                    ],

                plan=
                    first[
                        "plan"
                    ],

                hop=
                    first[
                        "hop"
                    ],

                candidate_rows=
                    rows
            )
        )


        start = int(
            group_ptr[
                group_index
            ]
        )

        end = int(
            group_ptr[
                group_index
                + 1
            ]
        )


        reference_logits = (
            frozen_logits[
                start:end
            ]
        )


        runtime_logits_group = (
            runtime_output[
                "logits"
            ]
        )


        assert reference_logits.shape == (
            runtime_logits_group.shape
        )


        max_logit_diff = float(
            np.max(
                np.abs(
                    reference_logits.astype(
                        np.float64
                    )
                    -
                    runtime_logits_group.astype(
                        np.float64
                    )
                )
            )
        )


        results.append(
            {
                "group_index":
                    int(
                        group_index
                    ),

                "source_index":
                    source_index,

                "hop":
                    int(
                        first[
                            "hop"
                        ]
                    ),

                "candidate_count":
                    int(
                        len(
                            rows
                        )
                    ),

                "max_logit_diff":
                    max_logit_diff,
            }
        )


    return results


webqsp_n_groups = (
    len(
        webqsp_frozen_final[
            "group_ptr"
        ]
    )
    - 1
)

cwq_n_groups = (
    len(
        cwq_frozen_final[
            "group_ptr"
        ]
    )
    - 1
)


webqsp_spot_groups = sorted(
    set(
        [
            0,
            webqsp_n_groups // 4,
            webqsp_n_groups // 2,
            (
                3
                * webqsp_n_groups
            )
            // 4,
            webqsp_n_groups - 1,
        ]
    )
)


cwq_spot_groups = sorted(
    set(
        [
            0,
            cwq_n_groups // 4,
            cwq_n_groups // 2,
            (
                3
                * cwq_n_groups
            )
            // 4,
            cwq_n_groups - 1,
        ]
    )
)


webqsp_callback_check = (
    callback_spotcheck(
        dataset_name=
            "webqsp",

        dataset_rows=
            webqsp_val_runtime,

        labels_file=
            WEBQSP_LABEL_FILE,

        frozen_features=
            webqsp_frozen_final,

        frozen_logits=
            webqsp_frozen_logits,

        requested_group_indices=
            webqsp_spot_groups
    )
)


cwq_callback_check = (
    callback_spotcheck(
        dataset_name=
            "cwq",

        dataset_rows=
            cwq_val_runtime,

        labels_file=
            CWQ_LABEL_FILE,

        frozen_features=
            cwq_frozen_final,

        frozen_logits=
            cwq_frozen_logits,

        requested_group_indices=
            cwq_spot_groups
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "ONLINE CALLBACK SPOT-CHECK"
)

print(
    "=" * 100
)


for dataset_name, rows in [
    (
        "WebQSP",
        webqsp_callback_check
    ),
    (
        "CWQ",
        cwq_callback_check
    ),
]:

    print(
        f"\n{dataset_name}"
    )


    for row in rows:

        print(
            "  "
            f"group={row['group_index']:<5} "
            f"hop={row['hop']:<2} "
            f"n={row['candidate_count']:<4} "
            f"max_logit_diff="
            f"{row['max_logit_diff']:.10g}"
        )


        assert (
            row[
                "max_logit_diff"
            ]
            <= LOGIT_ATOL
        )


print(
    "\nOnline scorer callback: PASSED"
)


# ======================================================================
# 19. FINAL RUNTIME OBJECT GATES
# ======================================================================

assert callable(
    AFP_RUNTIME_SCORE_GROUP
)

assert set(
    AFP_RUNTIME_SCORERS.keys()
) == {
    "webqsp",
    "cwq",
}

assert set(
    AFP_RUNTIME_STANDARDIZERS.keys()
) == {
    "webqsp",
    "cwq",
}


AFP_RUNTIME_SCORER_READY = True


print(
    "\nRuntime scorer objects: READY"
)


# ======================================================================
# 20. SAVE B4 MANIFEST
# ======================================================================

TRAVERSAL_DIR = (
    ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


B4_MANIFEST = {
    "cell":
        "RQ2_12C_B4_REVISED",

    "runtime_device":
        "cpu",

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "feature_dim":
        27,

    "scorer_version":
        "afp_mlp_v1",

    "selected_checkpoints": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_CKPT
                ),

            "sha256":
                webqsp_ckpt_sha,

            "hidden_dim":
                32,

            "loss":
                "branch_bce",

            "deployment_seed":
                42,
        },

        "cwq": {
            "path":
                str(
                    CWQ_CKPT
                ),

            "sha256":
                cwq_ckpt_sha,

            "hidden_dim":
                64,

            "loss":
                "branch_bce",

            "deployment_seed":
                42,
        },
    },

    "persisted_class_recovery": {
        "afp_scorer_cell":
            int(
                scorer_cell_idx
            ),

        "afp_standardizer_cell":
            int(
                std_cell_idx
            ),

        "afp_input_dim":
            27,

        "class_namespace_dependency_recovery":
            True,
    },

    "standardization": {
        "source":
            "selected_checkpoint_train_only_state",

        "formula":
            "(X - mean) / std",
    },

    "webqsp_logit_fidelity":
        webqsp_logit_fidelity,

    "cwq_logit_fidelity":
        cwq_logit_fidelity,

    "logit_atol":
        LOGIT_ATOL,

    "callback_spotcheck": {
        "webqsp":
            webqsp_callback_check,

        "cwq":
            cwq_callback_check,
    },

    "online_score_callback_ready":
        True,

    "gold_used_by_runtime_callback":
        False,

    "pruning_run":
        False,

    "afp_hyperparameter_tuning_run":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,
}


B4_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b4_scorer_runtime_fidelity.json"
)


with open(
    B4_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        B4_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 21. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "=== RQ2 CELL 12C-B4: ONLINE AFP SCORER RUNTIME VERIFIED ==="
)

print(
    "=" * 116
)


print(
    "\nWebQSP selected scorer:"
)

print(
    "  hidden dimension: 32"
)

print(
    "  loss:             branch_bce"
)

print(
    "  deployment seed:  42"
)

print(
    "  checkpoint SHA:  ",
    webqsp_ckpt_sha
)


print(
    "\nCWQ selected scorer:"
)

print(
    "  hidden dimension: 64"
)

print(
    "  loss:             branch_bce"
)

print(
    "  deployment seed:  42"
)

print(
    "  checkpoint SHA:  ",
    cwq_ckpt_sha
)


print(
    "\nRuntime integration:"
)

print(
    "  Exact persisted scorer class: RESTORED"
)

print(
    "  AFP_INPUT_DIM dependency:     RESTORED"
)

print(
    "  Frozen train standardizers:   RESTORED"
)

print(
    "  Selected scorer weights:      RESTORED"
)

print(
    "  Frozen-vs-online logits:      VERIFIED"
)

print(
    "  Online group callback:        VERIFIED"
)

print(
    "  Gold in runtime callback:     NO"
)

print(
    "  Pruning run:                  NO"
)

print(
    "  Hyperparameter tuning:        NO"
)

print(
    "  TEST examples accessed:       NO"
)

print(
    "  Complete AFP frozen:          NO"
)


print(
    "\nManifest:",
    B4_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 13 — VALIDATION-ONLY hyperparameter tuning "
    "for AFP (T, gamma_min) and controlled baselines."
)

Runtime device: cpu
Feature runtime: afp_features_v2_masked_entity_semantics
Feature dimension: 27

Selected checkpoints:
  WebQSP: /kaggle/working/step3_rq2_dev_v1/07_final_scorer/webqsp_afp_scorer_selected.pt
  CWQ:    /kaggle/working/step3_rq2_dev_v1/07_final_scorer/cwq_afp_scorer_selected.pt

Checkpoint SHA256
  WebQSP: bee65146403d565661b5105c41831af3e81d54ed5fba1b2a99bcb37382420d9f
  CWQ:    91a531c057bb02d0b311ef8b78bf02248a63cd66e57fe8959ef86e7297d9d0a1
Selected checkpoint identity: PASSED

Recovered classes:
  AFPScorer: cell 175
  AFPFeatureStandardizer: cell 175

Recovered scorer namespace dependencies:
  AFP_INPUT_DIM = 27
  Persisted literal AFP_* constants:
    AFP_DROPOUT = 0.0
    AFP_HIDDEN_CANDIDATES = [32, 64, 128]
    AFP_SCORER_VERSION = 'afp_mlp_v1'

Exact persisted classes: LOADED

AFPScorer signature:
(input_dim=27, hidden_dim=64, dropout=0.0)
AFPFeatureStandardizer signature:
(eps=1e-08)
AFPScorer input-dimension gate: PASSED

Checkpoint top-level types:
  WebQ

## Validation hyperparameter tuning — \(T\), \(\gamma_{\min}\), Top-\(B\), threshold

In [27]:
# ======================================================================
# RQ2 CELL 13 — REVISED v2
# VALIDATION-ONLY HYPERPARAMETER TUNING
# ======================================================================
#
# FIX IN THIS VERSION
# -------------------
# RQ1 metric distinction is now explicit:
#
#   active_hop_rows:
#       number of non-empty plan-hop profiler rows
#
#   active_prefixes:
#       sum_h |P_h| across those rows
#
# Verified RQ1 validation references:
#
#   WebQSP:
#       active_hop_rows = 971
#       active_prefixes = 2440
#
#   CWQ:
#       active_hop_rows = 16564
#       active_prefixes = 57841
#
# TUNING
# ------
# AFP:
#   T
#   gamma_min
#
# Fixed Top-B:
#   B
#
# Fixed Threshold:
#   tau
#
# PREDECLARED SELECTION RULE
# --------------------------
# Primary:
#     maximize SSR subject to validation AR >= 0.99
#
# Fallback:
#     maximize AR first, then SSR
#
# IMPORTANT
# ---------
# - validation only
# - selected scorer already frozen
# - Feature-v2 already frozen
# - relation plans frozen
# - exact RoG graph semantics frozen
# - final-hop protection enabled
# - singleton bypass enabled
# - gold NEVER enters scorer or selector
# - gold used only after traversal for validation Answer Retention
# - no TEST examples accessed
#
# CPU is sufficient.
#
# This version additionally uses:
#   - batched semantic prefill
#   - question-local graph-expansion cache
#   - question-local scorer cache
#
# These are COMPUTATIONAL optimizations only.
# They do not alter any method metric or selection behavior.
# ======================================================================

import hashlib
import json
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "AFP_RUNTIME_SCORE_GROUP",
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_SCORERS",
    "AFP_RUNTIME_STANDARDIZERS",

    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",

    "has_readable_entity_surface_runtime",
    "entity_surface_text_runtime",
    "relation_surface_text_runtime",
    "relation_sequence_text_runtime",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required runtime objects:\n  "
    + "\n  ".join(missing)
)

assert int(
    AFP_RUNTIME_FEATURE_DIM
) == 27


print(
    "Cell 13 prerequisites: PASSED"
)

print(
    "Runtime scorer: READY"
)

print(
    "Runtime device: CPU"
)

print(
    "Feature version:",
    AFP_RUNTIME_FEATURE_VERSION
)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/"
    "step3_rq2_dev_v1"
)

TUNING_DIR = (
    ROOT
    / "11_validation_tuning"
)

TUNING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PREDECLARED TUNING GRID
# ======================================================================

AR_FLOOR = 0.99


T_GRID = [
    0.5,
    1.0,
    2.0,
]


GAMMA_MIN_GRID = [
    0.50,
    0.70,
    0.80,
    0.90,
    0.95,
]


TOP_B_GRID = [
    1,
    2,
    4,
    8,
    16,
    32,
]


THRESHOLD_GRID = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]


TIE_ATOL = 1e-8


print(
    "\n"
    + "=" * 108
)

print(
    "PREDECLARED VALIDATION MODEL-SELECTION RULE"
)

print(
    "=" * 108
)


print(
    "Primary objective: maximize SSR "
    f"subject to AR >= {AR_FLOOR}"
)

print(
    "Fallback: maximize AR, then SSR, "
    "if no configuration satisfies AR floor."
)


print(
    "\nAFP grid"
)

print(
    "  T:         ",
    T_GRID
)

print(
    "  gamma_min: ",
    GAMMA_MIN_GRID
)


print(
    "\nFixed Top-B grid:"
)

print(
    " ",
    TOP_B_GRID
)


print(
    "\nFixed Threshold grid:"
)

print(
    " ",
    THRESHOLD_GRID
)


print(
    "\nRandom-B: inherits selected Fixed Top-B B; "
    "NOT independently tuned."
)

print(
    "Adaptive-Budget Random: inherits selected AFP adaptive "
    "budgets; NOT independently tuned."
)


# ======================================================================
# 4. CONFIGURATION LIST
# ======================================================================

CONFIGS = []


for T in T_GRID:

    for gamma_min in GAMMA_MIN_GRID:

        CONFIGS.append(
            {
                "config_id":
                    f"afp_T{T:g}_g{gamma_min:g}",

                "family":
                    "afp",

                "T":
                    float(
                        T
                    ),

                "gamma_min":
                    float(
                        gamma_min
                    ),
            }
        )


for B in TOP_B_GRID:

    CONFIGS.append(
        {
            "config_id":
                f"fixed_top_b_B{B}",

            "family":
                "fixed_top_b",

            "B":
                int(
                    B
                ),
        }
    )


for tau in THRESHOLD_GRID:

    CONFIGS.append(
        {
            "config_id":
                f"fixed_threshold_tau{tau:g}",

            "family":
                "fixed_threshold",

            "tau":
                float(
                    tau
                ),
        }
    )


assert len(
    CONFIGS
) == (
    len(
        T_GRID
    )
    * len(
        GAMMA_MIN_GRID
    )
    + len(
        TOP_B_GRID
    )
    + len(
        THRESHOLD_GRID
    )
)


CONFIG_BY_ID = {
    config[
        "config_id"
    ]:
        config

    for config in CONFIGS
}


print(
    "\nTotal validation configurations:",
    len(
        CONFIGS
    )
)


# ======================================================================
# 5. TUNING SPECIFICATION FINGERPRINT
# ======================================================================

TUNING_SPEC = {
    "version":
        "rq2_cell13_validation_tuning_v2_metric_corrected",

    "selection_rule": {
        "primary":
            "maximize_ssr_subject_to_ar_floor",

        "ar_floor":
            AR_FLOOR,

        "fallback":
            "maximize_ar_then_ssr",
    },

    "metric_definitions": {
        "active_hop_rows":
            "number_of_nonempty_plan_hop_states",

        "active_prefixes":
            "sum_of_active_path_prefix_counts_across_active_hop_rows",

        "primary_search_cost":
            "edges_examined",

        "ssr":
            "1_minus_method_edges_over_rog_edges",

        "answer_retention":
            "method_reachable_questions_over_rog_reachable_questions",
    },

    "afp": {
        "T_grid":
            T_GRID,

        "gamma_min_grid":
            GAMMA_MIN_GRID,

        "tie_atol":
            TIE_ATOL,

        "fully_tied_behavior":
            "retain_all",

        "cutoff_tie_behavior":
            "expand_cutoff_ties",
    },

    "fixed_top_b": {
        "B_grid":
            TOP_B_GRID,

        "cutoff_tie_behavior":
            "expand_cutoff_ties",
    },

    "fixed_threshold": {
        "tau_grid":
            THRESHOLD_GRID,

        "forced_top1":
            False,
    },

    "random_b": {
        "tuned":
            False,

        "budget_source":
            "selected_fixed_top_b",
    },

    "adaptive_budget_random": {
        "tuned":
            False,

        "budget_source":
            "selected_afp_dynamic_budget",
    },

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "test_examples_accessed":
        False,
}


TUNING_SPEC_JSON = json.dumps(
    TUNING_SPEC,
    sort_keys=True,
    separators=(
        ",",
        ":"
    )
)


TUNING_SPEC_SHA = hashlib.sha256(
    TUNING_SPEC_JSON.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Tuning specification SHA256:",
    TUNING_SPEC_SHA
)


# ======================================================================
# 6. BASIC FIELD NORMALIZATION
# ======================================================================

def as_entity_list(
    value
):

    if value is None:

        return []


    if isinstance(
        value,
        str
    ):

        return [
            value
        ]


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        )
    ):

        return [
            str(
                x
            )
            for x in value
        ]


    return [
        str(
            value
        )
    ]


def normalize_relation_plans(
    value
):

    if value is None:

        return []


    assert isinstance(
        value,
        (
            list,
            tuple,
        )
    )


    plans = []


    for plan in value:

        if plan is None:

            plans.append(
                []
            )

            continue


        assert isinstance(
            plan,
            (
                list,
                tuple,
            )
        )


        plans.append(
            [
                str(
                    relation
                )
                for relation in plan
            ]
        )


    return plans


# ======================================================================
# 7. EXACT RoG GRAPH CONSTRUCTION
# ======================================================================
#
# Official RoG semantics already verified in Cell 12B-R:
#
#   undirected simple graph
#   one edge per unordered entity pair
#   later duplicate pair overwrites relation
#   original neighbor insertion order preserved
# ======================================================================

def build_exact_rog_adjacency(
    triples
):

    adjacency = {}


    for triple in triples:

        assert len(
            triple
        ) == 3


        h, r, t = triple


        h = str(
            h
        )

        t = str(
            t
        )

        r = str(
            r
        ).strip()


        if h not in adjacency:

            adjacency[
                h
            ] = {}


        if t not in adjacency:

            adjacency[
                t
            ] = {}


        # Assignment to an existing dictionary key overwrites the
        # relation but preserves the original insertion position.
        adjacency[
            h
        ][
            t
        ] = r


        adjacency[
            t
        ][
            h
        ] = r


    return adjacency


# ======================================================================
# 8. EXACT UNPRUNED RoG TRAVERSAL
# ======================================================================

def traverse_rog_unpruned(
    adjacency,
    topic_entities,
    plan,
    gold_answers=None
):

    if len(
        plan
    ) == 0:

        return {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable":
                False,

            "final_prefixes":
                [],
        }


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    active_hop_rows = 0
    active_prefixes_total = 0

    edges_examined = 0
    candidate_branches = 0


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        # Number of non-empty plan-hop states.
        active_hop_rows += 1


        # Sum of actual path prefixes present at each active hop.
        active_prefixes_total += len(
            active
        )


        candidates = []


        for prefix in active:

            endpoint = prefix[
                -1
            ]


            neighbors = adjacency.get(
                endpoint,
                {}
            )


            # Every unique neighbor is examined.
            edges_examined += len(
                neighbors
            )


            for neighbor, relation in (
                neighbors.items()
            ):

                if relation == target_relation:

                    candidates.append(
                        prefix
                        + (
                            neighbor,
                        )
                    )


        candidate_branches += len(
            candidates
        )


        active = candidates


    reachable = False


    if (
        gold_answers is not None
        and
        active
    ):

        answer_set = {
            str(
                x
            )
            for x in gold_answers
        }


        reachable = any(
            prefix[
                -1
            ]
            in answer_set

            for prefix in active
        )


    return {
        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "reachable":
            bool(
                reachable
            ),

        "final_prefixes":
            active,
    }


# ======================================================================
# 9. CORRECTED RQ1 RoG FIDELITY REFERENCES
# ======================================================================

EXPECTED_ROG = {
    "webqsp": {
        "questions":
            246,

        "total_predicted_plans":
            721,

        "nonempty_plans":
            721,

        "empty_plans":
            0,

        "active_hop_rows":
            971,

        "active_prefixes":
            2440,

        "edges_examined":
            341526,

        "candidate_branches":
            7983,

        "reachable_plans":
            345,

        "reachable_questions":
            205,
    },

    "cwq": {
        "questions":
            3519,

        "total_predicted_plans":
            10536,

        "nonempty_plans":
            10529,

        "empty_plans":
            7,

        "active_hop_rows":
            16564,

        "active_prefixes":
            57841,

        "edges_examined":
            5257272,

        "candidate_branches":
            247161,

        "reachable_plans":
            3971,

        "reachable_questions":
            2425,
    },
}


# ======================================================================
# 10. CORRECTED RoG FIDELITY GATE
# ======================================================================

def rog_fidelity_gate(
    dataset_name,
    planning_rows
):

    expected = EXPECTED_ROG[
        dataset_name
    ]


    assert len(
        planning_rows
    ) == expected[
        "questions"
    ]


    totals = {
        "questions":
            len(
                planning_rows
            ),

        "total_predicted_plans":
            0,

        "nonempty_plans":
            0,

        "empty_plans":
            0,

        "active_hop_rows":
            0,

        "active_prefixes":
            0,

        "edges_examined":
            0,

        "candidate_branches":
            0,

        "reachable_plans":
            0,

        "reachable_questions":
            0,
    }


    for rec in tqdm(
        planning_rows,
        desc=(
            f"{dataset_name} RoG fidelity"
        )
    ):

        adjacency = (
            build_exact_rog_adjacency(
                rec[
                    "graph"
                ]
            )
        )


        topic_entities = (
            as_entity_list(
                rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                rec[
                    "predicted_paths"
                ]
            )
        )


        totals[
            "total_predicted_plans"
        ] += len(
            plans
        )


        question_reachable = False


        for plan in plans:

            if len(
                plan
            ) == 0:

                totals[
                    "empty_plans"
                ] += 1

                continue


            totals[
                "nonempty_plans"
            ] += 1


            result = (
                traverse_rog_unpruned(
                    adjacency=
                        adjacency,

                    topic_entities=
                        topic_entities,

                    plan=
                        plan,

                    gold_answers=
                        gold_answers
                )
            )


            totals[
                "active_hop_rows"
            ] += result[
                "active_hop_rows"
            ]


            totals[
                "active_prefixes"
            ] += result[
                "active_prefixes"
            ]


            totals[
                "edges_examined"
            ] += result[
                "edges_examined"
            ]


            totals[
                "candidate_branches"
            ] += result[
                "candidate_branches"
            ]


            if result[
                "reachable"
            ]:

                totals[
                    "reachable_plans"
                ] += 1

                question_reachable = True


        if question_reachable:

            totals[
                "reachable_questions"
            ] += 1


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} RoG FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in (
        totals.items()
    ):

        print(
            f"{key:<25}",
            value
        )


    fidelity_keys = [
        "questions",
        "total_predicted_plans",
        "nonempty_plans",
        "empty_plans",
        "active_hop_rows",
        "active_prefixes",
        "edges_examined",
        "candidate_branches",
        "reachable_plans",
        "reachable_questions",
    ]


    for key in fidelity_keys:

        assert (
            totals[
                key
            ]
            ==
            expected[
                key
            ]
        ), (
            f"{dataset_name}: RoG fidelity mismatch "
            f"for {key}: "
            f"{totals[key]} != {expected[key]}"
        )


    print(
        f"{dataset_name.upper()} RoG fidelity: PASSED"
    )


    return totals


# ======================================================================
# 11. RUN RoG FIDELITY GATES
# ======================================================================

webqsp_rog_reference = (
    rog_fidelity_gate(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows
    )
)


cwq_rog_reference = (
    rog_fidelity_gate(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows
    )
)


print(
    "\nExact RoG validation fidelity gate: PASSED"
)


# ======================================================================
# 12. BATCHED ONLINE VALIDATION SEMANTIC PREFILL
# ======================================================================
#
# All dynamic pruned candidate groups are subsets of the unpruned RoG
# candidate universe.
#
# We therefore collect validation semantic texts from the UNPRUNED
# traversal and batch-prefill MiniLM before tuning.
#
# This is only a speed optimization.
# ======================================================================

def collect_online_semantic_inventory(
    dataset_name,
    planning_rows,
    question_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans_mapping = {}
    suffixes = {}


    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} semantic inventory"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )

        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        questions[
            question_id
        ] = str(
            question_rec[
                "question"
            ]
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        for entity in topic_entities:

            if (
                has_readable_entity_surface_runtime(
                    entity
                )
            ):

                entities[
                    str(
                        entity
                    )
                ] = (
                    entity_surface_text_runtime(
                        entity
                    )
                )


        relation_plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        for plan in relation_plans:

            if len(
                plan
            ) == 0:

                continue


            # ----------------------------------------------------------
            # Relations
            # ----------------------------------------------------------

            for relation in plan:

                relations[
                    str(
                        relation
                    )
                ] = (
                    relation_surface_text_runtime(
                        relation
                    )
                )


            # ----------------------------------------------------------
            # Full plan
            # ----------------------------------------------------------

            plan_key = "||".join(
                str(
                    x
                )
                for x in plan
            )


            plans_mapping[
                plan_key
            ] = (
                relation_sequence_text_runtime(
                    plan
                )
            )


            # ----------------------------------------------------------
            # Every possible intermediate-hop remaining suffix
            # ----------------------------------------------------------

            for hop in range(
                max(
                    0,
                    len(
                        plan
                    )
                    - 1
                )
            ):

                suffix = plan[
                    hop + 1:
                ]


                suffix_key = "||".join(
                    str(
                        x
                    )
                    for x in suffix
                )


                suffixes[
                    suffix_key
                ] = (
                    relation_sequence_text_runtime(
                        suffix
                    )
                )


            # ----------------------------------------------------------
            # Unpruned relation-constrained entity universe
            # ----------------------------------------------------------

            active = [
                (
                    str(
                        entity
                    ),
                )
                for entity in topic_entities
            ]


            for target_relation in plan:

                if not active:

                    break


                candidates = []


                for prefix in active:

                    endpoint = prefix[
                        -1
                    ]


                    neighbors = adjacency.get(
                        endpoint,
                        {}
                    )


                    for neighbor, relation in (
                        neighbors.items()
                    ):

                        if relation != target_relation:

                            continue


                        candidates.append(
                            prefix
                            + (
                                neighbor,
                            )
                        )


                        if (
                            has_readable_entity_surface_runtime(
                                neighbor
                            )
                        ):

                            entities[
                                str(
                                    neighbor
                                )
                            ] = (
                                entity_surface_text_runtime(
                                    neighbor
                                )
                            )


                active = candidates


    return {
        "questions":
            questions,

        "entities":
            entities,

        "relations":
            relations,

        "plans":
            plans_mapping,

        "suffixes":
            suffixes,
    }


print(
    "\nCollecting complete validation runtime semantic universe..."
)


webqsp_online_inventory = (
    collect_online_semantic_inventory(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime
    )
)


cwq_online_inventory = (
    collect_online_semantic_inventory(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime
    )
)


def merge_dicts_in_order(
    *dicts
):

    result = {}


    for mapping in dicts:

        result.update(
            mapping
        )


    return result


ALL_VALIDATION_QUESTIONS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "questions"
        ],

        cwq_online_inventory[
            "questions"
        ]
    )
)


ALL_VALIDATION_ENTITIES = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "entities"
        ],

        cwq_online_inventory[
            "entities"
        ]
    )
)


ALL_VALIDATION_RELATIONS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "relations"
        ],

        cwq_online_inventory[
            "relations"
        ]
    )
)


ALL_VALIDATION_PLANS = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "plans"
        ],

        cwq_online_inventory[
            "plans"
        ]
    )
)


ALL_VALIDATION_SUFFIXES = (
    merge_dicts_in_order(
        webqsp_online_inventory[
            "suffixes"
        ],

        cwq_online_inventory[
            "suffixes"
        ]
    )
)


print(
    "\nValidation online semantic inventory"
)

print(
    "  Questions:         ",
    len(
        ALL_VALIDATION_QUESTIONS
    )
)

print(
    "  Readable entities: ",
    len(
        ALL_VALIDATION_ENTITIES
    )
)

print(
    "  Relations:         ",
    len(
        ALL_VALIDATION_RELATIONS
    )
)

print(
    "  Plans:             ",
    len(
        ALL_VALIDATION_PLANS
    )
)

print(
    "  Suffixes:          ",
    len(
        ALL_VALIDATION_SUFFIXES
    )
)


print(
    "\nBatch-prefilling semantic runtime..."
)


for namespace, mapping in [
    (
        "question",
        ALL_VALIDATION_QUESTIONS
    ),

    (
        "entity",
        ALL_VALIDATION_ENTITIES
    ),

    (
        "relation",
        ALL_VALIDATION_RELATIONS
    ),

    (
        "plan",
        ALL_VALIDATION_PLANS
    ),

    (
        "suffix",
        ALL_VALIDATION_SUFFIXES
    ),
]:

    print(
        f"  {namespace:<10} "
        f"{len(mapping):>7}"
    )


    AFP_RUNTIME_SEMANTIC_ENCODER.prefill(
        namespace,
        mapping
    )


print(
    "Validation semantic prefill: READY"
)


# ======================================================================
# 13. NUMERICAL HELPERS
# ======================================================================

def stable_sigmoid(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    output = np.empty_like(
        logits
    )


    positive = (
        logits >= 0
    )


    output[
        positive
    ] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -logits[
                    positive
                ]
            )
        )
    )


    exp_values = np.exp(
        logits[
            ~positive
        ]
    )


    output[
        ~positive
    ] = (
        exp_values
        /
        (
            1.0
            +
            exp_values
        )
    )


    return output


def temperature_softmax(
    logits,
    temperature
):

    temperature = float(
        temperature
    )


    assert temperature > 0


    values = (
        np.asarray(
            logits,
            dtype=np.float64
        )
        /
        temperature
    )


    values = (
        values
        -
        np.max(
            values
        )
    )


    exp_values = np.exp(
        values
    )


    denominator = float(
        np.sum(
            exp_values
        )
    )


    assert denominator > 0
    assert np.isfinite(
        denominator
    )


    return (
        exp_values
        /
        denominator
    )


def normalized_entropy(
    probabilities
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )


    n = len(
        probabilities
    )


    if n <= 1:

        return 0.0


    safe_probabilities = np.clip(
        probabilities,
        1e-12,
        1.0
    )


    entropy = -float(
        np.sum(
            safe_probabilities
            * np.log(
                safe_probabilities
            )
        )
    )


    denominator = math.log(
        n
    )


    if denominator <= 0:

        return 0.0


    return float(
        np.clip(
            entropy
            /
            denominator,
            0.0,
            1.0
        )
    )


# ======================================================================
# 14. TIE-AWARE HELPERS
# ======================================================================

def all_logits_tied(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    if len(
        logits
    ) <= 1:

        return True


    return bool(
        (
            np.max(
                logits
            )
            -
            np.min(
                logits
            )
        )
        <= TIE_ATOL
    )


def stable_descending_order(
    logits
):

    return np.argsort(
        -np.asarray(
            logits,
            dtype=np.float64
        ),
        kind="stable"
    )


def tie_expanded_top_k_indices(
    logits,
    requested_k
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    requested_k = int(
        requested_k
    )


    if requested_k >= n:

        return list(
            range(
                n
            )
        )


    assert requested_k >= 1


    order = (
        stable_descending_order(
            logits
        )
    )


    cutoff_index = order[
        requested_k
        - 1
    ]


    cutoff_score = logits[
        cutoff_index
    ]


    selected = []


    for index, score in enumerate(
        logits
    ):

        if (
            score
            >
            cutoff_score
            or
            np.isclose(
                score,
                cutoff_score,
                atol=TIE_ATOL,
                rtol=0.0
            )
        ):

            selected.append(
                index
            )


    return selected


# ======================================================================
# 15. POLICY SELECTION
# ======================================================================

def select_policy_indices(
    config,
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    assert n > 1


    family = config[
        "family"
    ]


    # ------------------------------------------------------------------
    # Fixed Top-B
    # ------------------------------------------------------------------

    if family == "fixed_top_b":

        B = int(
            config[
                "B"
            ]
        )


        requested_budget = min(
            B,
            n
        )


        selected = (
            tie_expanded_top_k_indices(
                logits=
                    logits,

                requested_k=
                    requested_budget
            )
        )


        return {
            "selected_indices":
                selected,

            "requested_budget":
                requested_budget,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                all_logits_tied(
                    logits
                ),
        }


    # ------------------------------------------------------------------
    # Fixed Threshold
    # ------------------------------------------------------------------

    if family == "fixed_threshold":

        tau = float(
            config[
                "tau"
            ]
        )


        probabilities = (
            stable_sigmoid(
                logits
            )
        )


        selected = [
            index
            for index, probability
            in enumerate(
                probabilities
            )
            if probability
            >= tau
        ]


        # No forced top-1.
        return {
            "selected_indices":
                selected,

            "requested_budget":
                None,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                all_logits_tied(
                    logits
                ),
        }


    # ------------------------------------------------------------------
    # AFP
    # ------------------------------------------------------------------

    assert family == "afp"


    T = float(
        config[
            "T"
        ]
    )


    gamma_min = float(
        config[
            "gamma_min"
        ]
    )


    assert 0.0 <= gamma_min <= 1.0


    # Explicit abstention under complete score tie.
    if all_logits_tied(
        logits
    ):

        return {
            "selected_indices":
                list(
                    range(
                        n
                    )
                ),

            "requested_budget":
                n,

            "retained_count":
                n,

            "uncertainty":
                1.0,

            "gamma":
                1.0,

            "fully_tied":
                True,
        }


    probabilities = (
        temperature_softmax(
            logits=
                logits,

            temperature=
                T
        )
    )


    uncertainty = (
        normalized_entropy(
            probabilities
        )
    )


    gamma = (
        gamma_min
        +
        uncertainty
        * (
            1.0
            -
            gamma_min
        )
    )


    order = (
        stable_descending_order(
            logits
        )
    )


    sorted_probabilities = (
        probabilities[
            order
        ]
    )


    cumulative = np.cumsum(
        sorted_probabilities
    )


    requested_budget = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        )
        + 1
    )


    requested_budget = min(
        max(
            requested_budget,
            1
        ),
        n
    )


    selected = (
        tie_expanded_top_k_indices(
            logits=
                logits,

            requested_k=
                requested_budget
        )
    )


    return {
        "selected_indices":
            selected,

        "requested_budget":
            requested_budget,

        "retained_count":
            len(
                selected
            ),

        "uncertainty":
            float(
                uncertainty
            ),

        "gamma":
            float(
                gamma
            ),

        "fully_tied":
            False,
    }


# ======================================================================
# 16. RELATION-EXPANSION CACHE
# ======================================================================
#
# The same dynamic frontier may be encountered by multiple tuning
# configurations.
#
# We cache the actual candidate construction once.
#
# IMPORTANT:
# edges_cost is still added independently to EVERY method, so measured
# graph-search cost remains method-correct.
# ======================================================================

def frontier_cache_key(
    plan_index,
    hop,
    active_prefixes
):

    return (
        int(
            plan_index
        ),

        int(
            hop
        ),

        tuple(
            tuple(
                prefix
            )
            for prefix in active_prefixes
        ),
    )


def get_cached_relation_expansion(
    adjacency,
    plan_index,
    hop,
    target_relation,
    active_prefixes,
    expansion_cache
):

    key = (
        frontier_cache_key(
            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    if key in expansion_cache:

        return expansion_cache[
            key
        ]


    candidates = []
    candidate_rows = []

    edges_cost = 0


    for parent_index, prefix in enumerate(
        active_prefixes
    ):

        endpoint = prefix[
            -1
        ]


        neighbors = adjacency.get(
            endpoint,
            {}
        )


        edges_cost += len(
            neighbors
        )


        for neighbor, relation in (
            neighbors.items()
        ):

            if relation != target_relation:

                continue


            candidate_prefix = (
                prefix
                + (
                    neighbor,
                )
            )


            candidates.append(
                candidate_prefix
            )


            candidate_rows.append(
                {
                    "prefix_entities":
                        list(
                            prefix
                        ),

                    "candidate_entity":
                        neighbor,

                    "parent_prefix_index":
                        int(
                            parent_index
                        ),
                }
            )


    result = {
        "candidates":
            candidates,

        "candidate_rows":
            candidate_rows,

        "edges_cost":
            int(
                edges_cost
            ),
    }


    expansion_cache[
        key
    ] = result


    return result


# ======================================================================
# 17. ONLINE SCORER CACHE
# ======================================================================

def get_group_logits(
    dataset_name,
    question_id,
    question,
    plan_index,
    plan,
    hop,
    active_prefixes,
    candidate_rows,
    score_cache
):

    key = (
        frontier_cache_key(
            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    if key in score_cache:

        cached = score_cache[
            key
        ]


        assert (
            cached[
                "candidate_count"
            ]
            ==
            len(
                candidate_rows
            )
        )


        return cached[
            "logits"
        ]


    output = (
        AFP_RUNTIME_SCORE_GROUP(
            dataset_name=
                dataset_name,

            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )


    logits = np.asarray(
        output[
            "logits"
        ],
        dtype=np.float32
    )


    assert logits.shape == (
        len(
            candidate_rows
        ),
    )


    score_cache[
        key
    ] = {
        "candidate_count":
            len(
                candidate_rows
            ),

        "logits":
            logits,
    }


    return logits


# ======================================================================
# 18. POLICY-AWARE ONLINE TRAVERSAL
# ======================================================================

def traverse_with_policy(
    dataset_name,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    config,
    expansion_cache,
    score_cache
):

    assert len(
        plan
    ) > 0


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    edges_examined = 0
    candidate_branches = 0

    active_hop_rows = 0
    active_prefixes_total = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    L = len(
        plan
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes_total += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        # Even though expansion computation is cached, this is the
        # graph work THIS method would perform.
        edges_examined += expansion[
            "edges_cost"
        ]


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        is_final_hop = (
            hop
            ==
            L - 1
        )


        if is_final_hop:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # INTERMEDIATE DECISION
        # --------------------------------------------------------------

        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            select_policy_indices(
                config=
                    config,

                logits=
                    logits
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )


            requested_budget_observations += 1


        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )


            uncertainty_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 19. GOLD-ONLY POST-TRAVERSAL REACHABILITY
# ======================================================================
#
# Gold is deliberately kept OUT of traverse_with_policy().
# ======================================================================

def final_prefixes_reach_answer(
    final_prefixes,
    gold_answers
):

    if not final_prefixes:

        return False


    answer_set = {
        str(
            answer
        )
        for answer in gold_answers
    }


    return any(
        prefix[
            -1
        ]
        in answer_set

        for prefix in final_prefixes
    )


# ======================================================================
# 20. INITIAL CONFIG STATS
# ======================================================================

def fresh_config_stats():

    return {
        config[
            "config_id"
        ]: {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }

        for config in CONFIGS
    }


# ======================================================================
# 21. RESUMABLE VALIDATION SWEEP
# ======================================================================

def tune_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    # Version-specific resume file prevents accidental reuse of any
    # older Cell-13 state.
    resume_path = (
        TUNING_DIR
        / (
            f"{dataset_name}_"
            "cell13_v2_metric_corrected_resume.pkl"
        )
    )


    if resume_path.exists():

        with open(
            resume_path,
            "rb"
        ) as f:

            state = pickle.load(
                f
            )


        assert (
            state[
                "tuning_spec_sha256"
            ]
            ==
            TUNING_SPEC_SHA
        ), (
            "Resume file belongs to a different tuning specification."
        )


        assert (
            state[
                "dataset"
            ]
            ==
            dataset_name
        )


        start_index = int(
            state[
                "next_index"
            ]
        )


        stats = state[
            "stats"
        ]


        print(
            f"\n{dataset_name.upper()} resume state detected."
        )


        print(
            "Resuming from question:",
            start_index,
            "/",
            len(
                planning_rows
            )
        )


    else:

        start_index = 0

        stats = (
            fresh_config_stats()
        )


    start_time = time.time()


    for source_index in tqdm(
        range(
            start_index,
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} tuning"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # --------------------------------------------------------------
        # Question-local computational caches.
        # --------------------------------------------------------------

        expansion_cache = {}

        score_cache = {}


        question_reachability = {
            config[
                "config_id"
            ]:
                False

            for config in CONFIGS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for config in CONFIGS:

                config_id = (
                    config[
                        "config_id"
                    ]
                )


                result = (
                    traverse_with_policy(
                        dataset_name=
                            dataset_name,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        config=
                            config,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    config_id
                ]


                s[
                    "active_hop_rows"
                ] += result[
                    "active_hop_rows"
                ]


                s[
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                s[
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                s[
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


                s[
                    "decision_hops"
                ] += result[
                    "decision_hops"
                ]


                s[
                    "retained_after_decision"
                ] += result[
                    "retained_after_decision"
                ]


                s[
                    "requested_budget_total"
                ] += result[
                    "requested_budget_total"
                ]


                s[
                    "requested_budget_observations"
                ] += result[
                    "requested_budget_observations"
                ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "fully_tied_decisions"
                ] += result[
                    "fully_tied_decisions"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # GOLD USED ONLY HERE, AFTER TRAVERSAL.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachability[
                        config_id
                    ] = True


        # --------------------------------------------------------------
        # Question-level Answer Retention indicator.
        # --------------------------------------------------------------

        for config_id, reachable in (
            question_reachability.items()
        ):

            if reachable:

                stats[
                    config_id
                ][
                    "reachable_questions"
                ] += 1


        # --------------------------------------------------------------
        # RESUME CHECKPOINT EVERY 100 QUESTIONS
        # --------------------------------------------------------------

        if (
            (
                source_index
                + 1
            )
            % 100
            == 0
            or
            source_index
            ==
            (
                len(
                    planning_rows
                )
                - 1
            )
        ):

            resume_state = {
                "dataset":
                    dataset_name,

                "tuning_spec_sha256":
                    TUNING_SPEC_SHA,

                "next_index":
                    int(
                        source_index
                        + 1
                    ),

                "stats":
                    stats,
            }


            with open(
                resume_path,
                "wb"
            ) as f:

                pickle.dump(
                    resume_state,
                    f
                )


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} tuning sweep completed."
    )


    print(
        f"Elapsed this run: {elapsed/60:.2f} min"
    )


    # ==================================================================
    # AGGREGATED RESULT TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable_questions = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    assert rog_edges > 0
    assert rog_reachable_questions > 0


    for config in CONFIGS:

        config_id = (
            config[
                "config_id"
            ]
        )


        s = stats[
            config_id
        ]


        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_questions = int(
            s[
                "reachable_questions"
            ]
        )


        ssr = (
            1.0
            -
            edges
            /
            rog_edges
        )


        answer_retention = (
            reachable_questions
            /
            rog_reachable_questions
        )


        avg_requested_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "config_id":
                    config_id,

                "family":
                    config[
                        "family"
                    ],

                "T":
                    config.get(
                        "T",
                        np.nan
                    ),

                "gamma_min":
                    config.get(
                        "gamma_min",
                        np.nan
                    ),

                "B":
                    config.get(
                        "B",
                        np.nan
                    ),

                "tau":
                    config.get(
                        "tau",
                        np.nan
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "edges_examined":
                    edges,

                "rog_edges_examined":
                    int(
                        rog_reference[
                            "edges_examined"
                        ]
                    ),

                "ssr":
                    float(
                        ssr
                    ),

                "reachable_questions":
                    reachable_questions,

                "rog_reachable_questions":
                    rog_reachable_questions,

                "answer_retention":
                    float(
                        answer_retention
                    ),

                "coverage_all_questions":
                    float(
                        reachable_questions
                        /
                        len(
                            planning_rows
                        )
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "retained_after_decision":
                    int(
                        s[
                            "retained_after_decision"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_requested_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),

                "ar_floor_feasible":
                    bool(
                        answer_retention
                        >= AR_FLOOR
                    ),
            }
        )


    return (
        pd.DataFrame(
            rows
        ),
        stats
    )


# ======================================================================
# 22. START VALIDATION TUNING
# ======================================================================

print(
    "\n"
    + "=" * 112
)

print(
    "STARTING VALIDATION-ONLY CONFIGURATION SWEEP"
)

print(
    "=" * 112
)


webqsp_tuning_df, webqsp_tuning_stats = (
    tune_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference
    )
)


cwq_tuning_df, cwq_tuning_stats = (
    tune_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference
    )
)


# ======================================================================
# 23. PREDECLARED CONFIGURATION SELECTION
# ======================================================================

def select_family_config(
    result_df,
    family
):

    subset = (
        result_df[
            result_df[
                "family"
            ]
            ==
            family
        ]
        .copy()
    )


    assert len(
        subset
    ) > 0


    feasible = (
        subset[
            subset[
                "answer_retention"
            ]
            >= AR_FLOOR
        ]
        .copy()
    )


    if len(
        feasible
    ) > 0:

        selection_mode = (
            "maximize_ssr_subject_to_ar_floor"
        )


        ranked = feasible.sort_values(
            by=[
                "ssr",
                "answer_retention",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    else:

        selection_mode = (
            "fallback_maximize_ar_then_ssr"
        )


        ranked = subset.sort_values(
            by=[
                "answer_retention",
                "ssr",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    selected = (
        ranked.iloc[
            0
        ].to_dict()
    )


    selected[
        "selection_mode"
    ] = selection_mode


    return (
        selected,
        ranked
    )


def select_dataset_configs(
    result_df
):

    selected = {}
    ranked = {}


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        (
            selected_family,
            ranked_family
        ) = (
            select_family_config(
                result_df=
                    result_df,

                family=
                    family
            )
        )


        selected[
            family
        ] = selected_family


        ranked[
            family
        ] = ranked_family


    return (
        selected,
        ranked
    )


(
    webqsp_selected,
    webqsp_ranked
) = (
    select_dataset_configs(
        webqsp_tuning_df
    )
)


(
    cwq_selected,
    cwq_ranked
) = (
    select_dataset_configs(
        cwq_tuning_df
    )
)


# ======================================================================
# 24. DISPLAY RESULTS
# ======================================================================

DISPLAY_COLUMNS = [
    "config_id",
    "family",
    "T",
    "gamma_min",
    "B",
    "tau",
    "ssr",
    "answer_retention",
    "reachable_questions",
    "edges_examined",
    "active_hop_rows",
    "active_prefixes",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "ar_floor_feasible",
]


def show_family_results(
    dataset_name,
    result_df
):

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} VALIDATION TUNING RESULTS"
    )

    print(
        "=" * 112
    )


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        subset = (
            result_df[
                result_df[
                    "family"
                ]
                ==
                family
            ]
            .sort_values(
                [
                    "answer_retention",
                    "ssr",
                ],

                ascending=[
                    False,
                    False,
                ]
            )
        )


        print(
            f"\n--- {family} ---"
        )


        print(
            subset[
                DISPLAY_COLUMNS
            ].to_string(
                index=False,

                float_format=lambda x:
                    f"{x:.6f}"
            )
        )


show_family_results(
    "webqsp",
    webqsp_tuning_df
)


show_family_results(
    "cwq",
    cwq_tuning_df
)


# ======================================================================
# 25. DISPLAY SELECTED CONFIGURATIONS
# ======================================================================

def print_selected(
    dataset_name,
    selected
):

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} "
        "SELECTED DEVELOPMENT CONFIGURATIONS"
    )

    print(
        "=" * 112
    )


    for family in [
        "afp",
        "fixed_top_b",
        "fixed_threshold",
    ]:

        row = selected[
            family
        ]


        print(
            f"\n{family}"
        )


        print(
            "  config:",
            row[
                "config_id"
            ]
        )


        print(
            "  selection mode:",
            row[
                "selection_mode"
            ]
        )


        print(
            "  AR:",
            f"{row['answer_retention']:.6f}"
        )


        print(
            "  SSR:",
            f"{row['ssr']:.6f}"
        )


        print(
            "  edges:",
            int(
                row[
                    "edges_examined"
                ]
            )
        )


        if family == "afp":

            print(
                "  T:",
                row[
                    "T"
                ]
            )


            print(
                "  gamma_min:",
                row[
                    "gamma_min"
                ]
            )


        elif family == "fixed_top_b":

            print(
                "  B:",
                int(
                    row[
                        "B"
                    ]
                )
            )


        elif family == "fixed_threshold":

            print(
                "  tau:",
                row[
                    "tau"
                ]
            )


print_selected(
    "webqsp",
    webqsp_selected
)


print_selected(
    "cwq",
    cwq_selected
)


# ======================================================================
# 26. SELECTED-CONFIG GRID SANITY
# ======================================================================

for dataset_name, selected in [
    (
        "WebQSP",
        webqsp_selected
    ),

    (
        "CWQ",
        cwq_selected
    ),
]:

    selected_afp = selected[
        "afp"
    ]


    assert float(
        selected_afp[
            "T"
        ]
    ) in T_GRID


    assert float(
        selected_afp[
            "gamma_min"
        ]
    ) in GAMMA_MIN_GRID


    assert int(
        selected[
            "fixed_top_b"
        ][
            "B"
        ]
    ) in TOP_B_GRID


    assert float(
        selected[
            "fixed_threshold"
        ][
            "tau"
        ]
    ) in THRESHOLD_GRID


print(
    "\nSelected configurations belong to "
    "predeclared grids: PASSED"
)


# ======================================================================
# 27. EXPORT FULL TUNING CSV
# ======================================================================

WEBQSP_TUNING_CSV = (
    TUNING_DIR
    / "webqsp_validation_tuning_v2.csv"
)


CWQ_TUNING_CSV = (
    TUNING_DIR
    / "cwq_validation_tuning_v2.csv"
)


webqsp_tuning_df.to_csv(
    WEBQSP_TUNING_CSV,
    index=False
)


cwq_tuning_df.to_csv(
    CWQ_TUNING_CSV,
    index=False
)


# ======================================================================
# 28. JSON-SAFE SELECTED RESULT
# ======================================================================

def json_safe_selected(
    selected
):

    output = {}


    for family, row in (
        selected.items()
    ):

        clean = {}


        for key, value in (
            row.items()
        ):

            if isinstance(
                value,
                np.integer
            ):

                clean[
                    key
                ] = int(
                    value
                )


            elif isinstance(
                value,
                np.floating
            ):

                if np.isnan(
                    value
                ):

                    clean[
                        key
                    ] = None

                else:

                    clean[
                        key
                    ] = float(
                        value
                    )


            elif isinstance(
                value,
                np.bool_
            ):

                clean[
                    key
                ] = bool(
                    value
                )


            elif (
                isinstance(
                    value,
                    float
                )
                and
                math.isnan(
                    value
                )
            ):

                clean[
                    key
                ] = None


            else:

                clean[
                    key
                ] = value


        output[
            family
        ] = clean


    return output


# ======================================================================
# 29. DEVELOPMENT-SELECTION MANIFEST
# ======================================================================

CELL13_MANIFEST = {
    "cell":
        "RQ2_CELL13_VALIDATION_TUNING_REVISED",

    "version":
        "rq2_cell13_validation_tuning_v2_metric_corrected",

    "tuning_spec_sha256":
        TUNING_SPEC_SHA,

    "selection_rule":
        TUNING_SPEC[
            "selection_rule"
        ],

    "metric_definitions":
        TUNING_SPEC[
            "metric_definitions"
        ],

    "grids": {
        "T":
            T_GRID,

        "gamma_min":
            GAMMA_MIN_GRID,

        "top_B":
            TOP_B_GRID,

        "threshold":
            THRESHOLD_GRID,
    },

    "rog_validation_fidelity": {
        "webqsp":
            webqsp_rog_reference,

        "cwq":
            cwq_rog_reference,
    },

    "selected": {
        "webqsp":
            json_safe_selected(
                webqsp_selected
            ),

        "cwq":
            json_safe_selected(
                cwq_selected
            ),
    },

    "random_b_policy": {
        "independently_tuned":
            False,

        "budget_source":
            "selected_fixed_top_b",
    },

    "adaptive_budget_random_policy": {
        "independently_tuned":
            False,

        "budget_source":
            "selected_afp_dynamic_budget",
    },

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "gold_used_for": [
        "validation_answer_retention_after_traversal_only"
    ],

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,

    "next_step":
        "RQ2_Cell14_validation_controlled_comparison",
}


CELL13_MANIFEST_PATH = (
    TUNING_DIR
    / "cell13_validation_tuning_v2_manifest.json"
)


with open(
    CELL13_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL13_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 30. EXPOSE SELECTED PARAMETERS FOR CELL 14
# ======================================================================

AFP_VALIDATION_SELECTED = {
    "webqsp": {
        "T":
            float(
                webqsp_selected[
                    "afp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                webqsp_selected[
                    "afp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                cwq_selected[
                    "afp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                cwq_selected[
                    "afp"
                ][
                    "gamma_min"
                ]
            ),
    },
}


FIXED_TOP_B_VALIDATION_SELECTED = {
    "webqsp":
        int(
            webqsp_selected[
                "fixed_top_b"
            ][
                "B"
            ]
        ),

    "cwq":
        int(
            cwq_selected[
                "fixed_top_b"
            ][
                "B"
            ]
        ),
}


FIXED_THRESHOLD_VALIDATION_SELECTED = {
    "webqsp":
        float(
            webqsp_selected[
                "fixed_threshold"
            ][
                "tau"
            ]
        ),

    "cwq":
        float(
            cwq_selected[
                "fixed_threshold"
            ][
                "tau"
            ]
        ),
}


CELL13_TUNING_COMPLETE = True


# ======================================================================
# 31. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "=== RQ2 CELL 13: "
    "VALIDATION-ONLY HYPERPARAMETER TUNING COMPLETE ==="
)

print(
    "=" * 118
)


print(
    "\nMetric correction:"
)

print(
    "  active_hop_rows != active_prefixes"
)

print(
    "  WebQSP reference: 971 hop rows / 2440 prefixes"
)

print(
    "  CWQ reference:    16564 hop rows / 57841 prefixes"
)


print(
    "\nSelection criterion:"
)

print(
    f"  maximize SSR subject to AR >= {AR_FLOOR}"
)

print(
    "  fallback: maximize AR, then SSR"
)


print(
    "\nSelected AFP:"
)

print(
    "  WebQSP:",
    AFP_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nSelected Fixed Top-B:"
)

print(
    "  WebQSP:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nSelected Fixed Threshold:"
)

print(
    "  WebQSP:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "cwq"
    ]
)


print(
    "\nRandom baselines:"
)

print(
    "  Random-B inherits selected Fixed Top-B B."
)

print(
    "  Adaptive-Budget Random inherits selected AFP budgets."
)


print(
    "\nDevelopment status:"
)

print(
    "  Feature-v2:                   FROZEN"
)

print(
    "  Scorer architecture/weights: FROZEN"
)

print(
    "  Selector hyperparameters:     VALIDATION-SELECTED"
)

print(
    "  Controlled comparison:        NEXT"
)

print(
    "  Ablations:                    NOT YET"
)

print(
    "  TEST examples accessed:       NO"
)

print(
    "  Complete AFP frozen:          NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_TUNING_CSV
)

print(
    " ",
    CWQ_TUNING_CSV
)

print(
    " ",
    CELL13_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 14 — validation controlled comparison:"
)

print(
    "RoG vs Fixed Top-B vs Fixed Threshold "
    "vs Random-B vs Adaptive-Budget Random vs AFP."
)

Cell 13 prerequisites: PASSED
Runtime scorer: READY
Runtime device: CPU
Feature version: afp_features_v2_masked_entity_semantics

PREDECLARED VALIDATION MODEL-SELECTION RULE
Primary objective: maximize SSR subject to AR >= 0.99
Fallback: maximize AR, then SSR, if no configuration satisfies AR floor.

AFP grid
  T:          [0.5, 1.0, 2.0]
  gamma_min:  [0.5, 0.7, 0.8, 0.9, 0.95]

Fixed Top-B grid:
  [1, 2, 4, 8, 16, 32]

Fixed Threshold grid:
  [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

Random-B: inherits selected Fixed Top-B B; NOT independently tuned.
Adaptive-Budget Random: inherits selected AFP adaptive budgets; NOT independently tuned.

Total validation configurations: 30
Tuning specification SHA256: d2d990a09c9d9494e8941ad3f9fb561768a8a891467d23a2e55ed993e35f4510


webqsp RoG fidelity:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP RoG FIDELITY
questions                 246
total_predicted_plans     721
nonempty_plans            721
empty_plans               0
active_hop_rows           971
active_prefixes           2440
edges_examined            341526
candidate_branches        7983
reachable_plans           345
reachable_questions       205
WEBQSP RoG fidelity: PASSED


cwq RoG fidelity:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ RoG FIDELITY
questions                 3519
total_predicted_plans     10536
nonempty_plans            10529
empty_plans               7
active_hop_rows           16564
active_prefixes           57841
edges_examined            5257272
candidate_branches        247161
reachable_plans           3971
reachable_questions       2425
CWQ RoG fidelity: PASSED

Exact RoG validation fidelity gate: PASSED



...


Validation online semantic inventory
  Questions:          3765
  Readable entities:  25248
  Relations:          762
  Plans:              2720
  Suffixes:           724

Batch-prefilling semantic runtime...
  question      3765
  entity       25248
  relation       762
  plan          2720
  suffix         724
Validation semantic prefill: READY

STARTING VALIDATION-ONLY CONFIGURATION SWEEP


webqsp tuning:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP tuning sweep completed.
Elapsed this run: 0.04 min


cwq tuning:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ tuning sweep completed.
Elapsed this run: 2.23 min

WEBQSP VALIDATION TUNING RESULTS

--- afp ---
     config_id family        T  gamma_min   B  tau      ssr  answer_retention  reachable_questions  edges_examined  active_hop_rows  active_prefixes  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  ar_floor_feasible
 afp_T0.5_g0.5    afp 0.500000   0.500000 NaN  NaN 0.001403          1.000000                  205          341047              971             2425            147             10.789116         0.988268                    97               True
 afp_T0.5_g0.7    afp 0.500000   0.700000 NaN  NaN 0.000820          1.000000                  205          341246              971             2428            147             10.809524         0.988268                    97               True
 afp_T0.5_g0.8    afp 0.500000   0.800000 NaN  NaN 0.000217          1.000000                  205          341452              971             2432            147  

## RQ2 validation comparison, SSR(Search Space Reduction) and Answer Retentation 

In [28]:
# ======================================================================
# RQ2 CELL 14
# VALIDATION CONTROLLED COMPARISON
# ======================================================================
#
# METHODS
# -------
# 1. RoG
# 2. Fixed Top-B
# 3. Fixed Threshold
# 4. Random-B                     seeds 42,43,44
# 5. Adaptive-Budget Random       seeds 42,43,44
# 6. AFP
#
# CONTROL
# -------
# Same:
#   questions
#   topic entities
#   per-question graph
#   relation plans
#   exact RoG graph semantics
#   Feature-v2
#   selected scorer
#   final-hop protection
#
# Random-B:
#   inherits selected Fixed Top-B B.
#
# Adaptive-Budget Random:
#   uses scorer ONLY to determine AFP's adaptive retained COUNT;
#   branch identity is then selected uniformly at random.
#
# This directly tests whether AFP's learned branch ranking adds value
# beyond its adaptive budget.
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL13_TUNING_COMPLETE",

    "AFP_VALIDATION_SELECTED",
    "FIXED_TOP_B_VALIDATION_SELECTED",
    "FIXED_THRESHOLD_VALIDATION_SELECTED",

    "AFP_RUNTIME_SCORE_GROUP",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "select_policy_indices",
    "get_cached_relation_expansion",
    "get_group_logits",
    "final_prefixes_reach_answer",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell-13/B4 objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13_TUNING_COMPLETE is True

print("Cell 14 prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

COMPARE_DIR = (
    ROOT
    / "12_validation_comparison"
)

COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. FIXED RANDOM SEEDS
# ======================================================================

RANDOM_SEEDS = [
    42,
    43,
    44,
]

print(
    "Random baseline seeds:",
    RANDOM_SEEDS
)


# ======================================================================
# 4. DISPLAY FROZEN DEVELOPMENT CONFIGURATIONS
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "FROZEN VALIDATION-SELECTED CONFIGURATIONS"
)

print(
    "=" * 108
)


for dataset in [
    "webqsp",
    "cwq",
]:

    print(
        f"\n{dataset.upper()}"
    )

    print(
        "  AFP:",
        AFP_VALIDATION_SELECTED[
            dataset
        ]
    )

    print(
        "  Fixed Top-B:",
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset
        ]
    )

    print(
        "  Fixed Threshold:",
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset
        ]
    )


# ======================================================================
# 5. METHOD SPECIFICATION
# ======================================================================

def build_method_specs(
    dataset_name
):

    afp_params = (
        AFP_VALIDATION_SELECTED[
            dataset_name
        ]
    )

    selected_B = int(
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset_name
        ]
    )

    selected_tau = float(
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    methods = [
        {
            "method":
                "RoG",

            "family":
                "rog",

            "seed":
                None,
        },

        {
            "method":
                "Fixed-Top-B",

            "family":
                "fixed_top_b",

            "B":
                selected_B,

            "seed":
                None,
        },

        {
            "method":
                "Fixed-Threshold",

            "family":
                "fixed_threshold",

            "tau":
                selected_tau,

            "seed":
                None,
        },

        {
            "method":
                "AFP",

            "family":
                "afp",

            "T":
                float(
                    afp_params[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    afp_params[
                        "gamma_min"
                    ]
                ),

            "seed":
                None,
        },
    ]


    for seed in RANDOM_SEEDS:

        methods.append(
            {
                "method":
                    "Random-B",

                "family":
                    "random_b",

                "B":
                    selected_B,

                "seed":
                    int(
                        seed
                    ),
            }
        )


    for seed in RANDOM_SEEDS:

        methods.append(
            {
                "method":
                    "Adaptive-Budget-Random",

                "family":
                    "adaptive_budget_random",

                "T":
                    float(
                        afp_params[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        afp_params[
                            "gamma_min"
                        ]
                    ),

                "seed":
                    int(
                        seed
                    ),
            }
        )


    return methods


WEBQSP_METHODS = (
    build_method_specs(
        "webqsp"
    )
)

CWQ_METHODS = (
    build_method_specs(
        "cwq"
    )
)


# ======================================================================
# 6. DETERMINISTIC RANDOM GENERATOR
# ======================================================================
#
# Do NOT use Python hash().
#
# RNG seed derives from:
#
#   fixed seed
#   dataset
#   question ID
#   plan index
#   hop
#   active frontier
#
# Therefore runs are reproducible independent of process/hash state.
# ======================================================================

def deterministic_group_rng(
    seed,
    dataset_name,
    question_id,
    plan_index,
    hop,
    active_prefixes
):

    payload = {
        "seed":
            int(
                seed
            ),

        "dataset":
            str(
                dataset_name
            ),

        "question_id":
            str(
                question_id
            ),

        "plan_index":
            int(
                plan_index
            ),

        "hop":
            int(
                hop
            ),

        "active_prefixes":
            [
                list(
                    prefix
                )
                for prefix in active_prefixes
            ],
    }


    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":"
        ),
        ensure_ascii=False
    )


    digest = hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).digest()


    rng_seed = int.from_bytes(
        digest[
            :8
        ],
        byteorder="big",
        signed=False
    )


    return np.random.default_rng(
        rng_seed
    )


# ======================================================================
# 7. RANDOM SELECTION
# ======================================================================

def uniform_random_indices(
    n,
    k,
    rng
):

    n = int(
        n
    )

    k = int(
        k
    )


    assert n >= 1
    assert 0 <= k <= n


    if k == 0:

        return []


    if k >= n:

        return list(
            range(
                n
            )
        )


    selected = rng.choice(
        n,
        size=k,
        replace=False
    )


    # Preserve original candidate order after sampling.
    return sorted(
        int(
            x
        )
        for x in selected
    )


# ======================================================================
# 8. METHOD-SPECIFIC SELECTION
# ======================================================================

def controlled_selection(
    dataset_name,
    method_spec,
    question_id,
    plan_index,
    hop,
    active_prefixes,
    logits
):

    family = method_spec[
        "family"
    ]


    n = len(
        logits
    )


    assert n > 1


    # ------------------------------------------------------------------
    # Fixed Top-B
    # ------------------------------------------------------------------

    if family == "fixed_top_b":

        config = {
            "family":
                "fixed_top_b",

            "B":
                int(
                    method_spec[
                        "B"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "scorer_ranked_fixed_budget",
        }


    # ------------------------------------------------------------------
    # Fixed Threshold
    # ------------------------------------------------------------------

    if family == "fixed_threshold":

        config = {
            "family":
                "fixed_threshold",

            "tau":
                float(
                    method_spec[
                        "tau"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "scorer_threshold",
        }


    # ------------------------------------------------------------------
    # AFP
    # ------------------------------------------------------------------

    if family == "afp":

        config = {
            "family":
                "afp",

            "T":
                float(
                    method_spec[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    method_spec[
                        "gamma_min"
                    ]
                ),
        }


        output = (
            select_policy_indices(
                config,
                logits
            )
        )


        return {
            **output,

            "selection_type":
                "adaptive_scorer_ranked",
        }


    # ------------------------------------------------------------------
    # Random-B
    # ------------------------------------------------------------------

    if family == "random_b":

        B = int(
            method_spec[
                "B"
            ]
        )


        retained_count = min(
            B,
            n
        )


        rng = (
            deterministic_group_rng(
                seed=
                    method_spec[
                        "seed"
                    ],

                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active_prefixes
            )
        )


        selected = (
            uniform_random_indices(
                n=
                    n,

                k=
                    retained_count,

                rng=
                    rng
            )
        )


        return {
            "selected_indices":
                selected,

            "requested_budget":
                retained_count,

            "retained_count":
                len(
                    selected
                ),

            "uncertainty":
                None,

            "gamma":
                None,

            "fully_tied":
                False,

            "selection_type":
                "uniform_random_fixed_budget",
        }


    # ------------------------------------------------------------------
    # Adaptive-Budget Random
    # ------------------------------------------------------------------
    #
    # IMPORTANT:
    #
    # First compute what AFP would do on THIS encountered group.
    #
    # We use AFP's ACTUAL retained count after tie expansion as the
    # matched local budget.
    #
    # Then branch identities are selected uniformly at random.
    # ------------------------------------------------------------------

    assert family == "adaptive_budget_random"


    afp_config = {
        "family":
            "afp",

        "T":
            float(
                method_spec[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                method_spec[
                    "gamma_min"
                ]
            ),
    }


    afp_budget_output = (
        select_policy_indices(
            afp_config,
            logits
        )
    )


    matched_count = int(
        afp_budget_output[
            "retained_count"
        ]
    )


    assert (
        1
        <= matched_count
        <= n
    )


    rng = (
        deterministic_group_rng(
            seed=
                method_spec[
                    "seed"
                ],

            dataset_name=
                dataset_name,

            question_id=
                question_id,

            plan_index=
                plan_index,

            hop=
                hop,

            active_prefixes=
                active_prefixes
        )
    )


    selected = (
        uniform_random_indices(
            n=
                n,

            k=
                matched_count,

            rng=
                rng
        )
    )


    return {
        "selected_indices":
            selected,

        "requested_budget":
            int(
                afp_budget_output[
                    "requested_budget"
                ]
            ),

        "retained_count":
            len(
                selected
            ),

        "uncertainty":
            afp_budget_output[
                "uncertainty"
            ],

        "gamma":
            afp_budget_output[
                "gamma"
            ],

        "fully_tied":
            afp_budget_output[
                "fully_tied"
            ],

        "selection_type":
            "uniform_random_afp_matched_retained_count",
    }


# ======================================================================
# 9. CONTROLLED METHOD TRAVERSAL
# ======================================================================

def traverse_controlled_method(
    dataset_name,
    method_spec,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    score_cache
):

    family = method_spec[
        "family"
    ]


    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    L = len(
        plan
    )


    active_hop_rows = 0
    active_prefixes_total = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0
    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes_total += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        # Search cost THIS method would pay.
        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # RoG never prunes.
        # --------------------------------------------------------------

        if family == "rog":

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # Final-hop protection for ALL pruning methods.
        # --------------------------------------------------------------

        is_final_hop = (
            hop
            ==
            L - 1
        )


        if is_final_hop:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # Singleton bypass for ALL pruning methods.
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            controlled_selection(
                dataset_name=
                    dataset_name,

                method_spec=
                    method_spec,

                question_id=
                    question_id,

                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active,

                logits=
                    logits
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )

            requested_budget_observations += 1


        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )

            uncertainty_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes_total
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 10. RUN CONTROLLED COMPARISON
# ======================================================================

def run_controlled_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    method_specs
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    aggregate = {}


    per_question_rows = []


    for spec_index, spec in enumerate(
        method_specs
    ):

        run_id = (
            f"{spec['method']}"
            if spec[
                "seed"
            ]
            is None
            else
            f"{spec['method']}_seed{spec['seed']}"
        )


        aggregate[
            run_id
        ] = {
            "method":
                spec[
                    "method"
                ],

            "family":
                spec[
                    "family"
                ],

            "seed":
                spec[
                    "seed"
                ],

            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} controlled comparison"
        )
    ):

        plan_rec = planning_rows[
            source_index
        ]


        question_rec = question_rows[
            source_index
        ]


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # Shared computational caches only.
        expansion_cache = {}
        score_cache = {}


        question_metrics = {}


        for spec in method_specs:

            run_id = (
                f"{spec['method']}"
                if spec[
                    "seed"
                ]
                is None
                else
                f"{spec['method']}_seed{spec['seed']}"
            )


            question_metrics[
                run_id
            ] = {
                "edges_examined":
                    0,

                "active_prefixes":
                    0,

                "candidate_branches":
                    0,

                "reachable":
                    False,
            }


        # ==============================================================
        # PLAN LOOP
        # ==============================================================

        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for spec in method_specs:

                run_id = (
                    f"{spec['method']}"
                    if spec[
                        "seed"
                    ]
                    is None
                    else
                    f"{spec['method']}_seed{spec['seed']}"
                )


                result = (
                    traverse_controlled_method(
                        dataset_name=
                            dataset_name,

                        method_spec=
                            spec,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                stats = aggregate[
                    run_id
                ]


                stats[
                    "active_hop_rows"
                ] += result[
                    "active_hop_rows"
                ]


                stats[
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                stats[
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                stats[
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


                stats[
                    "decision_hops"
                ] += result[
                    "decision_hops"
                ]


                stats[
                    "retained_after_decision"
                ] += result[
                    "retained_after_decision"
                ]


                stats[
                    "requested_budget_total"
                ] += result[
                    "requested_budget_total"
                ]


                stats[
                    "requested_budget_observations"
                ] += result[
                    "requested_budget_observations"
                ]


                stats[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                stats[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                stats[
                    "fully_tied_decisions"
                ] += result[
                    "fully_tied_decisions"
                ]


                stats[
                    "peak_frontier"
                ] = max(
                    stats[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                reachable = (
                    final_prefixes_reach_answer(
                        result[
                            "final_prefixes"
                        ],
                        gold_answers
                    )
                )


                if reachable:

                    stats[
                        "reachable_plans"
                    ] += 1


                    question_metrics[
                        run_id
                    ][
                        "reachable"
                    ] = True


                question_metrics[
                    run_id
                ][
                    "edges_examined"
                ] += result[
                    "edges_examined"
                ]


                question_metrics[
                    run_id
                ][
                    "active_prefixes"
                ] += result[
                    "active_prefixes"
                ]


                question_metrics[
                    run_id
                ][
                    "candidate_branches"
                ] += result[
                    "candidate_branches"
                ]


        # ==============================================================
        # QUESTION-LEVEL AGGREGATION
        # ==============================================================

        for run_id, qstats in (
            question_metrics.items()
        ):

            if qstats[
                "reachable"
            ]:

                aggregate[
                    run_id
                ][
                    "reachable_questions"
                ] += 1


            per_question_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "question_index":
                        int(
                            source_index
                        ),

                    "question_id":
                        question_id,

                    "run_id":
                        run_id,

                    "method":
                        aggregate[
                            run_id
                        ][
                            "method"
                        ],

                    "seed":
                        aggregate[
                            run_id
                        ][
                            "seed"
                        ],

                    "edges_examined":
                        int(
                            qstats[
                                "edges_examined"
                            ]
                        ),

                    "active_prefixes":
                        int(
                            qstats[
                                "active_prefixes"
                            ]
                        ),

                    "candidate_branches":
                        int(
                            qstats[
                                "candidate_branches"
                            ]
                        ),

                    "reachable":
                        bool(
                            qstats[
                                "reachable"
                            ]
                        ),
                }
            )


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} controlled comparison completed."
    )

    print(
        f"Elapsed: {elapsed/60:.2f} min"
    )


    # ==================================================================
    # AGGREGATE TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    for run_id, stats in (
        aggregate.items()
    ):

        edges = int(
            stats[
                "edges_examined"
            ]
        )


        reachable_q = int(
            stats[
                "reachable_questions"
            ]
        )


        avg_budget = (
            stats[
                "requested_budget_total"
            ]
            /
            stats[
                "requested_budget_observations"
            ]

            if
            stats[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            stats[
                "uncertainty_total"
            ]
            /
            stats[
                "uncertainty_observations"
            ]

            if
            stats[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "run_id":
                    run_id,

                "method":
                    stats[
                        "method"
                    ],

                "family":
                    stats[
                        "family"
                    ],

                "seed":
                    stats[
                        "seed"
                    ],

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "coverage_all_questions":
                    float(
                        reachable_q
                        /
                        len(
                            planning_rows
                        )
                    ),

                "reachable_plans":
                    int(
                        stats[
                            "reachable_plans"
                        ]
                    ),

                "active_hop_rows":
                    int(
                        stats[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        stats[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        stats[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        stats[
                            "decision_hops"
                        ]
                    ),

                "retained_after_decision":
                    int(
                        stats[
                            "retained_after_decision"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        stats[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        stats[
                            "peak_frontier"
                        ]
                    ),
            }
        )


    aggregate_df = pd.DataFrame(
        rows
    )


    per_question_df = pd.DataFrame(
        per_question_rows
    )


    return (
        aggregate_df,
        per_question_df
    )


# ======================================================================
# 11. RUN BOTH DATASETS
# ======================================================================

webqsp_comparison_df, webqsp_question_df = (
    run_controlled_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        method_specs=
            WEBQSP_METHODS
    )
)


cwq_comparison_df, cwq_question_df = (
    run_controlled_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        method_specs=
            CWQ_METHODS
    )
)


# ======================================================================
# 12. RoG CONTROL FIDELITY
# ======================================================================

def assert_rog_control(
    dataset_name,
    df,
    reference
):

    rog_row = (
        df[
            df[
                "method"
            ]
            ==
            "RoG"
        ]
        .iloc[
            0
        ]
    )


    assert int(
        rog_row[
            "edges_examined"
        ]
    ) == int(
        reference[
            "edges_examined"
        ]
    )


    assert int(
        rog_row[
            "reachable_questions"
        ]
    ) == int(
        reference[
            "reachable_questions"
        ]
    )


    assert int(
        rog_row[
            "active_prefixes"
        ]
    ) == int(
        reference[
            "active_prefixes"
        ]
    )


    assert abs(
        float(
            rog_row[
                "ssr"
            ]
        )
    ) <= 1e-12


    assert abs(
        float(
            rog_row[
                "answer_retention"
            ]
        )
        -
        1.0
    ) <= 1e-12


    print(
        f"{dataset_name} RoG control fidelity: PASSED"
    )


assert_rog_control(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_rog_reference
)


assert_rog_control(
    "CWQ",
    cwq_comparison_df,
    cwq_rog_reference
)


# ======================================================================
# 13. VERIFY DETERMINISTIC METHODS MATCH CELL 13
# ======================================================================

def get_cell13_selected_row(
    tuning_df,
    family
):

    if family == "afp":

        selected_config = None

        # reconstruct from exposed selected params
        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        params = (
            AFP_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                "afp"
            )
            &
            np.isclose(
                tuning_df[
                    "T"
                ],
                params[
                    "T"
                ]
            )
            &
            np.isclose(
                tuning_df[
                    "gamma_min"
                ],
                params[
                    "gamma_min"
                ]
            )
        )


    elif family == "fixed_top_b":

        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        selected_B = (
            FIXED_TOP_B_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                family
            )
            &
            (
                tuning_df[
                    "B"
                ]
                ==
                selected_B
            )
        )


    else:

        dataset = str(
            tuning_df[
                "dataset"
            ].iloc[
                0
            ]
        )


        selected_tau = (
            FIXED_THRESHOLD_VALIDATION_SELECTED[
                dataset
            ]
        )


        mask = (
            (
                tuning_df[
                    "family"
                ]
                ==
                "fixed_threshold"
            )
            &
            np.isclose(
                tuning_df[
                    "tau"
                ],
                selected_tau
            )
        )


    rows = tuning_df[
        mask
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def deterministic_method_fidelity(
    dataset_name,
    comparison_df,
    tuning_df
):

    mapping = {
        "AFP":
            "afp",

        "Fixed-Top-B":
            "fixed_top_b",

        "Fixed-Threshold":
            "fixed_threshold",
    }


    for method, family in (
        mapping.items()
    ):

        comparison = (
            comparison_df[
                comparison_df[
                    "method"
                ]
                ==
                method
            ]
            .iloc[
                0
            ]
        )


        tuning = (
            get_cell13_selected_row(
                tuning_df,
                family
            )
        )


        assert int(
            comparison[
                "edges_examined"
            ]
        ) == int(
            tuning[
                "edges_examined"
            ]
        ), (
            f"{dataset_name} {method} edge mismatch "
            "vs Cell 13."
        )


        assert int(
            comparison[
                "reachable_questions"
            ]
        ) == int(
            tuning[
                "reachable_questions"
            ]
        ), (
            f"{dataset_name} {method} AR mismatch "
            "vs Cell 13."
        )


    print(
        f"{dataset_name} deterministic-method "
        "Cell-13 fidelity: PASSED"
    )


deterministic_method_fidelity(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_tuning_df
)


deterministic_method_fidelity(
    "CWQ",
    cwq_comparison_df,
    cwq_tuning_df
)


# ======================================================================
# 14. RANDOM BASELINE SUMMARY
# ======================================================================

def summarize_method_runs(
    comparison_df
):

    summary_rows = []


    method_order = [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
        "Random-B",
        "Adaptive-Budget-Random",
        "AFP",
    ]


    for method in method_order:

        rows = comparison_df[
            comparison_df[
                "method"
            ]
            ==
            method
        ]


        assert len(
            rows
        ) > 0


        summary_rows.append(
            {
                "method":
                    method,

                "n_runs":
                    int(
                        len(
                            rows
                        )
                    ),

                "edges_mean":
                    float(
                        rows[
                            "edges_examined"
                        ].mean()
                    ),

                "edges_sd":
                    float(
                        rows[
                            "edges_examined"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "ssr_mean":
                    float(
                        rows[
                            "ssr"
                        ].mean()
                    ),

                "ssr_sd":
                    float(
                        rows[
                            "ssr"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "ar_mean":
                    float(
                        rows[
                            "answer_retention"
                        ].mean()
                    ),

                "ar_sd":
                    float(
                        rows[
                            "answer_retention"
                        ].std(
                            ddof=1
                        )
                    )
                    if len(
                        rows
                    ) > 1
                    else
                    0.0,

                "reachable_q_mean":
                    float(
                        rows[
                            "reachable_questions"
                        ].mean()
                    ),

                "active_prefixes_mean":
                    float(
                        rows[
                            "active_prefixes"
                        ].mean()
                    ),

                "candidate_branches_mean":
                    float(
                        rows[
                            "candidate_branches"
                        ].mean()
                    ),

                "decision_hops_mean":
                    float(
                        rows[
                            "decision_hops"
                        ].mean()
                    ),

                "avg_requested_budget_mean":
                    float(
                        rows[
                            "avg_requested_budget"
                        ].mean()
                    )
                    if rows[
                        "avg_requested_budget"
                    ].notna().any()
                    else
                    np.nan,
            }
        )


    return pd.DataFrame(
        summary_rows
    )


webqsp_summary_df = (
    summarize_method_runs(
        webqsp_comparison_df
    )
)


cwq_summary_df = (
    summarize_method_runs(
        cwq_comparison_df
    )
)


# ======================================================================
# 15. DISPLAY RUN-LEVEL RESULTS
# ======================================================================

RUN_COLUMNS = [
    "method",
    "seed",
    "edges_examined",
    "ssr",
    "reachable_questions",
    "answer_retention",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "fully_tied_decisions",
    "peak_frontier",
]


def display_results(
    dataset_name,
    comparison_df,
    summary_df
):

    print(
        "\n"
        + "=" * 118
    )

    print(
        f"{dataset_name.upper()} "
        "CONTROLLED VALIDATION COMPARISON — RUN LEVEL"
    )

    print(
        "=" * 118
    )


    print(
        comparison_df[
            RUN_COLUMNS
        ].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


    print(
        "\n"
        + "=" * 118
    )

    print(
        f"{dataset_name.upper()} "
        "CONTROLLED VALIDATION COMPARISON — SUMMARY"
    )

    print(
        "=" * 118
    )


    print(
        summary_df.to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_results(
    "webqsp",
    webqsp_comparison_df,
    webqsp_summary_df
)


display_results(
    "cwq",
    cwq_comparison_df,
    cwq_summary_df
)


# ======================================================================
# 16. KEY CAUSAL COMPARISONS
# ======================================================================

def method_summary_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def print_causal_diagnostics(
    dataset_name,
    summary_df
):

    afp = method_summary_row(
        summary_df,
        "AFP"
    )


    fixed = method_summary_row(
        summary_df,
        "Fixed-Top-B"
    )


    random_b = method_summary_row(
        summary_df,
        "Random-B"
    )


    adaptive_random = (
        method_summary_row(
            summary_df,
            "Adaptive-Budget-Random"
        )
    )


    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} CAUSAL DIAGNOSTICS"
    )

    print(
        "=" * 112
    )


    print(
        "\nAFP:"
    )

    print(
        f"  AR  = {afp['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {afp['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Top-B:"
    )

    print(
        f"  AR  = {fixed['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed['ssr_mean']:.6f}"
    )


    print(
        "\nRandom-B:"
    )

    print(
        f"  AR  = {random_b['ar_mean']:.6f} "
        f"± {random_b['ar_sd']:.6f}"
    )

    print(
        f"  SSR = {random_b['ssr_mean']:.6f} "
        f"± {random_b['ssr_sd']:.6f}"
    )


    print(
        "\nAdaptive-Budget Random:"
    )

    print(
        f"  AR  = {adaptive_random['ar_mean']:.6f} "
        f"± {adaptive_random['ar_sd']:.6f}"
    )

    print(
        f"  SSR = {adaptive_random['ssr_mean']:.6f} "
        f"± {adaptive_random['ssr_sd']:.6f}"
    )


    print(
        "\nRanking-value diagnostic:"
    )

    print(
        "  AFP AR - Adaptive-Random AR =",
        f"{afp['ar_mean'] - adaptive_random['ar_mean']:.6f}"
    )

    print(
        "  AFP SSR - Adaptive-Random SSR =",
        f"{afp['ssr_mean'] - adaptive_random['ssr_mean']:.6f}"
    )


    print(
        "\nFixed-ranking diagnostic:"
    )

    print(
        "  Fixed Top-B AR - Random-B AR =",
        f"{fixed['ar_mean'] - random_b['ar_mean']:.6f}"
    )

    print(
        "  Fixed Top-B SSR - Random-B SSR =",
        f"{fixed['ssr_mean'] - random_b['ssr_mean']:.6f}"
    )


print_causal_diagnostics(
    "webqsp",
    webqsp_summary_df
)


print_causal_diagnostics(
    "cwq",
    cwq_summary_df
)


# ======================================================================
# 17. EXPORT ARTIFACTS
# ======================================================================

WEBQSP_RUN_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_runs.csv"
)

CWQ_RUN_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_runs.csv"
)

WEBQSP_SUMMARY_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_summary.csv"
)

CWQ_SUMMARY_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_summary.csv"
)

WEBQSP_QUESTION_CSV = (
    COMPARE_DIR
    / "webqsp_validation_controlled_per_question.csv"
)

CWQ_QUESTION_CSV = (
    COMPARE_DIR
    / "cwq_validation_controlled_per_question.csv"
)


webqsp_comparison_df.to_csv(
    WEBQSP_RUN_CSV,
    index=False
)

cwq_comparison_df.to_csv(
    CWQ_RUN_CSV,
    index=False
)

webqsp_summary_df.to_csv(
    WEBQSP_SUMMARY_CSV,
    index=False
)

cwq_summary_df.to_csv(
    CWQ_SUMMARY_CSV,
    index=False
)

webqsp_question_df.to_csv(
    WEBQSP_QUESTION_CSV,
    index=False
)

cwq_question_df.to_csv(
    CWQ_QUESTION_CSV,
    index=False
)


# ======================================================================
# 18. MANIFEST
# ======================================================================

CELL14_MANIFEST = {
    "cell":
        "RQ2_CELL14_VALIDATION_CONTROLLED_COMPARISON",

    "version":
        "rq2_cell14_controlled_comparison_v1",

    "datasets": [
        "webqsp",
        "cwq",
    ],

    "methods": [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
        "Random-B",
        "Adaptive-Budget-Random",
        "AFP",
    ],

    "random_seeds":
        RANDOM_SEEDS,

    "selected_parameters": {
        "webqsp": {
            "afp":
                AFP_VALIDATION_SELECTED[
                    "webqsp"
                ],

            "fixed_top_b":
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "webqsp"
                ],

            "fixed_threshold":
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "webqsp"
                ],
        },

        "cwq": {
            "afp":
                AFP_VALIDATION_SELECTED[
                    "cwq"
                ],

            "fixed_top_b":
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "cwq"
                ],

            "fixed_threshold":
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "cwq"
                ],
        },
    },

    "random_b": {
        "budget":
            "selected_fixed_top_b",

        "selection":
            "uniform_without_replacement",

        "preserve_candidate_order_after_sampling":
            True,
    },

    "adaptive_budget_random": {
        "budget":
            "AFP_actual_retained_count_after_tie_expansion",

        "scorer_use":
            "budget_only",

        "selection":
            "uniform_without_replacement",

        "preserve_candidate_order_after_sampling":
            True,
    },

    "rng": {
        "type":
            "sha256_deterministic_group_rng",

        "python_hash_used":
            False,
    },

    "final_hop_protection":
        True,

    "singleton_bypass":
        True,

    "primary_search_cost":
        "edges_examined",

    "answer_retention_reference":
        "RoG_reachable_questions",

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_reachability_only",

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,

    "next_step":
        "RQ2_Cell15_ablations_and_final_development_freeze",
}


CELL14_MANIFEST_PATH = (
    COMPARE_DIR
    / "cell14_validation_controlled_comparison_manifest.json"
)


with open(
    CELL14_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL14_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 19. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 14: VALIDATION CONTROLLED COMPARISON COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nControlled methods:"
)

print(
    "  RoG"
)

print(
    "  Fixed Top-B"
)

print(
    "  Fixed Threshold"
)

print(
    "  Random-B [42,43,44]"
)

print(
    "  Adaptive-Budget Random [42,43,44]"
)

print(
    "  AFP"
)


print(
    "\nControl gates:"
)

print(
    "  RoG fidelity:                 PASSED"
)

print(
    "  Cell-13 deterministic match:  PASSED"
)

print(
    "  Random selection reproducible: SHA256-based"
)

print(
    "  Gold in scorer/selector:      NO"
)

print(
    "  TEST examples accessed:       NO"
)


print(
    "\nDevelopment status:"
)

print(
    "  Controlled comparison: COMPLETE"
)

print(
    "  Complete AFP frozen:    NO"
)

print(
    "  Next: causal ablations + final development decision"
)


print(
    "\nOutputs:"
)

for path in [
    WEBQSP_RUN_CSV,
    CWQ_RUN_CSV,
    WEBQSP_SUMMARY_CSV,
    CWQ_SUMMARY_CSV,
    WEBQSP_QUESTION_CSV,
    CWQ_QUESTION_CSV,
    CELL14_MANIFEST_PATH,
]:

    print(
        " ",
        path
    )

Cell 14 prerequisites: PASSED
Random baseline seeds: [42, 43, 44]

FROZEN VALIDATION-SELECTED CONFIGURATIONS

WEBQSP
  AFP: {'T': 0.5, 'gamma_min': 0.5}
  Fixed Top-B: 1
  Fixed Threshold: 0.3

CWQ
  AFP: {'T': 0.5, 'gamma_min': 0.5}
  Fixed Top-B: 8
  Fixed Threshold: 0.1


webqsp controlled comparison:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP controlled comparison completed.
Elapsed: 0.04 min


cwq controlled comparison:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ controlled comparison completed.
Elapsed: 2.16 min
WebQSP RoG control fidelity: PASSED
CWQ RoG control fidelity: PASSED
WebQSP deterministic-method Cell-13 fidelity: PASSED
CWQ deterministic-method Cell-13 fidelity: PASSED

WEBQSP CONTROLLED VALIDATION COMPARISON — RUN LEVEL
                method      seed  edges_examined      ssr  reachable_questions  answer_retention  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  fully_tied_decisions  peak_frontier
                   RoG       NaN          341526 0.000000                  205          1.000000             2440                7983              0                   NaN                     0            533
           Fixed-Top-B       NaN          335061 0.018930                  205          1.000000             2011                5962            147              1.000000                    97            145
       Fixed-Threshold       NaN          338714 0.008234                  203          0.99024

In [29]:
# ======================================================================
# RQ2 CELL 13B
# ONE-TIME AFP SELECTOR BOUNDARY EXPANSION
# ======================================================================
#
# WHY THIS CELL EXISTS
# --------------------
# Original Cell 13 selected:
#
#       T = 0.5
#       gamma_min = 0.5
#
# on BOTH WebQSP and CWQ.
#
# Both values were the LOWER BOUNDARIES of the original grid.
#
# Cell 14 further showed that AFP remains extremely conservative:
#
#   WebQSP SSR = 0.001403 at AR = 1.000
#   CWQ    SSR = 0.002092 at AR = 1.000
#
# Therefore we perform ONE predeclared lower-bound expansion before
# freezing the method.
#
# THIS IS THE ONLY BOUNDARY-EXPANSION PASS.
# We will NOT iteratively expand the grid again based on its outcome.
#
# NEW GRID
# --------
# T:
#       0.05, 0.10, 0.20, 0.30, 0.50
#
# gamma_min:
#       0.10, 0.30, 0.50
#
# The original selected point (0.5, 0.5) is included as an anchor and
# MUST exactly reproduce Cell-13 results.
#
# SELECTION RULE IS UNCHANGED:
#
#       maximize SSR subject to AR >= 0.99
#
# fallback:
#
#       maximize AR, then SSR
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ======================================================================
# 1. PREREQUISITES
# ======================================================================

required = [
    "CELL13_TUNING_COMPLETE",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "traverse_with_policy",
    "final_prefixes_reach_answer",

    "webqsp_tuning_df",
    "cwq_tuning_df",

    "AFP_VALIDATION_SELECTED",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell-13 runtime objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13_TUNING_COMPLETE is True

print("Cell 13B prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

BOUNDARY_DIR = (
    ROOT
    / "11_validation_tuning"
    / "selector_boundary_expansion"
)

BOUNDARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PREDECLARE THE SINGLE EXPANDED GRID
# ======================================================================

AR_FLOOR_13B = 0.99


EXPANDED_T_GRID = [
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]


EXPANDED_GAMMA_MIN_GRID = [
    0.10,
    0.30,
    0.50,
]


EXPANDED_CONFIGS = [
    {
        "config_id":
            f"afp_T{T:g}_g{gamma:g}",

        "family":
            "afp",

        "T":
            float(T),

        "gamma_min":
            float(gamma),
    }

    for T in EXPANDED_T_GRID
    for gamma in EXPANDED_GAMMA_MIN_GRID
]


assert len(
    EXPANDED_CONFIGS
) == 15


assert any(
    np.isclose(
        c["T"],
        0.5
    )
    and
    np.isclose(
        c["gamma_min"],
        0.5
    )
    for c in EXPANDED_CONFIGS
)


print(
    "\n"
    + "=" * 110
)

print(
    "ONE-TIME AFP SELECTOR BOUNDARY EXPANSION"
)

print(
    "=" * 110
)


print(
    "Selection rule: maximize SSR "
    "subject to AR >= 0.99"
)

print(
    "Fallback: maximize AR, then SSR"
)

print(
    "\nT grid:",
    EXPANDED_T_GRID
)

print(
    "gamma_min grid:",
    EXPANDED_GAMMA_MIN_GRID
)

print(
    "Configurations:",
    len(
        EXPANDED_CONFIGS
    )
)

print(
    "\nIMPORTANT: no further automatic "
    "grid expansion after this cell."
)


# ======================================================================
# 4. SPECIFICATION FINGERPRINT
# ======================================================================

CELL13B_SPEC = {
    "version":
        "rq2_cell13b_selector_boundary_expansion_v1",

    "reason":
        "original_selection_hit_both_lower_grid_boundaries",

    "original_selected": {
        "webqsp":
            AFP_VALIDATION_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_VALIDATION_SELECTED[
                "cwq"
            ],
    },

    "expanded_T_grid":
        EXPANDED_T_GRID,

    "expanded_gamma_min_grid":
        EXPANDED_GAMMA_MIN_GRID,

    "selection_rule":
        "maximize_ssr_subject_to_ar_0.99",

    "fallback":
        "maximize_ar_then_ssr",

    "one_time_expansion_only":
        True,

    "test_examples_accessed":
        False,
}


CELL13B_SPEC_SHA = hashlib.sha256(
    json.dumps(
        CELL13B_SPEC,
        sort_keys=True,
        separators=(",", ":")
    ).encode(
        "utf-8"
    )
).hexdigest()


print(
    "\nCell 13B specification SHA256:",
    CELL13B_SPEC_SHA
)


# ======================================================================
# 5. FRESH AFP STATISTICS
# ======================================================================

def fresh_13b_stats():

    return {
        config[
            "config_id"
        ]: {
            "active_hop_rows":
                0,

            "active_prefixes":
                0,

            "edges_examined":
                0,

            "candidate_branches":
                0,

            "reachable_plans":
                0,

            "reachable_questions":
                0,

            "decision_hops":
                0,

            "retained_after_decision":
                0,

            "requested_budget_total":
                0,

            "requested_budget_observations":
                0,

            "uncertainty_total":
                0.0,

            "uncertainty_observations":
                0,

            "fully_tied_decisions":
                0,

            "peak_frontier":
                0,
        }

        for config in EXPANDED_CONFIGS
    }


# ======================================================================
# 6. RUN ONE DATASET
# ======================================================================

def run_13b_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    stats = fresh_13b_stats()


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} Cell13B"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        # --------------------------------------------------------------
        # Computational caches only.
        # Method metrics are still accumulated independently.
        # --------------------------------------------------------------

        expansion_cache = {}

        score_cache = {}


        question_reachable = {
            config[
                "config_id"
            ]:
                False

            for config in EXPANDED_CONFIGS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for config in EXPANDED_CONFIGS:

                config_id = (
                    config[
                        "config_id"
                    ]
                )


                result = (
                    traverse_with_policy(
                        dataset_name=
                            dataset_name,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        config=
                            config,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    config_id
                ]


                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[
                        key
                    ] += result[
                        key
                    ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # GOLD IS CONSULTED ONLY AFTER TRAVERSAL.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachable[
                        config_id
                    ] = True


        for config_id, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    config_id
                ][
                    "reachable_questions"
                ] += 1


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} Cell13B completed "
        f"in {elapsed/60:.2f} min"
    )


    # ==================================================================
    # BUILD RESULT TABLE
    # ==================================================================

    rows = []


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable_questions = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    for config in EXPANDED_CONFIGS:

        config_id = (
            config[
                "config_id"
            ]
        )


        s = stats[
            config_id
        ]


        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_questions = int(
            s[
                "reachable_questions"
            ]
        )


        ssr = (
            1.0
            -
            edges
            /
            rog_edges
        )


        ar = (
            reachable_questions
            /
            rog_reachable_questions
        )


        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "config_id":
                    config_id,

                "T":
                    float(
                        config[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        config[
                            "gamma_min"
                        ]
                    ),

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        ssr
                    ),

                "reachable_questions":
                    reachable_questions,

                "rog_reachable_questions":
                    rog_reachable_questions,

                "answer_retention":
                    float(
                        ar
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),

                "ar_floor_feasible":
                    bool(
                        ar
                        >=
                        AR_FLOOR_13B
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================
# 7. RUN WEBQSP + CWQ
# ======================================================================

webqsp_13b_df = (
    run_13b_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference
    )
)


cwq_13b_df = (
    run_13b_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference
    )
)


# ======================================================================
# 8. ORIGINAL-POINT SOFTWARE FIDELITY GATE
# ======================================================================
#
# T=0.5, gamma=0.5 MUST reproduce original Cell 13 exactly.
# ======================================================================

def original_afp_row(
    tuning_df
):

    rows = tuning_df[
        (
            tuning_df[
                "family"
            ]
            ==
            "afp"
        )
        &
        np.isclose(
            tuning_df[
                "T"
            ],
            0.5
        )
        &
        np.isclose(
            tuning_df[
                "gamma_min"
            ],
            0.5
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def expanded_anchor_row(
    df
):

    rows = df[
        np.isclose(
            df[
                "T"
            ],
            0.5
        )
        &
        np.isclose(
            df[
                "gamma_min"
            ],
            0.5
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


for dataset_name, old_df, new_df in [
    (
        "WebQSP",
        webqsp_tuning_df,
        webqsp_13b_df
    ),

    (
        "CWQ",
        cwq_tuning_df,
        cwq_13b_df
    ),
]:

    old_row = (
        original_afp_row(
            old_df
        )
    )


    new_row = (
        expanded_anchor_row(
            new_df
        )
    )


    assert int(
        old_row[
            "edges_examined"
        ]
    ) == int(
        new_row[
            "edges_examined"
        ]
    )


    assert int(
        old_row[
            "reachable_questions"
        ]
    ) == int(
        new_row[
            "reachable_questions"
        ]
    )


    assert np.isclose(
        float(
            old_row[
                "ssr"
            ]
        ),
        float(
            new_row[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            old_row[
                "answer_retention"
            ]
        ),
        float(
            new_row[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} original AFP anchor fidelity: PASSED"
    )


# ======================================================================
# 9. SAME PREDECLARED SELECTION RULE
# ======================================================================

def select_13b(
    df
):

    feasible = df[
        df[
            "answer_retention"
        ]
        >=
        AR_FLOOR_13B
    ].copy()


    if len(
        feasible
    ) > 0:

        mode = (
            "maximize_ssr_subject_to_ar_floor"
        )


        ranked = feasible.sort_values(
            by=[
                "ssr",
                "answer_retention",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    else:

        mode = (
            "fallback_maximize_ar_then_ssr"
        )


        ranked = df.sort_values(
            by=[
                "answer_retention",
                "ssr",
                "config_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],

            kind="stable"
        )


    selected = (
        ranked.iloc[
            0
        ].to_dict()
    )


    selected[
        "selection_mode"
    ] = mode


    return (
        selected,
        ranked
    )


webqsp_13b_selected, webqsp_13b_ranked = (
    select_13b(
        webqsp_13b_df
    )
)


cwq_13b_selected, cwq_13b_ranked = (
    select_13b(
        cwq_13b_df
    )
)


# ======================================================================
# 10. DISPLAY ALL EXPANDED RESULTS
# ======================================================================

DISPLAY_COLUMNS = [
    "config_id",
    "T",
    "gamma_min",
    "ssr",
    "answer_retention",
    "reachable_questions",
    "edges_examined",
    "active_prefixes",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "ar_floor_feasible",
]


def show_13b_results(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 116
    )

    print(
        f"{dataset_name.upper()} "
        "AFP BOUNDARY-EXPANSION RESULTS"
    )

    print(
        "=" * 116
    )


    ordered = df.sort_values(
        by=[
            "answer_retention",
            "ssr",
        ],

        ascending=[
            False,
            False,
        ]
    )


    print(
        ordered[
            DISPLAY_COLUMNS
        ].to_string(
            index=False,

            float_format=lambda x:
                f"{x:.6f}"
        )
    )


show_13b_results(
    "webqsp",
    webqsp_13b_df
)


show_13b_results(
    "cwq",
    cwq_13b_df
)


# ======================================================================
# 11. COMPARE OLD vs NEW SELECTED AFP
# ======================================================================

def print_selection_comparison(
    dataset_name,
    old_tuning_df,
    new_selected
):

    old = (
        original_afp_row(
            old_tuning_df
        )
    )


    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} OLD vs EXPANDED AFP"
    )

    print(
        "=" * 112
    )


    print(
        "\nOLD"
    )

    print(
        "  T:",
        float(
            old[
                "T"
            ]
        )
    )

    print(
        "  gamma_min:",
        float(
            old[
                "gamma_min"
            ]
        )
    )

    print(
        "  AR:",
        f"{float(old['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(old['ssr']):.6f}"
    )


    print(
        "\nEXPANDED SELECTED"
    )

    print(
        "  T:",
        new_selected[
            "T"
        ]
    )

    print(
        "  gamma_min:",
        new_selected[
            "gamma_min"
        ]
    )

    print(
        "  AR:",
        f"{new_selected['answer_retention']:.6f}"
    )

    print(
        "  SSR:",
        f"{new_selected['ssr']:.6f}"
    )

    print(
        "  avg budget:",
        f"{new_selected['avg_requested_budget']:.6f}"
    )

    print(
        "  avg uncertainty:",
        f"{new_selected['avg_uncertainty']:.6f}"
    )

    print(
        "  selection mode:",
        new_selected[
            "selection_mode"
        ]
    )


print_selection_comparison(
    "webqsp",
    webqsp_tuning_df,
    webqsp_13b_selected
)


print_selection_comparison(
    "cwq",
    cwq_tuning_df,
    cwq_13b_selected
)


# ======================================================================
# 12. EXPOSE CANDIDATE SELECTED PARAMETERS
# ======================================================================
#
# DO NOT overwrite AFP_VALIDATION_SELECTED yet.
#
# We first inspect the result scientifically.
# ======================================================================

AFP_BOUNDARY_EXPANSION_SELECTED = {
    "webqsp": {
        "T":
            float(
                webqsp_13b_selected[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                webqsp_13b_selected[
                    "gamma_min"
                ]
            ),

        "answer_retention":
            float(
                webqsp_13b_selected[
                    "answer_retention"
                ]
            ),

        "ssr":
            float(
                webqsp_13b_selected[
                    "ssr"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                cwq_13b_selected[
                    "T"
                ]
            ),

        "gamma_min":
            float(
                cwq_13b_selected[
                    "gamma_min"
                ]
            ),

        "answer_retention":
            float(
                cwq_13b_selected[
                    "answer_retention"
                ]
            ),

        "ssr":
            float(
                cwq_13b_selected[
                    "ssr"
                ]
            ),
    },
}


# ======================================================================
# 13. SAVE ARTIFACTS
# ======================================================================

WEBQSP_13B_CSV = (
    BOUNDARY_DIR
    / "webqsp_afp_boundary_expansion.csv"
)


CWQ_13B_CSV = (
    BOUNDARY_DIR
    / "cwq_afp_boundary_expansion.csv"
)


webqsp_13b_df.to_csv(
    WEBQSP_13B_CSV,
    index=False
)


cwq_13b_df.to_csv(
    CWQ_13B_CSV,
    index=False
)


CELL13B_MANIFEST = {
    "cell":
        "RQ2_CELL13B_AFP_SELECTOR_BOUNDARY_EXPANSION",

    "version":
        "rq2_cell13b_selector_boundary_expansion_v1",

    "spec_sha256":
        CELL13B_SPEC_SHA,

    "trigger":
        "Cell13 selected both lower grid boundaries on both datasets",

    "one_time_expansion_only":
        True,

    "expanded_grid": {
        "T":
            EXPANDED_T_GRID,

        "gamma_min":
            EXPANDED_GAMMA_MIN_GRID,
    },

    "selection_rule":
        "maximize SSR subject to AR >= 0.99; "
        "fallback maximize AR then SSR",

    "selected_candidate": {
        "webqsp":
            AFP_BOUNDARY_EXPANSION_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_BOUNDARY_EXPANSION_SELECTED[
                "cwq"
            ],
    },

    "original_anchor_fidelity":
        True,

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_AR_only",

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,
}


CELL13B_MANIFEST_PATH = (
    BOUNDARY_DIR
    / "cell13b_selector_boundary_expansion_manifest.json"
)


with open(
    CELL13B_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL13B_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL13B_COMPLETE = True


# ======================================================================
# 14. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 13B: "
    "ONE-TIME AFP SELECTOR BOUNDARY EXPANSION COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nCandidate selected AFP parameters:"
)

print(
    "  WebQSP:",
    AFP_BOUNDARY_EXPANSION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_BOUNDARY_EXPANSION_SELECTED[
        "cwq"
    ]
)


print(
    "\nScientific status:"
)

print(
    "  Original Cell-13 anchor reproduced: YES"
)

print(
    "  Validation-only expansion:          YES"
)

print(
    "  Further automatic grid expansion:   NO"
)

print(
    "  AFP parameters overwritten:         NO"
)

print(
    "  TEST examples accessed:             NO"
)

print(
    "  Final AFP frozen:                   NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_13B_CSV
)

print(
    " ",
    CWQ_13B_CSV
)

print(
    " ",
    CELL13B_MANIFEST_PATH
)


print(
    "\nNEXT:"
)

print(
    "Review the expanded selector result before "
    "re-running the controlled comparison or proceeding to ablation/freeze."
)

Cell 13B prerequisites: PASSED

ONE-TIME AFP SELECTOR BOUNDARY EXPANSION
Selection rule: maximize SSR subject to AR >= 0.99
Fallback: maximize AR, then SSR

T grid: [0.05, 0.1, 0.2, 0.3, 0.5]
gamma_min grid: [0.1, 0.3, 0.5]
Configurations: 15

IMPORTANT: no further automatic grid expansion after this cell.

Cell 13B specification SHA256: 5e882cc11191c1d477465d938da3a65648a9e60409a1b49cf287c0a84c6d5dd2


webqsp Cell13B:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP Cell13B completed in 0.04 min


cwq Cell13B:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ Cell13B completed in 2.37 min
WebQSP original AFP anchor fidelity: PASSED
CWQ original AFP anchor fidelity: PASSED

WEBQSP AFP BOUNDARY-EXPANSION RESULTS
     config_id        T  gamma_min      ssr  answer_retention  reachable_questions  edges_examined  active_prefixes  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  ar_floor_feasible
afp_T0.05_g0.1 0.050000   0.100000 0.010272          1.000000                  205          338018             2293            147              9.891156         0.894528                    97               True
afp_T0.05_g0.3 0.050000   0.300000 0.009349          1.000000                  205          338333             2300            147              9.938776         0.894528                    97               True
afp_T0.05_g0.5 0.050000   0.500000 0.007973          1.000000                  205          338803             2312            147             10.020408         0.894528                    97               Tr

In [30]:
# ======================================================================
# RQ2 CELL 14B
# CONTROLLED VALIDATION COMPARISON WITH EXPANDED AFP SELECTOR
# ======================================================================
#
# RUN AS A NEW CELL.
# DO NOT REPLACE CELL 13, CELL 13B, OR CELL 14.
# DO NOT RESTART THE KERNEL.
#
# PURPOSE
# -------
# Re-run the controlled validation comparison after the single
# predeclared AFP selector boundary expansion.
#
# UPDATED AFP:
#
#   WebQSP:
#       T = 0.05
#       gamma_min = 0.10
#
#   CWQ:
#       T = 0.10
#       gamma_min = 0.10
#
# UNCHANGED:
#   RoG
#   Fixed Top-B
#   Fixed Threshold
#   Random-B
#   scorer
#   Feature-v2
#   plans
#   graphs
#   traversal
#   seeds
#   final-hop protection
#
# Adaptive-Budget Random inherits the NEW AFP adaptive retained counts.
#
# VALIDATION ONLY.
# NO TEST examples accessed.
# NO further hyperparameter search.
# ======================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL13B_COMPLETE",
    "AFP_BOUNDARY_EXPANSION_SELECTED",

    "FIXED_TOP_B_VALIDATION_SELECTED",
    "FIXED_THRESHOLD_VALIDATION_SELECTED",

    "webqsp_13b_df",
    "cwq_13b_df",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    # Cell 14 controlled-comparison functions
    "run_controlled_dataset",
    "summarize_method_runs",
    "assert_rog_control",
    "display_results",
    "print_causal_diagnostics",

    # Selected scorer artifact identity
    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required Cell 13B / Cell 14 objects:\n  "
    + "\n  ".join(missing)
)

assert CELL13B_COMPLETE is True

print("Cell 14B prerequisites: PASSED")


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

COMPARE14B_DIR = (
    ROOT
    / "12_validation_comparison"
    / "expanded_afp"
)

COMPARE14B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. PROMOTE THE ONE-TIME EXPANSION RESULT FOR CONTROLLED EVALUATION
# ======================================================================
#
# This does NOT freeze final AFP.
#
# These are now the development-selected selector candidates because:
#
#   1. the expansion was triggered only because the original optimum
#      landed at both lower grid boundaries;
#   2. expansion was declared one-time;
#   3. the same AR >= 0.99 selection rule was retained;
#   4. the original grid anchor reproduced exactly.
# ======================================================================

AFP_14B_SELECTED = {
    "webqsp": {
        "T":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "webqsp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "webqsp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "cwq"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_BOUNDARY_EXPANSION_SELECTED[
                    "cwq"
                ][
                    "gamma_min"
                ]
            ),
    },
}


assert AFP_14B_SELECTED[
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert AFP_14B_SELECTED[
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


print(
    "\nExpanded AFP selected for Cell 14B:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nUnchanged baselines:"
)

print(
    "  WebQSP Fixed Top-B:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  WebQSP threshold:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ Fixed Top-B:",
    FIXED_TOP_B_VALIDATION_SELECTED[
        "cwq"
    ]
)

print(
    "  CWQ threshold:",
    FIXED_THRESHOLD_VALIDATION_SELECTED[
        "cwq"
    ]
)


# ======================================================================
# 4. RANDOM SEEDS REMAIN FROZEN
# ======================================================================

RANDOM_SEEDS_14B = [
    42,
    43,
    44,
]


# ======================================================================
# 5. BUILD UPDATED METHOD SPECS
# ======================================================================

def build_method_specs_14b(
    dataset_name
):

    afp_params = (
        AFP_14B_SELECTED[
            dataset_name
        ]
    )


    selected_B = int(
        FIXED_TOP_B_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    selected_tau = float(
        FIXED_THRESHOLD_VALIDATION_SELECTED[
            dataset_name
        ]
    )


    methods = [
        # --------------------------------------------------------------
        # Unpruned control
        # --------------------------------------------------------------
        {
            "method":
                "RoG",

            "family":
                "rog",

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Frozen fixed-budget baseline
        # --------------------------------------------------------------
        {
            "method":
                "Fixed-Top-B",

            "family":
                "fixed_top_b",

            "B":
                selected_B,

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Frozen threshold baseline
        # --------------------------------------------------------------
        {
            "method":
                "Fixed-Threshold",

            "family":
                "fixed_threshold",

            "tau":
                selected_tau,

            "seed":
                None,
        },

        # --------------------------------------------------------------
        # Expanded-validation-selected AFP
        # --------------------------------------------------------------
        {
            "method":
                "AFP",

            "family":
                "afp",

            "T":
                float(
                    afp_params[
                        "T"
                    ]
                ),

            "gamma_min":
                float(
                    afp_params[
                        "gamma_min"
                    ]
                ),

            "seed":
                None,
        },
    ]


    # ------------------------------------------------------------------
    # Random-B:
    # unchanged selected Fixed Top-B budget.
    # ------------------------------------------------------------------

    for seed in RANDOM_SEEDS_14B:

        methods.append(
            {
                "method":
                    "Random-B",

                "family":
                    "random_b",

                "B":
                    selected_B,

                "seed":
                    int(
                        seed
                    ),
            }
        )


    # ------------------------------------------------------------------
    # Adaptive-Budget Random:
    #
    # uses NEW AFP T/gamma_min.
    #
    # At each encountered dynamic group it receives AFP's actual
    # tie-expanded retained count, but randomly chooses WHICH candidates
    # survive.
    # ------------------------------------------------------------------

    for seed in RANDOM_SEEDS_14B:

        methods.append(
            {
                "method":
                    "Adaptive-Budget-Random",

                "family":
                    "adaptive_budget_random",

                "T":
                    float(
                        afp_params[
                            "T"
                        ]
                    ),

                "gamma_min":
                    float(
                        afp_params[
                            "gamma_min"
                        ]
                    ),

                "seed":
                    int(
                        seed
                    ),
            }
        )


    return methods


WEBQSP_METHODS_14B = (
    build_method_specs_14b(
        "webqsp"
    )
)


CWQ_METHODS_14B = (
    build_method_specs_14b(
        "cwq"
    )
)


# ======================================================================
# 6. RUN CONTROLLED COMPARISON
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "RUNNING CELL 14B CONTROLLED VALIDATION COMPARISON"
)

print(
    "=" * 118
)


webqsp_comparison_14b_df, webqsp_question_14b_df = (
    run_controlled_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        method_specs=
            WEBQSP_METHODS_14B
    )
)


cwq_comparison_14b_df, cwq_question_14b_df = (
    run_controlled_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        method_specs=
            CWQ_METHODS_14B
    )
)


# ======================================================================
# 7. RoG CONTROL FIDELITY
# ======================================================================

assert_rog_control(
    "WebQSP",
    webqsp_comparison_14b_df,
    webqsp_rog_reference
)


assert_rog_control(
    "CWQ",
    cwq_comparison_14b_df,
    cwq_rog_reference
)


print(
    "\nCell 14B RoG control fidelity: PASSED"
)


# ======================================================================
# 8. FIXED BASELINES MUST MATCH ORIGINAL CELL 14 EXACTLY
# ======================================================================

def deterministic_baseline_match(
    dataset_name,
    old_df,
    new_df
):

    for method in [
        "RoG",
        "Fixed-Top-B",
        "Fixed-Threshold",
    ]:

        old_rows = old_df[
            old_df[
                "method"
            ]
            ==
            method
        ]


        new_rows = new_df[
            new_df[
                "method"
            ]
            ==
            method
        ]


        assert len(
            old_rows
        ) == 1

        assert len(
            new_rows
        ) == 1


        old = old_rows.iloc[
            0
        ]

        new = new_rows.iloc[
            0
        ]


        for key in [
            "edges_examined",
            "reachable_questions",
            "active_prefixes",
            "candidate_branches",
        ]:

            assert int(
                old[
                    key
                ]
            ) == int(
                new[
                    key
                ]
            ), (
                f"{dataset_name} {method} mismatch "
                f"for {key}"
            )


    print(
        f"{dataset_name} unchanged deterministic "
        "baseline fidelity: PASSED"
    )


deterministic_baseline_match(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_comparison_14b_df
)


deterministic_baseline_match(
    "CWQ",
    cwq_comparison_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 9. RANDOM-B MUST ALSO MATCH ORIGINAL CELL 14
# ======================================================================
#
# Random-B parameters and RNG are unchanged.
# ======================================================================

def random_b_match(
    dataset_name,
    old_df,
    new_df
):

    for seed in RANDOM_SEEDS_14B:

        old_rows = old_df[
            (
                old_df[
                    "method"
                ]
                ==
                "Random-B"
            )
            &
            (
                old_df[
                    "seed"
                ]
                ==
                seed
            )
        ]


        new_rows = new_df[
            (
                new_df[
                    "method"
                ]
                ==
                "Random-B"
            )
            &
            (
                new_df[
                    "seed"
                ]
                ==
                seed
            )
        ]


        assert len(
            old_rows
        ) == 1

        assert len(
            new_rows
        ) == 1


        old = old_rows.iloc[
            0
        ]

        new = new_rows.iloc[
            0
        ]


        for key in [
            "edges_examined",
            "reachable_questions",
            "active_prefixes",
            "candidate_branches",
        ]:

            assert int(
                old[
                    key
                ]
            ) == int(
                new[
                    key
                ]
            ), (
                f"{dataset_name} Random-B seed {seed} "
                f"mismatch for {key}"
            )


    print(
        f"{dataset_name} Random-B reproducibility: PASSED"
    )


random_b_match(
    "WebQSP",
    webqsp_comparison_df,
    webqsp_comparison_14b_df
)


random_b_match(
    "CWQ",
    cwq_comparison_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 10. NEW AFP MUST EXACTLY MATCH CELL 13B SELECTED POINT
# ======================================================================

def selected_13b_row(
    dataset_name,
    df
):

    params = (
        AFP_14B_SELECTED[
            dataset_name
        ]
    )


    rows = df[
        np.isclose(
            df[
                "T"
            ],
            params[
                "T"
            ]
        )
        &
        np.isclose(
            df[
                "gamma_min"
            ],
            params[
                "gamma_min"
            ]
        )
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def afp_13b_fidelity(
    dataset_name,
    comparison_df,
    boundary_df
):

    afp_rows = comparison_df[
        comparison_df[
            "method"
        ]
        ==
        "AFP"
    ]


    assert len(
        afp_rows
    ) == 1


    comparison = afp_rows.iloc[
        0
    ]


    boundary = (
        selected_13b_row(
            dataset_name,
            boundary_df
        )
    )


    assert int(
        comparison[
            "edges_examined"
        ]
    ) == int(
        boundary[
            "edges_examined"
        ]
    )


    assert int(
        comparison[
            "reachable_questions"
        ]
    ) == int(
        boundary[
            "reachable_questions"
        ]
    )


    assert np.isclose(
        float(
            comparison[
                "ssr"
            ]
        ),
        float(
            boundary[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            comparison[
                "answer_retention"
            ]
        ),
        float(
            boundary[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} AFP Cell-13B fidelity: PASSED"
    )


afp_13b_fidelity(
    "webqsp",
    webqsp_comparison_14b_df,
    webqsp_13b_df
)


afp_13b_fidelity(
    "cwq",
    cwq_comparison_14b_df,
    cwq_13b_df
)


# ======================================================================
# 11. SUMMARY TABLES
# ======================================================================

webqsp_summary_14b_df = (
    summarize_method_runs(
        webqsp_comparison_14b_df
    )
)


cwq_summary_14b_df = (
    summarize_method_runs(
        cwq_comparison_14b_df
    )
)


display_results(
    "webqsp",
    webqsp_comparison_14b_df,
    webqsp_summary_14b_df
)


display_results(
    "cwq",
    cwq_comparison_14b_df,
    cwq_summary_14b_df
)


# ======================================================================
# 12. UPDATED CAUSAL DIAGNOSTICS
# ======================================================================

print_causal_diagnostics(
    "webqsp",
    webqsp_summary_14b_df
)


print_causal_diagnostics(
    "cwq",
    cwq_summary_14b_df
)


# ======================================================================
# 13. EXTRA PARETO / DOMINANCE DIAGNOSTIC
# ======================================================================

def single_summary_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def dominance_report(
    dataset_name,
    summary_df
):

    afp = (
        single_summary_row(
            summary_df,
            "AFP"
        )
    )


    fixed_b = (
        single_summary_row(
            summary_df,
            "Fixed-Top-B"
        )
    )


    fixed_threshold = (
        single_summary_row(
            summary_df,
            "Fixed-Threshold"
        )
    )


    adaptive_random = (
        single_summary_row(
            summary_df,
            "Adaptive-Budget-Random"
        )
    )


    print(
        "\n"
        + "=" * 116
    )

    print(
        f"{dataset_name.upper()} DEVELOPMENT DOMINANCE DIAGNOSTIC"
    )

    print(
        "=" * 116
    )


    print(
        "\nAFP:"
    )

    print(
        f"  AR  = {afp['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {afp['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Top-B:"
    )

    print(
        f"  AR  = {fixed_b['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed_b['ssr_mean']:.6f}"
    )


    print(
        "\nFixed Threshold:"
    )

    print(
        f"  AR  = {fixed_threshold['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {fixed_threshold['ssr_mean']:.6f}"
    )


    print(
        "\nAdaptive-Budget Random:"
    )

    print(
        f"  AR  = {adaptive_random['ar_mean']:.6f}"
    )

    print(
        f"  SSR = {adaptive_random['ssr_mean']:.6f}"
    )


    # --------------------------------------------------------------
    # Simple deterministic dominance statement for Fixed Top-B vs AFP
    # --------------------------------------------------------------

    fixed_dominates_afp = (
        float(
            fixed_b[
                "ar_mean"
            ]
        )
        >=
        float(
            afp[
                "ar_mean"
            ]
        )
        and
        float(
            fixed_b[
                "ssr_mean"
            ]
        )
        >=
        float(
            afp[
                "ssr_mean"
            ]
        )
        and
        (
            float(
                fixed_b[
                    "ar_mean"
                ]
            )
            >
            float(
                afp[
                    "ar_mean"
                ]
            )
            or
            float(
                fixed_b[
                    "ssr_mean"
                ]
            )
            >
            float(
                afp[
                    "ssr_mean"
                ]
            )
        )
    )


    print(
        "\nFixed Top-B Pareto-dominates AFP:",
        bool(
            fixed_dominates_afp
        )
    )


    print(
        "\nAFP minus Adaptive-Random:"
    )

    print(
        "  AR delta:",
        f"{float(afp['ar_mean'] - adaptive_random['ar_mean']):.6f}"
    )

    print(
        "  SSR delta:",
        f"{float(afp['ssr_mean'] - adaptive_random['ssr_mean']):.6f}"
    )


dominance_report(
    "webqsp",
    webqsp_summary_14b_df
)


dominance_report(
    "cwq",
    cwq_summary_14b_df
)


# ======================================================================
# 14. EXPORT ARTIFACTS
# ======================================================================

WEBQSP_14B_RUN_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_controlled_runs.csv"
)


CWQ_14B_RUN_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_controlled_runs.csv"
)


WEBQSP_14B_SUMMARY_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_controlled_summary.csv"
)


CWQ_14B_SUMMARY_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_controlled_summary.csv"
)


WEBQSP_14B_Q_CSV = (
    COMPARE14B_DIR
    / "webqsp_expanded_afp_per_question.csv"
)


CWQ_14B_Q_CSV = (
    COMPARE14B_DIR
    / "cwq_expanded_afp_per_question.csv"
)


webqsp_comparison_14b_df.to_csv(
    WEBQSP_14B_RUN_CSV,
    index=False
)


cwq_comparison_14b_df.to_csv(
    CWQ_14B_RUN_CSV,
    index=False
)


webqsp_summary_14b_df.to_csv(
    WEBQSP_14B_SUMMARY_CSV,
    index=False
)


cwq_summary_14b_df.to_csv(
    CWQ_14B_SUMMARY_CSV,
    index=False
)


webqsp_question_14b_df.to_csv(
    WEBQSP_14B_Q_CSV,
    index=False
)


cwq_question_14b_df.to_csv(
    CWQ_14B_Q_CSV,
    index=False
)


# ======================================================================
# 15. CELL 14B MANIFEST
# ======================================================================

CELL14B_MANIFEST = {
    "cell":
        "RQ2_CELL14B_EXPANDED_AFP_CONTROLLED_COMPARISON",

    "version":
        "rq2_cell14b_expanded_afp_controlled_comparison_v1",

    "selector_source":
        "one_time_Cell13B_boundary_expansion",

    "afp_selected": {
        "webqsp":
            AFP_14B_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_14B_SELECTED[
                "cwq"
            ],
    },

    "fixed_top_b": {
        "webqsp":
            int(
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "webqsp"
                ]
            ),

        "cwq":
            int(
                FIXED_TOP_B_VALIDATION_SELECTED[
                    "cwq"
                ]
            ),
    },

    "fixed_threshold": {
        "webqsp":
            float(
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "webqsp"
                ]
            ),

        "cwq":
            float(
                FIXED_THRESHOLD_VALIDATION_SELECTED[
                    "cwq"
                ]
            ),
    },

    "random_seeds":
        RANDOM_SEEDS_14B,

    "adaptive_budget_random":
        {
            "budget_source":
                "new_AFP_actual_retained_count_after_tie_expansion",

            "branch_selection":
                "uniform_without_replacement",
        },

    "no_further_hyperparameter_search":
        True,

    "feature_version":
        "afp_features_v2_masked_entity_semantics",

    "selected_scorer_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "control_gates": {
        "rog_fidelity":
            True,

        "unchanged_fixed_baselines_match_Cell14":
            True,

        "random_b_matches_Cell14":
            True,

        "afp_matches_Cell13B":
            True,
    },

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,

    "next":
        "Cell15_ablations_and_final_development_decision",
}


CELL14B_MANIFEST_PATH = (
    COMPARE14B_DIR
    / "cell14b_expanded_afp_controlled_comparison_manifest.json"
)


with open(
    CELL14B_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL14B_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL14B_COMPLETE = True


# ======================================================================
# 16. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "=== RQ2 CELL 14B: "
    "EXPANDED-AFP CONTROLLED VALIDATION COMPARISON COMPLETE ==="
)

print(
    "=" * 120
)


print(
    "\nAFP development-selected candidates:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nControl status:"
)

print(
    "  RoG fidelity:                    PASSED"
)

print(
    "  Fixed baselines unchanged:       PASSED"
)

print(
    "  Random-B reproducibility:         PASSED"
)

print(
    "  AFP matches Cell 13B selection:   PASSED"
)

print(
    "  Further hyperparameter search:    NO"
)

print(
    "  TEST examples accessed:           NO"
)

print(
    "  Final AFP frozen:                 NO"
)


print(
    "\nNext:"
)

print(
    "Cell 15 — causal ablations + final development decision/freeze."
)


print(
    "\nManifest:"
)

print(
    CELL14B_MANIFEST_PATH
)

Cell 14B prerequisites: PASSED

Expanded AFP selected for Cell 14B:
  WebQSP: {'T': 0.05, 'gamma_min': 0.1}
  CWQ:    {'T': 0.1, 'gamma_min': 0.1}

Unchanged baselines:
  WebQSP Fixed Top-B: 1
  WebQSP threshold: 0.3
  CWQ Fixed Top-B: 8
  CWQ threshold: 0.1

RUNNING CELL 14B CONTROLLED VALIDATION COMPARISON


webqsp controlled comparison:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP controlled comparison completed.
Elapsed: 0.04 min


cwq controlled comparison:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ controlled comparison completed.
Elapsed: 2.21 min
WebQSP RoG control fidelity: PASSED
CWQ RoG control fidelity: PASSED

Cell 14B RoG control fidelity: PASSED
WebQSP unchanged deterministic baseline fidelity: PASSED
CWQ unchanged deterministic baseline fidelity: PASSED
WebQSP Random-B reproducibility: PASSED
CWQ Random-B reproducibility: PASSED
webqsp AFP Cell-13B fidelity: PASSED
cwq AFP Cell-13B fidelity: PASSED

WEBQSP CONTROLLED VALIDATION COMPARISON — RUN LEVEL
                method      seed  edges_examined      ssr  reachable_questions  answer_retention  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  fully_tied_decisions  peak_frontier
                   RoG       NaN          341526 0.000000                  205          1.000000             2440                7983              0                   NaN                     0            533
           Fixed-Top-B       NaN          335061 0.018930                  205          1.000000            

##  Matched retraining feature-group ablations, selector/component ablations and final development freeze

In [32]:
# ======================================================================
# MATCHED FEATURE-FAMILY RETRAINING ABLATIONS
# ======================================================================
# PURPOSE
# -------
# Proper feature-family causal ablation by RETRAINING the scorer after
# removing one feature family.
#
# VARIANTS
# --------
# 1. Full Feature-v2 retrain control
# 2. -Semantic
# 3. -Path
# 4. -Structural
# 5. -Progress
#
# METHODOLOGICAL CONTROLS
# -----------------------
# - exact frozen train Feature-v2 matrices
# - same feasible-decision training population
# - same selected hidden dimension
# - same branch BCE
# - same optimizer
# - same LR / weight decay / epochs
# - same seed = 42
# - no class weighting
# - no early stopping
# - no ablation-specific hyperparameter tuning
# - same already-selected AFP selector parameters
#
# IMPORTANT SOFTWARE FIX
# ----------------------
# Persisted AFPScorer and AFPFeatureStandardizer were originally defined
# with global AFP_INPUT_DIM = 27.
#
# Feature-family ablations have dimensions 19 / 20 / 23.
#
# We DO NOT rewrite either algorithm.
#
# Instead, the EXACT persisted class source is instantiated in an
# isolated namespace with AFP_INPUT_DIM equal to the ablation dimension.
#
# Full 27-D behavior is software-gated against the frozen standardizer.
#
# VALIDATION ONLY.
# NO TEST access.
# NO final AFP freeze in this cell.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import math
import re
import time
import typing
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    "CELL14B_COMPLETE",
    "CELL13B_COMPLETE",

    "AFP_14B_SELECTED",

    "AFP_RUNTIME_FEATURE_EXTRACTOR",
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    "webqsp_comparison_14b_df",
    "cwq_comparison_14b_df",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "get_cached_relation_expansion",
    "frontier_cache_key",
    "select_policy_indices",
    "final_prefixes_reach_answer",

    "webqsp_ckpt_obj",
    "cwq_ckpt_obj",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required prior-cell objects:\n  "
    + "\n  ".join(missing)
)

assert CELL14B_COMPLETE is True
assert CELL13B_COMPLETE is True
assert int(AFP_RUNTIME_FEATURE_DIM) == 27

print("Cell 15A prerequisites: PASSED")
print("Feature version:", AFP_RUNTIME_FEATURE_VERSION)
print("Feature dimension:", AFP_RUNTIME_FEATURE_DIM)


# ======================================================================
# 2. PATHS
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_ROOT = (
    ROOT
    / "03_features"
)

ABLATION_DIR = (
    ROOT
    / "13_feature_ablations"
)

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert FEATURE_ROOT.exists()
assert NOTEBOOK_PATH.exists()

print("Feature root:", FEATURE_ROOT)
print("Ablation output:", ABLATION_DIR)


# ======================================================================
# 3. VERIFIED FROZEN TRAIN REFERENCES
# ======================================================================

EXPECTED_TRAIN = {
    "webqsp": {
        "branches": 18437,
        "groups": 1457,
        "positive": 7301,
        "negative": 11136,
    },

    "cwq": {
        "branches": 218544,
        "groups": 15937,
        "positive": 52746,
        "negative": 165798,
    },
}


# ======================================================================
# 4. SHA256 HELPER
# ======================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================
# 5. DISCOVER EXACT FROZEN TRAIN NPZs
# ======================================================================

def inspect_npz_candidate(path, expected):

    try:
        data = np.load(
            path,
            allow_pickle=False
        )

    except Exception:
        return None

    try:

        files = set(data.files)

        if not {
            "X",
            "y",
            "group_ptr",
        }.issubset(files):
            return None

        X = data["X"]
        y = data["y"]
        group_ptr = data["group_ptr"]

        if tuple(X.shape) != (
            expected["branches"],
            27
        ):
            return None

        if tuple(y.shape) != (
            expected["branches"],
        ):
            return None

        if len(group_ptr) != (
            expected["groups"] + 1
        ):
            return None

        y_int = np.asarray(
            y,
            dtype=np.int64
        )

        positive = int(
            np.sum(y_int == 1)
        )

        negative = int(
            np.sum(y_int == 0)
        )

        if positive != expected["positive"]:
            return None

        if negative != expected["negative"]:
            return None

        return {
            "path": path,
            "shape": tuple(X.shape),
            "groups": int(len(group_ptr) - 1),
            "positive": positive,
            "negative": negative,
            "sha256": sha256_file(path),
        }

    finally:
        data.close()


def discover_training_npz(dataset_name):

    expected = EXPECTED_TRAIN[
        dataset_name
    ]

    candidates = []

    for path in FEATURE_ROOT.rglob(
        "*.npz"
    ):

        result = inspect_npz_candidate(
            path,
            expected
        )

        if result is not None:
            candidates.append(result)

    assert len(candidates) >= 1, (
        f"{dataset_name}: no train NPZ matched "
        "the frozen training reference."
    )

    print(
        f"\n{dataset_name.upper()} "
        "matching train NPZ candidate(s):"
    )

    for row in candidates:

        print(" ", row["path"])
        print("    SHA:", row["sha256"])

    if len(candidates) == 1:
        return candidates[0]["path"]

    hashes = {
        row["sha256"]
        for row in candidates
    }

    if len(hashes) == 1:

        chosen = sorted(
            [
                row["path"]
                for row in candidates
            ],
            key=lambda p: str(p)
        )[0]

        print(
            f"{dataset_name}: byte-identical "
            f"duplicate copies; using {chosen}"
        )

        return chosen

    # Scientific-array identity gate.
    reference_path = candidates[0]["path"]

    with np.load(
        reference_path,
        allow_pickle=False
    ) as ref:

        ref_X = np.asarray(ref["X"])
        ref_y = np.asarray(ref["y"])
        ref_ptr = np.asarray(ref["group_ptr"])

    for row in candidates[1:]:

        with np.load(
            row["path"],
            allow_pickle=False
        ) as other:

            assert np.array_equal(
                ref_X,
                other["X"]
            )

            assert np.array_equal(
                ref_y,
                other["y"]
            )

            assert np.array_equal(
                ref_ptr,
                other["group_ptr"]
            )

    chosen = sorted(
        [
            row["path"]
            for row in candidates
        ],
        key=lambda p: str(p)
    )[0]

    print(
        f"{dataset_name}: scientifically identical "
        f"copies; using {chosen}"
    )

    return chosen


WEBQSP_TRAIN_NPZ = (
    discover_training_npz(
        "webqsp"
    )
)

CWQ_TRAIN_NPZ = (
    discover_training_npz(
        "cwq"
    )
)

print(
    "\nFrozen train feature files identified: PASSED"
)


# ======================================================================
# 6. LOAD FROZEN TRAIN MATRICES
# ======================================================================

def load_train_matrix(
    path,
    dataset_name
):

    expected = EXPECTED_TRAIN[
        dataset_name
    ]

    with np.load(
        path,
        allow_pickle=False
    ) as data:

        X = np.asarray(
            data["X"],
            dtype=np.float32
        )

        y = np.asarray(
            data["y"],
            dtype=np.float32
        )

        group_ptr = np.asarray(
            data["group_ptr"],
            dtype=np.int64
        )

    assert X.shape == (
        expected["branches"],
        27
    )

    assert y.shape == (
        expected["branches"],
    )

    assert len(group_ptr) == (
        expected["groups"] + 1
    )

    assert int(
        np.sum(y == 1)
    ) == expected["positive"]

    assert int(
        np.sum(y == 0)
    ) == expected["negative"]

    assert np.all(
        np.isfinite(X)
    )

    return {
        "X": X,
        "y": y,
        "group_ptr": group_ptr,
    }


webqsp_train_ablation = (
    load_train_matrix(
        WEBQSP_TRAIN_NPZ,
        "webqsp"
    )
)

cwq_train_ablation = (
    load_train_matrix(
        CWQ_TRAIN_NPZ,
        "cwq"
    )
)


print(
    "\nTraining matrices:"
)

print(
    "  WebQSP:",
    webqsp_train_ablation["X"].shape,
    "| groups:",
    len(
        webqsp_train_ablation["group_ptr"]
    ) - 1
)

print(
    "  CWQ:   ",
    cwq_train_ablation["X"].shape,
    "| groups:",
    len(
        cwq_train_ablation["group_ptr"]
    ) - 1
)

print(
    "Frozen training-cache gate: PASSED"
)


# ======================================================================
# 7. FEATURE-FAMILY DEFINITIONS
# ======================================================================

FEATURE_GROUPS = {
    "semantic": list(range(0, 8)),
    "path": list(range(8, 15)),
    "structural": list(range(15, 23)),
    "progress": list(range(23, 27)),
}

all_feature_columns = (
    FEATURE_GROUPS["semantic"]
    + FEATURE_GROUPS["path"]
    + FEATURE_GROUPS["structural"]
    + FEATURE_GROUPS["progress"]
)

assert sorted(
    all_feature_columns
) == list(range(27))

assert len(
    set(all_feature_columns)
) == 27


ABLATION_VARIANTS = {
    "full_v2_retrain":
        list(range(27)),

    "minus_semantic":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["semantic"]
        ],

    "minus_path":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["path"]
        ],

    "minus_structural":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["structural"]
        ],

    "minus_progress":
        [
            i
            for i in range(27)
            if i not in FEATURE_GROUPS["progress"]
        ],
}


EXPECTED_DIMS = {
    "full_v2_retrain": 27,
    "minus_semantic": 19,
    "minus_path": 20,
    "minus_structural": 19,
    "minus_progress": 23,
}


for name, columns in (
    ABLATION_VARIANTS.items()
):

    assert len(columns) == EXPECTED_DIMS[name]


print(
    "\nFeature ablation variants:"
)

for name, columns in (
    ABLATION_VARIANTS.items()
):

    print(
        f"  {name:<20} "
        f"{len(columns):>2} features"
    )


# ======================================================================
# 8. LOAD PERSISTED NOTEBOOK
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook_15a = json.load(f)


# ======================================================================
# 9. RECOVER EXACT PERSISTED CLASS SOURCES
# ======================================================================

def recover_latest_class_source(
    notebook,
    class_name
):

    matches = []

    for cell_idx, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get(
            "cell_type"
        ) != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if (
            f"class {class_name}"
            not in source
        ):
            continue

        try:
            tree = ast.parse(source)

        except Exception:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.ClassDef)
                and
                node.name == class_name
            ):

                class_source = (
                    ast.get_source_segment(
                        source,
                        node
                    )
                )

                if class_source is not None:

                    matches.append(
                        {
                            "cell_idx":
                                int(cell_idx),

                            "class_source":
                                class_source,

                            "cell_source":
                                source,
                        }
                    )

    assert len(matches) >= 1, (
        f"Could not recover {class_name} "
        "from persisted notebook."
    )

    # Latest persisted definition.
    return matches[-1]


scorer_recovery = (
    recover_latest_class_source(
        notebook_15a,
        "AFPScorer"
    )
)

standardizer_recovery = (
    recover_latest_class_source(
        notebook_15a,
        "AFPFeatureStandardizer"
    )
)


SCORER_SOURCE_15A = (
    scorer_recovery[
        "class_source"
    ]
)

STANDARDIZER_SOURCE_15A = (
    standardizer_recovery[
        "class_source"
    ]
)


print(
    "\nRecovered persisted classes:"
)

print(
    "  AFPScorer: cell",
    scorer_recovery[
        "cell_idx"
    ]
)

print(
    "  AFPFeatureStandardizer: cell",
    standardizer_recovery[
        "cell_idx"
    ]
)


# ======================================================================
# 10. RECOVER LITERAL AFP_* DEPENDENCIES FROM CLASS CELLS
# ======================================================================

def recover_literal_afp_constants(
    source
):

    recovered = {}

    try:
        tree = ast.parse(source)

    except Exception:
        return recovered

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            targets = []

            if isinstance(
                node,
                ast.Assign
            ):
                targets = node.targets
                value_node = node.value

            else:
                targets = [node.target]
                value_node = node.value

            if value_node is None:
                continue

            for target in targets:

                if not isinstance(
                    target,
                    ast.Name
                ):
                    continue

                name = target.id

                if not name.startswith(
                    "AFP_"
                ):
                    continue

                try:
                    value = ast.literal_eval(
                        value_node
                    )

                except Exception:
                    continue

                recovered[name] = value

    return recovered


PERSISTED_AFP_CONSTANTS = {}

PERSISTED_AFP_CONSTANTS.update(
    recover_literal_afp_constants(
        scorer_recovery[
            "cell_source"
        ]
    )
)

PERSISTED_AFP_CONSTANTS.update(
    recover_literal_afp_constants(
        standardizer_recovery[
            "cell_source"
        ]
    )
)


print(
    "\nRecovered persisted class constants:"
)

for key in sorted(
    PERSISTED_AFP_CONSTANTS
):

    print(
        f"  {key} = "
        f"{PERSISTED_AFP_CONSTANTS[key]!r}"
    )


# ======================================================================
# 11. DIMENSION-AWARE EXACT CLASS FACTORIES
# ======================================================================
#
# The original classes were created with AFP_INPUT_DIM = 27.
#
# For ablation dimensions we recreate the EXACT source in an isolated
# globals namespace and override ONLY AFP_INPUT_DIM.
# ======================================================================

_DIMENSIONAL_STANDARDIZER_CLASSES = {}
_DIMENSIONAL_SCORER_CLASSES = {}


def build_exact_class_namespace(
    input_dim
):

    input_dim = int(input_dim)

    # Copy current globals to preserve harmless imported dependencies,
    # then inject persisted literals and the dimension-specific invariant.
    ns = dict(globals())

    ns.update(
        {
            "np": np,
            "torch": torch,
            "nn": nn,
            "F": F,
            "Path": Path,
            "typing": typing,
        }
    )

    ns.update(
        PERSISTED_AFP_CONSTANTS
    )

    # Critical dimension specialization.
    ns[
        "AFP_INPUT_DIM"
    ] = input_dim

    return ns


def get_exact_standardizer_class_for_dim(
    input_dim
):

    input_dim = int(input_dim)

    if input_dim in (
        _DIMENSIONAL_STANDARDIZER_CLASSES
    ):

        return (
            _DIMENSIONAL_STANDARDIZER_CLASSES[
                input_dim
            ]
        )

    ns = build_exact_class_namespace(
        input_dim
    )

    exec(
        STANDARDIZER_SOURCE_15A,
        ns
    )

    cls = ns[
        "AFPFeatureStandardizer"
    ]

    _DIMENSIONAL_STANDARDIZER_CLASSES[
        input_dim
    ] = cls

    return cls


def get_exact_scorer_class_for_dim(
    input_dim
):

    input_dim = int(input_dim)

    if input_dim in (
        _DIMENSIONAL_SCORER_CLASSES
    ):

        return (
            _DIMENSIONAL_SCORER_CLASSES[
                input_dim
            ]
        )

    ns = build_exact_class_namespace(
        input_dim
    )

    exec(
        SCORER_SOURCE_15A,
        ns
    )

    cls = ns[
        "AFPScorer"
    ]

    _DIMENSIONAL_SCORER_CLASSES[
        input_dim
    ] = cls

    return cls


ABLATION_INPUT_DIMS = sorted(
    set(
        EXPECTED_DIMS.values()
    )
)


assert ABLATION_INPUT_DIMS == [
    19,
    20,
    23,
    27,
]


print(
    "\nPreparing exact dimension-aware classes:"
)


for dim in ABLATION_INPUT_DIMS:

    std_cls = (
        get_exact_standardizer_class_for_dim(
            dim
        )
    )

    scorer_cls = (
        get_exact_scorer_class_for_dim(
            dim
        )
    )

    print(
        f"  {dim:>2}-D -> "
        f"{std_cls.__name__}, "
        f"{scorer_cls.__name__}"
    )


print(
    "Dimension-aware exact classes: READY"
)


# ======================================================================
# 12. DIMENSION-AWARE SOFTWARE SANITY GATE
# ======================================================================

for dim in ABLATION_INPUT_DIMS:

    rng = np.random.default_rng(
        15000 + dim
    )

    X_sanity = rng.normal(
        size=(
            8,
            dim
        )
    ).astype(
        np.float32
    )

    StdClass = (
        get_exact_standardizer_class_for_dim(
            dim
        )
    )

    std = StdClass()

    std.fit(
        X_sanity
    )

    Z_sanity = np.asarray(
        std.transform(
            X_sanity
        ),
        dtype=np.float32
    )

    assert Z_sanity.shape == (
        8,
        dim
    )

    assert np.all(
        np.isfinite(
            Z_sanity
        )
    )

    ScorerClass = (
        get_exact_scorer_class_for_dim(
            dim
        )
    )

    # The actual hidden dimension is dataset-specific later.
    scorer = ScorerClass(
        input_dim=dim,
        hidden_dim=32,
        dropout=0.0
    )

    with torch.inference_mode():

        out = scorer(
            torch.from_numpy(
                Z_sanity
            )
        )

    assert out.reshape(
        -1
    ).shape == (
        8,
    )


print(
    "Dimension-aware scorer/standardizer "
    "sanity gate: PASSED"
)


# ======================================================================
# 13. STANDARDIZER HELPERS
# ======================================================================

def fit_exact_standardizer(
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    assert X.ndim == 2

    input_dim = int(
        X.shape[1]
    )

    StandardizerClass = (
        get_exact_standardizer_class_for_dim(
            input_dim
        )
    )

    standardizer = (
        StandardizerClass()
    )

    assert hasattr(
        standardizer,
        "fit"
    )

    assert hasattr(
        standardizer,
        "transform"
    )

    standardizer.fit(
        X
    )

    Z = np.asarray(
        standardizer.transform(
            X
        ),
        dtype=np.float32
    )

    assert Z.shape == X.shape

    assert np.all(
        np.isfinite(Z)
    )

    return (
        standardizer,
        Z
    )


def transform_exact_standardizer(
    standardizer,
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    Z = np.asarray(
        standardizer.transform(
            X
        ),
        dtype=np.float32
    )

    assert Z.shape == X.shape

    assert np.all(
        np.isfinite(Z)
    )

    return Z


# ======================================================================
# 14. STANDARDIZER STATE EXTRACTION
# ======================================================================

def extract_standardizer_arrays(
    standardizer,
    expected_dim
):

    candidate_dicts = []

    if hasattr(
        standardizer,
        "state_dict"
    ):

        try:

            state = (
                standardizer.state_dict()
            )

            if isinstance(
                state,
                dict
            ):
                candidate_dicts.append(
                    state
                )

        except Exception:
            pass

    if hasattr(
        standardizer,
        "__dict__"
    ):

        candidate_dicts.append(
            vars(
                standardizer
            )
        )

    mean = None
    std = None

    for mapping in candidate_dicts:

        for key, value in (
            mapping.items()
        ):

            key_lower = str(
                key
            ).lower()

            try:

                if torch.is_tensor(
                    value
                ):

                    arr = (
                        value
                        .detach()
                        .cpu()
                        .numpy()
                    )

                else:

                    arr = np.asarray(
                        value
                    )

            except Exception:
                continue

            if arr.shape != (
                expected_dim,
            ):
                continue

            if "mean" in key_lower:

                mean = np.asarray(
                    arr,
                    dtype=np.float32
                )

            if (
                "std" in key_lower
                or
                "scale" in key_lower
            ):

                std = np.asarray(
                    arr,
                    dtype=np.float32
                )

    assert mean is not None, (
        "Could not recover fitted standardizer mean."
    )

    assert std is not None, (
        "Could not recover fitted standardizer std."
    )

    assert mean.shape == (
        expected_dim,
    )

    assert std.shape == (
        expected_dim,
    )

    return {
        "mean": mean,
        "std": std,
    }


def checkpoint_standardizer_arrays(
    checkpoint,
    expected_dim
):

    assert (
        "standardizer_state"
        in checkpoint
    )

    state = checkpoint[
        "standardizer_state"
    ]

    assert isinstance(
        state,
        dict
    )

    mean = None
    std = None

    for key, value in (
        state.items()
    ):

        key_lower = str(
            key
        ).lower()

        if torch.is_tensor(
            value
        ):

            arr = (
                value
                .detach()
                .cpu()
                .numpy()
            )

        else:

            arr = np.asarray(
                value
            )

        if arr.shape != (
            expected_dim,
        ):
            continue

        if "mean" in key_lower:

            mean = np.asarray(
                arr,
                dtype=np.float32
            )

        if (
            "std" in key_lower
            or
            "scale" in key_lower
        ):

            std = np.asarray(
                arr,
                dtype=np.float32
            )

    assert mean is not None
    assert std is not None

    return {
        "mean": mean,
        "std": std,
    }


# ======================================================================
# 15. 27-D STANDARDIZER SOFTWARE FIDELITY
# ======================================================================

for (
    dataset_name,
    train_data,
    checkpoint
) in [
    (
        "WebQSP",
        webqsp_train_ablation,
        webqsp_ckpt_obj
    ),
    (
        "CWQ",
        cwq_train_ablation,
        cwq_ckpt_obj
    ),
]:

    standardizer, _ = (
        fit_exact_standardizer(
            train_data["X"]
        )
    )

    fitted = (
        extract_standardizer_arrays(
            standardizer,
            27
        )
    )

    frozen = (
        checkpoint_standardizer_arrays(
            checkpoint,
            27
        )
    )

    mean_diff = float(
        np.max(
            np.abs(
                fitted["mean"].astype(
                    np.float64
                )
                -
                frozen["mean"].astype(
                    np.float64
                )
            )
        )
    )

    std_diff = float(
        np.max(
            np.abs(
                fitted["std"].astype(
                    np.float64
                )
                -
                frozen["std"].astype(
                    np.float64
                )
            )
        )
    )

    print(
        f"\n{dataset_name} full-v2 "
        "standardizer fidelity"
    )

    print(
        "  max mean diff:",
        mean_diff
    )

    print(
        "  max std diff: ",
        std_diff
    )

    assert mean_diff <= 1e-7
    assert std_diff <= 1e-7


print(
    "\nTrain-only standardizer "
    "reproduction: PASSED"
)


# ======================================================================
# 16. RECOVER SELECTED-CHECKPOINT TRAINING METADATA
# ======================================================================

def checkpoint_training_metadata(
    checkpoint,
    dataset_name
):

    required_keys = [
        "epochs",
        "learning_rate",
        "weight_decay",
        "loss_name",
        "seed",
        "hidden_dim",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in checkpoint
    ]

    assert not missing_keys, (
        f"{dataset_name}: checkpoint missing "
        f"{missing_keys}"
    )

    return {
        "epochs":
            int(
                checkpoint["epochs"]
            ),

        "learning_rate":
            float(
                checkpoint[
                    "learning_rate"
                ]
            ),

        "weight_decay":
            float(
                checkpoint[
                    "weight_decay"
                ]
            ),

        "loss_name":
            str(
                checkpoint[
                    "loss_name"
                ]
            ),

        "seed":
            int(
                checkpoint["seed"]
            ),

        "hidden_dim":
            int(
                checkpoint[
                    "hidden_dim"
                ]
            ),
    }


WEBQSP_TRAIN_META = (
    checkpoint_training_metadata(
        webqsp_ckpt_obj,
        "WebQSP"
    )
)

CWQ_TRAIN_META = (
    checkpoint_training_metadata(
        cwq_ckpt_obj,
        "CWQ"
    )
)


print(
    "\nRecovered selected-checkpoint "
    "training metadata:"
)

print(
    "  WebQSP:",
    WEBQSP_TRAIN_META
)

print(
    "  CWQ:   ",
    CWQ_TRAIN_META
)


for (
    dataset_name,
    meta,
    expected_H
) in [
    (
        "WebQSP",
        WEBQSP_TRAIN_META,
        32
    ),
    (
        "CWQ",
        CWQ_TRAIN_META,
        64
    ),
]:

    assert meta[
        "epochs"
    ] == 80

    assert np.isclose(
        meta[
            "learning_rate"
        ],
        1e-3
    )

    assert np.isclose(
        meta[
            "weight_decay"
        ],
        1e-4
    )

    assert meta[
        "loss_name"
    ] == "branch_bce"

    assert meta[
        "seed"
    ] == 42

    assert meta[
        "hidden_dim"
    ] == expected_H


print(
    "Frozen training-metadata gate: PASSED"
)


# ======================================================================
# 17. RECOVER ORIGINAL OPTIMIZER FROM NOTEBOOK
# ======================================================================

optimizer_hits = []


for cell_idx, cell in enumerate(
    notebook_15a["cells"]
):

    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    if "torch.optim." not in source:
        continue

    training_context = (
        "branch_bce" in source
        or
        "AFPScorer" in source
        or
        "loss_name" in source
    )

    if not training_context:
        continue

    names = re.findall(
        r"torch\.optim\.(AdamW|Adam)\s*\(",
        source
    )

    for name in names:

        optimizer_hits.append(
            {
                "cell_idx":
                    int(cell_idx),

                "optimizer":
                    name,
            }
        )


print(
    "\nPersisted AFP-training optimizer evidence:"
)

for row in optimizer_hits:

    print(
        "  cell",
        row["cell_idx"],
        "->",
        row["optimizer"]
    )


optimizer_names = {
    row["optimizer"]
    for row in optimizer_hits
}


assert len(
    optimizer_names
) == 1, (
    "Could not uniquely recover original AFP optimizer. "
    "Do NOT guess."
)


AFP_ABLATION_OPTIMIZER_NAME = (
    next(
        iter(optimizer_names)
    )
)


if AFP_ABLATION_OPTIMIZER_NAME == "AdamW":

    AFP_ABLATION_OPTIMIZER_CLASS = (
        torch.optim.AdamW
    )

elif AFP_ABLATION_OPTIMIZER_NAME == "Adam":

    AFP_ABLATION_OPTIMIZER_CLASS = (
        torch.optim.Adam
    )

else:

    raise AssertionError(
        "Unsupported recovered optimizer."
    )


print(
    "Recovered optimizer:",
    AFP_ABLATION_OPTIMIZER_NAME
)


# ======================================================================
# 18. SCHEDULER GATE
# ======================================================================

optimizer_cells = {
    row["cell_idx"]
    for row in optimizer_hits
}

scheduler_evidence = []

for cell_idx in sorted(
    optimizer_cells
):

    source = "".join(
        notebook_15a[
            "cells"
        ][
            cell_idx
        ].get(
            "source",
            []
        )
    )

    if (
        "lr_scheduler" in source
        or
        "scheduler.step" in source
    ):

        scheduler_evidence.append(
            cell_idx
        )


assert not scheduler_evidence, (
    "Scheduler detected in recovered AFP "
    "training cell. Stop rather than omit it."
)

print(
    "Learning-rate scheduler in AFP training: NONE"
)

print(
    "Original optimizer recovery: PASSED"
)


# ======================================================================
# 19. TRAINING DEVICE
# ======================================================================

ABLATION_DEVICE = torch.device(
    "cpu"
)

print(
    "Matched ablation training device:",
    ABLATION_DEVICE
)


# ======================================================================
# 20. DETERMINISTIC SEED HELPER
# ======================================================================

def set_training_seed(seed):

    seed = int(seed)

    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


# ======================================================================
# 21. MATCHED RETRAINING FUNCTION
# ======================================================================

def train_ablation_model(
    dataset_name,
    X_full,
    y,
    columns,
    meta,
    variant_name
):

    columns = list(columns)

    X = np.asarray(
        X_full[
            :,
            columns
        ],
        dtype=np.float32
    )

    y_np = np.asarray(
        y,
        dtype=np.float32
    )

    input_dim = int(
        X.shape[1]
    )

    assert input_dim == len(
        columns
    )

    # --------------------------------------------------------------
    # Fresh TRAIN-only standardizer.
    # --------------------------------------------------------------

    standardizer, Z = (
        fit_exact_standardizer(
            X
        )
    )

    # --------------------------------------------------------------
    # Same initialization seed for every matched variant.
    # --------------------------------------------------------------

    set_training_seed(
        meta["seed"]
    )

    ScorerClass = (
        get_exact_scorer_class_for_dim(
            input_dim
        )
    )

    model = ScorerClass(
        input_dim=input_dim,
        hidden_dim=int(
            meta["hidden_dim"]
        ),
        dropout=0.0
    ).to(
        ABLATION_DEVICE
    )

    model.train()

    optimizer = (
        AFP_ABLATION_OPTIMIZER_CLASS(
            model.parameters(),
            lr=float(
                meta["learning_rate"]
            ),
            weight_decay=float(
                meta["weight_decay"]
            )
        )
    )

    criterion = (
        nn.BCEWithLogitsLoss()
    )

    X_tensor = (
        torch.from_numpy(Z)
        .to(ABLATION_DEVICE)
    )

    y_tensor = (
        torch.from_numpy(y_np)
        .to(ABLATION_DEVICE)
    )

    losses = []

    for epoch in range(
        int(meta["epochs"])
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(
                X_tensor
            )
            .reshape(-1)
        )

        assert logits.shape == (
            len(y_np),
        )

        loss = criterion(
            logits,
            y_tensor
        )

        loss.backward()

        optimizer.step()

        losses.append(
            float(
                loss
                .detach()
                .cpu()
                .item()
            )
        )

    model.eval()

    standardizer_state = (
        extract_standardizer_arrays(
            standardizer,
            input_dim
        )
    )

    print(
        f"{dataset_name:<7} "
        f"{variant_name:<20} "
        f"dim={input_dim:<2} "
        f"loss0={losses[0]:.6f} "
        f"loss80={losses[-1]:.6f}"
    )

    return {
        "model":
            model,

        "standardizer":
            standardizer,

        "standardizer_state":
            standardizer_state,

        "columns":
            columns,

        "input_dim":
            input_dim,

        "loss_history":
            losses,
    }


# ======================================================================
# 22. TRAIN ALL MATCHED VARIANTS
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "TRAINING MATCHED FEATURE ABLATIONS"
)

print(
    "=" * 116
)


ABLATION_MODELS = {
    "webqsp": {},
    "cwq": {},
}


training_start = time.time()


for (
    dataset_name,
    train_data,
    meta
) in [
    (
        "webqsp",
        webqsp_train_ablation,
        WEBQSP_TRAIN_META
    ),
    (
        "cwq",
        cwq_train_ablation,
        CWQ_TRAIN_META
    ),
]:

    print(
        f"\n{dataset_name.upper()}"
    )

    for variant_name, columns in (
        ABLATION_VARIANTS.items()
    ):

        result = (
            train_ablation_model(
                dataset_name=
                    dataset_name,

                X_full=
                    train_data["X"],

                y=
                    train_data["y"],

                columns=
                    columns,

                meta=
                    meta,

                variant_name=
                    variant_name
            )
        )

        ABLATION_MODELS[
            dataset_name
        ][
            variant_name
        ] = result


print(
    "\nMatched ablation training elapsed:",
    f"{(time.time()-training_start)/60:.2f} min"
)


# ======================================================================
# 23. SAVE ABLATION CHECKPOINTS
# ======================================================================

ABLATION_CHECKPOINT_PATHS = {
    "webqsp": {},
    "cwq": {},
}


for dataset_name in [
    "webqsp",
    "cwq",
]:

    dataset_dir = (
        ABLATION_DIR
        / dataset_name
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    meta = (
        WEBQSP_TRAIN_META
        if dataset_name == "webqsp"
        else
        CWQ_TRAIN_META
    )

    for variant_name, bundle in (
        ABLATION_MODELS[
            dataset_name
        ].items()
    ):

        path = (
            dataset_dir
            / f"{variant_name}.pt"
        )

        torch.save(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant_name,

                "feature_version":
                    AFP_RUNTIME_FEATURE_VERSION,

                "feature_columns":
                    bundle["columns"],

                "input_dim":
                    bundle["input_dim"],

                "hidden_dim":
                    meta["hidden_dim"],

                "seed":
                    meta["seed"],

                "epochs":
                    meta["epochs"],

                "learning_rate":
                    meta["learning_rate"],

                "weight_decay":
                    meta["weight_decay"],

                "loss_name":
                    "branch_bce",

                "optimizer":
                    AFP_ABLATION_OPTIMIZER_NAME,

                "model_state_dict":
                    bundle[
                        "model"
                    ].state_dict(),

                "standardizer_state":
                    bundle[
                        "standardizer_state"
                    ],

                "final_training_loss":
                    bundle[
                        "loss_history"
                    ][-1],

                "ablation_specific_tuning":
                    False,

                "test_examples_accessed":
                    False,
            },
            path
        )

        ABLATION_CHECKPOINT_PATHS[
            dataset_name
        ][
            variant_name
        ] = path


print(
    "\nAblation checkpoints saved: PASSED"
)


# ======================================================================
# 24. RUNTIME FEATURE EXTRACTOR SIGNATURE
# ======================================================================

RUNTIME_EXTRACTOR_SIGNATURE = (
    inspect.signature(
        AFP_RUNTIME_FEATURE_EXTRACTOR
    )
)


print(
    "\nRuntime Feature-v2 extractor signature:"
)

print(
    " ",
    RUNTIME_EXTRACTOR_SIGNATURE
)


# ======================================================================
# 25. SAFE ADAPTER TO EXACT RUNTIME FEATURE EXTRACTOR
# ======================================================================
#
# We do not guess the exact persisted function signature.
#
# Only arguments explicitly accepted by the recovered callable are
# supplied. Unknown REQUIRED arguments trigger a hard failure.
# ======================================================================

def call_runtime_feature_extractor(
    dataset_name,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    available = {
        "dataset_name":
            dataset_name,

        "dataset":
            dataset_name,

        "question_id":
            question_id,

        "qid":
            question_id,

        "question":
            question,

        "plan":
            list(plan),

        "hop":
            int(hop),

        "candidate_rows":
            candidate_rows,

        "semantic_encoder":
            AFP_RUNTIME_SEMANTIC_ENCODER,

        "entity_name_map":
            None,
    }

    sig = inspect.signature(
        AFP_RUNTIME_FEATURE_EXTRACTOR
    )

    kwargs = {}

    has_var_keyword = False

    for name, parameter in (
        sig.parameters.items()
    ):

        if parameter.kind == (
            inspect.Parameter.VAR_KEYWORD
        ):

            has_var_keyword = True
            continue

        if parameter.kind == (
            inspect.Parameter.VAR_POSITIONAL
        ):
            continue

        if name in available:

            kwargs[name] = (
                available[name]
            )

            continue

        if parameter.default is (
            inspect.Parameter.empty
        ):

            raise AssertionError(
                "Unknown required runtime feature "
                f"extractor argument: {name}. "
                "Do not guess the mapping."
            )

    # For **kwargs-style wrapper, supplying our normal names is safe.
    if has_var_keyword:

        for key, value in available.items():

            if key not in kwargs:

                # Avoid supplying aliases simultaneously.
                if key in {
                    "dataset",
                    "qid",
                }:
                    continue

                kwargs[key] = value

    X = (
        AFP_RUNTIME_FEATURE_EXTRACTOR(
            **kwargs
        )
    )

    return np.asarray(
        X,
        dtype=np.float32
    )


# ======================================================================
# 26. FULL FEATURE CACHE FOR DYNAMIC VALIDATION GROUPS
# ======================================================================

def get_full_runtime_group_features(
    dataset_name,
    question_id,
    question,
    plan_index,
    plan,
    hop,
    active_prefixes,
    candidate_rows,
    feature_cache
):

    key = frontier_cache_key(
        plan_index=
            plan_index,

        hop=
            hop,

        active_prefixes=
            active_prefixes
    )

    if key in feature_cache:

        cached = feature_cache[
            key
        ]

        assert cached.shape == (
            len(candidate_rows),
            27
        )

        return cached

    X = (
        call_runtime_feature_extractor(
            dataset_name=
                dataset_name,

            question_id=
                question_id,

            question=
                question,

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                candidate_rows
        )
    )

    assert X.shape == (
        len(candidate_rows),
        27
    ), (
        "Runtime Feature-v2 shape mismatch: "
        f"{X.shape}"
    )

    assert np.all(
        np.isfinite(X)
    )

    feature_cache[
        key
    ] = X

    return X


# ======================================================================
# 27. ABLATION SCORING
# ======================================================================

def score_ablation_group(
    bundle,
    X_full
):

    columns = bundle[
        "columns"
    ]

    X_subset = np.asarray(
        X_full[
            :,
            columns
        ],
        dtype=np.float32
    )

    assert X_subset.shape[1] == (
        bundle["input_dim"]
    )

    Z = (
        transform_exact_standardizer(
            bundle[
                "standardizer"
            ],
            X_subset
        )
    )

    with torch.inference_mode():

        tensor = (
            torch.from_numpy(Z)
            .to(ABLATION_DEVICE)
        )

        logits = (
            bundle["model"](
                tensor
            )
            .reshape(-1)
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    assert logits.shape == (
        X_subset.shape[0],
    )

    assert np.all(
        np.isfinite(logits)
    )

    return logits


# ======================================================================
# 28. ABLATION TRAVERSAL
# ======================================================================

def traverse_ablation_variant(
    dataset_name,
    bundle,
    selector_params,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    feature_cache,
    logit_cache
):

    active = [
        (
            str(entity),
        )
        for entity in topic_entities
    ]

    L = len(plan)

    active_hop_rows = 0
    active_prefixes = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(active)

    for hop, target_relation in enumerate(
        plan
    ):

        if not active:
            break

        active_hop_rows += 1
        active_prefixes += len(active)

        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )

        candidates = expansion[
            "candidates"
        ]

        candidate_rows = expansion[
            "candidate_rows"
        ]

        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )

        candidate_branches += len(
            candidates
        )

        if not candidates:

            active = []
            break

        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        if hop == L - 1:

            active = candidates

            peak_frontier = max(
                peak_frontier,
                len(active)
            )

            continue

        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(candidates) <= 1:

            active = candidates

            peak_frontier = max(
                peak_frontier,
                len(active)
            )

            continue

        decision_hops += 1

        frontier_key = (
            frontier_cache_key(
                plan_index=
                    plan_index,

                hop=
                    hop,

                active_prefixes=
                    active
            )
        )

        variant_key = (
            id(
                bundle["model"]
            ),
            frontier_key,
        )

        if variant_key in logit_cache:

            logits = (
                logit_cache[
                    variant_key
                ]
            )

        else:

            X_full = (
                get_full_runtime_group_features(
                    dataset_name=
                        dataset_name,

                    question_id=
                        question_id,

                    question=
                        question,

                    plan_index=
                        plan_index,

                    plan=
                        plan,

                    hop=
                        hop,

                    active_prefixes=
                        active,

                    candidate_rows=
                        candidate_rows,

                    feature_cache=
                        feature_cache
                )
            )

            logits = (
                score_ablation_group(
                    bundle=
                        bundle,

                    X_full=
                        X_full
                )
            )

            logit_cache[
                variant_key
            ] = logits

        selector_config = {
            "family":
                "afp",

            "T":
                float(
                    selector_params["T"]
                ),

            "gamma_min":
                float(
                    selector_params[
                        "gamma_min"
                    ]
                ),
        }

        selection = (
            select_policy_indices(
                selector_config,
                logits
            )
        )

        selected_indices = (
            selection[
                "selected_indices"
            ]
        )

        active = [
            candidates[index]
            for index in selected_indices
        ]

        retained_after_decision += len(
            active
        )

        if (
            selection[
                "requested_budget"
            ]
            is not None
        ):

            requested_budget_total += int(
                selection[
                    "requested_budget"
                ]
            )

            requested_budget_observations += 1

        if (
            selection[
                "uncertainty"
            ]
            is not None
        ):

            uncertainty_total += float(
                selection[
                    "uncertainty"
                ]
            )

            uncertainty_observations += 1

        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1

        peak_frontier = max(
            peak_frontier,
            len(active)
        )

    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(active_hop_rows),

        "active_prefixes":
            int(active_prefixes),

        "edges_examined":
            int(edges_examined),

        "candidate_branches":
            int(candidate_branches),

        "decision_hops":
            int(decision_hops),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(peak_frontier),
    }


# ======================================================================
# 29. RUN ONE DATASET'S FEATURE ABLATIONS
# ======================================================================

def run_feature_ablation_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    selector_params,
    variant_bundles
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )

    stats = {
        variant: {
            "active_hop_rows": 0,
            "active_prefixes": 0,
            "edges_examined": 0,
            "candidate_branches": 0,
            "reachable_plans": 0,
            "reachable_questions": 0,
            "decision_hops": 0,
            "retained_after_decision": 0,
            "requested_budget_total": 0,
            "requested_budget_observations": 0,
            "uncertainty_total": 0.0,
            "uncertainty_observations": 0,
            "fully_tied_decisions": 0,
            "peak_frontier": 0,
        }
        for variant in variant_bundles
    }

    start_time = time.time()

    for source_index in tqdm(
        range(len(planning_rows)),
        desc=(
            f"{dataset_name} feature ablations"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )

        question_rec = (
            question_rows[
                source_index
            ]
        )

        assert str(
            plan_rec["id"]
        ) == str(
            question_rec["id"]
        )

        question_id = str(
            plan_rec["id"]
        )

        question = str(
            question_rec[
                "question"
            ]
        )

        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )

        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )

        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )

        adjacency = (
            build_exact_rog_adjacency(
                plan_rec["graph"]
            )
        )

        # Computational caches shared only within this question.
        expansion_cache = {}
        feature_cache = {}
        logit_cache = {}

        question_reachable = {
            variant: False
            for variant in variant_bundles
        }

        for plan_index, plan in enumerate(
            plans
        ):

            if len(plan) == 0:
                continue

            for variant_name, bundle in (
                variant_bundles.items()
            ):

                result = (
                    traverse_ablation_variant(
                        dataset_name=
                            dataset_name,

                        bundle=
                            bundle,

                        selector_params=
                            selector_params,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        feature_cache=
                            feature_cache,

                        logit_cache=
                            logit_cache
                    )
                )

                s = stats[
                    variant_name
                ]

                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[key] += result[key]

                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]

                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]

                s[
                    "peak_frontier"
                ] = max(
                    s["peak_frontier"],
                    result[
                        "peak_frontier"
                    ]
                )

                # ------------------------------------------------------
                # Gold used ONLY after traversal.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )

                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1

                    question_reachable[
                        variant_name
                    ] = True

        for variant_name, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    variant_name
                ][
                    "reachable_questions"
                ] += 1

    elapsed = (
        time.time() - start_time
    )

    print(
        f"\n{dataset_name.upper()} feature "
        f"ablation evaluation completed "
        f"in {elapsed/60:.2f} min"
    )

    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )

    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )

    rows = []

    for variant_name, s in (
        stats.items()
    ):

        edges = int(
            s["edges_examined"]
        )

        reachable_q = int(
            s[
                "reachable_questions"
            ]
        )

        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]
            if
            s[
                "requested_budget_observations"
            ] > 0
            else
            np.nan
        )

        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]
            if
            s[
                "uncertainty_observations"
            ] > 0
            else
            np.nan
        )

        rows.append(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant_name,

                "input_dim":
                    variant_bundles[
                        variant_name
                    ][
                        "input_dim"
                    ],

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(avg_budget),

                "avg_uncertainty":
                    float(avg_uncertainty),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ======================================================================
# 30. RUN VALIDATION ABLATIONS
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "RUNNING VALIDATION FEATURE-FAMILY ABLATIONS"
)

print(
    "=" * 116
)


webqsp_feature_ablation_df = (
    run_feature_ablation_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        selector_params=
            AFP_14B_SELECTED[
                "webqsp"
            ],

        variant_bundles=
            ABLATION_MODELS[
                "webqsp"
            ]
    )
)


cwq_feature_ablation_df = (
    run_feature_ablation_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        selector_params=
            AFP_14B_SELECTED[
                "cwq"
            ],

        variant_bundles=
            ABLATION_MODELS[
                "cwq"
            ]
    )
)


# ======================================================================
# 31. ORIGINAL CELL-14B AFP REFERENCE
# ======================================================================

def original_afp_reference_row(
    comparison_df
):

    rows = comparison_df[
        comparison_df[
            "method"
        ] == "AFP"
    ]

    assert len(rows) == 1

    return rows.iloc[0]


webqsp_original_afp = (
    original_afp_reference_row(
        webqsp_comparison_14b_df
    )
)

cwq_original_afp = (
    original_afp_reference_row(
        cwq_comparison_14b_df
    )
)


# ======================================================================
# 32. DELTAS RELATIVE TO MATCHED FULL RETRAIN CONTROL
# ======================================================================

def add_ablation_deltas(df):

    df = df.copy()

    full_rows = df[
        df["variant"]
        ==
        "full_v2_retrain"
    ]

    assert len(full_rows) == 1

    full = full_rows.iloc[0]

    df[
        "delta_ar_vs_full_retrain"
    ] = (
        df["answer_retention"]
        -
        float(
            full[
                "answer_retention"
            ]
        )
    )

    df[
        "delta_ssr_vs_full_retrain"
    ] = (
        df["ssr"]
        -
        float(
            full["ssr"]
        )
    )

    df[
        "delta_edges_vs_full_retrain"
    ] = (
        df["edges_examined"]
        -
        int(
            full[
                "edges_examined"
            ]
        )
    )

    return df


webqsp_feature_ablation_df = (
    add_ablation_deltas(
        webqsp_feature_ablation_df
    )
)

cwq_feature_ablation_df = (
    add_ablation_deltas(
        cwq_feature_ablation_df
    )
)


# ======================================================================
# 33. FULL-RETRAIN REPRODUCIBILITY CONTROL
# ======================================================================
#
# Original selected model was trained previously on CUDA.
# This matched ablation protocol uses CPU.
#
# Therefore we REPORT rather than assume exact equality.
#
# The causal feature comparison itself is against full_v2_retrain,
# which shares exactly the same retraining protocol as every ablation.
# ======================================================================

def print_full_retrain_control(
    dataset_name,
    ablation_df,
    original_afp
):

    full = (
        ablation_df[
            ablation_df[
                "variant"
            ]
            ==
            "full_v2_retrain"
        ]
        .iloc[0]
    )

    print(
        "\n"
        + "=" * 112
    )

    print(
        f"{dataset_name.upper()} "
        "FULL-RETRAIN REPRODUCIBILITY CONTROL"
    )

    print(
        "=" * 112
    )

    print(
        "\nOriginal selected AFP (Cell 14B)"
    )

    print(
        "  AR:",
        f"{float(original_afp['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(original_afp['ssr']):.6f}"
    )

    print(
        "  edges:",
        int(
            original_afp[
                "edges_examined"
            ]
        )
    )

    print(
        "\nFull Feature-v2 matched retrain"
    )

    print(
        "  AR:",
        f"{float(full['answer_retention']):.6f}"
    )

    print(
        "  SSR:",
        f"{float(full['ssr']):.6f}"
    )

    print(
        "  edges:",
        int(
            full[
                "edges_examined"
            ]
        )
    )

    print(
        "\nDifference (retrain - original)"
    )

    print(
        "  AR delta:",
        f"{float(full['answer_retention'] - original_afp['answer_retention']):.6f}"
    )

    print(
        "  SSR delta:",
        f"{float(full['ssr'] - original_afp['ssr']):.6f}"
    )

    print(
        "  edge delta:",
        int(
            full[
                "edges_examined"
            ]
            -
            original_afp[
                "edges_examined"
            ]
        )
    )


print_full_retrain_control(
    "webqsp",
    webqsp_feature_ablation_df,
    webqsp_original_afp
)

print_full_retrain_control(
    "cwq",
    cwq_feature_ablation_df,
    cwq_original_afp
)


# ======================================================================
# 34. DISPLAY FEATURE ABLATION TABLES
# ======================================================================

ABLATION_DISPLAY_COLUMNS = [
    "variant",
    "input_dim",
    "answer_retention",
    "ssr",
    "edges_examined",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "fully_tied_decisions",
    "delta_ar_vs_full_retrain",
    "delta_ssr_vs_full_retrain",
    "delta_edges_vs_full_retrain",
]


def display_ablation_table(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 124
    )

    print(
        f"{dataset_name.upper()} "
        "FEATURE-FAMILY RETRAINING ABLATION"
    )

    print(
        "=" * 124
    )

    ordered_names = [
        "full_v2_retrain",
        "minus_semantic",
        "minus_path",
        "minus_structural",
        "minus_progress",
    ]

    ordered = (
        df.set_index(
            "variant"
        )
        .loc[
            ordered_names
        ]
        .reset_index()
    )

    print(
        ordered[
            ABLATION_DISPLAY_COLUMNS
        ].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_ablation_table(
    "webqsp",
    webqsp_feature_ablation_df
)

display_ablation_table(
    "cwq",
    cwq_feature_ablation_df
)


# ======================================================================
# 35. CAUSAL FEATURE-FAMILY SUMMARY
# ======================================================================
#
# Interpret AR and SSR jointly.
#
# Greater SSR is NOT automatically beneficial if AR falls.
# ======================================================================

def causal_feature_summary(df):

    rows = []

    for _, row in df.iterrows():

        if (
            row["variant"]
            ==
            "full_v2_retrain"
        ):
            continue

        rows.append(
            {
                "variant":
                    row["variant"],

                "delta_AR":
                    float(
                        row[
                            "delta_ar_vs_full_retrain"
                        ]
                    ),

                "delta_SSR":
                    float(
                        row[
                            "delta_ssr_vs_full_retrain"
                        ]
                    ),

                "removal_harmed_AR":
                    bool(
                        row[
                            "delta_ar_vs_full_retrain"
                        ] < 0
                    ),

                "removal_improved_AR":
                    bool(
                        row[
                            "delta_ar_vs_full_retrain"
                        ] > 0
                    ),
            }
        )

    return pd.DataFrame(rows)


webqsp_feature_causal_df = (
    causal_feature_summary(
        webqsp_feature_ablation_df
    )
)

cwq_feature_causal_df = (
    causal_feature_summary(
        cwq_feature_ablation_df
    )
)


print(
    "\nWEBQSP causal feature deltas"
)

print(
    webqsp_feature_causal_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


print(
    "\nCWQ causal feature deltas"
)

print(
    cwq_feature_causal_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ======================================================================
# 36. EXPORT RESULT CSVs
# ======================================================================

WEBQSP_ABLATION_CSV = (
    ABLATION_DIR
    / "webqsp_feature_family_retraining_ablation.csv"
)

CWQ_ABLATION_CSV = (
    ABLATION_DIR
    / "cwq_feature_family_retraining_ablation.csv"
)

WEBQSP_CAUSAL_CSV = (
    ABLATION_DIR
    / "webqsp_feature_family_causal_deltas.csv"
)

CWQ_CAUSAL_CSV = (
    ABLATION_DIR
    / "cwq_feature_family_causal_deltas.csv"
)


webqsp_feature_ablation_df.to_csv(
    WEBQSP_ABLATION_CSV,
    index=False
)

cwq_feature_ablation_df.to_csv(
    CWQ_ABLATION_CSV,
    index=False
)

webqsp_feature_causal_df.to_csv(
    WEBQSP_CAUSAL_CSV,
    index=False
)

cwq_feature_causal_df.to_csv(
    CWQ_CAUSAL_CSV,
    index=False
)


# ======================================================================
# 37. CHECKPOINT MANIFEST HELPER
# ======================================================================

def model_checkpoint_manifest(
    dataset_name
):

    output = {}

    for variant_name, path in (
        ABLATION_CHECKPOINT_PATHS[
            dataset_name
        ].items()
    ):

        output[
            variant_name
        ] = {
            "path":
                str(path),

            "sha256":
                sha256_file(path),
        }

    return output


# ======================================================================
# 38. CELL 15A MANIFEST
# ======================================================================

CELL15A_MANIFEST = {
    "cell":
        "RQ2_CELL15A_FEATURE_FAMILY_RETRAINING_ABLATIONS",

    "version":
        "rq2_cell15a_feature_ablation_v2_dimension_aware_exact_classes",

    "methodology":
        "remove_feature_family_then_retrain",

    "dimension_handling":
        {
            "approach":
                "exact_persisted_class_source_reinstantiated_with_variant_AFP_INPUT_DIM",

            "scorer_algorithm_changed":
                False,

            "standardizer_algorithm_changed":
                False,
        },

    "variants": {
        variant: {
            "feature_columns":
                columns,

            "input_dim":
                len(columns),
        }
        for variant, columns in (
            ABLATION_VARIANTS.items()
        )
    },

    "feature_groups": {
        key: value
        for key, value in (
            FEATURE_GROUPS.items()
        )
    },

    "training_protocol": {
        "optimizer":
            AFP_ABLATION_OPTIMIZER_NAME,

        "loss":
            "branch_bce",

        "epochs":
            80,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-4,

        "seed":
            42,

        "dropout":
            0.0,

        "class_weighting":
            False,

        "early_stopping":
            False,

        "device":
            str(
                ABLATION_DEVICE
            ),

        "ablation_specific_tuning":
            False,
    },

    "hidden_dimensions": {
        "webqsp":
            32,

        "cwq":
            64,
    },

    "selector_parameters_fixed_from_Cell13B": {
        "webqsp":
            AFP_14B_SELECTED[
                "webqsp"
            ],

        "cwq":
            AFP_14B_SELECTED[
                "cwq"
            ],
    },

    "train_feature_npz": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TRAIN_NPZ
                ),

            "sha256":
                sha256_file(
                    WEBQSP_TRAIN_NPZ
                ),
        },

        "cwq": {
            "path":
                str(
                    CWQ_TRAIN_NPZ
                ),

            "sha256":
                sha256_file(
                    CWQ_TRAIN_NPZ
                ),
        },
    },

    "ablation_checkpoints": {
        "webqsp":
            model_checkpoint_manifest(
                "webqsp"
            ),

        "cwq":
            model_checkpoint_manifest(
                "cwq"
            ),
    },

    "original_selected_checkpoint_sha256": {
        "webqsp":
            webqsp_ckpt_sha,

        "cwq":
            cwq_ckpt_sha,
    },

    "full_27d_standardizer_fidelity":
        True,

    "gold_used_by_scorer":
        False,

    "gold_used_by_selector":
        False,

    "gold_used_for":
        "post_traversal_validation_reachability_only",

    "test_examples_accessed":
        False,

    "final_AFP_frozen":
        False,

    "next":
        "Cell15B_selector_component_ablations_and_final_development_freeze",
}


CELL15A_MANIFEST_PATH = (
    ABLATION_DIR
    / "cell15a_feature_family_retraining_ablation_manifest.json"
)


with open(
    CELL15A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL15A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL15A_COMPLETE = True


# ======================================================================
# 39. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "=== RQ2 CELL 15A: "
    "FEATURE-FAMILY RETRAINING ABLATIONS COMPLETE ==="
)

print(
    "=" * 124
)


print(
    "\nAblation methodology:"
)

print(
    "  Full Feature-v2 retraining control"
)

print(
    "  -Semantic   -> retrained"
)

print(
    "  -Path       -> retrained"
)

print(
    "  -Structural -> retrained"
)

print(
    "  -Progress   -> retrained"
)


print(
    "\nDimension handling:"
)

print(
    "  Exact persisted AFPScorer source:          YES"
)

print(
    "  Exact persisted standardizer source:       YES"
)

print(
    "  Variant-specific AFP_INPUT_DIM:             YES"
)

print(
    "  Scorer algorithm modified:                  NO"
)

print(
    "  Standardizer algorithm modified:            NO"
)


print(
    "\nTraining controls:"
)

print(
    "  Optimizer:",
    AFP_ABLATION_OPTIMIZER_NAME
)

print(
    "  Loss: branch_bce"
)

print(
    "  Epochs: 80"
)

print(
    "  LR: 1e-3"
)

print(
    "  Weight decay: 1e-4"
)

print(
    "  Seed: 42"
)

print(
    "  Ablation-specific tuning: NO"
)


print(
    "\nSelector remains fixed:"
)

print(
    "  WebQSP:",
    AFP_14B_SELECTED[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    AFP_14B_SELECTED[
        "cwq"
    ]
)


print(
    "\nScientific status:"
)

print(
    "  Proper feature-removal retraining: COMPLETE"
)

print(
    "  Gold in scorer/selector:           NO"
)

print(
    "  TEST examples accessed:            NO"
)

print(
    "  Final AFP frozen:                  NO"
)


print(
    "\nOutputs:"
)

print(
    " ",
    WEBQSP_ABLATION_CSV
)

print(
    " ",
    CWQ_ABLATION_CSV
)

print(
    " ",
    WEBQSP_CAUSAL_CSV
)

print(
    " ",
    CWQ_CAUSAL_CSV
)

print(
    " ",
    CELL15A_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 15B — selector/component ablations "
    "+ final development decision/freeze."
)

Cell 15A prerequisites: PASSED
Feature version: afp_features_v2_masked_entity_semantics
Feature dimension: 27
Feature root: /kaggle/working/step3_rq2_dev_v1/03_features
Ablation output: /kaggle/working/step3_rq2_dev_v1/13_feature_ablations

WEBQSP matching train NPZ candidate(s):
  /kaggle/working/step3_rq2_dev_v1/03_features/webqsp/webqsp_train_afp_features_v2.npz
    SHA: 55db1698520e6c6be3f330a11c4c38334adc02923de650a6a76ce0cc757800b0

CWQ matching train NPZ candidate(s):
  /kaggle/working/step3_rq2_dev_v1/03_features/cwq/cwq_train_afp_features_v2.npz
    SHA: 1c5a273ca6aa46ce2ad153a0921d824ce3e62eff817c935f468c0d93d8602508

Frozen train feature files identified: PASSED

Training matrices:
  WebQSP: (18437, 27) | groups: 1457
  CWQ:    (218544, 27) | groups: 15937
Frozen training-cache gate: PASSED

Feature ablation variants:
  full_v2_retrain      27 features
  minus_semantic       19 features
  minus_path           20 features
  minus_structural     19 features
  minus_progress   

webqsp feature ablations:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP feature ablation evaluation completed in 0.04 min


cwq feature ablations:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ feature ablation evaluation completed in 2.50 min

WEBQSP FULL-RETRAIN REPRODUCIBILITY CONTROL

Original selected AFP (Cell 14B)
  AR: 1.000000
  SSR: 0.010272
  edges: 338018

Full Feature-v2 matched retrain
  AR: 1.000000
  SSR: 0.010304
  edges: 338007

Difference (retrain - original)
  AR delta: 0.000000
  SSR delta: 0.000032
  edge delta: -11

CWQ FULL-RETRAIN REPRODUCIBILITY CONTROL

Original selected AFP (Cell 14B)
  AR: 0.990103
  SSR: 0.062031
  edges: 4931160

Full Feature-v2 matched retrain
  AR: 0.990103
  SSR: 0.062182
  edges: 4930362

Difference (retrain - original)
  AR delta: 0.000000
  SSR delta: 0.000152
  edge delta: -798

WEBQSP FEATURE-FAMILY RETRAINING ABLATION
         variant  input_dim  answer_retention      ssr  edges_examined  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  avg_uncertainty  fully_tied_decisions  delta_ar_vs_full_retrain  delta_ssr_vs_full_retrain  delta_edges_vs_full_retrain
 full_v2_retrain         27         

In [33]:
# ======================================================================
# RQ2 CELL 15B
# SELECTOR / COMPONENT ABLATIONS + FINAL AFP DEVELOPMENT FREEZE
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT replace Cell 15A.
# DO NOT restart the kernel.
# DO NOT rerun Cells 13 / 13B / 14 / 14B / 15A.
#
# ======================================================================
# PURPOSE
# ======================================================================
#
# Complete the remaining RQ2 development-stage causal analysis and
# freeze the final AFP configuration BEFORE any TEST evaluation.
#
# SELECTOR / COMPONENT ABLATIONS
# --------------------------------
#
# A. Full AFP
#    selected scorer + selected T + selected gamma_min
#
# B. No uncertainty adaptation
#    same scorer
#    same temperature T
#    same gamma_min
#    BUT:
#
#       gamma_h = gamma_min
#
#    instead of:
#
#       gamma_h = gamma_min + u_h (1 - gamma_min)
#
#    Fully tied groups STILL retain all.
#    Tie-expansion behavior unchanged.
#
# C. No temperature scaling
#    same scorer
#    same gamma_min
#    same uncertainty-adaptive rule
#    BUT:
#
#       T = 1.0
#
#    This isolates the contribution of validation-selected temperature
#    calibration.
#
#
# EXISTING CAUSAL CONTROLS FROM CELL 14B
# --------------------------------------
#
# D. Adaptive-Budget Random
#    same AFP retained-count mechanism, random branch identity
#    -> ranking-value control
#
# E. Fixed Top-B
#    fixed-budget controlled baseline
#    -> adaptive-budget comparison
#
#
# IMPORTANT
# ---------
# - NO further hyperparameter search
# - NO feature redesign
# - NO retraining for selector ablations
# - NO TEST access
# - Ablations are diagnostic only
# - Final method remains:
#
#       full Feature-v2
#       original selected scorer checkpoint
#       selected expanded AFP selector parameters
#
# After this cell:
#
#       AFP_DEVELOPMENT_FROZEN = True
#
# and NO development changes are allowed before TEST.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import hashlib
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD PREREQUISITES
# ======================================================================

required = [
    # --------------------------------------------------------------
    # Development stage
    # --------------------------------------------------------------
    "CELL15A_COMPLETE",
    "CELL14B_COMPLETE",
    "CELL13B_COMPLETE",

    # --------------------------------------------------------------
    # Final selected AFP parameters
    # --------------------------------------------------------------
    "AFP_14B_SELECTED",

    # --------------------------------------------------------------
    # Selected original scorer runtime
    # --------------------------------------------------------------
    "AFP_RUNTIME_SCORE_GROUP",
    "AFP_RUNTIME_FEATURE_VERSION",

    # --------------------------------------------------------------
    # Frozen validation data
    # --------------------------------------------------------------
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "webqsp_rog_reference",
    "cwq_rog_reference",

    # --------------------------------------------------------------
    # Exact traversal utilities
    # --------------------------------------------------------------
    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",

    "get_cached_relation_expansion",
    "get_group_logits",
    "frontier_cache_key",

    "final_prefixes_reach_answer",

    # --------------------------------------------------------------
    # Cell 14B controlled evidence
    # --------------------------------------------------------------
    "webqsp_comparison_14b_df",
    "cwq_comparison_14b_df",

    "webqsp_summary_14b_df",
    "cwq_summary_14b_df",

    # --------------------------------------------------------------
    # Cell 15A causal feature evidence
    # --------------------------------------------------------------
    "webqsp_feature_ablation_df",
    "cwq_feature_ablation_df",

    # --------------------------------------------------------------
    # Original selected checkpoints
    # --------------------------------------------------------------
    "webqsp_ckpt_obj",
    "cwq_ckpt_obj",

    "webqsp_ckpt_sha",
    "cwq_ckpt_sha",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required prior-cell objects:\n  "
    + "\n  ".join(missing)
)

assert CELL15A_COMPLETE is True
assert CELL14B_COMPLETE is True
assert CELL13B_COMPLETE is True


print(
    "Cell 15B prerequisites: PASSED"
)

print(
    "Feature version:",
    AFP_RUNTIME_FEATURE_VERSION
)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

CELL15B_DIR = (
    ROOT
    / "14_selector_ablations_and_final_freeze"
)

CELL15B_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Cell 15B output:",
    CELL15B_DIR
)


# ======================================================================
# 3. FINAL DEVELOPMENT-SELECTED AFP PARAMETERS
# ======================================================================

FINAL_SELECTOR_PARAMS = {
    "webqsp": {
        "T":
            float(
                AFP_14B_SELECTED[
                    "webqsp"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_14B_SELECTED[
                    "webqsp"
                ][
                    "gamma_min"
                ]
            ),
    },

    "cwq": {
        "T":
            float(
                AFP_14B_SELECTED[
                    "cwq"
                ][
                    "T"
                ]
            ),

        "gamma_min":
            float(
                AFP_14B_SELECTED[
                    "cwq"
                ][
                    "gamma_min"
                ]
            ),
    },
}


assert FINAL_SELECTOR_PARAMS[
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert FINAL_SELECTOR_PARAMS[
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


print(
    "\nFinal development-selected selector:"
)

print(
    "  WebQSP:",
    FINAL_SELECTOR_PARAMS[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FINAL_SELECTOR_PARAMS[
        "cwq"
    ]
)


# ======================================================================
# 4. PREDECLARE SELECTOR ABLATIONS
# ======================================================================
#
# These are NOT candidate configurations for selection.
#
# They are fixed causal diagnostics.
# ======================================================================

SELECTOR_ABLATIONS = [
    "full_afp",
    "no_uncertainty_adaptation",
    "no_temperature_scaling",
]


print(
    "\nSelector/component ablations:"
)

for name in SELECTOR_ABLATIONS:

    print(
        " ",
        name
    )


print(
    "\nNo ablation-specific tuning: YES"
)


# ======================================================================
# 5. NUMERICAL SELECTOR HELPERS
# ======================================================================

TIE_ATOL_15B = 1e-8


def stable_softmax_15b(
    logits,
    temperature
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    temperature = float(
        temperature
    )

    assert temperature > 0.0


    values = (
        logits
        /
        temperature
    )


    values = (
        values
        -
        np.max(
            values
        )
    )


    exp_values = np.exp(
        values
    )


    denominator = float(
        np.sum(
            exp_values
        )
    )


    assert denominator > 0
    assert np.isfinite(
        denominator
    )


    return (
        exp_values
        /
        denominator
    )


def normalized_entropy_15b(
    probabilities
):

    p = np.asarray(
        probabilities,
        dtype=np.float64
    )


    n = len(
        p
    )


    if n <= 1:

        return 0.0


    safe = np.clip(
        p,
        1e-12,
        1.0
    )


    entropy = -float(
        np.sum(
            safe
            *
            np.log(
                safe
            )
        )
    )


    denominator = math.log(
        n
    )


    if denominator <= 0:

        return 0.0


    return float(
        np.clip(
            entropy
            /
            denominator,
            0.0,
            1.0
        )
    )


def all_logits_tied_15b(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    if len(
        logits
    ) <= 1:

        return True


    return bool(
        (
            np.max(
                logits
            )
            -
            np.min(
                logits
            )
        )
        <=
        TIE_ATOL_15B
    )


def stable_descending_order_15b(
    logits
):

    return np.argsort(
        -np.asarray(
            logits,
            dtype=np.float64
        ),
        kind="stable"
    )


def tie_expanded_top_k_15b(
    logits,
    requested_k
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    requested_k = int(
        requested_k
    )


    assert requested_k >= 1


    if requested_k >= n:

        return list(
            range(
                n
            )
        )


    order = (
        stable_descending_order_15b(
            logits
        )
    )


    cutoff_index = order[
        requested_k
        - 1
    ]


    cutoff_score = logits[
        cutoff_index
    ]


    selected = []


    for index, score in enumerate(
        logits
    ):

        if (
            score
            >
            cutoff_score
            or
            np.isclose(
                score,
                cutoff_score,
                atol=TIE_ATOL_15B,
                rtol=0.0
            )
        ):

            selected.append(
                index
            )


    return selected


# ======================================================================
# 6. SELECTOR COMPONENT ABLATION
# ======================================================================

def select_component_variant(
    logits,
    selected_T,
    gamma_min,
    variant
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )


    n = len(
        logits
    )


    assert n > 1


    selected_T = float(
        selected_T
    )

    gamma_min = float(
        gamma_min
    )


    assert variant in (
        SELECTOR_ABLATIONS
    )


    # --------------------------------------------------------------
    # Preserve the full-method abstention behavior under complete tie.
    #
    # This is NOT removed because we want to isolate uncertainty
    # adaptation and temperature calibration separately.
    # --------------------------------------------------------------

    if all_logits_tied_15b(
        logits
    ):

        return {
            "selected_indices":
                list(
                    range(
                        n
                    )
                ),

            "requested_budget":
                n,

            "retained_count":
                n,

            "uncertainty":
                1.0,

            "gamma":
                1.0,

            "temperature_used":
                (
                    1.0
                    if
                    variant
                    ==
                    "no_temperature_scaling"
                    else
                    selected_T
                ),

            "fully_tied":
                True,
        }


    # --------------------------------------------------------------
    # Temperature component
    # --------------------------------------------------------------

    if (
        variant
        ==
        "no_temperature_scaling"
    ):

        temperature = 1.0

    else:

        temperature = (
            selected_T
        )


    probabilities = (
        stable_softmax_15b(
            logits=
                logits,

            temperature=
                temperature
        )
    )


    uncertainty = (
        normalized_entropy_15b(
            probabilities
        )
    )


    # --------------------------------------------------------------
    # Uncertainty-adaptation component
    # --------------------------------------------------------------

    if (
        variant
        ==
        "no_uncertainty_adaptation"
    ):

        gamma = (
            gamma_min
        )

    else:

        gamma = (
            gamma_min
            +
            uncertainty
            *
            (
                1.0
                -
                gamma_min
            )
        )


    gamma = float(
        np.clip(
            gamma,
            0.0,
            1.0
        )
    )


    order = (
        stable_descending_order_15b(
            logits
        )
    )


    sorted_probabilities = (
        probabilities[
            order
        ]
    )


    cumulative = np.cumsum(
        sorted_probabilities
    )


    requested_budget = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        )
        + 1
    )


    requested_budget = min(
        max(
            requested_budget,
            1
        ),
        n
    )


    selected_indices = (
        tie_expanded_top_k_15b(
            logits=
                logits,

            requested_k=
                requested_budget
        )
    )


    return {
        "selected_indices":
            selected_indices,

        "requested_budget":
            requested_budget,

        "retained_count":
            len(
                selected_indices
            ),

        "uncertainty":
            float(
                uncertainty
            ),

        "gamma":
            float(
                gamma
            ),

        "temperature_used":
            float(
                temperature
            ),

        "fully_tied":
            False,
    }


# ======================================================================
# 7. COMPONENT TRAVERSAL
# ======================================================================

def traverse_selector_ablation(
    dataset_name,
    variant,
    selector_params,
    question_id,
    question,
    adjacency,
    topic_entities,
    plan_index,
    plan,
    expansion_cache,
    score_cache
):

    active = [
        (
            str(
                entity
            ),
        )
        for entity in topic_entities
    ]


    L = len(
        plan
    )


    active_hop_rows = 0
    active_prefixes = 0

    edges_examined = 0
    candidate_branches = 0

    decision_hops = 0

    retained_after_decision = 0

    requested_budget_total = 0
    requested_budget_observations = 0

    uncertainty_total = 0.0
    uncertainty_observations = 0

    gamma_total = 0.0
    gamma_observations = 0

    fully_tied_decisions = 0

    peak_frontier = len(
        active
    )


    for hop, target_relation in enumerate(
        plan
    ):

        if not active:

            break


        active_hop_rows += 1

        active_prefixes += len(
            active
        )


        expansion = (
            get_cached_relation_expansion(
                adjacency=
                    adjacency,

                plan_index=
                    plan_index,

                hop=
                    hop,

                target_relation=
                    target_relation,

                active_prefixes=
                    active,

                expansion_cache=
                    expansion_cache
            )
        )


        candidates = expansion[
            "candidates"
        ]


        candidate_rows = expansion[
            "candidate_rows"
        ]


        edges_examined += int(
            expansion[
                "edges_cost"
            ]
        )


        candidate_branches += len(
            candidates
        )


        if not candidates:

            active = []

            break


        # --------------------------------------------------------------
        # FINAL-HOP PROTECTION
        # --------------------------------------------------------------

        if hop == (
            L
            -
            1
        ):

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        # --------------------------------------------------------------
        # SINGLETON BYPASS
        # --------------------------------------------------------------

        if len(
            candidates
        ) <= 1:

            active = candidates


            peak_frontier = max(
                peak_frontier,
                len(
                    active
                )
            )


            continue


        decision_hops += 1


        logits = (
            get_group_logits(
                dataset_name=
                    dataset_name,

                question_id=
                    question_id,

                question=
                    question,

                plan_index=
                    plan_index,

                plan=
                    plan,

                hop=
                    hop,

                active_prefixes=
                    active,

                candidate_rows=
                    candidate_rows,

                score_cache=
                    score_cache
            )
        )


        selection = (
            select_component_variant(
                logits=
                    logits,

                selected_T=
                    selector_params[
                        "T"
                    ],

                gamma_min=
                    selector_params[
                        "gamma_min"
                    ],

                variant=
                    variant
            )
        )


        selected_indices = (
            selection[
                "selected_indices"
            ]
        )


        active = [
            candidates[
                index
            ]
            for index
            in selected_indices
        ]


        retained_after_decision += len(
            active
        )


        requested_budget_total += int(
            selection[
                "requested_budget"
            ]
        )


        requested_budget_observations += 1


        uncertainty_total += float(
            selection[
                "uncertainty"
            ]
        )


        uncertainty_observations += 1


        gamma_total += float(
            selection[
                "gamma"
            ]
        )


        gamma_observations += 1


        if selection[
            "fully_tied"
        ]:

            fully_tied_decisions += 1


        peak_frontier = max(
            peak_frontier,
            len(
                active
            )
        )


    return {
        "final_prefixes":
            active,

        "active_hop_rows":
            int(
                active_hop_rows
            ),

        "active_prefixes":
            int(
                active_prefixes
            ),

        "edges_examined":
            int(
                edges_examined
            ),

        "candidate_branches":
            int(
                candidate_branches
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "retained_after_decision":
            int(
                retained_after_decision
            ),

        "requested_budget_total":
            int(
                requested_budget_total
            ),

        "requested_budget_observations":
            int(
                requested_budget_observations
            ),

        "uncertainty_total":
            float(
                uncertainty_total
            ),

        "uncertainty_observations":
            int(
                uncertainty_observations
            ),

        "gamma_total":
            float(
                gamma_total
            ),

        "gamma_observations":
            int(
                gamma_observations
            ),

        "fully_tied_decisions":
            int(
                fully_tied_decisions
            ),

        "peak_frontier":
            int(
                peak_frontier
            ),
    }


# ======================================================================
# 8. RUN ONE DATASET
# ======================================================================

def run_selector_ablation_dataset(
    dataset_name,
    planning_rows,
    question_rows,
    rog_reference,
    selector_params
):

    assert len(
        planning_rows
    ) == len(
        question_rows
    )


    stats = {
        variant: {
            "active_hop_rows": 0,
            "active_prefixes": 0,
            "edges_examined": 0,
            "candidate_branches": 0,
            "reachable_plans": 0,
            "reachable_questions": 0,
            "decision_hops": 0,
            "retained_after_decision": 0,
            "requested_budget_total": 0,
            "requested_budget_observations": 0,
            "uncertainty_total": 0.0,
            "uncertainty_observations": 0,
            "gamma_total": 0.0,
            "gamma_observations": 0,
            "fully_tied_decisions": 0,
            "peak_frontier": 0,
        }

        for variant
        in SELECTOR_ABLATIONS
    }


    start_time = time.time()


    for source_index in tqdm(
        range(
            len(
                planning_rows
            )
        ),
        desc=(
            f"{dataset_name} selector ablations"
        )
    ):

        plan_rec = (
            planning_rows[
                source_index
            ]
        )


        question_rec = (
            question_rows[
                source_index
            ]
        )


        assert str(
            plan_rec[
                "id"
            ]
        ) == str(
            question_rec[
                "id"
            ]
        )


        question_id = str(
            plan_rec[
                "id"
            ]
        )


        question = str(
            question_rec[
                "question"
            ]
        )


        topic_entities = (
            as_entity_list(
                plan_rec[
                    "q_entity"
                ]
            )
        )


        gold_answers = (
            as_entity_list(
                plan_rec[
                    "a_entity"
                ]
            )
        )


        plans = (
            normalize_relation_plans(
                plan_rec[
                    "predicted_paths"
                ]
            )
        )


        adjacency = (
            build_exact_rog_adjacency(
                plan_rec[
                    "graph"
                ]
            )
        )


        expansion_cache = {}

        score_cache = {}


        question_reachable = {
            variant:
                False

            for variant
            in SELECTOR_ABLATIONS
        }


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                continue


            for variant in (
                SELECTOR_ABLATIONS
            ):

                result = (
                    traverse_selector_ablation(
                        dataset_name=
                            dataset_name,

                        variant=
                            variant,

                        selector_params=
                            selector_params,

                        question_id=
                            question_id,

                        question=
                            question,

                        adjacency=
                            adjacency,

                        topic_entities=
                            topic_entities,

                        plan_index=
                            plan_index,

                        plan=
                            plan,

                        expansion_cache=
                            expansion_cache,

                        score_cache=
                            score_cache
                    )
                )


                s = stats[
                    variant
                ]


                for key in [
                    "active_hop_rows",
                    "active_prefixes",
                    "edges_examined",
                    "candidate_branches",
                    "decision_hops",
                    "retained_after_decision",
                    "requested_budget_total",
                    "requested_budget_observations",
                    "fully_tied_decisions",
                ]:

                    s[
                        key
                    ] += result[
                        key
                    ]


                s[
                    "uncertainty_total"
                ] += result[
                    "uncertainty_total"
                ]


                s[
                    "uncertainty_observations"
                ] += result[
                    "uncertainty_observations"
                ]


                s[
                    "gamma_total"
                ] += result[
                    "gamma_total"
                ]


                s[
                    "gamma_observations"
                ] += result[
                    "gamma_observations"
                ]


                s[
                    "peak_frontier"
                ] = max(
                    s[
                        "peak_frontier"
                    ],
                    result[
                        "peak_frontier"
                    ]
                )


                # ------------------------------------------------------
                # Gold is used ONLY after traversal.
                # ------------------------------------------------------

                reachable = (
                    final_prefixes_reach_answer(
                        final_prefixes=
                            result[
                                "final_prefixes"
                            ],

                        gold_answers=
                            gold_answers
                    )
                )


                if reachable:

                    s[
                        "reachable_plans"
                    ] += 1


                    question_reachable[
                        variant
                    ] = True


        for variant, reachable in (
            question_reachable.items()
        ):

            if reachable:

                stats[
                    variant
                ][
                    "reachable_questions"
                ] += 1


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"\n{dataset_name.upper()} selector "
        f"ablations completed in "
        f"{elapsed/60:.2f} min"
    )


    rog_edges = float(
        rog_reference[
            "edges_examined"
        ]
    )


    rog_reachable = int(
        rog_reference[
            "reachable_questions"
        ]
    )


    rows = []


    for variant, s in (
        stats.items()
    ):

        edges = int(
            s[
                "edges_examined"
            ]
        )


        reachable_q = int(
            s[
                "reachable_questions"
            ]
        )


        avg_budget = (
            s[
                "requested_budget_total"
            ]
            /
            s[
                "requested_budget_observations"
            ]

            if
            s[
                "requested_budget_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_uncertainty = (
            s[
                "uncertainty_total"
            ]
            /
            s[
                "uncertainty_observations"
            ]

            if
            s[
                "uncertainty_observations"
            ]
            > 0

            else
            np.nan
        )


        avg_gamma = (
            s[
                "gamma_total"
            ]
            /
            s[
                "gamma_observations"
            ]

            if
            s[
                "gamma_observations"
            ]
            > 0

            else
            np.nan
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "variant":
                    variant,

                "selected_T":
                    float(
                        selector_params[
                            "T"
                        ]
                    ),

                "selected_gamma_min":
                    float(
                        selector_params[
                            "gamma_min"
                        ]
                    ),

                "edges_examined":
                    edges,

                "ssr":
                    float(
                        1.0
                        -
                        edges
                        /
                        rog_edges
                    ),

                "reachable_questions":
                    reachable_q,

                "rog_reachable_questions":
                    rog_reachable,

                "answer_retention":
                    float(
                        reachable_q
                        /
                        rog_reachable
                    ),

                "active_hop_rows":
                    int(
                        s[
                            "active_hop_rows"
                        ]
                    ),

                "active_prefixes":
                    int(
                        s[
                            "active_prefixes"
                        ]
                    ),

                "candidate_branches":
                    int(
                        s[
                            "candidate_branches"
                        ]
                    ),

                "reachable_plans":
                    int(
                        s[
                            "reachable_plans"
                        ]
                    ),

                "decision_hops":
                    int(
                        s[
                            "decision_hops"
                        ]
                    ),

                "avg_requested_budget":
                    float(
                        avg_budget
                    ),

                "avg_uncertainty":
                    float(
                        avg_uncertainty
                    ),

                "avg_gamma":
                    float(
                        avg_gamma
                    ),

                "fully_tied_decisions":
                    int(
                        s[
                            "fully_tied_decisions"
                        ]
                    ),

                "peak_frontier":
                    int(
                        s[
                            "peak_frontier"
                        ]
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ======================================================================
# 9. RUN WEBQSP + CWQ
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "RUNNING SELECTOR / COMPONENT ABLATIONS"
)

print(
    "=" * 118
)


webqsp_selector_ablation_df = (
    run_selector_ablation_dataset(
        dataset_name=
            "webqsp",

        planning_rows=
            webqsp_val_plan_rows,

        question_rows=
            webqsp_val_runtime,

        rog_reference=
            webqsp_rog_reference,

        selector_params=
            FINAL_SELECTOR_PARAMS[
                "webqsp"
            ]
    )
)


cwq_selector_ablation_df = (
    run_selector_ablation_dataset(
        dataset_name=
            "cwq",

        planning_rows=
            cwq_val_plan_rows,

        question_rows=
            cwq_val_runtime,

        rog_reference=
            cwq_rog_reference,

        selector_params=
            FINAL_SELECTOR_PARAMS[
                "cwq"
            ]
    )
)


# ======================================================================
# 10. FULL AFP SOFTWARE FIDELITY TO CELL 14B
# ======================================================================

def get_single_row(
    df,
    column,
    value
):

    rows = df[
        df[
            column
        ]
        ==
        value
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def full_afp_fidelity_gate(
    dataset_name,
    selector_df,
    comparison_df
):

    component = (
        get_single_row(
            selector_df,
            "variant",
            "full_afp"
        )
    )


    reference = (
        get_single_row(
            comparison_df,
            "method",
            "AFP"
        )
    )


    for key in [
        "edges_examined",
        "reachable_questions",
        "active_prefixes",
        "candidate_branches",
        "decision_hops",
    ]:

        assert int(
            component[
                key
            ]
        ) == int(
            reference[
                key
            ]
        ), (
            f"{dataset_name}: full AFP mismatch "
            f"for {key}: "
            f"{component[key]} != "
            f"{reference[key]}"
        )


    assert np.isclose(
        float(
            component[
                "ssr"
            ]
        ),
        float(
            reference[
                "ssr"
            ]
        ),
        atol=1e-12
    )


    assert np.isclose(
        float(
            component[
                "answer_retention"
            ]
        ),
        float(
            reference[
                "answer_retention"
            ]
        ),
        atol=1e-12
    )


    print(
        f"{dataset_name} full AFP "
        "Cell-14B fidelity: PASSED"
    )


full_afp_fidelity_gate(
    "WebQSP",
    webqsp_selector_ablation_df,
    webqsp_comparison_14b_df
)


full_afp_fidelity_gate(
    "CWQ",
    cwq_selector_ablation_df,
    cwq_comparison_14b_df
)


# ======================================================================
# 11. ADD DELTAS RELATIVE TO FULL AFP
# ======================================================================

def add_selector_deltas(
    df
):

    df = df.copy()


    full = (
        get_single_row(
            df,
            "variant",
            "full_afp"
        )
    )


    df[
        "delta_ar_vs_full"
    ] = (
        df[
            "answer_retention"
        ]
        -
        float(
            full[
                "answer_retention"
            ]
        )
    )


    df[
        "delta_ssr_vs_full"
    ] = (
        df[
            "ssr"
        ]
        -
        float(
            full[
                "ssr"
            ]
        )
    )


    df[
        "delta_edges_vs_full"
    ] = (
        df[
            "edges_examined"
        ]
        -
        int(
            full[
                "edges_examined"
            ]
        )
    )


    return df


webqsp_selector_ablation_df = (
    add_selector_deltas(
        webqsp_selector_ablation_df
    )
)


cwq_selector_ablation_df = (
    add_selector_deltas(
        cwq_selector_ablation_df
    )
)


# ======================================================================
# 12. DISPLAY SELECTOR ABLATIONS
# ======================================================================

SELECTOR_DISPLAY_COLUMNS = [
    "variant",
    "answer_retention",
    "ssr",
    "edges_examined",
    "active_prefixes",
    "candidate_branches",
    "decision_hops",
    "avg_requested_budget",
    "avg_uncertainty",
    "avg_gamma",
    "fully_tied_decisions",
    "delta_ar_vs_full",
    "delta_ssr_vs_full",
    "delta_edges_vs_full",
]


def display_selector_ablation(
    dataset_name,
    df
):

    print(
        "\n"
        + "=" * 124
    )

    print(
        f"{dataset_name.upper()} "
        "SELECTOR / COMPONENT ABLATIONS"
    )

    print(
        "=" * 124
    )


    ordered = (
        df.set_index(
            "variant"
        )
        .loc[
            SELECTOR_ABLATIONS
        ]
        .reset_index()
    )


    print(
        ordered[
            SELECTOR_DISPLAY_COLUMNS
        ].to_string(
            index=False,

            float_format=lambda x:
                f"{x:.6f}"
        )
    )


display_selector_ablation(
    "webqsp",
    webqsp_selector_ablation_df
)


display_selector_ablation(
    "cwq",
    cwq_selector_ablation_df
)


# ======================================================================
# 13. EXTRACT EXISTING CELL 14B CAUSAL CONTROLS
# ======================================================================

def summary_method_row(
    summary_df,
    method
):

    rows = summary_df[
        summary_df[
            "method"
        ]
        ==
        method
    ]


    assert len(
        rows
    ) == 1


    return rows.iloc[
        0
    ]


def build_component_evidence_table(
    dataset_name,
    selector_df,
    summary_14b_df
):

    rows = []


    # --------------------------------------------------------------
    # New selector ablations
    # --------------------------------------------------------------

    for _, row in (
        selector_df.iterrows()
    ):

        rows.append(
            {
                "dataset":
                    dataset_name,

                "component_test":
                    row[
                        "variant"
                    ],

                "type":
                    "selector_ablation",

                "AR":
                    float(
                        row[
                            "answer_retention"
                        ]
                    ),

                "SSR":
                    float(
                        row[
                            "ssr"
                        ]
                    ),

                "n_runs":
                    1,
            }
        )


    # --------------------------------------------------------------
    # Existing matched/random causal controls
    # --------------------------------------------------------------

    for method, label in [
        (
            "Adaptive-Budget-Random",
            "ranking_removed_matched_adaptive_budget"
        ),
        (
            "Fixed-Top-B",
            "fixed_budget_control"
        ),
        (
            "Random-B",
            "random_fixed_budget_control"
        ),
    ]:

        row = summary_method_row(
            summary_14b_df,
            method
        )


        rows.append(
            {
                "dataset":
                    dataset_name,

                "component_test":
                    label,

                "type":
                    "existing_control",

                "AR":
                    float(
                        row[
                            "ar_mean"
                        ]
                    ),

                "SSR":
                    float(
                        row[
                            "ssr_mean"
                        ]
                    ),

                "n_runs":
                    int(
                        row[
                            "n_runs"
                        ]
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


webqsp_component_evidence_df = (
    build_component_evidence_table(
        dataset_name=
            "webqsp",

        selector_df=
            webqsp_selector_ablation_df,

        summary_14b_df=
            webqsp_summary_14b_df
    )
)


cwq_component_evidence_df = (
    build_component_evidence_table(
        dataset_name=
            "cwq",

        selector_df=
            cwq_selector_ablation_df,

        summary_14b_df=
            cwq_summary_14b_df
    )
)


print(
    "\n"
    + "=" * 124
)

print(
    "WEBQSP COMPLETE COMPONENT EVIDENCE"
)

print(
    "=" * 124
)


print(
    webqsp_component_evidence_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


print(
    "\n"
    + "=" * 124
)

print(
    "CWQ COMPLETE COMPONENT EVIDENCE"
)

print(
    "=" * 124
)


print(
    cwq_component_evidence_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ======================================================================
# 14. VALIDATION DOMINANCE STATUS
# ======================================================================
#
# IMPORTANT:
#
# We explicitly preserve negative / inconvenient development evidence.
#
# If Fixed Top-B Pareto-dominates AFP, the final freeze manifest records
# that fact rather than hiding it.
# ======================================================================

def compute_fixed_top_b_dominance(
    summary_df
):

    afp = summary_method_row(
        summary_df,
        "AFP"
    )


    fixed = summary_method_row(
        summary_df,
        "Fixed-Top-B"
    )


    afp_ar = float(
        afp[
            "ar_mean"
        ]
    )


    afp_ssr = float(
        afp[
            "ssr_mean"
        ]
    )


    fixed_ar = float(
        fixed[
            "ar_mean"
        ]
    )


    fixed_ssr = float(
        fixed[
            "ssr_mean"
        ]
    )


    dominates = (
        fixed_ar
        >=
        afp_ar
        and
        fixed_ssr
        >=
        afp_ssr
        and
        (
            fixed_ar
            >
            afp_ar
            or
            fixed_ssr
            >
            afp_ssr
        )
    )


    return {
        "fixed_top_b_pareto_dominates_afp":
            bool(
                dominates
            ),

        "afp_AR":
            afp_ar,

        "afp_SSR":
            afp_ssr,

        "fixed_top_b_AR":
            fixed_ar,

        "fixed_top_b_SSR":
            fixed_ssr,
    }


WEBQSP_DOMINANCE = (
    compute_fixed_top_b_dominance(
        webqsp_summary_14b_df
    )
)


CWQ_DOMINANCE = (
    compute_fixed_top_b_dominance(
        cwq_summary_14b_df
    )
)


print(
    "\nDevelopment Pareto status:"
)

print(
    "  WebQSP:",
    WEBQSP_DOMINANCE
)

print(
    "  CWQ:   ",
    CWQ_DOMINANCE
)


# ======================================================================
# 15. FEATURE ABLATION FREEZE DECISION
# ======================================================================
#
# Ablations are diagnostic.
#
# We DO NOT use their outcome to redesign the feature set after seeing
# validation results.
#
# Final AFP retains the original full 27-D Feature-v2 representation.
# ======================================================================

FINAL_FEATURE_DECISION = {
    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "input_dim":
        27,

    "decision":
        "retain_full_feature_v2",

    "reason":
        (
            "Feature-family ablations are diagnostic only; "
            "no feature-removal variant is adopted post hoc. "
            "Full Feature-v2 remains the pre-existing method "
            "representation."
        ),
}


print(
    "\nFinal feature decision:"
)

print(
    "  retain full 27-D Feature-v2"
)


# ======================================================================
# 16. ORIGINAL SELECTED SCORER METADATA
# ======================================================================

def selected_checkpoint_metadata(
    checkpoint,
    checkpoint_sha
):

    required = [
        "dataset",
        "hidden_dim",
        "loss_name",
        "seed",
        "epochs",
        "learning_rate",
        "weight_decay",
        "feature_spec_sha256",
        "scorer_spec_sha256",
        "training_spec_sha256",
    ]


    missing_keys = [
        key
        for key in required
        if key not in checkpoint
    ]


    assert not missing_keys, (
        "Selected checkpoint missing metadata: "
        f"{missing_keys}"
    )


    return {
        "checkpoint_sha256":
            str(
                checkpoint_sha
            ),

        "dataset":
            str(
                checkpoint[
                    "dataset"
                ]
            ),

        "hidden_dim":
            int(
                checkpoint[
                    "hidden_dim"
                ]
            ),

        "loss_name":
            str(
                checkpoint[
                    "loss_name"
                ]
            ),

        "seed":
            int(
                checkpoint[
                    "seed"
                ]
            ),

        "epochs":
            int(
                checkpoint[
                    "epochs"
                ]
            ),

        "learning_rate":
            float(
                checkpoint[
                    "learning_rate"
                ]
            ),

        "weight_decay":
            float(
                checkpoint[
                    "weight_decay"
                ]
            ),

        "feature_spec_sha256":
            str(
                checkpoint[
                    "feature_spec_sha256"
                ]
            ),

        "scorer_spec_sha256":
            str(
                checkpoint[
                    "scorer_spec_sha256"
                ]
            ),

        "training_spec_sha256":
            str(
                checkpoint[
                    "training_spec_sha256"
                ]
            ),
    }


WEBQSP_SCORER_FREEZE = (
    selected_checkpoint_metadata(
        webqsp_ckpt_obj,
        webqsp_ckpt_sha
    )
)


CWQ_SCORER_FREEZE = (
    selected_checkpoint_metadata(
        cwq_ckpt_obj,
        cwq_ckpt_sha
    )
)


assert WEBQSP_SCORER_FREEZE[
    "hidden_dim"
] == 32


assert CWQ_SCORER_FREEZE[
    "hidden_dim"
] == 64


assert WEBQSP_SCORER_FREEZE[
    "seed"
] == 42


assert CWQ_SCORER_FREEZE[
    "seed"
] == 42


assert WEBQSP_SCORER_FREEZE[
    "loss_name"
] == "branch_bce"


assert CWQ_SCORER_FREEZE[
    "loss_name"
] == "branch_bce"


print(
    "\nSelected original scorer metadata: PASSED"
)


# ======================================================================
# 17. JSON-SAFE CONVERSION
# ======================================================================

def json_safe_15b(
    value
):

    if isinstance(
        value,
        dict
    ):

        return {
            str(
                key
            ):
                json_safe_15b(
                    item
                )

            for key, item
            in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
        )
    ):

        return [
            json_safe_15b(
                item
            )
            for item in value
        ]


    if isinstance(
        value,
        np.integer
    ):

        return int(
            value
        )


    if isinstance(
        value,
        np.floating
    ):

        value = float(
            value
        )

        if math.isnan(
            value
        ):

            return None

        return value


    if isinstance(
        value,
        np.bool_
    ):

        return bool(
            value
        )


    if isinstance(
        value,
        float
    ):

        if math.isnan(
            value
        ):

            return None


    return value


def df_records_json_safe(
    df
):

    return [
        json_safe_15b(
            row
        )

        for row
        in df.to_dict(
            orient="records"
        )
    ]


# ======================================================================
# 18. FINAL DEVELOPMENT FREEZE PAYLOAD
# ======================================================================
#
# THIS is the immutable configuration to carry to TEST.
#
# Note carefully:
#
# - Final scorer is the ORIGINAL selected checkpoint from Cell 9C/B4.
# - The full-v2 retrain in Cell 15A was ONLY an ablation-control model.
# - Ablation models are NOT promoted to the final method.
# ======================================================================

FINAL_AFP_CONFIG = {
    "framework":
        "AdaPruner-KGQA",

    "component":
        "Adaptive Frontier Pruning",

    "development_freeze_version":
        "afp_final_development_freeze_v1",

    # --------------------------------------------------------------
    # Architectural scope
    # --------------------------------------------------------------

    "scope": {
        "position":
            "inside_RoG_relation_constrained_retrieval",

        "relation_planner_changed":
            False,

        "reasoner_changed":
            False,

        "unrestricted_KG_search":
            False,

        "intermediate_hop_pruning_only":
            True,

        "final_hop_protection":
            True,

        "singleton_bypass":
            True,
    },

    # --------------------------------------------------------------
    # Representation
    # --------------------------------------------------------------

    "features": {
        "version":
            AFP_RUNTIME_FEATURE_VERSION,

        "dimension":
            27,

        "families": [
            "semantic",
            "path",
            "structural",
            "progress",
        ],

        "raw_MID_embedding":
            False,

        "external_entity_resolver":
            False,

        "future_candidate_neighborhood_features":
            False,

        "final_decision":
            "retain_full_feature_v2",
    },

    # --------------------------------------------------------------
    # Scorers
    # --------------------------------------------------------------

    "scorer": {
        "architecture":
            "AFPScorer_MLP",

        "activation":
            "ReLU",

        "dropout":
            0.0,

        "loss":
            "branch_bce",

        "deployment_seed":
            42,

        "webqsp":
            WEBQSP_SCORER_FREEZE,

        "cwq":
            CWQ_SCORER_FREEZE,

        "selected_checkpoint_source":
            "original_Cell9C_B4_selected_checkpoint",

        "Cell15A_full_retrain_promoted":
            False,
    },

    # --------------------------------------------------------------
    # Selector
    # --------------------------------------------------------------

    "selector": {
        "webqsp":
            FINAL_SELECTOR_PARAMS[
                "webqsp"
            ],

        "cwq":
            FINAL_SELECTOR_PARAMS[
                "cwq"
            ],

        "probability":
            "softmax(logits / T)",

        "uncertainty":
            "normalized_entropy",

        "gamma_rule":
            "gamma_min + uncertainty * (1 - gamma_min)",

        "budget_rule":
            (
                "smallest cumulative-probability top-B "
                "reaching gamma"
            ),

        "fully_tied_behavior":
            "retain_all",

        "cutoff_tie_behavior":
            "expand_cutoff_ties",

        "one_time_boundary_expansion_used":
            True,

        "further_hyperparameter_search_allowed":
            False,
    },

    # --------------------------------------------------------------
    # Training / leakage controls
    # --------------------------------------------------------------

    "leakage_controls": {
        "scorer_training_labels":
            "train_only",

        "selector_tuning":
            "validation_only",

        "test_used_for_training":
            False,

        "test_used_for_model_selection":
            False,

        "test_used_for_hyperparameter_tuning":
            False,

        "test_examples_accessed_during_development":
            False,
    },

    # --------------------------------------------------------------
    # Metrics
    # --------------------------------------------------------------

    "evaluation": {
        "primary_search_cost":
            "edges_examined",

        "SSR":
            "1 - E_method / E_RoG",

        "answer_retention":
            (
                "method_reachable_questions / "
                "RoG_reachable_questions"
            ),

        "AR_selection_floor":
            0.99,

        "active_hop_rows_distinct_from_active_prefixes":
            True,

        "candidate_branches_distinct_from_unique_entities":
            True,
    },

    # --------------------------------------------------------------
    # Development evidence
    # --------------------------------------------------------------

    "validation_selected_AFP": {
        "webqsp": {
            "AR":
                float(
                    get_single_row(
                        webqsp_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "answer_retention"
                    ]
                ),

            "SSR":
                float(
                    get_single_row(
                        webqsp_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "ssr"
                    ]
                ),
        },

        "cwq": {
            "AR":
                float(
                    get_single_row(
                        cwq_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "answer_retention"
                    ]
                ),

            "SSR":
                float(
                    get_single_row(
                        cwq_comparison_14b_df,
                        "method",
                        "AFP"
                    )[
                        "ssr"
                    ]
                ),
        },
    },

    "validation_fixed_top_b_dominance":
        {
            "webqsp":
                WEBQSP_DOMINANCE,

            "cwq":
                CWQ_DOMINANCE,

            "interpretation":
                (
                    "Validation evidence must not be reported "
                    "as showing AFP outperforming Fixed Top-B "
                    "when Fixed Top-B Pareto-dominates AFP."
                ),
        },

    "feature_ablation_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_feature_ablation_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_feature_ablation_df
            ),
    },

    "selector_ablation_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_selector_ablation_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_selector_ablation_df
            ),
    },

    "component_control_results": {
        "webqsp":
            df_records_json_safe(
                webqsp_component_evidence_df
            ),

        "cwq":
            df_records_json_safe(
                cwq_component_evidence_df
            ),
    },

    # --------------------------------------------------------------
    # Freeze state
    # --------------------------------------------------------------

    "development_status": {
        "feature_development_complete":
            True,

        "scorer_development_complete":
            True,

        "selector_development_complete":
            True,

        "controlled_validation_comparison_complete":
            True,

        "feature_ablations_complete":
            True,

        "selector_component_ablations_complete":
            True,

        "no_further_development_changes_before_test":
            True,

        "final_AFP_configuration_frozen":
            True,

        "test_evaluation_started":
            False,
    },
}


# ======================================================================
# 19. CANONICAL FREEZE HASH
# ======================================================================

FINAL_AFP_CONFIG_SAFE = (
    json_safe_15b(
        FINAL_AFP_CONFIG
    )
)


FREEZE_CANONICAL_JSON = (
    json.dumps(
        FINAL_AFP_CONFIG_SAFE,
        sort_keys=True,
        separators=(
            ",",
            ":"
        ),
        ensure_ascii=False
    )
)


FINAL_AFP_FREEZE_SHA256 = (
    hashlib.sha256(
        FREEZE_CANONICAL_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nFinal AFP development-freeze SHA256:"
)

print(
    " ",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 20. SAVE FINAL FREEZE ARTIFACT
# ======================================================================

FINAL_FREEZE_JSON_PATH = (
    CELL15B_DIR
    / "final_afp_development_freeze.json"
)


FINAL_FREEZE_SHA_PATH = (
    CELL15B_DIR
    / "final_afp_development_freeze.sha256"
)


with open(
    FINAL_FREEZE_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_AFP_CONFIG_SAFE,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    FINAL_FREEZE_SHA_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        FINAL_AFP_FREEZE_SHA256
        +
        "\n"
    )


# ======================================================================
# 21. SAVE ABLATION TABLES
# ======================================================================

WEBQSP_SELECTOR_ABLATION_CSV = (
    CELL15B_DIR
    / "webqsp_selector_component_ablation.csv"
)


CWQ_SELECTOR_ABLATION_CSV = (
    CELL15B_DIR
    / "cwq_selector_component_ablation.csv"
)


WEBQSP_COMPONENT_EVIDENCE_CSV = (
    CELL15B_DIR
    / "webqsp_complete_component_evidence.csv"
)


CWQ_COMPONENT_EVIDENCE_CSV = (
    CELL15B_DIR
    / "cwq_complete_component_evidence.csv"
)


webqsp_selector_ablation_df.to_csv(
    WEBQSP_SELECTOR_ABLATION_CSV,
    index=False
)


cwq_selector_ablation_df.to_csv(
    CWQ_SELECTOR_ABLATION_CSV,
    index=False
)


webqsp_component_evidence_df.to_csv(
    WEBQSP_COMPONENT_EVIDENCE_CSV,
    index=False
)


cwq_component_evidence_df.to_csv(
    CWQ_COMPONENT_EVIDENCE_CSV,
    index=False
)


# ======================================================================
# 22. FINAL FREEZE SOFTWARE GATES
# ======================================================================

assert FINAL_AFP_CONFIG_SAFE[
    "development_status"
][
    "final_AFP_configuration_frozen"
] is True


assert FINAL_AFP_CONFIG_SAFE[
    "development_status"
][
    "test_evaluation_started"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "further_hyperparameter_search_allowed"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "scorer"
][
    "Cell15A_full_retrain_promoted"
] is False


assert FINAL_AFP_CONFIG_SAFE[
    "features"
][
    "dimension"
] == 27


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "webqsp"
] == {
    "T": 0.05,
    "gamma_min": 0.1,
}


assert FINAL_AFP_CONFIG_SAFE[
    "selector"
][
    "cwq"
] == {
    "T": 0.1,
    "gamma_min": 0.1,
}


assert (
    sha256_file(
        Path(
            "/kaggle/working/"
            "step3_rq2_dev_v1/"
            "07_final_scorer/"
            "webqsp_afp_scorer_selected.pt"
        )
    )
    ==
    webqsp_ckpt_sha
)


assert (
    sha256_file(
        Path(
            "/kaggle/working/"
            "step3_rq2_dev_v1/"
            "07_final_scorer/"
            "cwq_afp_scorer_selected.pt"
        )
    )
    ==
    cwq_ckpt_sha
)


print(
    "\nFinal checkpoint identity gate: PASSED"
)


# ======================================================================
# 23. DEVELOPMENT FREEZE FLAGS
# ======================================================================

CELL15B_COMPLETE = True

AFP_DEVELOPMENT_FROZEN = True

FINAL_AFP_CONFIG_FROZEN = (
    FINAL_AFP_CONFIG_SAFE
)


# ======================================================================
# 24. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "=== RQ2 CELL 15B: "
    "SELECTOR ABLATIONS + FINAL AFP DEVELOPMENT FREEZE COMPLETE ==="
)

print(
    "=" * 126
)


print(
    "\nFINAL AFP CONFIGURATION"
)


print(
    "\nFeature representation:"
)

print(
    "  Feature-v2"
)

print(
    "  Dimension: 27"
)

print(
    "  Semantic + Path + Structural + Progress"
)


print(
    "\nSelected scorer checkpoints:"
)

print(
    "  WebQSP:"
)

print(
    "   H =",
    WEBQSP_SCORER_FREEZE[
        "hidden_dim"
    ]
)

print(
    "   SHA =",
    webqsp_ckpt_sha
)


print(
    "  CWQ:"
)

print(
    "   H =",
    CWQ_SCORER_FREEZE[
        "hidden_dim"
    ]
)

print(
    "   SHA =",
    cwq_ckpt_sha
)


print(
    "\nFinal AFP selector:"
)

print(
    "  WebQSP:",
    FINAL_SELECTOR_PARAMS[
        "webqsp"
    ]
)

print(
    "  CWQ:   ",
    FINAL_SELECTOR_PARAMS[
        "cwq"
    ]
)


print(
    "\nPolicy:"
)

print(
    "  intermediate pruning only"
)

print(
    "  final-hop protection = TRUE"
)

print(
    "  singleton bypass = TRUE"
)

print(
    "  fully tied scores = retain all"
)

print(
    "  cutoff ties = expand ties"
)


print(
    "\nDevelopment evidence status:"
)

print(
    "  Controlled validation comparison: COMPLETE"
)

print(
    "  Feature-family ablations:          COMPLETE"
)

print(
    "  Selector/component ablations:      COMPLETE"
)

print(
    "  Further hyperparameter search:     PROHIBITED"
)


print(
    "\nValidation caveat explicitly frozen:"
)

print(
    "  WebQSP Fixed Top-B Pareto-dominates AFP:",
    WEBQSP_DOMINANCE[
        "fixed_top_b_pareto_dominates_afp"
    ]
)

print(
    "  CWQ Fixed Top-B Pareto-dominates AFP:",
    CWQ_DOMINANCE[
        "fixed_top_b_pareto_dominates_afp"
    ]
)


print(
    "\nLeakage status:"
)

print(
    "  Train labels for scorer: TRAIN only"
)

print(
    "  Selector tuning: VALIDATION only"
)

print(
    "  TEST examples accessed: NO"
)

print(
    "  TEST evaluation started: NO"
)


print(
    "\nFREEZE STATUS:"
)

print(
    "  AFP_DEVELOPMENT_FROZEN = TRUE"
)

print(
    "  No further development changes before TEST."
)


print(
    "\nFreeze SHA256:"
)

print(
    " ",
    FINAL_AFP_FREEZE_SHA256
)


print(
    "\nArtifacts:"
)

print(
    " ",
    WEBQSP_SELECTOR_ABLATION_CSV
)

print(
    " ",
    CWQ_SELECTOR_ABLATION_CSV
)

print(
    " ",
    WEBQSP_COMPONENT_EVIDENCE_CSV
)

print(
    " ",
    CWQ_COMPONENT_EVIDENCE_CSV
)

print(
    " ",
    FINAL_FREEZE_JSON_PATH
)

print(
    " ",
    FINAL_FREEZE_SHA_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "FROZEN TEST evaluation."
)

print(
    "No tuning, retraining, feature changes, "
    "or selector changes are allowed after this point."
)

Cell 15B prerequisites: PASSED
Feature version: afp_features_v2_masked_entity_semantics
Cell 15B output: /kaggle/working/step3_rq2_dev_v1/14_selector_ablations_and_final_freeze

Final development-selected selector:
  WebQSP: {'T': 0.05, 'gamma_min': 0.1}
  CWQ:    {'T': 0.1, 'gamma_min': 0.1}

Selector/component ablations:
  full_afp
  no_uncertainty_adaptation
  no_temperature_scaling

No ablation-specific tuning: YES

RUNNING SELECTOR / COMPONENT ABLATIONS


webqsp selector ablations:   0%|          | 0/246 [00:00<?, ?it/s]


WEBQSP selector ablations completed in 0.03 min


cwq selector ablations:   0%|          | 0/3519 [00:00<?, ?it/s]


CWQ selector ablations completed in 2.18 min
WebQSP full AFP Cell-14B fidelity: PASSED
CWQ full AFP Cell-14B fidelity: PASSED

WEBQSP SELECTOR / COMPONENT ABLATIONS
                  variant  answer_retention      ssr  edges_examined  active_prefixes  candidate_branches  decision_hops  avg_requested_budget  avg_uncertainty  avg_gamma  fully_tied_decisions  delta_ar_vs_full  delta_ssr_vs_full  delta_edges_vs_full
                 full_afp          1.000000 0.010272          338018             2293                6659            147              9.891156         0.894528   0.905075                    97          0.000000           0.000000                    0
no_uncertainty_adaptation          1.000000 0.017855          335428             2110                6012            147              7.306122         0.894528   0.693878                    97          0.000000           0.007584                -2590
   no_temperature_scaling          1.000000 0.000123          341484             

## Final frozen-test RQ1 profiling


In [5]:
# ======================================================================
# PURPOSE
# -------
# Final TEST evaluation for:
#
# RQ1:
# "Where and to what extent does search-space growth occur during
#  RoG's relation-constrained graph retrieval?"
#
# ALSO performs POST-FREEZE oracle analysis:
#   - doomed-prefix work
#   - oracle-prunable intermediate branches
#   - oracle-avoidable downstream edges
#   - final-hop-protected doomed candidates
#
# IMPORTANT
# ---------
# AFP is ALREADY FROZEN.
#
# This cell:
#   - does NOT tune anything
#   - does NOT train anything
#   - does NOT change planner/scorer/selector
#   - uses TEST gold only for POST-HOC reachability/oracle analysis
#
# REQUIRED INPUT
# --------------
# Exact RoG TEST planning outputs with schema:
#
#   id
#   question
#   q_entity
#   a_entity
#   graph
#   predicted_paths
#
# The cell auto-detects already-materialized TEST planning rows/files.
#
# It WILL NOT invent or regenerate planner outputs using a new
# implementation.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import hashlib
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE PREREQUISITES
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "FINAL_AFP_CONFIG_FROZEN",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required frozen-development objects:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA
), (
    "AFP freeze SHA mismatch. "
    "STOP before TEST evaluation."
)


print("Cell 16 freeze gate: PASSED")
print("AFP freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FINAL_TEST_ROOT = (
    ROOT
    / "15_final_frozen_test"
)

RQ1_TEST_DIR = (
    FINAL_TEST_ROOT
    / "rq1_profiling"
)

RQ1_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. HASH HELPERS
# ======================================================================

def sha256_file_16(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def rows_sha256_16(rows):

    canonical = json.dumps(
        rows,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )

    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()


# ======================================================================
# 4. TEST PLANNING SCHEMA
# ======================================================================

REQUIRED_PLAN_FIELDS_16 = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
}


def planning_rows_valid_16(rows):

    if not isinstance(rows, list):

        return False

    if len(rows) == 0:

        return False

    sample_indices = sorted(
        set(
            [
                0,
                len(rows) // 2,
                len(rows) - 1,
            ]
        )
    )

    for idx in sample_indices:

        rec = rows[idx]

        if not isinstance(rec, dict):

            return False

        if not REQUIRED_PLAN_FIELDS_16.issubset(
            rec.keys()
        ):

            return False

    return True


# ======================================================================
# 5. LOAD JSONL
# ======================================================================

def read_jsonl_16(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            rows.append(
                json.loads(line)
            )

    return rows


# ======================================================================
# 6. DISCOVER ALREADY-MATERIALIZED TEST PLANNING OUTPUT
# ======================================================================

def discover_test_planning_16(
    dataset_name
):

    dataset_name = dataset_name.lower()

    # --------------------------------------------------------------
    # First: surviving globals.
    # --------------------------------------------------------------

    candidate_global_names = [
        f"{dataset_name}_test_plan_rows",
        f"{dataset_name}_test_planning_rows",
        f"{dataset_name}_planning_test_rows",
        f"{dataset_name}_test_plans",
    ]


    valid_globals = []


    for name in candidate_global_names:

        if name not in globals():

            continue

        value = globals()[name]

        if planning_rows_valid_16(
            value
        ):

            valid_globals.append(
                (
                    name,
                    value
                )
            )


    if len(valid_globals) == 1:

        name, rows = valid_globals[0]

        print(
            f"{dataset_name.upper()} TEST planning "
            f"loaded from global: {name}"
        )

        return (
            rows,
            None,
            rows_sha256_16(rows)
        )


    if len(valid_globals) > 1:

        hashes = {
            rows_sha256_16(rows)
            for _, rows in valid_globals
        }

        assert len(hashes) == 1, (
            f"{dataset_name}: multiple non-identical "
            "TEST planning globals found."
        )

        name, rows = valid_globals[0]

        print(
            f"{dataset_name.upper()}: multiple identical "
            f"globals found; using {name}"
        )

        return (
            rows,
            None,
            rows_sha256_16(rows)
        )


    # --------------------------------------------------------------
    # Second: frozen planning JSONL artifacts under /kaggle/working.
    # --------------------------------------------------------------

    candidates = []


    for path in Path(
        "/kaggle/working"
    ).rglob("*.jsonl"):

        lower = str(
            path
        ).lower()

        if dataset_name not in lower:

            continue

        if "test" not in lower:

            continue


        try:

            rows = read_jsonl_16(
                path
            )

        except Exception:

            continue


        if not planning_rows_valid_16(
            rows
        ):

            continue


        candidates.append(
            {
                "path":
                    path,

                "rows":
                    rows,

                "sha256":
                    sha256_file_16(
                        path
                    ),
            }
        )


    assert len(candidates) >= 1, (
        f"\n{dataset_name.upper()} frozen TEST planning "
        "artifact was not found.\n\n"
        "STOP HERE.\n"
        "Do NOT generate test relation plans with a new or "
        "approximate planner implementation.\n\n"
        "Materialize TEST plans using the EXACT frozen RoG "
        "planner-generation code/configuration used for validation, "
        "then rerun Cell 16."
    )


    if len(candidates) > 1:

        content_hashes = {
            rows_sha256_16(
                row["rows"]
            )
            for row in candidates
        }

        assert len(
            content_hashes
        ) == 1, (
            f"{dataset_name}: multiple non-identical TEST "
            "planning artifacts found. STOP."
        )


    chosen = sorted(
        candidates,
        key=lambda x:
            str(
                x["path"]
            )
    )[0]


    print(
        f"{dataset_name.upper()} TEST planning:"
    )

    print(
        " ",
        chosen[
            "path"
        ]
    )

    print(
        " SHA256:",
        chosen[
            "sha256"
        ]
    )


    return (
        chosen[
            "rows"
        ],
        chosen[
            "path"
        ],
        chosen[
            "sha256"
        ]
    )


(
    webqsp_test_plan_rows,
    WEBQSP_TEST_PLAN_PATH,
    WEBQSP_TEST_PLAN_SHA
) = discover_test_planning_16(
    "webqsp"
)


(
    cwq_test_plan_rows,
    CWQ_TEST_PLAN_PATH,
    CWQ_TEST_PLAN_SHA
) = discover_test_planning_16(
    "cwq"
)


# ======================================================================
# 7. TEST / VALIDATION DISJOINTNESS GATE
# ======================================================================

def id_set_16(rows):

    return {
        str(
            rec["id"]
        )
        for rec in rows
    }


assert not (
    id_set_16(
        webqsp_test_plan_rows
    )
    &
    id_set_16(
        webqsp_val_plan_rows
    )
), (
    "WebQSP TEST IDs overlap validation IDs."
)


assert not (
    id_set_16(
        cwq_test_plan_rows
    )
    &
    id_set_16(
        cwq_val_plan_rows
    )
), (
    "CWQ TEST IDs overlap validation IDs."
)


print(
    "\nTEST / validation ID-disjointness: PASSED"
)

print(
    "WebQSP TEST questions:",
    len(
        webqsp_test_plan_rows
    )
)

print(
    "CWQ TEST questions:",
    len(
        cwq_test_plan_rows
    )
)


# ======================================================================
# 8. EXACT GRAPH HELPERS
# ======================================================================

def matching_neighbors_16(
    adjacency,
    entity,
    relation
):

    return [
        neighbor
        for neighbor, edge_relation
        in adjacency.get(
            entity,
            {}
        ).items()
        if edge_relation == relation
    ]


def degree_16(
    adjacency,
    entity
):

    return len(
        adjacency.get(
            entity,
            {}
        )
    )


# ======================================================================
# 9. QUESTION-LOCAL ORACLE FUNCTIONS
# ======================================================================
#
# POST-FREEZE only.
#
# suffix_reachable(entity, suffix):
#   whether exact remaining relation sequence can reach gold.
#
# downstream_cost(entity, suffix):
#   exact future edge examinations RoG would perform from ONE prefix.
#
# Repeated prefixes are not merged because RoG traversal preserves
# path-prefix multiplicity.
# ======================================================================

def make_oracle_functions_16(
    adjacency,
    gold_answers
):

    gold_set = {
        str(x)
        for x in gold_answers
    }


    reachable_cache = {}

    cost_cache = {}


    def suffix_reachable(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suffix
        )

        key = (
            entity,
            suffix
        )

        if key in reachable_cache:

            return reachable_cache[
                key
            ]


        if len(
            suffix
        ) == 0:

            result = (
                entity
                in
                gold_set
            )

            reachable_cache[
                key
            ] = result

            return result


        relation = suffix[0]


        result = any(
            suffix_reachable(
                neighbor,
                suffix[
                    1:
                ]
            )

            for neighbor
            in matching_neighbors_16(
                adjacency,
                entity,
                relation
            )
        )


        reachable_cache[
            key
        ] = bool(
            result
        )


        return bool(
            result
        )


    def downstream_cost(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suffix
        )

        key = (
            entity,
            suffix
        )

        if key in cost_cache:

            return cost_cache[
                key
            ]


        if len(
            suffix
        ) == 0:

            cost_cache[
                key
            ] = 0

            return 0


        relation = suffix[0]


        total = degree_16(
            adjacency,
            entity
        )


        for neighbor in matching_neighbors_16(
            adjacency,
            entity,
            relation
        ):

            total += downstream_cost(
                neighbor,
                suffix[
                    1:
                ]
            )


        total = int(
            total
        )


        cost_cache[
            key
        ] = total


        return total


    return (
        suffix_reachable,
        downstream_cost
    )


# ======================================================================
# 10. EXACT RQ1 PROFILER
# ======================================================================

def profile_rq1_dataset_16(
    dataset_name,
    planning_rows
):

    hop_rows = []
    plan_rows = []
    question_rows = []


    oracle_totals = {
        "total_edges":
            0,

        "doomed_prefix_edges":
            0,

        "candidate_branches":
            0,

        "intermediate_candidate_branches":
            0,

        "oracle_prunable_intermediate_branches":
            0,

        "doomed_final_protected_branches":
            0,

        "oracle_avoidable_downstream_edges":
            0,
    }


    for q_index, rec in enumerate(
        tqdm(
            planning_rows,
            desc=f"{dataset_name} final TEST RQ1"
        )
    ):

        qid = str(
            rec["id"]
        )


        graph = rec[
            "graph"
        ]


        adjacency = (
            build_exact_rog_adjacency(
                graph
            )
        )


        topics = (
            as_entity_list(
                rec["q_entity"]
            )
        )


        gold = (
            as_entity_list(
                rec["a_entity"]
            )
        )


        plans = (
            normalize_relation_plans(
                rec[
                    "predicted_paths"
                ]
            )
        )


        (
            suffix_reachable,
            downstream_cost
        ) = make_oracle_functions_16(
            adjacency,
            gold
        )


        question_edges = 0
        question_candidates = 0
        question_active_prefixes = 0

        question_reachable = False

        q_unique_expanded_entities = set()

        q_peak_frontier = len(
            topics
        )

        q_has_intermediate_active = False
        q_has_decision = False


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                plan_rows.append(
                    {
                        "dataset":
                            dataset_name,

                        "question_id":
                            qid,

                        "question_index":
                            q_index,

                        "plan_index":
                            plan_index,

                        "plan_length":
                            0,

                        "empty_plan":
                            True,

                        "active_hop_rows":
                            0,

                        "active_prefixes":
                            0,

                        "edges_examined":
                            0,

                        "candidate_branches":
                            0,

                        "retrieved_paths":
                            0,

                        "peak_frontier":
                            len(
                                topics
                            ),

                        "reachable":
                            False,

                        "eligible_intermediate":
                            False,

                        "has_decision":
                            False,

                        "oracle_prunable_intermediate":
                            0,

                        "oracle_avoidable_downstream_edges":
                            0,
                    }
                )

                continue


            active = [
                (
                    str(
                        entity
                    ),
                )
                for entity in topics
            ]


            plan_active_rows = 0
            plan_active_prefixes = 0
            plan_edges = 0
            plan_candidates = 0

            plan_peak = len(
                active
            )

            plan_eligible = False
            plan_decision = False

            plan_oracle_prunable = 0
            plan_oracle_avoidable = 0


            L = len(
                plan
            )


            for hop, target_relation in enumerate(
                plan
            ):

                if not active:

                    break


                plan_active_rows += 1

                plan_active_prefixes += len(
                    active
                )


                q_peak_frontier = max(
                    q_peak_frontier,
                    len(
                        active
                    )
                )


                plan_peak = max(
                    plan_peak,
                    len(
                        active
                    )
                )


                # ------------------------------------------------------
                # Current active prefixes
                # ------------------------------------------------------

                current_unique_entities = {
                    prefix[-1]
                    for prefix in active
                }


                q_unique_expanded_entities.update(
                    current_unique_entities
                )


                edges_this_hop = sum(
                    degree_16(
                        adjacency,
                        prefix[-1]
                    )
                    for prefix in active
                )


                plan_edges += edges_this_hop

                question_edges += edges_this_hop

                oracle_totals[
                    "total_edges"
                ] += edges_this_hop


                # ------------------------------------------------------
                # Doomed-prefix current work
                # ------------------------------------------------------

                remaining_from_active = (
                    plan[
                        hop:
                    ]
                )


                doomed_prefix_edges = 0


                for prefix in active:

                    endpoint = prefix[
                        -1
                    ]


                    if not suffix_reachable(
                        endpoint,
                        remaining_from_active
                    ):

                        doomed_prefix_edges += (
                            degree_16(
                                adjacency,
                                endpoint
                            )
                        )


                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges


                # ------------------------------------------------------
                # Exact relation-constrained candidates
                # ------------------------------------------------------

                candidates = []


                for parent_index, prefix in enumerate(
                    active
                ):

                    endpoint = prefix[
                        -1
                    ]


                    for neighbor, relation in (
                        adjacency.get(
                            endpoint,
                            {}
                        ).items()
                    ):

                        if relation != target_relation:

                            continue


                        candidates.append(
                            prefix
                            +
                            (
                                neighbor,
                            )
                        )


                candidate_count = len(
                    candidates
                )


                plan_candidates += candidate_count

                question_candidates += (
                    candidate_count
                )


                oracle_totals[
                    "candidate_branches"
                ] += candidate_count


                unique_candidate_entities = {
                    prefix[-1]
                    for prefix in candidates
                }


                duplicate_ratio = (
                    1.0
                    -
                    len(
                        unique_candidate_entities
                    )
                    /
                    candidate_count

                    if candidate_count > 0

                    else 0.0
                )


                rho = (
                    candidate_count
                    /
                    len(
                        active
                    )
                )


                is_intermediate = (
                    hop
                    <
                    L - 1
                )


                if is_intermediate:

                    q_has_intermediate_active = True
                    plan_eligible = True

                    oracle_totals[
                        "intermediate_candidate_branches"
                    ] += candidate_count


                decision = (
                    is_intermediate
                    and
                    candidate_count > 1
                )


                if decision:

                    q_has_decision = True
                    plan_decision = True


                # ------------------------------------------------------
                # Oracle analysis AFTER current candidate generation.
                #
                # Current-hop edge cost has already been paid.
                # ------------------------------------------------------

                suffix_after_candidate = (
                    plan[
                        hop + 1:
                    ]
                )


                oracle_prunable_this_hop = 0
                oracle_avoidable_this_hop = 0
                downstream_work_all_candidates = 0


                if is_intermediate:

                    for candidate in candidates:

                        endpoint = candidate[
                            -1
                        ]


                        future_cost = (
                            downstream_cost(
                                endpoint,
                                suffix_after_candidate
                            )
                        )


                        downstream_work_all_candidates += (
                            future_cost
                        )


                        feasible = (
                            suffix_reachable(
                                endpoint,
                                suffix_after_candidate
                            )
                        )


                        if not feasible:

                            oracle_prunable_this_hop += 1

                            oracle_avoidable_this_hop += (
                                future_cost
                            )


                    oracle_totals[
                        "oracle_prunable_intermediate_branches"
                    ] += oracle_prunable_this_hop


                    oracle_totals[
                        "oracle_avoidable_downstream_edges"
                    ] += oracle_avoidable_this_hop


                    plan_oracle_prunable += (
                        oracle_prunable_this_hop
                    )


                    plan_oracle_avoidable += (
                        oracle_avoidable_this_hop
                    )


                else:

                    # Final candidate branches are protected.
                    doomed_final = sum(
                        1
                        for candidate
                        in candidates
                        if not suffix_reachable(
                            candidate[-1],
                            ()
                        )
                    )


                    oracle_totals[
                        "doomed_final_protected_branches"
                    ] += doomed_final


                hop_rows.append(
                    {
                        "dataset":
                            dataset_name,

                        "question_id":
                            qid,

                        "question_index":
                            q_index,

                        "plan_index":
                            plan_index,

                        "plan_length":
                            L,

                        "hop":
                            hop,

                        "is_intermediate":
                            is_intermediate,

                        "active_prefixes":
                            len(
                                active
                            ),

                        "unique_active_entities":
                            len(
                                current_unique_entities
                            ),

                        "edges_examined":
                            int(
                                edges_this_hop
                            ),

                        "candidate_branches":
                            int(
                                candidate_count
                            ),

                        "unique_candidate_entities":
                            len(
                                unique_candidate_entities
                            ),

                        "duplicate_endpoint_ratio":
                            float(
                                duplicate_ratio
                            ),

                        "rho":
                            float(
                                rho
                            ),

                        "decision_hop":
                            bool(
                                decision
                            ),

                        "oracle_prunable_candidates":
                            int(
                                oracle_prunable_this_hop
                            ),

                        "oracle_avoidable_downstream_edges":
                            int(
                                oracle_avoidable_this_hop
                            ),

                        "downstream_work_all_candidates":
                            int(
                                downstream_work_all_candidates
                            ),
                    }
                )


                active = candidates


                q_peak_frontier = max(
                    q_peak_frontier,
                    len(
                        active
                    )
                )


                plan_peak = max(
                    plan_peak,
                    len(
                        active
                    )
                )


            answer_set = {
                str(x)
                for x in gold
            }


            plan_reachable = any(
                prefix[-1]
                in
                answer_set

                for prefix in active
            )


            if plan_reachable:

                question_reachable = True


            plan_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "question_id":
                        qid,

                    "question_index":
                        q_index,

                    "plan_index":
                        plan_index,

                    "plan_length":
                        len(
                            plan
                        ),

                    "empty_plan":
                        False,

                    "active_hop_rows":
                        int(
                            plan_active_rows
                        ),

                    "active_prefixes":
                        int(
                            plan_active_prefixes
                        ),

                    "edges_examined":
                        int(
                            plan_edges
                        ),

                    "candidate_branches":
                        int(
                            plan_candidates
                        ),

                    "retrieved_paths":
                        int(
                            len(
                                active
                            )
                        ),

                    "peak_frontier":
                        int(
                            plan_peak
                        ),

                    "reachable":
                        bool(
                            plan_reachable
                        ),

                    "eligible_intermediate":
                        bool(
                            plan_eligible
                        ),

                    "has_decision":
                        bool(
                            plan_decision
                        ),

                    "oracle_prunable_intermediate":
                        int(
                            plan_oracle_prunable
                        ),

                    "oracle_avoidable_downstream_edges":
                        int(
                            plan_oracle_avoidable
                        ),
                }
            )


            question_active_prefixes += (
                plan_active_prefixes
            )


        question_rows.append(
            {
                "dataset":
                    dataset_name,

                "question_id":
                    qid,

                "question_index":
                    q_index,

                "predicted_plan_count":
                    len(
                        plans
                    ),

                "edges_examined":
                    int(
                        question_edges
                    ),

                "candidate_branches":
                    int(
                        question_candidates
                    ),

                "active_prefixes":
                    int(
                        question_active_prefixes
                    ),

                "unique_expanded_entities":
                    int(
                        len(
                            q_unique_expanded_entities
                        )
                    ),

                "peak_frontier":
                    int(
                        q_peak_frontier
                    ),

                "reachable":
                    bool(
                        question_reachable
                    ),

                "eligible_intermediate":
                    bool(
                        q_has_intermediate_active
                    ),

                "has_decision":
                    bool(
                        q_has_decision
                    ),
            }
        )


    hop_df = pd.DataFrame(
        hop_rows
    )


    plan_df = pd.DataFrame(
        plan_rows
    )


    question_df = pd.DataFrame(
        question_rows
    )


    return (
        hop_df,
        plan_df,
        question_df,
        oracle_totals
    )


# ======================================================================
# 11. VALIDATION ORACLE SOFTWARE GATE
# ======================================================================
#
# Before exposing TEST oracle results, reproduce the previously verified
# validation oracle totals.
# ======================================================================

EXPECTED_VAL_ORACLE_16 = {
    "webqsp": {
        "total_edges":
            341526,

        "candidate_branches":
            7983,

        "doomed_prefix_edges":
            159039,

        "intermediate_candidate_branches":
            1707,

        "oracle_prunable_intermediate_branches":
            1340,

        "doomed_final_protected_branches":
            4882,

        "oracle_avoidable_downstream_edges":
            11717,
    },

    "cwq": {
        "total_edges":
            5257272,

        "candidate_branches":
            247161,

        "doomed_prefix_edges":
            3197488,

        "intermediate_candidate_branches":
            41576,

        "oracle_prunable_intermediate_branches":
            34431,

        "doomed_final_protected_branches":
            195499,

        "oracle_avoidable_downstream_edges":
            1176294,
    },
}


def oracle_validation_gate_16(
    dataset_name,
    validation_rows
):

    (
        _hop,
        _plan,
        _question,
        oracle
    ) = profile_rq1_dataset_16(
        f"{dataset_name}_validation_gate",
        validation_rows
    )


    expected = (
        EXPECTED_VAL_ORACLE_16[
            dataset_name
        ]
    )


    for key, expected_value in (
        expected.items()
    ):

        actual = int(
            oracle[
                key
            ]
        )


        assert actual == int(
            expected_value
        ), (
            f"{dataset_name} oracle gate mismatch "
            f"for {key}: {actual} != {expected_value}"
        )


    print(
        f"{dataset_name.upper()} validation oracle "
        "software gate: PASSED"
    )


oracle_validation_gate_16(
    "webqsp",
    webqsp_val_plan_rows
)


oracle_validation_gate_16(
    "cwq",
    cwq_val_plan_rows
)


print(
    "\nRQ1 oracle implementation fidelity: PASSED"
)


# ======================================================================
# 12. FINAL TEST PROFILING
# ======================================================================

(
    webqsp_test_hop_df,
    webqsp_test_plan_df,
    webqsp_test_question_df,
    webqsp_test_oracle
) = profile_rq1_dataset_16(
    "webqsp",
    webqsp_test_plan_rows
)


(
    cwq_test_hop_df,
    cwq_test_plan_df,
    cwq_test_question_df,
    cwq_test_oracle
) = profile_rq1_dataset_16(
    "cwq",
    cwq_test_plan_rows
)


# ======================================================================
# 13. DESCRIPTIVE STATISTICS HELPERS
# ======================================================================

def qstats_16(
    series
):

    series = pd.Series(
        series
    ).dropna()


    if len(
        series
    ) == 0:

        return {
            "n": 0
        }


    return {
        "n":
            int(
                len(
                    series
                )
            ),

        "mean":
            float(
                series.mean()
            ),

        "median":
            float(
                series.median()
            ),

        "p75":
            float(
                series.quantile(
                    0.75
                )
            ),

        "p90":
            float(
                series.quantile(
                    0.90
                )
            ),

        "p95":
            float(
                series.quantile(
                    0.95
                )
            ),

        "max":
            float(
                series.max()
            ),
    }


# ======================================================================
# 14. RQ1 FINAL SUMMARY
# ======================================================================

def build_rq1_summary_16(
    dataset_name,
    hop_df,
    plan_df,
    question_df,
    oracle
):

    active = hop_df.copy()


    intermediate = (
        hop_df[
            hop_df[
                "is_intermediate"
            ]
        ].copy()
    )


    decision = (
        intermediate[
            intermediate[
                "decision_hop"
            ]
        ].copy()
    )


    total_plans = len(
        plan_df
    )


    nonempty_plans = int(
        (
            ~plan_df[
                "empty_plan"
            ]
        ).sum()
    )


    empty_plans = (
        total_plans
        -
        nonempty_plans
    )


    reachable_plans = int(
        plan_df[
            "reachable"
        ].sum()
    )


    reachable_questions = int(
        question_df[
            "reachable"
        ].sum()
    )


    eligible_plans = int(
        plan_df[
            "eligible_intermediate"
        ].sum()
    )


    decision_plans = int(
        plan_df[
            "has_decision"
        ].sum()
    )


    eligible_questions = int(
        question_df[
            "eligible_intermediate"
        ].sum()
    )


    decision_questions = int(
        question_df[
            "has_decision"
        ].sum()
    )


    intermediate_active_hops = len(
        intermediate
    )


    decision_hops = len(
        decision
    )


    total_edges = int(
        oracle[
            "total_edges"
        ]
    )


    summary = {
        "dataset":
            dataset_name,

        "questions":
            int(
                len(
                    question_df
                )
            ),

        "total_predicted_plans":
            int(
                total_plans
            ),

        "nonempty_plans":
            int(
                nonempty_plans
            ),

        "empty_plans":
            int(
                empty_plans
            ),

        "active_hop_rows":
            int(
                len(
                    active
                )
            ),

        "active_prefixes_total":
            int(
                active[
                    "active_prefixes"
                ].sum()
            ),

        "edges_examined_total":
            int(
                active[
                    "edges_examined"
                ].sum()
            ),

        "candidate_branches_total":
            int(
                active[
                    "candidate_branches"
                ].sum()
            ),

        "reachable_plans":
            reachable_plans,

        "reachable_plan_rate":
            (
                reachable_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "reachable_questions":
            reachable_questions,

        "reachable_question_rate":
            (
                reachable_questions
                /
                len(
                    question_df
                )
            ),

        "branching_rate_candidate_gt1_all":
            float(
                (
                    active[
                        "candidate_branches"
                    ]
                    >
                    1
                ).mean()
            ),

        "branching_rate_candidate_gt1_intermediate":
            (
                float(
                    (
                        intermediate[
                            "candidate_branches"
                        ]
                        >
                        1
                    ).mean()
                )
                if
                len(
                    intermediate
                ) > 0
                else 0.0
            ),

        "rho_gt1_rate":
            float(
                (
                    active[
                        "rho"
                    ]
                    >
                    1
                ).mean()
            ),

        "rho_eq1_rate":
            float(
                np.isclose(
                    active[
                        "rho"
                    ],
                    1.0
                ).mean()
            ),

        "rho_lt1_rate":
            float(
                (
                    active[
                        "rho"
                    ]
                    <
                    1
                ).mean()
            ),

        "eligible_plans":
            eligible_plans,

        "eligible_plan_rate":
            (
                eligible_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "decision_plans":
            decision_plans,

        "decision_plan_rate":
            (
                decision_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "eligible_questions":
            eligible_questions,

        "eligible_question_rate":
            (
                eligible_questions
                /
                len(
                    question_df
                )
            ),

        "decision_questions":
            decision_questions,

        "decision_question_rate":
            (
                decision_questions
                /
                len(
                    question_df
                )
            ),

        "intermediate_active_hops":
            int(
                intermediate_active_hops
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "decision_rate_among_intermediate_active":
            (
                decision_hops
                /
                intermediate_active_hops
                if intermediate_active_hops > 0
                else 0.0
            ),

        "duplicate_endpoint_hop_rate":
            float(
                (
                    active[
                        "duplicate_endpoint_ratio"
                    ]
                    >
                    0
                ).mean()
            ),

        "active_prefix_stats_all":
            qstats_16(
                active[
                    "active_prefixes"
                ]
            ),

        "edge_stats_all":
            qstats_16(
                active[
                    "edges_examined"
                ]
            ),

        "candidate_stats_all":
            qstats_16(
                active[
                    "candidate_branches"
                ]
            ),

        "unique_frontier_stats_all":
            qstats_16(
                active[
                    "unique_candidate_entities"
                ]
            ),

        "active_prefix_stats_intermediate":
            qstats_16(
                intermediate[
                    "active_prefixes"
                ]
            ),

        "edge_stats_intermediate":
            qstats_16(
                intermediate[
                    "edges_examined"
                ]
            ),

        "candidate_stats_intermediate":
            qstats_16(
                intermediate[
                    "candidate_branches"
                ]
            ),

        "question_edge_stats":
            qstats_16(
                question_df[
                    "edges_examined"
                ]
            ),

        "question_peak_frontier_stats":
            qstats_16(
                question_df[
                    "peak_frontier"
                ]
            ),

        "decision_downstream_work_stats":
            qstats_16(
                decision[
                    "downstream_work_all_candidates"
                ]
            ),

        "oracle": {
            **{
                key:
                    int(
                        value
                    )
                for key, value
                in oracle.items()
            },

            "doomed_prefix_edge_fraction":
                (
                    oracle[
                        "doomed_prefix_edges"
                    ]
                    /
                    total_edges
                    if total_edges > 0
                    else 0.0
                ),

            "oracle_avoidable_downstream_edge_fraction":
                (
                    oracle[
                        "oracle_avoidable_downstream_edges"
                    ]
                    /
                    total_edges
                    if total_edges > 0
                    else 0.0
                ),

            "oracle_prunable_intermediate_fraction":
                (
                    oracle[
                        "oracle_prunable_intermediate_branches"
                    ]
                    /
                    oracle[
                        "intermediate_candidate_branches"
                    ]
                    if
                    oracle[
                        "intermediate_candidate_branches"
                    ] > 0
                    else 0.0
                ),
        },
    }


    return summary


WEBQSP_FINAL_RQ1_SUMMARY = (
    build_rq1_summary_16(
        "webqsp",
        webqsp_test_hop_df,
        webqsp_test_plan_df,
        webqsp_test_question_df,
        webqsp_test_oracle
    )
)


CWQ_FINAL_RQ1_SUMMARY = (
    build_rq1_summary_16(
        "cwq",
        cwq_test_hop_df,
        cwq_test_plan_df,
        cwq_test_question_df,
        cwq_test_oracle
    )
)


FINAL_RQ1_TEST_REFERENCE = {
    "webqsp":
        WEBQSP_FINAL_RQ1_SUMMARY,

    "cwq":
        CWQ_FINAL_RQ1_SUMMARY,
}


# ======================================================================
# 15. PRINT CORE RQ1 RESULTS
# ======================================================================

def print_rq1_summary_16(
    summary
):

    print(
        "\n"
        + "=" * 118
    )

    print(
        summary[
            "dataset"
        ].upper(),
        "FINAL FROZEN-TEST RQ1"
    )

    print(
        "=" * 118
    )


    print(
        "questions:",
        summary[
            "questions"
        ]
    )

    print(
        "plans:",
        summary[
            "total_predicted_plans"
        ]
    )

    print(
        "active hop rows:",
        summary[
            "active_hop_rows"
        ]
    )

    print(
        "active prefixes:",
        summary[
            "active_prefixes_total"
        ]
    )

    print(
        "edges examined:",
        summary[
            "edges_examined_total"
        ]
    )

    print(
        "candidate branches:",
        summary[
            "candidate_branches_total"
        ]
    )

    print(
        "RoG reachable questions:",
        summary[
            "reachable_questions"
        ],
        f"({summary['reachable_question_rate']:.4%})"
    )

    print(
        "decision hops:",
        summary[
            "decision_hops"
        ],
        f"({summary['decision_rate_among_intermediate_active']:.4%} "
        "of intermediate-active hops)"
    )


    oracle = summary[
        "oracle"
    ]


    print(
        "\nPOST-FREEZE ORACLE"
    )

    print(
        "doomed-prefix edges:",
        oracle[
            "doomed_prefix_edges"
        ],
        f"({oracle['doomed_prefix_edge_fraction']:.4%})"
    )

    print(
        "oracle-prunable intermediate branches:",
        oracle[
            "oracle_prunable_intermediate_branches"
        ],
        "/",
        oracle[
            "intermediate_candidate_branches"
        ],
        f"({oracle['oracle_prunable_intermediate_fraction']:.4%})"
    )

    print(
        "oracle-avoidable downstream edges:",
        oracle[
            "oracle_avoidable_downstream_edges"
        ],
        f"({oracle['oracle_avoidable_downstream_edge_fraction']:.4%})"
    )

    print(
        "doomed final-hop protected branches:",
        oracle[
            "doomed_final_protected_branches"
        ]
    )


print_rq1_summary_16(
    WEBQSP_FINAL_RQ1_SUMMARY
)


print_rq1_summary_16(
    CWQ_FINAL_RQ1_SUMMARY
)


# ======================================================================
# 16. SAVE RAW TEST ARTIFACTS
# ======================================================================

WEBQSP_RQ1_HOP_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_hops.csv"
)

WEBQSP_RQ1_PLAN_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_plans.csv"
)

WEBQSP_RQ1_Q_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_questions.csv"
)


CWQ_RQ1_HOP_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_hops.csv"
)

CWQ_RQ1_PLAN_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_plans.csv"
)

CWQ_RQ1_Q_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_questions.csv"
)


webqsp_test_hop_df.to_csv(
    WEBQSP_RQ1_HOP_CSV,
    index=False
)

webqsp_test_plan_df.to_csv(
    WEBQSP_RQ1_PLAN_CSV,
    index=False
)

webqsp_test_question_df.to_csv(
    WEBQSP_RQ1_Q_CSV,
    index=False
)


cwq_test_hop_df.to_csv(
    CWQ_RQ1_HOP_CSV,
    index=False
)

cwq_test_plan_df.to_csv(
    CWQ_RQ1_PLAN_CSV,
    index=False
)

cwq_test_question_df.to_csv(
    CWQ_RQ1_Q_CSV,
    index=False
)


# ======================================================================
# 17. SAVE RQ1 SUMMARY JSON
# ======================================================================

RQ1_SUMMARY_JSON = (
    RQ1_TEST_DIR
    / "final_frozen_test_rq1_summary.json"
)


with open(
    RQ1_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "freeze_sha256":
                FINAL_AFP_FREEZE_SHA256,

            "webqsp_test_planning_sha256":
                WEBQSP_TEST_PLAN_SHA,

            "cwq_test_planning_sha256":
                CWQ_TEST_PLAN_SHA,

            "webqsp":
                WEBQSP_FINAL_RQ1_SUMMARY,

            "cwq":
                CWQ_FINAL_RQ1_SUMMARY,

            "oracle_status":
                "post_freeze_test_analysis",

            "test_used_for_development":
                False,
        },
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16_COMPLETE = True

FINAL_RQ1_TEST_COMPLETE = True


print(
    "\n"
    + "=" * 120
)

print(
    "=== CELL 16: FINAL FROZEN-TEST RQ1 PROFILING COMPLETE ==="
)

print(
    "=" * 120
)

print(
    "AFP changed after freeze: NO"
)

print(
    "TEST used for tuning: NO"
)

print(
    "POST-FREEZE oracle analysis: COMPLETE"
)

print(
    "RQ1 final test artifacts:",
    RQ1_TEST_DIR
)

print(
    "\nNEXT: Cell 17 — final frozen-test RQ2 controlled comparison."
)

Cell 16 freeze gate: PASSED
AFP freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116
WEBQSP TEST planning loaded from global: webqsp_test_plan_rows
CWQ TEST planning loaded from global: cwq_test_plan_rows

TEST / validation ID-disjointness: PASSED
WebQSP TEST questions: 1628
CWQ TEST questions: 3531


webqsp_validation_gate final TEST RQ1:   0%|          | 0/246 [00:00<?, ?it/s]

WEBQSP validation oracle software gate: PASSED


cwq_validation_gate final TEST RQ1:   0%|          | 0/3519 [00:00<?, ?it/s]

AssertionError: cwq oracle gate mismatch for oracle_avoidable_downstream_edges: 1341283 != 1176294

In [6]:
# ======================================================================
# CELL 16 ORACLE-MISMATCH DIAGNOSTIC
# RUN AS A NEW CELL AFTER THE CWQ ORACLE GATE FAILURE
# ======================================================================
#
# PURPOSE:
#   Recover evidence about the previously verified oracle implementation
#   and compare it with the currently live Cell-16 implementation.
#
# THIS CELL:
#   - does NOT touch TEST traversal
#   - does NOT tune anything
#   - does NOT modify AFP
#   - does NOT overwrite artifacts
# ======================================================================

from pathlib import Path
import ast
import hashlib
import inspect
import json
import re


print("=" * 100)
print("CELL 16 ORACLE-MISMATCH DIAGNOSTIC")
print("=" * 100)


# ----------------------------------------------------------------------
# 1. Freeze gate
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Known previously verified RQ1 oracle references
# ----------------------------------------------------------------------

KNOWN_WEB = {
    "total_edges": 341526,
    "doomed_prefix_edges": 159039,
    "oracle_avoidable_downstream_edges": 11717,
    "candidate_branches": 7983,
    "intermediate_candidates": 1707,
    "prunable_intermediate_candidates": 1340,
    "doomed_final_protected": 4882,
}

KNOWN_CWQ = {
    "total_edges": 5257272,
    "doomed_prefix_edges": 3197488,
    "oracle_avoidable_downstream_edges": 1176294,
    "candidate_branches": 247161,
    "intermediate_candidates": 41576,
    "prunable_intermediate_candidates": 34431,
    "doomed_final_protected": 195499,
}

print("\nPreviously verified CWQ oracle reference:")
for k, v in KNOWN_CWQ.items():
    print(f"  {k}: {v}")


# ----------------------------------------------------------------------
# 3. Inspect CURRENT live oracle functions
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("A. CURRENT LIVE ORACLE-RELATED FUNCTIONS")
print("=" * 100)

oracle_function_names = []

for name, obj in sorted(globals().items()):

    if (
        callable(obj)
        and
        (
            "oracle" in name.lower()
            or "suffix" in name.lower()
            or "avoidable" in name.lower()
        )
    ):
        oracle_function_names.append(name)


print("Found functions:")
for name in oracle_function_names:
    print(" ", name)


def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


for name in oracle_function_names:

    obj = globals()[name]

    print("\n" + "-" * 100)
    print("FUNCTION:", name)

    try:
        src = inspect.getsource(obj)

        print("SHA256:", sha_text(src))
        print(src)

    except Exception as e:
        print("Could not inspect source:", repr(e))


# ----------------------------------------------------------------------
# 4. Specifically inspect oracle_validation_gate_16
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("B. oracle_validation_gate_16")
print("=" * 100)

assert "oracle_validation_gate_16" in globals()

try:
    gate_src = inspect.getsource(
        oracle_validation_gate_16
    )

    print(
        "oracle_validation_gate_16 SHA:",
        sha_text(gate_src)
    )

    print(gate_src)

except Exception as e:
    print(
        "inspect.getsource failed:",
        repr(e)
    )


# ----------------------------------------------------------------------
# 5. Search persisted notebook for OLD oracle implementation/evidence
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("C. PERSISTED NOTEBOOK ORACLE SEARCH")
print("=" * 100)

assert "persisted_nb_16ar" in globals()


search_terms = [
    "1176294",
    "3197488",
    "34431",
    "195499",
    "oracle_avoidable_downstream_edges",
    "doomed_prefix",
    "doomed_final",
    "suffix_dp",
]


notebook_hits = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    matched = [
        term
        for term in search_terms
        if term in source
    ]

    if matched:

        notebook_hits.append(
            (
                cell_idx,
                matched,
                source
            )
        )


print(
    "Relevant persisted notebook cells:",
    [x[0] for x in notebook_hits]
)


for cell_idx, matched, source in notebook_hits:

    print("\n" + "-" * 100)
    print(
        f"CELL {cell_idx} | matched:",
        matched
    )
    print("-" * 100)

    # Avoid dumping gigantic cells unnecessarily.
    lines = source.splitlines()

    interesting_indices = set()

    for i, line in enumerate(lines):

        if any(
            term in line
            for term in search_terms
        ):
            for j in range(
                max(0, i - 8),
                min(len(lines), i + 15)
            ):
                interesting_indices.add(j)


    for i in sorted(interesting_indices):
        print(
            f"{i + 1:04d}: {lines[i]}"
        )


# ----------------------------------------------------------------------
# 6. Search saved RQ1 DEVELOPMENT artifacts for verified oracle values
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("D. SAVED RQ1 DEVELOPMENT ARTIFACT SEARCH")
print("=" * 100)

RQ1_ROOT = Path(
    "/kaggle/working/step2_rq1_dev"
)

assert RQ1_ROOT.exists()


target_strings = [
    "1176294",
    "3197488",
    "34431",
    "195499",
    "11717",
    "159039",
]


artifact_hits = []


allowed_suffixes = {
    ".json",
    ".jsonl",
    ".csv",
    ".txt",
    ".md",
}


for path in RQ1_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in allowed_suffixes:
        continue

    # Safety: don't scan huge graph files.
    try:
        if path.stat().st_size > 20 * 1024 * 1024:
            continue
    except Exception:
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        x
        for x in target_strings
        if x in text
    ]

    if matched:
        artifact_hits.append(
            (
                path,
                matched
            )
        )


print(
    "Files containing previously verified oracle values:"
)

for path, matched in artifact_hits:
    print(
        "\n ",
        path
    )
    print(
        "   matched:",
        matched
    )


# ----------------------------------------------------------------------
# 7. Search all small development manifests/results for oracle field names
# ----------------------------------------------------------------------

field_hits = []

for path in RQ1_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in allowed_suffixes:
        continue

    try:
        if path.stat().st_size > 20 * 1024 * 1024:
            continue

        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue

    if (
        "oracle_avoidable" in text
        or
        "doomed_prefix" in text
        or
        "prunable_intermediate" in text
    ):
        field_hits.append(path)


print(
    "\nFiles containing oracle field names:"
)

for path in field_hits:
    print(" ", path)


# ----------------------------------------------------------------------
# 8. Critical numerical diagnosis
# ----------------------------------------------------------------------

current_cwq = 1341283
frozen_cwq = 1176294

difference = (
    current_cwq
    -
    frozen_cwq
)

print("\n" + "=" * 100)
print("E. NUMERICAL DIFFERENCE")
print("=" * 100)

print(
    "Current CWQ avoidable downstream edges:",
    current_cwq
)

print(
    "Frozen verified value:",
    frozen_cwq
)

print(
    "Difference:",
    difference
)

print(
    "Relative excess:",
    f"{difference / frozen_cwq:.6%}"
)


# ----------------------------------------------------------------------
# 9. Status
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE")
print("=" * 100)

print(
    "\nNo TEST result was accepted."
)

print(
    "No expected oracle value was changed."
)

print(
    "No AFP parameter was changed."
)

print(
    "No development result was overwritten."
)

print(
    "\nPASTE THE OUTPUT OF THIS CELL HERE."
)

CELL 16 ORACLE-MISMATCH DIAGNOSTIC

Freeze gate: PASSED

Previously verified CWQ oracle reference:
  total_edges: 5257272
  doomed_prefix_edges: 3197488
  oracle_avoidable_downstream_edges: 1176294
  candidate_branches: 247161
  intermediate_candidates: 41576
  prunable_intermediate_candidates: 34431
  doomed_final_protected: 195499

A. CURRENT LIVE ORACLE-RELATED FUNCTIONS
Found functions:
  make_oracle_functions_16
  oracle_validation_gate_16

----------------------------------------------------------------------------------------------------
FUNCTION: make_oracle_functions_16
SHA256: 9ad6a28e261c24931764ba05120d5a31d9f389a266ed059b79b5cab0aca4cf69
def make_oracle_functions_16(
    adjacency,
    gold_answers
):

    gold_set = {
        str(x)
        for x in gold_answers
    }


    reachable_cache = {}

    cost_cache = {}


    def suffix_reachable(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suff

In [7]:
# ======================================================================
# CELL 16 ORACLE REPAIR — EXACT SOURCE EXTRACTION
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun Cell 16
#   - run Cell 17
#   - restart kernel
#   - change expected oracle values
#
# PURPOSE:
#   Recover the EXACT original oracle logic from development Cell 149
#   before modifying the reconstructed Cell-16 implementation.
# ======================================================================

import ast
import hashlib
import inspect


print("=" * 110)
print("EXACT ORIGINAL ORACLE SOURCE EXTRACTION")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Hard scientific gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Helpers
# ----------------------------------------------------------------------

def source_sha_16repair(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def get_cell_source_16repair(cell_index):

    cell = persisted_nb_16ar[
        "cells"
    ][cell_index]

    assert cell.get(
        "cell_type"
    ) == "code"

    return "".join(
        cell.get(
            "source",
            []
        )
    )


# ----------------------------------------------------------------------
# 3. Cell 149 exact function inventory
# ----------------------------------------------------------------------

cell149_source = (
    get_cell_source_16repair(
        149
    )
)

cell149_tree = ast.parse(
    cell149_source
)


cell149_functions = []


for node in cell149_tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        src = ast.get_source_segment(
            cell149_source,
            node
        )

        cell149_functions.append(
            {
                "name": node.name,
                "lineno": node.lineno,
                "end_lineno": getattr(
                    node,
                    "end_lineno",
                    None
                ),
                "sha256": source_sha_16repair(
                    src
                ),
                "source": src,
            }
        )


print(
    "\nCELL 149 FUNCTION INVENTORY"
)

for rec in cell149_functions:

    print(
        f"  {rec['name']:<40} "
        f"lines {rec['lineno']}-{rec['end_lineno']} "
        f"sha={rec['sha256']}"
    )


# ----------------------------------------------------------------------
# 4. Print ALL oracle-relevant Cell-149 functions verbatim
# ----------------------------------------------------------------------

oracle_relevant_149 = []


for rec in cell149_functions:

    src_lower = rec[
        "source"
    ].lower()

    if (
        "oracle" in src_lower
        or
        "productive" in src_lower
        or
        "doomed" in src_lower
        or
        "avoidable" in src_lower
        or
        "downstream" in src_lower
        or
        "suffix" in src_lower
    ):

        oracle_relevant_149.append(
            rec
        )


print(
    "\n"
    + "=" * 110
)

print(
    "ORIGINAL CELL 149 ORACLE-RELEVANT FUNCTIONS — VERBATIM"
)

print(
    "=" * 110
)


for rec in oracle_relevant_149:

    print(
        "\n"
        + "-" * 110
    )

    print(
        "FUNCTION:",
        rec["name"]
    )

    print(
        "LINES:",
        rec["lineno"],
        "-",
        rec["end_lineno"]
    )

    print(
        "SHA256:",
        rec["sha256"]
    )

    print(
        "-" * 110
    )

    print(
        rec["source"]
    )


# ----------------------------------------------------------------------
# 5. Cell 147 DP-function inventory
# ----------------------------------------------------------------------

cell147_source = (
    get_cell_source_16repair(
        147
    )
)

cell147_tree = ast.parse(
    cell147_source
)


cell147_functions = []


for node in cell147_tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        src = ast.get_source_segment(
            cell147_source,
            node
        )

        cell147_functions.append(
            {
                "name": node.name,
                "lineno": node.lineno,
                "end_lineno": getattr(
                    node,
                    "end_lineno",
                    None
                ),
                "sha256": source_sha_16repair(
                    src
                ),
                "source": src,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 147 SUFFIX/REACHABILITY FUNCTIONS"
)

print(
    "=" * 110
)


for rec in cell147_functions:

    src_lower = rec[
        "source"
    ].lower()

    if (
        "suffix" in src_lower
        or
        "reachable" in src_lower
        or
        "relation" in src_lower
    ):

        print(
            "\n"
            + "-" * 110
        )

        print(
            "FUNCTION:",
            rec["name"]
        )

        print(
            "LINES:",
            rec["lineno"],
            "-",
            rec["end_lineno"]
        )

        print(
            "SHA256:",
            rec["sha256"]
        )

        print(
            "-" * 110
        )

        print(
            rec["source"]
        )


# ----------------------------------------------------------------------
# 6. Current Cell-16 profile function
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "CURRENT profile_rq1_dataset_16 — ORACLE USAGE"
)

print(
    "=" * 110
)


assert (
    "profile_rq1_dataset_16"
    in globals()
)


current_profile_src = (
    inspect.getsource(
        profile_rq1_dataset_16
    )
)


print(
    "SHA256:",
    source_sha_16repair(
        current_profile_src
    )
)


# Print full function because the interaction between
# traversal and oracle accounting is what we need to compare.
print(
    current_profile_src
)


# ----------------------------------------------------------------------
# 7. Current helper dependencies
# ----------------------------------------------------------------------

for fname in [
    "matching_neighbors_16",
    "degree_16",
    "make_oracle_functions_16",
]:

    print(
        "\n"
        + "=" * 110
    )

    print(
        "CURRENT:",
        fname
    )

    print(
        "=" * 110
    )

    assert fname in globals()

    src = inspect.getsource(
        globals()[fname]
    )

    print(
        "SHA256:",
        source_sha_16repair(
            src
        )
    )

    print(
        src
    )


# ----------------------------------------------------------------------
# 8. Saved development summary — exact contents
# ----------------------------------------------------------------------

import pandas as pd
from pathlib import Path


print(
    "\n"
    + "=" * 110
)

print(
    "AUTHORITATIVE SAVED DEVELOPMENT ORACLE SUMMARIES"
)

print(
    "=" * 110
)


for dataset_name in [
    "webqsp",
    "cwq",
]:

    path = Path(
        f"/kaggle/working/step2_rq1_dev/"
        f"oracle_headroom_{dataset_name}_dev_summary.csv"
    )

    assert path.exists()

    df = pd.read_csv(
        path
    )

    print(
        f"\n{dataset_name.upper()}"
    )

    print(
        df.to_string(
            index=False
        )
    )


print(
    "\n"
    + "=" * 110
)

print(
    "EXACT SOURCE EXTRACTION COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nNo TEST evaluation performed."
)

print(
    "No oracle value changed."
)

print(
    "No AFP setting changed."
)

print(
    "No artifact overwritten."
)

print(
    "\nPASTE THIS OUTPUT HERE."
)

EXACT ORIGINAL ORACLE SOURCE EXTRACTION

Freeze gate: PASSED

CELL 149 FUNCTION INVENTORY
  oracle_headroom_for_plan                 lines 18-93 sha=42190812ea03aead6ab97b564ed1829e73ab043fc9678bd3cadf6ab584caef1a
  compute_oracle_headroom                  lines 96-166 sha=295d9d8a0d838167f5b570a818c617806e8076b5dc43d3897ee346e294ec8309
  check_oracle_vs_rq1                      lines 207-239 sha=04050bf25473141a8ec2d7cbaa3e6388b144031acffac4716949aa5fbc1dc22a
  safe_pct                                 lines 258-259 sha=1cc1805de72a0ceb0b0c63511cf158e2933b6422322d38dac344ca2a7c8f1301
  summarize_headroom                       lines 262-474 sha=c46230458a4799d67e71b4969886c6d1ed73fbe03c581e8afc212a26f8e60954
  get_summary_value                        lines 532-533 sha=8761e5ff0ae691de7e42d29d06baefac37331f17c750d7c6c720dc809e03ba60

ORIGINAL CELL 149 ORACLE-RELEVANT FUNCTIONS — VERBATIM

----------------------------------------------------------------------------------------------------

In [9]:
# ======================================================================
# CELL 16-R1
# EXACT ORIGINAL ORACLE-SEMANTICS REPAIR + VALIDATION GATE
# ======================================================================
#
# RUN AS A NEW CELL after the failed Cell 16 oracle gate.
#
# DO NOT:
#   - rerun Cell 16
#   - run Cell 17
#   - restart kernel
#
# This repairs ONLY the reconstructed Cell-16 oracle accounting.
# It does not alter AFP, frozen plans, scorer, selector, or TEST data.
# ======================================================================

import inspect
import textwrap
import hashlib
import json
from pathlib import Path

import pandas as pd


# ======================================================================
# 1. HARD FREEZE GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "profile_rq1_dataset_16" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("Cell 16-R1 freeze gate: PASSED")


# ======================================================================
# 2. VERIFY THAT WE ARE PATCHING THE EXACT FAILED IMPLEMENTATION
# ======================================================================

def sha256_text_16r1(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


profile_source_16r1 = textwrap.dedent(
    inspect.getsource(
        profile_rq1_dataset_16
    )
)


CURRENT_FAILED_PROFILE_SHA = (
    "cf1f74072bb777653005682d48e65caf8544bcc9d636752ff918a5fec792e6c9"
)


actual_failed_sha = sha256_text_16r1(
    profile_source_16r1
)


print("\nCurrent failed profile SHA:")
print(" ", actual_failed_sha)


assert (
    actual_failed_sha
    ==
    CURRENT_FAILED_PROFILE_SHA
), (
    "profile_rq1_dataset_16 is not the exact implementation "
    "diagnosed previously. Stop rather than patch unknown code."
)


print("Failed Cell-16 profile identity: PASSED")


# ======================================================================
# 3. PATCH 1
#
# ORIGINAL CELL-149 SEMANTICS:
#
# If the CURRENT prefix is doomed and h > 0, every adjacency edge
# examined from that prefix is avoidable because that prefix could
# have been pruned after the previous intermediate candidate set.
#
# Therefore:
#
# avoidable_this_hop =
#     doomed_prefix_edges if hop > 0 else 0
#
# It is NOT recursive future subtree cost assigned to the candidate.
# ======================================================================

old_1 = '''                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges
'''


new_1 = '''                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges


                # ------------------------------------------------------
                # EXACT ORIGINAL CELL-149 ORACLE SEMANTICS
                #
                # A doomed prefix at hop > 0 was generated at the
                # preceding intermediate hop and could therefore have
                # been removed before THIS expansion occurred.
                #
                # Count the actual adjacency work paid at this hop.
                # Do not recursively assign its whole future subtree
                # to the candidate that generated it.
                # ------------------------------------------------------

                oracle_avoidable_this_hop = int(
                    doomed_prefix_edges
                    if hop > 0
                    else 0
                )


                oracle_totals[
                    "oracle_avoidable_downstream_edges"
                ] += oracle_avoidable_this_hop


                plan_oracle_avoidable += (
                    oracle_avoidable_this_hop
                )
'''


assert profile_source_16r1.count(old_1) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_1,
        new_1,
        1
    )
)


# ======================================================================
# 4. PATCH 2
# Do not reset oracle_avoidable_this_hop after it was computed from
# the current doomed-prefix expansion.
# ======================================================================

old_2 = '''                oracle_prunable_this_hop = 0
                oracle_avoidable_this_hop = 0
                downstream_work_all_candidates = 0
'''


new_2 = '''                oracle_prunable_this_hop = 0

                # oracle_avoidable_this_hop was already computed above
                # from CURRENT doomed-prefix adjacency work, matching
                # the original Cell-149 definition.

                downstream_work_all_candidates = 0
'''


assert profile_source_16r1.count(old_2) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_2,
        new_2,
        1
    )
)


# ======================================================================
# 5. PATCH 3
# Candidate feasibility still determines whether an INTERMEDIATE
# candidate is oracle-prunable.
#
# But recursive future_cost must NOT be added to the authoritative
# oracle-avoidable edge counter.
# ======================================================================

old_3 = '''                        if not feasible:

                            oracle_prunable_this_hop += 1

                            oracle_avoidable_this_hop += (
                                future_cost
                            )
'''


new_3 = '''                        if not feasible:

                            oracle_prunable_this_hop += 1
'''


assert profile_source_16r1.count(old_3) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_3,
        new_3,
        1
    )
)


# ======================================================================
# 6. PATCH 4
# Remove the old recursive candidate-level addition to global oracle
# avoidable edges. It is now counted once at actual prefix expansion.
# ======================================================================

old_4 = '''                    oracle_totals[
                        "oracle_avoidable_downstream_edges"
                    ] += oracle_avoidable_this_hop
'''


assert profile_source_16r1.count(old_4) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_4,
        "",
        1
    )
)


# ======================================================================
# 7. PATCH 5
# Same correction for per-plan oracle avoidable work.
# ======================================================================

old_5 = '''                    plan_oracle_avoidable += (
                        oracle_avoidable_this_hop
                    )
'''


assert profile_source_16r1.count(old_5) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_5,
        "",
        1
    )
)


# ======================================================================
# 8. INSTALL REPAIRED FUNCTION IN CURRENT KERNEL
# ======================================================================

CELL16_REPAIRED_PROFILE_SOURCE = (
    profile_source_16r1
)

CELL16_REPAIRED_PROFILE_SHA256 = (
    sha256_text_16r1(
        CELL16_REPAIRED_PROFILE_SOURCE
    )
)


exec(
    CELL16_REPAIRED_PROFILE_SOURCE,
    globals()
)


print("\nRepaired profile installed.")
print(
    "Repaired profile SHA:",
    CELL16_REPAIRED_PROFILE_SHA256
)


# ======================================================================
# 9. AUTHORITATIVE FROZEN VALIDATION REFERENCES
# ======================================================================

EXPECTED_WEB_16R1 = {
    "total_edges":
        341526,

    "doomed_prefix_edges":
        159039,

    "oracle_avoidable_downstream_edges":
        11717,

    "candidate_branches":
        7983,

    "intermediate_candidate_branches":
        1707,

    "oracle_prunable_intermediate_branches":
        1340,

    "doomed_final_protected_branches":
        4882,
}


EXPECTED_CWQ_16R1 = {
    "total_edges":
        5257272,

    "doomed_prefix_edges":
        3197488,

    "oracle_avoidable_downstream_edges":
        1176294,

    "candidate_branches":
        247161,

    "intermediate_candidate_branches":
        41576,

    "oracle_prunable_intermediate_branches":
        34431,

    "doomed_final_protected_branches":
        195499,
}


# ======================================================================
# 10. RUN REPAIRED WEBQSP VALIDATION
# ======================================================================

print(
    "\nRunning repaired WebQSP oracle validation..."
)


(
    web_hop_16r1,
    web_plan_16r1,
    web_question_16r1,
    web_oracle_16r1,
) = profile_rq1_dataset_16(
    "webqsp_validation_repair_gate",
    webqsp_val_plan_rows
)


for key, expected in EXPECTED_WEB_16R1.items():

    actual = int(
        web_oracle_16r1[
            key
        ]
    )

    assert actual == expected, (
        f"WebQSP repaired oracle mismatch: "
        f"{key}: {actual} != {expected}"
    )


print(
    "WEBQSP repaired oracle totals: EXACT MATCH"
)


# ======================================================================
# 11. RUN REPAIRED CWQ VALIDATION
# ======================================================================

print(
    "\nRunning repaired CWQ oracle validation..."
)


(
    cwq_hop_16r1,
    cwq_plan_16r1,
    cwq_question_16r1,
    cwq_oracle_16r1,
) = profile_rq1_dataset_16(
    "cwq_validation_repair_gate",
    cwq_val_plan_rows
)


for key, expected in EXPECTED_CWQ_16R1.items():

    actual = int(
        cwq_oracle_16r1[
            key
        ]
    )

    assert actual == expected, (
        f"CWQ repaired oracle mismatch: "
        f"{key}: {actual} != {expected}"
    )


print(
    "CWQ repaired oracle totals: EXACT MATCH"
)


# ======================================================================
# 12. PER-HOP FIDELITY AGAINST SAVED ORIGINAL CELL-149 ARTIFACTS
#
# This verifies not only the total, but the HOP ATTRIBUTION.
# ======================================================================

def verify_per_hop_16r1(
    dataset_name,
    new_hop_df
):

    old_path = Path(
        "/kaggle/working/step2_rq1_dev/"
        f"oracle_headroom_{dataset_name}_dev_hop.csv"
    )

    assert old_path.exists(), old_path

    old_df = pd.read_csv(
        old_path
    )


    required_old = {
        "hop",
        "oracle_avoidable_downstream_edges",
        "oracle_prunable_branches",
    }

    assert required_old.issubset(
        set(old_df.columns)
    )


    new_agg = (
        new_hop_df
        .groupby(
            "hop",
            as_index=False
        )
        .agg(
            oracle_avoidable_downstream_edges=(
                "oracle_avoidable_downstream_edges",
                "sum"
            ),

            oracle_prunable_branches=(
                "oracle_prunable_candidates",
                "sum"
            ),
        )
    )


    old_cmp = (
        old_df[
            [
                "hop",
                "oracle_avoidable_downstream_edges",
                "oracle_prunable_branches",
            ]
        ]
        .copy()
    )


    all_hops = sorted(
        set(
            old_cmp["hop"].tolist()
        )
        |
        set(
            new_agg["hop"].tolist()
        )
    )


    old_cmp = (
        old_cmp
        .set_index("hop")
        .reindex(
            all_hops,
            fill_value=0
        )
    )


    new_cmp = (
        new_agg
        .set_index("hop")
        .reindex(
            all_hops,
            fill_value=0
        )
    )


    for col in [
        "oracle_avoidable_downstream_edges",
        "oracle_prunable_branches",
    ]:

        old_values = (
            old_cmp[col]
            .astype("int64")
            .tolist()
        )

        new_values = (
            new_cmp[col]
            .astype("int64")
            .tolist()
        )

        assert old_values == new_values, (
            f"{dataset_name} per-hop mismatch "
            f"for {col}:\n"
            f"old={old_values}\n"
            f"new={new_values}"
        )


    print(
        f"{dataset_name.upper()} per-hop "
        "oracle fidelity: PASSED"
    )


verify_per_hop_16r1(
    "webqsp",
    web_hop_16r1
)


verify_per_hop_16r1(
    "cwq",
    cwq_hop_16r1
)


# ======================================================================
# 13. ORDINARY RoG TRAVERSAL MUST REMAIN UNCHANGED
# ======================================================================

assert int(
    web_hop_16r1[
        "edges_examined"
    ].sum()
) == 341526

assert int(
    web_hop_16r1[
        "candidate_branches"
    ].sum()
) == 7983


assert int(
    cwq_hop_16r1[
        "edges_examined"
    ].sum()
) == 5257272

assert int(
    cwq_hop_16r1[
        "candidate_branches"
    ].sum()
) == 247161


assert int(
    web_question_16r1[
        "reachable"
    ].sum()
) == 205

assert int(
    cwq_question_16r1[
        "reachable"
    ].sum()
) == 2425


print(
    "\nOrdinary RoG validation traversal unchanged: PASSED"
)


# ======================================================================
# 14. SAVE REPAIR AUDIT MANIFEST
# ======================================================================

repair_manifest_16r1 = {
    "repair":
        "cell16_exact_original_oracle_semantics",

    "development_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "failed_profile_sha256":
        CURRENT_FAILED_PROFILE_SHA,

    "repaired_profile_sha256":
        CELL16_REPAIRED_PROFILE_SHA256,

    "original_cell149_oracle_headroom_for_plan_sha256":
        "42190812ea03aead6ab97b564ed1829e73ab043fc9678bd3cadf6ab584caef1a",

    "original_cell147_suffix_reachable_dp_sha256":
        "85a1fea859e8eb9e262efe6a807d8845ea9e69a287890353a279d21295551325",

    "webqsp_validation_oracle":
        {
            k: int(v)
            for k, v
            in web_oracle_16r1.items()
        },

    "cwq_validation_oracle":
        {
            k: int(v)
            for k, v
            in cwq_oracle_16r1.items()
        },

    "test_evaluated_in_this_repair":
        False,

    "afp_changed":
        False,

    "test_tuning":
        False,
}


repair_manifest_path_16r1 = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_oracle_repair_manifest.json"
)


repair_manifest_path_16r1.parent.mkdir(
    parents=True,
    exist_ok=True
)


with open(
    repair_manifest_path_16r1,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        repair_manifest_16r1,
        f,
        indent=2,
        sort_keys=True
    )


CELL16_ORACLE_REPAIR_VALIDATED = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16 ORACLE REPAIR: VALIDATED"
)

print(
    "=" * 110
)


print(
    "\nWebQSP oracle avoidable edges:",
    web_oracle_16r1[
        "oracle_avoidable_downstream_edges"
    ]
)

print(
    "CWQ oracle avoidable edges:",
    cwq_oracle_16r1[
        "oracle_avoidable_downstream_edges"
    ]
)


print(
    "\nExpected:"
)

print(
    " WebQSP = 11717"
)

print(
    " CWQ    = 1176294"
)


print(
    "\nPer-hop original Cell-149 fidelity: PASSED"
)

print(
    "Ordinary RoG traversal fidelity:   PASSED"
)

print(
    "AFP changed:                       NO"
)

print(
    "TEST evaluated by repair cell:     NO"
)

print(
    "TEST tuning:                       NO"
)


print(
    "\nManifest:"
)

print(
    " ",
    repair_manifest_path_16r1
)


print(
    "\nIMPORTANT:"
)

print(
    "DO NOT RERUN THE ORIGINAL CELL 16 YET."
)

print(
    "It would redefine the old reconstructed profiler."
)

print(
    "Paste this cell's final output here."
)

Cell 16-R1 freeze gate: PASSED


OSError: could not get source code

In [10]:
# ======================================================================
# CELL 16-R1A
# RECOVER / RESUME AFTER inspect.getsource OSError
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun Cell 16-R1
#   - rerun original Cell 16
#   - run Cell 17
#   - restart kernel
# ======================================================================

from pathlib import Path
import hashlib
import json


print("=" * 100)
print("CELL 16-R1 PARTIAL-EXECUTION STATUS")
print("=" * 100)


# ----------------------------------------------------------------------
# 1. Freeze state
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Did first Cell 16-R1 create repaired source?
# ----------------------------------------------------------------------

has_repaired_source = (
    "CELL16_REPAIRED_PROFILE_SOURCE"
    in globals()
)

has_repaired_sha = (
    "CELL16_REPAIRED_PROFILE_SHA256"
    in globals()
)

print(
    "\nCELL16_REPAIRED_PROFILE_SOURCE exists:",
    has_repaired_source
)

print(
    "CELL16_REPAIRED_PROFILE_SHA256 exists:",
    has_repaired_sha
)

if has_repaired_sha:

    print(
        "Repaired SHA:",
        CELL16_REPAIRED_PROFILE_SHA256
    )


# ----------------------------------------------------------------------
# 3. Is current profiler callable?
# ----------------------------------------------------------------------

print(
    "\nprofile_rq1_dataset_16 exists:",
    "profile_rq1_dataset_16"
    in globals()
)

print(
    "profile_rq1_dataset_16 callable:",
    callable(
        globals().get(
            "profile_rq1_dataset_16",
            None
        )
    )
)


# ----------------------------------------------------------------------
# 4. Did WebQSP/CWQ repaired validation already execute?
# ----------------------------------------------------------------------

for name in [
    "web_oracle_16r1",
    "cwq_oracle_16r1",
]:

    print(
        f"\n{name} exists:",
        name in globals()
    )

    if name in globals():

        print(
            globals()[name]
        )


print(
    "\nCELL16_ORACLE_REPAIR_VALIDATED:",
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
)


# ----------------------------------------------------------------------
# 5. Repair manifest
# ----------------------------------------------------------------------

manifest_path = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_oracle_repair_manifest.json"
)

print(
    "\nRepair manifest exists:",
    manifest_path.exists()
)

if manifest_path.exists():

    print(
        "Manifest path:",
        manifest_path
    )

    with open(
        manifest_path,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(f)

    print(
        "Manifest test_evaluated_in_this_repair:",
        manifest.get(
            "test_evaluated_in_this_repair"
        )
    )

    print(
        "Manifest AFP changed:",
        manifest.get(
            "afp_changed"
        )
    )


# ----------------------------------------------------------------------
# 6. Frozen TEST plans still safe
# ----------------------------------------------------------------------

WEB_TEST = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)

print("\nFrozen TEST planning files:")
print(" WebQSP exists:", WEB_TEST.exists())
print(" CWQ exists:   ", CWQ_TEST.exists())


# ----------------------------------------------------------------------
# 7. If repaired source exists, verify its stored SHA directly
# ----------------------------------------------------------------------

if has_repaired_source:

    actual_sha = hashlib.sha256(
        CELL16_REPAIRED_PROFILE_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()

    print(
        "\nStored repaired source SHA:",
        actual_sha
    )

    if has_repaired_sha:

        assert (
            actual_sha
            ==
            CELL16_REPAIRED_PROFILE_SHA256
        )

        print(
            "Stored repaired-source integrity: PASSED"
        )


print(
    "\n"
    + "=" * 100
)

print(
    "STATUS CHECK COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nPaste this output here."
)

CELL 16-R1 PARTIAL-EXECUTION STATUS

Freeze gate: PASSED

CELL16_REPAIRED_PROFILE_SOURCE exists: True
CELL16_REPAIRED_PROFILE_SHA256 exists: True
Repaired SHA: 3413dc5efa00a1be3f9c56f79677bbd97232293b9fb4d15d5a562bacb434287f

profile_rq1_dataset_16 exists: True
profile_rq1_dataset_16 callable: True

web_oracle_16r1 exists: True
{'total_edges': 341526, 'doomed_prefix_edges': 159039, 'candidate_branches': 7983, 'intermediate_candidate_branches': 1707, 'oracle_prunable_intermediate_branches': 1340, 'doomed_final_protected_branches': 4882, 'oracle_avoidable_downstream_edges': 11717}

cwq_oracle_16r1 exists: True
{'total_edges': 5257272, 'doomed_prefix_edges': 3197488, 'candidate_branches': 247161, 'intermediate_candidate_branches': 41576, 'oracle_prunable_intermediate_branches': 34431, 'doomed_final_protected_branches': 195499, 'oracle_avoidable_downstream_edges': 1176294}

CELL16_ORACLE_REPAIR_VALIDATED: True

Repair manifest exists: True
Manifest path: /kaggle/working/step2_rq1_test/cell

In [11]:
# ======================================================================
# CELL 16-R2
# RESUME ORIGINAL CELL 16 AFTER VALIDATED ORACLE REPAIR
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun original Cell 16
#   - rerun Cell 16-R1
#   - rerun M3
#   - restart kernel
#
# This executes ONLY the original Cell-16 statements that came AFTER
# the CWQ validation oracle gate.
# ======================================================================

import ast
import hashlib
from pathlib import Path


print("=" * 110)
print("CELL 16-R2 — SAFE RESUME")
print("=" * 110)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

assert "profile_rq1_dataset_16" in globals()
assert callable(profile_rq1_dataset_16)

assert "web_oracle_16r1" in globals()
assert "cwq_oracle_16r1" in globals()


# Exact repaired validation evidence
assert (
    int(
        web_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 11717
)

assert (
    int(
        cwq_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 1176294
)


print("Freeze gate:                 PASSED")
print("Oracle repair validation:    PASSED")
print("Repaired profiler in memory: YES")


# ======================================================================
# 2. VERIFY FROZEN TEST PLANNING ARTIFACTS
# ======================================================================

WEB_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)


assert WEB_TEST_PATH.exists()
assert CWQ_TEST_PATH.exists()


def file_sha256_16r2(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


WEB_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


web_sha_16r2 = file_sha256_16r2(
    WEB_TEST_PATH
)

cwq_sha_16r2 = file_sha256_16r2(
    CWQ_TEST_PATH
)


assert web_sha_16r2 == WEB_EXPECTED_SHA
assert cwq_sha_16r2 == CWQ_EXPECTED_SHA


print("\nFrozen TEST planning SHA gate: PASSED")
print(" WebQSP:", web_sha_16r2)
print(" CWQ:   ", cwq_sha_16r2)


# ======================================================================
# 3. RECOVER EXACT ORIGINAL CELL 16 FROM IPYTHON HISTORY
# ======================================================================

ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


cell16_candidates = []


for history_index, source in enumerate(
    history
):

    if not isinstance(source, str):
        continue

    # Strong signature of the original Cell 16.
    if (
        "def profile_rq1_dataset_16" in source
        and
        "def oracle_validation_gate_16" in source
        and
        "EXPECTED_VAL_ORACLE_16" in source
        and
        "oracle_validation_gate_16" in source
    ):

        cell16_candidates.append(
            (
                history_index,
                source
            )
        )


assert cell16_candidates, (
    "Could not find the executed original Cell 16 "
    "in IPython input history."
)


# Use the latest matching execution.
CELL16_HISTORY_INDEX, CELL16_ORIGINAL_SOURCE = (
    cell16_candidates[-1]
)


CELL16_ORIGINAL_SOURCE_SHA256 = (
    hashlib.sha256(
        CELL16_ORIGINAL_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nOriginal Cell 16 recovered from history:"
)

print(
    " history index:",
    CELL16_HISTORY_INDEX
)

print(
    " source SHA256:",
    CELL16_ORIGINAL_SOURCE_SHA256
)


# ======================================================================
# 4. FIND THE TWO TOP-LEVEL VALIDATION-GATE CALLS
# ======================================================================

cell16_tree = ast.parse(
    CELL16_ORIGINAL_SOURCE
)


gate_nodes = []


for body_index, node in enumerate(
    cell16_tree.body
):

    if not isinstance(
        node,
        ast.Expr
    ):
        continue

    call = node.value

    if not isinstance(
        call,
        ast.Call
    ):
        continue

    func = call.func

    if (
        isinstance(
            func,
            ast.Name
        )
        and
        func.id
        ==
        "oracle_validation_gate_16"
    ):

        gate_nodes.append(
            (
                body_index,
                node
            )
        )


print(
    "\nTop-level validation gates found:",
    len(gate_nodes)
)


assert len(gate_nodes) == 2, (
    "Expected exactly two top-level validation "
    "oracle gates (WebQSP and CWQ)."
)


first_gate_idx, first_gate_node = (
    gate_nodes[0]
)

second_gate_idx, second_gate_node = (
    gate_nodes[1]
)


print(
    " WebQSP gate line:",
    first_gate_node.lineno
)

print(
    " CWQ gate line:",
    second_gate_node.lineno
)


# ======================================================================
# 5. BUILD EXACT POST-GATE RESUME MODULE
# ======================================================================
#
# Original Cell 16 failed while executing the SECOND validation gate.
#
# Everything before it already executed successfully.
#
# We therefore execute only the top-level statements AFTER that gate.
# ======================================================================

resume_nodes = (
    cell16_tree.body[
        second_gate_idx + 1:
    ]
)


assert len(resume_nodes) > 0, (
    "No Cell-16 statements exist after the CWQ gate."
)


resume_module = ast.Module(
    body=resume_nodes,
    type_ignores=[]
)


ast.fix_missing_locations(
    resume_module
)


# Raw source after the exact second gate, for audit only.
source_lines = (
    CELL16_ORIGINAL_SOURCE
    .splitlines(
        keepends=True
    )
)


CELL16_POST_GATE_RAW_SOURCE = "".join(
    source_lines[
        second_gate_node.end_lineno:
    ]
)


CELL16_POST_GATE_SHA256 = (
    hashlib.sha256(
        CELL16_POST_GATE_RAW_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nPost-CWQ-gate resume source:"
)

print(
    " statements:",
    len(resume_nodes)
)

print(
    " SHA256:",
    CELL16_POST_GATE_SHA256
)


# ======================================================================
# 6. SAFETY CHECKS ON REMAINDER
# ======================================================================

# The remainder must NOT redefine the profiler we just repaired.
assert (
    "def profile_rq1_dataset_16"
    not in
    CELL16_POST_GATE_RAW_SOURCE
), (
    "Unsafe resume: remainder unexpectedly "
    "redefines profile_rq1_dataset_16."
)


# It must not redefine the oracle repair.
assert (
    "def make_oracle_functions_16"
    not in
    CELL16_POST_GATE_RAW_SOURCE
)


print(
    "Resume-source safety checks: PASSED"
)


# ======================================================================
# 7. EXECUTE THE ORIGINAL CELL-16 REMAINDER
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "RESUMING ORIGINAL CELL 16 AFTER CWQ VALIDATION GATE"
)

print(
    "=" * 110
)

print(
    "\nUsing repaired, validation-proven "
    "profile_rq1_dataset_16."
)


CELL16_R2_RESUME_STARTED = True


exec(
    compile(
        resume_module,
        filename="<CELL16_R2_EXACT_RESUME>",
        mode="exec"
    ),
    globals()
)


CELL16_R2_RESUME_COMPLETE = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2 RESUME COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nOriginal Cell 16 rerun: NO"
)

print(
    "M3 rerun:              NO"
)

print(
    "AFP changed:           NO"
)

print(
    "TEST tuning:           NO"
)

print(
    "Frozen TEST plans:     UNCHANGED"
)

print(
    "\nNEXT ACTION:"
)

print(
    "Paste the final Cell 16-R2 output here "
    "before running Cell 17."
)

CELL 16-R2 — SAFE RESUME
Freeze gate:                 PASSED
Oracle repair validation:    PASSED
Repaired profiler in memory: YES

Frozen TEST planning SHA gate: PASSED
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ:    95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

Original Cell 16 recovered from history:
 history index: 11
 source SHA256: 036e3f0d2e06d59edc9de544a2d92d08cf9d7fe6aa3337b5db46b9ab60f5b67a

Top-level validation gates found: 0


AssertionError: Expected exactly two top-level validation oracle gates (WebQSP and CWQ).

In [12]:
# ======================================================================
# CELL 16-R2A
# LOCATE ORIGINAL VALIDATION-GATE CALLS EXACTLY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# This is diagnostic only.
# No TEST profiling.
# No AFP changes.
# No files overwritten.
# ======================================================================

import ast
import hashlib


print("=" * 110)
print("CELL 16-R2A — EXACT VALIDATION-GATE LOCATION")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Scientific-state gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

print("\nFreeze + oracle repair gates: PASSED")


# ----------------------------------------------------------------------
# 2. Reuse exact Cell-16 source already recovered by R2
# ----------------------------------------------------------------------

if "CELL16_ORIGINAL_SOURCE" in globals():

    source = CELL16_ORIGINAL_SOURCE

    print(
        "\nUsing CELL16_ORIGINAL_SOURCE already recovered."
    )

else:

    ip = get_ipython()

    history = list(
        ip.history_manager.input_hist_raw
    )

    candidates = []

    for history_index, src in enumerate(history):

        if not isinstance(src, str):
            continue

        if (
            "def profile_rq1_dataset_16" in src
            and
            "def oracle_validation_gate_16" in src
            and
            "EXPECTED_VAL_ORACLE_16" in src
        ):

            candidates.append(
                (history_index, src)
            )

    assert candidates, (
        "Could not recover original Cell 16 source."
    )

    CELL16_HISTORY_INDEX, source = (
        candidates[-1]
    )

    CELL16_ORIGINAL_SOURCE = source


print(
    "Cell-16 SHA256:",
    hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()
)


# ----------------------------------------------------------------------
# 3. Parse source and build parent map
# ----------------------------------------------------------------------

tree = ast.parse(source)

parent = {}

for node in ast.walk(tree):

    for child in ast.iter_child_nodes(node):

        parent[child] = node


# ----------------------------------------------------------------------
# 4. Find ALL calls to oracle_validation_gate_16 anywhere
# ----------------------------------------------------------------------

gate_calls = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if (
        isinstance(func, ast.Name)
        and
        func.id == "oracle_validation_gate_16"
    ):

        gate_calls.append(node)


gate_calls = sorted(
    gate_calls,
    key=lambda n: (
        n.lineno,
        n.col_offset
    )
)


print(
    "\nTotal oracle_validation_gate_16 calls found:",
    len(gate_calls)
)


# ----------------------------------------------------------------------
# 5. Find enclosing top-level statement for each call
# ----------------------------------------------------------------------

def enclosing_top_level(node):

    current = node

    while current in parent:

        p = parent[current]

        if p is tree:

            return current

        current = p

    return None


lines = source.splitlines()


for i, call in enumerate(
    gate_calls,
    start=1
):

    top = enclosing_top_level(call)

    print(
        "\n"
        + "-" * 110
    )

    print(
        f"GATE CALL #{i}"
    )

    print(
        "call lines:",
        call.lineno,
        "-",
        getattr(
            call,
            "end_lineno",
            call.lineno
        )
    )

    print(
        "column:",
        call.col_offset
    )

    if top is not None:

        print(
            "enclosing TOP-LEVEL type:",
            type(top).__name__
        )

        print(
            "top-level lines:",
            top.lineno,
            "-",
            getattr(
                top,
                "end_lineno",
                top.lineno
            )
        )


    # Ancestor chain
    chain = []

    cur = call

    while cur in parent:

        cur = parent[cur]

        chain.append(
            type(cur).__name__
        )

        if cur is tree:
            break

    print(
        "ancestor chain:",
        " -> ".join(chain)
    )


    # Exact call source
    call_src = ast.get_source_segment(
        source,
        call
    )

    print(
        "\nExact call:"
    )

    print(
        call_src
    )


    # Surrounding source
    start = max(
        0,
        call.lineno - 15
    )

    end = min(
        len(lines),
        getattr(
            call,
            "end_lineno",
            call.lineno
        ) + 20
    )

    print(
        "\nSurrounding source:"
    )

    for line_no in range(
        start,
        end
    ):

        marker = (
            ">>>"
            if (
                call.lineno - 1
                <= line_no
                <=
                getattr(
                    call,
                    "end_lineno",
                    call.lineno
                ) - 1
            )
            else "   "
        )

        print(
            f"{marker} "
            f"{line_no + 1:04d}: "
            f"{lines[line_no]}"
        )


# ----------------------------------------------------------------------
# 6. Raw text occurrence check
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "RAW SOURCE OCCURRENCES"
)

print(
    "=" * 110
)


for line_no, line in enumerate(
    lines,
    start=1
):

    if (
        "oracle_validation_gate_16("
        in line
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ----------------------------------------------------------------------
# 7. Check what comes immediately after the second gate
# ----------------------------------------------------------------------

if len(gate_calls) >= 2:

    second = gate_calls[1]

    second_end = getattr(
        second,
        "end_lineno",
        second.lineno
    )

    print(
        "\n"
        + "=" * 110
    )

    print(
        "SOURCE AFTER SECOND VALIDATION GATE"
    )

    print(
        "=" * 110
    )

    for line_no in range(
        second_end,
        min(
            len(lines),
            second_end + 60
        )
    ):

        print(
            f"{line_no + 1:04d}: "
            f"{lines[line_no]}"
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2A COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nNo TEST profiling executed."
)

print(
    "No AFP setting changed."
)

print(
    "No artifact overwritten."
)

print(
    "\nPaste this output here."
)

CELL 16-R2A — EXACT VALIDATION-GATE LOCATION

Freeze + oracle repair gates: PASSED

Using CELL16_ORIGINAL_SOURCE already recovered.
Cell-16 SHA256: 036e3f0d2e06d59edc9de544a2d92d08cf9d7fe6aa3337b5db46b9ab60f5b67a

Total oracle_validation_gate_16 calls found: 0

RAW SOURCE OCCURRENCES

CELL 16-R2A COMPLETE

No TEST profiling executed.
No AFP setting changed.
No artifact overwritten.

Paste this output here.


In [13]:
# ======================================================================
# CELL 16-R2B
# ROBUST ORIGINAL CELL-16 LOCATOR + SAFE POST-GATE RESUME
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun original Cell 16
#   - rerun Cell 16-R1
#   - rerun Cell 16-R2 / R2A
#   - rerun M3
#   - restart kernel
#
# This locates the REAL original Cell 16 using AST structure,
# not string matching.
# ======================================================================

import ast
import hashlib
from pathlib import Path


print("=" * 110)
print("CELL 16-R2B — ROBUST SAFE RESUME")
print("=" * 110)


# ======================================================================
# 1. SCIENTIFIC STATE GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

assert "profile_rq1_dataset_16" in globals()
assert callable(profile_rq1_dataset_16)

assert "web_oracle_16r1" in globals()
assert "cwq_oracle_16r1" in globals()


assert (
    int(
        web_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 11717
)

assert (
    int(
        cwq_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 1176294
)


print("Freeze gate:              PASSED")
print("Oracle repair validation: PASSED")


# ======================================================================
# 2. VERIFY FROZEN TEST PLAN FILES AGAIN
# ======================================================================

WEB_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)


assert WEB_TEST_PATH.exists()
assert CWQ_TEST_PATH.exists()


def file_sha256_r2b(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


WEB_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


assert (
    file_sha256_r2b(WEB_TEST_PATH)
    ==
    WEB_EXPECTED_SHA
)

assert (
    file_sha256_r2b(CWQ_TEST_PATH)
    ==
    CWQ_EXPECTED_SHA
)


print("Frozen TEST plan SHA gate: PASSED")


# ======================================================================
# 3. SEARCH IPYTHON HISTORY BY ACTUAL AST DEFINITIONS
# ======================================================================

ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


real_candidates = []


for hist_index, source in enumerate(history):

    if not isinstance(source, str):
        continue

    if len(source) < 1000:
        continue

    try:

        tree = ast.parse(source)

    except Exception:

        continue


    # --------------------------------------------------------------
    # REAL function definitions, not text appearing inside strings
    # --------------------------------------------------------------

    top_level_function_names = {
        node.name
        for node in tree.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
    }


    if (
        "profile_rq1_dataset_16"
        not in
        top_level_function_names
    ):
        continue


    if (
        "oracle_validation_gate_16"
        not in
        top_level_function_names
    ):
        continue


    # --------------------------------------------------------------
    # REAL function calls anywhere in AST
    # --------------------------------------------------------------

    gate_calls = []


    for node in ast.walk(tree):

        if not isinstance(
            node,
            ast.Call
        ):
            continue

        func = node.func

        if (
            isinstance(
                func,
                ast.Name
            )
            and
            func.id
            ==
            "oracle_validation_gate_16"
        ):

            gate_calls.append(
                node
            )


    gate_calls = sorted(
        gate_calls,
        key=lambda x:
            (
                x.lineno,
                x.col_offset
            )
    )


    if len(gate_calls) != 2:
        continue


    real_candidates.append(
        {
            "history_index":
                hist_index,

            "source":
                source,

            "tree":
                tree,

            "gate_calls":
                gate_calls,

            "sha256":
                hashlib.sha256(
                    source.encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "chars":
                len(source),
        }
    )


print(
    "\nREAL Cell-16 candidates found:",
    len(real_candidates)
)


for candidate in real_candidates:

    print(
        " history=",
        candidate[
            "history_index"
        ],

        "| chars=",
        candidate[
            "chars"
        ],

        "| SHA=",
        candidate[
            "sha256"
        ],

        "| gates=",
        [
            x.lineno
            for x in
            candidate[
                "gate_calls"
            ]
        ]
    )


assert real_candidates, (
    "No IPython history entry actually defines "
    "profile_rq1_dataset_16 + oracle_validation_gate_16 "
    "and contains exactly two real gate calls."
)


# Prefer the latest actual definition cell.
original = sorted(
    real_candidates,
    key=lambda x:
        x["history_index"]
)[-1]


CELL16_TRUE_HISTORY_INDEX = (
    original[
        "history_index"
    ]
)

CELL16_TRUE_ORIGINAL_SOURCE = (
    original[
        "source"
    ]
)

CELL16_TRUE_ORIGINAL_SHA256 = (
    original[
        "sha256"
    ]
)

cell16_true_tree = (
    original[
        "tree"
    ]
)

gate_calls = (
    original[
        "gate_calls"
    ]
)


print(
    "\nSelected REAL original Cell 16:"
)

print(
    " history index:",
    CELL16_TRUE_HISTORY_INDEX
)

print(
    " characters:",
    len(
        CELL16_TRUE_ORIGINAL_SOURCE
    )
)

print(
    " SHA256:",
    CELL16_TRUE_ORIGINAL_SHA256
)

print(
    " gate lines:",
    [
        x.lineno
        for x in gate_calls
    ]
)


# ======================================================================
# 4. VERIFY BOTH GATE CALLS ARE TOP-LEVEL EXPRESSIONS
# ======================================================================

parent = {}


for node in ast.walk(
    cell16_true_tree
):

    for child in ast.iter_child_nodes(
        node
    ):

        parent[
            child
        ] = node


def enclosing_top_level_r2b(
    node
):

    current = node

    while current in parent:

        p = parent[
            current
        ]

        if p is cell16_true_tree:
            return current

        current = p

    return None


gate_top_nodes = []


for i, call in enumerate(
    gate_calls,
    start=1
):

    top = enclosing_top_level_r2b(
        call
    )

    assert top is not None

    gate_top_nodes.append(
        top
    )

    print(
        f"\nGate #{i}:"
    )

    print(
        " call line:",
        call.lineno
    )

    print(
        " top-level type:",
        type(top).__name__
    )

    print(
        " top-level lines:",
        top.lineno,
        "-",
        getattr(
            top,
            "end_lineno",
            top.lineno
        )
    )

    print(
        " exact call:"
    )

    print(
        ast.get_source_segment(
            CELL16_TRUE_ORIGINAL_SOURCE,
            call
        )
    )


# For the original Cell 16 design these validation calls should be
# standalone top-level expressions.
assert all(
    isinstance(
        x,
        ast.Expr
    )
    for x in gate_top_nodes
), (
    "Validation gate is nested inside a compound "
    "statement. Stop rather than resume ambiguously."
)


# ======================================================================
# 5. FIND SECOND GATE'S TOP-LEVEL BODY INDEX
# ======================================================================

second_gate_top = (
    gate_top_nodes[1]
)


second_gate_body_index = None


for idx, node in enumerate(
    cell16_true_tree.body
):

    if node is second_gate_top:

        second_gate_body_index = idx
        break


assert second_gate_body_index is not None


print(
    "\nSecond validation gate top-level body index:",
    second_gate_body_index
)


# ======================================================================
# 6. BUILD EXACT REMAINDER AFTER SECOND VALIDATION GATE
# ======================================================================

resume_nodes = (
    cell16_true_tree.body[
        second_gate_body_index + 1:
    ]
)


assert resume_nodes, (
    "Nothing remains after the second validation gate."
)


resume_module = ast.Module(
    body=resume_nodes,
    type_ignores=[]
)


ast.fix_missing_locations(
    resume_module
)


source_lines = (
    CELL16_TRUE_ORIGINAL_SOURCE
    .splitlines(
        keepends=True
    )
)


second_gate_end_line = getattr(
    second_gate_top,
    "end_lineno",
    second_gate_top.lineno
)


CELL16_R2B_POST_GATE_SOURCE = "".join(
    source_lines[
        second_gate_end_line:
    ]
)


CELL16_R2B_POST_GATE_SHA256 = (
    hashlib.sha256(
        CELL16_R2B_POST_GATE_SOURCE
        .encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nPost-gate remainder:"
)

print(
    " statements:",
    len(
        resume_nodes
    )
)

print(
    " characters:",
    len(
        CELL16_R2B_POST_GATE_SOURCE
    )
)

print(
    " SHA256:",
    CELL16_R2B_POST_GATE_SHA256
)


# ======================================================================
# 7. CRITICAL SAFETY CHECKS
# ======================================================================

assert (
    "def profile_rq1_dataset_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)

assert (
    "def make_oracle_functions_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)

assert (
    "def oracle_validation_gate_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)


# It should contain references to final TEST processing.
assert (
    "test"
    in
    CELL16_R2B_POST_GATE_SOURCE.lower()
), (
    "Post-gate source does not appear to contain "
    "the final TEST section."
)


print(
    "Post-gate safety checks: PASSED"
)


# ======================================================================
# 8. CONFIRM REPAIRED FUNCTION IS STILL ACTIVE
# ======================================================================

assert (
    globals().get(
        "CELL16_REPAIRED_PROFILE_SHA256"
    )
    ==
    "3413dc5efa00a1be3f9c56f79677bbd97232293b9fb4d15d5a562bacb434287f"
)


print(
    "Repaired profiler identity: PASSED"
)


# ======================================================================
# 9. RESUME ORIGINAL CELL 16
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "RESUMING ORIGINAL CELL 16 AFTER VALIDATED CWQ GATE"
)

print(
    "=" * 110
)

print(
    "\nThis will now execute the frozen TEST RQ1 section."
)

print(
    "No planner regeneration."
)

print(
    "No AFP retuning."
)


CELL16_R2B_RESUME_STARTED = True


exec(
    compile(
        resume_module,
        filename="<CELL16_R2B_TRUE_POST_GATE_RESUME>",
        mode="exec"
    ),
    globals()
)


CELL16_R2B_RESUME_COMPLETE = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2B RESUME COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nOriginal Cell 16 fully rerun: NO"
)

print(
    "M3 rerun:                   NO"
)

print(
    "Frozen TEST plans changed:  NO"
)

print(
    "AFP changed:                NO"
)

print(
    "TEST tuning:                NO"
)

print(
    "\nNEXT ACTION:"
)

print(
    "PASTE THE FINAL OUTPUT HERE BEFORE RUNNING CELL 17."
)

CELL 16-R2B — ROBUST SAFE RESUME
Freeze gate:              PASSED
Oracle repair validation: PASSED
Frozen TEST plan SHA gate: PASSED

REAL Cell-16 candidates found: 1
 history= 5 | chars= 51180 | SHA= 2f72f6ab2ac9ec11ac7d97780bd4ae580dc75908c1fca5391da39c9aec6a61e3 | gates= [1677, 1683]

Selected REAL original Cell 16:
 history index: 5
 characters: 51180
 SHA256: 2f72f6ab2ac9ec11ac7d97780bd4ae580dc75908c1fca5391da39c9aec6a61e3
 gate lines: [1677, 1683]

Gate #1:
 call line: 1677
 top-level type: Expr
 top-level lines: 1677 - 1680
 exact call:
oracle_validation_gate_16(
    "webqsp",
    webqsp_val_plan_rows
)

Gate #2:
 call line: 1683
 top-level type: Expr
 top-level lines: 1683 - 1686
 exact call:
oracle_validation_gate_16(
    "cwq",
    cwq_val_plan_rows
)

Second validation gate top-level body index: 41

Post-gate remainder:
 statements: 35
 characters: 16333
 SHA256: ceb29540b9522fd9b88706c8f99564cab7c96e44a4d329dfa1613e5eabbe61c8
Post-gate safety checks: PASSED
Repaired profile

webqsp final TEST RQ1:   0%|          | 0/1628 [00:00<?, ?it/s]

cwq final TEST RQ1:   0%|          | 0/3531 [00:00<?, ?it/s]


WEBQSP FINAL FROZEN-TEST RQ1
questions: 1628
plans: 4880
active hop rows: 4880
active prefixes: 4978
edges examined: 2254484
candidate branches: 0
RoG reachable questions: 0 (0.0000%)
decision hops: 0 (0.0000% of intermediate-active hops)

POST-FREEZE ORACLE
doomed-prefix edges: 2254484 (100.0000%)
oracle-prunable intermediate branches: 0 / 0 (0.0000%)
oracle-avoidable downstream edges: 0 (0.0000%)
doomed final-hop protected branches: 0

CWQ FINAL FROZEN-TEST RQ1
questions: 3531
plans: 10537
active hop rows: 10537
active prefixes: 16344
edges examined: 3461479
candidate branches: 0
RoG reachable questions: 0 (0.0000%)
decision hops: 0 (0.0000% of intermediate-active hops)

POST-FREEZE ORACLE
doomed-prefix edges: 3461479 (100.0000%)
oracle-prunable intermediate branches: 0 / 0 (0.0000%)
oracle-avoidable downstream edges: 0 (0.0000%)
doomed final-hop protected branches: 0

=== CELL 16: FINAL FROZEN-TEST RQ1 PROFILING COMPLETE ===
AFP changed after freeze: NO
TEST used for tuning: NO
POS

In [14]:
# ======================================================================
# CELL 16-R3
# TEST RELATION-MATCHING DIAGNOSTIC
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun Cell 16
#   - rerun M3
#   - restart kernel
#
# No tuning. No TEST plans changed. No artifacts overwritten.
# ======================================================================

from collections import Counter
import json
import re


print("=" * 110)
print("CELL 16-R3 — TEST RELATION-MATCHING DIAGNOSTIC")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Scientific gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()

print("\nFreeze / oracle repair gates: PASSED")


# ----------------------------------------------------------------------
# 2. Relation normalization probes ONLY
# ----------------------------------------------------------------------

def graph_relation_set(rec):

    return {
        str(triple[1]).strip()
        for triple in rec["graph"]
    }


def raw_plan_tokens(rec):

    plans = rec.get(
        "predicted_paths",
        []
    )

    out = []

    for p in plans:

        if isinstance(p, (list, tuple)):
            out.extend(
                str(x)
                for x in p
            )
        else:
            out.append(
                str(p)
            )

    return out


def token_variants(token):

    s = str(token)

    variants = {
        "raw":
            s,

        "strip":
            s.strip(),

        "strip_quotes":
            s.strip()
             .strip("'\""),

        "remove_path_tags":
            s.replace(
                "<PATH>",
                ""
            ).replace(
                "</PATH>",
                ""
            ).strip(),

        "remove_brackets":
            s.strip()
             .strip("[](){}")
             .strip(),

        "last_space_token":
            s.strip().split()[-1]
            if s.strip()
            else "",
    }

    return variants


# ----------------------------------------------------------------------
# 3. Print several raw TEST examples
# ----------------------------------------------------------------------

def inspect_examples(
    dataset_name,
    rows,
    n=8
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        dataset_name,
        "RAW EXAMPLES"
    )

    print(
        "=" * 110
    )

    shown = 0

    for rec in rows:

        tokens = raw_plan_tokens(
            rec
        )

        if not tokens:
            continue

        graph_rels = graph_relation_set(
            rec
        )

        print(
            "\nQuestion ID:",
            rec["id"]
        )

        print(
            "Question:",
            rec["question"][:180]
        )

        print(
            "predicted_paths:",
            repr(
                rec["predicted_paths"]
            )
        )

        print(
            "\nFirst planner token variants:"
        )

        first = tokens[0]

        for k, v in token_variants(
            first
        ).items():

            print(
                f"  {k:<20}: {repr(v)} "
                f"| exact graph match={v in graph_rels}"
            )

        print(
            "\nSample graph relations:"
        )

        for x in list(
            graph_rels
        )[:12]:
            print(
                " ",
                repr(x)
            )

        shown += 1

        if shown >= n:
            break


inspect_examples(
    "WEBQSP",
    webqsp_test_plan_rows
)

inspect_examples(
    "CWQ",
    cwq_test_plan_rows
)


# ----------------------------------------------------------------------
# 4. Dataset-wide overlap statistics
# ----------------------------------------------------------------------

def overlap_stats(
    dataset_name,
    rows
):

    stats = Counter()

    token_examples = {
        "raw_miss_strip_hit": [],
        "all_simple_miss": [],
    }


    for rec in rows:

        graph_rels = graph_relation_set(
            rec
        )

        tokens = raw_plan_tokens(
            rec
        )

        for token in tokens:

            stats[
                "tokens"
            ] += 1

            variants = token_variants(
                token
            )

            if variants[
                "raw"
            ] in graph_rels:

                stats[
                    "raw_match"
                ] += 1

            if variants[
                "strip"
            ] in graph_rels:

                stats[
                    "strip_match"
                ] += 1

            if variants[
                "strip_quotes"
            ] in graph_rels:

                stats[
                    "strip_quotes_match"
                ] += 1

            if variants[
                "remove_path_tags"
            ] in graph_rels:

                stats[
                    "remove_path_tags_match"
                ] += 1

            if variants[
                "remove_brackets"
            ] in graph_rels:

                stats[
                    "remove_brackets_match"
                ] += 1


            if (
                variants["raw"]
                not in graph_rels
                and
                variants["strip"]
                in graph_rels
                and
                len(
                    token_examples[
                        "raw_miss_strip_hit"
                    ]
                ) < 10
            ):

                token_examples[
                    "raw_miss_strip_hit"
                ].append(
                    (
                        token,
                        variants["strip"]
                    )
                )


            if not any(
                variants[k] in graph_rels

                for k in [
                    "raw",
                    "strip",
                    "strip_quotes",
                    "remove_path_tags",
                    "remove_brackets",
                ]
            ):

                if len(
                    token_examples[
                        "all_simple_miss"
                    ]
                ) < 15:

                    token_examples[
                        "all_simple_miss"
                    ].append(
                        {
                            "id":
                                rec["id"],

                            "token":
                                token,

                            "question":
                                rec["question"][:120],
                        }
                    )


    print(
        "\n"
        + "=" * 110
    )

    print(
        dataset_name,
        "RELATION OVERLAP"
    )

    print(
        "=" * 110
    )

    for key in [
        "tokens",
        "raw_match",
        "strip_match",
        "strip_quotes_match",
        "remove_path_tags_match",
        "remove_brackets_match",
    ]:

        print(
            f"{key:<28}:",
            stats[key]
        )


    denom = max(
        1,
        stats["tokens"]
    )

    print(
        "\nRaw match %:",
        100
        * stats["raw_match"]
        / denom
    )

    print(
        "Strip match %:",
        100
        * stats["strip_match"]
        / denom
    )


    print(
        "\nraw miss / strip hit examples:"
    )

    for x in token_examples[
        "raw_miss_strip_hit"
    ]:
        print(
            repr(x)
        )


    print(
        "\nSimple-normalization misses:"
    )

    for x in token_examples[
        "all_simple_miss"
    ]:
        print(
            x
        )


    return stats


WEB_REL_STATS_R3 = overlap_stats(
    "WEBQSP",
    webqsp_test_plan_rows
)

CWQ_REL_STATS_R3 = overlap_stats(
    "CWQ",
    cwq_test_plan_rows
)


# ----------------------------------------------------------------------
# 5. Compare TEST token shape to frozen VALIDATION token shape
# ----------------------------------------------------------------------

def token_shape_summary(
    rows,
    label
):

    examples = []

    lengths = Counter()

    contains_arrow = 0
    contains_space = 0
    contains_path_tag = 0
    contains_brackets = 0

    total = 0


    for rec in rows:

        for token in raw_plan_tokens(
            rec
        ):

            total += 1

            lengths[
                len(token)
            ] += 1

            if " -> " in token:
                contains_arrow += 1

            if " " in token:
                contains_space += 1

            if (
                "<PATH>" in token
                or
                "</PATH>" in token
            ):
                contains_path_tag += 1

            if any(
                x in token
                for x in "[](){}"
            ):
                contains_brackets += 1

            if len(examples) < 12:
                examples.append(
                    token
                )


    print(
        "\n"
        + "=" * 110
    )

    print(
        label,
        "TOKEN SHAPE"
    )

    print(
        "=" * 110
    )

    print(
        "tokens:",
        total
    )

    print(
        "contains ' -> ':",
        contains_arrow
    )

    print(
        "contains spaces:",
        contains_space
    )

    print(
        "contains PATH tags:",
        contains_path_tag
    )

    print(
        "contains brackets:",
        contains_brackets
    )

    print(
        "\nexamples:"
    )

    for x in examples:
        print(
            " ",
            repr(x)
        )


token_shape_summary(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

token_shape_summary(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

token_shape_summary(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)

token_shape_summary(
    cwq_test_plan_rows,
    "CWQ TEST"
)


# ----------------------------------------------------------------------
# 6. Final status
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R3 DIAGNOSTIC COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nCurrent Cell-16 TEST result is NOT accepted."
)

print(
    "No TEST plan regenerated."
)

print(
    "No tuning performed."
)

print(
    "No AFP parameter changed."
)

print(
    "\nPASTE THIS OUTPUT HERE."
)

CELL 16-R3 — TEST RELATION-MATCHING DIAGNOSTIC

Freeze / oracle repair gates: PASSED

WEBQSP RAW EXAMPLES

Question ID: WebQTest-0
Question: what does jamaican people speak
predicted_paths: [['location. location. l anguages _ sp oken'], ['location. language. count ries _ sp oken _ in'], ['location. language. main _ country']]

First planner token variants:
  raw                 : 'location. location. l anguages _ sp oken' | exact graph match=False
  strip               : 'location. location. l anguages _ sp oken' | exact graph match=False
  strip_quotes        : 'location. location. l anguages _ sp oken' | exact graph match=False
  remove_path_tags    : 'location. location. l anguages _ sp oken' | exact graph match=False
  remove_brackets     : 'location. location. l anguages _ sp oken' | exact graph match=False
  last_space_token    : 'oken' | exact graph match=False

Sample graph relations:
  'symbols.namesake.named_after'
  'olympics.olympic_participating_country.olympics_participat

In [17]:
# ======================================================================
# CELL 16-R4
# TOKENIZER / WHITESPACE-CANONICALIZATION FIDELITY DIAGNOSTIC
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - rewrite TEST JSONL files
#   - restart kernel
#
# This is diagnostic only.
# ======================================================================

import re
import sys
from collections import Counter, defaultdict

import transformers


print("=" * 115)
print("CELL 16-R4 — PLANNER DECODING / RELATION-ID FIDELITY DIAGNOSTIC")
print("=" * 115)


# ======================================================================
# 1. HARD GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert "tokenizer" in globals()

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()

assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()


print("\nFreeze / repair gates: PASSED")


# ======================================================================
# 2. CURRENT TOKENIZER ENVIRONMENT
# ======================================================================

print("\n" + "=" * 115)
print("A. CURRENT TOKENIZER ENVIRONMENT")
print("=" * 115)

print("Python:", sys.version.split()[0])
print("transformers:", transformers.__version__)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer module:", tokenizer.__class__.__module__)
print("is_fast:", getattr(tokenizer, "is_fast", None))
print("legacy:", getattr(tokenizer, "legacy", "ATTRIBUTE_NOT_PRESENT"))
print(
    "clean_up_tokenization_spaces:",
    getattr(
        tokenizer,
        "clean_up_tokenization_spaces",
        "ATTRIBUTE_NOT_PRESENT"
    )
)

print(
    "name_or_path:",
    getattr(
        tokenizer,
        "name_or_path",
        None
    )
)


# ======================================================================
# 3. CANONICAL RELATION-ID RULE
#
# Freebase relation identifiers in the graph are schema identifiers,
# not natural-language strings. We first establish empirically whether
# graph relation IDs contain whitespace.
# ======================================================================

def compact_relation_id_16r4(x):

    return re.sub(
        r"\s+",
        "",
        str(x)
    )


def graph_relations_16r4(rec):

    return {
        str(t[1]).strip()
        for t in rec["graph"]
    }


def plan_tokens_16r4(rec):

    out = []

    for plan in rec.get(
        "predicted_paths",
        []
    ):

        if isinstance(
            plan,
            (list, tuple)
        ):

            out.extend(
                str(x)
                for x in plan
            )

        else:

            out.append(
                str(plan)
            )

    return out


# ======================================================================
# 4. VERIFY GRAPH RELATION VOCABULARY ITSELF
# ======================================================================

def graph_relation_schema_audit(
    rows,
    label
):

    total = 0
    with_whitespace = 0
    examples = []


    for rec in rows:

        for triple in rec["graph"]:

            relation = str(
                triple[1]
            )

            total += 1

            if re.search(
                r"\s",
                relation
            ):

                with_whitespace += 1

                if len(examples) < 10:

                    examples.append(
                        relation
                    )


    print(f"\n{label}")
    print(" graph edge relations:", total)
    print(" relations containing whitespace:", with_whitespace)

    if examples:

        print(" examples:", examples)


    return {
        "total":
            total,

        "with_whitespace":
            with_whitespace,
    }


print("\n" + "=" * 115)
print("B. GRAPH RELATION-ID SCHEMA AUDIT")
print("=" * 115)


WEB_GRAPH_SCHEMA = graph_relation_schema_audit(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

CWQ_GRAPH_SCHEMA = graph_relation_schema_audit(
    cwq_test_plan_rows,
    "CWQ TEST"
)


assert WEB_GRAPH_SCHEMA[
    "with_whitespace"
] == 0

assert CWQ_GRAPH_SCHEMA[
    "with_whitespace"
] == 0


print(
    "\nGraph relation IDs are whitespace-free: PASSED"
)


# ======================================================================
# 5. RAW vs WHITESPACE-COMPACTED MATCHING
# ======================================================================

def compact_match_audit(
    rows,
    label
):

    stats = Counter()

    examples = []

    question_any_raw = 0
    question_any_compact = 0

    first_hop_raw = 0
    first_hop_compact = 0

    n_questions_with_plans = 0


    for rec in rows:

        relations = graph_relations_16r4(
            rec
        )

        tokens = plan_tokens_16r4(
            rec
        )


        any_raw = False
        any_compact = False


        if tokens:

            n_questions_with_plans += 1


        for token in tokens:

            raw = str(token)

            compact = compact_relation_id_16r4(
                raw
            )

            stats[
                "tokens"
            ] += 1


            if raw in relations:

                stats[
                    "raw_exact"
                ] += 1

                any_raw = True


            if compact in relations:

                stats[
                    "compact_exact"
                ] += 1

                any_compact = True


            if (
                raw not in relations
                and
                compact in relations
            ):

                stats[
                    "recovered_by_compaction"
                ] += 1

                if len(examples) < 20:

                    examples.append(
                        (
                            raw,
                            compact
                        )
                    )


        # --------------------------------------------------------------
        # First relation of each predicted plan
        # --------------------------------------------------------------

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):

                continue

            if len(plan) == 0:

                continue


            raw_first = str(
                plan[0]
            )

            compact_first = (
                compact_relation_id_16r4(
                    raw_first
                )
            )


            stats[
                "first_hop_tokens"
            ] += 1


            if raw_first in relations:

                first_hop_raw += 1


            if compact_first in relations:

                first_hop_compact += 1


        if any_raw:
            question_any_raw += 1

        if any_compact:
            question_any_compact += 1


    print("\n" + "-" * 115)
    print(label)
    print("-" * 115)

    for key in [
        "tokens",
        "raw_exact",
        "compact_exact",
        "recovered_by_compaction",
        "first_hop_tokens",
    ]:

        print(
            f"{key:<30}:",
            stats[key]
        )


    denom = max(
        1,
        stats["tokens"]
    )

    first_denom = max(
        1,
        stats["first_hop_tokens"]
    )


    print(
        f"raw exact %:      "
        f"{100 * stats['raw_exact'] / denom:.4f}%"
    )

    print(
        f"compact exact %:  "
        f"{100 * stats['compact_exact'] / denom:.4f}%"
    )

    print(
        f"first-hop raw %:  "
        f"{100 * first_hop_raw / first_denom:.4f}%"
    )

    print(
        f"first-hop compact %: "
        f"{100 * first_hop_compact / first_denom:.4f}%"
    )

    print(
        "questions with any raw graph relation:",
        question_any_raw,
        "/",
        n_questions_with_plans
    )

    print(
        "questions with any compact graph relation:",
        question_any_compact,
        "/",
        n_questions_with_plans
    )


    print(
        "\nRecovered examples:"
    )

    for raw, compact in examples:

        print(
            " RAW:    ",
            repr(raw)
        )

        print(
            " COMPACT:",
            repr(compact)
        )

        print()


    return {
        "stats":
            stats,

        "question_any_raw":
            question_any_raw,

        "question_any_compact":
            question_any_compact,

        "first_hop_raw":
            first_hop_raw,

        "first_hop_compact":
            first_hop_compact,
    }


print("\n" + "=" * 115)
print("C. TEST RAW-vs-COMPACT MATCH AUDIT")
print("=" * 115)


WEB_COMPACT_TEST = compact_match_audit(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

CWQ_COMPACT_TEST = compact_match_audit(
    cwq_test_plan_rows,
    "CWQ TEST"
)


# ======================================================================
# 6. VALIDATION INVARIANCE CHECK
#
# A legitimate representation repair should essentially be a no-op on
# the already-correct frozen validation planner outputs.
# ======================================================================

def validation_compaction_audit(
    rows,
    label
):

    tokens = 0
    changed = 0

    changed_examples = []


    for rec in rows:

        for token in plan_tokens_16r4(
            rec
        ):

            tokens += 1

            compact = compact_relation_id_16r4(
                token
            )

            if compact != token:

                changed += 1

                if len(
                    changed_examples
                ) < 20:

                    changed_examples.append(
                        (
                            token,
                            compact
                        )
                    )


    print(f"\n{label}")
    print(" tokens:", tokens)
    print(" tokens altered by whitespace compaction:", changed)

    if tokens:

        print(
            " changed %:",
            f"{100 * changed / tokens:.6f}%"
        )


    if changed_examples:

        print(
            "\n changed examples:"
        )

        for x in changed_examples:

            print(
                " ",
                repr(x[0]),
                "->",
                repr(x[1])
            )


    return {
        "tokens":
            tokens,

        "changed":
            changed,

        "examples":
            changed_examples,
    }


print("\n" + "=" * 115)
print("D. FROZEN VALIDATION INVARIANCE")
print("=" * 115)


WEB_VAL_COMPACT = (
    validation_compaction_audit(
        webqsp_val_plan_rows,
        "WEBQSP VALIDATION"
    )
)

CWQ_VAL_COMPACT = (
    validation_compaction_audit(
        cwq_val_plan_rows,
        "CWQ VALIDATION"
    )
)


# ======================================================================
# 7. TOKENIZER ROUND-TRIP PROBES
# ======================================================================

print("\n" + "=" * 115)
print("E. CURRENT TOKENIZER ROUND-TRIP PROBES")
print("=" * 115)


probe_relations = [
    "people.person.place_of_birth",
    "people.person.profession",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "sports.sports_team.team_mascot",
    "education.educational_institution.sports_teams",
    "olympics.olympic_participating_country.olympics_participated_in",
]


TOKENIZER_PROBE_16R4 = []


for relation in probe_relations:

    ids = tokenizer.encode(
        relation,
        add_special_tokens=False
    )

    tokens = tokenizer.convert_ids_to_tokens(
        ids
    )

    decoded = tokenizer.decode(
        ids,
        skip_special_tokens=True
    )

    rec = {
        "relation":
            relation,

        "ids":
            ids,

        "tokens":
            tokens,

        "decoded":
            decoded,

        "exact_roundtrip":
            decoded == relation,

        "compact_roundtrip":
            compact_relation_id_16r4(
                decoded
            )
            ==
            relation,
    }

    TOKENIZER_PROBE_16R4.append(
        rec
    )


    print("\nRELATION:")
    print(" ", relation)

    print("TOKENS:")
    print(" ", tokens)

    print("DECODED:")
    print(" ", repr(decoded))

    print(
        "exact round-trip:",
        decoded == relation
    )

    print(
        "compact round-trip:",
        compact_relation_id_16r4(
            decoded
        )
        ==
        relation
    )


# ======================================================================
# 8. CHECK WHETHER TEST CORRUPTION IS CHARACTER-DESTRUCTIVE
#
# If removing whitespace produces syntactically normal relation IDs,
# then characters themselves survived and only token-boundary spacing
# was injected.
# ======================================================================

relation_pattern = re.compile(
    r"^[A-Za-z0-9_#.\-]+$"
)


def compact_syntax_audit(
    rows,
    label
):

    total = 0
    valid_compact_syntax = 0
    invalid_examples = []


    for rec in rows:

        for token in plan_tokens_16r4(
            rec
        ):

            total += 1

            compact = (
                compact_relation_id_16r4(
                    token
                )
            )


            if relation_pattern.fullmatch(
                compact
            ):

                valid_compact_syntax += 1

            elif len(
                invalid_examples
            ) < 20:

                invalid_examples.append(
                    (
                        token,
                        compact
                    )
                )


    print(f"\n{label}")
    print(" tokens:", total)
    print(
        "valid relation-ID syntax after compaction:",
        valid_compact_syntax
    )

    print(
        "percentage:",
        (
            100
            * valid_compact_syntax
            / max(1, total)
        )
    )


    if invalid_examples:

        print(
            "\nInvalid compacted examples:"
        )

        for x in invalid_examples:
            print(x)


    return {
        "total":
            total,

        "valid":
            valid_compact_syntax,

        "invalid_examples":
            invalid_examples,
    }


print("\n" + "=" * 115)
print("F. CHARACTER-PRESERVATION / SYNTAX AUDIT")
print("=" * 115)


WEB_SYNTAX_16R4 = (
    compact_syntax_audit(
        webqsp_test_plan_rows,
        "WEBQSP TEST"
    )
)

CWQ_SYNTAX_16R4 = (
    compact_syntax_audit(
        cwq_test_plan_rows,
        "CWQ TEST"
    )
)


# ======================================================================
# 9. STATUS
# ======================================================================

CELL16_R4_COMPLETE = True


print(
    "\n"
    + "=" * 115
)

print(
    "CELL 16-R4 DIAGNOSTIC COMPLETE"
)

print(
    "=" * 115
)

print(
    "\nTEST JSONLs modified: NO"
)

print(
    "TEST plans regenerated: NO"
)

print(
    "AFP changed: NO"
)

print(
    "TEST tuning: NO"
)

print(
    "\nPASTE THE OUTPUT HERE BEFORE ANY REPAIR."
)

CELL 16-R4 — PLANNER DECODING / RELATION-ID FIDELITY DIAGNOSTIC

Freeze / repair gates: PASSED

A. CURRENT TOKENIZER ENVIRONMENT
Python: 3.12.13
transformers: 5.0.0
Tokenizer class: TokenizersBackend
Tokenizer module: transformers.tokenization_utils_tokenizers
is_fast: True
legacy: ATTRIBUTE_NOT_PRESENT
clean_up_tokenization_spaces: True
name_or_path: rmanluo/RoG

B. GRAPH RELATION-ID SCHEMA AUDIT

WEBQSP TEST
 graph edge relations: 7015502
 relations containing whitespace: 0

CWQ TEST
 graph edge relations: 15086427
 relations containing whitespace: 0

Graph relation IDs are whitespace-free: PASSED

C. TEST RAW-vs-COMPACT MATCH AUDIT

-------------------------------------------------------------------------------------------------------------------
WEBQSP TEST
-------------------------------------------------------------------------------------------------------------------
tokens                        : 7817
raw_exact                     : 0
compact_exact                 : 4160
reco

In [18]:
# ======================================================================
# CELL 16-R5
# ORIGINAL SLOW-TOKENIZER COMPATIBILITY / INPUT-ID FIDELITY AUDIT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - rewrite TEST JSONLs
#   - restart kernel
#
# This cell performs NO TEST evaluation and NO tuning.
# ======================================================================

from pathlib import Path
import hashlib
import re
import sys

import sentencepiece as spm


print("=" * 118)
print("CELL 16-R5 — ORIGINAL SLOW-TOKENIZER COMPATIBILITY AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "tokenizer" in globals()
assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("\nFreeze gate: PASSED")


# ======================================================================
# 2. VERIFY OFFICIAL RoG REQUIREMENT FILE
# ======================================================================

req_candidates = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "requirements.txt"
    ),
    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/requirements.txt"
    ),
]

REQ_PATH = next(
    (p for p in req_candidates if p.exists()),
    None
)

assert REQ_PATH is not None, (
    "Official local RoG requirements.txt not found."
)

requirements_text = REQ_PATH.read_text(
    encoding="utf-8"
)

assert "transformers==4.32.0" in requirements_text
assert "sentencepiece==0.1.99" in requirements_text

print("\nOfficial RoG dependency specification: PASSED")
print(" requirements:", REQ_PATH)
print(" expected transformers: 4.32.0")
print(" expected sentencepiece: 0.1.99")

print("\nCurrent environment:")
print(
    " transformers:",
    __import__("transformers").__version__
)
print(
    " sentencepiece:",
    spm.__version__
)
print(
    " tokenizer class:",
    tokenizer.__class__.__name__
)
print(
    " is_fast:",
    getattr(tokenizer, "is_fast", None)
)


# ======================================================================
# 3. FIND EXACT rmanluo/RoG tokenizer.model
# ======================================================================

tokenizer_model_candidates = []

# HF cache
hf_cache = Path.home() / ".cache/huggingface/hub"

if hf_cache.exists():

    tokenizer_model_candidates.extend(
        hf_cache.glob(
            "models--rmanluo--RoG/"
            "snapshots/*/tokenizer.model"
        )
    )

# Kaggle caches sometimes live elsewhere.
for root in [
    Path("/root/.cache/huggingface"),
    Path("/kaggle/working"),
]:

    if root.exists():

        tokenizer_model_candidates.extend(
            root.rglob("tokenizer.model")
        )


# Keep plausible RoG files, then verify vocab size later.
tokenizer_model_candidates = list(
    dict.fromkeys(
        tokenizer_model_candidates
    )
)


print(
    "\nFound tokenizer.model candidates:",
    len(tokenizer_model_candidates)
)


for p in tokenizer_model_candidates[:20]:
    print(" ", p)


assert tokenizer_model_candidates, (
    "Could not locate tokenizer.model downloaded for rmanluo/RoG."
)


# Prefer explicit rmanluo RoG cache path.
preferred = [
    p
    for p in tokenizer_model_candidates
    if (
        "models--rmanluo--RoG"
        in str(p)
    )
]


TOKENIZER_MODEL_PATH = (
    preferred[0]
    if preferred
    else tokenizer_model_candidates[0]
)


print(
    "\nSelected SentencePiece model:",
    TOKENIZER_MODEL_PATH
)


# ======================================================================
# 4. LOAD RAW SENTENCEPIECE MODEL
# ======================================================================

sp = spm.SentencePieceProcessor()

loaded = sp.load(
    str(TOKENIZER_MODEL_PATH)
)

assert loaded


print("\nSentencePiece model loaded: PASSED")
print(" vocab size:", sp.get_piece_size())
print(" bos id:", sp.bos_id())
print(" eos id:", sp.eos_id())
print(" unk id:", sp.unk_id())


print("\nTransformers tokenizer:")
print(
    " vocab size:",
    getattr(tokenizer, "vocab_size", None)
)
print(
    " bos_token_id:",
    getattr(tokenizer, "bos_token_id", None)
)
print(
    " eos_token_id:",
    getattr(tokenizer, "eos_token_id", None)
)


# ======================================================================
# 5. ID-ENCODING COMPARISON
#
# The critical question:
# Does Transformers 5 produce the same SentencePiece IDs?
# ======================================================================

def current_ids_no_special(text):

    return list(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )


def slow_sp_ids_no_special(text):

    return list(
        sp.encode(
            str(text),
            out_type=int
        )
    )


def compare_id_encoding(text):

    current = current_ids_no_special(
        text
    )

    slow = slow_sp_ids_no_special(
        text
    )

    return {
        "same":
            current == slow,

        "current":
            current,

        "slow":
            slow,
    }


probe_texts = [
    "people.person.place_of_birth",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "what does jamaican people speak",
    "what did james k polk do before he was president",
    (
        "Please generate a valid relation path that can be "
        "helpful for answering the following question: "
        "what does jamaican people speak"
    ),
]


print("\n" + "=" * 118)
print("A. DIRECT TOKEN-ID PROBES")
print("=" * 118)


for text in probe_texts:

    cmp = compare_id_encoding(
        text
    )

    print("\nTEXT:")
    print(repr(text))

    print(
        "same token IDs:",
        cmp["same"]
    )

    if not cmp["same"]:

        print(
            " current:",
            cmp["current"][:80]
        )
        print(
            " slow SP:",
            cmp["slow"][:80]
        )


# ======================================================================
# 6. LARGE-SAMPLE QUESTION ENCODING AUDIT
# ======================================================================

def audit_question_ids(
    rows,
    label,
    limit=500
):

    checked = 0
    identical = 0
    mismatched = 0
    examples = []


    for rec in rows[:limit]:

        text = str(
            rec["question"]
        )

        cmp = compare_id_encoding(
            text
        )

        checked += 1

        if cmp["same"]:

            identical += 1

        else:

            mismatched += 1

            if len(examples) < 10:

                examples.append(
                    {
                        "id":
                            rec["id"],

                        "question":
                            text,

                        "current":
                            cmp["current"],

                        "slow":
                            cmp["slow"],
                    }
                )


    print(f"\n{label}")
    print(" checked:", checked)
    print(" identical IDs:", identical)
    print(" mismatched IDs:", mismatched)

    print(
        " identity rate:",
        f"{100 * identical / max(1, checked):.6f}%"
    )

    if examples:

        print("\nMismatch examples:")

        for x in examples:
            print(" ", x["id"])
            print("   ", x["question"][:150])
            print("   current:", x["current"][:40])
            print("   slow:   ", x["slow"][:40])


    return {
        "checked":
            checked,

        "identical":
            identical,

        "mismatched":
            mismatched,
    }


print("\n" + "=" * 118)
print("B. QUESTION INPUT-ID FIDELITY")
print("=" * 118)


WEB_Q_ID_AUDIT = audit_question_ids(
    webqsp_test_plan_rows,
    "WEBQSP TEST questions",
    limit=500
)

CWQ_Q_ID_AUDIT = audit_question_ids(
    cwq_test_plan_rows,
    "CWQ TEST questions",
    limit=500
)


# ======================================================================
# 7. FULL PLANNER-PROMPT INPUT-ID AUDIT
#
# Use the exact recovered prompter if it survived M3.
# ======================================================================

print("\n" + "=" * 118)
print("C. FULL PLANNER-PROMPT INPUT-ID FIDELITY")
print("=" * 118)


PROMPT_AUDIT_AVAILABLE = (
    "prompter" in globals()
    and
    "INSTRUCTION" in globals()
)


print(
    "Exact prompter available:",
    PROMPT_AUDIT_AVAILABLE
)


def planner_prompt_for(question):

    return prompter.format(
        instruction=INSTRUCTION,
        message=question
    )


def audit_full_prompts(
    rows,
    label,
    limit=200
):

    checked = 0
    identical = 0
    mismatched = 0

    examples = []


    for rec in rows[:limit]:

        prompt = planner_prompt_for(
            rec["question"]
        )

        cmp = compare_id_encoding(
            prompt
        )

        checked += 1

        if cmp["same"]:

            identical += 1

        else:

            mismatched += 1

            if len(examples) < 5:

                examples.append(
                    (
                        rec["id"],
                        cmp["current"][:60],
                        cmp["slow"][:60],
                    )
                )


    print(f"\n{label}")
    print(" prompts checked:", checked)
    print(" identical IDs:", identical)
    print(" mismatched IDs:", mismatched)

    print(
        " identity rate:",
        f"{100 * identical / max(1, checked):.6f}%"
    )


    if examples:

        print("\nMismatch examples:")

        for x in examples:
            print(" ", x)


    return {
        "checked":
            checked,

        "identical":
            identical,

        "mismatched":
            mismatched,
    }


if PROMPT_AUDIT_AVAILABLE:

    WEB_PROMPT_ID_AUDIT = audit_full_prompts(
        webqsp_test_plan_rows,
        "WEBQSP full planner prompts",
        limit=200
    )

    CWQ_PROMPT_ID_AUDIT = audit_full_prompts(
        cwq_test_plan_rows,
        "CWQ full planner prompts",
        limit=200
    )

else:

    WEB_PROMPT_ID_AUDIT = None
    CWQ_PROMPT_ID_AUDIT = None

    print(
        "\nFull-prompt audit skipped because "
        "prompter/INSTRUCTION is not currently in RAM."
    )


# ======================================================================
# 8. DECODING COMPARISON
# ======================================================================

def compact_ws(x):

    return re.sub(
        r"\s+",
        "",
        str(x)
    )


print("\n" + "=" * 118)
print("D. FAST-v5 vs RAW SENTENCEPIECE DECODING")
print("=" * 118)


relations = [
    "people.person.place_of_birth",
    "people.person.profession",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "sports.sports_team.team_mascot",
    "education.educational_institution.sports_teams",
]


for relation in relations:

    ids = slow_sp_ids_no_special(
        relation
    )

    sp_decoded = sp.decode(
        ids
    )

    fast_decoded = tokenizer.decode(
        ids,
        skip_special_tokens=True
    )


    print("\nRELATION:")
    print(" ", relation)

    print("SentencePiece decode:")
    print(" ", repr(sp_decoded))

    print("Transformers-5 decode:")
    print(" ", repr(fast_decoded))

    print(
        "SP exact:",
        sp_decoded == relation
    )

    print(
        "v5 compact exact:",
        compact_ws(
            fast_decoded
        )
        ==
        relation
    )


# ======================================================================
# 9. DATASET-WIDE VALIDATION RELATION ROUND-TRIP
#
# Known-good frozen validation relation strings are used as the
# representation-fidelity reference, not TEST gold labels.
# ======================================================================

def relation_roundtrip_audit(
    rows,
    label
):

    total = 0
    whitespace_free = 0

    sp_exact = 0
    v5_exact = 0
    v5_compact_exact = 0

    failures = []


    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue


            for relation in plan:

                relation = str(
                    relation
                )

                total += 1


                # Primary reference is already-canonical
                # validation relation strings.
                if re.search(
                    r"\s",
                    relation
                ):
                    continue


                whitespace_free += 1

                ids = slow_sp_ids_no_special(
                    relation
                )

                sp_decoded = sp.decode(
                    ids
                )

                v5_decoded = tokenizer.decode(
                    ids,
                    skip_special_tokens=True
                )


                if sp_decoded == relation:
                    sp_exact += 1

                if v5_decoded == relation:
                    v5_exact += 1

                if (
                    compact_ws(
                        v5_decoded
                    )
                    ==
                    relation
                ):
                    v5_compact_exact += 1

                elif len(failures) < 15:

                    failures.append(
                        (
                            relation,
                            v5_decoded,
                            compact_ws(
                                v5_decoded
                            ),
                        )
                    )


    print(f"\n{label}")
    print(" all tokens:", total)
    print(
        " canonical whitespace-free tokens:",
        whitespace_free
    )

    print(
        " raw SentencePiece exact:",
        sp_exact
    )

    print(
        " Transformers-5 exact:",
        v5_exact
    )

    print(
        " Transformers-5 compact exact:",
        v5_compact_exact
    )

    print(
        " compact recovery rate:",
        f"{100 * v5_compact_exact / max(1, whitespace_free):.6f}%"
    )


    if failures:

        print("\nCompaction failures:")

        for item in failures:
            print(item)


    return {
        "total":
            total,

        "canonical":
            whitespace_free,

        "sp_exact":
            sp_exact,

        "v5_exact":
            v5_exact,

        "v5_compact_exact":
            v5_compact_exact,
    }


print("\n" + "=" * 118)
print("E. FROZEN-VALIDATION ROUND-TRIP FIDELITY")
print("=" * 118)


WEB_VAL_ROUNDTRIP = relation_roundtrip_audit(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

CWQ_VAL_ROUNDTRIP = relation_roundtrip_audit(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)


# ======================================================================
# 10. VALIDATION GRAPH-OVERLAP REFERENCE
#
# Compare TEST-compacted overlap later against the natural raw overlap
# level of frozen validation plans.
# ======================================================================

def graph_relations_r5(rec):

    return {
        str(t[1]).strip()
        for t in rec["graph"]
    }


def overlap_reference(
    rows,
    label
):

    tokens = 0
    exact = 0

    first_tokens = 0
    first_exact = 0

    q_with_plan = 0
    q_any = 0


    for rec in rows:

        graph_rels = graph_relations_r5(
            rec
        )

        any_match = False
        had_plan = False


        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue

            if len(plan) == 0:
                continue

            had_plan = True

            first_tokens += 1

            if str(plan[0]) in graph_rels:
                first_exact += 1


            for relation in plan:

                tokens += 1

                if str(relation) in graph_rels:

                    exact += 1
                    any_match = True


        if had_plan:
            q_with_plan += 1

        if any_match:
            q_any += 1


    print(f"\n{label}")
    print(" relation tokens:", tokens)
    print(" exact graph relations:", exact)

    print(
        " exact token %:",
        f"{100 * exact / max(1, tokens):.4f}%"
    )

    print(
        " first-hop exact %:",
        f"{100 * first_exact / max(1, first_tokens):.4f}%"
    )

    print(
        " questions with any exact relation:",
        q_any,
        "/",
        q_with_plan
    )


    return {
        "tokens":
            tokens,

        "exact":
            exact,

        "first_tokens":
            first_tokens,

        "first_exact":
            first_exact,

        "q_with_plan":
            q_with_plan,

        "q_any":
            q_any,
    }


print("\n" + "=" * 118)
print("F. FROZEN VALIDATION GRAPH-OVERLAP REFERENCE")
print("=" * 118)


WEB_VAL_OVERLAP_R5 = overlap_reference(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

CWQ_VAL_OVERLAP_R5 = overlap_reference(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)


# ======================================================================
# 11. FINAL STATUS — DO NOT AUTO-REPAIR
# ======================================================================

CELL16_R5_COMPLETE = True


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R5 AUDIT COMPLETE"
)

print(
    "=" * 118
)

print("\nNo TEST JSONL modified.")
print("No TEST plan regenerated.")
print("No AFP parameter changed.")
print("No TEST tuning performed.")

print(
    "\nPASTE THIS OUTPUT HERE."
)

print(
    "Do NOT run Cell 17 yet."
)

CELL 16-R5 — ORIGINAL SLOW-TOKENIZER COMPATIBILITY AUDIT

Freeze gate: PASSED

Official RoG dependency specification: PASSED
 requirements: /kaggle/working/reasoning-on-graphs/requirements.txt
 expected transformers: 4.32.0
 expected sentencepiece: 0.1.99

Current environment:
 transformers: 5.0.0
 sentencepiece: 0.2.1
 tokenizer class: TokenizersBackend
 is_fast: True

Found tokenizer.model candidates: 1
  /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model

Selected SentencePiece model: /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model

SentencePiece model loaded: PASSED
 vocab size: 32000
 bos id: 1
 eos id: 2
 unk id: 0

Transformers tokenizer:
 vocab size: 32000
 bos_token_id: 1
 eos_token_id: 2

A. DIRECT TOKEN-ID PROBES

TEXT:
'people.person.place_of_birth'
same token IDs: False
 current: [412, 459, 280, 29889, 546, 1100, 29889, 572, 815, 29918, 974

In [19]:
# ======================================================================
# CELL 16-R6
# ORIGINAL ENVIRONMENT + TRUE SLOW LLAMA TOKENIZER RECOVERY AUDIT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - modify TEST JSONLs
#   - restart kernel
#
# Diagnostic only.
# ======================================================================

import ast
import re
from pathlib import Path

import sentencepiece as spm
import transformers


print("=" * 118)
print("CELL 16-R6 — ORIGINAL ENVIRONMENT / SLOW TOKENIZER AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()
assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("\nFreeze gate: PASSED")


# ======================================================================
# 2. SEARCH ORIGINAL NOTEBOOK FOR DEPENDENCY-INSTALL CELLS
# ======================================================================

print("\n" + "=" * 118)
print("A. ORIGINAL NOTEBOOK DEPENDENCY SETUP")
print("=" * 118)


dependency_terms = [
    "pip install",
    "pip3 install",
    "requirements.txt",
    "transformers==",
    "sentencepiece",
    "graph-walker",
]


dependency_cells = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    lowered = source.lower()

    if any(
        term.lower() in lowered
        for term in dependency_terms
    ):

        dependency_cells.append(
            (
                cell_idx,
                source
            )
        )


print(
    "Relevant dependency/setup cells:",
    [x[0] for x in dependency_cells]
)


for cell_idx, source in dependency_cells:

    print(
        "\n"
        + "-" * 118
    )

    print(
        f"NOTEBOOK CELL {cell_idx}"
    )

    print(
        "-" * 118
    )

    for line in source.splitlines():

        if any(
            term.lower() in line.lower()
            for term in dependency_terms
        ):

            print(line)


# ======================================================================
# 3. OFFICIAL LOCAL REQUIREMENTS
# ======================================================================

req_candidates = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "requirements.txt"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/requirements.txt"
    ),
]


REQ_PATH = next(
    (
        p
        for p in req_candidates
        if p.exists()
    ),
    None
)


assert REQ_PATH is not None


requirements_text = (
    REQ_PATH.read_text(
        encoding="utf-8"
    )
)


print("\n" + "=" * 118)
print("B. OFFICIAL RoG REQUIREMENTS")
print("=" * 118)

for line in requirements_text.splitlines():

    if (
        "transformers" in line.lower()
        or
        "sentencepiece" in line.lower()
        or
        "tokenizers" in line.lower()
    ):

        print(line)


assert "transformers==4.32.0" in requirements_text
assert "sentencepiece==0.1.99" in requirements_text


print("\nCurrent environment:")
print(" transformers:", transformers.__version__)
print(" sentencepiece:", spm.__version__)


# ======================================================================
# 4. FIND RAW RoG SENTENCEPIECE MODEL
# ======================================================================

cache_root = Path(
    "/root/.cache/huggingface/hub"
)

sp_candidates = list(
    cache_root.glob(
        "models--rmanluo--RoG/"
        "snapshots/*/tokenizer.model"
    )
)


assert sp_candidates, (
    "RoG tokenizer.model not found."
)


SP_MODEL_PATH = sp_candidates[0]


sp = spm.SentencePieceProcessor()

assert sp.load(
    str(SP_MODEL_PATH)
)


print("\nSentencePiece model:")
print(" ", SP_MODEL_PATH)
print(" vocab:", sp.get_piece_size())


# ======================================================================
# 5. TRY TRUE SLOW LLAMA TOKENIZER DIRECTLY
#
# AutoTokenizer is the component that returned TokenizersBackend.
# Here we explicitly request the SentencePiece LlamaTokenizer.
# ======================================================================

print("\n" + "=" * 118)
print("C. DIRECT SLOW LLAMA TOKENIZER")
print("=" * 118)


SLOW_TOKENIZER_AVAILABLE = False
slow_tokenizer = None
slow_tokenizer_error = None


try:

    from transformers.models.llama.tokenization_llama import (
        LlamaTokenizer
    )

    try:

        slow_tokenizer = (
            LlamaTokenizer.from_pretrained(
                "rmanluo/RoG",
                legacy=True,
                clean_up_tokenization_spaces=True,
            )
        )

    except TypeError:

        # Some versions do not expose legacy as a constructor kwarg.
        slow_tokenizer = (
            LlamaTokenizer.from_pretrained(
                "rmanluo/RoG",
                clean_up_tokenization_spaces=True,
            )
        )


    SLOW_TOKENIZER_AVAILABLE = True


except Exception as e:

    slow_tokenizer_error = repr(e)


print(
    "Slow tokenizer available:",
    SLOW_TOKENIZER_AVAILABLE
)


if SLOW_TOKENIZER_AVAILABLE:

    print(
        " class:",
        slow_tokenizer.__class__.__name__
    )

    print(
        " module:",
        slow_tokenizer.__class__.__module__
    )

    print(
        " is_fast:",
        getattr(
            slow_tokenizer,
            "is_fast",
            False
        )
    )

    print(
        " vocab size:",
        slow_tokenizer.vocab_size
    )

else:

    print(
        " error:",
        slow_tokenizer_error
    )


# ======================================================================
# 6. RAW SP vs DIRECT SLOW-TOKENIZER ID FIDELITY
# ======================================================================

def raw_sp_ids(text):

    return list(
        sp.encode(
            str(text),
            out_type=int
        )
    )


def slow_ids_no_special(text):

    return list(
        slow_tokenizer.encode(
            str(text),
            add_special_tokens=False
        )
    )


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("D. DIRECT TOKEN-ID FIDELITY")
    print("=" * 118)


    probes = [
        "people.person.place_of_birth",
        "government.government_position_held.office_holder",
        "location.location.containedby",
        "what does jamaican people speak",
        "what did james k polk do before he was president",
    ]


    for text in probes:

        a = raw_sp_ids(
            text
        )

        b = slow_ids_no_special(
            text
        )

        print(
            "\n",
            repr(text)
        )

        print(
            " IDs identical:",
            a == b
        )

        if a != b:

            print(
                " raw SP:",
                a
            )

            print(
                " slow:  ",
                b
            )


# ======================================================================
# 7. DATASET-WIDE QUESTION ID FIDELITY
# ======================================================================

def question_id_fidelity(
    rows,
    label,
    limit=500
):

    checked = 0
    exact = 0

    mismatches = []


    for rec in rows[:limit]:

        question = str(
            rec["question"]
        )

        a = raw_sp_ids(
            question
        )

        b = slow_ids_no_special(
            question
        )

        checked += 1


        if a == b:

            exact += 1

        elif len(mismatches) < 5:

            mismatches.append(
                rec["id"]
            )


    print(f"\n{label}")
    print(" checked:", checked)
    print(" exact IDs:", exact)

    print(
        " identity rate:",
        f"{100 * exact / max(1, checked):.6f}%"
    )

    print(
        " mismatch examples:",
        mismatches
    )


    return {
        "checked": checked,
        "exact": exact,
    }


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("E. QUESTION INPUT-ID FIDELITY")
    print("=" * 118)


    WEB_SLOW_ID_AUDIT = (
        question_id_fidelity(
            webqsp_test_plan_rows,
            "WEBQSP TEST",
            500
        )
    )


    CWQ_SLOW_ID_AUDIT = (
        question_id_fidelity(
            cwq_test_plan_rows,
            "CWQ TEST",
            500
        )
    )


# ======================================================================
# 8. VALIDATION RELATION ROUND-TRIP
# ======================================================================

def validation_roundtrip(
    rows,
    label
):

    checked = 0
    exact = 0

    failures = []


    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):

                continue


            for relation in plan:

                relation = str(
                    relation
                )

                # Keep this audit on already-canonical relation IDs.
                if re.search(
                    r"\s",
                    relation
                ):

                    continue


                ids = slow_ids_no_special(
                    relation
                )

                decoded = (
                    slow_tokenizer.decode(
                        ids,
                        skip_special_tokens=True
                    )
                )


                checked += 1


                if decoded == relation:

                    exact += 1

                elif len(failures) < 10:

                    failures.append(
                        (
                            relation,
                            decoded
                        )
                    )


    print(f"\n{label}")
    print(
        " canonical relations checked:",
        checked
    )

    print(
        " exact round-trips:",
        exact
    )

    print(
        " exact rate:",
        f"{100 * exact / max(1, checked):.6f}%"
    )


    if failures:

        print(
            " failures:",
            failures
        )


    return {
        "checked": checked,
        "exact": exact,
    }


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("F. FROZEN VALIDATION RELATION ROUND-TRIP")
    print("=" * 118)


    WEB_SLOW_ROUNDTRIP = (
        validation_roundtrip(
            webqsp_val_plan_rows,
            "WEBQSP VALIDATION"
        )
    )


    CWQ_SLOW_ROUNDTRIP = (
        validation_roundtrip(
            cwq_val_plan_rows,
            "CWQ VALIDATION"
        )
    )


# ======================================================================
# 9. SPECIAL-TOKEN BEHAVIOR USED BY generate_seq()
#
# generate_seq uses tokenizer.encode(..., return_tensors="pt")
# without add_special_tokens=False.
# ======================================================================

if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("G. generate_seq ENCODE BEHAVIOR")
    print("=" * 118)


    probe = (
        "Please generate a valid relation path that can be "
        "helpful for answering the following question: "
        "what does jamaican people speak"
    )


    no_special = slow_ids_no_special(
        probe
    )

    with_special = list(
        slow_tokenizer.encode(
            probe
        )
    )


    print(
        "slow add_bos_token:",
        getattr(
            slow_tokenizer,
            "add_bos_token",
            None
        )
    )

    print(
        "slow add_eos_token:",
        getattr(
            slow_tokenizer,
            "add_eos_token",
            None
        )
    )

    print(
        "BOS id:",
        slow_tokenizer.bos_token_id
    )

    print(
        "EOS id:",
        slow_tokenizer.eos_token_id
    )

    print(
        "no-special length:",
        len(no_special)
    )

    print(
        "default encode length:",
        len(with_special)
    )

    print(
        "default encode prefix:",
        with_special[:10]
    )

    print(
        "expected raw-SP prefix:",
        (
            [slow_tokenizer.bos_token_id]
            + raw_sp_ids(probe)
        )[:10]
    )

    print(
        "default == BOS + raw SP:",
        with_special
        ==
        (
            [slow_tokenizer.bos_token_id]
            + raw_sp_ids(probe)
        )
    )


# ======================================================================
# 10. FINAL DECISION GATE
# ======================================================================

if SLOW_TOKENIZER_AVAILABLE:

    web_ids_ok = (
        WEB_SLOW_ID_AUDIT["exact"]
        ==
        WEB_SLOW_ID_AUDIT["checked"]
    )

    cwq_ids_ok = (
        CWQ_SLOW_ID_AUDIT["exact"]
        ==
        CWQ_SLOW_ID_AUDIT["checked"]
    )

    web_decode_ok = (
        WEB_SLOW_ROUNDTRIP["exact"]
        ==
        WEB_SLOW_ROUNDTRIP["checked"]
    )

    cwq_decode_ok = (
        CWQ_SLOW_ROUNDTRIP["exact"]
        ==
        CWQ_SLOW_ROUNDTRIP["checked"]
    )

    SLOW_TOKENIZER_FIDELITY_PASSED = (
        web_ids_ok
        and
        cwq_ids_ok
        and
        web_decode_ok
        and
        cwq_decode_ok
    )

else:

    SLOW_TOKENIZER_FIDELITY_PASSED = False


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R6 AUDIT COMPLETE"
)

print(
    "=" * 118
)

print(
    "\nSlow tokenizer fidelity passed:",
    SLOW_TOKENIZER_FIDELITY_PASSED
)

print("\nTEST JSONLs modified: NO")
print("TEST plans regenerated: NO")
print("AFP changed: NO")
print("TEST tuning: NO")

print(
    "\nPASTE THIS OUTPUT HERE."
)

print(
    "DO NOT RUN CELL 17."
)

CELL 16-R6 — ORIGINAL ENVIRONMENT / SLOW TOKENIZER AUDIT

Freeze gate: PASSED

A. ORIGINAL NOTEBOOK DEPENDENCY SETUP
Relevant dependency/setup cells: [1]

----------------------------------------------------------------------------------------------------------------------
NOTEBOOK CELL 1
----------------------------------------------------------------------------------------------------------------------
!pip install -q \
    --only-binary=transformers,tokenizers,peft,sentencepiece,accelerate,datasets,numpy,torch \
    "transformers==4.44.2" "tokenizers==0.19.1" "peft==0.12.0" \
    "sentencepiece==0.2.0" "accelerate==0.34.2" "datasets==2.20.0" \
    "numpy>=2.0,<2.1" "graph-walker==1.0.6" "{_torch_pin}"

B. OFFICIAL RoG REQUIREMENTS
transformers==4.32.0
sentencepiece==0.1.99

Current environment:
 transformers: 5.0.0
 sentencepiece: 0.2.1

SentencePiece model:
  /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model
 vocab

In [20]:
# ======================================================================
# CELL 16-R7
# EXACT ORIGINAL NOTEBOOK TOKENIZER ENVIRONMENT AUDIT
# transformers 4.44.2 / tokenizers 0.19.1 / sentencepiece 0.2.0
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - restart kernel
#   - modify TEST planning files
#
# This creates an isolated Python package directory and runs the
# tokenizer audit in a subprocess, leaving the current kernel untouched.
# ======================================================================

import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path


print("=" * 118)
print("CELL 16-R7 — EXACT ORIGINAL TOKENIZER ENVIRONMENT AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()


print("\nFreeze gate: PASSED")


# ======================================================================
# 2. EXACT ORIGINAL NOTEBOOK SOFTWARE SPEC
# ======================================================================

TARGET_TRANSFORMERS = "4.44.2"
TARGET_TOKENIZERS = "0.19.1"
TARGET_SENTENCEPIECE = "0.2.0"


print("\nExact original notebook environment:")
print(" transformers:  ", TARGET_TRANSFORMERS)
print(" tokenizers:    ", TARGET_TOKENIZERS)
print(" sentencepiece: ", TARGET_SENTENCEPIECE)


# ======================================================================
# 3. PREPARE ISOLATED ENVIRONMENT
# ======================================================================

ENV_DIR = Path(
    "/kaggle/working/rog_exact_tokenizer_env_4442"
)

ENV_DIR.mkdir(
    parents=True,
    exist_ok=True
)


marker = (
    ENV_DIR
    / "_INSTALL_COMPLETE.json"
)


need_install = True


if marker.exists():

    try:

        marker_data = json.loads(
            marker.read_text(
                encoding="utf-8"
            )
        )

        need_install = (
            marker_data.get("transformers")
            != TARGET_TRANSFORMERS
            or
            marker_data.get("tokenizers")
            != TARGET_TOKENIZERS
            or
            marker_data.get("sentencepiece")
            != TARGET_SENTENCEPIECE
        )

    except Exception:

        need_install = True


if need_install:

    print(
        "\nInstalling exact tokenizer environment "
        "into isolated directory..."
    )

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--target",
        str(ENV_DIR),

        "transformers==4.44.2",
        "tokenizers==0.19.1",
        "sentencepiece==0.2.0",

        # Notebook Cell 1 also pinned this numerical range.
        "numpy>=2.0,<2.1",
    ]


    subprocess.run(
        cmd,
        check=True
    )


    marker.write_text(
        json.dumps(
            {
                "transformers":
                    TARGET_TRANSFORMERS,

                "tokenizers":
                    TARGET_TOKENIZERS,

                "sentencepiece":
                    TARGET_SENTENCEPIECE,
            },
            indent=2
        ),
        encoding="utf-8"
    )


    print("Isolated installation: COMPLETE")

else:

    print(
        "\nExisting exact isolated environment reused."
    )


# ======================================================================
# 4. SERIALIZE NON-GOLD AUDIT INPUTS
#
# No answer labels are required.
# ======================================================================

AUDIT_DIR = Path(
    "/kaggle/working/step2_rq1_test/"
    "tokenizer_fidelity_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


audit_input_path = (
    AUDIT_DIR
    / "cell16_r7_inputs.json"
)


def canonical_relation_tokens(rows):

    out = []

    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue

            for relation in plan:

                relation = str(
                    relation
                )

                # Only already-canonical validation tokens.
                if not any(
                    c.isspace()
                    for c in relation
                ):
                    out.append(
                        relation
                    )

    return out


audit_payload = {
    "webqsp_questions": [
        {
            "id": str(x["id"]),
            "question": str(x["question"]),
        }
        for x in webqsp_test_plan_rows[:1000]
    ],

    "cwq_questions": [
        {
            "id": str(x["id"]),
            "question": str(x["question"]),
        }
        for x in cwq_test_plan_rows[:1000]
    ],

    "webqsp_validation_relations":
        canonical_relation_tokens(
            webqsp_val_plan_rows
        ),

    "cwq_validation_relations":
        canonical_relation_tokens(
            cwq_val_plan_rows
        ),
}


audit_input_path.write_text(
    json.dumps(
        audit_payload,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nAudit payload written:",
    audit_input_path
)


# ======================================================================
# 5. BUILD ISOLATED SUBPROCESS SCRIPT
# ======================================================================

audit_output_path = (
    AUDIT_DIR
    / "cell16_r7_results.json"
)


script_path = (
    AUDIT_DIR
    / "cell16_r7_subprocess.py"
)


script = r'''
import json
import sys
from pathlib import Path

import transformers
import tokenizers
import sentencepiece

from transformers import AutoTokenizer


INPUT_PATH = Path(sys.argv[1])
OUTPUT_PATH = Path(sys.argv[2])


payload = json.loads(
    INPUT_PATH.read_text(
        encoding="utf-8"
    )
)


tokenizer = AutoTokenizer.from_pretrained(
    "rmanluo/RoG",
    use_fast=False,
    clean_up_tokenization_spaces=True,
    local_files_only=True,
)


def audit_questions(rows):

    checked = 0
    unk_questions = 0
    unk_token_total = 0

    lengths = []

    examples_with_unk = []


    for rec in rows:

        ids = tokenizer.encode(
            rec["question"],
            add_special_tokens=False
        )

        checked += 1
        lengths.append(len(ids))

        unk_count = sum(
            int(x == tokenizer.unk_token_id)
            for x in ids
        )

        unk_token_total += unk_count


        if unk_count > 0:

            unk_questions += 1

            if len(examples_with_unk) < 10:

                examples_with_unk.append(
                    {
                        "id":
                            rec["id"],

                        "question":
                            rec["question"],

                        "unk_count":
                            unk_count,

                        "ids":
                            ids,
                    }
                )


    return {
        "checked":
            checked,

        "questions_with_unk":
            unk_questions,

        "unk_token_total":
            unk_token_total,

        "mean_length":
            (
                sum(lengths) / len(lengths)
                if lengths
                else 0
            ),

        "examples_with_unk":
            examples_with_unk,
    }


def audit_relations(relations):

    checked = 0
    exact = 0
    failures = []


    for relation in relations:

        ids = tokenizer.encode(
            relation,
            add_special_tokens=False
        )

        decoded = tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

        checked += 1


        if decoded == relation:

            exact += 1

        elif len(failures) < 20:

            failures.append(
                {
                    "relation":
                        relation,

                    "decoded":
                        decoded,

                    "ids":
                        ids,
                }
            )


    return {
        "checked":
            checked,

        "exact":
            exact,

        "exact_rate":
            (
                exact / checked
                if checked
                else 0
            ),

        "failures":
            failures,
    }


probe = (
    "people.person.place_of_birth"
)

probe_ids = tokenizer.encode(
    probe,
    add_special_tokens=False
)

probe_decoded = tokenizer.decode(
    probe_ids,
    skip_special_tokens=True
)


results = {
    "software": {
        "transformers":
            transformers.__version__,

        "tokenizers":
            tokenizers.__version__,

        "sentencepiece":
            sentencepiece.__version__,

        "tokenizer_class":
            tokenizer.__class__.__name__,

        "tokenizer_module":
            tokenizer.__class__.__module__,

        "is_fast":
            getattr(
                tokenizer,
                "is_fast",
                None
            ),

        "name_or_path":
            getattr(
                tokenizer,
                "name_or_path",
                None
            ),
    },

    "probe": {
        "input":
            probe,

        "ids":
            probe_ids,

        "decoded":
            probe_decoded,

        "exact":
            probe_decoded == probe,
    },

    "webqsp_questions":
        audit_questions(
            payload[
                "webqsp_questions"
            ]
        ),

    "cwq_questions":
        audit_questions(
            payload[
                "cwq_questions"
            ]
        ),

    "webqsp_validation_relations":
        audit_relations(
            payload[
                "webqsp_validation_relations"
            ]
        ),

    "cwq_validation_relations":
        audit_relations(
            payload[
                "cwq_validation_relations"
            ]
        ),

    "generate_seq_special_tokens": {
        "add_bos_token":
            getattr(
                tokenizer,
                "add_bos_token",
                None
            ),

        "add_eos_token":
            getattr(
                tokenizer,
                "add_eos_token",
                None
            ),

        "bos_token_id":
            tokenizer.bos_token_id,

        "eos_token_id":
            tokenizer.eos_token_id,
    },
}


OUTPUT_PATH.write_text(
    json.dumps(
        results,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)
'''


script_path.write_text(
    script,
    encoding="utf-8"
)


# ======================================================================
# 6. RUN EXACT ENVIRONMENT SUBPROCESS
# ======================================================================

env = os.environ.copy()


existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)


env[
    "PYTHONPATH"
] = (
    str(ENV_DIR)
    +
    (
        os.pathsep + existing_pythonpath
        if existing_pythonpath
        else ""
    )
)


print(
    "\nRunning isolated exact-environment audit..."
)


proc = subprocess.run(
    [
        sys.executable,
        str(script_path),
        str(audit_input_path),
        str(audit_output_path),
    ],
    env=env,
    text=True,
    capture_output=True,
)


print(
    "Subprocess return code:",
    proc.returncode
)


if proc.stdout.strip():

    print(
        "\nSUBPROCESS STDOUT:"
    )

    print(
        proc.stdout
    )


if proc.stderr.strip():

    print(
        "\nSUBPROCESS STDERR:"
    )

    print(
        proc.stderr[-5000:]
    )


assert proc.returncode == 0, (
    "Exact-environment subprocess failed. "
    "Do NOT change the main environment."
)


# ======================================================================
# 7. LOAD RESULTS
# ======================================================================

results = json.loads(
    audit_output_path.read_text(
        encoding="utf-8"
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "A. EXACT ENVIRONMENT"
)

print(
    "=" * 118
)


for k, v in results[
    "software"
].items():

    print(
        f"{k:<25}:",
        v
    )


# Exact software identity.
assert (
    results["software"]["transformers"]
    ==
    "4.44.2"
)

assert (
    results["software"]["tokenizers"]
    ==
    "0.19.1"
)

assert (
    results["software"]["sentencepiece"]
    ==
    "0.2.0"
)


# CRITICAL:
# use_fast=False must produce a genuine slow tokenizer.
assert (
    results[
        "software"
    ][
        "is_fast"
    ]
    is False
), (
    "Even exact 4.44.2 did not return a slow tokenizer."
)


print(
    "\nExact software + slow-tokenizer identity: PASSED"
)


# ======================================================================
# 8. PROBE
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "B. RELATION ROUND-TRIP PROBE"
)

print(
    "=" * 118
)


print(
    "input:",
    repr(
        results[
            "probe"
        ][
            "input"
        ]
    )
)

print(
    "decoded:",
    repr(
        results[
            "probe"
        ][
            "decoded"
        ]
    )
)

print(
    "exact:",
    results[
        "probe"
    ][
        "exact"
    ]
)


assert results[
    "probe"
][
    "exact"
] is True


# ======================================================================
# 9. VALIDATION RELATION FIDELITY
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "C. FROZEN VALIDATION RELATION FIDELITY"
)

print(
    "=" * 118
)


for name in [
    "webqsp_validation_relations",
    "cwq_validation_relations",
]:

    x = results[
        name
    ]

    print(
        f"\n{name}:"
    )

    print(
        " checked:",
        x["checked"]
    )

    print(
        " exact:",
        x["exact"]
    )

    print(
        " exact rate:",
        f"{100 * x['exact_rate']:.6f}%"
    )

    if x[
        "failures"
    ]:

        print(
            " failures:",
            x["failures"][:5]
        )


# These are the canonical validation tokens.
assert (
    results[
        "webqsp_validation_relations"
    ][
        "exact"
    ]
    ==
    results[
        "webqsp_validation_relations"
    ][
        "checked"
    ]
)

assert (
    results[
        "cwq_validation_relations"
    ][
        "exact"
    ]
    ==
    results[
        "cwq_validation_relations"
    ][
        "checked"
    ]
)


print(
    "\nFrozen validation canonical-token fidelity: PASSED"
)


# ======================================================================
# 10. QUESTION TOKENIZATION HEALTH
#
# We do NOT compare against raw SentencePiece here.
# The exact 4.44.2 AutoTokenizer is the reference implementation.
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "D. QUESTION TOKENIZATION HEALTH"
)

print(
    "=" * 118
)


for name in [
    "webqsp_questions",
    "cwq_questions",
]:

    x = results[
        name
    ]

    print(
        f"\n{name}:"
    )

    print(
        " checked:",
        x["checked"]
    )

    print(
        " questions with UNK:",
        x["questions_with_unk"]
    )

    print(
        " total UNK tokens:",
        x["unk_token_total"]
    )

    print(
        " mean tokenized length:",
        x["mean_length"]
    )

    if x[
        "examples_with_unk"
    ]:

        print(
            " UNK examples:",
            x["examples_with_unk"][:5]
        )


# SentencePiece should not explode ordinary natural-language questions
# into the v5 behavior we saw.
assert (
    results[
        "webqsp_questions"
    ][
        "questions_with_unk"
    ]
    == 0
)

assert (
    results[
        "cwq_questions"
    ][
        "questions_with_unk"
    ]
    == 0
)


print(
    "\nQuestion-tokenization health gate: PASSED"
)


# ======================================================================
# 11. SPECIAL TOKEN GATE
# ======================================================================

special = results[
    "generate_seq_special_tokens"
]


print(
    "\n"
    + "=" * 118
)

print(
    "E. generate_seq SPECIAL-TOKEN BEHAVIOR"
)

print(
    "=" * 118
)


for k, v in special.items():

    print(
        f"{k:<20}:",
        v
    )


assert special[
    "add_bos_token"
] is True

assert special[
    "add_eos_token"
] is False

assert special[
    "bos_token_id"
] == 1

assert special[
    "eos_token_id"
] == 2


print(
    "\ngenerate_seq tokenization behavior gate: PASSED"
)


# ======================================================================
# 12. FINAL DECISION
# ======================================================================

EXACT_4442_TOKENIZER_FIDELITY_PASSED = True


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R7 EXACT TOKENIZER AUDIT: PASSED"
)

print(
    "=" * 118
)

print(
    "\nExperimental reference environment:"
)

print(
    " transformers==4.44.2"
)

print(
    " tokenizers==0.19.1"
)

print(
    " sentencepiece==0.2.0"
)

print(
    "\nCurrent v5 M3 TEST realization valid:",
    "NO"
)

print(
    "Whitespace repair of M3 accepted:",
    "NO"
)

print(
    "TEST regeneration required:",
    "YES"
)

print(
    "\nTEST JSONLs modified:",
    "NO"
)

print(
    "AFP changed:",
    "NO"
)

print(
    "TEST tuning:",
    "NO"
)

print(
    "\nResults:",
    audit_output_path
)

print(
    "\nNEXT:"
)

print(
    "Paste this output here."
)

print(
    "Do NOT regenerate TEST plans until the "
    "exact generation environment is prepared."
)

CELL 16-R7 — EXACT ORIGINAL TOKENIZER ENVIRONMENT AUDIT

Freeze gate: PASSED

Exact original notebook environment:
 transformers:   4.44.2
 tokenizers:     0.19.1
 sentencepiece:  0.2.0

Installing exact tokenizer environment into isolated directory...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, bu

Isolated installation: COMPLETE

Audit payload written: /kaggle/working/step2_rq1_test/tokenizer_fidelity_audit/cell16_r7_inputs.json

Running isolated exact-environment audit...
Subprocess return code: 0

SUBPROCESS STDERR:
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.

0it [00:00, ?it/s]
0it [00:00, ?it/s]


A. EXACT ENVIRONMENT
transformers             : 4.44.2
tokenizers               : 0.19.1
sentencepiece            : 0.2.0
tokenizer_class          : LlamaTokenizer
tokenizer_module         : transformers.models.llama.tokenization_llama
is_fast                  : False
name_or_path             : rmanluo/RoG

Exact software + slow-tokenizer identity: PASSED

B. RELATION ROUND-TRIP PROBE
input: 'people.person.place_of_birth'
decoded: 'people.person.place_of_birth'
exact: True

C. FROZEN VALIDATION RE

In [21]:
# ======================================================================
# CELL 16-R8
# QUARANTINE INVALID v5 M3 + EXACT GENERATION-ENVIRONMENT PREFLIGHT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M3
#   - run Cell 17
#   - restart kernel
#   - overwrite either original M3 JSONL
#
# This cell:
#   1. verifies the invalid M3 artifacts by their frozen hashes
#   2. copies them into an explicit audit quarantine
#   3. records why they were rejected
#   4. recovers the original notebook software setup
#   5. audits current/exact subprocess Torch+CUDA state
#   6. recovers the ACTUAL executed M3 source from history if available
#
# NO model generation occurs.
# ======================================================================

import ast
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch


print("=" * 120)
print("CELL 16-R8 — INVALID-M3 QUARANTINE + GENERATION PREFLIGHT")
print("=" * 120)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert globals().get(
    "EXACT_4442_TOKENIZER_FIDELITY_PASSED",
    False
) is True

assert "persisted_nb_16ar" in globals()


print("\nFreeze gate:                    PASSED")
print("Oracle repair gate:             PASSED")
print("Exact 4.44.2 tokenizer gate:    PASSED")


# ======================================================================
# 2. ORIGINAL INVALID M3 FILES
# ======================================================================

WEB_M3_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_M3_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)

M3_MANIFEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16a_m3_materialization_manifest.json"
)


assert WEB_M3_PATH.exists()
assert CWQ_M3_PATH.exists()


WEB_M3_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_M3_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


def sha256_file_r8(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


web_m3_sha = sha256_file_r8(
    WEB_M3_PATH
)

cwq_m3_sha = sha256_file_r8(
    CWQ_M3_PATH
)


assert web_m3_sha == WEB_M3_EXPECTED_SHA
assert cwq_m3_sha == CWQ_M3_EXPECTED_SHA


print("\nInvalid M3 artifact identity: PASSED")
print(" WebQSP:", web_m3_sha)
print(" CWQ:   ", cwq_m3_sha)


# ======================================================================
# 3. LOAD R7 AUDIT EVIDENCE
# ======================================================================

R7_RESULTS_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "tokenizer_fidelity_audit/"
    "cell16_r7_results.json"
)

assert R7_RESULTS_PATH.exists()


r7 = json.loads(
    R7_RESULTS_PATH.read_text(
        encoding="utf-8"
    )
)


assert r7["software"]["transformers"] == "4.44.2"
assert r7["software"]["tokenizers"] == "0.19.1"
assert r7["software"]["sentencepiece"] == "0.2.0"
assert r7["software"]["is_fast"] is False

assert (
    r7["webqsp_validation_relations"]["exact"]
    ==
    r7["webqsp_validation_relations"]["checked"]
)

assert (
    r7["cwq_validation_relations"]["exact"]
    ==
    r7["cwq_validation_relations"]["checked"]
)


print("\nR7 evidence file: PASSED")


# ======================================================================
# 4. QUARANTINE COPY — NEVER DELETE ORIGINAL FAILED RUN
# ======================================================================

QUARANTINE_DIR = Path(
    "/kaggle/working/step2_rq1_test/"
    "invalid_transformers5_m3"
)

QUARANTINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


WEB_QUARANTINE = (
    QUARANTINE_DIR
    /
    "planning_webqsp_test.INVALID_TRANSFORMERS5.jsonl"
)

CWQ_QUARANTINE = (
    QUARANTINE_DIR
    /
    "planning_cwq_test.INVALID_TRANSFORMERS5.jsonl"
)


# copy2 is idempotent here; verify hashes afterwards.
shutil.copy2(
    WEB_M3_PATH,
    WEB_QUARANTINE
)

shutil.copy2(
    CWQ_M3_PATH,
    CWQ_QUARANTINE
)


assert (
    sha256_file_r8(
        WEB_QUARANTINE
    )
    ==
    WEB_M3_EXPECTED_SHA
)

assert (
    sha256_file_r8(
        CWQ_QUARANTINE
    )
    ==
    CWQ_M3_EXPECTED_SHA
)


if M3_MANIFEST_PATH.exists():

    shutil.copy2(
        M3_MANIFEST_PATH,
        QUARANTINE_DIR
        /
        "cell16a_m3_materialization_manifest.INVALID_TRANSFORMERS5.json"
    )


print("\nInvalid M3 quarantine copies: VERIFIED")


# ======================================================================
# 5. WRITE SOFTWARE-FIDELITY FAILURE MANIFEST
# ======================================================================

FAILURE_MANIFEST_PATH = (
    QUARANTINE_DIR
    /
    "invalid_m3_software_fidelity_manifest.json"
)


failure_manifest = {
    "status":
        "INVALID_DO_NOT_USE_FOR_RESULTS",

    "reason":
        (
            "TEST plans were generated under transformers 5.0.0, "
            "where AutoTokenizer(use_fast=False) resolved to "
            "TokenizersBackend / fast behavior rather than the "
            "experiment notebook's transformers 4.44.2 slow "
            "LlamaTokenizer behavior."
        ),

    "scientific_consequence":
        (
            "Planner INPUT token IDs differed materially from the "
            "original experiment tokenizer. Therefore this is not "
            "a decoding-only defect and whitespace post-processing "
            "is not an acceptable repair."
        ),

    "invalid_artifacts": {
        "webqsp": {
            "path":
                str(WEB_M3_PATH),

            "sha256":
                web_m3_sha,
        },

        "cwq": {
            "path":
                str(CWQ_M3_PATH),

            "sha256":
                cwq_m3_sha,
        },
    },

    "invalid_generation_environment": {
        "transformers":
            "5.0.0",

        "observed_tokenizer_class":
            "TokenizersBackend",

        "observed_is_fast":
            True,
    },

    "experimental_reference_environment": {
        "transformers":
            "4.44.2",

        "tokenizers":
            "0.19.1",

        "sentencepiece":
            "0.2.0",

        "tokenizer_class":
            "LlamaTokenizer",

        "is_fast":
            False,
    },

    "r7_evidence":
        str(R7_RESULTS_PATH),

    "afp_changed":
        False,

    "test_used_for_tuning":
        False,

    "failed_test_results_accepted":
        False,

    "whitespace_repair_accepted":
        False,

    "regeneration_required":
        True,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


FAILURE_MANIFEST_PATH.write_text(
    json.dumps(
        failure_manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "Failure manifest:",
    FAILURE_MANIFEST_PATH
)


# ======================================================================
# 6. RECOVER ORIGINAL NOTEBOOK CELL 1 SOFTWARE SETUP
# ======================================================================

setup_source = "".join(
    persisted_nb_16ar[
        "cells"
    ][1].get(
        "source",
        []
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "A. ORIGINAL NOTEBOOK SOFTWARE SETUP"
)

print(
    "=" * 120
)


print(
    setup_source
)


# ----------------------------------------------------------------------
# Try to recover _torch_pin without executing pip.
# ----------------------------------------------------------------------

torch_pin_expression = None

for line in setup_source.splitlines():

    if re.search(
        r"^\s*_torch_pin\s*=",
        line
    ):

        torch_pin_expression = line.strip()

        break


print(
    "\nRecovered _torch_pin assignment:"
)

print(
    torch_pin_expression
)


# ======================================================================
# 7. CURRENT MAIN-KERNEL TORCH/CUDA STATE
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "B. CURRENT MAIN-KERNEL GPU SOFTWARE"
)

print(
    "=" * 120
)


print(
    "Python:",
    sys.version.split()[0]
)

print(
    "torch:",
    torch.__version__
)

print(
    "torch CUDA build:",
    torch.version.cuda
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

    print(
        "GPU 0:",
        torch.cuda.get_device_name(0)
    )

    props = torch.cuda.get_device_properties(
        0
    )

    print(
        "GPU memory GiB:",
        round(
            props.total_memory
            /
            (1024 ** 3),
            3
        )
    )


# ======================================================================
# 8. EXACT ISOLATED ENVIRONMENT GPU PREFLIGHT
# ======================================================================

EXACT_ENV_DIR = Path(
    "/kaggle/working/"
    "rog_exact_tokenizer_env_4442"
)

assert EXACT_ENV_DIR.exists()


preflight_script = r'''
import json

import torch
import transformers
import tokenizers
import sentencepiece

from transformers import AutoTokenizer


tok = AutoTokenizer.from_pretrained(
    "rmanluo/RoG",
    use_fast=False,
    clean_up_tokenization_spaces=True,
    local_files_only=True,
)


result = {
    "torch":
        torch.__version__,

    "torch_cuda_build":
        torch.version.cuda,

    "cuda_available":
        torch.cuda.is_available(),

    "gpu_name":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),

    "transformers":
        transformers.__version__,

    "tokenizers":
        tokenizers.__version__,

    "sentencepiece":
        sentencepiece.__version__,

    "tokenizer_class":
        tok.__class__.__name__,

    "is_fast":
        getattr(
            tok,
            "is_fast",
            None
        ),

    "probe_decode":
        tok.decode(
            tok.encode(
                "people.person.place_of_birth",
                add_special_tokens=False
            ),
            skip_special_tokens=True
        ),
}


print(
    json.dumps(
        result
    )
)
'''


env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(EXACT_ENV_DIR)
    +
    (
        os.pathsep
        +
        existing_pythonpath

        if existing_pythonpath
        else ""
    )
)


gpu_preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        preflight_script,
    ],
    env=env,
    text=True,
    capture_output=True,
)


print(
    "\n"
    + "=" * 120
)

print(
    "C. EXACT 4.44.2 SUBPROCESS GPU PREFLIGHT"
)

print(
    "=" * 120
)


print(
    "return code:",
    gpu_preflight.returncode
)


if gpu_preflight.stderr.strip():

    print(
        "\nSTDERR:"
    )

    print(
        gpu_preflight.stderr[-4000:]
    )


assert gpu_preflight.returncode == 0


preflight_lines = [
    x.strip()
    for x in gpu_preflight.stdout.splitlines()
    if x.strip()
]


assert preflight_lines


gpu_exact = json.loads(
    preflight_lines[-1]
)


for k, v in gpu_exact.items():

    print(
        f"{k:<24}:",
        v
    )


assert gpu_exact["transformers"] == "4.44.2"
assert gpu_exact["tokenizers"] == "0.19.1"
assert gpu_exact["sentencepiece"] == "0.2.0"
assert gpu_exact["tokenizer_class"] == "LlamaTokenizer"
assert gpu_exact["is_fast"] is False

assert (
    gpu_exact["probe_decode"]
    ==
    "people.person.place_of_birth"
)

assert gpu_exact["cuda_available"] is True


print(
    "\nExact-env CUDA/tokenizer preflight: PASSED"
)


# ======================================================================
# 9. RECOVER THE ACTUAL EXECUTED M3 SOURCE FROM IPYTHON HISTORY
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "D. EXACT EXECUTED M3 SOURCE RECOVERY"
)

print(
    "=" * 120
)


ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


def assigned_names_r8(tree):

    names = set()

    for node in ast.walk(tree):

        if isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            targets = (
                node.targets

                if isinstance(
                    node,
                    ast.Assign
                )

                else [
                    node.target
                ]
            )

            for target in targets:

                if isinstance(
                    target,
                    ast.Name
                ):

                    names.add(
                        target.id
                    )

    return names


m3_candidates = []


for hist_idx, source in enumerate(
    history
):

    if not isinstance(
        source,
        str
    ):
        continue

    if len(source) < 1000:
        continue

    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    names = assigned_names_r8(
        tree
    )


    # Strong structural signatures of the actual M3 materialization cell.
    if (
        "CELL16AM3_COMPLETE"
        in names
        and
        "FROZEN_TEST_PLANS_MATERIALIZED"
        in names
    ):

        m3_candidates.append(
            {
                "history_index":
                    hist_idx,

                "source":
                    source,

                "sha256":
                    hashlib.sha256(
                        source.encode(
                            "utf-8"
                        )
                    ).hexdigest(),

                "chars":
                    len(source),
            }
        )


print(
    "M3 history candidates found:",
    len(m3_candidates)
)


for x in m3_candidates:

    print(
        " history=",
        x["history_index"],
        "| chars=",
        x["chars"],
        "| sha=",
        x["sha256"]
    )


M3_SOURCE_RECOVERED = (
    len(m3_candidates) >= 1
)


if M3_SOURCE_RECOVERED:

    selected_m3 = sorted(
        m3_candidates,
        key=lambda x:
            x["history_index"]
    )[-1]


    EXACT_EXECUTED_M3_SOURCE = (
        selected_m3[
            "source"
        ]
    )


    EXACT_EXECUTED_M3_SOURCE_SHA256 = (
        selected_m3[
            "sha256"
        ]
    )


    M3_SOURCE_PATH = (
        QUARANTINE_DIR
        /
        "executed_m3_source_transformers5_failure.py"
    )


    M3_SOURCE_PATH.write_text(
        EXACT_EXECUTED_M3_SOURCE,
        encoding="utf-8"
    )


    print(
        "\nSelected M3 history index:",
        selected_m3[
            "history_index"
        ]
    )

    print(
        "M3 source SHA256:",
        EXACT_EXECUTED_M3_SOURCE_SHA256
    )

    print(
        "M3 source saved:",
        M3_SOURCE_PATH
    )


else:

    EXACT_EXECUTED_M3_SOURCE = None
    EXACT_EXECUTED_M3_SOURCE_SHA256 = None

    print(
        "\nActual M3 source not found in current "
        "IPython history."
    )

    print(
        "This does NOT affect the quarantined artifacts."
    )


# ======================================================================
# 10. SAVE PREFLIGHT MANIFEST
# ======================================================================

PREFLIGHT_MANIFEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_r8_generation_environment_preflight.json"
)


preflight_manifest = {
    "invalid_m3_quarantined":
        True,

    "invalid_m3_webqsp_sha256":
        web_m3_sha,

    "invalid_m3_cwq_sha256":
        cwq_m3_sha,

    "exact_tokenizer_environment_passed":
        True,

    "exact_reference_environment": {
        "transformers":
            "4.44.2",

        "tokenizers":
            "0.19.1",

        "sentencepiece":
            "0.2.0",
    },

    "main_kernel": {
        "torch":
            torch.__version__,

        "torch_cuda_build":
            torch.version.cuda,

        "cuda_available":
            torch.cuda.is_available(),

        "gpu":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
    },

    "exact_subprocess": gpu_exact,

    "torch_pin_expression":
        torch_pin_expression,

    "m3_source_recovered":
        M3_SOURCE_RECOVERED,

    "m3_source_sha256":
        EXACT_EXECUTED_M3_SOURCE_SHA256,

    "model_generation_executed_in_r8":
        False,

    "test_tuning":
        False,

    "afp_changed":
        False,
}


PREFLIGHT_MANIFEST_PATH.write_text(
    json.dumps(
        preflight_manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


CELL16_R8_PREFLIGHT_PASSED = True


print(
    "\n"
    + "=" * 120
)

print(
    "CELL 16-R8 PREFLIGHT COMPLETE"
)

print(
    "=" * 120
)

print(
    "\nInvalid Transformers-5 M3: QUARANTINED"
)

print(
    "Original failed files deleted: NO"
)

print(
    "Original failed files overwritten: NO"
)

print(
    "Exact 4.44.2 tokenizer: PASSED"
)

print(
    "Exact-env CUDA access: PASSED"
)

print(
    "M3 source recovered:",
    M3_SOURCE_RECOVERED
)

print(
    "Model generation executed: NO"
)

print(
    "TEST tuning: NO"
)

print(
    "AFP changed: NO"
)

print(
    "\nPreflight manifest:",
    PREFLIGHT_MANIFEST_PATH
)

print(
    "\nNEXT ACTION:"
)

print(
    "Paste the full R8 output here."
)

print(
    "Do NOT run Cell 17 or regenerate TEST yet."
)

CELL 16-R8 — INVALID-M3 QUARANTINE + GENERATION PREFLIGHT

Freeze gate:                    PASSED
Oracle repair gate:             PASSED
Exact 4.44.2 tokenizer gate:    PASSED

Invalid M3 artifact identity: PASSED
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ:    95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

R7 evidence file: PASSED

Invalid M3 quarantine copies: VERIFIED
Failure manifest: /kaggle/working/step2_rq1_test/invalid_transformers5_m3/invalid_m3_software_fidelity_manifest.json

A. ORIGINAL NOTEBOOK SOFTWARE SETUP
import torch
_torch_pin = f"torch=={torch.__version__.split('+')[0]}"
print("Keeping installed PyTorch:", _torch_pin)

# accelerate==0.33.0 caps numpy<2.0, which drags Kaggle's numpy back from 2.x
# to 1.26.4 -- but Kaggle's pandas wheel is built against numpy 2.x's C ABI,
# so that downgrade breaks pandas at import time ("numpy.dtype size changed").
# accelerate>=0.34 dropped that cap, so bump it and pin numpy exp

In [22]:
# ======================================================================
# KAGGLE ACCOUNT MIGRATION BACKUP — PROJECT STATE
# ======================================================================
#
# RUN AS A NEW CELL IN THE CURRENT ACCOUNT.
#
# Does NOT:
#   - run experiments
#   - modify AFP
#   - modify TEST plans
#   - restart kernel
#
# Creates:
#   /kaggle/working/AdaPruner_KGQA_MIGRATION_20260901.zip
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile


ROOT = Path("/kaggle/working")

OUT_ZIP = ROOT / "AdaPruner_KGQA_MIGRATION_20260901.zip"

MANIFEST_PATH = ROOT / "AdaPruner_KGQA_MIGRATION_MANIFEST.json"


# ----------------------------------------------------------------------
# Critical project directories/files
# ----------------------------------------------------------------------

targets = [
    ROOT / "step2_rq1_dev",
    ROOT / "step2_rq1_test",
    ROOT / "step3_rq2_dev_v1",
    ROOT / "reasoning-on-graphs",
    ROOT / "rog_exact_tokenizer_env_4442",
]


# Include only things that actually exist.
targets = [
    p for p in targets
    if p.exists()
]


print("=" * 100)
print("ADAPRUNER-KGQA MIGRATION BACKUP")
print("=" * 100)

print("\nTargets:")

for p in targets:
    print(" ", p)


# ----------------------------------------------------------------------
# SHA helper
# ----------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ----------------------------------------------------------------------
# Inventory files
# ----------------------------------------------------------------------

files = []

for target in targets:

    if target.is_file():

        files.append(target)

    else:

        files.extend(
            p
            for p in target.rglob("*")
            if p.is_file()
        )


files = sorted(
    set(files),
    key=lambda p: str(p)
)


print("\nFiles to archive:", len(files))


# ----------------------------------------------------------------------
# Build manifest BEFORE ZIP
# ----------------------------------------------------------------------

manifest_files = []

total_bytes = 0


for i, path in enumerate(files, start=1):

    size = path.stat().st_size

    total_bytes += size

    rel = path.relative_to(ROOT)

    manifest_files.append(
        {
            "path": str(rel),
            "size_bytes": size,
            "sha256": sha256_file(path),
        }
    )

    if (
        i % 100 == 0
        or i == len(files)
    ):
        print(
            f"Hashed {i}/{len(files)} files"
        )


manifest = {
    "project": "AdaPruner-KGQA",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "root": str(ROOT),
    "file_count": len(files),
    "total_bytes": total_bytes,

    "scientific_state": {
        "afp_development_frozen": globals().get(
            "AFP_DEVELOPMENT_FROZEN"
        ),

        "final_afp_freeze_sha256": globals().get(
            "FINAL_AFP_FREEZE_SHA256"
        ),

        "oracle_repair_validated": globals().get(
            "CELL16_ORACLE_REPAIR_VALIDATED"
        ),

        "exact_4442_tokenizer_fidelity_passed": globals().get(
            "EXACT_4442_TOKENIZER_FIDELITY_PASSED"
        ),

        "invalid_transformers5_m3_not_accepted": True,

        "cell17_started": False,
    },

    "files": manifest_files,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nManifest:",
    MANIFEST_PATH
)


# ----------------------------------------------------------------------
# Add manifest itself
# ----------------------------------------------------------------------

files_with_manifest = (
    files
    +
    [MANIFEST_PATH]
)


# ----------------------------------------------------------------------
# ZIP
# ----------------------------------------------------------------------

if OUT_ZIP.exists():
    OUT_ZIP.unlink()


print("\nCreating ZIP...")


with zipfile.ZipFile(
    OUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for i, path in enumerate(
        files_with_manifest,
        start=1
    ):

        arcname = path.relative_to(ROOT)

        zf.write(
            path,
            arcname=str(arcname)
        )

        if (
            i % 100 == 0
            or i == len(files_with_manifest)
        ):

            print(
                f"Archived {i}/"
                f"{len(files_with_manifest)} files"
            )


# ----------------------------------------------------------------------
# Verify archive can be read
# ----------------------------------------------------------------------

with zipfile.ZipFile(
    OUT_ZIP,
    "r"
) as zf:

    bad = zf.testzip()


assert bad is None, (
    f"ZIP integrity failure at {bad}"
)


zip_sha = sha256_file(
    OUT_ZIP
)


print("\n" + "=" * 100)
print("MIGRATION BACKUP COMPLETE")
print("=" * 100)

print(
    "\nZIP:",
    OUT_ZIP
)

print(
    "ZIP size GiB:",
    round(
        OUT_ZIP.stat().st_size
        / (1024 ** 3),
        3
    )
)

print(
    "ZIP SHA256:",
    zip_sha
)

print(
    "Archived files:",
    len(files_with_manifest)
)

print(
    "\nZIP integrity: PASSED"
)

print(
    "\nDOWNLOAD THIS ZIP BEFORE LEAVING THIS ACCOUNT."
)

ADAPRUNER-KGQA MIGRATION BACKUP

Targets:
  /kaggle/working/step2_rq1_dev
  /kaggle/working/step2_rq1_test
  /kaggle/working/step3_rq2_dev_v1
  /kaggle/working/reasoning-on-graphs
  /kaggle/working/rog_exact_tokenizer_env_4442

Files to archive: 6576
Hashed 100/6576 files
Hashed 200/6576 files
Hashed 300/6576 files
Hashed 400/6576 files
Hashed 500/6576 files
Hashed 600/6576 files
Hashed 700/6576 files
Hashed 800/6576 files
Hashed 900/6576 files
Hashed 1000/6576 files
Hashed 1100/6576 files
Hashed 1200/6576 files
Hashed 1300/6576 files
Hashed 1400/6576 files
Hashed 1500/6576 files
Hashed 1600/6576 files
Hashed 1700/6576 files
Hashed 1800/6576 files
Hashed 1900/6576 files
Hashed 2000/6576 files
Hashed 2100/6576 files
Hashed 2200/6576 files
Hashed 2300/6576 files
Hashed 2400/6576 files
Hashed 2500/6576 files
Hashed 2600/6576 files
Hashed 2700/6576 files
Hashed 2800/6576 files
Hashed 2900/6576 files
Hashed 3000/6576 files
Hashed 3100/6576 files
Hashed 3200/6576 files
Hashed 3300/6576 files

In [16]:
# ======================================================================
# CELL 16-R4A
# EXACT TOKENIZER-ONLY RECOVERY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M3
#   - rerun Cell 16
#   - restart kernel
#
# After this PASSES, rerun Cell 16-R4 unchanged.
# ======================================================================

import ast
import os
from pathlib import Path

import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


print("=" * 105)
print("CELL 16-R4A — TOKENIZER-ONLY RECOVERY")
print("=" * 105)


# ----------------------------------------------------------------------
# 1. Scientific-state gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. If tokenizer somehow already returned, simply reuse it
# ----------------------------------------------------------------------

if (
    "tokenizer" in globals()
    and
    globals()["tokenizer"] is not None
):

    print("\nTokenizer already exists in globals.")

else:

    # ------------------------------------------------------------------
    # 3. Recover EXACT original tokenizer assignment from initial
    #    planner setup. We previously verified tokenizer assignment
    #    was in persisted notebook cell 13.
    # ------------------------------------------------------------------

    TARGET_CELL = 13

    cell_source = "".join(
        persisted_nb_16ar[
            "cells"
        ][TARGET_CELL].get(
            "source",
            []
        )
    )

    tree = ast.parse(
        cell_source
    )


    tokenizer_assignment = None


    for node in tree.body:

        targets = []

        if isinstance(
            node,
            ast.Assign
        ):
            targets = node.targets

        elif isinstance(
            node,
            ast.AnnAssign
        ):
            targets = [
                node.target
            ]


        for target in targets:

            if (
                isinstance(
                    target,
                    ast.Name
                )
                and
                target.id == "tokenizer"
            ):

                tokenizer_assignment = (
                    ast.get_source_segment(
                        cell_source,
                        node
                    )
                )

                break


        if tokenizer_assignment is not None:
            break


    assert tokenizer_assignment is not None, (
        "Exact tokenizer assignment was not found "
        "in persisted planner setup cell 13."
    )


    print(
        "\nExact tokenizer assignment recovered "
        "from persisted cell 13:"
    )

    print(
        tokenizer_assignment
    )


    # ------------------------------------------------------------------
    # 4. Build assignment index for preceding setup variables
    # ------------------------------------------------------------------

    assignment_index = {}


    for cell_idx in range(
        0,
        TARGET_CELL + 1
    ):

        src = "".join(
            persisted_nb_16ar[
                "cells"
            ][cell_idx].get(
                "source",
                []
            )
        )

        try:
            parsed = ast.parse(
                src
            )

        except Exception:
            continue


        for node in parsed.body:

            if isinstance(
                node,
                ast.Assign
            ):

                for target in node.targets:

                    if isinstance(
                        target,
                        ast.Name
                    ):

                        assignment_index[
                            target.id
                        ] = (
                            cell_idx,
                            ast.get_source_segment(
                                src,
                                node
                            )
                        )


            elif isinstance(
                node,
                ast.AnnAssign
            ):

                if isinstance(
                    node.target,
                    ast.Name
                ):

                    assignment_index[
                        node.target.id
                    ] = (
                        cell_idx,
                        ast.get_source_segment(
                            src,
                            node
                        )
                    )


    # ------------------------------------------------------------------
    # 5. Minimal exact-execution namespace
    # ------------------------------------------------------------------

    ns = {
        "os":
            os,

        "Path":
            Path,

        "torch":
            torch,

        "transformers":
            transformers,

        "AutoTokenizer":
            AutoTokenizer,

        "AutoModelForCausalLM":
            AutoModelForCausalLM,
    }


    # Reuse already surviving globals where appropriate.
    for name in [
        "MODEL_NAME",
        "model_name",
        "model_path",
        "MODEL_PATH",
    ]:

        if name in globals():

            ns[
                name
            ] = globals()[
                name
            ]


    # ------------------------------------------------------------------
    # 6. Recover missing assignment dependencies on demand
    # ------------------------------------------------------------------

    recovering = set()


    def recover_assignment_symbol(
        symbol
    ):

        if symbol in ns:
            return


        if symbol in globals():

            ns[
                symbol
            ] = globals()[
                symbol
            ]

            return


        assert symbol not in recovering, (
            f"Circular setup dependency while "
            f"recovering {symbol}"
        )


        assert symbol in assignment_index, (
            f"Could not recover setup dependency: "
            f"{symbol}"
        )


        recovering.add(
            symbol
        )


        cell_idx, src = (
            assignment_index[
                symbol
            ]
        )


        # Try execution. If another setup variable is missing,
        # recover that exact variable first.
        while True:

            try:

                exec(
                    src,
                    ns
                )

                break

            except NameError as e:

                missing_name = getattr(
                    e,
                    "name",
                    None
                )

                assert missing_name, (
                    f"Unresolved NameError while "
                    f"recovering {symbol}: {e}"
                )

                recover_assignment_symbol(
                    missing_name
                )


        recovering.remove(
            symbol
        )


    # ------------------------------------------------------------------
    # 7. Execute exact tokenizer assignment
    # ------------------------------------------------------------------

    while True:

        try:

            exec(
                tokenizer_assignment,
                ns
            )

            break

        except NameError as e:

            missing_name = getattr(
                e,
                "name",
                None
            )

            assert missing_name, (
                f"Tokenizer recovery failed: {e}"
            )

            recover_assignment_symbol(
                missing_name
            )


    assert "tokenizer" in ns

    tokenizer = ns[
        "tokenizer"
    ]

    globals()[
        "tokenizer"
    ] = tokenizer


# ----------------------------------------------------------------------
# 8. Identity audit
# ----------------------------------------------------------------------

assert tokenizer is not None


tokenizer_name = str(
    getattr(
        tokenizer,
        "name_or_path",
        ""
    )
)


print("\nTokenizer recovered:")
print(
    " class:",
    tokenizer.__class__.__name__
)

print(
    " module:",
    tokenizer.__class__.__module__
)

print(
    " name_or_path:",
    tokenizer_name
)

print(
    " is_fast:",
    getattr(
        tokenizer,
        "is_fast",
        None
    )
)


assert (
    "rmanluo/RoG".lower()
    in
    tokenizer_name.lower()
), (
    f"Unexpected tokenizer identity: "
    f"{tokenizer_name}"
)


print(
    "\nTokenizer model identity: PASSED"
)


# ----------------------------------------------------------------------
# 9. Small non-destructive round-trip probe
# ----------------------------------------------------------------------

probe = (
    "people.person.place_of_birth"
)

ids = tokenizer.encode(
    probe,
    add_special_tokens=False
)

decoded = tokenizer.decode(
    ids,
    skip_special_tokens=True
)


print("\nRound-trip probe:")
print(" original:", repr(probe))
print(" decoded: ", repr(decoded))


print(
    "\n"
    + "=" * 105
)

print(
    "CELL 16-R4A TOKENIZER RECOVERY COMPLETE"
)

print(
    "=" * 105
)

print("\nModel loaded: NO")
print("TEST plans modified: NO")
print("TEST plans regenerated: NO")
print("AFP changed: NO")
print("TEST tuning: NO")

print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16-R4 UNCHANGED."
)

CELL 16-R4A — TOKENIZER-ONLY RECOVERY

Freeze gate: PASSED

Exact tokenizer assignment recovered from persisted cell 13:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, clean_up_tokenization_spaces=True)

Tokenizer recovered:
 class: TokenizersBackend
 module: transformers.tokenization_utils_tokenizers
 name_or_path: rmanluo/RoG
 is_fast: True

Tokenizer model identity: PASSED

Round-trip probe:
 original: 'people.person.place_of_birth'
 decoded:  'pe op le. per son. pl ace _ of _ bir th'

CELL 16-R4A TOKENIZER RECOVERY COMPLETE

Model loaded: NO
TEST plans modified: NO
TEST plans regenerated: NO
AFP changed: NO
TEST tuning: NO

NEXT ACTION:
RERUN CELL 16-R4 UNCHANGED.


In [35]:
# ======================================================================
# RECOVER + MATERIALIZE EXACT FROZEN RoG TEST PLANS
# ======================================================================
# DO NOT:
#   - modify AFP
#   - tune anything
#   - change planner settings
#   - use TEST gold answers in planner input
#
# GOAL
# ----
# 1. Recover exact frozen RoG planner implementation/configuration.
# 2. Locate exact WebQSP/CWQ TEST examples.
# 3. Verify planner software against the frozen VALIDATION plan artifact.
# 4. Generate TEST predicted_paths.
# 5. Save:
#
#    planning_webqsp_test.jsonl
#    planning_cwq_test.jsonl
#
# Then rerun Cell 16 UNCHANGED.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import os
import re
import time
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE GATE
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    # Existing frozen validation planning rows
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing frozen objects:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16A = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16A
), (
    "AFP freeze SHA mismatch."
)


print("Cell 16A freeze gate: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. FROZEN RoG PLANNER IDENTITY
# ======================================================================
#
# These are DEVELOPMENT-FROZEN facts already established earlier.
# They are NOT being selected using TEST.
# ======================================================================

FROZEN_ROG_MODEL_ID = (
    "rmanluo/RoG"
)

FROZEN_ROG_TOP_K = 3

FROZEN_INSTRUCTION_SHA256 = (
    "e3687b4a5081c22c"
)

FROZEN_PLANNER_CONFIG_SHA_PREFIX = (
    "2c36bd"
)


print(
    "\nFrozen planner identity:"
)

print(
    " model:",
    FROZEN_ROG_MODEL_ID
)

print(
    " Top-K:",
    FROZEN_ROG_TOP_K
)

print(
    " instruction SHA prefix:",
    FROZEN_INSTRUCTION_SHA256
)

print(
    " planner config SHA prefix:",
    FROZEN_PLANNER_CONFIG_SHA_PREFIX
)


# ======================================================================
# 3. PATHS
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists(), (
    f"Persisted notebook not found: {NOTEBOOK_PATH}"
)


PLANNING_OUT_DIR = Path(
    "/kaggle/working/step2_rq1_test"
)

PLANNING_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR
    / "planning_cwq_test.jsonl"
)


# ======================================================================
# 4. LOAD PERSISTED NOTEBOOK
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16a = json.load(f)


print(
    "\nPersisted notebook loaded:",
    NOTEBOOK_PATH
)


# ======================================================================
# 5. SEARCH FOR EXACT PLANNING CELLS
# ======================================================================
#
# We locate cells containing the known validation planning artifact
# names and/or the frozen model ID.
# ======================================================================

planning_cell_hits = []


SEARCH_TERMS_16A = [
    "planning_webqsp_validation",
    "planning_cwq_validation",
    "rmanluo/RoG",
    "predicted_paths",
    "planning_time_sec",
]


for cell_idx, cell in enumerate(
    persisted_nb_16a[
        "cells"
    ]
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    score = sum(
        term in source
        for term in SEARCH_TERMS_16A
    )


    if score >= 2:

        planning_cell_hits.append(
            {
                "cell_idx":
                    int(
                        cell_idx
                    ),

                "score":
                    int(
                        score
                    ),

                "source":
                    source,
            }
        )


assert len(
    planning_cell_hits
) >= 1, (
    "Could not locate the persisted RoG planning implementation."
)


planning_cell_hits = sorted(
    planning_cell_hits,
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "cell_idx"
            ],
        )
)


print(
    "\nCandidate persisted planning cells:"
)


for hit in planning_cell_hits[
    :10
]:

    print(
        f"  cell {hit['cell_idx']}: "
        f"score={hit['score']}"
    )


# ======================================================================
# 6. RECOVER STRING LITERALS AND FIND FROZEN INSTRUCTION
# ======================================================================

def iter_string_literals_16a(
    source
):

    try:

        tree = ast.parse(
            source
        )

    except Exception:

        return []


    values = []


    for node in ast.walk(
        tree
    ):

        if (
            isinstance(
                node,
                ast.Constant
            )
            and
            isinstance(
                node.value,
                str
            )
        ):

            values.append(
                node.value
            )


    return values


def sha256_text_16a(
    text
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


instruction_candidates = []


for cell in persisted_nb_16a[
    "cells"
]:

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    for text in iter_string_literals_16a(
        source
    ):

        digest = sha256_text_16a(
            text
        )


        if digest.startswith(
            FROZEN_INSTRUCTION_SHA256
        ):

            instruction_candidates.append(
                text
            )


instruction_candidates = list(
    dict.fromkeys(
        instruction_candidates
    )
)


print(
    "\nFrozen instruction literal matches:",
    len(
        instruction_candidates
    )
)


if len(
    instruction_candidates
) == 1:

    FROZEN_ROG_INSTRUCTION = (
        instruction_candidates[
            0
        ]
    )


    print(
        "Exact frozen instruction recovered: PASSED"
    )


else:

    FROZEN_ROG_INSTRUCTION = None


# ======================================================================
# 7. RECOVER PLANNER-RELATED FUNCTION DEFINITIONS
# ======================================================================

PLANNER_FUNCTION_KEYWORDS = [
    "plan",
    "predict",
    "generate",
    "path",
    "relation",
    "parse",
]


recovered_functions_16a = []


for hit in planning_cell_hits:

    source = hit[
        "source"
    ]


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        if not isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            continue


        name_lower = (
            node.name.lower()
        )


        function_source = (
            ast.get_source_segment(
                source,
                node
            )
        )


        if function_source is None:

            continue


        keyword_score = sum(
            keyword
            in
            name_lower
            for keyword
            in PLANNER_FUNCTION_KEYWORDS
        )


        body_score = sum(
            token
            in
            function_source
            for token
            in [
                "generate(",
                "predicted_paths",
                "relation",
                "num_return_sequences",
                "num_beams",
                "tokenizer",
            ]
        )


        if (
            keyword_score
            +
            body_score
            >=
            2
        ):

            recovered_functions_16a.append(
                {
                    "cell_idx":
                        hit[
                            "cell_idx"
                        ],

                    "name":
                        node.name,

                    "source":
                        function_source,

                    "score":
                        keyword_score
                        +
                        body_score,
                }
            )


# Deduplicate by exact name/source.
dedup = {}

for row in recovered_functions_16a:

    key = (
        row[
            "name"
        ],
        row[
            "source"
        ],
    )

    dedup[
        key
    ] = row


recovered_functions_16a = sorted(
    dedup.values(),
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "cell_idx"
            ],
            x[
                "name"
            ],
        )
)


print(
    "\nRecovered planner-related function candidates:"
)


for row in recovered_functions_16a[
    :20
]:

    print(
        f"  cell {row['cell_idx']}: "
        f"{row['name']} "
        f"(score={row['score']})"
    )


# ======================================================================
# 8. SEARCH SURVIVING GLOBAL PLANNER FUNCTIONS
# ======================================================================

global_planner_candidates = []


for name, obj in list(
    globals().items()
):

    if not callable(
        obj
    ):

        continue


    lname = name.lower()


    if not any(
        keyword
        in
        lname
        for keyword
        in [
            "plan",
            "predict",
            "generate",
            "relation",
            "rog",
        ]
    ):

        continue


    try:

        signature = str(
            inspect.signature(
                obj
            )
        )

    except Exception:

        signature = "?"


    global_planner_candidates.append(
        (
            name,
            signature
        )
    )


print(
    "\nSurviving planner-like globals:"
)


for name, signature in (
    global_planner_candidates[
        :30
    ]
):

    print(
        f"  {name}{signature}"
    )


# ======================================================================
# 9. LOCATE RAW TEST DATA — GLOBALS FIRST
# ======================================================================
#
# Required raw TEST row fields:
#
#   id
#   question
#   q_entity
#   a_entity
#   graph
#
# predicted_paths must NOT be required yet.
# ======================================================================

RAW_REQUIRED_16A = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
}


def valid_raw_rows_16a(
    rows
):

    if not isinstance(
        rows,
        list
    ):

        return False


    if len(
        rows
    ) == 0:

        return False


    probes = sorted(
        set(
            [
                0,
                len(
                    rows
                )
                //
                2,
                len(
                    rows
                )
                -
                1,
            ]
        )
    )


    for idx in probes:

        rec = rows[
            idx
        ]


        if not isinstance(
            rec,
            dict
        ):

            return False


        if not RAW_REQUIRED_16A.issubset(
            rec.keys()
        ):

            return False


    return True


def discover_raw_test_global_16a(
    dataset_name
):

    dataset_name = (
        dataset_name.lower()
    )


    preferred_names = [
        f"{dataset_name}_test",
        f"{dataset_name}_test_rows",
        f"{dataset_name}_test_data",
        f"{dataset_name}_raw_test",
        f"{dataset_name}_test_examples",
    ]


    matches = []


    for name in preferred_names:

        if name not in globals():

            continue


        value = globals()[
            name
        ]


        if valid_raw_rows_16a(
            value
        ):

            matches.append(
                (
                    name,
                    value
                )
            )


    if len(
        matches
    ) == 1:

        print(
            f"{dataset_name.upper()} raw TEST "
            f"from global: {matches[0][0]}"
        )

        return matches[
            0
        ][
            1
        ]


    if len(
        matches
    ) > 1:

        id_sets = [
            {
                str(
                    row[
                        "id"
                    ]
                )
                for row in rows
            }
            for _, rows
            in matches
        ]


        assert all(
            id_sets[
                0
            ]
            ==
            ids
            for ids
            in id_sets[
                1:
            ]
        ), (
            f"{dataset_name}: multiple non-identical "
            "raw TEST globals."
        )


        print(
            f"{dataset_name.upper()}: multiple "
            "equivalent raw TEST globals found."
        )


        return matches[
            0
        ][
            1
        ]


    return None


webqsp_raw_test_16a = (
    discover_raw_test_global_16a(
        "webqsp"
    )
)


cwq_raw_test_16a = (
    discover_raw_test_global_16a(
        "cwq"
    )
)


# ======================================================================
# 10. LOCATE RAW TEST FILES IF GLOBALS ARE ABSENT
# ======================================================================

def read_json_or_jsonl_16a(
    path
):

    suffix = (
        path.suffix.lower()
    )


    if suffix == ".jsonl":

        rows = []

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = (
                    line.strip()
                )

                if not line:

                    continue

                rows.append(
                    json.loads(
                        line
                    )
                )


        return rows


    if suffix == ".json":

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(
                f
            )


        if isinstance(
            obj,
            list
        ):

            return obj


        if isinstance(
            obj,
            dict
        ):

            for key in [
                "data",
                "test",
                "examples",
                "rows",
            ]:

                if (
                    key in obj
                    and
                    isinstance(
                        obj[
                            key
                        ],
                        list
                    )
                ):

                    return obj[
                        key
                    ]


    return None


def discover_raw_test_file_16a(
    dataset_name
):

    dataset_name = (
        dataset_name.lower()
    )


    candidate_paths = []


    search_roots = [
        Path(
            "/kaggle/working"
        ),
        Path(
            "/kaggle/input"
        ),
    ]


    for root in search_roots:

        if not root.exists():

            continue


        for pattern in [
            f"*{dataset_name}*test*.json",
            f"*{dataset_name}*test*.jsonl",
            f"*{dataset_name.upper()}*test*.json",
            f"*{dataset_name.upper()}*test*.jsonl",
        ]:

            for path in root.rglob(
                pattern
            ):

                candidate_paths.append(
                    path
                )


    candidate_paths = sorted(
        set(
            candidate_paths
        )
    )


    valid = []


    for path in candidate_paths:

        # Skip output plan files themselves.
        if (
            "planning_"
            in
            path.name.lower()
        ):

            continue


        try:

            rows = (
                read_json_or_jsonl_16a(
                    path
                )
            )

        except Exception:

            continue


        if valid_raw_rows_16a(
            rows
        ):

            valid.append(
                (
                    path,
                    rows
                )
            )


    if len(
        valid
    ) == 0:

        return (
            None,
            None
        )


    print(
        f"\n{dataset_name.upper()} raw TEST "
        "file candidate(s):"
    )


    for path, rows in valid:

        print(
            " ",
            path,
            "| rows:",
            len(
                rows
            )
        )


    id_sets = [
        {
            str(
                row[
                    "id"
                ]
            )
            for row in rows
        }

        for _,
        rows in valid
    ]


    if len(
        valid
    ) > 1:

        assert all(
            id_sets[
                0
            ]
            ==
            ids
            for ids in id_sets[
                1:
            ]
        ), (
            f"{dataset_name}: multiple raw TEST files "
            "have different question sets. "
            "STOP rather than guessing."
        )


    return valid[
        0
    ]


if webqsp_raw_test_16a is None:

    (
        WEBQSP_RAW_TEST_PATH_16A,
        webqsp_raw_test_16a
    ) = discover_raw_test_file_16a(
        "webqsp"
    )

else:

    WEBQSP_RAW_TEST_PATH_16A = None


if cwq_raw_test_16a is None:

    (
        CWQ_RAW_TEST_PATH_16A,
        cwq_raw_test_16a
    ) = discover_raw_test_file_16a(
        "cwq"
    )

else:

    CWQ_RAW_TEST_PATH_16A = None


# ======================================================================
# 11. TEST DATA GATES
# ======================================================================

assert webqsp_raw_test_16a is not None, (
    "WebQSP raw TEST rows were not found."
)


assert cwq_raw_test_16a is not None, (
    "CWQ raw TEST rows were not found."
)


print(
    "\nRaw TEST datasets located:"
)

print(
    " WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    " CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


# Validation/test overlap check.
web_val_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in webqsp_val_plan_rows
}


cwq_val_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in cwq_val_plan_rows
}


web_test_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in webqsp_raw_test_16a
}


cwq_test_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in cwq_raw_test_16a
}


assert not (
    web_val_ids_16a
    &
    web_test_ids_16a
), (
    "WebQSP validation / TEST overlap."
)


assert not (
    cwq_val_ids_16a
    &
    cwq_test_ids_16a
), (
    "CWQ validation / TEST overlap."
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 12. FIND THE EXACT PLANNER CALLABLE
# ======================================================================
#
# Preferred path:
#   reuse a surviving callable that the notebook used.
#
# We inspect candidate names and require exactly one compatible callable
# or recover exact functions from the persisted notebook.
# ======================================================================

def callable_source_16a(
    obj
):

    try:

        return inspect.getsource(
            obj
        )

    except Exception:

        return ""


surviving_exact_candidates = []


for name, signature in (
    global_planner_candidates
):

    obj = globals()[
        name
    ]


    source = (
        callable_source_16a(
            obj
        )
    )


    combined = (
        name.lower()
        +
        " "
        +
        source.lower()
    )


    score = sum(
        token in combined
        for token in [
            "predicted_paths",
            "generate",
            "num_return_sequences",
            "relation",
            "tokenizer",
            "planning",
        ]
    )


    if score >= 2:

        surviving_exact_candidates.append(
            {
                "name":
                    name,

                "object":
                    obj,

                "signature":
                    signature,

                "score":
                    score,
            }
        )


surviving_exact_candidates = sorted(
    surviving_exact_candidates,
    key=lambda x:
        -x[
            "score"
        ]
)


print(
    "\nHigh-confidence surviving planner callable candidate(s):"
)


for row in surviving_exact_candidates[
    :20
]:

    print(
        " ",
        row[
            "name"
        ],
        row[
            "signature"
        ],
        "score=",
        row[
            "score"
        ]
    )


# ======================================================================
# 13. RECOVER EXACT PLANNER NAMESPACE FROM PERSISTED FUNCTION SOURCES
# ======================================================================
#
# We execute ONLY function/class definitions and literal assignments from
# the high-confidence persisted planning cells.
#
# We do NOT execute notebook top-level planning loops.
# ======================================================================

planner_ns_16a = dict(
    globals()
)


for hit in planning_cell_hits:

    source = hit[
        "source"
    ]


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    safe_nodes = []


    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
                ast.Import,
                ast.ImportFrom,
            )
        ):

            safe_nodes.append(
                node
            )


        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            # Literal-only constants.
            value_node = (
                node.value
            )


            try:

                ast.literal_eval(
                    value_node
                )

            except Exception:

                continue


            safe_nodes.append(
                node
            )


    if not safe_nodes:

        continue


    safe_module = ast.Module(
        body=
            safe_nodes,

        type_ignores=[]
    )


    ast.fix_missing_locations(
        safe_module
    )


    try:

        exec(
            compile(
                safe_module,
                filename=(
                    f"<planner_cell_"
                    f"{hit['cell_idx']}>"
                ),
                mode="exec"
            ),
            planner_ns_16a
        )

    except Exception as exc:

        print(
            f"Planner definition recovery warning "
            f"for cell {hit['cell_idx']}: "
            f"{type(exc).__name__}: {exc}"
        )


# ======================================================================
# 14. IDENTIFY EXACT SINGLE-QUESTION PLANNER FUNCTION
# ======================================================================

recovered_callable_candidates = []


for name, obj in (
    planner_ns_16a.items()
):

    if not callable(
        obj
    ):

        continue


    lname = (
        str(
            name
        ).lower()
    )


    if not any(
        token
        in
        lname
        for token in [
            "plan",
            "predict",
            "generate",
            "path",
            "relation",
        ]
    ):

        continue


    try:

        sig = inspect.signature(
            obj
        )

    except Exception:

        continue


    params = set(
        sig.parameters.keys()
    )


    useful_inputs = {
        "question",
        "query",
        "instruction",
        "model",
        "tokenizer",
        "top_k",
        "num_return_sequences",
    }


    overlap = len(
        params
        &
        useful_inputs
    )


    if overlap == 0:

        continue


    try:

        source = inspect.getsource(
            obj
        )

    except Exception:

        source = ""


    score = (
        overlap
        +
        sum(
            token in source
            for token in [
                ".generate(",
                "num_return_sequences",
                "predicted_paths",
                "relation",
            ]
        )
    )


    recovered_callable_candidates.append(
        {
            "name":
                name,

            "object":
                obj,

            "signature":
                sig,

            "score":
                score,
        }
    )


recovered_callable_candidates = sorted(
    recovered_callable_candidates,
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "name"
            ],
        )
)


print(
    "\nRecovered callable candidates:"
)


for row in recovered_callable_candidates[
    :30
]:

    print(
        f"  {row['name']}"
        f"{row['signature']} "
        f"score={row['score']}"
    )


# ======================================================================
# 15. FIND EXISTING PLANNER MODEL/TOKENIZER OBJECTS
# ======================================================================

def find_model_like_16a():

    preferred = [
        "planner_model",
        "model",
        "rog_model",
        "planning_model",
    ]


    for name in preferred:

        if name not in globals():

            continue


        obj = globals()[
            name
        ]


        if hasattr(
            obj,
            "generate"
        ):

            return (
                name,
                obj
            )


    for name, obj in globals().items():

        if hasattr(
            obj,
            "generate"
        ):

            module = (
                type(
                    obj
                ).__module__
            )


            if (
                "transformers"
                in
                module
            ):

                return (
                    name,
                    obj
                )


    return (
        None,
        None
    )


def find_tokenizer_like_16a():

    preferred = [
        "planner_tokenizer",
        "tokenizer",
        "rog_tokenizer",
    ]


    for name in preferred:

        if name not in globals():

            continue


        obj = globals()[
            name
        ]


        if callable(
            obj
        ) and hasattr(
            obj,
            "decode"
        ):

            return (
                name,
                obj
            )


    for name, obj in globals().items():

        if (
            callable(
                obj
            )
            and
            hasattr(
                obj,
                "decode"
            )
        ):

            module = (
                type(
                    obj
                ).__module__
            )


            if (
                "transformers"
                in
                module
            ):

                return (
                    name,
                    obj
                )


    return (
        None,
        None
    )


(
    EXISTING_PLANNER_MODEL_NAME_16A,
    EXISTING_PLANNER_MODEL_16A
) = find_model_like_16a()


(
    EXISTING_PLANNER_TOKENIZER_NAME_16A,
    EXISTING_PLANNER_TOKENIZER_16A
) = find_tokenizer_like_16a()


print(
    "\nExisting planner model:",
    EXISTING_PLANNER_MODEL_NAME_16A
)

print(
    "Existing planner tokenizer:",
    EXISTING_PLANNER_TOKENIZER_NAME_16A
)


# ======================================================================
# 16. CHECK FOR FROZEN PLANNER CONFIG / MANIFEST ARTIFACTS
# ======================================================================

planner_manifest_candidates_16a = []


for root in [
    Path(
        "/kaggle/working"
    ),
]:

    if not root.exists():

        continue


    for path in root.rglob(
        "*.json"
    ):

        lower = str(
            path
        ).lower()


        if not any(
            token
            in
            lower
            for token in [
                "planner",
                "planning",
                "relation_plan",
            ]
        ):

            continue


        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                obj = json.load(
                    f
                )

        except Exception:

            continue


        text = json.dumps(
            obj,
            sort_keys=True
        )


        if (
            FROZEN_ROG_MODEL_ID
            in
            text
            or
            FROZEN_PLANNER_CONFIG_SHA_PREFIX
            in
            text
            or
            "top_k"
            in
            text.lower()
        ):

            planner_manifest_candidates_16a.append(
                (
                    path,
                    obj
                )
            )


print(
    "\nPlanner/config artifact candidates:"
)


for path, _ in (
    planner_manifest_candidates_16a[
        :20
    ]
):

    print(
        " ",
        path
    )


# ======================================================================
# 17. SOFTWARE-RECOVERY STATUS
# ======================================================================

RECOVERY_REPORT_16A = {
    "freeze_sha":
        FINAL_AFP_FREEZE_SHA256,

    "model_id":
        FROZEN_ROG_MODEL_ID,

    "top_k":
        FROZEN_ROG_TOP_K,

    "instruction_recovered":
        FROZEN_ROG_INSTRUCTION
        is not None,

    "instruction_sha_prefix":
        FROZEN_INSTRUCTION_SHA256,

    "planning_cell_indices":
        [
            row[
                "cell_idx"
            ]
            for row in planning_cell_hits
        ],

    "recovered_planner_functions":
        [
            {
                "name":
                    row[
                        "name"
                    ],

                "signature":
                    str(
                        row[
                            "signature"
                        ]
                    ),

                "score":
                    int(
                        row[
                            "score"
                        ]
                    ),
            }

            for row in
            recovered_callable_candidates[
                :30
            ]
        ],

    "existing_model_name":
        EXISTING_PLANNER_MODEL_NAME_16A,

    "existing_tokenizer_name":
        EXISTING_PLANNER_TOKENIZER_NAME_16A,

    "webqsp_test_questions":
        len(
            webqsp_raw_test_16a
        ),

    "cwq_test_questions":
        len(
            cwq_raw_test_16a
        ),

    "webqsp_raw_test_path":
        (
            str(
                WEBQSP_RAW_TEST_PATH_16A
            )
            if
            WEBQSP_RAW_TEST_PATH_16A
            is not None
            else
            None
        ),

    "cwq_raw_test_path":
        (
            str(
                CWQ_RAW_TEST_PATH_16A
            )
            if
            CWQ_RAW_TEST_PATH_16A
            is not None
            else
            None
        ),
}


RECOVERY_REPORT_PATH_16A = (
    PLANNING_OUT_DIR
    / "cell16a_planner_recovery_report.json"
)


with open(
    RECOVERY_REPORT_PATH_16A,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RECOVERY_REPORT_16A,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 18. DO NOT GENERATE UNTIL RECOVERY IS UNAMBIGUOUS
# ======================================================================
#
# This is an intentional gate.
#
# The output from this cell tells us EXACTLY:
#
#   - which persisted planning function exists
#   - its signature
#   - whether the exact instruction was recovered
#   - whether model/tokenizer survived
#   - where the raw TEST data lives
#
# We will then invoke THAT exact function.
#
# This is safer than guessing the RoG generation wrapper after freeze.
# ======================================================================

print(
    "\n"
    + "=" * 122
)

print(
    "CELL 16A RECOVERY REPORT"
)

print(
    "=" * 122
)


print(
    "\nExact frozen instruction recovered:",
    FROZEN_ROG_INSTRUCTION
    is not None
)


if (
    FROZEN_ROG_INSTRUCTION
    is not None
):

    print(
        "Instruction SHA256:",
        sha256_text_16a(
            FROZEN_ROG_INSTRUCTION
        )
    )


print(
    "\nRaw TEST data:"
)

print(
    " WebQSP questions:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    " CWQ questions:",
    len(
        cwq_raw_test_16a
    )
)


print(
    "\nRecovered exact planner-callable candidates:"
)


for row in recovered_callable_candidates[
    :15
]:

    print(
        " ",
        row[
            "name"
        ],
        row[
            "signature"
        ],
        "score=",
        row[
            "score"
        ]
    )


print(
    "\nExisting model:",
    EXISTING_PLANNER_MODEL_NAME_16A
)

print(
    "Existing tokenizer:",
    EXISTING_PLANNER_TOKENIZER_NAME_16A
)


print(
    "\nRecovery report saved:"
)

print(
    " ",
    RECOVERY_REPORT_PATH_16A
)


print(
    "\nSTATUS:"
)

print(
    "  TEST planning generation has NOT started."
)

print(
    "  TEST gold has NOT been used by the planner."
)

print(
    "  AFP freeze remains unchanged."
)


print(
    "\nNEXT ACTION:"
)

print(
    "Send me this Cell 16A output."
)

print(
    "I will map the recovered exact planner callable "
    "to the TEST rows and give you the short materialization "
    "cell, then you rerun Cell 16 unchanged."
)

Cell 16A freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen planner identity:
 model: rmanluo/RoG
 Top-K: 3
 instruction SHA prefix: e3687b4a5081c22c
 planner config SHA prefix: 2c36bd

Persisted notebook loaded: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb

Candidate persisted planning cells:
  cell 139: score=4
  cell 21: score=2
  cell 27: score=2
  cell 47: score=2
  cell 49: score=2
  cell 63: score=2
  cell 65: score=2
  cell 83: score=2
  cell 87: score=2
  cell 105: score=2

Frozen instruction literal matches: 0

Recovered planner-related function candidates:
  cell 139: run_planning_checkpointed (score=3)
  cell 87: run_retrieval (score=2)

Surviving planner-like globals:
  select_rog(candidate_count)
  extract_plan_container(row)
  plan_is_empty(plan)
  audit_plan_rows(rows, dataset_name)
  discover_plan_files(dataset_name)
  restore_validation_plan_rows(dataset_name, existing_global_names)
  locate_r

AssertionError: WebQSP raw TEST rows were not found.

In [36]:
# ======================================================================
# PURPOSE
# -------
# Fix Cell 16A failure:
#
#   WebQSP raw TEST rows were not found
#
# We now load the OFFICIAL RoG Hugging Face datasets directly:
#
#   rmanluo/RoG-webqsp
#   rmanluo/RoG-cwq
#
# Then:
#   1. verify validation identity against frozen validation planning rows
#   2. expose exact TEST rows
#   3. recover run_planning_checkpointed from persisted notebook cell 139
#   4. inspect exact signature/source dependencies
#
# THIS CELL DOES NOT GENERATE TEST PLANS YET.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset


# ======================================================================
# 1. HARD FREEZE GATE
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing frozen prerequisites:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AR = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AR
)


print("Cell 16A-R freeze gate: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. OFFICIAL FROZEN DATASET IDENTITIES
# ======================================================================

OFFICIAL_ROG_DATASETS_16AR = {
    "webqsp":
        "rmanluo/RoG-webqsp",

    "cwq":
        "rmanluo/RoG-cwq",
}


EXPECTED_SPLIT_COUNTS_16AR = {
    "webqsp": {
        "train": 2826,
        "validation": 246,
        "test": 1628,
    },

    "cwq": {
        "train": 27639,
        "validation": 3519,
        "test": 3531,
    },
}


print(
    "\nFrozen official data sources:"
)

for dataset_name, repo in (
    OFFICIAL_ROG_DATASETS_16AR.items()
):

    print(
        f"  {dataset_name}: {repo}"
    )


# ======================================================================
# 3. LOAD OFFICIAL VALIDATION + TEST SPLITS
# ======================================================================
#
# Validation is loaded deliberately so that we can prove that the
# official HF source is the SAME source used by our frozen development
# pipeline before opening TEST results.
#
# No TEST labels are used for any model selection.
# ======================================================================

def load_official_split_16ar(
    dataset_name,
    split
):

    repo = (
        OFFICIAL_ROG_DATASETS_16AR[
            dataset_name
        ]
    )


    print(
        f"\nLoading {repo} [{split}] ..."
    )


    ds = load_dataset(
        repo,
        split=split
    )


    expected = (
        EXPECTED_SPLIT_COUNTS_16AR[
            dataset_name
        ][
            split
        ]
    )


    assert len(
        ds
    ) == expected, (
        f"{dataset_name} {split} count mismatch: "
        f"{len(ds)} != {expected}"
    )


    required_fields = {
        "id",
        "question",
        "q_entity",
        "a_entity",
        "graph",
    }


    assert required_fields.issubset(
        set(
            ds.column_names
        )
    ), (
        f"{dataset_name} {split}: missing expected fields. "
        f"Found {ds.column_names}"
    )


    print(
        f"{dataset_name.upper()} {split}: "
        f"{len(ds)} rows"
    )


    return ds


webqsp_official_val_16ar = (
    load_official_split_16ar(
        "webqsp",
        "validation"
    )
)


cwq_official_val_16ar = (
    load_official_split_16ar(
        "cwq",
        "validation"
    )
)


webqsp_official_test_16ar = (
    load_official_split_16ar(
        "webqsp",
        "test"
    )
)


cwq_official_test_16ar = (
    load_official_split_16ar(
        "cwq",
        "test"
    )
)


# ======================================================================
# 4. NORMALIZATION HELPERS FOR SOURCE-IDENTITY GATE
# ======================================================================

def norm_sequence_16ar(
    value
):

    if value is None:

        return []


    if isinstance(
        value,
        np.ndarray
    ):

        value = value.tolist()


    if isinstance(
        value,
        tuple
    ):

        value = list(
            value
        )


    if not isinstance(
        value,
        list
    ):

        value = [
            value
        ]


    return [
        str(
            x
        )
        for x in value
    ]


def canonical_graph_16ar(
    graph
):

    if isinstance(
        graph,
        np.ndarray
    ):

        graph = graph.tolist()


    return [
        [
            str(
                x
            )
            for x in triplet
        ]
        for triplet in graph
    ]


def graph_sha_16ar(
    graph
):

    canonical = json.dumps(
        canonical_graph_16ar(
            graph
        ),
        ensure_ascii=False,
        separators=(",", ":")
    )


    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


# ======================================================================
# 5. VALIDATION DATA-SOURCE IDENTITY GATE
# ======================================================================
#
# This is important:
#
# We do NOT merely assume the HF repo is the same dataset source.
#
# We verify:
#   - all IDs
#   - exact questions
#   - q_entity
#   - a_entity
#
# and exact graph equality on deterministic samples.
# ======================================================================

def validate_source_identity_16ar(
    dataset_name,
    official_val,
    frozen_val_rows
):

    official_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in official_val
    }


    frozen_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in frozen_val_rows
    }


    assert set(
        official_by_id
    ) == set(
        frozen_by_id
    ), (
        f"{dataset_name}: validation ID sets differ "
        "between official HF source and frozen plans."
    )


    mismatch_question = 0
    mismatch_q_entity = 0
    mismatch_a_entity = 0


    sorted_ids = sorted(
        frozen_by_id.keys()
    )


    for qid in sorted_ids:

        hf = official_by_id[
            qid
        ]


        frozen = frozen_by_id[
            qid
        ]


        if str(
            hf[
                "question"
            ]
        ) != str(
            frozen[
                "question"
            ]
        ):

            mismatch_question += 1


        if norm_sequence_16ar(
            hf[
                "q_entity"
            ]
        ) != norm_sequence_16ar(
            frozen[
                "q_entity"
            ]
        ):

            mismatch_q_entity += 1


        if norm_sequence_16ar(
            hf[
                "a_entity"
            ]
        ) != norm_sequence_16ar(
            frozen[
                "a_entity"
            ]
        ):

            mismatch_a_entity += 1


    assert mismatch_question == 0
    assert mismatch_q_entity == 0
    assert mismatch_a_entity == 0


    # --------------------------------------------------------------
    # Deterministic graph equality sample.
    #
    # Full CWQ validation graph serialization is very large, so exact
    # graph equality is checked at deterministic spread-out positions.
    # --------------------------------------------------------------

    n = len(
        sorted_ids
    )


    sample_positions = sorted(
        set(
            [
                0,
                1,
                n // 10,
                n // 4,
                n // 2,
                (3 * n) // 4,
                (9 * n) // 10,
                n - 2,
                n - 1,
            ]
        )
    )


    graph_checks = 0


    for pos in sample_positions:

        qid = sorted_ids[
            pos
        ]


        hf_graph = official_by_id[
            qid
        ][
            "graph"
        ]


        frozen_graph = frozen_by_id[
            qid
        ][
            "graph"
        ]


        hf_sha = graph_sha_16ar(
            hf_graph
        )


        frozen_sha = graph_sha_16ar(
            frozen_graph
        )


        assert hf_sha == frozen_sha, (
            f"{dataset_name}: graph mismatch for "
            f"validation qid={qid}"
        )


        graph_checks += 1


    print(
        f"{dataset_name.upper()} official-source "
        "validation identity: PASSED"
    )

    print(
        "  IDs:",
        len(
            sorted_ids
        )
    )

    print(
        "  question/entity mismatches: 0"
    )

    print(
        "  exact graph samples:",
        graph_checks,
        "/",
        graph_checks
    )


validate_source_identity_16ar(
    "webqsp",
    webqsp_official_val_16ar,
    webqsp_val_plan_rows
)


validate_source_identity_16ar(
    "cwq",
    cwq_official_val_16ar,
    cwq_val_plan_rows
)


print(
    "\nOfficial RoG dataset-source identity gate: PASSED"
)


# ======================================================================
# 6. MATERIALIZE RAW TEST ROWS IN NOTEBOOK MEMORY
# ======================================================================

def dataset_to_rows_16ar(
    ds
):

    rows = []


    for row in ds:

        rows.append(
            {
                "id":
                    str(
                        row[
                            "id"
                        ]
                    ),

                "question":
                    str(
                        row[
                            "question"
                        ]
                    ),

                "answer":
                    norm_sequence_16ar(
                        row.get(
                            "answer",
                            []
                        )
                    ),

                "q_entity":
                    norm_sequence_16ar(
                        row[
                            "q_entity"
                        ]
                    ),

                "a_entity":
                    norm_sequence_16ar(
                        row[
                            "a_entity"
                        ]
                    ),

                "graph":
                    canonical_graph_16ar(
                        row[
                            "graph"
                        ]
                    ),

                "choices":
                    row.get(
                        "choices",
                        []
                    ),
            }
        )


    return rows


webqsp_raw_test_16ar = (
    dataset_to_rows_16ar(
        webqsp_official_test_16ar
    )
)


cwq_raw_test_16ar = (
    dataset_to_rows_16ar(
        cwq_official_test_16ar
    )
)


assert len(
    webqsp_raw_test_16ar
) == 1628


assert len(
    cwq_raw_test_16ar
) == 3531


# Expose standard names for later planner/materialization cells.
webqsp_raw_test_16a = (
    webqsp_raw_test_16ar
)

cwq_raw_test_16a = (
    cwq_raw_test_16ar
)


print(
    "\nRaw TEST rows materialized:"
)

print(
    "  WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    "  CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


# ======================================================================
# 7. TEST / VALIDATION DISJOINTNESS
# ======================================================================

def ids_16ar(
    rows
):

    return {
        str(
            row[
                "id"
            ]
        )
        for row in rows
    }


assert not (
    ids_16ar(
        webqsp_raw_test_16a
    )
    &
    ids_16ar(
        webqsp_val_plan_rows
    )
)


assert not (
    ids_16ar(
        cwq_raw_test_16a
    )
    &
    ids_16ar(
        cwq_val_plan_rows
    )
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 8. LOAD PERSISTED NOTEBOOK
# ======================================================================

NOTEBOOK_PATH_16AR = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)


assert NOTEBOOK_PATH_16AR.exists()


with open(
    NOTEBOOK_PATH_16AR,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16ar = (
        json.load(
            f
        )
    )


# ======================================================================
# 9. EXTRACT CELL 139 EXACT SOURCE
# ======================================================================

PLANNER_CELL_INDEX_16AR = 139


assert (
    PLANNER_CELL_INDEX_16AR
    <
    len(
        persisted_nb_16ar[
            "cells"
        ]
    )
)


planner_cell_139_16ar = (
    persisted_nb_16ar[
        "cells"
    ][
        PLANNER_CELL_INDEX_16AR
    ]
)


assert (
    planner_cell_139_16ar.get(
        "cell_type"
    )
    ==
    "code"
)


planner_cell_source_16ar = "".join(
    planner_cell_139_16ar.get(
        "source",
        []
    )
)


assert (
    "run_planning_checkpointed"
    in
    planner_cell_source_16ar
), (
    "Cell 139 no longer contains "
    "run_planning_checkpointed."
)


print(
    "\nPersisted planner cell 139 recovered: PASSED"
)


# ======================================================================
# 10. EXTRACT EXACT run_planning_checkpointed SOURCE
# ======================================================================

tree_16ar = ast.parse(
    planner_cell_source_16ar
)


run_planning_source_16ar = None


for node in tree_16ar.body:

    if (
        isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
        and
        node.name
        ==
        "run_planning_checkpointed"
    ):

        run_planning_source_16ar = (
            ast.get_source_segment(
                planner_cell_source_16ar,
                node
            )
        )

        break


assert run_planning_source_16ar is not None


print(
    "Exact run_planning_checkpointed "
    "function source recovered: PASSED"
)


# ======================================================================
# 11. RECOVER ALL SAFE DEFINITIONS FROM CELL 139
# ======================================================================
#
# Only definitions/imports/literal assignments are executed.
# Top-level planner loops are NOT executed.
# ======================================================================

planner_ns_16ar = dict(
    globals()
)


safe_nodes_16ar = []


for node in tree_16ar.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
            ast.Import,
            ast.ImportFrom,
        )
    ):

        safe_nodes_16ar.append(
            node
        )


    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):

        value_node = node.value


        if value_node is None:

            continue


        try:

            ast.literal_eval(
                value_node
            )

        except Exception:

            continue


        safe_nodes_16ar.append(
            node
        )


safe_module_16ar = ast.Module(
    body=
        safe_nodes_16ar,

    type_ignores=[]
)


ast.fix_missing_locations(
    safe_module_16ar
)


exec(
    compile(
        safe_module_16ar,
        filename="<persisted_planner_cell_139>",
        mode="exec"
    ),
    planner_ns_16ar
)


assert (
    "run_planning_checkpointed"
    in
    planner_ns_16ar
)


RUN_PLANNING_CHECKPOINTED_16AR = (
    planner_ns_16ar[
        "run_planning_checkpointed"
    ]
)


# ======================================================================
# 12. EXACT SIGNATURE
# ======================================================================

RUN_PLANNING_SIGNATURE_16AR = (
    inspect.signature(
        RUN_PLANNING_CHECKPOINTED_16AR
    )
)


print(
    "\nExact run_planning_checkpointed signature:"
)

print(
    " ",
    RUN_PLANNING_SIGNATURE_16AR
)


# ======================================================================
# 13. SOURCE DEPENDENCY SCAN
# ======================================================================

function_tree_16ar = ast.parse(
    run_planning_source_16ar
)


called_names_16ar = set()


for node in ast.walk(
    function_tree_16ar
):

    if isinstance(
        node,
        ast.Call
    ):

        if isinstance(
            node.func,
            ast.Name
        ):

            called_names_16ar.add(
                node.func.id
            )


defined_or_builtin_16ar = {
    "len",
    "str",
    "int",
    "float",
    "bool",
    "list",
    "dict",
    "set",
    "tuple",
    "enumerate",
    "range",
    "print",
    "open",
    "min",
    "max",
    "sum",
    "sorted",
}


external_called_names_16ar = sorted(
    name
    for name in called_names_16ar

    if (
        name
        not in
        defined_or_builtin_16ar
    )
)


print(
    "\nDirect function dependencies referenced by "
    "run_planning_checkpointed:"
)


for name in external_called_names_16ar:

    status = (
        "AVAILABLE"
        if
        name
        in
        planner_ns_16ar
        else
        "MISSING"
    )

    print(
        f"  {name:<40} {status}"
    )


# ======================================================================
# 14. PRINT EXACT FUNCTION SOURCE
# ======================================================================
#
# This is deliberately shown now because it is the final point before
# TEST planner generation.
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "EXACT PERSISTED run_planning_checkpointed SOURCE"
)

print(
    "=" * 120
)


print(
    run_planning_source_16ar
)


# ======================================================================
# 15. SAVE RAW TEST SNAPSHOT + RECOVERY REPORT
# ======================================================================

RECOVERY_DIR_16AR = Path(
    "/kaggle/working/"
    "step2_rq1_test/"
    "planner_recovery"
)


RECOVERY_DIR_16AR.mkdir(
    parents=True,
    exist_ok=True
)


def sha_json_rows_16ar(
    rows
):

    h = hashlib.sha256()


    for row in rows:

        # Avoid one giant in-memory string.
        text = json.dumps(
            row,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False
        )


        h.update(
            text.encode(
                "utf-8"
            )
        )

        h.update(
            b"\n"
        )


    return h.hexdigest()


WEBQSP_RAW_TEST_SHA_16AR = (
    sha_json_rows_16ar(
        webqsp_raw_test_16a
    )
)


CWQ_RAW_TEST_SHA_16AR = (
    sha_json_rows_16ar(
        cwq_raw_test_16a
    )
)


CELL16AR_REPORT = {
    "freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "official_dataset_sources": {
        "webqsp":
            "rmanluo/RoG-webqsp",

        "cwq":
            "rmanluo/RoG-cwq",
    },

    "counts": {
        "webqsp_test":
            len(
                webqsp_raw_test_16a
            ),

        "cwq_test":
            len(
                cwq_raw_test_16a
            ),
    },

    "raw_test_sha256": {
        "webqsp":
            WEBQSP_RAW_TEST_SHA_16AR,

        "cwq":
            CWQ_RAW_TEST_SHA_16AR,
    },

    "validation_source_identity":
        True,

    "validation_test_disjoint":
        True,

    "planner_cell_index":
        139,

    "planner_function":
        "run_planning_checkpointed",

    "planner_signature":
        str(
            RUN_PLANNING_SIGNATURE_16AR
        ),

    "direct_dependencies":
        external_called_names_16ar,

    "test_planning_started":
        False,

    "test_used_for_tuning":
        False,
}


CELL16AR_REPORT_PATH = (
    RECOVERY_DIR_16AR
    / "cell16ar_exact_dataset_and_planner_recovery.json"
)


with open(
    CELL16AR_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL16AR_REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AR_COMPLETE = True


# ======================================================================
# 16. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "=== CELL 16A-R: RAW TEST + EXACT PLANNER RECOVERY COMPLETE ==="
)

print(
    "=" * 124
)


print(
    "\nOfficial source identity:"
)

print(
    "  WebQSP validation match: PASSED"
)

print(
    "  CWQ validation match:    PASSED"
)


print(
    "\nRaw TEST:"
)

print(
    "  WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    "  CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


print(
    "\nPlanner:"
)

print(
    "  persisted cell: 139"
)

print(
    "  callable: run_planning_checkpointed"
)

print(
    "  signature:",
    RUN_PLANNING_SIGNATURE_16AR
)


print(
    "\nTEST planner generation started: NO"
)

print(
    "AFP freeze changed: NO"
)


print(
    "\nReport:"
)

print(
    " ",
    CELL16AR_REPORT_PATH
)


print(
    "\nNEXT:"
)

print(
    "Use the exact signature/source printed above "
    "to materialize frozen TEST relation plans."
)

Cell 16A-R freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen official data sources:
  webqsp: rmanluo/RoG-webqsp
  cwq: rmanluo/RoG-cwq

Loading rmanluo/RoG-webqsp [validation] ...


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

CWQ validation: 3519 rows

Loading rmanluo/RoG-webqsp [test] ...
WEBQSP test: 1628 rows

Loading rmanluo/RoG-cwq [test] ...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

CWQ test: 3531 rows
WEBQSP official-source validation identity: PASSED
  IDs: 246
  question/entity mismatches: 0
  exact graph samples: 9 / 9
CWQ official-source validation identity: PASSED
  IDs: 3519
  question/entity mismatches: 0
  exact graph samples: 9 / 9

Official RoG dataset-source identity gate: PASSED

Raw TEST rows materialized:
  WebQSP: 1628
  CWQ:    3531
Validation / TEST disjointness: PASSED

Persisted planner cell 139 recovered: PASSED
Exact run_planning_checkpointed function source recovered: PASSED

Exact run_planning_checkpointed signature:
  (dataset_split, ckpt_path, desc)

Direct function dependencies referenced by run_planning_checkpointed:
  generate_seq                             MISSING
  load_checkpoint                          MISSING
  parse_prediction                         MISSING
  tqdm                                     AVAILABLE

EXACT PERSISTED run_planning_checkpointed SOURCE
def run_planning_checkpointed(dataset_split, ckpt_path, desc):
    do

In [37]:
# ======================================================================
# CELL 16A-MATERIALIZE
# EXACT FROZEN RoG TEST RELATION-PLAN MATERIALIZATION
# ======================================================================
#
# RUN AS A NEW CELL AFTER CELL 16A-R.
#
# DO NOT:
#   - restart kernel
#   - replace Cell 16
#   - rerun development cells
#   - modify AFP
#   - tune using TEST
#
# THIS CELL:
#   1. recovers exact persisted planner dependencies
#   2. recovers exact INSTRUCTION / N_BEAM / prompter
#   3. recovers or restores exact rmanluo/RoG model + tokenizer
#   4. verifies planner software/configuration identity
#   5. generates TEST plans using the EXACT recovered wrapper:
#
#          run_planning_checkpointed(
#              dataset_split,
#              ckpt_path,
#              desc
#          )
#
#   6. saves checkpointed:
#
#      /kaggle/working/step2_rq1_test/
#          planning_webqsp_test.jsonl
#
#      /kaggle/working/step2_rq1_test/
#          planning_cwq_test.jsonl
#
#   7. exposes:
#
#      webqsp_test_plan_rows
#      cwq_test_plan_rows
#
# After this cell completes successfully:
#
#      RERUN CELL 16 UNCHANGED.
#
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import builtins
import hashlib
import inspect
import json
import math
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE + CELL 16A-R GATES
# ======================================================================

required = [
    "CELL16AR_COMPLETE",

    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "persisted_nb_16ar",

    "run_planning_source_16ar",
    "planner_cell_source_16ar",

    "RUN_PLANNING_SIGNATURE_16AR",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell 16A-R / frozen prerequisites:\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AM = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AM
), (
    "AFP freeze SHA mismatch. STOP."
)


print(
    "Cell 16A-MATERIALIZE freeze gate: PASSED"
)

print(
    "Freeze SHA:",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 2. FROZEN PLANNER FACTS
# ======================================================================

FROZEN_MODEL_ID_16AM = (
    "rmanluo/RoG"
)

FROZEN_TOP_K_16AM = 3

FROZEN_INSTRUCTION_SHA_PREFIX_16AM = (
    "e3687b4a5081c22c"
)

FROZEN_PLANNER_CONFIG_SHA_PREFIX_16AM = (
    "2c36bd"
)


assert (
    "do_sample=True"
    in
    run_planning_source_16ar
)

assert (
    "max_new_tokens=100"
    in
    run_planning_source_16ar
)

assert (
    "num_beam=N_BEAM"
    in
    run_planning_source_16ar
)


print(
    "\nFrozen wrapper constants:"
)

print(
    "  model:",
    FROZEN_MODEL_ID_16AM
)

print(
    "  beam:",
    FROZEN_TOP_K_16AM
)

print(
    "  do_sample: True"
)

print(
    "  max_new_tokens: 100"
)


# ======================================================================
# 3. OUTPUT PATHS
# ======================================================================

PLANNING_OUT_DIR_16AM = Path(
    "/kaggle/working/step2_rq1_test"
)

PLANNING_OUT_DIR_16AM.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR_16AM
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR_16AM
    / "planning_cwq_test.jsonl"
)


MATERIALIZE_MANIFEST_PATH_16AM = (
    PLANNING_OUT_DIR_16AM
    / "cell16a_test_plan_materialization_manifest.json"
)


# ======================================================================
# 4. SHA HELPERS
# ======================================================================

def sha256_text_16am(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def sha256_file_16am(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================
# 5. AST HELPERS
# ======================================================================

PLANNER_MAX_CELL_16AM = 139


def target_names_16am(
    target
):

    names = []


    if isinstance(
        target,
        ast.Name
    ):

        names.append(
            target.id
        )


    elif isinstance(
        target,
        (
            ast.Tuple,
            ast.List,
        )
    ):

        for element in target.elts:

            names.extend(
                target_names_16am(
                    element
                )
            )


    return names


def assignment_target_names_16am(
    node
):

    names = []


    if isinstance(
        node,
        ast.Assign
    ):

        for target in node.targets:

            names.extend(
                target_names_16am(
                    target
                )
            )


    elif isinstance(
        node,
        ast.AnnAssign
    ):

        names.extend(
            target_names_16am(
                node.target
            )
        )


    return names


# ======================================================================
# 6. BUILD INDEX OF TOP-LEVEL NOTEBOOK DEFINITIONS
# ======================================================================

NOTEBOOK_INDEX_16AM = {
    "functions":
        {},

    "classes":
        {},

    "assignments":
        {},

    "imports":
        [],

    "seed_calls":
        [],
}


for cell_idx in range(
    min(
        PLANNER_MAX_CELL_16AM + 1,
        len(
            persisted_nb_16ar[
                "cells"
            ]
        )
    )
):

    cell = (
        persisted_nb_16ar[
            "cells"
        ][
            cell_idx
        ]
    )


    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        segment = (
            ast.get_source_segment(
                source,
                node
            )
        )


        if not segment:

            continue


        # --------------------------------------------------------------
        # Imports
        # --------------------------------------------------------------

        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
            )
        ):

            NOTEBOOK_INDEX_16AM[
                "imports"
            ].append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Functions
        # --------------------------------------------------------------

        elif isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            NOTEBOOK_INDEX_16AM[
                "functions"
            ].setdefault(
                node.name,
                []
            ).append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Classes
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.ClassDef
        ):

            NOTEBOOK_INDEX_16AM[
                "classes"
            ].setdefault(
                node.name,
                []
            ).append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Assignments
        # --------------------------------------------------------------

        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            names = (
                assignment_target_names_16am(
                    node
                )
            )


            for name in names:

                NOTEBOOK_INDEX_16AM[
                    "assignments"
                ].setdefault(
                    name,
                    []
                ).append(
                    {
                        "cell_idx":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


        # --------------------------------------------------------------
        # Explicit seed calls
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.Expr
        ):

            lower = (
                segment.lower()
            )


            if any(
                token
                in
                lower
                for token
                in [
                    "set_seed(",
                    "manual_seed(",
                    "random.seed(",
                    "np.random.seed(",
                    "numpy.random.seed(",
                ]
            ):

                NOTEBOOK_INDEX_16AM[
                    "seed_calls"
                ].append(
                    {
                        "cell_idx":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


print(
    "\nPersisted notebook AST index: READY"
)


# ======================================================================
# 7. CREATE ISOLATED EXACT PLANNER NAMESPACE
# ======================================================================

planner_ns_16am = {
    "__builtins__":
        builtins.__dict__,

    "ast":
        ast,

    "hashlib":
        hashlib,

    "inspect":
        inspect,

    "json":
        json,

    "math":
        math,

    "os":
        os,

    "random":
        random,

    "re":
        re,

    "time":
        time,

    "Path":
        Path,

    "np":
        np,

    "numpy":
        np,

    "torch":
        torch,

    "tqdm":
        tqdm,
}


# ======================================================================
# 8. EXECUTE PERSISTED IMPORTS
# ======================================================================
#
# Import failures are recorded rather than silently treated as success.
# Many irrelevant notebook imports may legitimately fail; required
# planner imports will be checked later.
# ======================================================================

IMPORT_FAILURES_16AM = []


for row in (
    NOTEBOOK_INDEX_16AM[
        "imports"
    ]
):

    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except Exception as exc:

        IMPORT_FAILURES_16AM.append(
            {
                "cell_idx":
                    row[
                        "cell_idx"
                    ],

                "source":
                    row[
                        "source"
                    ],

                "error":
                    (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            }
        )


print(
    "Persisted imports attempted."
)

print(
    "Non-critical import failures:",
    len(
        IMPORT_FAILURES_16AM
    )
)


# ======================================================================
# 9. EXECUTE ALL PERSISTED CLASS DEFINITIONS
# ======================================================================

for class_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "classes"
    ].items()
):

    # Latest persisted definition before planner cell.
    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    try:

        exec(
            (
                "from __future__ import annotations\n"
                +
                row[
                    "source"
                ]
            ),
            planner_ns_16am
        )

    except Exception:

        # Not every notebook class belongs to planning.
        pass


# ======================================================================
# 10. EXECUTE ALL PERSISTED FUNCTION DEFINITIONS
# ======================================================================
#
# Definitions have no planning-loop side effect.
# ======================================================================

for function_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "functions"
    ].items()
):

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    try:

        exec(
            (
                "from __future__ import annotations\n"
                +
                row[
                    "source"
                ]
            ),
            planner_ns_16am
        )

    except Exception:

        pass


# ======================================================================
# 11. EXECUTE SAFE LITERAL ASSIGNMENTS
# ======================================================================

for target_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "assignments"
    ].items()
):

    for row in sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    ):

        source = row[
            "source"
        ]


        try:

            tree = ast.parse(
                source
            )


            node = tree.body[
                0
            ]


            value_node = node.value


            ast.literal_eval(
                value_node
            )


            exec(
                source,
                planner_ns_16am
            )


        except Exception:

            pass


# ======================================================================
# 12. FORCE EXACT REQUIRED FUNCTION DEFINITIONS
# ======================================================================

REQUIRED_FUNCTIONS_16AM = [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
    "run_planning_checkpointed",
]


def recover_latest_function_16am(
    name
):

    rows = (
        NOTEBOOK_INDEX_16AM[
            "functions"
        ].get(
            name,
            []
        )
    )


    assert rows, (
        f"Could not recover persisted function: {name}"
    )


    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    exec(
        (
            "from __future__ import annotations\n"
            +
            row[
                "source"
            ]
        ),
        planner_ns_16am
    )


    assert callable(
        planner_ns_16am[
            name
        ]
    )


    return row


RECOVERED_FUNCTION_ROWS_16AM = {}


for name in REQUIRED_FUNCTIONS_16AM:

    RECOVERED_FUNCTION_ROWS_16AM[
        name
    ] = recover_latest_function_16am(
        name
    )


print(
    "\nExact planner functions recovered:"
)


for name in REQUIRED_FUNCTIONS_16AM:

    row = (
        RECOVERED_FUNCTION_ROWS_16AM[
            name
        ]
    )


    print(
        f"  {name:<28} "
        f"cell={row['cell_idx']} "
        f"SHA={sha256_text_16am(row['source'])[:16]}"
    )


# ======================================================================
# 13. GENERIC ASSIGNMENT RECOVERY
# ======================================================================

def missing_name_from_error_16am(
    exc
):

    match = re.search(
        r"name '([^']+)' is not defined",
        str(
            exc
        )
    )


    if not match:

        return None


    return match.group(
        1
    )


RECOVERY_STACK_16AM = set()


def recover_assignment_target_16am(
    target_name,
    cutoff_cell=PLANNER_MAX_CELL_16AM,
    preferred_filter=None
):

    if target_name in (
        RECOVERY_STACK_16AM
    ):

        raise RuntimeError(
            f"Circular assignment recovery: "
            f"{target_name}"
        )


    rows = [
        row
        for row
        in NOTEBOOK_INDEX_16AM[
            "assignments"
        ].get(
            target_name,
            []
        )
        if row[
            "cell_idx"
        ]
        <=
        cutoff_cell
    ]


    if preferred_filter is not None:

        preferred = [
            row
            for row
            in rows
            if preferred_filter(
                row
            )
        ]


        if preferred:

            rows = preferred


    assert rows, (
        f"No persisted assignment found for "
        f"{target_name}"
    )


    # Latest matching assignment first.
    rows = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ],
        reverse=True
    )


    last_error = None


    RECOVERY_STACK_16AM.add(
        target_name
    )


    try:

        for row in rows:

            source = row[
                "source"
            ]


            for attempt in range(
                12
            ):

                try:

                    exec(
                        source,
                        planner_ns_16am
                    )


                    if target_name in (
                        planner_ns_16am
                    ):

                        return row


                    break


                except NameError as exc:

                    missing_name = (
                        missing_name_from_error_16am(
                            exc
                        )
                    )


                    if not missing_name:

                        last_error = exc
                        break


                    # Function / class definition?
                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "functions"
                        ]
                    ):

                        recover_latest_function_16am(
                            missing_name
                        )

                        continue


                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "classes"
                        ]
                    ):

                        class_row = sorted(
                            NOTEBOOK_INDEX_16AM[
                                "classes"
                            ][
                                missing_name
                            ],
                            key=lambda x:
                                x[
                                    "cell_idx"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            (
                                "from __future__ "
                                "import annotations\n"
                                +
                                class_row[
                                    "source"
                                ]
                            ),
                            planner_ns_16am
                        )

                        continue


                    # Recover exact persisted assignment.
                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "assignments"
                        ]
                    ):

                        recover_assignment_target_16am(
                            missing_name,
                            cutoff_cell=
                                row[
                                    "cell_idx"
                                ]
                        )

                        continue


                    last_error = exc
                    break


                except Exception as exc:

                    last_error = exc
                    break


    finally:

        RECOVERY_STACK_16AM.remove(
            target_name
        )


    raise AssertionError(
        f"Failed to recover assignment for "
        f"{target_name}.\n"
        f"Last error: "
        f"{type(last_error).__name__ if last_error else 'unknown'}: "
        f"{last_error}"
    )


# ======================================================================
# 14. RECOVER EXACT INSTRUCTION
# ======================================================================

instruction_rows = (
    NOTEBOOK_INDEX_16AM[
        "assignments"
    ].get(
        "INSTRUCTION",
        []
    )
)


assert instruction_rows, (
    "No persisted INSTRUCTION assignment found."
)


INSTRUCTION_RECOVERY_ROW_16AM = None


for row in sorted(
    instruction_rows,
    key=lambda x:
        x[
            "cell_idx"
        ],
    reverse=True
):

    # Work in current exact namespace.
    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except NameError:

        try:

            recover_assignment_target_16am(
                "INSTRUCTION",
                cutoff_cell=
                    row[
                        "cell_idx"
                ]
            )

        except Exception:

            continue


    except Exception:

        continue


    if (
        "INSTRUCTION"
        not in
        planner_ns_16am
    ):

        continue


    value = planner_ns_16am[
        "INSTRUCTION"
    ]


    if not isinstance(
        value,
        str
    ):

        continue


    digest = sha256_text_16am(
        value
    )


    if digest.startswith(
        FROZEN_INSTRUCTION_SHA_PREFIX_16AM
    ):

        INSTRUCTION_RECOVERY_ROW_16AM = row

        break


assert (
    INSTRUCTION_RECOVERY_ROW_16AM
    is not None
), (
    "Exact frozen INSTRUCTION could not be recovered "
    "with expected SHA prefix."
)


INSTRUCTION_16AM = (
    planner_ns_16am[
        "INSTRUCTION"
    ]
)


INSTRUCTION_SHA_16AM = (
    sha256_text_16am(
        INSTRUCTION_16AM
    )
)


print(
    "\nExact frozen instruction: PASSED"
)

print(
    "  cell:",
    INSTRUCTION_RECOVERY_ROW_16AM[
        "cell_idx"
    ]
)

print(
    "  SHA256:",
    INSTRUCTION_SHA_16AM
)


# ======================================================================
# 15. RECOVER EXACT N_BEAM
# ======================================================================

N_BEAM_RECOVERY_ROW_16AM = (
    recover_assignment_target_16am(
        "N_BEAM"
    )
)


N_BEAM_16AM = int(
    planner_ns_16am[
        "N_BEAM"
    ]
)


assert (
    N_BEAM_16AM
    ==
    FROZEN_TOP_K_16AM
), (
    f"N_BEAM mismatch: "
    f"{N_BEAM_16AM} != {FROZEN_TOP_K_16AM}"
)


print(
    "\nExact N_BEAM: PASSED"
)

print(
    "  N_BEAM =",
    N_BEAM_16AM
)


# ======================================================================
# 16. RECOVER EXACT PROMPTER
# ======================================================================

PROMPTER_RECOVERY_ROW_16AM = (
    recover_assignment_target_16am(
        "prompter"
    )
)


prompter_16am = (
    planner_ns_16am[
        "prompter"
    ]
)


assert hasattr(
    prompter_16am,
    "format"
), (
    "Recovered prompter lacks format()."
)


# Verify prompt creation on frozen validation text.
validation_prompt_probe_16am = (
    prompter_16am.format(
        instruction=
            INSTRUCTION_16AM,

        message=
            str(
                webqsp_val_plan_rows[
                    0
                ][
                    "question"
                ]
            )
    )
)


assert isinstance(
    validation_prompt_probe_16am,
    str
)

assert len(
    validation_prompt_probe_16am
) > 0


PROMPT_PROBE_SHA_16AM = (
    sha256_text_16am(
        validation_prompt_probe_16am
    )
)


print(
    "\nExact prompter recovered: PASSED"
)

print(
    "  assignment cell:",
    PROMPTER_RECOVERY_ROW_16AM[
        "cell_idx"
    ]
)

print(
    "  validation prompt probe SHA:",
    PROMPT_PROBE_SHA_16AM
)


# ======================================================================
# 17. MODEL-ID EVIDENCE HELPER
# ======================================================================

def assignment_references_model_id_16am(
    row
):

    source = row[
        "source"
    ]


    if (
        FROZEN_MODEL_ID_16AM.lower()
        in
        source.lower()
    ):

        return True


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        return False


    names = {
        node.id
        for node in ast.walk(
            tree
        )
        if isinstance(
            node,
            ast.Name
        )
    }


    for name in names:

        if name not in (
            planner_ns_16am
        ):

            continue


        value = planner_ns_16am[
            name
        ]


        if (
            isinstance(
                value,
                str
            )
            and
            FROZEN_MODEL_ID_16AM.lower()
            in
            value.lower()
        ):

            return True


    return False


# ======================================================================
# 18. RUNTIME MODEL/TOKENIZER IDENTITY
# ======================================================================

def runtime_identity_strings_16am(
    obj
):

    values = []


    for attr in [
        "name_or_path",
        "_name_or_path",
    ]:

        value = getattr(
            obj,
            attr,
            None
        )


        if isinstance(
            value,
            str
        ):

            values.append(
                value
            )


    config = getattr(
        obj,
        "config",
        None
    )


    if config is not None:

        for attr in [
            "name_or_path",
            "_name_or_path",
        ]:

            value = getattr(
                config,
                attr,
                None
            )


            if isinstance(
                value,
                str
            ):

                values.append(
                    value
                )


    return list(
        dict.fromkeys(
            values
        )
    )


def runtime_matches_model_id_16am(
    obj
):

    return any(
        FROZEN_MODEL_ID_16AM.lower()
        in
        value.lower()

        for value in
        runtime_identity_strings_16am(
            obj
        )
    )


# ======================================================================
# 19. USE SURVIVING EXACT MODEL/TOKENIZER IF AVAILABLE
# ======================================================================

SURVIVING_MODEL_16AM = None
SURVIVING_TOKENIZER_16AM = None


# Preferred exact global names.
for name in [
    "model",
    "planner_model",
    "rog_model",
    "planning_model",
]:

    if name not in globals():

        continue


    obj = globals()[
        name
    ]


    if (
        hasattr(
            obj,
            "generate"
        )
        and
        runtime_matches_model_id_16am(
            obj
        )
    ):

        SURVIVING_MODEL_16AM = obj

        print(
            "\nUsing surviving exact planner model:",
            name
        )

        break


for name in [
    "tokenizer",
    "planner_tokenizer",
    "rog_tokenizer",
]:

    if name not in globals():

        continue


    obj = globals()[
        name
    ]


    if (
        callable(
            obj
        )
        and
        hasattr(
            obj,
            "decode"
        )
        and
        runtime_matches_model_id_16am(
            obj
        )
    ):

        SURVIVING_TOKENIZER_16AM = obj

        print(
            "Using surviving exact planner tokenizer:",
            name
        )

        break


# ======================================================================
# 20. RECOVER EXACT MODEL ASSIGNMENT IF NEEDED
# ======================================================================

MODEL_RECOVERY_ROW_16AM = None


if SURVIVING_MODEL_16AM is None:

    model_rows = (
        NOTEBOOK_INDEX_16AM[
            "assignments"
        ].get(
            "model",
            []
        )
    )


    model_evidence_rows = [
        row
        for row
        in model_rows
        if assignment_references_model_id_16am(
            row
        )
    ]


    assert model_evidence_rows, (
        "No unambiguous persisted planner-model assignment "
        "referencing rmanluo/RoG was found.\n"
        "STOP rather than loading a guessed model implementation."
    )


    MODEL_RECOVERY_ROW_16AM = (
        sorted(
            model_evidence_rows,
            key=lambda x:
                x[
                    "cell_idx"
                ],
            reverse=True
        )[
            0
        ]
    )


    # Exact source may assign model + tokenizer together.
    source = (
        MODEL_RECOVERY_ROW_16AM[
            "source"
        ]
    )


    print(
        "\nExecuting exact persisted model assignment "
        f"from cell "
        f"{MODEL_RECOVERY_ROW_16AM['cell_idx']}..."
    )


    # Recover missing assignment dependencies recursively.
    for attempt in range(
        20
    ):

        try:

            exec(
                source,
                planner_ns_16am
            )

            break


        except NameError as exc:

            missing_name = (
                missing_name_from_error_16am(
                    exc
                )
            )


            assert missing_name is not None


            if missing_name in (
                NOTEBOOK_INDEX_16AM[
                    "assignments"
                ]
            ):

                recover_assignment_target_16am(
                    missing_name,
                    cutoff_cell=
                        MODEL_RECOVERY_ROW_16AM[
                            "cell_idx"
                        ]
                )

                continue


            if missing_name in (
                NOTEBOOK_INDEX_16AM[
                    "functions"
                ]
            ):

                recover_latest_function_16am(
                    missing_name
                )

                continue


            raise


    assert (
        "model"
        in
        planner_ns_16am
    )


    SURVIVING_MODEL_16AM = (
        planner_ns_16am[
            "model"
        ]
    )


# ======================================================================
# 21. RECOVER EXACT TOKENIZER ASSIGNMENT IF NEEDED
# ======================================================================

TOKENIZER_RECOVERY_ROW_16AM = None


if SURVIVING_TOKENIZER_16AM is None:

    # It may already have been created by model tuple assignment.
    if (
        "tokenizer"
        in
        planner_ns_16am
        and
        hasattr(
            planner_ns_16am[
                "tokenizer"
            ],
            "decode"
        )
    ):

        SURVIVING_TOKENIZER_16AM = (
            planner_ns_16am[
                "tokenizer"
            ]
        )


    else:

        tokenizer_rows = (
            NOTEBOOK_INDEX_16AM[
                "assignments"
            ].get(
                "tokenizer",
                []
            )
        )


        tokenizer_evidence_rows = [
            row
            for row
            in tokenizer_rows
            if assignment_references_model_id_16am(
                row
            )
        ]


        assert tokenizer_evidence_rows, (
            "No unambiguous persisted tokenizer assignment "
            "referencing rmanluo/RoG was found."
        )


        TOKENIZER_RECOVERY_ROW_16AM = (
            sorted(
                tokenizer_evidence_rows,
                key=lambda x:
                    x[
                        "cell_idx"
                    ],
                reverse=True
            )[
                0
            ]
        )


        source = (
            TOKENIZER_RECOVERY_ROW_16AM[
                "source"
            ]
        )


        print(
            "Executing exact persisted tokenizer assignment "
            f"from cell "
            f"{TOKENIZER_RECOVERY_ROW_16AM['cell_idx']}..."
        )


        for attempt in range(
            20
        ):

            try:

                exec(
                    source,
                    planner_ns_16am
                )

                break


            except NameError as exc:

                missing_name = (
                    missing_name_from_error_16am(
                        exc
                    )
                )


                assert missing_name is not None


                if missing_name in (
                    NOTEBOOK_INDEX_16AM[
                        "assignments"
                    ]
                ):

                    recover_assignment_target_16am(
                        missing_name,
                        cutoff_cell=
                            TOKENIZER_RECOVERY_ROW_16AM[
                                "cell_idx"
                            ]
                    )

                    continue


                if missing_name in (
                    NOTEBOOK_INDEX_16AM[
                        "functions"
                    ]
                ):

                    recover_latest_function_16am(
                        missing_name
                    )

                    continue


                raise


        assert (
            "tokenizer"
            in
            planner_ns_16am
        )


        SURVIVING_TOKENIZER_16AM = (
            planner_ns_16am[
                "tokenizer"
            ]
        )


# ======================================================================
# 22. HARD MODEL / TOKENIZER SOFTWARE GATES
# ======================================================================

assert SURVIVING_MODEL_16AM is not None
assert SURVIVING_TOKENIZER_16AM is not None

assert hasattr(
    SURVIVING_MODEL_16AM,
    "generate"
)

assert hasattr(
    SURVIVING_TOKENIZER_16AM,
    "decode"
)


model_runtime_ids_16am = (
    runtime_identity_strings_16am(
        SURVIVING_MODEL_16AM
    )
)


tokenizer_runtime_ids_16am = (
    runtime_identity_strings_16am(
        SURVIVING_TOKENIZER_16AM
    )
)


model_assignment_evidence_16am = (
    MODEL_RECOVERY_ROW_16AM
    is not None
    and
    assignment_references_model_id_16am(
        MODEL_RECOVERY_ROW_16AM
    )
)


tokenizer_assignment_evidence_16am = (
    TOKENIZER_RECOVERY_ROW_16AM
    is not None
    and
    assignment_references_model_id_16am(
        TOKENIZER_RECOVERY_ROW_16AM
    )
)


model_identity_verified_16am = (
    runtime_matches_model_id_16am(
        SURVIVING_MODEL_16AM
    )
    or
    model_assignment_evidence_16am
)


tokenizer_identity_verified_16am = (
    runtime_matches_model_id_16am(
        SURVIVING_TOKENIZER_16AM
    )
    or
    tokenizer_assignment_evidence_16am
    or
    runtime_matches_model_id_16am(
        SURVIVING_MODEL_16AM
    )
)


assert model_identity_verified_16am, (
    "Planner model identity could not be tied "
    "to rmanluo/RoG."
)


assert tokenizer_identity_verified_16am, (
    "Planner tokenizer identity could not be tied "
    "to rmanluo/RoG."
)


print(
    "\nPlanner model/tokenizer identity: PASSED"
)

print(
    "  model runtime IDs:",
    model_runtime_ids_16am
)

print(
    "  tokenizer runtime IDs:",
    tokenizer_runtime_ids_16am
)


# ======================================================================
# 23. INJECT EXACT REQUIRED GLOBALS INTO RECOVERED WRAPPER NAMESPACE
# ======================================================================

planner_ns_16am[
    "model"
] = SURVIVING_MODEL_16AM

planner_ns_16am[
    "tokenizer"
] = SURVIVING_TOKENIZER_16AM

planner_ns_16am[
    "prompter"
] = prompter_16am

planner_ns_16am[
    "INSTRUCTION"
] = INSTRUCTION_16AM

planner_ns_16am[
    "N_BEAM"
] = N_BEAM_16AM

planner_ns_16am[
    "time"
] = time

planner_ns_16am[
    "json"
] = json

planner_ns_16am[
    "tqdm"
] = tqdm


# Re-execute exact required functions LAST so their __globals__
# point to the final exact planner namespace.
for name in REQUIRED_FUNCTIONS_16AM:

    row = (
        RECOVERED_FUNCTION_ROWS_16AM[
            name
        ]
    )


    exec(
        (
            "from __future__ import annotations\n"
            +
            row[
                "source"
            ]
        ),
        planner_ns_16am
    )


GENERATE_SEQ_16AM = (
    planner_ns_16am[
        "generate_seq"
    ]
)


LOAD_CHECKPOINT_16AM = (
    planner_ns_16am[
        "load_checkpoint"
    ]
)


PARSE_PREDICTION_16AM = (
    planner_ns_16am[
        "parse_prediction"
    ]
)


RUN_PLANNING_16AM = (
    planner_ns_16am[
        "run_planning_checkpointed"
    ]
)


print(
    "\nExact planner namespace injection: PASSED"
)


# ======================================================================
# 24. SIGNATURE + SOURCE SOFTWARE GATES
# ======================================================================

assert str(
    inspect.signature(
        RUN_PLANNING_16AM
    )
) == (
    "(dataset_split, ckpt_path, desc)"
)


assert str(
    inspect.signature(
        RUN_PLANNING_16AM
    )
) == str(
    RUN_PLANNING_SIGNATURE_16AR
)


print(
    "run_planning_checkpointed signature fidelity: PASSED"
)


# ======================================================================
# 25. DIRECT DEPENDENCY AVAILABILITY GATE
# ======================================================================

for name in [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
    "prompter",
    "INSTRUCTION",
    "model",
    "tokenizer",
    "N_BEAM",
    "time",
]:

    assert name in (
        RUN_PLANNING_16AM.__globals__
    ), (
        f"Missing planner global dependency: {name}"
    )


print(
    "Planner direct dependencies: PASSED"
)


# ======================================================================
# 26. GENERATE_SEQ CONFIGURATION SOURCE AUDIT
# ======================================================================

GENERATE_SEQ_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "generate_seq"
    ][
        "source"
    ]
)


PARSE_PREDICTION_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "parse_prediction"
    ][
        "source"
    ]
)


LOAD_CHECKPOINT_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "load_checkpoint"
    ][
        "source"
    ]
)


RUN_PLANNING_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "run_planning_checkpointed"
    ][
        "source"
    ]
)


FUNCTION_SHA_16AM = {
    "generate_seq":
        sha256_text_16am(
            GENERATE_SEQ_SOURCE_16AM
        ),

    "parse_prediction":
        sha256_text_16am(
            PARSE_PREDICTION_SOURCE_16AM
        ),

    "load_checkpoint":
        sha256_text_16am(
            LOAD_CHECKPOINT_SOURCE_16AM
        ),

    "run_planning_checkpointed":
        sha256_text_16am(
            RUN_PLANNING_SOURCE_16AM
        ),
}


print(
    "\nExact planner function SHA256:"
)


for name, digest in (
    FUNCTION_SHA_16AM.items()
):

    print(
        f"  {name:<28} {digest}"
    )


# ======================================================================
# 27. STOCHASTIC-SEED AUDIT
# ======================================================================
#
# Wrapper explicitly uses do_sample=True.
#
# We therefore inspect seed calls in the recovered planning setup.
#
# IMPORTANT:
# We DO NOT invent a seed if the original notebook did not explicitly
# establish one in the planning setup.
# ======================================================================

setup_cells_16am = [
    INSTRUCTION_RECOVERY_ROW_16AM[
        "cell_idx"
    ],

    N_BEAM_RECOVERY_ROW_16AM[
        "cell_idx"
    ],

    PROMPTER_RECOVERY_ROW_16AM[
        "cell_idx"
    ],
]


if MODEL_RECOVERY_ROW_16AM is not None:

    setup_cells_16am.append(
        MODEL_RECOVERY_ROW_16AM[
            "cell_idx"
        ]
    )


if TOKENIZER_RECOVERY_ROW_16AM is not None:

    setup_cells_16am.append(
        TOKENIZER_RECOVERY_ROW_16AM[
            "cell_idx"
        ]
    )


PLANNER_SETUP_START_CELL_16AM = min(
    setup_cells_16am
)


PLANNER_SEED_CALLS_16AM = [
    row
    for row
    in NOTEBOOK_INDEX_16AM[
        "seed_calls"
    ]
    if (
        row[
            "cell_idx"
        ]
        >=
        PLANNER_SETUP_START_CELL_16AM
        and
        row[
            "cell_idx"
        ]
        <=
        PLANNER_MAX_CELL_16AM
    )
]


print(
    "\nRecovered explicit seed calls "
    "inside planner setup window:"
)


if PLANNER_SEED_CALLS_16AM:

    for row in (
        PLANNER_SEED_CALLS_16AM
    ):

        print(
            f"  cell {row['cell_idx']}: "
            f"{row['source']}"
        )


else:

    print(
        "  NONE"
    )


# Execute exactly recovered planner-window seed calls in notebook order.
for row in sorted(
    PLANNER_SEED_CALLS_16AM,
    key=lambda x:
        x[
            "cell_idx"
        ]
):

    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except Exception as exc:

        raise AssertionError(
            "Recovered explicit planner seed call "
            "could not be reproduced:\n"
            f"{row['source']}\n"
            f"{type(exc).__name__}: {exc}"
        )


if PLANNER_SEED_CALLS_16AM:

    print(
        "Exact recovered planner seed setup: APPLIED"
    )


else:

    print(
        "\nIMPORTANT REPRODUCIBILITY NOTE:"
    )

    print(
        "The persisted planning setup contains no explicit "
        "seed call in the recovered planner setup window."
    )

    print(
        "No new seed will be invented."
    )

    print(
        "The materialized TEST plan files themselves will "
        "be frozen by SHA256 after generation."
    )


# ======================================================================
# 28. FINAL PRE-GENERATION SAFETY GATES
# ======================================================================

assert len(
    webqsp_raw_test_16a
) == 1628


assert len(
    cwq_raw_test_16a
) == 3531


assert len(
    webqsp_official_test_16ar
) == 1628


assert len(
    cwq_official_test_16ar
) == 3531


# Exact wrapper uses only sample["question"] for planner input.
wrapper_compact_16am = (
    re.sub(
        r"\s+",
        "",
        RUN_PLANNING_SOURCE_16AM
    )
)


assert (
    'message=sample["question"]'
    in
    wrapper_compact_16am
), (
    "Could not verify that planner prompt uses "
    "question-only input."
)


# Ensure gold is NOT used in planner prompt/generation expression.
prompt_generation_prefix_16am = (
    RUN_PLANNING_SOURCE_16AM.split(
        "rec =",
        1
    )[
        0
    ]
)


assert (
    'sample["a_entity"]'
    not in
    prompt_generation_prefix_16am
)


assert (
    'sample["answer"]'
    not in
    prompt_generation_prefix_16am
)


print(
    "\nPRE-GENERATION SAFETY GATES: PASSED"
)

print(
    "  TEST gold used in planner prompt: NO"
)

print(
    "  TEST gold used in generation:     NO"
)

print(
    "  AFP changed:                      NO"
)

print(
    "  hyperparameter tuning:            NO"
)


# ======================================================================
# 29. CHECK EXISTING PARTIAL CHECKPOINTS
# ======================================================================

def checkpoint_count_16am(
    path
):

    if not path.exists():

        return 0


    count = 0


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                count += 1


    return count


web_existing_16am = (
    checkpoint_count_16am(
        WEBQSP_TEST_PLAN_PATH
    )
)


cwq_existing_16am = (
    checkpoint_count_16am(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nCheckpoint status before generation:"
)

print(
    "  WebQSP:",
    web_existing_16am,
    "/ 1628"
)

print(
    "  CWQ:   ",
    cwq_existing_16am,
    "/ 3531"
)


# ======================================================================
# 30. MATERIALIZE WEBQSP TEST PLANS
# ======================================================================
#
# Exact persisted wrapper.
#
# Checkpointed / resume-safe.
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "MATERIALIZING FROZEN WEBQSP TEST RELATION PLANS"
)

print(
    "=" * 124
)


webqsp_materialize_start_16am = (
    time.time()
)


webqsp_test_plan_rows = (
    RUN_PLANNING_16AM(
        dataset_split=
            webqsp_official_test_16ar,

        ckpt_path=
            str(
                WEBQSP_TEST_PLAN_PATH
            ),

        desc=
            "WebQSP frozen TEST planning"
    )
)


webqsp_materialize_elapsed_16am = (
    time.time()
    -
    webqsp_materialize_start_16am
)


print(
    "\nWebQSP planning elapsed:",
    f"{webqsp_materialize_elapsed_16am/60:.2f} min"
)


# ======================================================================
# 31. MATERIALIZE CWQ TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "MATERIALIZING FROZEN CWQ TEST RELATION PLANS"
)

print(
    "=" * 124
)


cwq_materialize_start_16am = (
    time.time()
)


cwq_test_plan_rows = (
    RUN_PLANNING_16AM(
        dataset_split=
            cwq_official_test_16ar,

        ckpt_path=
            str(
                CWQ_TEST_PLAN_PATH
            ),

        desc=
            "CWQ frozen TEST planning"
    )
)


cwq_materialize_elapsed_16am = (
    time.time()
    -
    cwq_materialize_start_16am
)


print(
    "\nCWQ planning elapsed:",
    f"{cwq_materialize_elapsed_16am/60:.2f} min"
)


# ======================================================================
# 32. STRICT MATERIALIZED-PLAN AUDIT
# ======================================================================

REQUIRED_OUTPUT_FIELDS_16AM = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
    "planning_time_sec",
}


def audit_materialized_plans_16am(
    dataset_name,
    rows,
    raw_rows,
    expected_count
):

    assert len(
        rows
    ) == expected_count, (
        f"{dataset_name}: output row count "
        f"{len(rows)} != {expected_count}"
    )


    ids = [
        str(
            row[
                "id"
            ]
        )
        for row in rows
    ]


    assert len(
        ids
    ) == len(
        set(
            ids
        )
    ), (
        f"{dataset_name}: duplicate planning IDs."
    )


    raw_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row
        for row in raw_rows
    }


    assert set(
        ids
    ) == set(
        raw_by_id.keys()
    ), (
        f"{dataset_name}: materialized ID set "
        "does not equal raw TEST ID set."
    )


    empty_plan_records = 0

    total_predicted_paths = 0

    max_predicted_paths = 0


    for row in rows:

        assert (
            REQUIRED_OUTPUT_FIELDS_16AM.issubset(
                row.keys()
            )
        ), (
            f"{dataset_name}: missing planning fields "
            f"for id={row.get('id')}"
        )


        qid = str(
            row[
                "id"
            ]
        )


        raw = raw_by_id[
            qid
        ]


        assert str(
            row[
                "question"
            ]
        ) == str(
            raw[
                "question"
            ]
        )


        assert list(
            row[
                "q_entity"
            ]
        ) == list(
            raw[
                "q_entity"
            ]
        )


        assert list(
            row[
                "a_entity"
            ]
        ) == list(
            raw[
                "a_entity"
            ]
        )


        paths = row[
            "predicted_paths"
        ]


        assert isinstance(
            paths,
            list
        ), (
            f"{dataset_name}: predicted_paths "
            f"is not list for {qid}"
        )


        total_predicted_paths += len(
            paths
        )


        max_predicted_paths = max(
            max_predicted_paths,
            len(
                paths
            )
        )


        if len(
            paths
        ) == 0:

            empty_plan_records += 1


        # Frozen Top-K planning.
        assert len(
            paths
        ) <= FROZEN_TOP_K_16AM, (
            f"{dataset_name}: >Top-K plans "
            f"for id={qid}: {len(paths)}"
        )


        planning_time = float(
            row[
                "planning_time_sec"
            ]
        )


        assert (
            planning_time
            >=
            0.0
        )


    summary = {
        "questions":
            int(
                len(
                    rows
                )
            ),

        "empty_plan_questions":
            int(
                empty_plan_records
            ),

        "total_predicted_paths":
            int(
                total_predicted_paths
            ),

        "mean_predicted_paths_per_question":
            float(
                total_predicted_paths
                /
                len(
                    rows
                )
            ),

        "max_predicted_paths_per_question":
            int(
                max_predicted_paths
            ),
    }


    print(
        f"\n{dataset_name.upper()} materialized-plan audit: PASSED"
    )


    print(
        "  questions:",
        summary[
            "questions"
        ]
    )


    print(
        "  total plans:",
        summary[
            "total_predicted_paths"
        ]
    )


    print(
        "  mean plans/question:",
        f"{summary['mean_predicted_paths_per_question']:.6f}"
    )


    print(
        "  max plans/question:",
        summary[
            "max_predicted_paths_per_question"
        ]
    )


    print(
        "  empty-plan questions:",
        summary[
            "empty_plan_questions"
        ]
    )


    return summary


WEBQSP_TEST_PLAN_AUDIT_16AM = (
    audit_materialized_plans_16am(
        dataset_name=
            "webqsp",

        rows=
            webqsp_test_plan_rows,

        raw_rows=
            webqsp_raw_test_16a,

        expected_count=
            1628
    )
)


CWQ_TEST_PLAN_AUDIT_16AM = (
    audit_materialized_plans_16am(
        dataset_name=
            "cwq",

        rows=
            cwq_test_plan_rows,

        raw_rows=
            cwq_raw_test_16a,

        expected_count=
            3531
    )
)


# ======================================================================
# 33. VERIFY FILE-LEVEL CHECKPOINT CONTENT
# ======================================================================

assert WEBQSP_TEST_PLAN_PATH.exists()
assert CWQ_TEST_PLAN_PATH.exists()


assert (
    checkpoint_count_16am(
        WEBQSP_TEST_PLAN_PATH
    )
    ==
    1628
)


assert (
    checkpoint_count_16am(
        CWQ_TEST_PLAN_PATH
    )
    ==
    3531
)


WEBQSP_TEST_PLAN_SHA = (
    sha256_file_16am(
        WEBQSP_TEST_PLAN_PATH
    )
)


CWQ_TEST_PLAN_SHA = (
    sha256_file_16am(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nFrozen TEST planning artifact SHA256:"
)

print(
    "  WebQSP:",
    WEBQSP_TEST_PLAN_SHA
)

print(
    "  CWQ:   ",
    CWQ_TEST_PLAN_SHA
)


# ======================================================================
# 34. VALIDATION/TEST DISJOINTNESS AGAIN AFTER MATERIALIZATION
# ======================================================================

web_val_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    webqsp_val_plan_rows
}


cwq_val_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    cwq_val_plan_rows
}


web_test_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    webqsp_test_plan_rows
}


cwq_test_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    cwq_test_plan_rows
}


assert not (
    web_val_ids_16am
    &
    web_test_ids_16am
)


assert not (
    cwq_val_ids_16am
    &
    cwq_test_ids_16am
)


print(
    "\nFinal validation / TEST disjointness: PASSED"
)


# ======================================================================
# 35. PLANNER CONFIG FINGERPRINT
# ======================================================================
#
# This does NOT replace the earlier frozen planner config hash.
# It records the exact implementation used to generate TEST artifacts.
# ======================================================================

TEST_PLANNER_IMPLEMENTATION_PAYLOAD_16AM = {
    "model_id":
        FROZEN_MODEL_ID_16AM,

    "instruction_sha256":
        INSTRUCTION_SHA_16AM,

    "N_BEAM":
        N_BEAM_16AM,

    "do_sample":
        True,

    "max_new_tokens":
        100,

    "prompter_assignment_cell":
        int(
            PROMPTER_RECOVERY_ROW_16AM[
                "cell_idx"
            ]
        ),

    "function_sha256":
        FUNCTION_SHA_16AM,

    "seed_calls":
        [
            {
                "cell_idx":
                    int(
                        row[
                            "cell_idx"
                        ]
                    ),

                "source":
                    row[
                        "source"
                    ],
            }

            for row in
            PLANNER_SEED_CALLS_16AM
        ],
}


TEST_PLANNER_IMPLEMENTATION_JSON_16AM = (
    json.dumps(
        TEST_PLANNER_IMPLEMENTATION_PAYLOAD_16AM,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )
)


TEST_PLANNER_IMPLEMENTATION_SHA_16AM = (
    sha256_text_16am(
        TEST_PLANNER_IMPLEMENTATION_JSON_16AM
    )
)


print(
    "\nTEST planner implementation fingerprint:"
)

print(
    " ",
    TEST_PLANNER_IMPLEMENTATION_SHA_16AM
)


# ======================================================================
# 36. SAVE MATERIALIZATION MANIFEST
# ======================================================================

CELL16A_MATERIALIZE_MANIFEST = {
    "cell":
        "CELL16A_MATERIALIZE_EXACT_FROZEN_ROG_TEST_PLANS",

    "afp_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "development_planner_identity": {
        "model_id":
            FROZEN_MODEL_ID_16AM,

        "top_k":
            FROZEN_TOP_K_16AM,

        "instruction_sha256":
            INSTRUCTION_SHA_16AM,

        "instruction_expected_sha_prefix":
            FROZEN_INSTRUCTION_SHA_PREFIX_16AM,

        "prior_planner_config_sha_prefix":
            FROZEN_PLANNER_CONFIG_SHA_PREFIX_16AM,
    },

    "generation_configuration": {
        "N_BEAM":
            N_BEAM_16AM,

        "do_sample":
            True,

        "max_new_tokens":
            100,

        "generation_order": [
            "webqsp",
            "cwq",
        ],

        "explicit_recovered_seed_calls":
            [
                {
                    "cell_idx":
                        int(
                            row[
                                "cell_idx"
                            ]
                        ),

                    "source":
                        row[
                            "source"
                        ],
                }

                for row in
                PLANNER_SEED_CALLS_16AM
            ],

        "new_seed_invented":
            False,
    },

    "software_identity": {
        "planner_function_signature":
            str(
                inspect.signature(
                    RUN_PLANNING_16AM
                )
            ),

        "function_sha256":
            FUNCTION_SHA_16AM,

        "test_planner_implementation_sha256":
            TEST_PLANNER_IMPLEMENTATION_SHA_16AM,

        "model_runtime_identity":
            model_runtime_ids_16am,

        "tokenizer_runtime_identity":
            tokenizer_runtime_ids_16am,

        "model_identity_verified":
            bool(
                model_identity_verified_16am
            ),

        "tokenizer_identity_verified":
            bool(
                tokenizer_identity_verified_16am
            ),

        "prompt_probe_sha256":
            PROMPT_PROBE_SHA_16AM,
    },

    "test_input": {
        "webqsp_questions":
            1628,

        "cwq_questions":
            3531,

        "official_source_validation_identity_gate":
            True,

        "validation_test_disjoint":
            True,
    },

    "output": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TEST_PLAN_PATH
                ),

            "sha256":
                WEBQSP_TEST_PLAN_SHA,

            **WEBQSP_TEST_PLAN_AUDIT_16AM,
        },

        "cwq": {
            "path":
                str(
                    CWQ_TEST_PLAN_PATH
                ),

            "sha256":
                CWQ_TEST_PLAN_SHA,

            **CWQ_TEST_PLAN_AUDIT_16AM,
        },
    },

    "leakage": {
        "test_gold_used_in_planner_prompt":
            False,

        "test_gold_used_in_generation":
            False,

        "test_used_for_parameter_selection":
            False,

        "test_used_for_hyperparameter_tuning":
            False,

        "test_triggered_method_change":
            False,
    },

    "afp_changed_after_freeze":
        False,

    "next":
        "rerun_Cell16_unchanged",
}


with open(
    MATERIALIZE_MANIFEST_PATH_16AM,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL16A_MATERIALIZE_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16A_MATERIALIZE_COMPLETE = True

FROZEN_TEST_PLANS_MATERIALIZED = True


# ======================================================================
# 37. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 128
)

print(
    "=== CELL 16A-MATERIALIZE: "
    "FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ==="
)

print(
    "=" * 128
)


print(
    "\nPlanner identity:"
)

print(
    "  model:",
    FROZEN_MODEL_ID_16AM
)

print(
    "  instruction SHA:",
    INSTRUCTION_SHA_16AM
)

print(
    "  N_BEAM:",
    N_BEAM_16AM
)

print(
    "  do_sample:",
    True
)

print(
    "  max_new_tokens:",
    100
)


print(
    "\nTEST plans:"
)

print(
    "  WebQSP:",
    len(
        webqsp_test_plan_rows
    ),
    "questions"
)

print(
    "    file:",
    WEBQSP_TEST_PLAN_PATH
)

print(
    "    SHA:",
    WEBQSP_TEST_PLAN_SHA
)


print(
    "  CWQ:",
    len(
        cwq_test_plan_rows
    ),
    "questions"
)

print(
    "    file:",
    CWQ_TEST_PLAN_PATH
)

print(
    "    SHA:",
    CWQ_TEST_PLAN_SHA
)


print(
    "\nLeakage / freeze status:"
)

print(
    "  TEST gold in planner prompt: NO"
)

print(
    "  TEST tuning:                 NO"
)

print(
    "  TEST-triggered changes:      NO"
)

print(
    "  AFP changed after freeze:    NO"
)


print(
    "\nMaterialization manifest:"
)

print(
    " ",
    MATERIALIZE_MANIFEST_PATH_16AM
)


print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16 UNCHANGED."
)

print(
    "Do NOT move directly to Cell 17."
)

Cell 16A-MATERIALIZE freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen wrapper constants:
  model: rmanluo/RoG
  beam: 3
  do_sample: True
  max_new_tokens: 100

Persisted notebook AST index: READY
Persisted imports attempted.
Non-critical import failures: 9


AssertionError: Could not recover persisted function: generate_seq

In [38]:
# ======================================================================
# CELL 16A-D
# EXACT PLANNER DEPENDENCY ORIGIN RECOVERY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT restart kernel.
# DO NOT rerun Cell 16 yet.
#
# PURPOSE
# -------
# Determine EXACTLY where these persisted planner dependencies came from:
#
#   generate_seq
#   load_checkpoint
#   parse_prediction
#
# They may be:
#   - imported with `from module import ...`
#   - imported with `from module import *`
#   - assigned/aliased
#   - defined in another local .py module
#   - defined in notebook code not represented as top-level FunctionDef
#
# NO TEST generation occurs here.
# ======================================================================

import ast
import importlib
import importlib.util
import inspect
import json
import os
import re
import sys
from pathlib import Path


# ======================================================================
# 1. HARD GATE
# ======================================================================

required = [
    "CELL16AR_COMPLETE",
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "persisted_nb_16ar",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Missing prerequisites:\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("Cell 16A-D freeze gate: PASSED")


# ======================================================================
# 2. TARGET SYMBOLS
# ======================================================================

TARGETS_16AD = [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
]


# ======================================================================
# 3. COLLECT NOTEBOOK OCCURRENCES
# ======================================================================

occurrences_16ad = {
    name: []
    for name in TARGETS_16AD
}

sys_path_statements_16ad = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    # Only relevant pre-planner history.
    if cell_idx > 139:
        continue

    # --------------------------------------------------------------
    # Recover sys.path mutations because imported helper modules may
    # depend on them.
    # --------------------------------------------------------------

    for line in source.splitlines():

        stripped = line.strip()

        if (
            stripped.startswith("sys.path.append(")
            or
            stripped.startswith("sys.path.insert(")
        ):
            sys_path_statements_16ad.append(
                {
                    "cell_idx": cell_idx,
                    "source": stripped,
                }
            )

    # --------------------------------------------------------------
    # Any textual occurrence of target symbols.
    # --------------------------------------------------------------

    for name in TARGETS_16AD:

        if name not in source:
            continue

        info = {
            "cell_idx": cell_idx,
            "full_source": source,
            "definitions": [],
            "imports": [],
            "assignments": [],
        }

        try:
            tree = ast.parse(source)

        except Exception:
            occurrences_16ad[name].append(info)
            continue

        for node in tree.body:

            segment = ast.get_source_segment(
                source,
                node
            )

            if not segment:
                continue

            # Direct function definition.
            if (
                isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    )
                )
                and
                node.name == name
            ):
                info["definitions"].append(
                    segment
                )

            # from module import symbol
            elif isinstance(
                node,
                ast.ImportFrom
            ):

                imported_names = [
                    alias.name
                    for alias in node.names
                ]

                aliases = [
                    alias.asname
                    for alias in node.names
                ]

                if (
                    name in imported_names
                    or
                    name in aliases
                    or
                    "*" in imported_names
                ):
                    info["imports"].append(
                        segment
                    )

            # import module
            elif isinstance(
                node,
                ast.Import
            ):

                if name in segment:
                    info["imports"].append(
                        segment
                    )

            # Assignment / alias.
            elif isinstance(
                node,
                (
                    ast.Assign,
                    ast.AnnAssign,
                )
            ):

                if name in segment:
                    info["assignments"].append(
                        segment
                    )

        occurrences_16ad[name].append(info)


# ======================================================================
# 4. PRINT NOTEBOOK ORIGIN EVIDENCE
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "NOTEBOOK SYMBOL ORIGIN EVIDENCE"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    print(
        f"\n### {name}"
    )

    rows = occurrences_16ad[name]

    print(
        "occurrence cells:",
        [
            row["cell_idx"]
            for row in rows
        ]
    )

    for row in rows:

        meaningful = (
            row["definitions"]
            or
            row["imports"]
            or
            row["assignments"]
        )

        if not meaningful:
            continue

        print(
            f"\n--- cell {row['cell_idx']} ---"
        )

        for src in row["definitions"]:
            print(
                "[DEFINITION]"
            )
            print(src)

        for src in row["imports"]:
            print(
                "[IMPORT]"
            )
            print(src)

        for src in row["assignments"]:
            print(
                "[ASSIGNMENT]"
            )
            print(src)


# ======================================================================
# 5. REPLAY ONLY EXACT sys.path MUTATIONS
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "RECOVERED sys.path MUTATIONS"
)

print(
    "=" * 120
)


for row in sys_path_statements_16ad:

    print(
        f"cell {row['cell_idx']}: "
        f"{row['source']}"
    )


for row in sorted(
    sys_path_statements_16ad,
    key=lambda x: x["cell_idx"]
):

    try:

        exec(
            row["source"],
            {
                "sys": sys,
                "os": os,
                "Path": Path,
            }
        )

    except Exception as exc:

        print(
            "Could not replay:",
            row["source"]
        )

        print(
            type(exc).__name__,
            exc
        )


# ======================================================================
# 6. ATTEMPT EXACT NOTEBOOK IMPORTS
# ======================================================================

resolved_16ad = {}

import_attempts_16ad = []


for name in TARGETS_16AD:

    # --------------------------------------------------------------
    # First: maybe function already exists in live global namespace.
    # --------------------------------------------------------------

    live = globals().get(
        name
    )

    if callable(live):

        resolved_16ad[name] = {
            "object": live,
            "origin":
                "surviving_notebook_global",
            "source":
                (
                    inspect.getsource(live)
                    if inspect.isfunction(live)
                    else None
                ),
        }

        continue

    # --------------------------------------------------------------
    # Replay exact persisted import statements that mention the target.
    # --------------------------------------------------------------

    candidate_imports = []

    for row in occurrences_16ad[name]:

        candidate_imports.extend(
            row["imports"]
        )

    candidate_imports = list(
        dict.fromkeys(
            candidate_imports
        )
    )

    for statement in candidate_imports:

        local_ns = {}

        try:

            exec(
                statement,
                globals(),
                local_ns
            )

            obj = (
                local_ns.get(name)
                or
                globals().get(name)
            )

            # Star imports may place it in locals.
            if callable(obj):

                resolved_16ad[name] = {
                    "object": obj,
                    "origin":
                        f"exact_import: {statement}",
                    "source":
                        (
                            inspect.getsource(obj)
                            if inspect.isfunction(obj)
                            else None
                        ),
                }

                import_attempts_16ad.append(
                    {
                        "symbol": name,
                        "statement": statement,
                        "status": "RESOLVED",
                    }
                )

                break

            import_attempts_16ad.append(
                {
                    "symbol": name,
                    "statement": statement,
                    "status":
                        "import succeeded but symbol absent",
                }
            )

        except Exception as exc:

            import_attempts_16ad.append(
                {
                    "symbol": name,
                    "statement": statement,
                    "status":
                        (
                            f"{type(exc).__name__}: "
                            f"{exc}"
                        ),
                }
            )


# ======================================================================
# 7. SEARCH LOCAL PYTHON SOURCES
# ======================================================================
#
# Only searches code-sized roots, not dataset parquet content.
# ======================================================================

SEARCH_ROOTS_16AD = [
    Path.cwd(),
    Path(
        "/kaggle/working"
    ),
    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap"
    ),
]


file_hits_16ad = {
    name: []
    for name in TARGETS_16AD
}


seen_files_16ad = set()


for root in SEARCH_ROOTS_16AD:

    if not root.exists():
        continue

    try:
        py_files = list(
            root.rglob("*.py")
        )

    except Exception:
        continue

    for path in py_files:

        path_key = str(
            path.resolve()
        )

        if path_key in seen_files_16ad:
            continue

        seen_files_16ad.add(
            path_key
        )

        # Avoid giant accidental source files.
        try:

            if path.stat().st_size > 5_000_000:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception:
            continue

        for name in TARGETS_16AD:

            patterns = [
                rf"\bdef\s+{re.escape(name)}\s*\(",
                rf"\b{name}\s*=",
                rf"\bimport\s+{re.escape(name)}\b",
                rf"\bimport\s+.*\b{re.escape(name)}\b",
            ]

            if any(
                re.search(pattern, text)
                for pattern in patterns
            ):

                file_hits_16ad[name].append(
                    path
                )


# ======================================================================
# 8. PRINT FILE SEARCH
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "LOCAL PYTHON SOURCE HITS"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    print(
        f"\n{name}:"
    )

    if not file_hits_16ad[name]:

        print(
            "  NONE"
        )

    else:

        for path in file_hits_16ad[name]:

            print(
                " ",
                path
            )


# ======================================================================
# 9. INSPECT IMPORT ATTEMPTS
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "EXACT IMPORT ATTEMPTS"
)

print(
    "=" * 120
)


if not import_attempts_16ad:

    print(
        "No direct import statements found."
    )


for row in import_attempts_16ad:

    print(
        f"\n{row['symbol']}"
    )

    print(
        " statement:",
        row["statement"]
    )

    print(
        " status:",
        row["status"]
    )


# ======================================================================
# 10. RESOLVED FUNCTION REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "RESOLVED EXACT PLANNER DEPENDENCIES"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    if name not in resolved_16ad:

        print(
            f"\n{name}: NOT YET RESOLVED"
        )

        continue

    item = resolved_16ad[name]

    obj = item["object"]

    print(
        f"\n{name}: RESOLVED"
    )

    print(
        " origin:",
        item["origin"]
    )

    try:

        print(
            " module:",
            obj.__module__
        )

    except Exception:
        pass

    try:

        print(
            " signature:",
            inspect.signature(
                obj
            )
        )

    except Exception:
        pass

    try:

        src = inspect.getsource(
            obj
        )

        print(
            "\nSOURCE:"
        )

        print(src)

    except Exception as exc:

        print(
            " source unavailable:",
            type(exc).__name__,
            exc
        )


# ======================================================================
# 11. SAVE DIAGNOSTIC REPORT
# ======================================================================

OUT_DIR_16AD = Path(
    "/kaggle/working/"
    "step2_rq1_test/"
    "planner_recovery"
)

OUT_DIR_16AD.mkdir(
    parents=True,
    exist_ok=True
)


REPORT_PATH_16AD = (
    OUT_DIR_16AD
    / "cell16ad_dependency_origin_report.json"
)


report_16ad = {
    "freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "targets":
        TARGETS_16AD,

    "resolved": {
        name:
            {
                "resolved":
                    name in resolved_16ad,

                "origin":
                    (
                        resolved_16ad[
                            name
                        ][
                            "origin"
                        ]
                        if name in resolved_16ad
                        else None
                    ),

                "module":
                    (
                        getattr(
                            resolved_16ad[
                                name
                            ][
                                "object"
                            ],
                            "__module__",
                            None
                        )
                        if name in resolved_16ad
                        else None
                    ),
            }

        for name in TARGETS_16AD
    },

    "notebook_occurrence_cells": {
        name:
            [
                row["cell_idx"]
                for row in
                occurrences_16ad[name]
            ]

        for name in TARGETS_16AD
    },

    "file_hits": {
        name:
            [
                str(path)
                for path in
                file_hits_16ad[name]
            ]

        for name in TARGETS_16AD
    },

    "import_attempts":
        import_attempts_16ad,

    "test_planning_started":
        False,

    "afp_changed":
        False,
}


with open(
    REPORT_PATH_16AD,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report_16ad,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AD_COMPLETE = True

RESOLVED_PLANNER_DEPENDENCIES_16AD = (
    resolved_16ad
)


print(
    "\n"
    + "=" * 120
)

print(
    "=== CELL 16A-D COMPLETE ==="
)

print(
    "=" * 120
)

print(
    "\nTEST planning started: NO"
)

print(
    "AFP changed: NO"
)

print(
    "\nResolved:",
    {
        name:
            name in resolved_16ad
        for name in TARGETS_16AD
    }
)

print(
    "\nReport:"
)

print(
    " ",
    REPORT_PATH_16AD
)

Cell 16A-D freeze gate: PASSED

NOTEBOOK SYMBOL ORIGIN EVIDENCE

### generate_seq
occurrence cells: [19, 21, 47, 83, 105, 139]

--- cell 19 ---
[IMPORT]
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

### load_checkpoint
occurrence cells: [47, 57, 105, 115, 139]

--- cell 47 ---
[DEFINITION]
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done[rec["id"]] = rec
    return done
[ASSIGNMENT]
planning_done = load_checkpoint(PLANNING_CKPT)

--- cell 57 ---
[ASSIGNMENT]
reasoning_done = load_checkpoint(REASON_CKPT)

--- cell 105 ---
[ASSIGNMENT]
cwq_planning_done = load_checkpoint(CWQ_PLANNING_CKPT)

### parse_prediction
occurrence cells: [19, 21, 41, 47, 83, 99, 105, 139]

--- cell 19 ---
[IMPORT]
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

R

In [4]:
# ======================================================================
# CELL 16A-M3
# EXACT FROZEN RoG TEST PLAN MATERIALIZATION
# DIRECT-SOURCE RECOVERY — NO utils PACKAGE IMPORT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M2/M2R
#   - restart kernel
#   - run Cell 16 yet
#   - install/upgrade RoG dependencies
#   - change AFP
#
# WHY THIS VERSION
# ----------------
# RoG's gen_rule_path.py imports the full "utils" package.
# That indirectly imports graph_utils -> walker.
#
# But our frozen planning wrapper uses only:
#
#   generate_seq
#   parse_prediction
#   INSTRUCTION
#
# Those symbols do NOT need graph-walker.
#
# Therefore this cell extracts their EXACT source definitions directly
# from the SHA-verified local gen_rule_path.py instead of importing the
# whole module.
#
# It also extracts the exact InstructFormater implementation directly
# from src/utils/utils.py without importing src/utils/__init__.py.
#
# NO source code is modified.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import math
import os
import random
import re
import sys
import time
import types
from pathlib import Path

import numpy as np
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

try:
    from peft import (
        AutoPeftModelForCausalLM,
    )
except Exception:
    AutoPeftModelForCausalLM = None


# ======================================================================
# 1. HARD GATES
# ======================================================================

required = [
    "CELL16AR_COMPLETE",
    "CELL16AD_COMPLETE",

    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "persisted_nb_16ar",
    "run_planning_source_16ar",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    x
    for x in required
    if x not in globals()
]

assert not missing, (
    "Missing prerequisite object(s):\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert CELL16AD_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AM3 = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AM3
)


print(
    "Cell 16A-M3 freeze gate: PASSED"
)

print(
    "Freeze SHA:",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 2. FROZEN PLANNER FACTS
# ======================================================================

FROZEN_MODEL_ID_16AM3 = (
    "rmanluo/RoG"
)

FROZEN_N_BEAM_16AM3 = 3

FROZEN_INSTRUCTION_SHA_PREFIX_16AM3 = (
    "e3687b4a5081c22c"
)


assert (
    "do_sample=True"
    in
    run_planning_source_16ar
)

assert (
    "max_new_tokens=100"
    in
    run_planning_source_16ar
)

assert (
    "num_beam=N_BEAM"
    in
    run_planning_source_16ar
)


print(
    "\nFrozen planner:"
)

print(
    " model:",
    FROZEN_MODEL_ID_16AM3
)

print(
    " N_BEAM:",
    FROZEN_N_BEAM_16AM3
)

print(
    " do_sample:",
    True
)

print(
    " max_new_tokens:",
    100
)


# ======================================================================
# 3. CUDA GATE
# ======================================================================
#
# Exact persisted generate_seq uses:
#
#     .to("cuda")
#
# Therefore CPU substitution would change the implementation.
# ======================================================================

assert torch.cuda.is_available(), (
    "CUDA is unavailable.\n"
    "Exact recovered generate_seq hardcodes .to('cuda').\n"
    "Enable a Kaggle GPU before materializing TEST plans."
)


print(
    "\nCUDA gate: PASSED"
)

print(
    " GPU:",
    torch.cuda.get_device_name(0)
)


# ======================================================================
# 4. SHA HELPERS
# ======================================================================

def sha256_file_16am3(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_text_16am3(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


# ======================================================================
# 5. EXACT SOURCE FILES
# ======================================================================

ROG_REPO_16AM3 = Path(
    "/kaggle/working/"
    "reasoning-on-graphs"
)


GEN_RULE_PATH_16AM3 = (
    ROG_REPO_16AM3
    / "src"
    / "qa_prediction"
    / "gen_rule_path.py"
)


UTILS_UTILS_PATH_16AM3 = (
    ROG_REPO_16AM3
    / "src"
    / "utils"
    / "utils.py"
)


assert GEN_RULE_PATH_16AM3.exists()
assert UTILS_UTILS_PATH_16AM3.exists()


EXPECTED_GEN_RULE_SHA_16AM3 = (
    "e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af"
)


ACTUAL_GEN_RULE_SHA_16AM3 = (
    sha256_file_16am3(
        GEN_RULE_PATH_16AM3
    )
)


assert (
    ACTUAL_GEN_RULE_SHA_16AM3
    ==
    EXPECTED_GEN_RULE_SHA_16AM3
)


print(
    "\nExact gen_rule_path.py SHA gate: PASSED"
)

print(
    " ",
    ACTUAL_GEN_RULE_SHA_16AM3
)


# ======================================================================
# 6. READ EXACT gen_rule_path.py AS TEXT
# ======================================================================
#
# IMPORTANT:
# We DO NOT import the module.
#
# Therefore:
#
#   utils
#   graph_utils
#   walker
#
# are never imported.
# ======================================================================

GEN_RULE_SOURCE_16AM3 = (
    GEN_RULE_PATH_16AM3.read_text(
        encoding="utf-8"
    )
)


GEN_RULE_AST_16AM3 = ast.parse(
    GEN_RULE_SOURCE_16AM3
)


# ======================================================================
# 7. EXTRACT EXACT CONSTANTS + FUNCTIONS
# ======================================================================

INSTRUCTION_NODE_16AM3 = None
PATH_RE_NODE_16AM3 = None

GENERATE_SEQ_NODE_16AM3 = None
PARSE_PREDICTION_NODE_16AM3 = None


for node in GEN_RULE_AST_16AM3.body:

    # --------------------------------------------------------------
    # Constants
    # --------------------------------------------------------------

    if isinstance(
        node,
        ast.Assign
    ):

        target_names = [
            target.id
            for target in node.targets
            if isinstance(
                target,
                ast.Name
            )
        ]


        if "INSTRUCTION" in target_names:

            INSTRUCTION_NODE_16AM3 = node


        if "PATH_RE" in target_names:

            PATH_RE_NODE_16AM3 = node


    # --------------------------------------------------------------
    # Functions
    # --------------------------------------------------------------

    elif isinstance(
        node,
        ast.FunctionDef
    ):

        if node.name == "generate_seq":

            GENERATE_SEQ_NODE_16AM3 = node


        elif node.name == "parse_prediction":

            PARSE_PREDICTION_NODE_16AM3 = node


assert INSTRUCTION_NODE_16AM3 is not None
assert PATH_RE_NODE_16AM3 is not None
assert GENERATE_SEQ_NODE_16AM3 is not None
assert PARSE_PREDICTION_NODE_16AM3 is not None


INSTRUCTION_16AM3 = ast.literal_eval(
    INSTRUCTION_NODE_16AM3.value
)


PATH_RE_16AM3 = ast.literal_eval(
    PATH_RE_NODE_16AM3.value
)


GENERATE_SEQ_SOURCE_16AM3 = (
    ast.get_source_segment(
        GEN_RULE_SOURCE_16AM3,
        GENERATE_SEQ_NODE_16AM3
    )
)


PARSE_PREDICTION_SOURCE_16AM3 = (
    ast.get_source_segment(
        GEN_RULE_SOURCE_16AM3,
        PARSE_PREDICTION_NODE_16AM3
    )
)


assert GENERATE_SEQ_SOURCE_16AM3
assert PARSE_PREDICTION_SOURCE_16AM3


# ======================================================================
# 8. INSTRUCTION FIDELITY
# ======================================================================

INSTRUCTION_SHA_16AM3 = (
    sha256_text_16am3(
        INSTRUCTION_16AM3
    )
)


assert INSTRUCTION_SHA_16AM3.startswith(
    FROZEN_INSTRUCTION_SHA_PREFIX_16AM3
), (
    "Exact local INSTRUCTION does not match "
    "the development-frozen SHA."
)


print(
    "\nInstruction fidelity: PASSED"
)

print(
    " SHA:",
    INSTRUCTION_SHA_16AM3
)


# ======================================================================
# 9. EXECUTE ONLY EXACT REQUIRED PLANNER HELPERS
# ======================================================================

helper_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "torch":
        torch,

    "re":
        re,

    "PATH_RE":
        PATH_RE_16AM3,
}


exec(
    GENERATE_SEQ_SOURCE_16AM3,
    helper_ns_16am3
)


exec(
    PARSE_PREDICTION_SOURCE_16AM3,
    helper_ns_16am3
)


generate_seq_16am3 = (
    helper_ns_16am3[
        "generate_seq"
    ]
)


parse_prediction_16am3 = (
    helper_ns_16am3[
        "parse_prediction"
    ]
)


assert callable(
    generate_seq_16am3
)

assert callable(
    parse_prediction_16am3
)


print(
    "\nExact planner helpers reconstructed: PASSED"
)

print(
    " generate_seq:",
    inspect.signature(
        generate_seq_16am3
    )
)

print(
    " parse_prediction:",
    inspect.signature(
        parse_prediction_16am3
    )
)


print(
    " generate_seq SHA:",
    sha256_text_16am3(
        GENERATE_SEQ_SOURCE_16AM3
    )
)

print(
    " parse_prediction SHA:",
    sha256_text_16am3(
        PARSE_PREDICTION_SOURCE_16AM3
    )
)


# ======================================================================
# 10. EXTRACT EXACT InstructFormater WITHOUT IMPORTING utils PACKAGE
# ======================================================================

UTILS_SOURCE_16AM3 = (
    UTILS_UTILS_PATH_16AM3.read_text(
        encoding="utf-8"
    )
)


UTILS_AST_16AM3 = ast.parse(
    UTILS_SOURCE_16AM3
)


READ_PROMPT_SOURCE_16AM3 = None
INSTRUCT_FORMATTER_SOURCE_16AM3 = None


for node in UTILS_AST_16AM3.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "read_prompt"
    ):

        READ_PROMPT_SOURCE_16AM3 = (
            ast.get_source_segment(
                UTILS_SOURCE_16AM3,
                node
            )
        )


    elif (
        isinstance(
            node,
            ast.ClassDef
        )
        and
        node.name
        ==
        "InstructFormater"
    ):

        INSTRUCT_FORMATTER_SOURCE_16AM3 = (
            ast.get_source_segment(
                UTILS_SOURCE_16AM3,
                node
            )
        )


assert READ_PROMPT_SOURCE_16AM3
assert INSTRUCT_FORMATTER_SOURCE_16AM3


formatter_ns_16am3 = {
    "__builtins__":
        __builtins__,
}


exec(
    READ_PROMPT_SOURCE_16AM3,
    formatter_ns_16am3
)


exec(
    INSTRUCT_FORMATTER_SOURCE_16AM3,
    formatter_ns_16am3
)


InstructFormater_16AM3 = (
    formatter_ns_16am3[
        "InstructFormater"
    ]
)


read_prompt_16am3 = (
    formatter_ns_16am3[
        "read_prompt"
    ]
)


# Mimic exact notebook "utils.InstructFormater" reference,
# without importing utils/__init__.py.
utils_exact_16am3 = (
    types.SimpleNamespace(
        InstructFormater=
            InstructFormater_16AM3
    )
)


print(
    "\nExact InstructFormater source: PASSED"
)

print(
    " source:",
    UTILS_UTILS_PATH_16AM3
)


# ======================================================================
# 11. INDEX INITIAL PLANNER-SETUP NOTEBOOK CELLS
# ======================================================================
#
# Cell 19 is the confirmed exact import:
#
# from qa_prediction.gen_rule_path import
#     generate_seq, parse_prediction, INSTRUCTION
#
# We restrict planner model/prompt recovery to the initial planning
# setup region through cell 21, avoiding later reasoner assignments.
# ======================================================================

SETUP_MAX_CELL_16AM3 = 21


assignment_index_16am3 = {}

function_index_16am3 = {}

class_index_16am3 = {}

seed_statements_16am3 = []


def target_names_16am3(
    target
):

    if isinstance(
        target,
        ast.Name
    ):

        return [
            target.id
        ]


    if isinstance(
        target,
        (
            ast.Tuple,
            ast.List,
        )
    ):

        result = []


        for element in target.elts:

            result.extend(
                target_names_16am3(
                    element
                )
            )


        return result


    return []


for cell_idx in range(
    min(
        SETUP_MAX_CELL_16AM3 + 1,
        len(
            persisted_nb_16ar[
                "cells"
            ]
        )
    )
):

    cell = (
        persisted_nb_16ar[
            "cells"
        ][
            cell_idx
        ]
    )


    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        segment = ast.get_source_segment(
            source,
            node
        )


        if not segment:

            continue


        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            function_index_16am3.setdefault(
                node.name,
                []
            ).append(
                {
                    "cell":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        elif isinstance(
            node,
            ast.ClassDef
        ):

            class_index_16am3.setdefault(
                node.name,
                []
            ).append(
                {
                    "cell":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            if isinstance(
                node,
                ast.Assign
            ):

                targets = []


                for target in node.targets:

                    targets.extend(
                        target_names_16am3(
                            target
                        )
                    )


            else:

                targets = target_names_16am3(
                    node.target
                )


            for name in targets:

                assignment_index_16am3.setdefault(
                    name,
                    []
                ).append(
                    {
                        "cell":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


        elif isinstance(
            node,
            ast.Expr
        ):

            lower = segment.lower()


            if any(
                token in lower
                for token in [
                    "set_seed(",
                    "manual_seed(",
                    "random.seed(",
                    "np.random.seed(",
                    "numpy.random.seed(",
                ]
            ):

                seed_statements_16am3.append(
                    {
                        "cell":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


print(
    "\nInitial planner-setup index: READY"
)


# ======================================================================
# 12. PRINT CRITICAL RECOVERED ASSIGNMENT LOCATIONS
# ======================================================================

print(
    "\nCritical setup assignment locations:"
)


for symbol in [
    "N_BEAM",
    "prompter",
    "model",
    "tokenizer",
]:

    rows = assignment_index_16am3.get(
        symbol,
        []
    )


    print(
        f" {symbol}:",
        [
            row[
                "cell"
            ]
            for row in rows
        ]
    )


# ======================================================================
# 13. EXACT RECOVERY NAMESPACE
# ======================================================================

planner_setup_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "os":
        os,

    "sys":
        sys,

    "json":
        json,

    "time":
        time,

    "random":
        random,

    "re":
        re,

    "np":
        np,

    "numpy":
        np,

    "torch":
        torch,

    "Path":
        Path,

    "tqdm":
        tqdm,

    "AutoTokenizer":
        AutoTokenizer,

    "AutoModelForCausalLM":
        AutoModelForCausalLM,

    "AutoPeftModelForCausalLM":
        AutoPeftModelForCausalLM,

    # Exact source-derived replacement for utils.InstructFormater.
    "utils":
        utils_exact_16am3,

    "InstructFormater":
        InstructFormater_16AM3,

    # Exact recovered planner helpers.
    "generate_seq":
        generate_seq_16am3,

    "parse_prediction":
        parse_prediction_16am3,

    "INSTRUCTION":
        INSTRUCTION_16AM3,

    # Exact repository root.
    "REPO_DIR":
        str(
            ROG_REPO_16AM3
        ),
}


# ======================================================================
# 14. EXECUTE DEFINITIONS FROM SETUP REGION
# ======================================================================

for rows in class_index_16am3.values():

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ]
    )[
        -1
    ]


    try:

        exec(
            row[
                "source"
            ],
            planner_setup_ns_16am3
        )

    except Exception:

        pass


for rows in function_index_16am3.values():

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ]
    )[
        -1
    ]


    try:

        exec(
            row[
                "source"
            ],
            planner_setup_ns_16am3
        )

    except Exception:

        pass


# ======================================================================
# 15. RECURSIVE ASSIGNMENT RECOVERY
# ======================================================================

def extract_missing_name_16am3(
    exc
):

    match = re.search(
        r"name '([^']+)' is not defined",
        str(
            exc
        )
    )


    return (
        match.group(
            1
        )
        if match
        else None
    )


recovery_stack_16am3 = set()


def recover_symbol_16am3(
    symbol,
    max_cell=SETUP_MAX_CELL_16AM3
):

    if symbol in planner_setup_ns_16am3:

        return None


    if symbol in recovery_stack_16am3:

        raise RuntimeError(
            f"Circular recovery dependency: "
            f"{symbol}"
        )


    rows = [
        row
        for row in
        assignment_index_16am3.get(
            symbol,
            []
        )
        if row[
            "cell"
        ]
        <=
        max_cell
    ]


    assert rows, (
        f"No initial-planning assignment found "
        f"for required symbol '{symbol}'."
    )


    rows = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ],
        reverse=True
    )


    recovery_stack_16am3.add(
        symbol
    )


    last_error = None


    original_cwd = os.getcwd()


    try:

        # Relative prompt/model paths in the original repository
        # were intended to resolve from the RoG project root.
        os.chdir(
            ROG_REPO_16AM3
        )


        for row in rows:

            for _ in range(
                20
            ):

                try:

                    exec(
                        row[
                            "source"
                        ],
                        planner_setup_ns_16am3
                    )


                    if symbol in planner_setup_ns_16am3:

                        return row


                    break


                except NameError as exc:

                    last_error = exc


                    missing = (
                        extract_missing_name_16am3(
                            exc
                        )
                    )


                    if (
                        missing
                        and
                        missing in
                        assignment_index_16am3
                    ):

                        recover_symbol_16am3(
                            missing,
                            max_cell=
                                row[
                                    "cell"
                                ]
                        )

                        continue


                    if (
                        missing
                        and
                        missing in
                        function_index_16am3
                    ):

                        frow = sorted(
                            function_index_16am3[
                                missing
                            ],
                            key=lambda x:
                                x[
                                    "cell"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            frow[
                                "source"
                            ],
                            planner_setup_ns_16am3
                        )

                        continue


                    if (
                        missing
                        and
                        missing in
                        class_index_16am3
                    ):

                        crow = sorted(
                            class_index_16am3[
                                missing
                            ],
                            key=lambda x:
                                x[
                                    "cell"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            crow[
                                "source"
                            ],
                            planner_setup_ns_16am3
                        )

                        continue


                    break


                except Exception as exc:

                    last_error = exc

                    break


    finally:

        os.chdir(
            original_cwd
        )

        recovery_stack_16am3.remove(
            symbol
        )


    raise AssertionError(
        f"Could not recover '{symbol}'.\n"
        f"Last error: "
        f"{type(last_error).__name__ if last_error else 'unknown'}: "
        f"{last_error}"
    )


# ======================================================================
# 16. RECOVER N_BEAM
# ======================================================================

N_BEAM_ROW_16AM3 = (
    recover_symbol_16am3(
        "N_BEAM"
    )
)


N_BEAM_16AM3 = int(
    planner_setup_ns_16am3[
        "N_BEAM"
    ]
)


assert (
    N_BEAM_16AM3
    ==
    FROZEN_N_BEAM_16AM3
)


print(
    "\nN_BEAM recovery: PASSED"
)

print(
    " N_BEAM:",
    N_BEAM_16AM3
)


# ======================================================================
# 17. RECOVER PROMPTER
# ======================================================================

PROMPTER_ROW_16AM3 = (
    recover_symbol_16am3(
        "prompter"
    )
)


prompter_16am3 = (
    planner_setup_ns_16am3[
        "prompter"
    ]
)


assert hasattr(
    prompter_16am3,
    "format"
)


prompt_probe_16am3 = (
    prompter_16am3.format(
        instruction=
            INSTRUCTION_16AM3,

        message=
            str(
                webqsp_val_plan_rows[
                    0
                ][
                    "question"
                ]
            )
    )
)


assert isinstance(
    prompt_probe_16am3,
    str
)

assert len(
    prompt_probe_16am3
) > 0


PROMPT_PROBE_SHA_16AM3 = (
    sha256_text_16am3(
        prompt_probe_16am3
    )
)


print(
    "\nPrompter recovery: PASSED"
)

print(
    " assignment cell:",
    (
        PROMPTER_ROW_16AM3[
            "cell"
        ]
        if PROMPTER_ROW_16AM3
        else
        "already available"
    )
)

print(
    " prompt probe SHA:",
    PROMPT_PROBE_SHA_16AM3
)


# ======================================================================
# 18. MODEL IDENTITY HELPERS
# ======================================================================

def identity_strings_16am3(
    obj
):

    values = []


    for attr in [
        "name_or_path",
        "_name_or_path",
    ]:

        value = getattr(
            obj,
            attr,
            None
        )


        if isinstance(
            value,
            str
        ):

            values.append(
                value
            )


    config = getattr(
        obj,
        "config",
        None
    )


    if config is not None:

        for attr in [
            "name_or_path",
            "_name_or_path",
        ]:

            value = getattr(
                config,
                attr,
                None
            )


            if isinstance(
                value,
                str
            ):

                values.append(
                    value
                )


    return list(
        dict.fromkeys(
            values
        )
    )


def is_rog_runtime_16am3(
    obj
):

    return any(
        FROZEN_MODEL_ID_16AM3.lower()
        in value.lower()

        for value in
        identity_strings_16am3(
            obj
        )
    )


# ======================================================================
# 19. REUSE SURVIVING RoG MODEL IF PRESENT
# ======================================================================

model_16am3 = None
tokenizer_16am3 = None


for name, obj in list(
    globals().items()
):

    try:

        if (
            hasattr(
                obj,
                "generate"
            )
            and
            is_rog_runtime_16am3(
                obj
            )
        ):

            model_16am3 = obj

            print(
                "\nUsing surviving frozen RoG model:",
                name
            )

            break


    except Exception:

        pass


for name, obj in list(
    globals().items()
):

    try:

        if (
            callable(
                obj
            )
            and
            hasattr(
                obj,
                "decode"
            )
            and
            is_rog_runtime_16am3(
                obj
            )
        ):

            tokenizer_16am3 = obj

            print(
                "Using surviving frozen RoG tokenizer:",
                name
            )

            break


    except Exception:

        pass


# ======================================================================
# 20. OTHERWISE RECOVER EXACT INITIAL MODEL ASSIGNMENTS
# ======================================================================

if model_16am3 is None:

    # Ensure a stale unrelated "model" does not block recursive recovery.
    planner_setup_ns_16am3.pop(
        "model",
        None
    )


    MODEL_ROW_16AM3 = (
        recover_symbol_16am3(
            "model"
        )
    )


    model_16am3 = (
        planner_setup_ns_16am3[
            "model"
        ]
    )


else:

    MODEL_ROW_16AM3 = None


if tokenizer_16am3 is None:

    candidate = planner_setup_ns_16am3.get(
        "tokenizer"
    )


    if (
        candidate is not None
        and
        callable(
            candidate
        )
        and
        hasattr(
            candidate,
            "decode"
        )
    ):

        tokenizer_16am3 = candidate


    else:

        planner_setup_ns_16am3.pop(
            "tokenizer",
            None
        )


        TOKENIZER_ROW_16AM3 = (
            recover_symbol_16am3(
                "tokenizer"
            )
        )


        tokenizer_16am3 = (
            planner_setup_ns_16am3[
                "tokenizer"
            ]
        )


else:

    TOKENIZER_ROW_16AM3 = None


assert hasattr(
    model_16am3,
    "generate"
)


assert callable(
    tokenizer_16am3
)

assert hasattr(
    tokenizer_16am3,
    "decode"
)


MODEL_IDS_16AM3 = (
    identity_strings_16am3(
        model_16am3
    )
)


TOKENIZER_IDS_16AM3 = (
    identity_strings_16am3(
        tokenizer_16am3
    )
)


assert is_rog_runtime_16am3(
    model_16am3
), (
    "Recovered planning model is not rmanluo/RoG.\n"
    f"Runtime identity: {MODEL_IDS_16AM3}"
)


print(
    "\nPlanner model identity: PASSED"
)

print(
    " model IDs:",
    MODEL_IDS_16AM3
)

print(
    " tokenizer IDs:",
    TOKENIZER_IDS_16AM3
)


# ======================================================================
# 21. ENSURE MODEL IS USABLE FOR EXACT generate_seq
# ======================================================================

model_16am3.eval()


print(
    "Model eval mode: READY"
)


# ======================================================================
# 22. EXACT load_checkpoint FROM NOTEBOOK CELL 47
# ======================================================================

CELL47_SOURCE_16AM3 = "".join(
    persisted_nb_16ar[
        "cells"
    ][
        47
    ].get(
        "source",
        []
    )
)


CELL47_AST_16AM3 = ast.parse(
    CELL47_SOURCE_16AM3
)


LOAD_CHECKPOINT_SOURCE_16AM3 = None


for node in CELL47_AST_16AM3.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "load_checkpoint"
    ):

        LOAD_CHECKPOINT_SOURCE_16AM3 = (
            ast.get_source_segment(
                CELL47_SOURCE_16AM3,
                node
            )
        )

        break


assert LOAD_CHECKPOINT_SOURCE_16AM3


runtime_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "os":
        os,

    "json":
        json,
}


exec(
    LOAD_CHECKPOINT_SOURCE_16AM3,
    runtime_ns_16am3
)


load_checkpoint_16am3 = (
    runtime_ns_16am3[
        "load_checkpoint"
    ]
)


print(
    "\nload_checkpoint recovery: PASSED"
)

print(
    " SHA:",
    sha256_text_16am3(
        LOAD_CHECKPOINT_SOURCE_16AM3
    )
)


# ======================================================================
# 23. EXACT PLANNING WRAPPER FROM CELL 139
# ======================================================================

planning_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "load_checkpoint":
        load_checkpoint_16am3,

    "generate_seq":
        generate_seq_16am3,

    "parse_prediction":
        parse_prediction_16am3,

    "INSTRUCTION":
        INSTRUCTION_16AM3,

    "prompter":
        prompter_16am3,

    "model":
        model_16am3,

    "tokenizer":
        tokenizer_16am3,

    "N_BEAM":
        N_BEAM_16AM3,

    "time":
        time,

    "json":
        json,

    "tqdm":
        tqdm,
}


exec(
    run_planning_source_16ar,
    planning_ns_16am3
)


run_planning_checkpointed_16am3 = (
    planning_ns_16am3[
        "run_planning_checkpointed"
    ]
)


assert str(
    inspect.signature(
        run_planning_checkpointed_16am3
    )
) == (
    "(dataset_split, ckpt_path, desc)"
)


print(
    "\nExact Cell-139 planner wrapper: PASSED"
)

print(
    " SHA:",
    sha256_text_16am3(
        run_planning_source_16ar
    )
)


# ======================================================================
# 24. GOLD-LEAKAGE SOURCE GATE
# ======================================================================

generation_section_16am3 = (
    run_planning_source_16ar
    .split(
        "rec =",
        1
    )[
        0
    ]
)


assert (
    'sample["a_entity"]'
    not in
    generation_section_16am3
)


assert (
    'sample["answer"]'
    not in
    generation_section_16am3
)


assert (
    'sample["question"]'
    in
    generation_section_16am3
)


print(
    "\nLeakage source gate: PASSED"
)

print(
    " TEST gold in prompt/generation: NO"
)


# ======================================================================
# 25. SEED AUDIT
# ======================================================================
#
# do_sample=True, therefore seed behavior matters.
#
# Apply only seed statements explicitly present in the original
# initial-planning setup. Never invent seed 42 here.
# ======================================================================

print(
    "\nExplicit original initial-planning seed statements:"
)


if seed_statements_16am3:

    for row in seed_statements_16am3:

        print(
            f" cell {row['cell']}: "
            f"{row['source']}"
        )


else:

    print(
        " NONE"
    )


for row in sorted(
    seed_statements_16am3,
    key=lambda x:
        x[
            "cell"
        ]
):

    exec(
        row[
            "source"
        ],
        planner_setup_ns_16am3
    )


if not seed_statements_16am3:

    print(
        "No new random seed invented."
    )


# ======================================================================
# 26. OUTPUT PATHS
# ======================================================================

TEST_PLANNING_DIR_16AM3 = Path(
    "/kaggle/working/"
    "step2_rq1_test"
)


TEST_PLANNING_DIR_16AM3.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    TEST_PLANNING_DIR_16AM3
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    TEST_PLANNING_DIR_16AM3
    / "planning_cwq_test.jsonl"
)


MANIFEST_PATH_16AM3 = (
    TEST_PLANNING_DIR_16AM3
    / "cell16a_m3_materialization_manifest.json"
)


# ======================================================================
# 27. CHECKPOINT COUNTER
# ======================================================================

def checkpoint_count_16am3(
    path
):

    if not path.exists():

        return 0


    count = 0


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                count += 1


    return count


print(
    "\nExisting TEST planning checkpoints:"
)

print(
    " WebQSP:",
    checkpoint_count_16am3(
        WEBQSP_TEST_PLAN_PATH
    ),
    "/ 1628"
)

print(
    " CWQ:",
    checkpoint_count_16am3(
        CWQ_TEST_PLAN_PATH
    ),
    "/ 3531"
)


# ======================================================================
# 28. INPUT COUNT / DISJOINTNESS GATE
# ======================================================================

assert len(
    webqsp_official_test_16ar
) == 1628


assert len(
    cwq_official_test_16ar
) == 3531


web_val_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    webqsp_val_plan_rows
}


cwq_val_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    cwq_val_plan_rows
}


web_test_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    webqsp_raw_test_16a
}


cwq_test_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    cwq_raw_test_16a
}


assert not (
    web_val_ids_16am3
    &
    web_test_ids_16am3
)


assert not (
    cwq_val_ids_16am3
    &
    cwq_test_ids_16am3
)


print(
    "\nTEST dataset gate: PASSED"
)

print(
    " WebQSP: 1628"
)

print(
    " CWQ:    3531"
)

print(
    " validation/test overlap: 0"
)


# ======================================================================
# 29. FINAL PRE-GENERATION AUDIT
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "FINAL PRE-GENERATION FROZEN PLANNER AUDIT"
)

print(
    "=" * 126
)


print(
    "gen_rule_path source SHA:      PASSED"
)

print(
    "generate_seq exact source:     PASSED"
)

print(
    "parse_prediction exact source: PASSED"
)

print(
    "INSTRUCTION frozen SHA:        PASSED"
)

print(
    "InstructFormater exact source: PASSED"
)

print(
    "N_BEAM = 3:                    PASSED"
)

print(
    "rmanluo/RoG model identity:    PASSED"
)

print(
    "Cell-47 checkpoint loader:     PASSED"
)

print(
    "Cell-139 planning wrapper:     PASSED"
)

print(
    "TEST gold in generation:       NO"
)

print(
    "AFP changed:                   NO"
)

print(
    "TEST tuning:                   NO"
)


# ======================================================================
# 30. MATERIALIZE WEBQSP TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "MATERIALIZING WEBQSP FROZEN TEST PLANS"
)

print(
    "=" * 126
)


web_start_16am3 = time.time()


webqsp_test_plan_rows = (
    run_planning_checkpointed_16am3(
        dataset_split=
            webqsp_official_test_16ar,

        ckpt_path=
            str(
                WEBQSP_TEST_PLAN_PATH
            ),

        desc=
            "WebQSP frozen TEST planning"
    )
)


WEBQSP_PLANNING_SECONDS_16AM3 = (
    time.time()
    -
    web_start_16am3
)


print(
    "\nWebQSP planning elapsed:",
    f"{WEBQSP_PLANNING_SECONDS_16AM3/60:.2f} min"
)


# ======================================================================
# 31. MATERIALIZE CWQ TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "MATERIALIZING CWQ FROZEN TEST PLANS"
)

print(
    "=" * 126
)


cwq_start_16am3 = time.time()


cwq_test_plan_rows = (
    run_planning_checkpointed_16am3(
        dataset_split=
            cwq_official_test_16ar,

        ckpt_path=
            str(
                CWQ_TEST_PLAN_PATH
            ),

        desc=
            "CWQ frozen TEST planning"
    )
)


CWQ_PLANNING_SECONDS_16AM3 = (
    time.time()
    -
    cwq_start_16am3
)


print(
    "\nCWQ planning elapsed:",
    f"{CWQ_PLANNING_SECONDS_16AM3/60:.2f} min"
)


# ======================================================================
# 32. STRICT OUTPUT AUDIT
# ======================================================================

REQUIRED_OUTPUT_FIELDS_16AM3 = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
    "planning_time_sec",
}


def audit_plans_16am3(
    dataset_name,
    rows,
    raw_rows,
    expected_n
):

    assert len(
        rows
    ) == expected_n


    raw_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in raw_rows
    }


    output_ids = [
        str(
            row[
                "id"
            ]
        )

        for row in rows
    ]


    assert len(
        output_ids
    ) == len(
        set(
            output_ids
        )
    )


    assert set(
        output_ids
    ) == set(
        raw_by_id.keys()
    )


    total_plans = 0
    empty_questions = 0

    lengths = []


    for row in rows:

        assert REQUIRED_OUTPUT_FIELDS_16AM3.issubset(
            row.keys()
        )


        qid = str(
            row[
                "id"
            ]
        )


        raw = raw_by_id[
            qid
        ]


        assert str(
            row[
                "question"
            ]
        ) == str(
            raw[
                "question"
            ]
        )


        assert list(
            row[
                "q_entity"
            ]
        ) == list(
            raw[
                "q_entity"
            ]
        )


        assert list(
            row[
                "a_entity"
            ]
        ) == list(
            raw[
                "a_entity"
            ]
        )


        paths = row[
            "predicted_paths"
        ]


        assert isinstance(
            paths,
            list
        )


        assert len(
            paths
        ) <= 3


        total_plans += len(
            paths
        )


        if len(
            paths
        ) == 0:

            empty_questions += 1


        for plan in paths:

            assert isinstance(
                plan,
                list
            )

            lengths.append(
                len(
                    plan
                )
            )


        assert float(
            row[
                "planning_time_sec"
            ]
        ) >= 0.0


    result = {
        "questions":
            int(
                len(
                    rows
                )
            ),

        "total_predicted_plans":
            int(
                total_plans
            ),

        "mean_plans_per_question":
            float(
                total_plans
                /
                len(
                    rows
                )
            ),

        "empty_plan_questions":
            int(
                empty_questions
            ),

        "min_plan_length":
            (
                int(
                    min(
                        lengths
                    )
                )
                if lengths
                else None
            ),

        "max_plan_length":
            (
                int(
                    max(
                        lengths
                    )
                )
                if lengths
                else None
            ),

        "mean_plan_length":
            (
                float(
                    np.mean(
                        lengths
                    )
                )
                if lengths
                else None
            ),
    }


    print(
        f"\n{dataset_name.upper()} audit: PASSED"
    )

    print(
        " questions:",
        result[
            "questions"
        ]
    )

    print(
        " predicted plans:",
        result[
            "total_predicted_plans"
        ]
    )

    print(
        " mean plans/question:",
        f"{result['mean_plans_per_question']:.6f}"
    )

    print(
        " empty-plan questions:",
        result[
            "empty_plan_questions"
        ]
    )

    print(
        " plan-length min/max:",
        result[
            "min_plan_length"
        ],
        "/",
        result[
            "max_plan_length"
        ]
    )


    return result


WEBQSP_PLAN_AUDIT_16AM3 = (
    audit_plans_16am3(
        "webqsp",
        webqsp_test_plan_rows,
        webqsp_raw_test_16a,
        1628
    )
)


CWQ_PLAN_AUDIT_16AM3 = (
    audit_plans_16am3(
        "cwq",
        cwq_test_plan_rows,
        cwq_raw_test_16a,
        3531
    )
)


# ======================================================================
# 33. FILE COMPLETENESS + SHA
# ======================================================================

assert (
    checkpoint_count_16am3(
        WEBQSP_TEST_PLAN_PATH
    )
    ==
    1628
)


assert (
    checkpoint_count_16am3(
        CWQ_TEST_PLAN_PATH
    )
    ==
    3531
)


WEBQSP_TEST_PLAN_SHA = (
    sha256_file_16am3(
        WEBQSP_TEST_PLAN_PATH
    )
)


CWQ_TEST_PLAN_SHA = (
    sha256_file_16am3(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nFrozen TEST planning SHA256:"
)

print(
    " WebQSP:",
    WEBQSP_TEST_PLAN_SHA
)

print(
    " CWQ:",
    CWQ_TEST_PLAN_SHA
)


# ======================================================================
# 34. IMPLEMENTATION FINGERPRINT
# ======================================================================

PLANNER_IMPLEMENTATION_16AM3 = {
    "model_id":
        FROZEN_MODEL_ID_16AM3,

    "gen_rule_path_file_sha256":
        ACTUAL_GEN_RULE_SHA_16AM3,

    "instruction_sha256":
        INSTRUCTION_SHA_16AM3,

    "generate_seq_source_sha256":
        sha256_text_16am3(
            GENERATE_SEQ_SOURCE_16AM3
        ),

    "parse_prediction_source_sha256":
        sha256_text_16am3(
            PARSE_PREDICTION_SOURCE_16AM3
        ),

    "InstructFormater_source_sha256":
        sha256_text_16am3(
            INSTRUCT_FORMATTER_SOURCE_16AM3
        ),

    "load_checkpoint_source_sha256":
        sha256_text_16am3(
            LOAD_CHECKPOINT_SOURCE_16AM3
        ),

    "run_planning_checkpointed_source_sha256":
        sha256_text_16am3(
            run_planning_source_16ar
        ),

    "N_BEAM":
        N_BEAM_16AM3,

    "do_sample":
        True,

    "max_new_tokens":
        100,

    "model_runtime_identity":
        MODEL_IDS_16AM3,

    "tokenizer_runtime_identity":
        TOKENIZER_IDS_16AM3,

    "prompt_probe_sha256":
        PROMPT_PROBE_SHA_16AM3,

    "seed_statements":
        seed_statements_16am3,

    "invented_new_seed":
        False,

    "graph_walker_required_for_planning_helpers":
        False,

    "full_utils_package_imported":
        False,
}


PLANNER_IMPLEMENTATION_SHA_16AM3 = (
    sha256_text_16am3(
        json.dumps(
            PLANNER_IMPLEMENTATION_16AM3,
            sort_keys=True,
            separators=(
                ",",
                ":"
            ),
            ensure_ascii=False
        )
    )
)


print(
    "\nPlanner implementation fingerprint:"
)

print(
    " ",
    PLANNER_IMPLEMENTATION_SHA_16AM3
)


# ======================================================================
# 35. SAVE MATERIALIZATION MANIFEST
# ======================================================================

MATERIALIZATION_MANIFEST_16AM3 = {
    "stage":
        "frozen_test_relation_plan_materialization",

    "afp_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "planner":
        {
            **PLANNER_IMPLEMENTATION_16AM3,

            "implementation_sha256":
                PLANNER_IMPLEMENTATION_SHA_16AM3,
        },

    "origin_evidence": {
        "generate_seq":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "parse_prediction":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "INSTRUCTION":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "InstructFormater":
            (
                "exact AST extraction from "
                "src/utils/utils.py; "
                "utils package __init__ not imported"
            ),

        "load_checkpoint":
            "exact notebook cell 47 source",

        "run_planning_checkpointed":
            "exact notebook cell 139 source",
    },

    "outputs": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TEST_PLAN_PATH
                ),

            "sha256":
                WEBQSP_TEST_PLAN_SHA,

            **WEBQSP_PLAN_AUDIT_16AM3,
        },

        "cwq": {
            "path":
                str(
                    CWQ_TEST_PLAN_PATH
                ),

            "sha256":
                CWQ_TEST_PLAN_SHA,

            **CWQ_PLAN_AUDIT_16AM3,
        },
    },

    "leakage": {
        "test_gold_in_prompt":
            False,

        "test_gold_in_generation":
            False,

        "test_used_for_tuning":
            False,

        "test_used_for_model_selection":
            False,

        "test_triggered_method_change":
            False,
    },

    "afp_changed_after_freeze":
        False,

    "next":
        "rerun_Cell16_unchanged",
}


with open(
    MANIFEST_PATH_16AM3,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MATERIALIZATION_MANIFEST_16AM3,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AM3_COMPLETE = True

FROZEN_TEST_PLANS_MATERIALIZED = True


# ======================================================================
# 36. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 130
)

print(
    "=== CELL 16A-M3: "
    "FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ==="
)

print(
    "=" * 130
)


print(
    "\nRecovery strategy:"
)

print(
    " gen_rule_path module imported: NO"
)

print(
    " utils package imported:        NO"
)

print(
    " graph-walker required:         NO"
)

print(
    " exact source extracted:        YES"
)


print(
    "\nFrozen planner:"
)

print(
    " model:",
    FROZEN_MODEL_ID_16AM3
)

print(
    " N_BEAM:",
    N_BEAM_16AM3
)

print(
    " instruction SHA:",
    INSTRUCTION_SHA_16AM3
)


print(
    "\nTEST planning files:"
)

print(
    " WebQSP:"
)

print(
    "  questions:",
    len(
        webqsp_test_plan_rows
    )
)

print(
    "  file:",
    WEBQSP_TEST_PLAN_PATH
)

print(
    "  SHA:",
    WEBQSP_TEST_PLAN_SHA
)


print(
    " CWQ:"
)

print(
    "  questions:",
    len(
        cwq_test_plan_rows
    )
)

print(
    "  file:",
    CWQ_TEST_PLAN_PATH
)

print(
    "  SHA:",
    CWQ_TEST_PLAN_SHA
)


print(
    "\nScientific integrity:"
)

print(
    " TEST gold in generation: NO"
)

print(
    " TEST tuning:             NO"
)

print(
    " AFP changed:             NO"
)

print(
    " new seed invented:       NO"
)


print(
    "\nManifest:"
)

print(
    " ",
    MANIFEST_PATH_16AM3
)


print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16 UNCHANGED."
)

print(
    "Then Cell 17, then Cell 18."
)

Cell 16A-M3 freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen planner:
 model: rmanluo/RoG
 N_BEAM: 3
 do_sample: True
 max_new_tokens: 100

CUDA gate: PASSED
 GPU: Tesla T4

Exact gen_rule_path.py SHA gate: PASSED
  e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af

Instruction fidelity: PASSED
 SHA: e3687b4a5081c22cecf474857676a2d10a7132d667507eac5bebc79263e5096d

Exact planner helpers reconstructed: PASSED
 generate_seq: (model, input_text, tokenizer, num_beam=3, do_sample=False, max_new_tokens=100)
 parse_prediction: (prediction)
 generate_seq SHA: 23deb37187b09722b84af94c3c7b146b37d2d65a1755bc31c149533c57aa09d9
 parse_prediction SHA: 6fc2edaf54a0a0388cb4ccc0cb68f3cc7cbc3b6c3657c5d13f0e7d38dd9ee4ad

Exact InstructFormater source: PASSED
 source: /kaggle/working/reasoning-on-graphs/src/utils/utils.py

Initial planner-setup index: READY

Critical setup assignment locations:
 N_BEAM: [9]
 prompter: [19]
 model: [13]

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...


Planner model identity: PASSED
 model IDs: ['rmanluo/RoG']
 tokenizer IDs: ['rmanluo/RoG']
Model eval mode: READY

load_checkpoint recovery: PASSED
 SHA: a50d1a3b3f62b72c6745467a0ba96dc8b9313183865671d6ad1b7448d22e889c

Exact Cell-139 planner wrapper: PASSED
 SHA: 8dc630d5e3cb8c448e7858c03f66b9143af1599e7f8bb196f81439f917bc2bdd

Leakage source gate: PASSED
 TEST gold in prompt/generation: NO

Explicit original initial-planning seed statements:
 NONE
No new random seed invented.

Existing TEST planning checkpoints:
 WebQSP: 0 / 1628
 CWQ: 0 / 3531

TEST dataset gate: PASSED
 WebQSP: 1628
 CWQ:    3531
 validation/test overlap: 0

FINAL PRE-GENERATION FROZEN PLANNER AUDIT
gen_rule_path source SHA:      PASSED
generate_seq exact source:     PASSED
parse_prediction exact source: PASSED
INSTRUCTION frozen SHA:        PASSED
InstructFormater exact source: PASSED
N_BEAM = 3:                    PASSED
rmanluo/RoG model identity:    PASSED
Cell-47 checkpoint loader:     PASSED
Cell-139 plannin

WebQSP frozen TEST planning:   0%|          | 0/1628 [00:00<?, ?it/s]


WebQSP planning elapsed: 54.98 min

MATERIALIZING CWQ FROZEN TEST PLANS
Resuming: 0 / 3531 already planned (CWQ frozen TEST planning).


CWQ frozen TEST planning:   0%|          | 0/3531 [00:00<?, ?it/s]


CWQ planning elapsed: 139.98 min

WEBQSP audit: PASSED
 questions: 1628
 predicted plans: 4880
 mean plans/question: 2.997543
 empty-plan questions: 0
 plan-length min/max: 1 / 4

CWQ audit: PASSED
 questions: 3531
 predicted plans: 10537
 mean plans/question: 2.984140
 empty-plan questions: 10
 plan-length min/max: 1 / 5

Frozen TEST planning SHA256:
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ: 95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

Planner implementation fingerprint:
  8d5537a822cf4e8e28d586bd804f8d02c47d87592e2ecea2b12fe768ea2be058

=== CELL 16A-M3: FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ===

Recovery strategy:
 gen_rule_path module imported: NO
 utils package imported:        NO
 graph-walker required:         NO
 exact source extracted:        YES

Frozen planner:
 model: rmanluo/RoG
 N_BEAM: 3
 instruction SHA: e3687b4a5081c22cecf474857676a2d10a7132d667507eac5bebc79263e5096d

TEST planning files:
 WebQSP:
  que

In [42]:
# ======================================================================
# CPU → GPU SAFETY BACKUP
# RUN THIS BEFORE CHANGING KAGGLE ACCELERATOR
# ======================================================================

from pathlib import Path
import zipfile
import hashlib
import os

BACKUP_ZIP = Path(
    "/kaggle/working/AdaPruner_CPU_to_GPU_backup.zip"
)

critical_paths = [
    # --------------------------------------------------------------
    # Final frozen AFP development boundary
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "14_selector_ablations_and_final_freeze"
    ),

    # --------------------------------------------------------------
    # Selected scorer checkpoints
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "07_final_scorer"
    ),

    # --------------------------------------------------------------
    # Frozen selector / baseline definitions
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "08_adaptive_selector"
    ),

    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "09_controlled_baselines"
    ),

    # --------------------------------------------------------------
    # Exact validation traversal/fidelity artifacts
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "10_validation_traversal"
    ),

    # --------------------------------------------------------------
    # Validation feature/scorer evidence
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "13_feature_ablations"
    ),

    # --------------------------------------------------------------
    # Frozen validation relation plans
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step2_rq1_dev/"
        "planning_webqsp_validation.jsonl"
    ),

    Path(
        "/kaggle/working/step2_rq1_dev/"
        "planning_cwq_validation.jsonl"
    ),

    # --------------------------------------------------------------
    # Any test-recovery material created so far
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step2_rq1_test"
    ),
]


def add_path_to_zip(zf, path):
    if not path.exists():
        print("NOT FOUND:", path)
        return

    if path.is_file():
        arcname = str(path).lstrip("/")
        zf.write(path, arcname)
        print("ADDED FILE:", path)
        return

    for file_path in path.rglob("*"):
        if file_path.is_file():
            arcname = str(file_path).lstrip("/")
            zf.write(file_path, arcname)

    print("ADDED DIR :", path)


with zipfile.ZipFile(
    BACKUP_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:

    for path in critical_paths:
        add_path_to_zip(zf, path)


# SHA256
h = hashlib.sha256()

with open(BACKUP_ZIP, "rb") as f:
    while True:
        chunk = f.read(1024 * 1024)

        if not chunk:
            break

        h.update(chunk)


backup_sha = h.hexdigest()
backup_mb = BACKUP_ZIP.stat().st_size / (1024 ** 2)


print("\n" + "=" * 90)
print("BACKUP COMPLETE")
print("=" * 90)

print("File:", BACKUP_ZIP)
print(f"Size: {backup_mb:.2f} MB")
print("SHA256:", backup_sha)

print("\nIMPORTANT:")
print("Download this ZIP to your computer BEFORE switching CPU → GPU.")

ADDED DIR : /kaggle/working/step3_rq2_dev_v1/14_selector_ablations_and_final_freeze
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/07_final_scorer
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/08_adaptive_selector
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/09_controlled_baselines
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/10_validation_traversal
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/13_feature_ablations
ADDED FILE: /kaggle/working/step2_rq1_dev/planning_webqsp_validation.jsonl
ADDED FILE: /kaggle/working/step2_rq1_dev/planning_cwq_validation.jsonl
ADDED DIR : /kaggle/working/step2_rq1_test

BACKUP COMPLETE
File: /kaggle/working/AdaPruner_CPU_to_GPU_backup.zip
Size: 189.68 MB
SHA256: 3422b3f86f2e9aef00e8b00feb9cf3be6850da1705e8d997aca4601a0c47bf2b

IMPORTANT:
Download this ZIP to your computer BEFORE switching CPU → GPU.


In [43]:
# ============================================================
# VERIFY CPU→GPU BACKUP BEFORE SWITCHING SESSION
# ============================================================

from pathlib import Path
import zipfile

backup = Path(
    "/kaggle/working/AdaPruner_CPU_to_GPU_backup.zip"
)

assert backup.exists(), "Backup ZIP is missing."

required_fragments = [
    # Final freeze
    "final_afp_development_freeze.json",
    "final_afp_development_freeze.sha256",

    # Selected scorers
    "webqsp_afp_scorer_selected.pt",
    "cwq_afp_scorer_selected.pt",

    # Validation planning
    "planning_webqsp_validation.jsonl",
    "planning_cwq_validation.jsonl",

    # Controlled baseline / selector
    "controlled_baseline_definition_manifest.json",

    # Final selector/freeze development artifacts
    "webqsp_selector_component_ablation.csv",
    "cwq_selector_component_ablation.csv",
]

with zipfile.ZipFile(backup, "r") as zf:
    names = zf.namelist()

print("Backup contains", len(names), "files.")

missing = []

for fragment in required_fragments:
    matches = [
        name for name in names
        if name.endswith(fragment)
    ]

    if matches:
        print("FOUND:", fragment)
    else:
        print("MISSING:", fragment)
        missing.append(fragment)

assert not missing, (
    "\nCritical experiment files missing from backup:\n"
    + "\n".join(missing)
)

print("\n" + "=" * 80)
print("CRITICAL EXPERIMENT BACKUP VERIFIED")
print("=" * 80)
print("It is safe to keep this ZIP as the frozen-development backup.")

Backup contains 41 files.
FOUND: final_afp_development_freeze.json
FOUND: final_afp_development_freeze.sha256
FOUND: webqsp_afp_scorer_selected.pt
FOUND: cwq_afp_scorer_selected.pt
FOUND: planning_webqsp_validation.jsonl
FOUND: planning_cwq_validation.jsonl
FOUND: controlled_baseline_definition_manifest.json
FOUND: webqsp_selector_component_ablation.csv
FOUND: cwq_selector_component_ablation.csv

CRITICAL EXPERIMENT BACKUP VERIFIED
It is safe to keep this ZIP as the frozen-development backup.


In [1]:
# ======================================================================
# GPU RESTART RECOVERY CELL
# Reconstruct frozen state needed for Cell 16A-M3 and Cell 16
# ======================================================================
#
# RUN AS A NEW CELL IN THE NEW GPU SESSION.
#
# This does NOT:
#   - retrain AFP
#   - retune anything
#   - change the frozen configuration
#   - generate TEST plans
#
# It restores/reconstructs:
#   - frozen AFP boundary
#   - validation planning rows
#   - persisted notebook
#   - exact planner wrapper source
#   - official RoG validation/test datasets
#   - raw TEST rows
#   - exact traversal helpers needed by Cell 16
#   - local RoG repository copy needed by M3
#
# AFTER THIS PASSES:
#   run Cell 16A-M3 unchanged.
# ======================================================================

import ast
import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch
import networkx as nx

from datasets import load_dataset


# ======================================================================
# 1. GPU CHECK
# ======================================================================

assert torch.cuda.is_available(), (
    "GPU is still unavailable. "
    "Check Kaggle Accelerator settings."
)

print("GPU recovery session: READY")
print("GPU:", torch.cuda.get_device_name(0))


# ======================================================================
# 2. FIND / RESTORE BACKUP IF NECESSARY
# ======================================================================

WORK = Path("/kaggle/working")

DEV_ROOT = (
    WORK
    / "step3_rq2_dev_v1"
)

FREEZE_DIR = (
    DEV_ROOT
    / "14_selector_ablations_and_final_freeze"
)

FREEZE_JSON = (
    FREEZE_DIR
    / "final_afp_development_freeze.json"
)

FREEZE_SHA_FILE = (
    FREEZE_DIR
    / "final_afp_development_freeze.sha256"
)


def find_backup_zip():
    candidates = []

    search_roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    names = [
        "AdaPruner_CPU_to_GPU_backup.zip",
        "AdaPruner_FULL_experiment_backup.zip",
    ]

    for root in search_roots:
        if not root.exists():
            continue

        for name in names:
            candidates.extend(
                root.rglob(name)
            )

    return candidates


# If files survived the restart, no extraction is necessary.
if FREEZE_JSON.exists() and FREEZE_SHA_FILE.exists():

    print("\nExisting experiment files survived GPU restart.")

else:

    backup_candidates = find_backup_zip()

    assert backup_candidates, (
        "\nExperiment files are not currently under /kaggle/working "
        "and I cannot find your backup ZIP in this GPU session.\n\n"
        "Upload your downloaded "
        "AdaPruner_CPU_to_GPU_backup.zip "
        "to Kaggle, then rerun THIS recovery cell.\n\n"
        "Do NOT rerun development experiments."
    )

    # Prefer full backup if available.
    backup_candidates = sorted(
        backup_candidates,
        key=lambda p:
            (
                "FULL" not in p.name,
                str(p)
            )
    )

    BACKUP_USED = backup_candidates[0]

    print("\nRestoring backup:")
    print(" ", BACKUP_USED)

    with zipfile.ZipFile(
        BACKUP_USED,
        "r"
    ) as zf:

        # ZIP paths were stored like:
        # kaggle/working/step3...
        #
        # Extract to temporary folder, then copy back to /
        tmp_restore = (
            WORK
            / "_adapruner_gpu_restore"
        )

        if tmp_restore.exists():
            shutil.rmtree(tmp_restore)

        tmp_restore.mkdir(
            parents=True,
            exist_ok=True
        )

        zf.extractall(
            tmp_restore
        )


    restored_work = (
        tmp_restore
        / "kaggle"
        / "working"
    )

    assert restored_work.exists(), (
        "Backup structure was not recognized."
    )


    for item in restored_work.iterdir():

        destination = (
            WORK
            / item.name
        )

        if item.is_dir():

            shutil.copytree(
                item,
                destination,
                dirs_exist_ok=True
            )

        else:

            shutil.copy2(
                item,
                destination
            )


    shutil.rmtree(
        tmp_restore
    )

    print("Backup restoration: COMPLETE")


# ======================================================================
# 3. VERIFY DEVELOPMENT FREEZE
# ======================================================================

EXPECTED_FREEZE_SHA = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert FREEZE_JSON.exists()
assert FREEZE_SHA_FILE.exists()


with open(
    FREEZE_SHA_FILE,
    "r",
    encoding="utf-8"
) as f:

    freeze_sha_text = (
        f.read()
        .strip()
    )


assert freeze_sha_text == EXPECTED_FREEZE_SHA, (
    "Frozen-development SHA mismatch."
)


with open(
    FREEZE_JSON,
    "r",
    encoding="utf-8"
) as f:

    FINAL_AFP_CONFIG_FROZEN = json.load(f)


FINAL_AFP_FREEZE_SHA256 = (
    EXPECTED_FREEZE_SHA
)

AFP_DEVELOPMENT_FROZEN = True


print("\nAFP frozen boundary: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 4. SELECTED SCORER CHECKPOINT IDENTITY
# ======================================================================

SCORER_DIR = (
    DEV_ROOT
    / "07_final_scorer"
)

WEB_SCORER = (
    SCORER_DIR
    / "webqsp_afp_scorer_selected.pt"
)

CWQ_SCORER = (
    SCORER_DIR
    / "cwq_afp_scorer_selected.pt"
)


assert WEB_SCORER.exists()
assert CWQ_SCORER.exists()


def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


webqsp_ckpt_sha = file_sha256(
    WEB_SCORER
)

cwq_ckpt_sha = file_sha256(
    CWQ_SCORER
)


assert (
    webqsp_ckpt_sha
    ==
    "bee65146403d565661b5105c41831af3e81d54ed5fba1b2a99bcb37382420d9f"
)

assert (
    cwq_ckpt_sha
    ==
    "91a531c057bb02d0b311ef8b78bf02248a63cd66e57fe8959ef86e7297d9d0a1"
)


print("Selected scorer checkpoints: PASSED")


# ======================================================================
# 5. RESTORE FROZEN VALIDATION PLAN ROWS
# ======================================================================

RQ1_DEV = (
    WORK
    / "step2_rq1_dev"
)

WEB_VAL_PATH = (
    RQ1_DEV
    / "planning_webqsp_validation.jsonl"
)

CWQ_VAL_PATH = (
    RQ1_DEV
    / "planning_cwq_validation.jsonl"
)


assert WEB_VAL_PATH.exists()
assert CWQ_VAL_PATH.exists()


def read_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


webqsp_val_plan_rows = (
    read_jsonl(
        WEB_VAL_PATH
    )
)

cwq_val_plan_rows = (
    read_jsonl(
        CWQ_VAL_PATH
    )
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


print(
    "\nFrozen validation planning rows:"
)

print(
    " WebQSP:",
    len(webqsp_val_plan_rows)
)

print(
    " CWQ:",
    len(cwq_val_plan_rows)
)


# ======================================================================
# 6. LOAD PERSISTED NOTEBOOK
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists(), (
    "Persisted source notebook not found."
)


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16ar = (
        json.load(f)
    )


print(
    "\nPersisted notebook recovered:",
    NOTEBOOK_PATH
)


# ======================================================================
# 7. RECOVER EXACT CELL-139 PLANNER WRAPPER SOURCE
# ======================================================================

CELL139_SOURCE = "".join(
    persisted_nb_16ar[
        "cells"
    ][139].get(
        "source",
        []
    )
)


tree139 = ast.parse(
    CELL139_SOURCE
)


run_planning_source_16ar = None


for node in tree139.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "run_planning_checkpointed"
    ):

        run_planning_source_16ar = (
            ast.get_source_segment(
                CELL139_SOURCE,
                node
            )
        )

        break


assert run_planning_source_16ar is not None


print(
    "Exact Cell-139 planning wrapper: RECOVERED"
)


# ======================================================================
# 8. RESTORE EXACT RoG REPOSITORY TO /kaggle/working
# ======================================================================
#
# M3 expects:
#
# /kaggle/working/reasoning-on-graphs
#
# The persisted notebook input contains the same repository copy.
# ======================================================================

INPUT_ROG_REPO = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "reasoning-on-graphs"
)

WORKING_ROG_REPO = Path(
    "/kaggle/working/"
    "reasoning-on-graphs"
)


assert INPUT_ROG_REPO.exists()


if not WORKING_ROG_REPO.exists():

    print(
        "\nRestoring exact RoG repository "
        "from persisted notebook input..."
    )

    shutil.copytree(
        INPUT_ROG_REPO,
        WORKING_ROG_REPO
    )


GEN_RULE = (
    WORKING_ROG_REPO
    / "src"
    / "qa_prediction"
    / "gen_rule_path.py"
)


assert GEN_RULE.exists()


assert (
    file_sha256(
        GEN_RULE
    )
    ==
    "e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af"
)


print(
    "Exact RoG repository SHA gate: PASSED"
)


# ======================================================================
# 9. LOAD OFFICIAL RoG VALIDATION + TEST SPLITS
# ======================================================================

print(
    "\nLoading official RoG datasets..."
)


webqsp_official_val_16ar = (
    load_dataset(
        "rmanluo/RoG-webqsp",
        split="validation"
    )
)

cwq_official_val_16ar = (
    load_dataset(
        "rmanluo/RoG-cwq",
        split="validation"
    )
)

webqsp_official_test_16ar = (
    load_dataset(
        "rmanluo/RoG-webqsp",
        split="test"
    )
)

cwq_official_test_16ar = (
    load_dataset(
        "rmanluo/RoG-cwq",
        split="test"
    )
)


assert len(
    webqsp_official_val_16ar
) == 246

assert len(
    cwq_official_val_16ar
) == 3519

assert len(
    webqsp_official_test_16ar
) == 1628

assert len(
    cwq_official_test_16ar
) == 3531


print(
    " WebQSP val/test:",
    246,
    "/",
    1628
)

print(
    " CWQ val/test:",
    3519,
    "/",
    3531
)


# ======================================================================
# 10. DATA NORMALIZATION
# ======================================================================

def norm_seq_gpu_recovery(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    if not isinstance(
        value,
        list
    ):
        value = [value]

    return [
        str(x)
        for x in value
    ]


def canonical_graph_gpu_recovery(graph):

    if isinstance(
        graph,
        np.ndarray
    ):
        graph = graph.tolist()

    return [
        [
            str(x)
            for x in triple
        ]
        for triple in graph
    ]


def official_to_rows_gpu_recovery(ds):

    rows = []

    for row in ds:

        rows.append(
            {
                "id":
                    str(row["id"]),

                "question":
                    str(row["question"]),

                "answer":
                    norm_seq_gpu_recovery(
                        row.get(
                            "answer",
                            []
                        )
                    ),

                "q_entity":
                    norm_seq_gpu_recovery(
                        row["q_entity"]
                    ),

                "a_entity":
                    norm_seq_gpu_recovery(
                        row["a_entity"]
                    ),

                "graph":
                    canonical_graph_gpu_recovery(
                        row["graph"]
                    ),

                "choices":
                    row.get(
                        "choices",
                        []
                    ),
            }
        )

    return rows


webqsp_raw_test_16a = (
    official_to_rows_gpu_recovery(
        webqsp_official_test_16ar
    )
)

cwq_raw_test_16a = (
    official_to_rows_gpu_recovery(
        cwq_official_test_16ar
    )
)


assert len(
    webqsp_raw_test_16a
) == 1628

assert len(
    cwq_raw_test_16a
) == 3531


# ======================================================================
# 11. VALIDATION SOURCE-IDENTITY GATE
# ======================================================================

def verify_val_identity_gpu_recovery(
    official_ds,
    frozen_rows,
    dataset_name
):

    official = {
        str(row["id"]):
            row
        for row in official_ds
    }

    frozen = {
        str(row["id"]):
            row
        for row in frozen_rows
    }


    assert set(
        official
    ) == set(
        frozen
    )


    for qid in frozen:

        assert str(
            official[qid]["question"]
        ) == str(
            frozen[qid]["question"]
        )

        assert norm_seq_gpu_recovery(
            official[qid]["q_entity"]
        ) == norm_seq_gpu_recovery(
            frozen[qid]["q_entity"]
        )

        assert norm_seq_gpu_recovery(
            official[qid]["a_entity"]
        ) == norm_seq_gpu_recovery(
            frozen[qid]["a_entity"]
        )


    print(
        dataset_name,
        "validation source identity: PASSED"
    )


verify_val_identity_gpu_recovery(
    webqsp_official_val_16ar,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_val_identity_gpu_recovery(
    cwq_official_val_16ar,
    cwq_val_plan_rows,
    "CWQ"
)


# ======================================================================
# 12. VALIDATION / TEST DISJOINTNESS
# ======================================================================

assert not (
    {
        str(x["id"])
        for x in webqsp_val_plan_rows
    }
    &
    {
        str(x["id"])
        for x in webqsp_raw_test_16a
    }
)

assert not (
    {
        str(x["id"])
        for x in cwq_val_plan_rows
    }
    &
    {
        str(x["id"])
        for x in cwq_raw_test_16a
    }
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 13. RECOVER EXACT CELL-16 TRAVERSAL HELPERS
# ======================================================================

HELPERS_TO_RECOVER = [
    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",
]


def recover_latest_function_from_notebook(
    function_name
):

    hits = []


    for cell_idx, cell in enumerate(
        persisted_nb_16ar[
            "cells"
        ]
    ):

        if cell.get(
            "cell_type"
        ) != "code":
            continue


        source = "".join(
            cell.get(
                "source",
                []
            )
        )


        if function_name not in source:
            continue


        try:
            tree = ast.parse(
                source
            )

        except Exception:
            continue


        for node in tree.body:

            if (
                isinstance(
                    node,
                    ast.FunctionDef
                )
                and
                node.name
                ==
                function_name
            ):

                segment = (
                    ast.get_source_segment(
                        source,
                        node
                    )
                )


                if segment:

                    hits.append(
                        (
                            cell_idx,
                            segment
                        )
                    )


    assert hits, (
        f"Could not recover function: "
        f"{function_name}"
    )


    # latest persisted definition
    cell_idx, source = sorted(
        hits,
        key=lambda x:
            x[0]
    )[-1]


    ns = {
        "np": np,
        "nx": nx,
        "json": json,
        "re": __import__("re"),
    }


    exec(
        source,
        ns
    )


    print(
        f"Recovered {function_name} "
        f"from cell {cell_idx}"
    )


    return ns[
        function_name
    ]


build_exact_rog_adjacency = (
    recover_latest_function_from_notebook(
        "build_exact_rog_adjacency"
    )
)

as_entity_list = (
    recover_latest_function_from_notebook(
        "as_entity_list"
    )
)

normalize_relation_plans = (
    recover_latest_function_from_notebook(
        "normalize_relation_plans"
    )
)


print(
    "Cell-16 traversal helpers: READY"
)


# ======================================================================
# 14. RECONSTRUCT SUCCESS FLAGS FROM VERIFIED STATE
# ======================================================================
#
# These flags do not pretend RAM survived.
# They indicate that their required scientific gates were reconstructed
# successfully in this GPU session.
# ======================================================================

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True


# ======================================================================
# 15. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GPU RESTART RECOVERY COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nFrozen development:"
)

print(
    " AFP freeze SHA: VERIFIED"
)

print(
    " WebQSP scorer:   VERIFIED"
)

print(
    " CWQ scorer:      VERIFIED"
)


print(
    "\nFrozen validation planning:"
)

print(
    " WebQSP:",
    len(webqsp_val_plan_rows)
)

print(
    " CWQ:",
    len(cwq_val_plan_rows)
)


print(
    "\nFinal TEST inputs:"
)

print(
    " WebQSP:",
    len(webqsp_raw_test_16a)
)

print(
    " CWQ:",
    len(cwq_raw_test_16a)
)


print(
    "\nGPU:"
)

print(
    " ",
    torch.cuda.get_device_name(0)
)


print(
    "\nAFP retrained: NO"
)

print(
    "AFP retuned:   NO"
)

print(
    "TEST plans generated: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

GPU recovery session: READY
GPU: Tesla T4

Existing experiment files survived GPU restart.

AFP frozen boundary: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116
Selected scorer checkpoints: PASSED

Frozen validation planning rows:
 WebQSP: 246
 CWQ: 3519

Persisted notebook recovered: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Exact Cell-139 planning wrapper: RECOVERED
Exact RoG repository SHA gate: PASSED

Loading official RoG datasets...


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/3531 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

 WebQSP val/test: 246 / 1628
 CWQ val/test: 3519 / 3531
WebQSP validation source identity: PASSED
CWQ validation source identity: PASSED
Validation / TEST disjointness: PASSED


AssertionError: Could not recover function: build_exact_rog_adjacency

In [2]:
# ======================================================================
# GPU RESTART RECOVERY PATCH
# Exact RoG traversal-helper reconstruction + validation fidelity gate
# ======================================================================
#
# RUN AS A NEW CELL after the GPU recovery cell stopped.
#
# DO NOT:
#   - rerun the full GPU recovery cell
#   - retrain/tune anything
#   - run Cell 16 yet
#
# WHY THIS PATCH EXISTS
# ---------------------
# build_exact_rog_adjacency was a later verified experimental helper
# and is not present in the old persisted notebook snapshot.
#
# We reconstruct it from RoG's EXACT build_graph source and then require
# reproduction of the previously frozen validation traversal totals.
#
# If BOTH WebQSP and CWQ totals match exactly, recovery is behaviorally
# equivalent to the pre-restart traversal implementation.
# ======================================================================

import ast
import json
from pathlib import Path

import numpy as np
import networkx as nx


# ======================================================================
# 1. HARD GATES
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "persisted_nb_16ar",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "run_planning_source_16ar",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing GPU-recovery prerequisite(s):\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("GPU recovery patch freeze gate: PASSED")


# ======================================================================
# 2. RECOVER EXACT RoG build_graph SOURCE
# ======================================================================

GRAPH_UTILS_CANDIDATES = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "src/utils/graph_utils.py"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/src/utils/graph_utils.py"
    ),
]


GRAPH_UTILS_PATH = next(
    (
        path
        for path in GRAPH_UTILS_CANDIDATES
        if path.exists()
    ),
    None
)

assert GRAPH_UTILS_PATH is not None, (
    "Exact RoG graph_utils.py not found."
)


graph_utils_source = GRAPH_UTILS_PATH.read_text(
    encoding="utf-8"
)

graph_utils_tree = ast.parse(
    graph_utils_source
)


exact_build_graph_source = None


for node in graph_utils_tree.body:

    if (
        isinstance(node, ast.FunctionDef)
        and
        node.name == "build_graph"
    ):

        exact_build_graph_source = (
            ast.get_source_segment(
                graph_utils_source,
                node
            )
        )

        break


assert exact_build_graph_source is not None


build_graph_ns = {
    "nx": nx
}


exec(
    exact_build_graph_source,
    build_graph_ns
)


_exact_rog_build_graph = (
    build_graph_ns[
        "build_graph"
    ]
)


print("\nExact RoG build_graph recovered:")
print(exact_build_graph_source)


# ======================================================================
# 3. EXACT ADJACENCY ADAPTER
# ======================================================================
#
# RoG build_graph uses an UNDIRECTED simple nx.Graph:
#
#   G.add_edge(h, t, relation=r.strip())
#
# Consequences intentionally preserved:
#
# - undirected traversal
# - later duplicate unordered pair overwrites edge relation
# - neighbor insertion order is preserved by NetworkX
# - exact relation equality
# ======================================================================

def build_exact_rog_adjacency(triples):

    G = _exact_rog_build_graph(
        triples
    )

    adjacency = {}

    for node in G.nodes:

        adjacency[node] = {}

        for neighbor, attrs in G[node].items():

            adjacency[node][neighbor] = (
                attrs["relation"]
            )

    return adjacency


# ======================================================================
# 4. ENTITY NORMALIZER
# ======================================================================

def as_entity_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    if isinstance(
        value,
        set
    ):
        value = list(value)

    if isinstance(
        value,
        list
    ):
        return value

    return [value]


# ======================================================================
# 5. RELATION-PLAN NORMALIZER
# ======================================================================

def normalize_relation_plans(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    # Defensive support for serialized containers.
    if isinstance(
        value,
        str
    ):

        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(
                text
            )

            return normalize_relation_plans(
                parsed
            )

        except Exception:
            return [[text]]

    if not isinstance(
        value,
        list
    ):
        return [[value]]

    if len(value) == 0:
        return []

    # Flat relation-token list means one plan.
    if all(
        isinstance(x, str)
        for x in value
    ):
        return [
            list(value)
        ]

    plans = []

    for plan in value:

        if plan is None:
            continue

        if isinstance(
            plan,
            np.ndarray
        ):
            plan = plan.tolist()

        if isinstance(
            plan,
            tuple
        ):
            plan = list(plan)

        if isinstance(
            plan,
            str
        ):
            plans.append(
                [plan]
            )

        elif isinstance(
            plan,
            list
        ):
            # Preserve empty predicted plans.
            plans.append(
                list(plan)
            )

        else:
            plans.append(
                [plan]
            )

    return plans


# ======================================================================
# 6. EXACT VALIDATION TRAVERSAL FIDELITY CHECK
# ======================================================================

def validate_exact_rog_recovery(
    dataset_name,
    planning_rows
):

    active_hop_rows = 0
    active_prefixes_total = 0
    edges_examined_total = 0
    candidate_branches_total = 0

    reachable_plans = 0
    reachable_questions = 0

    total_plans = 0


    for rec in planning_rows:

        adjacency = (
            build_exact_rog_adjacency(
                rec["graph"]
            )
        )

        topics = (
            as_entity_list(
                rec["q_entity"]
            )
        )

        answers = set(
            as_entity_list(
                rec["a_entity"]
            )
        )

        plans = (
            normalize_relation_plans(
                rec["predicted_paths"]
            )
        )

        question_reachable = False


        for plan in plans:

            total_plans += 1

            # Frozen RQ1 convention:
            # empty relation plan contributes no active traversal row.
            if len(plan) == 0:
                continue


            active = [
                (entity,)
                for entity in topics
            ]


            for relation in plan:

                if not active:
                    break


                active_hop_rows += 1

                active_prefixes_total += (
                    len(active)
                )


                # RoG examines every unique neighbor of every
                # active PATH PREFIX endpoint before relation filtering.
                edges_examined_total += sum(
                    len(
                        adjacency.get(
                            prefix[-1],
                            {}
                        )
                    )
                    for prefix in active
                )


                candidates = []


                for prefix in active:

                    endpoint = prefix[-1]

                    for (
                        neighbor,
                        edge_relation
                    ) in adjacency.get(
                        endpoint,
                        {}
                    ).items():

                        if (
                            edge_relation
                            ==
                            relation
                        ):

                            candidates.append(
                                prefix
                                +
                                (neighbor,)
                            )


                candidate_branches_total += (
                    len(candidates)
                )


                active = candidates


            plan_reachable = any(
                prefix[-1] in answers
                for prefix in active
            )


            if plan_reachable:

                reachable_plans += 1

                question_reachable = True


        if question_reachable:

            reachable_questions += 1


    return {
        "questions":
            len(planning_rows),

        "plans":
            total_plans,

        "active_hop_rows":
            active_hop_rows,

        "active_prefixes":
            active_prefixes_total,

        "edges_examined":
            edges_examined_total,

        "candidate_branches":
            candidate_branches_total,

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            reachable_questions,
    }


# ======================================================================
# 7. PREVIOUSLY VERIFIED FROZEN VALIDATION REFERENCES
# ======================================================================

EXPECTED_WEBQSP = {
    "questions": 246,
    "plans": 721,
    "active_hop_rows": 971,
    "active_prefixes": 2440,
    "edges_examined": 341526,
    "candidate_branches": 7983,
    "reachable_plans": 345,
    "reachable_questions": 205,
}


EXPECTED_CWQ = {
    "questions": 3519,
    "plans": 10529,
    "active_hop_rows": 16564,
    "active_prefixes": 57841,
    "edges_examined": 5257272,
    "candidate_branches": 247161,
    "reachable_plans": 3971,
    "reachable_questions": 2425,
}


print(
    "\nRunning WebQSP exact RoG validation fidelity gate..."
)

WEB_RECOVERED = (
    validate_exact_rog_recovery(
        "webqsp",
        webqsp_val_plan_rows
    )
)


print(
    WEB_RECOVERED
)


print(
    "\nRunning CWQ exact RoG validation fidelity gate..."
)

CWQ_RECOVERED = (
    validate_exact_rog_recovery(
        "cwq",
        cwq_val_plan_rows
    )
)


print(
    CWQ_RECOVERED
)


# ======================================================================
# 8. EXACT VALUE-BY-VALUE ASSERTIONS
# ======================================================================

for key, expected in EXPECTED_WEBQSP.items():

    actual = WEB_RECOVERED[
        key
    ]

    assert actual == expected, (
        f"WebQSP recovery mismatch for {key}: "
        f"{actual} != {expected}"
    )


for key, expected in EXPECTED_CWQ.items():

    actual = CWQ_RECOVERED[
        key
    ]

    assert actual == expected, (
        f"CWQ recovery mismatch for {key}: "
        f"{actual} != {expected}"
    )


print(
    "\n"
    + "=" * 100
)

print(
    "EXACT RoG VALIDATION FIDELITY: PASSED"
)

print(
    "=" * 100
)

print(
    "WebQSP:"
)

print(
    " active-hop rows: 971"
)

print(
    " active prefixes: 2440"
)

print(
    " edges:           341526"
)

print(
    " candidates:      7983"
)

print(
    " reachable plans: 345"
)

print(
    " reachable q:     205"
)


print(
    "\nCWQ:"
)

print(
    " active-hop rows: 16564"
)

print(
    " active prefixes: 57841"
)

print(
    " edges:           5257272"
)

print(
    " candidates:      247161"
)

print(
    " reachable plans: 3971"
)

print(
    " reachable q:     2425"
)


# ======================================================================
# 9. RECONSTRUCT GPU-RECOVERY COMPLETION FLAGS
# ======================================================================

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True

EXACT_ROG_HELPERS_RECOVERED = True


print(
    "\n"
    + "=" * 100
)

print(
    "GPU RESTART RECOVERY PATCH COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nScientific status:"
)

print(
    " AFP freeze: VERIFIED"
)

print(
    " exact RoG traversal: VERIFIED"
)

print(
    " validation totals reproduced exactly: YES"
)

print(
    " TEST plans generated: NO"
)

print(
    " TEST tuning: NO"
)

print(
    " development changed: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

GPU recovery patch freeze gate: PASSED

Exact RoG build_graph recovered:
def build_graph(graph: list) -> nx.Graph:
    G = nx.Graph()
    for triplet in graph:
        h, r, t = triplet
        G.add_edge(h, t, relation=r.strip())
    return G

Running WebQSP exact RoG validation fidelity gate...
{'questions': 246, 'plans': 721, 'active_hop_rows': 971, 'active_prefixes': 2440, 'edges_examined': 341526, 'candidate_branches': 7983, 'reachable_plans': 345, 'reachable_questions': 205}

Running CWQ exact RoG validation fidelity gate...
{'questions': 3519, 'plans': 10536, 'active_hop_rows': 16564, 'active_prefixes': 57841, 'edges_examined': 5257272, 'candidate_branches': 247161, 'reachable_plans': 3971, 'reachable_questions': 2425}


AssertionError: CWQ recovery mismatch for plans: 10536 != 10529

In [3]:
# ======================================================================
# GPU RECOVERY PATCH-FIX
# Correct CWQ total-plan vs non-empty-plan reference
# ======================================================================
#
# RUN AS A NEW CELL.
#
# Do NOT rerun the full GPU recovery.
# Do NOT rerun the previous large recovery patch.
#
# No TEST generation occurs here.
# ======================================================================


# ----------------------------------------------------------------------
# 1. Frozen validation references — corrected
# ----------------------------------------------------------------------

EXPECTED_WEBQSP_CORRECTED = {
    "questions": 246,
    "plans": 721,
    "active_hop_rows": 971,
    "active_prefixes": 2440,
    "edges_examined": 341526,
    "candidate_branches": 7983,
    "reachable_plans": 345,
    "reachable_questions": 205,
}


EXPECTED_CWQ_CORRECTED = {
    "questions": 3519,

    # IMPORTANT:
    # 10,536 = ALL predicted relation plans
    # 10,529 = NON-EMPTY predicted relation plans
    # 7      = empty relation plans
    "plans": 10536,

    "active_hop_rows": 16564,
    "active_prefixes": 57841,
    "edges_examined": 5257272,
    "candidate_branches": 247161,
    "reachable_plans": 3971,
    "reachable_questions": 2425,
}


# ----------------------------------------------------------------------
# 2. Verify already-computed traversal results
# ----------------------------------------------------------------------

assert "WEB_RECOVERED" in globals()
assert "CWQ_RECOVERED" in globals()


for key, expected in EXPECTED_WEBQSP_CORRECTED.items():

    actual = WEB_RECOVERED[key]

    assert actual == expected, (
        f"WebQSP mismatch for {key}: "
        f"{actual} != {expected}"
    )


for key, expected in EXPECTED_CWQ_CORRECTED.items():

    actual = CWQ_RECOVERED[key]

    assert actual == expected, (
        f"CWQ mismatch for {key}: "
        f"{actual} != {expected}"
    )


# ----------------------------------------------------------------------
# 3. Explicitly verify CWQ empty/non-empty plan distinction
# ----------------------------------------------------------------------

cwq_total_plans = 0
cwq_empty_plans = 0
cwq_nonempty_plans = 0


for rec in cwq_val_plan_rows:

    plans = normalize_relation_plans(
        rec["predicted_paths"]
    )

    for plan in plans:

        cwq_total_plans += 1

        if len(plan) == 0:
            cwq_empty_plans += 1
        else:
            cwq_nonempty_plans += 1


assert cwq_total_plans == 10536
assert cwq_empty_plans == 7
assert cwq_nonempty_plans == 10529


print("CWQ plan accounting: PASSED")
print("  total plans:    ", cwq_total_plans)
print("  non-empty plans:", cwq_nonempty_plans)
print("  empty plans:    ", cwq_empty_plans)


# ----------------------------------------------------------------------
# 4. WebQSP plan accounting
# ----------------------------------------------------------------------

web_total_plans = 0
web_empty_plans = 0
web_nonempty_plans = 0


for rec in webqsp_val_plan_rows:

    plans = normalize_relation_plans(
        rec["predicted_paths"]
    )

    for plan in plans:

        web_total_plans += 1

        if len(plan) == 0:
            web_empty_plans += 1
        else:
            web_nonempty_plans += 1


assert web_total_plans == 721


print("\nWebQSP plan accounting: PASSED")
print("  total plans:    ", web_total_plans)
print("  non-empty plans:", web_nonempty_plans)
print("  empty plans:    ", web_empty_plans)


# ----------------------------------------------------------------------
# 5. Mark reconstructed traversal state as verified
# ----------------------------------------------------------------------

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True
EXACT_ROG_HELPERS_RECOVERED = True
EXACT_ROG_VALIDATION_FIDELITY_PASSED = True


print(
    "\n"
    + "=" * 100
)

print(
    "EXACT RoG VALIDATION FIDELITY: PASSED"
)

print(
    "=" * 100
)


print("\nWebQSP")
print("  plans:             721")
print("  active-hop rows:   971")
print("  active prefixes:   2440")
print("  edges:              341526")
print("  candidates:         7983")
print("  reachable plans:    345")
print("  reachable questions:205")


print("\nCWQ")
print("  total plans:        10536")
print("  non-empty plans:    10529")
print("  empty plans:        7")
print("  active-hop rows:    16564")
print("  active prefixes:    57841")
print("  edges:               5257272")
print("  candidates:          247161")
print("  reachable plans:     3971")
print("  reachable questions: 2425")


print(
    "\nGPU RESTART RECOVERY: COMPLETE"
)

print(
    "Development changed: NO"
)

print(
    "TEST plans generated: NO"
)

print(
    "TEST tuning: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

CWQ plan accounting: PASSED
  total plans:     10536
  non-empty plans: 10529
  empty plans:     7

WebQSP plan accounting: PASSED
  total plans:     721
  non-empty plans: 721
  empty plans:     0

EXACT RoG VALIDATION FIDELITY: PASSED

WebQSP
  plans:             721
  active-hop rows:   971
  active prefixes:   2440
  edges:              341526
  candidates:         7983
  reachable plans:    345
  reachable questions:205

CWQ
  total plans:        10536
  non-empty plans:    10529
  empty plans:        7
  active-hop rows:    16564
  active prefixes:    57841
  edges:               5257272
  candidates:          247161
  reachable plans:     3971
  reachable questions: 2425

GPU RESTART RECOVERY: COMPLETE
Development changed: NO
TEST plans generated: NO
TEST tuning: NO

NEXT ACTION:
RUN CELL 16A-M3 UNCHANGED.


## Final frozen-test RQ2 controlled comparison

## Final RQ2 tables/statistics/artifact saving